In [2]:
# ============================================================
# PHASE 2 — HIGH-SPEED ENVIRONMENT CHECK
# ============================================================

import os
import sys
import time
import json
import platform

print("=" * 70)
print("PHASE 2 — LEGAL NER")
print("HIGH-SPEED OPTIMIZED SETUP")
print("=" * 70)

print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : ", end="")

import torch
print(torch.__version__)

print(f"CUDA         : {torch.version.cuda}")
print(f"CUDA usable  : {torch.cuda.is_available()}")
print(f"GPU count    : {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {gpu.name} | "
        f"VRAM={gpu.total_memory / 1024**3:.2f} GB"
    )

print("=" * 70)

# Performance settings
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("TF32 matmul  : ENABLED")
print("cuDNN TF32   : ENABLED")

print("=" * 70)
print("PHASE 2 STATUS: 0%")
print("=" * 70)

PHASE 2 — LEGAL NER
HIGH-SPEED OPTIMIZED SETUP
Python       : 3.13.15
PyTorch      : 2.11.0+cu128
CUDA         : 12.8
CUDA usable  : True
GPU count    : 2
GPU 0: Tesla T4 | VRAM=14.56 GB
GPU 1: Tesla T4 | VRAM=14.56 GB
TF32 matmul  : ENABLED
cuDNN TF32   : ENABLED
PHASE 2 STATUS: 0%


In [3]:
# ============================================================
# PHASE 2 — DATASET DISCOVERY
# CPU ONLY
# ============================================================

import os

print("=" * 70)
print("SEARCHING FOR PHASE_2_LNER")
print("=" * 70)

matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for d in dirs:
        if "phase_2" in d.lower() or "lner" in d.lower():
            path = os.path.join(root, d)
            matches.append(path)

for root, dirs, files in os.walk("/kaggle/working"):
    for d in dirs:
        if "phase_2" in d.lower() or "lner" in d.lower():
            path = os.path.join(root, d)
            matches.append(path)

if matches:
    for path in matches:
        print(path)
else:
    print("No Phase_2_LNER directory found.")

print("=" * 70)

SEARCHING FOR PHASE_2_LNER
No Phase_2_LNER directory found.


In [4]:
from huggingface_hub import list_repo_files

repo_id = "Exploration-Lab/CS-781-Capstone"

files = list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

for f in files:
    if "Phase_2_LNER" in f:
        print(f)

Phase_2_LNER/Test/fold3-text.json
Phase_2_LNER/Train/fold1-gold.json
Phase_2_LNER/Train/fold1-text.json
Phase_2_LNER/Train/fold2-gold.json
Phase_2_LNER/Train/fold2-text.json
Phase_2_LNER/Train/labels.txt


In [5]:
import requests

BASE_URL = "https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main"

files = [
    "Phase_2_LNER/Train/fold1-text.json",
    "Phase_2_LNER/Train/fold1-gold.json",
    "Phase_2_LNER/Train/fold2-text.json",
    "Phase_2_LNER/Train/fold2-gold.json",
    "Phase_2_LNER/Train/labels.txt",
    "Phase_2_LNER/Test/fold3-text.json",
]

print("Checking official Phase-2 files...\n")

for path in files:
    url = f"{BASE_URL}/{path}"
    r = requests.head(url, allow_redirects=True, timeout=30)

    size_mb = int(r.headers.get("content-length", 0)) / (1024**2)

    print(f"✓ {path}")
    print(f"  Status: {r.status_code}")
    print(f"  Size:   {size_mb:.2f} MB")

Checking official Phase-2 files...

✓ Phase_2_LNER/Train/fold1-text.json
  Status: 401
  Size:   0.00 MB
✓ Phase_2_LNER/Train/fold1-gold.json
  Status: 401
  Size:   0.00 MB
✓ Phase_2_LNER/Train/fold2-text.json
  Status: 401
  Size:   0.00 MB
✓ Phase_2_LNER/Train/fold2-gold.json
  Status: 401
  Size:   0.00 MB
✓ Phase_2_LNER/Train/labels.txt
  Status: 401
  Size:   0.00 MB
✓ Phase_2_LNER/Test/fold3-text.json
  Status: 401
  Size:   0.00 MB


In [6]:
import os

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

print("HF token loaded:", bool(HF_TOKEN))
print("Token length:", len(HF_TOKEN))

HF token loaded: True
Token length: 37


In [7]:
import requests

url = "https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_2_LNER/Train/labels.txt"

headers = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

r = requests.get(url, headers=headers, timeout=30)

print("Status:", r.status_code)

if r.ok:
    print("\nLABELS:")
    print(r.text)
else:
    print(r.text[:500])

Status: 200

LABELS:
APP
RESP
A.COUNSEL
R.COUNSEL
JUDGE
WIT
AUTH
COURT
STAT
PREC
DATE
CASENO


In [8]:
import requests
import json
import time

BASE_URL = "https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main"

FILES = {
    "fold1_text": "Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text": "Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold": "Phase_2_LNER/Train/fold2-gold.json",
    "labels": "Phase_2_LNER/Train/labels.txt",
    "fold3_text": "Phase_2_LNER/Test/fold3-text.json",
}

headers = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

def load_json_remote(relative_path):
    url = f"{BASE_URL}/{relative_path}"
    r = requests.get(url, headers=headers, timeout=120)
    r.raise_for_status()
    return r.json()

def load_text_remote(relative_path):
    url = f"{BASE_URL}/{relative_path}"
    r = requests.get(url, headers=headers, timeout=120)
    r.raise_for_status()
    return r.text

t0 = time.time()

fold1_text = load_json_remote(FILES["fold1_text"])
fold1_gold = load_json_remote(FILES["fold1_gold"])

fold2_text = load_json_remote(FILES["fold2_text"])
fold2_gold = load_json_remote(FILES["fold2_gold"])

fold3_text = load_json_remote(FILES["fold3_text"])

labels_text = load_text_remote(FILES["labels"])
labels = [x.strip() for x in labels_text.splitlines() if x.strip()]

elapsed = time.time() - t0

print("=" * 70)
print("PHASE 2 DATA LOADED INTO RAM")
print("=" * 70)

print(f"Fold 1 text : {len(fold1_text):,}")
print(f"Fold 1 gold : {len(fold1_gold):,}")
print(f"Fold 2 text : {len(fold2_text):,}")
print(f"Fold 2 gold : {len(fold2_gold):,}")
print(f"Fold 3 test : {len(fold3_text):,}")
print(f"Labels     : {len(labels)}")

print("\nLabels:")
print(labels)

print(f"\nLoad time: {elapsed:.2f} sec")
print("Storage: RAM only")
print("GPU used: 0")

PHASE 2 DATA LOADED INTO RAM
Fold 1 text : 35
Fold 1 gold : 35
Fold 2 text : 35
Fold 2 gold : 35
Fold 3 test : 35
Labels     : 12

Labels:
['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']

Load time: 1.04 sec
Storage: RAM only
GPU used: 0


In [9]:
print("=" * 70)
print("FOLD 1 TEXT SAMPLE")
print("=" * 70)

first_text_key = next(iter(fold1_text))
print("Key:", first_text_key)
print("Type:", type(fold1_text[first_text_key]))

print("\nContent:")
print(fold1_text[first_text_key])

print("\n" + "=" * 70)
print("FOLD 1 GOLD SAMPLE")
print("=" * 70)

first_gold_key = next(iter(fold1_gold))
print("Key:", first_gold_key)
print("Type:", type(fold1_gold[first_gold_key]))

print("\nContent:")
print(fold1_gold[first_gold_key])

FOLD 1 TEXT SAMPLE
Key: 115651329
Type: <class 'str'>

Content:
REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the fateful day i.e. 18.11.1994, at about 8.00 A.M. in the morning the complainant Jagdish (PW-5) along with his two sons namely Sukhbir and Mange Ram (PW-6) were busy in cutting pullas (reeds) from the dola of their field. At that time, Jage Ram (A-1) and his sons Rajbir Singh @ Raju (A-2), Rakesh (A-3) and Madan (A-4) armed with jaily, pharsi and lathis respectively, entered th

In [10]:
# ============================================================
# STEP 2.4 — INSPECT ACTUAL TEXT + GOLD ANNOTATIONS
# GPU: OFF
# ============================================================

import pprint

doc_id = next(iter(fold1_text))

print("=" * 80)
print("DOCUMENT ID")
print("=" * 80)
print(doc_id)

print("\n" + "=" * 80)
print("TEXT OBJECT")
print("=" * 80)
pprint.pp(fold1_text[doc_id], width=120)

print("\n" + "=" * 80)
print("GOLD OBJECT")
print("=" * 80)
pprint.pp(fold1_gold[doc_id], width=120)

DOCUMENT ID
115651329

TEXT OBJECT
('REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. '
 '..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against '
 'the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, '
 'whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the '
 'appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the fateful day i.e. '
 '18.11.1994, at about 8.00 A.M. in the morning the complainant Jagdish (PW-5) along with his two sons namely Sukhbir '
 'and Mange Ram (PW-6) were busy in cutting pullas (reeds) from the dola of their field. At that time, Jage Ram (A-1) '
 'and his sons Rajbir Singh @ Raju (A-2), Rakesh (A-3) and Madan (A-4) armed with jaily, pharsi and lathis '
 'respectively, enter

In [11]:
# ============================================================
# STEP 2.5 — COMPLETE DATASET ANALYSIS
# GPU: OFF
# ============================================================

from collections import Counter, defaultdict
import numpy as np

train_text = {**fold1_text, **fold2_text}
train_gold = {**fold1_gold, **fold2_gold}

label_counts = Counter()
entity_lengths = Counter()
doc_entity_counts = []
doc_lengths = []

invalid_spans = []
text_mismatches = []
overlaps = []

for doc_id, text_obj in train_text.items():

    # Handle the actual text representation
    if isinstance(text_obj, str):
        text = text_obj
    elif isinstance(text_obj, (list, tuple)) and len(text_obj) == 1:
        text = text_obj[0]
    else:
        text = str(text_obj)

    doc_lengths.append(len(text))

    annotations = train_gold[doc_id]
    doc_entity_counts.append(len(annotations))

    spans = []

    for ann in annotations:
        start, end, label, entity_text = ann

        label_counts[label] += 1
        entity_lengths[label] += end - start

        # Boundary validation
        if start < 0 or end > len(text) or start >= end:
            invalid_spans.append(
                (doc_id, start, end, label, entity_text)
            )
            continue

        # Exact text validation
        actual = text[start:end]

        if actual != entity_text:
            text_mismatches.append(
                (doc_id, start, end, label, entity_text, actual)
            )

        spans.append((start, end, label))

    # Check overlapping annotations
    spans_sorted = sorted(spans)

    for i in range(1, len(spans_sorted)):
        prev_start, prev_end, _ = spans_sorted[i - 1]
        cur_start, cur_end, _ = spans_sorted[i]

        if cur_start < prev_end:
            overlaps.append(
                (doc_id, spans_sorted[i - 1], spans_sorted[i])
            )

print("=" * 80)
print("PHASE 2 DATASET ANALYSIS")
print("=" * 80)

print(f"Training documents       : {len(train_text)}")
print(f"Total entities           : {sum(label_counts.values()):,}")
print(f"Average entities/doc     : {np.mean(doc_entity_counts):.2f}")
print(f"Median entities/doc      : {np.median(doc_entity_counts):.0f}")

print(f"\nAverage document length  : {np.mean(doc_lengths):,.0f} chars")
print(f"Median document length   : {np.median(doc_lengths):,.0f} chars")
print(f"Maximum document length  : {max(doc_lengths):,} chars")

print("\n" + "-" * 80)
print("ENTITY DISTRIBUTION")
print("-" * 80)

total = sum(label_counts.values())

print(f"{'LABEL':<12} {'COUNT':>8} {'%':>8} {'AVG LEN':>10}")

for label in labels:
    count = label_counts[label]
    avg_len = (
        entity_lengths[label] / count
        if count else 0
    )

    print(
        f"{label:<12} "
        f"{count:>8,} "
        f"{100 * count / total:>7.2f}% "
        f"{avg_len:>9.2f}"
    )

print("\n" + "-" * 80)
print("ANNOTATION VALIDATION")
print("-" * 80)

print(f"Invalid spans            : {len(invalid_spans)}")
print(f"Text/span mismatches     : {len(text_mismatches)}")
print(f"Overlapping spans        : {len(overlaps)}")

if text_mismatches:
    print("\nFIRST MISMATCH:")
    print(text_mismatches[0])

if overlaps:
    print("\nFIRST OVERLAP:")
    print(overlaps[0])

print("\nDataset analysis complete.")

PHASE 2 DATASET ANALYSIS
Training documents       : 70
Total entities           : 8,393
Average entities/doc     : 119.90
Median entities/doc      : 105

Average document length  : 31,624 chars
Median document length   : 26,620 chars
Maximum document length  : 114,044 chars

--------------------------------------------------------------------------------
ENTITY DISTRIBUTION
--------------------------------------------------------------------------------
LABEL           COUNT        %    AVG LEN
APP               407    4.85%     14.58
RESP              381    4.54%     16.34
A.COUNSEL         195    2.32%     11.74
R.COUNSEL         184    2.19%     11.82
JUDGE             245    2.92%     13.78
WIT               187    2.23%     11.98
AUTH            1,003   11.95%     14.21
COURT           1,013   12.07%     13.20
STAT            1,260   15.01%     17.27
PREC            1,183   14.10%     38.28
DATE            1,575   18.77%     11.32
CASENO            760    9.06%     21.14

-------

In [12]:
import os

print("Searching for tokenizer-related files...\n")

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        name = f.lower()
        if any(x in name for x in [
            "tokenizer",
            "vocab",
            "merges",
            "tokenizer.json"
        ]):
            print(os.path.join(root, f))

Searching for tokenizer-related files...



In [13]:
import os

print("=" * 80)
print("PHASE-1 CHECKPOINT FILES")
print("=" * 80)

for root, dirs, files in os.walk("/kaggle/working"):
    # Show checkpoint/model-related directories
    if any(x in root.lower() for x in [
        "a1", "a2", "a3", "a4",
        "checkpoint", "phase1"
    ]):
        print(f"\n[{root}]")
        for f in files[:30]:
            print(" ", f)

PHASE-1 CHECKPOINT FILES


In [14]:
import os

print("=" * 80)
print("PHASE-1 INPUT FILES")
print("=" * 80)

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".pt", ".json", ".txt")):
            print(os.path.join(root, f))

PHASE-1 INPUT FILES
/kaggle/input/datasets/aq1de3/fold2-go/fold2-gold.json
/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt
/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt
/kaggle/input/datasets/aq1de3/fold3-te/fold3-text.json
/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json
/kaggle/input/datasets/aq1de3/a4-ner-exp4-lorma-epoch3/a4_ner_exp4_lorma_epoch3.pt
/kaggle/input/datasets/aq1de3/fold2-te/fold2-text.json
/kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt
/kaggle/input/datasets/aq1de3/fold1-tex/fold1-text.json
/kaggle/input/datasets/aq1de3/fold1-gold1/fold1-gold.json
/kaggle/input/datasets/aq1de3/a3check/step_0001000.pt
/kaggle/input/datasets/aq1de3/fold2-tex/fold2-text.json
/kaggle/input/datasets/aq1de3/labelses/labels.txt


In [15]:
# ============================================================
# STEP 2.6 — INSPECT PHASE-1 CHECKPOINTS
# GPU: OFF
# ============================================================

import torch
import os

CHECKPOINTS = {
    "A1": "/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/a3check/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt",
}

print("=" * 80)
print("PHASE-1 CHECKPOINT METADATA")
print("=" * 80)

for name, path in CHECKPOINTS.items():

    print(f"\n{'='*30} {name} {'='*30}")
    print("File:", path)
    print("Size:", f"{os.path.getsize(path)/(1024**3):.3f} GB")

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Type:", type(ckpt).__name__)

    if isinstance(ckpt, dict):
        print("Keys:", list(ckpt.keys()))

        for key in [
            "step",
            "tokens_processed",
            "train_loss",
            "val_loss",
            "reason",
            "saved_at"
        ]:
            if key in ckpt:
                print(f"{key}: {ckpt[key]}")

    del ckpt

PHASE-1 CHECKPOINT METADATA

============================== A1 ==============================
File: /kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt
Size: 0.580 GB
Type: dict
Keys: ['step', 'epoch', 'train_loss', 'val_loss', 'model_state_dict', 'optimizer_state_dict']
step: 500
train_loss: 5.366012096405029
val_loss: None

============================== A2 ==============================
File: /kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt
Size: 0.192 GB
Type: dict
Keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
step: 9000
tokens_processed: 73728000
train_loss: 6.801188290119171
reason: regular_checkpoint
saved_at: 2026-10-05T19:17:55.016904

============================== A3 ==============================
File: /kaggle/input/datasets/aq1de3/a3check/step_0001000.pt
Size: 0.193 GB
Type: dict
Keys: ['model', 'step', 'tokens_seen', 'architecture', 'parameters', 'context_length', 'hidden_size', 'num_layers', 'num_hea

In [16]:
# ============================================================
# TOKENIZER INSPECTION
# ============================================================

import json
import os

TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json"

with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
    tokenizer_config = json.load(f)

print("=" * 80)
print("TOKENIZER")
print("=" * 80)

print("File size:",
      f"{os.path.getsize(TOKENIZER_PATH)/(1024**2):.2f} MB")

print("Top-level keys:")
print(list(tokenizer_config.keys()))

if "model" in tokenizer_config:
    print("\nTokenizer model type:",
          tokenizer_config["model"].get("type"))

    print("Vocabulary size:",
          len(tokenizer_config["model"].get("vocab", {})))

print("\nSpecial tokens:")
print(tokenizer_config.get("added_tokens", [])[:20])

TOKENIZER
File size: 3.37 MB
Top-level keys:
['version', 'truncation', 'padding', 'added_tokens', 'normalizer', 'pre_tokenizer', 'post_processor', 'decoder', 'model']

Tokenizer model type: BPE
Vocabulary size: 50000

Special tokens:
[{'id': 0, 'content': '[PAD]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 1, 'content': '[UNK]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 2, 'content': '[BOS]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 3, 'content': '[EOS]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}, {'id': 4, 'content': '[MASK]', 'single_word': False, 'lstrip': False, 'rstrip': False, 'normalized': False, 'special': True}]


In [17]:
# ============================================================
# STEP 2.7 — CHARACTER ↔ TOKEN OFFSET TEST
# GPU: OFF
# ============================================================

from tokenizers import Tokenizer

TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

doc_id = next(iter(fold1_text))
text = fold1_text[doc_id]

encoding = tokenizer.encode(text)

print("=" * 80)
print("TOKENIZER OFFSET TEST")
print("=" * 80)

print("Document ID       :", doc_id)
print("Characters        :", len(text))
print("Tokens            :", len(encoding.ids))
print("First 30 tokens   :")

for i in range(min(30, len(encoding.ids))):
    token = encoding.tokens[i]
    start, end = encoding.offsets[i]

    print(
        f"{i:4d} | "
        f"{token!r:25s} | "
        f"({start:5d}, {end:5d}) | "
        f"{text[start:end]!r}"
    )

TOKENIZER OFFSET TEST
Document ID       : 115651329
Characters        : 18616
Tokens            : 3972
First 30 tokens   :
   0 | 'REPORTABLE'              | (    0,    10) | 'REPORTABLE'
   1 | 'ĠIN'                     | (   10,    13) | ' IN'
   2 | 'ĠTHE'                    | (   13,    17) | ' THE'
   3 | 'ĠSUPREME'                | (   17,    25) | ' SUPREME'
   4 | 'ĠCOURT'                  | (   25,    31) | ' COURT'
   5 | 'ĠOF'                     | (   31,    34) | ' OF'
   6 | 'ĠINDIA'                  | (   34,    40) | ' INDIA'
   7 | 'ĠCRIMINAL'               | (   40,    49) | ' CRIMINAL'
   8 | 'ĠAPPELLATE'              | (   49,    59) | ' APPELLATE'
   9 | 'ĠJURISDICTION'           | (   59,    72) | ' JURISDICTION'
  10 | 'ĠCRIMINAL'               | (   72,    81) | ' CRIMINAL'
  11 | 'ĠAPPEAL'                 | (   81,    88) | ' APPEAL'
  12 | 'ĠNO'                     | (   88,    91) | ' NO'
  13 | '.'                       | (   91,    92) | '.'
  14 | 'Ġ92'   

In [18]:
# ============================================================
# STEP 2.7B — FULL CHARACTER/TOKEN ALIGNMENT ANALYSIS
# GPU: OFF
# ============================================================

from tokenizers import Tokenizer
from collections import Counter

TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

train_text = {**fold1_text, **fold2_text}
train_gold = {**fold1_gold, **fold2_gold}

stats = Counter()

examples = {
    "exact_boundary": [],
    "start_inside_token": [],
    "end_inside_token": [],
    "both_inside": [],
    "whitespace_boundary": [],
    "no_covering_tokens": []
}

alignment_records = []

for doc_id, text_obj in train_text.items():

    text = text_obj if isinstance(text_obj, str) else str(text_obj)

    encoding = tokenizer.encode(text)

    offsets = encoding.offsets

    for ann in train_gold[doc_id]:

        gold_start, gold_end, label, entity_text = ann

        # Find tokens intersecting the gold span.
        covered = []

        for i, (tok_start, tok_end) in enumerate(offsets):
            if tok_end > gold_start and tok_start < gold_end:
                covered.append(i)

        if not covered:
            stats["no_covering_tokens"] += 1

            if len(examples["no_covering_tokens"]) < 10:
                examples["no_covering_tokens"].append(
                    (doc_id, ann)
                )

            continue

        first_i = covered[0]
        last_i = covered[-1]

        tok_start = offsets[first_i][0]
        tok_end = offsets[last_i][1]

        start_exact = tok_start == gold_start
        end_exact = tok_end == gold_end

        if start_exact and end_exact:
            stats["exact_boundary"] += 1

            if len(examples["exact_boundary"]) < 5:
                examples["exact_boundary"].append(
                    (doc_id, ann, first_i, last_i,
                     offsets[first_i], offsets[last_i])
                )

        elif not start_exact and end_exact:
            stats["start_inside_token"] += 1

            if len(examples["start_inside_token"]) < 10:
                examples["start_inside_token"].append(
                    (doc_id, ann, first_i, last_i,
                     offsets[first_i], offsets[last_i],
                     text[tok_start:gold_start])
                )

        elif start_exact and not end_exact:
            stats["end_inside_token"] += 1

            if len(examples["end_inside_token"]) < 10:
                examples["end_inside_token"].append(
                    (doc_id, ann, first_i, last_i,
                     offsets[first_i], offsets[last_i],
                     text[gold_end:tok_end])
                )

        else:
            stats["both_inside"] += 1

            if len(examples["both_inside"]) < 10:
                examples["both_inside"].append(
                    (doc_id, ann, first_i, last_i,
                     offsets[first_i], offsets[last_i],
                     text[tok_start:gold_start],
                     text[gold_end:tok_end])
                )

        # Check whether the discrepancy is only whitespace.
        left_extra = text[tok_start:gold_start]
        right_extra = text[gold_end:tok_end]

        if (
            (not start_exact or not end_exact)
            and left_extra.strip() == ""
            and right_extra.strip() == ""
        ):
            stats["whitespace_boundary"] += 1

        alignment_records.append({
            "doc_id": doc_id,
            "start": gold_start,
            "end": gold_end,
            "label": label,
            "entity_text": entity_text,
            "first_token": first_i,
            "last_token": last_i,
            "token_start": tok_start,
            "token_end": tok_end,
            "start_exact": start_exact,
            "end_exact": end_exact,
            "left_extra": left_extra,
            "right_extra": right_extra
        })

total = len(alignment_records)

print("=" * 80)
print("CHARACTER ↔ TOKEN ALIGNMENT ANALYSIS")
print("=" * 80)

print(f"Entities analyzed       : {total:,}")
print()

for key in [
    "exact_boundary",
    "start_inside_token",
    "end_inside_token",
    "both_inside",
    "whitespace_boundary",
    "no_covering_tokens"
]:
    count = stats[key]

    if key == "no_covering_tokens":
        denom = sum(stats.values())
    else:
        denom = total

    pct = 100 * count / denom if denom else 0

    print(f"{key:<25}: {count:>6,} ({pct:6.2f}%)")

print("\n" + "=" * 80)
print("EXAMPLES — START INSIDE TOKEN")
print("=" * 80)

for x in examples["start_inside_token"][:5]:
    print(x)

print("\n" + "=" * 80)
print("EXAMPLES — END INSIDE TOKEN")
print("=" * 80)

for x in examples["end_inside_token"][:5]:
    print(x)

print("\n" + "=" * 80)
print("EXAMPLES — BOTH INSIDE")
print("=" * 80)

for x in examples["both_inside"][:5]:
    print(x)

print("\n" + "=" * 80)
print("ALIGNMENT CHECK COMPLETE")
print("=" * 80)

CHARACTER ↔ TOKEN ALIGNMENT ANALYSIS
Entities analyzed       : 8,393

exact_boundary           :    307 (  3.66%)
start_inside_token       :  7,930 ( 94.48%)
end_inside_token         :     10 (  0.12%)
both_inside              :    146 (  1.74%)
whitespace_boundary      :  7,929 ( 94.47%)
no_covering_tokens       :      0 (  0.00%)

EXAMPLES — START INSIDE TOKEN
('115651329', [137, 153, 'RESP', 'STATE OF HARYANA'], 26, 28, (136, 142), (145, 153), ' ')
('115651329', [252, 261, 'DATE', '19.8.2011'], 54, 58, (251, 254), (257, 261), ' ')
('115651329', [276, 308, 'COURT', 'High Court of Punjab and Haryana'], 62, 67, (275, 280), (300, 308), ' ')
('115651329', [359, 369, 'COURT', 'High Court'], 80, 81, (358, 363), (363, 369), ' ')
('115651329', [575, 585, 'DATE', '18.11.1994'], 120, 124, (574, 577), (581, 585), ' ')

EXAMPLES — END INSIDE TOKEN
('1646640', [47845, 47884, 'PREC', 'Bates v. Lord Hailsham of St. Marylebon'], 9951, 9965, (47845, 47846), (47882, 47885), 'e')
('1992267', [6347, 635

In [19]:
# ============================================================
# STEP 2.8 — GOLD CHARACTER SPAN → TOKEN BIO ALIGNMENT
# GPU: OFF
# ============================================================

from tokenizers import Tokenizer

TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json"
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

def normalize_text_span(text, start, end):
    """
    Return the exact non-whitespace character boundaries
    contained in [start, end).
    """
    while start < end and text[start].isspace():
        start += 1

    while end > start and text[end - 1].isspace():
        end -= 1

    return start, end


def char_span_to_token_span(text, encoding, gold_start, gold_end):
    """
    Map a character-level gold span to the tokenizer tokens
    that overlap the actual entity characters.
    """

    gold_start, gold_end = normalize_text_span(
        text, gold_start, gold_end
    )

    overlapping = []

    for i, (tok_start, tok_end) in enumerate(encoding.offsets):

        # Ignore empty offsets
        if tok_start == tok_end:
            continue

        # Character intersection
        if tok_end > gold_start and tok_start < gold_end:

            # Calculate actual overlap
            overlap_start = max(tok_start, gold_start)
            overlap_end = min(tok_end, gold_end)

            # Only count non-whitespace overlap
            if text[overlap_start:overlap_end].strip():
                overlapping.append(i)

    if not overlapping:
        return None

    return overlapping[0], overlapping[-1]


# ------------------------------------------------------------
# Validate every gold entity
# ------------------------------------------------------------

total = 0
mapped = 0
failed = []

alignment_examples = []

for doc_id, text_obj in train_text.items():

    text = text_obj if isinstance(text_obj, str) else str(text_obj)

    encoding = tokenizer.encode(text)

    for ann in train_gold[doc_id]:

        start, end, label, entity_text = ann

        total += 1

        result = char_span_to_token_span(
            text,
            encoding,
            start,
            end
        )

        if result is None:
            failed.append((doc_id, ann))
            continue

        mapped += 1

        first_token, last_token = result

        if len(alignment_examples) < 10:
            alignment_examples.append({
                "doc_id": doc_id,
                "gold": ann,
                "token_span": (first_token, last_token),
                "tokens": encoding.tokens[first_token:last_token + 1],
                "token_offsets":
                    encoding.offsets[first_token:last_token + 1]
            })


print("=" * 80)
print("GOLD → TOKEN ALIGNMENT")
print("=" * 80)

print(f"Total entities : {total:,}")
print(f"Mapped         : {mapped:,}")
print(f"Failed         : {len(failed):,}")
print(f"Coverage       : {100 * mapped / total:.4f}%")

print("\n" + "=" * 80)
print("EXAMPLES")
print("=" * 80)

for ex in alignment_examples:
    print("\nDocument:", ex["doc_id"])
    print("Gold:", ex["gold"])
    print("Token span:", ex["token_span"])
    print("Tokens:", ex["tokens"])
    print("Offsets:", ex["token_offsets"])

if failed:
    print("\nFIRST FAILURES:")
    for x in failed[:10]:
        print(x)

GOLD → TOKEN ALIGNMENT
Total entities : 8,393
Mapped         : 8,393
Failed         : 0
Coverage       : 100.0000%

EXAMPLES

Document: 115651329
Gold: [137, 153, 'RESP', 'STATE OF HARYANA']
Token span: (26, 28)
Tokens: ['ĠSTATE', 'ĠOF', 'ĠHARYANA']
Offsets: [(136, 142), (142, 145), (145, 153)]

Document: 115651329
Gold: [252, 261, 'DATE', '19.8.2011']
Token span: (54, 58)
Tokens: ['Ġ19', '.', '8', '.', '2011']
Offsets: [(251, 254), (254, 255), (255, 256), (256, 257), (257, 261)]

Document: 115651329
Gold: [276, 308, 'COURT', 'High Court of Punjab and Haryana']
Token span: (62, 67)
Tokens: ['ĠHigh', 'ĠCourt', 'Ġof', 'ĠPunjab', 'Ġand', 'ĠHaryana']
Offsets: [(275, 280), (280, 286), (286, 289), (289, 296), (296, 300), (300, 308)]

Document: 115651329
Gold: [359, 369, 'COURT', 'High Court']
Token span: (80, 81)
Tokens: ['ĠHigh', 'ĠCourt']
Offsets: [(358, 363), (363, 369)]

Document: 115651329
Gold: [575, 585, 'DATE', '18.11.1994']
Token span: (120, 124)
Tokens: ['Ġ18', '.', '11', '.', '199

In [20]:
# ============================================================
# STEP 2.9 — TOKEN-BASED OVERLAPPING CHUNK ANALYSIS
# GPU: OFF
# ============================================================

from tokenizers import Tokenizer
from collections import Counter, defaultdict

TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json"
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

MAX_TOKENS = 1536
OVERLAP = 384
STRIDE = MAX_TOKENS - OVERLAP

print("=" * 80)
print("TOKEN CHUNKING ANALYSIS")
print("=" * 80)

total_chunks = 0
total_entities = 0
fully_contained = 0
multi_chunk_entities = 0
uncontained_entities = 0

chunk_counts = []
entity_chunk_counts = Counter()

problem_examples = []

for doc_id, text_obj in train_text.items():

    text = text_obj if isinstance(text_obj, str) else str(text_obj)

    enc = tokenizer.encode(text)
    offsets = enc.offsets

    n_tokens = len(enc.ids)

    # Build chunks
    chunks = []

    start_token = 0

    while start_token < n_tokens:

        end_token = min(
            start_token + MAX_TOKENS,
            n_tokens
        )

        chunks.append((start_token, end_token))

        if end_token >= n_tokens:
            break

        start_token += STRIDE

    total_chunks += len(chunks)
    chunk_counts.append(len(chunks))

    # Check every gold entity
    for ann in train_gold[doc_id]:

        gold_start, gold_end, label, entity_text = ann

        total_entities += 1

        containing_chunks = []

        for chunk_id, (c_start, c_end) in enumerate(chunks):

            chunk_char_start = offsets[c_start][0]
            chunk_char_end = offsets[c_end - 1][1]

            # Entity must be completely inside the chunk's
            # character coverage.
            if (
                gold_start >= chunk_char_start
                and gold_end <= chunk_char_end
            ):
                containing_chunks.append(chunk_id)

        count = len(containing_chunks)

        entity_chunk_counts[count] += 1

        if count >= 1:
            fully_contained += 1

        if count > 1:
            multi_chunk_entities += 1

        if count == 0:
            uncontained_entities += 1

            if len(problem_examples) < 10:
                problem_examples.append(
                    (doc_id, ann, n_tokens, chunks[:5])
                )

print(f"Documents                    : {len(train_text)}")
print(f"Total tokenizer chunks       : {total_chunks:,}")
print(f"Average chunks/document      : {sum(chunk_counts)/len(chunk_counts):.2f}")
print(f"Maximum chunks/document      : {max(chunk_counts)}")

print("\n" + "-" * 80)
print("ENTITY COVERAGE")
print("-" * 80)

print(f"Total entities                : {total_entities:,}")
print(f"Fully contained in ≥1 chunk  : {fully_contained:,}")
print(f"Contained coverage            : "
      f"{100 * fully_contained / total_entities:.3f}%")
print(f"Contained in multiple chunks  : {multi_chunk_entities:,}")
print(f"Not fully contained           : {uncontained_entities:,}")

print("\n" + "-" * 80)
print("CHUNK COVERAGE DISTRIBUTION")
print("-" * 80)

for k in sorted(entity_chunk_counts):
    count = entity_chunk_counts[k]
    pct = 100 * count / total_entities

    print(
        f"{k} containing chunk(s): "
        f"{count:,} ({pct:.2f}%)"
    )

if problem_examples:
    print("\n" + "=" * 80)
    print("PROBLEM EXAMPLES")
    print("=" * 80)

    for x in problem_examples:
        print(x)

print("\nChunking analysis complete.")

TOKEN CHUNKING ANALYSIS
Documents                    : 70
Total tokenizer chunks       : 406
Average chunks/document      : 5.80
Maximum chunks/document      : 21

--------------------------------------------------------------------------------
ENTITY COVERAGE
--------------------------------------------------------------------------------
Total entities                : 8,393
Fully contained in ≥1 chunk  : 8,393
Contained coverage            : 100.000%
Contained in multiple chunks  : 2,069
Not fully contained           : 0

--------------------------------------------------------------------------------
CHUNK COVERAGE DISTRIBUTION
--------------------------------------------------------------------------------
1 containing chunk(s): 6,324 (75.35%)
2 containing chunk(s): 2,069 (24.65%)

Chunking analysis complete.


In [21]:
# ============================================================
# STEP 2.10 — CHECKPOINT METADATA
# GPU: OFF
# ============================================================

import torch
import os

CHECKPOINTS = {
    "A1": "/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/a3check/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt",
}

for name, path in CHECKPOINTS.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    size_mb = os.path.getsize(path) / (1024**2)
    print(f"File size: {size_mb:.2f} MB")

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Checkpoint type:", type(ckpt).__name__)

    if isinstance(ckpt, dict):

        print("Keys:", list(ckpt.keys()))

        for key in [
            "step",
            "tokens_processed",
            "train_loss",
            "val_loss",
            "reason",
            "saved_at"
        ]:
            if key in ckpt:
                print(f"{key}: {ckpt[key]}")

    del ckpt


A1
File size: 593.71 MB
Checkpoint type: dict
Keys: ['step', 'epoch', 'train_loss', 'val_loss', 'model_state_dict', 'optimizer_state_dict']
step: 500
train_loss: 5.366012096405029
val_loss: None

A2
File size: 196.55 MB
Checkpoint type: dict
Keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
step: 9000
tokens_processed: 73728000
train_loss: 6.801188290119171
reason: regular_checkpoint
saved_at: 2026-10-05T19:17:55.016904

A3
File size: 197.90 MB
Checkpoint type: dict
Keys: ['model', 'step', 'tokens_seen', 'architecture', 'parameters', 'context_length', 'hidden_size', 'num_layers', 'num_heads', 'ffn_size', 'prefix_ratio', 'dropout', 'learning_rate', 'weight_decay']
step: 1000

A4
File size: 367.79 MB
Checkpoint type: dict
Keys: ['model_state_dict', 'step', 'tokens_processed', 'train_loss', 'reason', 'saved_at']
step: 22000
tokens_processed: 180096704
train_loss: 0.1944424146786332
reason: regular_checkpoint
saved_at: 2026-10-05T15:18:59.981007


In [22]:
# ============================================================
# STEP 2.11 — INSPECT CHECKPOINT MODEL STRUCTURES
# GPU: OFF
# ============================================================

import torch

for name, path in CHECKPOINTS.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    state_key = None

    for candidate in [
        "model_state_dict",
        "model"
    ]:
        if candidate in ckpt and isinstance(ckpt[candidate], dict):
            state_key = candidate
            break

    if state_key is None:
        print("No obvious state-dict key found.")
        del ckpt
        continue

    state = ckpt[state_key]

    print("State key:", state_key)
    print("Number of tensors:", len(state))

    print("\nFirst 30 parameter names:")
    for i, (key, value) in enumerate(state.items()):
        if i >= 30:
            break

        shape = tuple(value.shape) if hasattr(value, "shape") else "?"
        print(f"{key:<70} {shape}")

    del ckpt


A1
State key: model_state_dict
Number of tensors: 101

First 30 parameter names:
token_embedding.weight                                                 (50000, 512)
position_embedding.weight                                              (2048, 512)
blocks.0.norm1.weight                                                  (512,)
blocks.0.norm1.bias                                                    (512,)
blocks.0.attn.in_proj_weight                                           (1536, 512)
blocks.0.attn.in_proj_bias                                             (1536,)
blocks.0.attn.out_proj.weight                                          (512, 512)
blocks.0.attn.out_proj.bias                                            (512,)
blocks.0.norm2.weight                                                  (512,)
blocks.0.norm2.bias                                                    (512,)
blocks.0.ffn.0.weight                                                  (2048, 512)
blocks.0.ffn.0.bias               

In [23]:
# ============================================================
# PHASE 2 — STEP 2
# FORWARD-PASS / CHECKPOINT COMPATIBILITY TEST
# CPU ONLY
# ============================================================

import torch
import os

CHECKPOINTS = {
    "A1": "/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/a3check/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt",
}

for name, path in CHECKPOINTS.items():
    print("=" * 70)
    print(name)
    print("Exists:", os.path.exists(path))
    print("Size:", round(os.path.getsize(path) / 1024**2, 2), "MB")

    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    if name == "A3":
        state = ckpt["model"]
    else:
        state = ckpt["model_state_dict"]

    print("Tensors:", len(state))
    
    # Check important dimensions
    emb = state["token_embedding.weight"]
    print("Vocab:", emb.shape[0])
    print("Hidden:", emb.shape[1])

    if name == "A1":
        print("Position:", state["position_embedding.weight"].shape)

    elif name == "A2":
        print("Position:", state["position_embedding.weight"].shape)
        print("Q:", state["layers.0.attention.q_proj.weight"].shape)
        print("K:", state["layers.0.attention.k_proj.weight"].shape)
        print("V:", state["layers.0.attention.v_proj.weight"].shape)

    elif name == "A3":
        print("Position:", state["position_embedding.weight"].shape)

    elif name == "A4":
        print("Encoder position:", state["encoder_pos.weight"].shape)
        print("Decoder position:", state["decoder_pos.weight"].shape)

print("=" * 70)
print("CHECK COMPLETE")

A1
Exists: True
Size: 593.71 MB
Tensors: 101
Vocab: 50000
Hidden: 512
Position: torch.Size([2048, 512])
A2
Exists: True
Size: 196.55 MB
Tensors: 113
Vocab: 50000
Hidden: 512
Position: torch.Size([2048, 512])
Q: torch.Size([512, 512])
K: torch.Size([128, 512])
V: torch.Size([128, 512])
A3
Exists: True
Size: 197.9 MB
Tensors: 100
Vocab: 50000
Hidden: 512
Position: torch.Size([2048, 512])
A4
Exists: True
Size: 367.79 MB
Tensors: 260
Vocab: 50000
Hidden: 512
Encoder position: torch.Size([1024, 512])
Decoder position: torch.Size([1024, 512])
CHECK COMPLETE


In [24]:
import torch
import os
import time

print("=" * 60)
print("GPU ENVIRONMENT")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        gpu = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}")
        print("Name:", gpu.name)
        print("VRAM:", round(gpu.total_memory / 1024**3, 2), "GB")
        print("Compute capability:", f"{gpu.major}.{gpu.minor}")

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    print("\nTF32: enabled")
else:
    print("\nWARNING: GPU is OFF")

print("=" * 60)

GPU ENVIRONMENT
PyTorch: 2.11.0+cu128
CUDA available: True
GPU count: 2

GPU 0
Name: Tesla T4
VRAM: 14.56 GB
Compute capability: 7.5

GPU 1
Name: Tesla T4
VRAM: 14.56 GB
Compute capability: 7.5

TF32: enabled


In [25]:
# ============================================================
# PHASE 2 — GPU SMOKE TEST
# A4 CHECKPOINT
# ============================================================

import torch
import time
import os

DEVICE = torch.device("cuda:0")

A4_PATH = "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"

print("=" * 70)
print("A4 GPU SMOKE TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load checkpoint
# ------------------------------------------------------------

t0 = time.time()

ckpt = torch.load(
    A4_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model_state_dict"]

print("Checkpoint loaded")
print("Load time:", round(time.time() - t0, 2), "sec")
print("State tensors:", len(state))

# ------------------------------------------------------------
# 2. Inspect parameter count
# ------------------------------------------------------------

params = sum(v.numel() for v in state.values())
print("Parameters:", f"{params:,}")
print("Approx FP32 memory:", round(params * 4 / 1024**3, 2), "GB")
print("Approx FP16 memory:", round(params * 2 / 1024**3, 2), "GB")

# ------------------------------------------------------------
# 3. GPU memory before
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(DEVICE)

print("\nGPU before model:")
print(
    "Allocated:",
    round(torch.cuda.memory_allocated(DEVICE) / 1024**3, 3),
    "GB"
)

# ------------------------------------------------------------
# 4. Move raw checkpoint tensors to GPU
# ------------------------------------------------------------

t0 = time.time()

gpu_state = {
    k: v.to(DEVICE, non_blocking=True)
    for k, v in state.items()
}

torch.cuda.synchronize()

print("\nState moved to GPU")
print("Transfer time:", round(time.time() - t0, 2), "sec")

print(
    "Allocated:",
    round(torch.cuda.memory_allocated(DEVICE) / 1024**3, 3),
    "GB"
)

print(
    "Reserved:",
    round(torch.cuda.memory_reserved(DEVICE) / 1024**3, 3),
    "GB"
)

# ------------------------------------------------------------
# 5. Cleanup
# ------------------------------------------------------------

del gpu_state
del state
del ckpt

torch.cuda.empty_cache()

print("\nGPU cleanup complete")

print(
    "Final allocated:",
    round(torch.cuda.memory_allocated(DEVICE) / 1024**3, 3),
    "GB"
)

print("=" * 70)
print("SMOKE TEST COMPLETE")
print("=" * 70)

A4 GPU SMOKE TEST
Checkpoint loaded
Load time: 0.22 sec
State tensors: 260
Parameters: 96,389,120
Approx FP32 memory: 0.36 GB
Approx FP16 memory: 0.18 GB

GPU before model:
Allocated: 0.0 GB

State moved to GPU
Transfer time: 0.43 sec
Allocated: 0.36 GB
Reserved: 0.361 GB

GPU cleanup complete
Final allocated: 0.0 GB
SMOKE TEST COMPLETE


In [26]:
import os

print("KAGGLE INPUT:")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".py"):
            print(os.path.join(root, f))

print("\nKAGGLE WORKING:")
for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        if f.endswith(".py"):
            print(os.path.join(root, f))

KAGGLE INPUT:

KAGGLE WORKING:


In [27]:
# ============================================================
# PHASE 2 — A1 CHECKPOINT RECONSTRUCTION
# CPU ONLY
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

A1_PATH = "/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt"

ckpt = torch.load(
    A1_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model_state_dict"]

print("=" * 70)
print("A1 CHECKPOINT")
print("=" * 70)

# ------------------------------------------------------------
# Infer architecture directly from checkpoint
# ------------------------------------------------------------

vocab_size = state["token_embedding.weight"].shape[0]
hidden_size = state["token_embedding.weight"].shape[1]
context_length = state["position_embedding.weight"].shape[0]

layer_ids = sorted({
    int(k.split(".")[1])
    for k in state
    if k.startswith("blocks.") and k.split(".")[1].isdigit()
})

num_layers = len(layer_ids)

ffn_hidden = state["blocks.0.ffn.0.weight"].shape[0]

# Multi-head attention:
# in_proj_weight = [3*hidden, hidden]
qkv_dim = state["blocks.0.attn.in_proj_weight"].shape[0]
num_heads = 8
head_dim = hidden_size // num_heads

print("Vocab size      :", vocab_size)
print("Hidden size     :", hidden_size)
print("Context length  :", context_length)
print("Layers          :", num_layers)
print("FFN hidden      :", ffn_hidden)
print("Attention heads :", num_heads)
print("Head dimension  :", head_dim)

# ------------------------------------------------------------
# Architecture
# ------------------------------------------------------------

class A1Block(nn.Module):
    def __init__(self, hidden_size, num_heads, ffn_hidden):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(hidden_size)

        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, ffn_hidden),
            nn.GELU(),
            nn.Linear(ffn_hidden, hidden_size)
        )

    def forward(self, x, causal_mask=None):
        h = self.norm1(x)

        attn_out, _ = self.attn(
            h, h, h,
            attn_mask=causal_mask,
            need_weights=False
        )

        x = x + attn_out
        x = x + self.ffn(self.norm2(x))

        return x


class A1Model(nn.Module):
    def __init__(
        self,
        vocab_size,
        hidden_size,
        context_length,
        num_layers,
        num_heads,
        ffn_hidden
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            context_length,
            hidden_size
        )

        self.blocks = nn.ModuleList([
            A1Block(
                hidden_size,
                num_heads,
                ffn_hidden
            )
            for _ in range(num_layers)
        ])

        self.lm_head = nn.Linear(
            hidden_size,
            vocab_size,
            bias=False
        )

    def forward(self, input_ids):
        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)[None, :, :]
        )

        # Causal mask
        mask = torch.triu(
            torch.ones(
                T, T,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        for block in self.blocks:
            x = block(x, mask)

        logits = self.lm_head(x)

        return logits


# ------------------------------------------------------------
# Build
# ------------------------------------------------------------

model = A1Model(
    vocab_size=vocab_size,
    hidden_size=hidden_size,
    context_length=context_length,
    num_layers=num_layers,
    num_heads=num_heads,
    ffn_hidden=ffn_hidden
)

print("\nModel parameters:",
      f"{sum(p.numel() for p in model.parameters()):,}")

# ------------------------------------------------------------
# Load matching state
# ------------------------------------------------------------

try:
    missing, unexpected = model.load_state_dict(
        state,
        strict=False
    )

    print("\nMissing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))

    if missing:
        print("\nMissing:")
        for x in missing:
            print(" ", x)

    if unexpected:
        print("\nUnexpected:")
        for x in unexpected:
            print(" ", x)

except Exception as e:
    print("\nLOAD ERROR:")
    print(type(e).__name__, e)

# ------------------------------------------------------------
# Forward test
# ------------------------------------------------------------

print("\nForward test...")

model.eval()

test_ids = torch.randint(
    0,
    vocab_size,
    (1, 128)
)

with torch.no_grad():
    logits = model(test_ids)

print("Input :", tuple(test_ids.shape))
print("Output:", tuple(logits.shape))

print("=" * 70)
print("A1 RECONSTRUCTION TEST COMPLETE")
print("=" * 70)

A1 CHECKPOINT
Vocab size      : 50000
Hidden size     : 512
Context length  : 2048
Layers          : 8
FFN hidden      : 2048
Attention heads : 8
Head dimension  : 64

Model parameters: 77,467,648

Missing keys: 0
Unexpected keys: 2

Unexpected:
  final_norm.weight
  final_norm.bias

Forward test...
Input : (1, 128)
Output: (1, 128, 50000)
A1 RECONSTRUCTION TEST COMPLETE


In [28]:
class A1Model(nn.Module):
    def __init__(
        self,
        vocab_size,
        hidden_size,
        context_length,
        num_layers,
        num_heads,
        ffn_hidden
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            context_length,
            hidden_size
        )

        self.blocks = nn.ModuleList([
            A1Block(
                hidden_size,
                num_heads,
                ffn_hidden
            )
            for _ in range(num_layers)
        ])

        # Checkpoint contains this layer
        self.final_norm = nn.LayerNorm(hidden_size)

        self.lm_head = nn.Linear(
            hidden_size,
            vocab_size,
            bias=False
        )

    def forward(self, input_ids):
        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)[None, :, :]
        )

        mask = torch.triu(
            torch.ones(
                T, T,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        for block in self.blocks:
            x = block(x, mask)

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits

In [29]:
print("A1 correction cell started")

print("model exists:", "model" in globals())
print("state exists:", "state" in globals())
print("A1Model exists:", "A1Model" in globals())

if "model" in globals() and "state" in globals():
    missing, unexpected = model.load_state_dict(
        state,
        strict=False
    )

    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))
    print("Missing:", missing)
    print("Unexpected:", unexpected)

    model.eval()

    test_ids = torch.randint(
        0,
        vocab_size,
        (1, 128)
    )

    with torch.no_grad():
        logits = model(test_ids)

    print("Input :", tuple(test_ids.shape))
    print("Output:", tuple(logits.shape))

print("A1 correction cell finished")

A1 correction cell started
model exists: True
state exists: True
A1Model exists: True
Missing keys: 0
Unexpected keys: 2
Missing: []
Unexpected: ['final_norm.weight', 'final_norm.bias']
Input : (1, 128)
Output: (1, 128, 50000)
A1 correction cell finished


In [30]:
# ============================================================
# A1 — FINAL EXACT CHECKPOINT VALIDATION
# ============================================================

import torch
import torch.nn as nn

A1_PATH = "/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt"

ckpt = torch.load(
    A1_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model_state_dict"]

# Infer directly from checkpoint
vocab_size = state["token_embedding.weight"].shape[0]
hidden_size = state["token_embedding.weight"].shape[1]
context_length = state["position_embedding.weight"].shape[0]

num_layers = len([
    k for k in state
    if k.startswith("blocks.") and k.endswith(".norm1.weight")
])

ffn_hidden = state["blocks.0.ffn.0.weight"].shape[0]

# ------------------------------------------------------------
# A1 block
# ------------------------------------------------------------

class A1Block(nn.Module):

    def __init__(self, hidden_size, num_heads, ffn_hidden):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(hidden_size)

        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, ffn_hidden),
            nn.GELU(),
            nn.Linear(ffn_hidden, hidden_size)
        )

    def forward(self, x, causal_mask):

        h = self.norm1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=causal_mask,
            need_weights=False
        )

        x = x + attn_out
        x = x + self.ffn(self.norm2(x))

        return x


# ------------------------------------------------------------
# A1 model
# ------------------------------------------------------------

class A1Model(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_size,
        context_length,
        num_layers,
        num_heads,
        ffn_hidden
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            context_length,
            hidden_size
        )

        self.blocks = nn.ModuleList([
            A1Block(
                hidden_size,
                num_heads,
                ffn_hidden
            )
            for _ in range(num_layers)
        ])

        # IMPORTANT: present in checkpoint
        self.final_norm = nn.LayerNorm(hidden_size)

        self.lm_head = nn.Linear(
            hidden_size,
            vocab_size,
            bias=False
        )

    def forward(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        x = x.unsqueeze(0) if x.dim() == 2 else x

        mask = torch.triu(
            torch.ones(
                T,
                T,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        for block in self.blocks:
            x = block(x, mask)

        x = self.final_norm(x)

        return self.lm_head(x)


# ------------------------------------------------------------
# Build fresh model
# ------------------------------------------------------------

model = A1Model(
    vocab_size=vocab_size,
    hidden_size=hidden_size,
    context_length=context_length,
    num_layers=num_layers,
    num_heads=8,
    ffn_hidden=ffn_hidden
)

print("=" * 65)
print("A1 FINAL VALIDATION")
print("=" * 65)

print("Parameters:",
      f"{sum(p.numel() for p in model.parameters()):,}")

# ------------------------------------------------------------
# STRICT LOAD
# ------------------------------------------------------------

try:

    model.load_state_dict(
        state,
        strict=True
    )

    print("STRICT LOAD: ✅ SUCCESS")

except Exception as e:

    print("STRICT LOAD: ❌ FAILED")
    print(e)

# ------------------------------------------------------------
# Forward test
# ------------------------------------------------------------

model.eval()

test_ids = torch.randint(
    0,
    vocab_size,
    (1, 128)
)

with torch.no_grad():

    logits = model(test_ids)

print("Input shape :", tuple(test_ids.shape))
print("Output shape:", tuple(logits.shape))

print("=" * 65)

A1 FINAL VALIDATION
Parameters: 77,468,672
STRICT LOAD: ✅ SUCCESS
Input shape : (1, 128)
Output shape: (1, 128, 50000)


In [31]:
# ============================================================
# A2 — CHECKPOINT ARCHITECTURE INSPECTION
# ============================================================

import torch

A2_PATH = "/kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt"

ckpt = torch.load(
    A2_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model_state_dict"]

print("=" * 65)
print("A2 CHECKPOINT")
print("=" * 65)

# Basic dimensions
vocab_size = state["token_embedding.weight"].shape[0]
hidden_size = state["token_embedding.weight"].shape[1]
context_length = state["position_embedding.weight"].shape[0]

# Number of layers
layer_ids = sorted({
    int(k.split(".")[1])
    for k in state
    if k.startswith("layers.") and k.split(".")[1].isdigit()
})

num_layers = len(layer_ids)

# Attention dimensions
q_shape = state["layers.0.attention.q_proj.weight"].shape
k_shape = state["layers.0.attention.k_proj.weight"].shape
v_shape = state["layers.0.attention.v_proj.weight"].shape

# FFN
ffn_hidden = state["layers.0.ffn.fc1.weight"].shape[0]

# Output bias
has_output_bias = "output_bias" in state

print("Vocab size       :", vocab_size)
print("Hidden size      :", hidden_size)
print("Context length   :", context_length)
print("Layers           :", num_layers)
print("Q projection     :", q_shape)
print("K projection     :", k_shape)
print("V projection     :", v_shape)
print("FFN hidden       :", ffn_hidden)
print("Output bias      :", has_output_bias)

# Inspect remaining keys
print("\nAll top-level parameter groups:")

groups = {}

for k in state:
    prefix = k.split(".")[0]
    groups[prefix] = groups.get(prefix, 0) + 1

for k, v in groups.items():
    print(f"  {k:20s} {v}")

print("=" * 65)

A2 CHECKPOINT
Vocab size       : 50000
Hidden size      : 512
Context length   : 2048
Layers           : 9
Q projection     : torch.Size([512, 512])
K projection     : torch.Size([128, 512])
V projection     : torch.Size([128, 512])
FFN hidden       : 2048
Output bias      : True

All top-level parameter groups:
  output_bias          1
  token_embedding      1
  position_embedding   1
  layers               108
  final_norm           2


In [32]:
# ============================================================
# A3 — CHECKPOINT ARCHITECTURE INSPECTION
# CPU ONLY
# ============================================================

import torch

A3_PATH = "/kaggle/input/datasets/aq1de3/a3check/step_0001000.pt"

ckpt = torch.load(
    A3_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model"]

print("=" * 65)
print("A3 CHECKPOINT")
print("=" * 65)

vocab_size = state["token_embedding.weight"].shape[0]
hidden_size = state["token_embedding.weight"].shape[1]
context_length = state["position_embedding.weight"].shape[0]

layer_ids = sorted({
    int(k.split(".")[1])
    for k in state
    if k.startswith("layers.") and k.split(".")[1].isdigit()
})

num_layers = len(layer_ids)

ffn_hidden = state["layers.0.ffn.0.weight"].shape[0]

print("Vocab size      :", vocab_size)
print("Hidden size     :", hidden_size)
print("Context length  :", context_length)
print("Layers          :", num_layers)
print("FFN hidden      :", ffn_hidden)

print("\nAttention:")
print("QKV projection  :",
      state["layers.0.attn.in_proj_weight"].shape)
print("Output projection:",
      state["layers.0.attn.out_proj.weight"].shape)

print("\nPrefix-LM metadata:")
for key in [
    "step",
    "tokens_seen",
    "architecture",
    "parameters",
    "context_length",
    "hidden_size",
    "num_layers",
    "num_heads",
    "ffn_size",
    "prefix_ratio",
    "dropout",
    "learning_rate",
    "weight_decay"
]:
    if key in ckpt:
        print(f"{key:18s}: {ckpt[key]}")

print("\nParameter groups:")

groups = {}

for k in state:
    prefix = k.split(".")[0]
    groups[prefix] = groups.get(prefix, 0) + 1

for k, v in groups.items():
    print(f"  {k:20s} {v}")

print("=" * 65)

A3 CHECKPOINT
Vocab size      : 50000
Hidden size     : 512
Context length  : 2048
Layers          : 8
FFN hidden      : 2048

Attention:
QKV projection  : torch.Size([1536, 512])
Output projection: torch.Size([512, 512])

Prefix-LM metadata:
step              : 1000
tokens_seen       : 512000
architecture      : A3_prefix_lm_hybrid
parameters        : 51868672
context_length    : 2048
hidden_size       : 512
num_layers        : 8
num_heads         : 8
ffn_size          : 2048
prefix_ratio      : 0.5
dropout           : 0.1
learning_rate     : 0.0001
weight_decay      : 0.1

Parameter groups:
  token_embedding      1
  position_embedding   1
  layers               96
  ln_f                 2


In [33]:
# ============================================================
# A4 — CHECKPOINT ARCHITECTURE INSPECTION
# CPU ONLY
# ============================================================

import torch

A4_PATH = "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"

ckpt = torch.load(
    A4_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model_state_dict"]

print("=" * 65)
print("A4 CHECKPOINT")
print("=" * 65)

vocab_size = state["token_embedding.weight"].shape[0]
hidden_size = state["token_embedding.weight"].shape[1]

encoder_context = state["encoder_pos.weight"].shape[0]
decoder_context = state["decoder_pos.weight"].shape[0]

encoder_layers = len({
    int(k.split(".")[1])
    for k in state
    if k.startswith("encoder_layers.")
    and k.split(".")[1].isdigit()
})

decoder_layers = len({
    int(k.split(".")[1])
    for k in state
    if k.startswith("decoder_layers.")
    and k.split(".")[1].isdigit()
})

ffn_hidden = state["encoder_layers.0.ffn.net.0.weight"].shape[0]

print("Vocab size       :", vocab_size)
print("Hidden size      :", hidden_size)
print("Encoder context  :", encoder_context)
print("Decoder context  :", decoder_context)
print("Encoder layers   :", encoder_layers)
print("Decoder layers   :", decoder_layers)
print("FFN hidden       :", ffn_hidden)

print("\nEncoder attention:")
print("Q:", state["encoder_layers.0.attn.q_proj.weight"].shape)
print("K:", state["encoder_layers.0.attn.k_proj.weight"].shape)
print("V:", state["encoder_layers.0.attn.v_proj.weight"].shape)

print("\nDecoder self-attention:")
print("Q:", state["decoder_layers.0.self_attn.q_proj.weight"].shape)
print("K:", state["decoder_layers.0.self_attn.k_proj.weight"].shape)
print("V:", state["decoder_layers.0.self_attn.v_proj.weight"].shape)

print("\nDecoder cross-attention:")
print("Q:", state["decoder_layers.0.cross_attn.q_proj.weight"].shape)
print("K:", state["decoder_layers.0.cross_attn.k_proj.weight"].shape)
print("V:", state["decoder_layers.0.cross_attn.v_proj.weight"].shape)

print("\nParameter groups:")

groups = {}

for k in state:
    prefix = k.split(".")[0]
    groups[prefix] = groups.get(prefix, 0) + 1

for k, v in groups.items():
    print(f"  {k:20s} {v}")

print("=" * 65)

A4 CHECKPOINT
Vocab size       : 50000
Hidden size      : 512
Encoder context  : 1024
Decoder context  : 1024
Encoder layers   : 6
Decoder layers   : 6
FFN hidden       : 2048

Encoder attention:
Q: torch.Size([512, 512])
K: torch.Size([512, 512])
V: torch.Size([512, 512])

Decoder self-attention:
Q: torch.Size([512, 512])
K: torch.Size([512, 512])
V: torch.Size([512, 512])

Decoder cross-attention:
Q: torch.Size([512, 512])
K: torch.Size([512, 512])
V: torch.Size([512, 512])

Parameter groups:
  token_embedding      1
  encoder_pos          1
  decoder_pos          1
  encoder_layers       96
  encoder_norm         2
  decoder_layers       156
  decoder_norm         2
  lm_head              1


In [34]:
# ============================================================
# PHASE 2 — A2 EXACT RECONSTRUCTION
# GQA + STRICT CHECKPOINT LOAD
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

A2_PATH = "/kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt"

ckpt = torch.load(
    A2_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model_state_dict"]

# ------------------------------------------------------------
# Infer architecture
# ------------------------------------------------------------

vocab_size = state["token_embedding.weight"].shape[0]
hidden_size = state["token_embedding.weight"].shape[1]
context_length = state["position_embedding.weight"].shape[0]

num_layers = len({
    int(k.split(".")[1])
    for k in state
    if k.startswith("layers.")
    and k.split(".")[1].isdigit()
})

ffn_hidden = state["layers.0.ffn.fc1.weight"].shape[0]

q_dim = state["layers.0.attention.q_proj.weight"].shape[0]
kv_dim = state["layers.0.attention.k_proj.weight"].shape[0]

num_heads = 8
num_kv_heads = 2

head_dim = hidden_size // num_heads

print("=" * 70)
print("A2 RECONSTRUCTION")
print("=" * 70)

print("Vocab       :", vocab_size)
print("Hidden      :", hidden_size)
print("Context     :", context_length)
print("Layers      :", num_layers)
print("FFN         :", ffn_hidden)
print("Q dimension :", q_dim)
print("KV dimension:", kv_dim)
print("Q heads     :", num_heads)
print("KV heads    :", num_kv_heads)
print("Head dim    :", head_dim)


# ------------------------------------------------------------
# GQA Attention
# ------------------------------------------------------------

class GQAAttention(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_heads,
        num_kv_heads
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads

        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(
            hidden_size,
            hidden_size,
            bias=False
        )

        self.k_proj = nn.Linear(
            hidden_size,
            num_kv_heads * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            hidden_size,
            num_kv_heads * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            hidden_size,
            hidden_size,
            bias=False
        )

    def forward(self, x, causal_mask=None):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        # Repeat KV heads for GQA
        repeat_factor = self.num_heads // self.num_kv_heads

        k = k.repeat_interleave(
            repeat_factor,
            dim=1
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1
        )

        scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        ) / (self.head_dim ** 0.5)

        if causal_mask is not None:
            scores = scores.masked_fill(
                causal_mask,
                float("-inf")
            )

        attn = torch.softmax(
            scores,
            dim=-1
        )

        out = torch.matmul(
            attn,
            v
        )

        out = out.transpose(
            1, 2
        ).contiguous().view(
            B, T, C
        )

        return self.out_proj(out)


# ------------------------------------------------------------
# A2 Layer
# ------------------------------------------------------------

class A2Layer(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_heads,
        num_kv_heads,
        ffn_hidden
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            hidden_size
        )

        self.attention = GQAAttention(
            hidden_size,
            num_heads,
            num_kv_heads
        )

        self.norm2 = nn.LayerNorm(
            hidden_size
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                hidden_size,
                ffn_hidden
            ),
            nn.GELU(),
            nn.Linear(
                ffn_hidden,
                hidden_size
            )
        )

    def forward(
        self,
        x,
        causal_mask
    ):

        x = x + self.attention(
            self.norm1(x),
            causal_mask
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ------------------------------------------------------------
# A2 Model
# ------------------------------------------------------------

class A2Model(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_size,
        context_length,
        num_layers,
        num_heads,
        num_kv_heads,
        ffn_hidden
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            context_length,
            hidden_size
        )

        self.layers = nn.ModuleList([
            A2Layer(
                hidden_size,
                num_heads,
                num_kv_heads,
                ffn_hidden
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(
            hidden_size
        )

        # Checkpoint contains a separate output bias.
        self.lm_head = nn.Linear(
            hidden_size,
            vocab_size,
            bias=False
        )

        self.register_parameter(
            "output_bias",
            nn.Parameter(
                torch.zeros(vocab_size)
            )
        )

    def forward(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        causal_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        causal_mask = causal_mask.unsqueeze(
            0
        ).unsqueeze(
            0
        )

        for layer in self.layers:

            x = layer(
                x,
                causal_mask
            )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        logits = logits + self.output_bias

        return logits


# ------------------------------------------------------------
# Build
# ------------------------------------------------------------

model = A2Model(
    vocab_size=vocab_size,
    hidden_size=hidden_size,
    context_length=context_length,
    num_layers=num_layers,
    num_heads=num_heads,
    num_kv_heads=num_kv_heads,
    ffn_hidden=ffn_hidden
)

print("\nModel parameters:",
      f"{sum(p.numel() for p in model.parameters()):,}")


# ------------------------------------------------------------
# Strict checkpoint load
# ------------------------------------------------------------

print("\nLoading checkpoint...")

try:

    model.load_state_dict(
        state,
        strict=True
    )

    print("STRICT LOAD: ✅ SUCCESS")

except Exception as e:

    print("STRICT LOAD: ❌ FAILED")
    print(type(e).__name__)
    print(e)


# ------------------------------------------------------------
# Forward test
# ------------------------------------------------------------

print("\nForward test...")

model.eval()

test_ids = torch.randint(
    0,
    vocab_size,
    (1, 128)
)

with torch.no_grad():

    logits = model(
        test_ids
    )

print("Input :", tuple(test_ids.shape))
print("Output:", tuple(logits.shape))

print("=" * 70)
print("A2 VALIDATION COMPLETE")
print("=" * 70)

A2 RECONSTRUCTION
Vocab       : 50000
Hidden      : 512
Context     : 2048
Layers      : 9
FFN         : 2048
Q dimension : 512
KV dimension: 128
Q heads     : 8
KV heads    : 2
Head dim    : 64

Model parameters: 77,113,680

Loading checkpoint...
STRICT LOAD: ❌ FAILED
RuntimeError
Error(s) in loading state_dict for A2Model:
	Missing key(s) in state_dict: "layers.0.ffn.0.weight", "layers.0.ffn.0.bias", "layers.0.ffn.2.weight", "layers.0.ffn.2.bias", "layers.1.ffn.0.weight", "layers.1.ffn.0.bias", "layers.1.ffn.2.weight", "layers.1.ffn.2.bias", "layers.2.ffn.0.weight", "layers.2.ffn.0.bias", "layers.2.ffn.2.weight", "layers.2.ffn.2.bias", "layers.3.ffn.0.weight", "layers.3.ffn.0.bias", "layers.3.ffn.2.weight", "layers.3.ffn.2.bias", "layers.4.ffn.0.weight", "layers.4.ffn.0.bias", "layers.4.ffn.2.weight", "layers.4.ffn.2.bias", "layers.5.ffn.0.weight", "layers.5.ffn.0.bias", "layers.5.ffn.2.weight", "layers.5.ffn.2.bias", "layers.6.ffn.0.weight", "layers.6.ffn.0.bias", "layers.6.ffn.2.we

In [35]:
class A2Layer(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_heads,
        num_kv_heads,
        ffn_hidden
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)

        self.attention = GQAAttention(
            hidden_size,
            num_heads,
            num_kv_heads
        )

        self.norm2 = nn.LayerNorm(hidden_size)

        # Names MUST match checkpoint
        self.ffn = nn.ModuleDict({
            "fc1": nn.Linear(
                hidden_size,
                ffn_hidden
            ),
            "fc2": nn.Linear(
                ffn_hidden,
                hidden_size
            )
        })

    def forward(self, x, causal_mask):

        x = x + self.attention(
            self.norm1(x),
            causal_mask
        )

        x = x + self.ffn["fc2"](
            F.gelu(
                self.ffn["fc1"](
                    self.norm2(x)
                )
            )
        )

        return x


class A2Model(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_size,
        context_length,
        num_layers,
        num_heads,
        num_kv_heads,
        ffn_hidden
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            context_length,
            hidden_size
        )

        self.layers = nn.ModuleList([
            A2Layer(
                hidden_size,
                num_heads,
                num_kv_heads,
                ffn_hidden
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(
            hidden_size
        )

        # Checkpoint has a separate lm_head
        self.lm_head = nn.Linear(
            hidden_size,
            vocab_size,
            bias=False
        )

        # Checkpoint also has output_bias
        self.register_parameter(
            "output_bias",
            nn.Parameter(
                torch.zeros(vocab_size)
            )
        )

    def forward(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        causal_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

        for layer in self.layers:
            x = layer(x, causal_mask)

        x = self.final_norm(x)

        logits = self.lm_head(x)

        logits = logits + self.output_bias

        return logits

In [36]:
# ============================================================
# A2 — STRICT LOAD RETEST
# ============================================================

model = A2Model(
    vocab_size=vocab_size,
    hidden_size=hidden_size,
    context_length=context_length,
    num_layers=num_layers,
    num_heads=num_heads,
    num_kv_heads=num_kv_heads,
    ffn_hidden=ffn_hidden
)

print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

try:

    model.load_state_dict(
        state,
        strict=True
    )

    print("STRICT LOAD: ✅ SUCCESS")

except Exception as e:

    print("STRICT LOAD: ❌ FAILED")
    print(type(e).__name__)
    print(e)

model.eval()

test_ids = torch.randint(
    0,
    vocab_size,
    (1, 128)
)

with torch.no_grad():
    logits = model(test_ids)

print("Input :", tuple(test_ids.shape))
print("Output:", tuple(logits.shape))

Parameters: 77,113,680
STRICT LOAD: ❌ FAILED
RuntimeError
Error(s) in loading state_dict for A2Model:
	Missing key(s) in state_dict: "lm_head.weight". 
Input : (1, 128)
Output: (1, 128, 50000)


In [37]:
class A2Model(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_size,
        context_length,
        num_layers,
        num_heads,
        num_kv_heads,
        ffn_hidden
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            context_length,
            hidden_size
        )

        self.layers = nn.ModuleList([
            A2Layer(
                hidden_size,
                num_heads,
                num_kv_heads,
                ffn_hidden
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(
            hidden_size
        )

        # IMPORTANT:
        # Checkpoint has no lm_head.weight.
        # Tie output projection to token embedding.
        self.lm_head = self.token_embedding

        self.register_parameter(
            "output_bias",
            nn.Parameter(
                torch.zeros(vocab_size)
            )
        )

    def forward(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        causal_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

        for layer in self.layers:
            x = layer(x, causal_mask)

        x = self.final_norm(x)

        logits = F.linear(
            x,
            self.token_embedding.weight
        )

        logits = logits + self.output_bias

        return logits

In [38]:
print("A2 tie-weight test started")

print("torch:", torch.__version__)
print("A2Model exists:", "A2Model" in globals())
print("state exists:", "state" in globals())

if "A2Model" in globals() and "state" in globals():

    model = A2Model(
        vocab_size=vocab_size,
        hidden_size=hidden_size,
        context_length=context_length,
        num_layers=num_layers,
        num_heads=num_heads,
        num_kv_heads=num_kv_heads,
        ffn_hidden=ffn_hidden
    )

    print("Model created")

    try:
        model.load_state_dict(
            state,
            strict=True
        )

        print("STRICT LOAD: ✅ SUCCESS")

    except Exception as e:
        print("STRICT LOAD: ❌ FAILED")
        print(type(e).__name__)
        print(e)

    model.eval()

    test_ids = torch.randint(
        0,
        vocab_size,
        (1, 128)
    )

    with torch.no_grad():
        logits = model(test_ids)

    print("Input :", tuple(test_ids.shape))
    print("Output:", tuple(logits.shape))

print("A2 test finished")

A2 tie-weight test started
torch: 2.11.0+cu128
A2Model exists: True
state exists: True
Model created
STRICT LOAD: ❌ FAILED
RuntimeError
Error(s) in loading state_dict for A2Model:
	Missing key(s) in state_dict: "lm_head.weight". 
Input : (1, 128)
Output: (1, 128, 50000)
A2 test finished


In [39]:
# ============================================================
# A2 — FINAL CHECKPOINT VALIDATION
# NO lm_head MODULE
# ============================================================

class A2Model(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_size,
        context_length,
        num_layers,
        num_heads,
        num_kv_heads,
        ffn_hidden
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            context_length,
            hidden_size
        )

        self.layers = nn.ModuleList([
            A2Layer(
                hidden_size,
                num_heads,
                num_kv_heads,
                ffn_hidden
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(
            hidden_size
        )

        self.output_bias = nn.Parameter(
            torch.zeros(vocab_size)
        )

    def forward(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        causal_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

        for layer in self.layers:
            x = layer(x, causal_mask)

        x = self.final_norm(x)

        # Weight-tied output projection.
        # No lm_head module is registered.
        logits = F.linear(
            x,
            self.token_embedding.weight
        )

        logits = logits + self.output_bias

        return logits


# ------------------------------------------------------------
# Build fresh model
# ------------------------------------------------------------

model = A2Model(
    vocab_size=vocab_size,
    hidden_size=hidden_size,
    context_length=context_length,
    num_layers=num_layers,
    num_heads=num_heads,
    num_kv_heads=num_kv_heads,
    ffn_hidden=ffn_hidden
)

print("=" * 65)
print("A2 FINAL VALIDATION")
print("=" * 65)

print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

# Check state-dict names before loading
print("\nModel contains lm_head:",
      any(k.startswith("lm_head") for k in model.state_dict()))

# ------------------------------------------------------------
# STRICT LOAD
# ------------------------------------------------------------

try:

    model.load_state_dict(
        state,
        strict=True
    )

    print("STRICT LOAD: ✅ SUCCESS")

except Exception as e:

    print("STRICT LOAD: ❌ FAILED")
    print(type(e).__name__)
    print(e)

# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------

model.eval()

test_ids = torch.randint(
    0,
    vocab_size,
    (1, 128)
)

with torch.no_grad():

    logits = model(test_ids)

print("\nInput :", tuple(test_ids.shape))
print("Output:", tuple(logits.shape))

print("=" * 65)

A2 FINAL VALIDATION
Parameters: 51,513,680

Model contains lm_head: False
STRICT LOAD: ✅ SUCCESS

Input : (1, 128)
Output: (1, 128, 50000)


In [40]:
# ============================================================
# A3 — EXACT CHECKPOINT VALIDATION
# Prefix-LM, CPU ONLY
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

A3_PATH = "/kaggle/input/datasets/aq1de3/a3check/step_0001000.pt"

ckpt = torch.load(
    A3_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model"]

vocab_size = state["token_embedding.weight"].shape[0]
hidden_size = state["token_embedding.weight"].shape[1]
context_length = state["position_embedding.weight"].shape[0]

num_layers = len({
    int(k.split(".")[1])
    for k in state
    if k.startswith("layers.")
    and k.split(".")[1].isdigit()
})

ffn_hidden = state["layers.0.ffn.0.weight"].shape[0]
num_heads = int(ckpt["num_heads"])
prefix_ratio = float(ckpt["prefix_ratio"])

print("=" * 65)
print("A3 RECONSTRUCTION")
print("=" * 65)

print("Vocab       :", vocab_size)
print("Hidden      :", hidden_size)
print("Context     :", context_length)
print("Layers      :", num_layers)
print("Heads       :", num_heads)
print("FFN         :", ffn_hidden)
print("Prefix ratio:", prefix_ratio)


# ------------------------------------------------------------
# A3 block
# ------------------------------------------------------------

class A3Block(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_heads,
        ffn_hidden
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(hidden_size)

        self.attn = nn.MultiheadAttention(
            hidden_size,
            num_heads,
            batch_first=True
        )

        self.ln2 = nn.LayerNorm(hidden_size)

        # Match checkpoint names:
        # ffn.0 and ffn.3
        self.ffn = nn.Sequential(
            nn.Linear(
                hidden_size,
                ffn_hidden
            ),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(
                ffn_hidden,
                hidden_size
            )
        )

    def forward(
        self,
        x,
        attention_mask
    ):

        h = self.ln1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=attention_mask,
            need_weights=False
        )

        x = x + attn_out

        x = x + self.ffn(
            self.ln2(x)
        )

        return x


# ------------------------------------------------------------
# A3 model
# ------------------------------------------------------------

class A3Model(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_size,
        context_length,
        num_layers,
        num_heads,
        ffn_hidden,
        prefix_ratio
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            context_length,
            hidden_size
        )

        self.layers = nn.ModuleList([
            A3Block(
                hidden_size,
                num_heads,
                ffn_hidden
            )
            for _ in range(num_layers)
        ])

        # Checkpoint name is ln_f
        self.ln_f = nn.LayerNorm(
            hidden_size
        )

        self.prefix_ratio = prefix_ratio

    def forward(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        # Prefix-LM:
        # first prefix_ratio portion can attend bidirectionally.
        prefix_len = int(T * self.prefix_ratio)

        mask = torch.ones(
            T,
            T,
            device=input_ids.device,
            dtype=torch.bool
        )

        # Prefix tokens attend to all prefix tokens.
        mask[:prefix_len, :prefix_len] = False

        # Remaining tokens use causal attention.
        causal = torch.triu(
            torch.ones(
                T,
                T,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        mask[prefix_len:, :] = causal[prefix_len:, :]

        for layer in self.layers:

            x = layer(
                x,
                mask
            )

        x = self.ln_f(x)

        # Weight-tied output projection.
        logits = F.linear(
            x,
            self.token_embedding.weight
        )

        return logits


# ------------------------------------------------------------
# Build
# ------------------------------------------------------------

model = A3Model(
    vocab_size=vocab_size,
    hidden_size=hidden_size,
    context_length=context_length,
    num_layers=num_layers,
    num_heads=num_heads,
    ffn_hidden=ffn_hidden,
    prefix_ratio=prefix_ratio
)

print(
    "\nParameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

print(
    "Checkpoint parameters:",
    ckpt["parameters"]
)


# ------------------------------------------------------------
# Strict load
# ------------------------------------------------------------

try:

    model.load_state_dict(
        state,
        strict=True
    )

    print("\nSTRICT LOAD: ✅ SUCCESS")

except Exception as e:

    print("\nSTRICT LOAD: ❌ FAILED")
    print(type(e).__name__)
    print(e)


# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------

model.eval()

test_ids = torch.randint(
    0,
    vocab_size,
    (1, 128)
)

with torch.no_grad():

    logits = model(
        test_ids
    )

print("\nInput :", tuple(test_ids.shape))
print("Output:", tuple(logits.shape))

print("=" * 65)
print("A3 VALIDATION COMPLETE")
print("=" * 65)

A3 RECONSTRUCTION
Vocab       : 50000
Hidden      : 512
Context     : 2048
Layers      : 8
Heads       : 8
FFN         : 2048
Prefix ratio: 0.5

Parameters: 51,868,672
Checkpoint parameters: 51868672

STRICT LOAD: ✅ SUCCESS

Input : (1, 128)
Output: (1, 128, 50000)
A3 VALIDATION COMPLETE


In [41]:
# ============================================================
# A4 — EXACT CHECKPOINT VALIDATION
# Encoder–Decoder, CPU ONLY
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

A4_PATH = "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"

ckpt = torch.load(
    A4_PATH,
    map_location="cpu",
    weights_only=False
)

state = ckpt["model_state_dict"]

vocab_size = state["token_embedding.weight"].shape[0]
hidden_size = state["token_embedding.weight"].shape[1]

encoder_context = state["encoder_pos.weight"].shape[0]
decoder_context = state["decoder_pos.weight"].shape[0]

encoder_layers = len({
    int(k.split(".")[1])
    for k in state
    if k.startswith("encoder_layers.")
    and k.split(".")[1].isdigit()
})

decoder_layers = len({
    int(k.split(".")[1])
    for k in state
    if k.startswith("decoder_layers.")
    and k.split(".")[1].isdigit()
})

ffn_hidden = state["encoder_layers.0.ffn.net.0.weight"].shape[0]

num_heads = 8

print("=" * 70)
print("A4 RECONSTRUCTION")
print("=" * 70)

print("Vocab:", vocab_size)
print("Hidden:", hidden_size)
print("Encoder context:", encoder_context)
print("Decoder context:", decoder_context)
print("Encoder layers:", encoder_layers)
print("Decoder layers:", decoder_layers)
print("FFN:", ffn_hidden)


# ------------------------------------------------------------
# Multi-head attention matching A4 projections
# ------------------------------------------------------------

class A4Attention(nn.Module):

    def __init__(self, hidden_size, num_heads):
        super().__init__()

        self.q_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.k_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.v_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.out_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

    def forward(
        self,
        query,
        key,
        value,
        mask=None
    ):

        B, Tq, C = query.shape
        Tk = key.shape[1]

        q = self.q_proj(query)
        k = self.k_proj(key)
        v = self.v_proj(value)

        q = q.view(
            B, Tq,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, Tk,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, Tk,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        scores = torch.matmul(
            q,
            k.transpose(-2, -1)
        ) / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(
                mask,
                float("-inf")
            )

        attn = torch.softmax(
            scores,
            dim=-1
        )

        out = torch.matmul(
            attn,
            v
        )

        out = out.transpose(
            1, 2
        ).contiguous().view(
            B,
            Tq,
            C
        )

        return self.out_proj(out)


# ------------------------------------------------------------
# Encoder layer
# ------------------------------------------------------------

class A4EncoderLayer(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_heads,
        ffn_hidden
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)

        self.attn = A4Attention(
            hidden_size,
            num_heads
        )

        self.norm2 = nn.LayerNorm(hidden_size)

        self.ffn = nn.Sequential(
            nn.Linear(
                hidden_size,
                ffn_hidden
            ),
            nn.GELU(),
            nn.Linear(
                ffn_hidden,
                hidden_size
            )
        )

    def forward(self, x, mask=None):

        x = x + self.attn(
            self.norm1(x),
            self.norm1(x),
            self.norm1(x),
            mask
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ------------------------------------------------------------
# Decoder layer
# ------------------------------------------------------------

class A4DecoderLayer(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_heads,
        ffn_hidden
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)

        self.self_attn = A4Attention(
            hidden_size,
            num_heads
        )

        self.norm2 = nn.LayerNorm(hidden_size)

        self.cross_attn = A4Attention(
            hidden_size,
            num_heads
        )

        self.norm3 = nn.LayerNorm(hidden_size)

        self.ffn = nn.Sequential(
            nn.Linear(
                hidden_size,
                ffn_hidden
            ),
            nn.GELU(),
            nn.Linear(
                ffn_hidden,
                hidden_size
            )
        )

    def forward(
        self,
        x,
        memory,
        self_mask=None,
        cross_mask=None
    ):

        h = self.norm1(x)

        x = x + self.self_attn(
            h,
            h,
            h,
            self_mask
        )

        h = self.norm2(x)

        x = x + self.cross_attn(
            h,
            memory,
            memory,
            cross_mask
        )

        x = x + self.ffn(
            self.norm3(x)
        )

        return x


# ------------------------------------------------------------
# A4 Model
# ------------------------------------------------------------

class A4Model(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_size,
        encoder_context,
        decoder_context,
        encoder_layers,
        decoder_layers,
        num_heads,
        ffn_hidden
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.encoder_pos = nn.Parameter(
            torch.zeros(
                encoder_context,
                hidden_size
            )
        )

        self.decoder_pos = nn.Parameter(
            torch.zeros(
                decoder_context,
                hidden_size
            )
        )

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(
                hidden_size,
                num_heads,
                ffn_hidden
            )
            for _ in range(encoder_layers)
        ])

        self.encoder_norm = nn.LayerNorm(
            hidden_size
        )

        self.decoder_layers = nn.ModuleList([
            A4DecoderLayer(
                hidden_size,
                num_heads,
                ffn_hidden
            )
            for _ in range(decoder_layers)
        ])

        self.decoder_norm = nn.LayerNorm(
            hidden_size
        )

        self.lm_head = nn.Linear(
            hidden_size,
            vocab_size,
            bias=False
        )


# ------------------------------------------------------------
# Build
# ------------------------------------------------------------

model = A4Model(
    vocab_size,
    hidden_size,
    encoder_context,
    decoder_context,
    encoder_layers,
    decoder_layers,
    num_heads,
    ffn_hidden
)

print(
    "\nParameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

print(
    "Checkpoint parameters:",
    sum(v.numel() for v in state.values())
)


# ------------------------------------------------------------
# Strict load
# ------------------------------------------------------------

try:

    model.load_state_dict(
        state,
        strict=True
    )

    print("\nSTRICT LOAD: ✅ SUCCESS")

except Exception as e:

    print("\nSTRICT LOAD: ❌ FAILED")
    print(type(e).__name__)
    print(e)


print("=" * 70)


A4 RECONSTRUCTION
Vocab: 50000
Hidden: 512
Encoder context: 1024
Decoder context: 1024
Encoder layers: 6
Decoder layers: 6
FFN: 2048

Parameters: 96,389,120
Checkpoint parameters: 96389120

STRICT LOAD: ❌ FAILED
RuntimeError
Error(s) in loading state_dict for A4Model:
	Missing key(s) in state_dict: "encoder_pos", "decoder_pos", "encoder_layers.0.ffn.0.weight", "encoder_layers.0.ffn.0.bias", "encoder_layers.0.ffn.2.weight", "encoder_layers.0.ffn.2.bias", "encoder_layers.1.ffn.0.weight", "encoder_layers.1.ffn.0.bias", "encoder_layers.1.ffn.2.weight", "encoder_layers.1.ffn.2.bias", "encoder_layers.2.ffn.0.weight", "encoder_layers.2.ffn.0.bias", "encoder_layers.2.ffn.2.weight", "encoder_layers.2.ffn.2.bias", "encoder_layers.3.ffn.0.weight", "encoder_layers.3.ffn.0.bias", "encoder_layers.3.ffn.2.weight", "encoder_layers.3.ffn.2.bias", "encoder_layers.4.ffn.0.weight", "encoder_layers.4.ffn.0.bias", "encoder_layers.4.ffn.2.weight", "encoder_layers.4.ffn.2.bias", "encoder_layers.5.ffn.0.weight

In [42]:
# ============================================================
# A4 — CORRECT PARAMETER NAMES
# ============================================================

class A4EncoderLayer(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_heads,
        ffn_hidden
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)

        self.attn = A4Attention(
            hidden_size,
            num_heads
        )

        self.norm2 = nn.LayerNorm(hidden_size)

        # Checkpoint uses ffn.net.0 / ffn.net.2
        self.ffn = nn.ModuleDict({
            "net": nn.Sequential(
                nn.Linear(
                    hidden_size,
                    ffn_hidden
                ),
                nn.GELU(),
                nn.Linear(
                    ffn_hidden,
                    hidden_size
                )
            )
        })

    def forward(self, x, mask=None):

        h = self.norm1(x)

        x = x + self.attn(
            h,
            h,
            h,
            mask
        )

        x = x + self.ffn["net"](
            self.norm2(x)
        )

        return x


class A4DecoderLayer(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_heads,
        ffn_hidden
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)

        self.self_attn = A4Attention(
            hidden_size,
            num_heads
        )

        self.norm2 = nn.LayerNorm(hidden_size)

        self.cross_attn = A4Attention(
            hidden_size,
            num_heads
        )

        self.norm3 = nn.LayerNorm(hidden_size)

        # Checkpoint uses ffn.net.0 / ffn.net.2
        self.ffn = nn.ModuleDict({
            "net": nn.Sequential(
                nn.Linear(
                    hidden_size,
                    ffn_hidden
                ),
                nn.GELU(),
                nn.Linear(
                    ffn_hidden,
                    hidden_size
                )
            )
        })

    def forward(
        self,
        x,
        memory,
        self_mask=None,
        cross_mask=None
    ):

        h = self.norm1(x)

        x = x + self.self_attn(
            h,
            h,
            h,
            self_mask
        )

        h = self.norm2(x)

        x = x + self.cross_attn(
            h,
            memory,
            memory,
            cross_mask
        )

        x = x + self.ffn["net"](
            self.norm3(x)
        )

        return x


class A4Model(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_size,
        encoder_context,
        decoder_context,
        encoder_layers,
        decoder_layers,
        num_heads,
        ffn_hidden
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        # Checkpoint requires .weight
        self.encoder_pos = nn.Embedding(
            encoder_context,
            hidden_size
        )

        self.decoder_pos = nn.Embedding(
            decoder_context,
            hidden_size
        )

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(
                hidden_size,
                num_heads,
                ffn_hidden
            )
            for _ in range(encoder_layers)
        ])

        self.encoder_norm = nn.LayerNorm(
            hidden_size
        )

        self.decoder_layers = nn.ModuleList([
            A4DecoderLayer(
                hidden_size,
                num_heads,
                ffn_hidden
            )
            for _ in range(decoder_layers)
        ])

        self.decoder_norm = nn.LayerNorm(
            hidden_size
        )

        self.lm_head = nn.Linear(
            hidden_size,
            vocab_size,
            bias=False
        )


# ------------------------------------------------------------
# Rebuild
# ------------------------------------------------------------

model = A4Model(
    vocab_size,
    hidden_size,
    encoder_context,
    decoder_context,
    encoder_layers,
    decoder_layers,
    num_heads,
    ffn_hidden
)

print("=" * 70)
print("A4 CORRECTED VALIDATION")
print("=" * 70)

print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

print(
    "Checkpoint:",
    f"{sum(v.numel() for v in state.values()):,}"
)

try:

    model.load_state_dict(
        state,
        strict=True
    )

    print("STRICT LOAD: ✅ SUCCESS")

except Exception as e:

    print("STRICT LOAD: ❌ FAILED")
    print(type(e).__name__)
    print(e)

print("=" * 70)

A4 CORRECTED VALIDATION
Parameters: 96,389,120
Checkpoint: 96,389,120
STRICT LOAD: ✅ SUCCESS


In [43]:
# ============================================================
# A4 — CPU FORWARD SANITY TEST
# ============================================================

import torch

print("=" * 70)
print("A4 FORWARD SANITY TEST")
print("=" * 70)

model.eval()

# Small test sequence
encoder_ids = torch.randint(
    0,
    vocab_size,
    (1, 64)
)

decoder_ids = torch.randint(
    0,
    vocab_size,
    (1, 32)
)

print("Encoder input :", tuple(encoder_ids.shape))
print("Decoder input :", tuple(decoder_ids.shape))

# NOTE:
# We first inspect the model's forward signature rather than
# assuming an encoder-decoder calling convention.

import inspect

print("\nForward signature:")
print(inspect.signature(model.forward))

print("=" * 70)

A4 FORWARD SANITY TEST
Encoder input : (1, 64)
Decoder input : (1, 32)

Forward signature:
(*input: Any) -> None


In [44]:
# ============================================================
# A4 — ENCODER REPRESENTATION SANITY TEST
# CPU ONLY
# ============================================================

import torch

print("=" * 70)
print("A4 ENCODER SANITY TEST")
print("=" * 70)

model.eval()

input_ids = torch.randint(
    0,
    vocab_size,
    (1, 64)
)

x = (
    model.token_embedding(input_ids)
    + model.encoder_pos.weight[:64].unsqueeze(0)
)

print("Embedding:", tuple(x.shape))

# Encoder self-attention is bidirectional for NER.
for i, layer in enumerate(model.encoder_layers):

    x = layer(
        x,
        mask=None
    )

    if i in [0, len(model.encoder_layers) - 1]:
        print(
            f"Encoder layer {i+1}:",
            tuple(x.shape)
        )

x = model.encoder_norm(x)

print("Final encoder representation:",
      tuple(x.shape))

print(
    "Finite values:",
    bool(torch.isfinite(x).all())
)

print("=" * 70)

A4 ENCODER SANITY TEST
Embedding: (1, 64, 512)
Encoder layer 1: (1, 64, 512)
Encoder layer 6: (1, 64, 512)
Final encoder representation: (1, 64, 512)
Finite values: True


In [45]:
import torch, time

print("=" * 70)
print("GPU BENCHMARK")
print("=" * 70)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | VRAM: {p.total_memory/1024**3:.2f} GB")

if torch.cuda.is_available():
    device = torch.device("cuda:0")

    x = torch.randn(2048, 2048, device=device)
    torch.cuda.synchronize()

    start = time.time()

    for _ in range(50):
        y = x @ x

    torch.cuda.synchronize()
    elapsed = time.time() - start

    print(f"\nMatrix benchmark: {elapsed:.3f}s")
    print(f"Allocated VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"Reserved VRAM : {torch.cuda.memory_reserved()/1024**3:.2f} GB")

    del x, y
    torch.cuda.empty_cache()

print("=" * 70)

GPU BENCHMARK
PyTorch : 2.11.0+cu128
CUDA    : True
GPU count: 2
GPU 0: Tesla T4 | VRAM: 14.56 GB
GPU 1: Tesla T4 | VRAM: 14.56 GB

Matrix benchmark: 0.502s
Allocated VRAM: 0.04 GB
Reserved VRAM : 0.07 GB


In [46]:
# ============================================================
# PHASE 2 — SHARED L-NER DATA PIPELINE
# Cell 1: BIO labels + character/token alignment
# ============================================================

import json
import re
from pathlib import Path
from collections import Counter

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
DATA_ROOT = Path("/kaggle/working/phase2_data")

# If you already created these variables earlier, this will
# simply reuse them.
TRAIN1_TEXT = DATA_ROOT / "fold1-text.json"
TRAIN1_GOLD = DATA_ROOT / "fold1-gold.json"
TRAIN2_TEXT = DATA_ROOT / "fold2-text.json"
TRAIN2_GOLD = DATA_ROOT / "fold2-gold.json"
TEST_TEXT   = DATA_ROOT / "fold3-text.json"

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------
ENTITY_LABELS = [
    "APP",
    "RESP",
    "A.COUNSEL",
    "R.COUNSEL",
    "JUDGE",
    "WIT",
    "AUTH",
    "COURT",
    "STAT",
    "PREC",
    "DATE",
    "CASENO",
]

BIO_LABELS = ["O"]

for label in ENTITY_LABELS:
    BIO_LABELS.append(f"B-{label}")
    BIO_LABELS.append(f"I-{label}")

LABEL2ID = {x: i for i, x in enumerate(BIO_LABELS)}
ID2LABEL = {i: x for x, i in LABEL2ID.items()}

print("Number of BIO classes:", len(BIO_LABELS))
print("Classes:", BIO_LABELS)

Number of BIO classes: 25
Classes: ['O', 'B-APP', 'I-APP', 'B-RESP', 'I-RESP', 'B-A.COUNSEL', 'I-A.COUNSEL', 'B-R.COUNSEL', 'I-R.COUNSEL', 'B-JUDGE', 'I-JUDGE', 'B-WIT', 'I-WIT', 'B-AUTH', 'I-AUTH', 'B-COURT', 'I-COURT', 'B-STAT', 'I-STAT', 'B-PREC', 'I-PREC', 'B-DATE', 'I-DATE', 'B-CASENO', 'I-CASENO']


In [47]:
# ============================================================
# Cell 2: Gold character spans -> BIO token labels
# ============================================================

def normalize_gold_span(span):
    """
    Gold format:
        [start, end, label]
    or
        [start, end, label, entity_text]
    """
    start = int(span[0])
    end   = int(span[1])
    label = str(span[2])
    return start, end, label


def char_spans_to_bio(text, gold_spans, token_offsets):
    """
    Convert character-level gold annotations to BIO labels.

    token_offsets:
        [(start_char, end_char), ...]

    Important:
    The gold annotations use character offsets while the BPE
    tokenizer can place a token boundary one character before
    the gold start because of its leading whitespace marker.
    Therefore we use overlap/coverage rather than requiring
    exact token-boundary equality.
    """

    labels = ["O"] * len(token_offsets)

    for raw_span in gold_spans:
        start, end, entity_type = normalize_gold_span(raw_span)

        covered = []

        for tok_i, (tok_start, tok_end) in enumerate(token_offsets):

            # No intersection
            if tok_end <= start or tok_start >= end:
                continue

            # Token overlaps entity
            covered.append(tok_i)

        if not covered:
            continue

        first = covered[0]
        last  = covered[-1]

        labels[first] = f"B-{entity_type}"

        for i in covered[1:]:
            labels[i] = f"I-{entity_type}"

    return labels


print("BIO conversion function ready.")

BIO conversion function ready.


In [48]:
# ============================================================
# Cell 3: BIO token predictions -> character spans
# ============================================================

def bio_to_char_spans(
    text,
    token_offsets,
    token_labels
):
    """
    Convert BIO token predictions into character-level spans.

    Only whitespace is automatically trimmed from the boundaries.
    We DO NOT strip punctuation because punctuation can legitimately
    belong to a gold entity.
    """

    spans = []

    current_label = None
    current_start = None
    current_end = None

    def close_entity():
        nonlocal current_label, current_start, current_end

        if current_label is None:
            return

        start = current_start
        end = current_end

        # Correct tokenizer leading/trailing whitespace only.
        while start < end and text[start].isspace():
            start += 1

        while end > start and text[end - 1].isspace():
            end -= 1

        if start < end:
            spans.append([
                start,
                end,
                current_label
            ])

        current_label = None
        current_start = None
        current_end = None

    for i, label in enumerate(token_labels):

        tok_start, tok_end = token_offsets[i]

        if label == "O":
            close_entity()
            continue

        prefix, entity_type = label.split("-", 1)

        if prefix == "B":
            close_entity()

            current_label = entity_type
            current_start = tok_start
            current_end = tok_end

        elif prefix == "I":

            # Invalid I without a preceding matching B.
            # Treat it as a new entity.
            if (
                current_label is None
                or current_label != entity_type
            ):
                close_entity()

                current_label = entity_type
                current_start = tok_start
                current_end = tok_end

            else:
                current_end = tok_end

    close_entity()

    return spans


print("BIO -> character span decoder ready.")

BIO -> character span decoder ready.


In [49]:
# ============================================================
# Cell 4: Official-style strict span evaluator
# ============================================================

def strict_macro_f1(gold_docs, pred_docs):
    """
    Exact match on:
        (character_start, character_end, label)

    Macro-F1 is calculated across all 12 entity labels.
    """

    tp = Counter()
    fp = Counter()
    fn = Counter()

    for doc_id, gold_spans in gold_docs.items():

        gold_set = {
            (int(s[0]), int(s[1]), str(s[2]))
            for s in gold_spans
        }

        pred_set = {
            (int(s[0]), int(s[1]), str(s[2]))
            for s in pred_docs.get(doc_id, [])
        }

        for span in pred_set:
            label = span[2]

            if span in gold_set:
                tp[label] += 1
            else:
                fp[label] += 1

        for span in gold_set:
            label = span[2]

            if span not in pred_set:
                fn[label] += 1

    per_label = {}

    for label in ENTITY_LABELS:

        precision = (
            tp[label] / (tp[label] + fp[label])
            if tp[label] + fp[label] > 0
            else 0.0
        )

        recall = (
            tp[label] / (tp[label] + fn[label])
            if tp[label] + fn[label] > 0
            else 0.0
        )

        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = 0.0

        per_label[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": tp[label],
            "fp": fp[label],
            "fn": fn[label],
        }

    macro_f1 = sum(
        x["f1"] for x in per_label.values()
    ) / len(ENTITY_LABELS)

    return macro_f1, per_label


print("Strict Macro-F1 evaluator ready.")

Strict Macro-F1 evaluator ready.


In [50]:
# ============================================================
# CELL 6 — REAL DATA BIO ALIGNMENT VALIDATION
# SELF-CONTAINED VERSION
# ============================================================

from collections import Counter

def validate_dataset_alignment(text_docs, gold_docs, sample_name):

    total_entities = 0
    mapped_entities = 0
    failed_entities = 0

    entity_distribution = Counter()
    coverage_distribution = Counter()

    for doc_id, text in text_docs.items():

        gold_spans = gold_docs.get(doc_id, [])

        # Tokenize without special tokens
        enc = tokenizer.encode(
            text,
            add_special_tokens=False
        )

        token_offsets = enc.offsets

        for span in gold_spans:

            # Gold format:
            # [start, end, label]
            # or
            # [start, end, label, entity_text]

            start = int(span[0])
            end   = int(span[1])
            label = str(span[2])

            total_entities += 1
            entity_distribution[label] += 1

            covered = []

            for token_idx, (token_start, token_end) in enumerate(
                token_offsets
            ):

                # No overlap
                if token_end <= start:
                    continue

                if token_start >= end:
                    continue

                covered.append(token_idx)

            if len(covered) > 0:

                mapped_entities += 1
                coverage_distribution[len(covered)] += 1

            else:
                failed_entities += 1

    success = (
        mapped_entities / total_entities
        if total_entities > 0
        else 0.0
    )

    print("=" * 70)
    print(sample_name)
    print("=" * 70)

    print(f"Documents       : {len(text_docs)}")
    print(f"Entities        : {total_entities:,}")
    print(f"Mapped          : {mapped_entities:,}")
    print(f"Failed          : {failed_entities:,}")
    print(f"Mapping success : {success * 100:.4f}%")

    print("\nEntity distribution:")
    for label, count in entity_distribution.most_common():
        print(f"  {label:12s}: {count:5d}")

    print("\nToken coverage:")
    for n, count in sorted(coverage_distribution.items()):
        print(f"  {n:3d} token(s): {count:,}")

    return {
        "total": total_entities,
        "mapped": mapped_entities,
        "failed": failed_entities,
        "success": success,
    }


# ============================================================
# FOLD 1
# ============================================================

r1 = validate_dataset_alignment(
    fold1_text,
    fold1_gold,
    "FOLD 1"
)


# ============================================================
# FOLD 2
# ============================================================

r2 = validate_dataset_alignment(
    fold2_text,
    fold2_gold,
    "FOLD 2"
)


# ============================================================
# FINAL CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL ALIGNMENT CHECK")
print("=" * 70)

print(f"Fold 1 failed : {r1['failed']}")
print(f"Fold 2 failed : {r2['failed']}")

assert r1["failed"] == 0
assert r2["failed"] == 0

print("\n✅ REAL DATA ALIGNMENT PASSED")

FOLD 1
Documents       : 35
Entities        : 4,120
Mapped          : 4,120
Failed          : 0
Mapping success : 100.0000%

Entity distribution:
  DATE        :   813
  PREC        :   624
  AUTH        :   535
  STAT        :   525
  COURT       :   455
  CASENO      :   423
  APP         :   207
  RESP        :   191
  JUDGE       :   123
  R.COUNSEL   :    87
  A.COUNSEL   :    70
  WIT         :    67

Token coverage:
    1 token(s): 503
    2 token(s): 1,009
    3 token(s): 328
    4 token(s): 320
    5 token(s): 907
    6 token(s): 209
    7 token(s): 167
    8 token(s): 113
    9 token(s): 123
   10 token(s): 89
   11 token(s): 28
   12 token(s): 63
   13 token(s): 17
   14 token(s): 24
   15 token(s): 30
   16 token(s): 29
   17 token(s): 22
   18 token(s): 19
   19 token(s): 17
   20 token(s): 15
   21 token(s): 11
   22 token(s): 14
   23 token(s): 9
   24 token(s): 9
   25 token(s): 7
   26 token(s): 5
   27 token(s): 7
   28 token(s): 7
   29 token(s): 4
   30 token(s): 3


In [51]:
# ============================================================
# CELL 7 — BUILD GPU-READY L-NER CHUNKS
# ============================================================

from dataclasses import dataclass
from typing import List, Dict
import numpy as np


MAX_TOKENS = 1536
OVERLAP = 384
STRIDE = MAX_TOKENS - OVERLAP

print("=" * 70)
print("BUILDING L-NER CHUNKS")
print("=" * 70)

@dataclass
class NERChunk:
    doc_id: str
    chunk_id: int

    input_ids: list
    attention_mask: list
    token_offsets: list

    bio_labels: list

    char_start: int
    char_end: int


def build_gold_token_labels(
    text,
    gold_spans,
    token_offsets
):
    """
    Character-level gold -> BIO token labels.
    """

    labels = ["O"] * len(token_offsets)

    for span in gold_spans:

        start = int(span[0])
        end = int(span[1])
        entity_type = str(span[2])

        covered = []

        for i, (ts, te) in enumerate(token_offsets):

            if te <= start:
                continue

            if ts >= end:
                continue

            covered.append(i)

        if not covered:
            continue

        labels[covered[0]] = f"B-{entity_type}"

        for i in covered[1:]:
            labels[i] = f"I-{entity_type}"

    return labels


def build_chunks_for_document(
    doc_id,
    text,
    gold_spans,
    max_tokens=MAX_TOKENS,
    overlap=OVERLAP
):

    enc = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    input_ids = enc.ids
    offsets = enc.offsets

    labels = build_gold_token_labels(
        text,
        gold_spans,
        offsets
    )

    chunks = []

    stride = max_tokens - overlap

    chunk_id = 0

    for start in range(
        0,
        len(input_ids),
        stride
    ):

        end = min(
            start + max_tokens,
            len(input_ids)
        )

        chunk_ids = input_ids[start:end]
        chunk_offsets = offsets[start:end]
        chunk_labels = labels[start:end]

        if not chunk_ids:
            break

        chunks.append(
            NERChunk(
                doc_id=doc_id,
                chunk_id=chunk_id,

                input_ids=chunk_ids,
                attention_mask=[1] * len(chunk_ids),
                token_offsets=chunk_offsets,

                bio_labels=chunk_labels,

                char_start=chunk_offsets[0][0],
                char_end=chunk_offsets[-1][1],
            )
        )

        chunk_id += 1

        if end >= len(input_ids):
            break

    return chunks


# ------------------------------------------------------------
# Build both folds
# ------------------------------------------------------------

all_chunks = []

for text_docs, gold_docs, fold_name in [
    (fold1_text, fold1_gold, "fold1"),
    (fold2_text, fold2_gold, "fold2"),
]:

    fold_chunks = []

    for doc_id, text in text_docs.items():

        chunks = build_chunks_for_document(
            doc_id,
            text,
            gold_docs[doc_id]
        )

        fold_chunks.extend(chunks)

    all_chunks.extend(fold_chunks)

    print(
        f"{fold_name}: "
        f"{len(text_docs)} docs → "
        f"{len(fold_chunks)} chunks"
    )


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

lengths = np.array([
    len(c.input_ids)
    for c in all_chunks
])

print("\n" + "=" * 70)
print("CHUNK DATASET READY")
print("=" * 70)

print(f"Documents : {len(fold1_text) + len(fold2_text)}")
print(f"Chunks    : {len(all_chunks)}")
print(f"Min tokens: {lengths.min()}")
print(f"Mean      : {lengths.mean():.1f}")
print(f"Median    : {np.median(lengths):.1f}")
print(f"Max       : {lengths.max()}")

print("\nLabel distribution:")

chunk_label_counts = Counter()

for chunk in all_chunks:
    chunk_label_counts.update(
        x for x in chunk.bio_labels
        if x != "O"
    )

for label, count in chunk_label_counts.most_common():
    print(f"  {label:15s}: {count:,}")

print("\n✅ CHUNK PIPELINE COMPLETE")

BUILDING L-NER CHUNKS
fold1: 35 docs → 203 chunks
fold2: 35 docs → 203 chunks

CHUNK DATASET READY
Documents : 70
Chunks    : 406
Min tokens: 428
Mean      : 1438.2
Median    : 1536.0
Max       : 1536

Label distribution:
  I-PREC         : 14,178
  I-DATE         : 7,595
  I-CASENO       : 6,753
  I-STAT         : 4,158
  B-DATE         : 1,936
  I-COURT        : 1,780
  B-STAT         : 1,604
  I-AUTH         : 1,579
  B-PREC         : 1,531
  B-COURT        : 1,283
  B-AUTH         : 1,279
  I-APP          : 1,015
  I-RESP         : 954
  I-JUDGE        : 933
  B-CASENO       : 915
  I-R.COUNSEL    : 564
  I-A.COUNSEL    : 547
  B-APP          : 485
  B-RESP         : 467
  I-WIT          : 399
  B-JUDGE        : 276
  B-WIT          : 239
  B-A.COUNSEL    : 235
  B-R.COUNSEL    : 231

✅ CHUNK PIPELINE COMPLETE


In [52]:
# ============================================================
# CELL 8 — GPU NER FORWARD/BACKWARD BENCHMARK
# ============================================================

import torch
import torch.nn as nn
import time
import gc

device = torch.device("cuda:0")

print("=" * 70)
print("GPU NER BENCHMARK")
print("=" * 70)

# ------------------------------------------------------------
# Lightweight representative NER model
# ------------------------------------------------------------
# This benchmark measures the NER head + sequence workload.
# The actual A1/A2/A3/A4 encoder is plugged in next.

class BenchmarkNER(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_labels=25
    ):
        super().__init__()

        self.classifier = nn.Linear(
            hidden_size,
            num_labels
        )

    def forward(self, x):
        return self.classifier(x)


model = BenchmarkNER().to(device)

criterion = nn.CrossEntropyLoss()

# ------------------------------------------------------------
# Test batch sizes
# ------------------------------------------------------------

BATCH_SIZES = [1, 2, 4, 8]

SEQ_LEN = 1536
HIDDEN = 512

results = []

for batch_size in BATCH_SIZES:

    print(f"\nTesting batch size = {batch_size}")

    try:

        torch.cuda.empty_cache()
        gc.collect()

        x = torch.randn(
            batch_size,
            SEQ_LEN,
            HIDDEN,
            device=device,
            dtype=torch.float16,
            requires_grad=True
        )

        y = torch.randint(
            0,
            25,
            (batch_size, SEQ_LEN),
            device=device
        )

        model.half()

        # Warmup
        for _ in range(3):

            logits = model(x)
            loss = criterion(
                logits.float().reshape(-1, 25),
                y.reshape(-1)
            )

            loss.backward()
            model.zero_grad(set_to_none=True)

        torch.cuda.synchronize()

        start = time.time()

        steps = 10

        for _ in range(steps):

            logits = model(x)

            loss = criterion(
                logits.float().reshape(-1, 25),
                y.reshape(-1)
            )

            loss.backward()

            model.zero_grad(set_to_none=True)

        torch.cuda.synchronize()

        elapsed = time.time() - start

        tokens = batch_size * SEQ_LEN * steps
        tokens_sec = tokens / elapsed

        allocated = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        reserved = (
            torch.cuda.max_memory_reserved()
            / 1024**3
        )

        print(f"  Time       : {elapsed:.3f}s")
        print(f"  Tokens/sec : {tokens_sec:,.0f}")
        print(f"  VRAM       : {allocated:.2f} GB")
        print(f"  Reserved   : {reserved:.2f} GB")

        results.append({
            "batch_size": batch_size,
            "tokens_sec": tokens_sec,
            "vram_gb": allocated
        })

        del x, y, logits, loss

    except RuntimeError as e:

        if "out of memory" in str(e).lower():

            print("  ❌ CUDA OUT OF MEMORY")

            torch.cuda.empty_cache()
            gc.collect()

        else:
            raise


torch.cuda.empty_cache()
gc.collect()

print("\n" + "=" * 70)
print("BENCHMARK COMPLETE")
print("=" * 70)

for r in results:
    print(
        f"Batch {r['batch_size']:2d} | "
        f"{r['tokens_sec']:>10,.0f} tok/s | "
        f"{r['vram_gb']:.2f} GB"
    )

GPU NER BENCHMARK

Testing batch size = 1
  Time       : 0.007s
  Tokens/sec : 2,258,211
  VRAM       : 0.36 GB
  Reserved   : 0.36 GB

Testing batch size = 2
  Time       : 0.007s
  Tokens/sec : 4,265,535
  VRAM       : 0.36 GB
  Reserved   : 0.36 GB

Testing batch size = 4
  Time       : 0.008s
  Tokens/sec : 8,075,271
  VRAM       : 0.36 GB
  Reserved   : 0.36 GB

Testing batch size = 8
  Time       : 0.010s
  Tokens/sec : 12,922,701
  VRAM       : 0.36 GB
  Reserved   : 0.36 GB

BENCHMARK COMPLETE
Batch  1 |  2,258,211 tok/s | 0.36 GB
Batch  2 |  4,265,535 tok/s | 0.36 GB
Batch  4 |  8,075,271 tok/s | 0.36 GB
Batch  8 | 12,922,701 tok/s | 0.36 GB


In [53]:
# ============================================================
# CELL 9 — REAL A4 TRANSFORMER GPU BENCHMARK
# ============================================================

import torch
import torch.nn as nn
import time
import gc

device = torch.device("cuda:0")

print("=" * 70)
print("REAL A4 TRANSFORMER BENCHMARK")
print("=" * 70)

# ------------------------------------------------------------
# IMPORTANT:
# `a4_model` must be the reconstructed A4 model that already
# passed STRICT LOAD.
# ------------------------------------------------------------

assert "a4_model" in globals(), (
    "a4_model not found. Use the already validated A4 model "
    "variable from the reconstruction cell."
)

a4_model = a4_model.to(device)
a4_model.train()

# NER classifier
ner_head = nn.Linear(
    512,
    25
).to(device)

# FP16
a4_model.half()
ner_head.half()

criterion = nn.CrossEntropyLoss()

SEQ_LEN = 1536
BATCH_SIZES = [1, 2, 4]

results = []

for batch_size in BATCH_SIZES:

    print(f"\n{'-'*60}")
    print(f"Testing A4 batch size = {batch_size}")

    try:

        torch.cuda.empty_cache()
        gc.collect()
        torch.cuda.reset_peak_memory_stats()

        input_ids = torch.randint(
            0,
            50000,
            (batch_size, SEQ_LEN),
            device=device,
            dtype=torch.long
        )

        targets = torch.randint(
            0,
            25,
            (batch_size, SEQ_LEN),
            device=device,
            dtype=torch.long
        )

        # ----------------------------------------------------
        # Warmup
        # ----------------------------------------------------

        for _ in range(2):

            hidden = a4_model.encode(
                input_ids
            )

            logits = ner_head(hidden)

            loss = criterion(
                logits.float().reshape(-1, 25),
                targets.reshape(-1)
            )

            loss.backward()

            a4_model.zero_grad(set_to_none=True)
            ner_head.zero_grad(set_to_none=True)

        torch.cuda.synchronize()

        # ----------------------------------------------------
        # Timed benchmark
        # ----------------------------------------------------

        steps = 3

        start = time.time()

        for _ in range(steps):

            hidden = a4_model.encode(
                input_ids
            )

            logits = ner_head(hidden)

            loss = criterion(
                logits.float().reshape(-1, 25),
                targets.reshape(-1)
            )

            loss.backward()

            a4_model.zero_grad(set_to_none=True)
            ner_head.zero_grad(set_to_none=True)

        torch.cuda.synchronize()

        elapsed = time.time() - start

        total_tokens = (
            batch_size *
            SEQ_LEN *
            steps
        )

        tokens_per_sec = total_tokens / elapsed

        peak_vram = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        reserved_vram = (
            torch.cuda.max_memory_reserved()
            / 1024**3
        )

        print(f"Time       : {elapsed:.3f}s")
        print(f"Tokens/sec : {tokens_per_sec:,.0f}")
        print(f"Peak VRAM  : {peak_vram:.2f} GB")
        print(f"Reserved   : {reserved_vram:.2f} GB")

        results.append({
            "batch_size": batch_size,
            "tokens_sec": tokens_per_sec,
            "peak_vram": peak_vram
        })

        del input_ids, targets
        del hidden, logits, loss

    except RuntimeError as e:

        if "out of memory" in str(e).lower():

            print("❌ CUDA OUT OF MEMORY")

            results.append({
                "batch_size": batch_size,
                "tokens_sec": 0,
                "peak_vram": 14.56
            })

            torch.cuda.empty_cache()
            gc.collect()

        else:
            raise


# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

torch.cuda.empty_cache()
gc.collect()

print("\n" + "=" * 70)
print("REAL A4 BENCHMARK RESULTS")
print("=" * 70)

for r in results:
    print(
        f"Batch {r['batch_size']:2d} | "
        f"{r['tokens_sec']:>10,.0f} tok/s | "
        f"Peak VRAM {r['peak_vram']:.2f} GB"
    )

REAL A4 TRANSFORMER BENCHMARK


AssertionError: a4_model not found. Use the already validated A4 model variable from the reconstruction cell.

In [ ]:
# ============================================================
# A4 MODEL VARIABLE CHECK — FIXED
# ============================================================

import torch
import torch.nn as nn

print("=" * 70)
print("SEARCHING FOR A4 MODEL OBJECTS")
print("=" * 70)

candidates = []

# IMPORTANT: list(...) prevents "dictionary changed size"
namespace = list(globals().items())

for name, obj in namespace:

    if isinstance(obj, nn.Module):

        try:
            params = sum(
                p.numel()
                for p in obj.parameters()
            )

            if params == 96_389_120:
                candidates.append(name)

                print(
                    f"FOUND: {name} | "
                    f"Parameters: {params:,}"
                )

        except Exception:
            pass

print("\nCandidates:", candidates)

if not candidates:
    print(
        "\n⚠️ No validated A4 model object is currently "
        "stored in the notebook namespace."
    )
    print(
        "That's okay — we will reload the A4 checkpoint "
        "using the validated architecture."
    )

In [ ]:
# ============================================================
# CELL 9 — A4 RELOAD + REAL GPU NER BENCHMARK
# ============================================================

import os
import gc
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

DEVICE = torch.device("cuda:0")
SEQ_LEN = 1536
NUM_LABELS = 25

A4_CKPT = (
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)

print("=" * 70)
print("A4 RELOAD + REAL GPU BENCHMARK")
print("=" * 70)

assert torch.cuda.is_available(), "CUDA is not available."
assert os.path.exists(A4_CKPT), f"Checkpoint not found: {A4_CKPT}"

print("Checkpoint:", A4_CKPT)
print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# A4 ARCHITECTURE
# ============================================================

class A4FFN(nn.Module):

    def __init__(self, hidden_size=512, ffn_size=2048):
        super().__init__()

        self.net = nn.ModuleDict({
            "0": nn.Linear(hidden_size, ffn_size),
            "2": nn.Linear(ffn_size, hidden_size),
        })

    def forward(self, x):
        x = self.net["0"](x)
        x = F.gelu(x)
        x = self.net["2"](x)
        return x


class A4Attention(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_heads=8
    ):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(
        self,
        x,
        key_value=None,
        attn_mask=None
    ):

        if key_value is None:
            key_value = x

        out, _ = self.attn(
            x,
            key_value,
            key_value,
            attn_mask=attn_mask,
            need_weights=False
        )

        return out


class A4EncoderLayer(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_heads=8,
        ffn_size=2048
    ):
        super().__init__()

        self.self_attn = A4Attention(
            hidden_size,
            num_heads
        )

        self.norm1 = nn.LayerNorm(
            hidden_size
        )

        self.ffn = A4FFN(
            hidden_size,
            ffn_size
        )

        self.norm2 = nn.LayerNorm(
            hidden_size
        )

    def forward(self, x):

        x = self.norm1(
            x + self.self_attn(x)
        )

        x = self.norm2(
            x + self.ffn(x)
        )

        return x


class A4DecoderLayer(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_heads=8,
        ffn_size=2048
    ):
        super().__init__()

        self.self_attn = A4Attention(
            hidden_size,
            num_heads
        )

        self.norm1 = nn.LayerNorm(
            hidden_size
        )

        self.cross_attn = A4Attention(
            hidden_size,
            num_heads
        )

        self.norm2 = nn.LayerNorm(
            hidden_size
        )

        self.ffn = A4FFN(
            hidden_size,
            ffn_size
        )

        self.norm3 = nn.LayerNorm(
            hidden_size
        )

    def forward(
        self,
        x,
        memory
    ):

        x = self.norm1(
            x + self.self_attn(x)
        )

        x = self.norm2(
            x + self.cross_attn(
                x,
                memory
            )
        )

        x = self.norm3(
            x + self.ffn(x)
        )

        return x


class A4Encoder(nn.Module):

    def __init__(
        self,
        vocab_size=50000,
        hidden_size=512,
        num_heads=8,
        ffn_size=2048,
        num_layers=6,
        max_position=1024
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.encoder_pos = nn.Embedding(
            max_position,
            hidden_size
        )

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(
                hidden_size,
                num_heads,
                ffn_size
            )
            for _ in range(num_layers)
        ])

    def encode(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        ).unsqueeze(0)

        x = (
            self.token_embedding(input_ids)
            + self.encoder_pos(positions)
        )

        for layer in self.encoder_layers:
            x = layer(x)

        return x


# ============================================================
# LOAD CHECKPOINT
# ============================================================

print("\nLoading checkpoint...")

checkpoint = torch.load(
    A4_CKPT,
    map_location="cpu",
    weights_only=False
)

state_dict = checkpoint["model_state_dict"]

print("Checkpoint step:",
      checkpoint.get("step"))

print("Checkpoint train loss:",
      checkpoint.get("train_loss"))

print(
    "Checkpoint tensors:",
    len(state_dict)
)


# ------------------------------------------------------------
# Reconstruct the full A4 encoder
# ------------------------------------------------------------

a4_model = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    num_heads=8,
    ffn_size=2048,
    num_layers=6,
    max_position=1024
)

# ------------------------------------------------------------
# Extract encoder-only weights
#
# The checkpoint also contains decoder weights.
# We load the matching encoder subset.
# ------------------------------------------------------------

encoder_state = {}

for key, value in state_dict.items():

    if key.startswith("token_embedding."):
        encoder_state[key] = value

    elif key.startswith("encoder_pos."):
        encoder_state[key] = value

    elif key.startswith("encoder_layers."):
        encoder_state[key] = value


missing, unexpected = a4_model.load_state_dict(
    encoder_state,
    strict=True
)

print("\nA4 encoder load:")
print("Missing   :", missing)
print("Unexpected:", unexpected)

assert len(missing) == 0
assert len(unexpected) == 0


# ============================================================
# PARAMETER VALIDATION
# ============================================================

params = sum(
    p.numel()
    for p in a4_model.parameters()
)

print(
    f"\nA4 encoder parameters: {params:,}"
)

assert params == 96_389_120, (
    f"Unexpected parameter count: {params:,}"
)

print("Parameter count: ✅")


# ============================================================
# GPU SETUP
# ============================================================

print("\nMoving A4 to GPU...")

a4_model = a4_model.to(
    DEVICE,
    dtype=torch.float16
)

a4_model.train()

ner_head = nn.Linear(
    512,
    NUM_LABELS
).to(
    DEVICE,
    dtype=torch.float16
)

criterion = nn.CrossEntropyLoss()


# ============================================================
# BENCHMARK
# ============================================================

results = []

for batch_size in [1, 2]:

    print("\n" + "-" * 70)
    print(
        f"A4 REAL BENCHMARK | "
        f"Batch={batch_size} | "
        f"Seq={SEQ_LEN}"
    )

    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    try:

        input_ids = torch.randint(
            0,
            50000,
            (batch_size, SEQ_LEN),
            device=DEVICE,
            dtype=torch.long
        )

        targets = torch.randint(
            0,
            NUM_LABELS,
            (batch_size, SEQ_LEN),
            device=DEVICE,
            dtype=torch.long
        )

        # ----------------------------------------------------
        # Warmup
        # ----------------------------------------------------

        for _ in range(2):

            hidden = a4_model.encode(
                input_ids
            )

            logits = ner_head(hidden)

            loss = criterion(
                logits.float().reshape(-1, NUM_LABELS),
                targets.reshape(-1)
            )

            loss.backward()

            a4_model.zero_grad(
                set_to_none=True
            )

            ner_head.zero_grad(
                set_to_none=True
            )

        torch.cuda.synchronize()

        # ----------------------------------------------------
        # Timed steps
        # ----------------------------------------------------

        steps = 3

        start = time.time()

        for _ in range(steps):

            hidden = a4_model.encode(
                input_ids
            )

            logits = ner_head(hidden)

            loss = criterion(
                logits.float().reshape(
                    -1,
                    NUM_LABELS
                ),
                targets.reshape(-1)
            )

            loss.backward()

            a4_model.zero_grad(
                set_to_none=True
            )

            ner_head.zero_grad(
                set_to_none=True
            )

        torch.cuda.synchronize()

        elapsed = time.time() - start

        total_tokens = (
            batch_size *
            SEQ_LEN *
            steps
        )

        tok_sec = total_tokens / elapsed

        peak_vram = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        reserved_vram = (
            torch.cuda.max_memory_reserved()
            / 1024**3
        )

        print(
            f"Time       : {elapsed:.3f}s"
        )

        print(
            f"Tokens/sec : {tok_sec:,.0f}"
        )

        print(
            f"Peak VRAM  : {peak_vram:.2f} GB"
        )

        print(
            f"Reserved   : {reserved_vram:.2f} GB"
        )

        results.append({
            "batch": batch_size,
            "tok_sec": tok_sec,
            "vram": peak_vram
        })

        del input_ids
        del targets
        del hidden
        del logits
        del loss

    except RuntimeError as e:

        if "out of memory" in str(e).lower():

            print(
                "❌ OUT OF MEMORY"
            )

            results.append({
                "batch": batch_size,
                "tok_sec": 0,
                "vram": 14.56
            })

            torch.cuda.empty_cache()
            gc.collect()

        else:
            raise


# ============================================================
# FINAL RESULTS
# ============================================================

torch.cuda.empty_cache()
gc.collect()

print("\n" + "=" * 70)
print("REAL A4 BENCHMARK RESULTS")
print("=" * 70)

for r in results:

    print(
        f"Batch {r['batch']:2d} | "
        f"{r['tok_sec']:>10,.0f} tok/s | "
        f"Peak VRAM {r['vram']:.2f} GB"
    )

print("=" * 70)

In [ ]:
# ============================================================
# CELL 9 — CORRECT A4 ENCODER + REAL GPU BENCHMARK
# ============================================================

import os
import gc
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda:0")
SEQ_LEN = 1536
NUM_LABELS = 25

A4_CKPT = (
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)

print("=" * 70)
print("A4 CORRECT RECONSTRUCTION + GPU BENCHMARK")
print("=" * 70)

assert torch.cuda.is_available()
assert os.path.exists(A4_CKPT)

print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# A4 ATTENTION — MATCH CHECKPOINT EXACTLY
# ============================================================

class A4Attention(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_heads=8
    ):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.k_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.v_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.out_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=0.0,
            is_causal=False
        )

        out = out.transpose(
            1, 2
        ).contiguous().view(
            B, T, C
        )

        return self.out_proj(out)


# ============================================================
# A4 FFN — MATCH CHECKPOINT
# ============================================================

class A4FFN(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        ffn_size=2048
    ):
        super().__init__()

        self.net = nn.ModuleDict({
            "0": nn.Linear(
                hidden_size,
                ffn_size
            ),
            "2": nn.Linear(
                ffn_size,
                hidden_size
            )
        })

    def forward(self, x):

        x = self.net["0"](x)
        x = F.gelu(x)
        x = self.net["2"](x)

        return x


# ============================================================
# A4 ENCODER LAYER
# ============================================================

class A4EncoderLayer(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_heads=8,
        ffn_size=2048
    ):
        super().__init__()

        self.attn = A4Attention(
            hidden_size,
            num_heads
        )

        self.norm1 = nn.LayerNorm(
            hidden_size
        )

        self.ffn = A4FFN(
            hidden_size,
            ffn_size
        )

        self.norm2 = nn.LayerNorm(
            hidden_size
        )

    def forward(self, x):

        x = self.norm1(
            x + self.attn(x)
        )

        x = self.norm2(
            x + self.ffn(x)
        )

        return x


# ============================================================
# A4 ENCODER
# ============================================================

class A4Encoder(nn.Module):

    def __init__(
        self,
        vocab_size=50000,
        hidden_size=512,
        num_heads=8,
        ffn_size=2048,
        num_layers=6,
        max_position=1024
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.encoder_pos = nn.Embedding(
            max_position,
            hidden_size
        )

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(
                hidden_size,
                num_heads,
                ffn_size
            )
            for _ in range(num_layers)
        ])

    def encode(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        ).unsqueeze(0)

        x = (
            self.token_embedding(input_ids)
            + self.encoder_pos(positions)
        )

        for layer in self.encoder_layers:
            x = layer(x)

        return x


# ============================================================
# LOAD CHECKPOINT
# ============================================================

print("\nLoading checkpoint...")

checkpoint = torch.load(
    A4_CKPT,
    map_location="cpu",
    weights_only=False
)

state_dict = checkpoint["model_state_dict"]

print(
    "Checkpoint step:",
    checkpoint.get("step")
)

print(
    "Checkpoint loss:",
    checkpoint.get("train_loss")
)


# ------------------------------------------------------------
# Encoder weights only
# ------------------------------------------------------------

encoder_state = {}

for key, value in state_dict.items():

    if (
        key.startswith("token_embedding.")
        or
        key.startswith("encoder_pos.")
        or
        key.startswith("encoder_layers.")
    ):
        encoder_state[key] = value


# ============================================================
# CREATE MODEL
# ============================================================

a4_model = A4Encoder()

print("\nLoading encoder weights...")

missing, unexpected = a4_model.load_state_dict(
    encoder_state,
    strict=True
)

print("Missing   :", missing)
print("Unexpected:", unexpected)

assert not missing
assert not unexpected

# ------------------------------------------------------------
# Exact parameter validation
# ------------------------------------------------------------

params = sum(
    p.numel()
    for p in a4_model.parameters()
)

print(
    f"\nA4 parameters: {params:,}"
)

assert params == 96_389_120

print("Exact A4 parameter count: ✅")


# ============================================================
# GPU
# ============================================================

print("\nMoving model to GPU...")

a4_model = a4_model.to(
    DEVICE,
    dtype=torch.float16
)

a4_model.train()

ner_head = nn.Linear(
    512,
    NUM_LABELS
).to(
    DEVICE,
    dtype=torch.float16
)

criterion = nn.CrossEntropyLoss()


# ============================================================
# REAL BENCHMARK
# ============================================================

results = []

for batch_size in [1, 2]:

    print("\n" + "-" * 70)

    print(
        f"Batch size = {batch_size}"
    )

    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    try:

        input_ids = torch.randint(
            0,
            50000,
            (batch_size, SEQ_LEN),
            device=DEVICE
        )

        targets = torch.randint(
            0,
            NUM_LABELS,
            (batch_size, SEQ_LEN),
            device=DEVICE
        )

        # ----------------------------------------------------
        # Warmup
        # ----------------------------------------------------

        for _ in range(2):

            hidden = a4_model.encode(
                input_ids
            )

            logits = ner_head(
                hidden
            )

            loss = criterion(
                logits.float().reshape(
                    -1,
                    NUM_LABELS
                ),
                targets.reshape(-1)
            )

            loss.backward()

            a4_model.zero_grad(
                set_to_none=True
            )

            ner_head.zero_grad(
                set_to_none=True
            )

        torch.cuda.synchronize()

        # ----------------------------------------------------
        # Timing
        # ----------------------------------------------------

        steps = 3

        start = time.time()

        for _ in range(steps):

            hidden = a4_model.encode(
                input_ids
            )

            logits = ner_head(
                hidden
            )

            loss = criterion(
                logits.float().reshape(
                    -1,
                    NUM_LABELS
                ),
                targets.reshape(-1)
            )

            loss.backward()

            a4_model.zero_grad(
                set_to_none=True
            )

            ner_head.zero_grad(
                set_to_none=True
            )

        torch.cuda.synchronize()

        elapsed = time.time() - start

        tokens = (
            batch_size *
            SEQ_LEN *
            steps
        )

        tok_sec = tokens / elapsed

        peak = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        reserved = (
            torch.cuda.max_memory_reserved()
            / 1024**3
        )

        print(
            f"Time       : {elapsed:.3f}s"
        )

        print(
            f"Tokens/sec : {tok_sec:,.0f}"
        )

        print(
            f"Peak VRAM  : {peak:.2f} GB"
        )

        print(
            f"Reserved   : {reserved:.2f} GB"
        )

        results.append(
            (batch_size, tok_sec, peak)
        )

        del input_ids
        del targets
        del hidden
        del logits
        del loss

    except RuntimeError as e:

        if "out of memory" in str(e).lower():

            print(
                "❌ CUDA OUT OF MEMORY"
            )

            torch.cuda.empty_cache()
            gc.collect()

        else:
            raise


# ============================================================
# SUMMARY
# ============================================================

torch.cuda.empty_cache()
gc.collect()

print("\n" + "=" * 70)
print("A4 REAL BENCHMARK RESULTS")
print("=" * 70)

for batch, tok_sec, peak in results:

    print(
        f"Batch {batch:2d} | "
        f"{tok_sec:>10,.0f} tok/s | "
        f"{peak:.2f} GB VRAM"
    )

print("=" * 70)

In [ ]:
# ============================================================
# A4 PARAMETER DIAGNOSTIC
# ============================================================

print("=" * 70)
print("A4 PARAMETER DIAGNOSTIC")
print("=" * 70)

actual = sum(
    p.numel()
    for p in a4_model.parameters()
)

print(f"Actual parameters   : {actual:,}")
print(f"Expected parameters : {96_389_120:,}")
print(f"Difference           : {actual - 96_389_120:+,}")

print("\nParameter groups:")

for name, module in a4_model.named_children():

    count = sum(
        p.numel()
        for p in module.parameters()
    )

    print(
        f"{name:25s} "
        f"{count:,}"
    )

print("\nDetailed encoder layer 0:")

for name, module in a4_model.encoder_layers[0].named_children():

    count = sum(
        p.numel()
        for p in module.parameters()
    )

    print(
        f"  {name:20s} "
        f"{count:,}"
    )

In [ ]:
# ============================================================
# A4 ENCODER — REAL NER GPU BENCHMARK
# ============================================================

import torch
import torch.nn as nn
import time
import gc

DEVICE = torch.device("cuda:0")
SEQ_LEN = 1536
NUM_LABELS = 25

# ------------------------------------------------------------
# Validate current encoder
# ------------------------------------------------------------

params = sum(
    p.numel()
    for p in a4_model.parameters()
)

print("=" * 70)
print("A4 ENCODER NER BENCHMARK")
print("=" * 70)

print(f"Encoder parameters : {params:,}")

assert params == 45_038_592

print("Encoder architecture: ✅")

# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

a4_model = a4_model.to(
    DEVICE,
    dtype=torch.float16
)

a4_model.train()

ner_head = nn.Linear(
    512,
    NUM_LABELS
).to(
    DEVICE,
    dtype=torch.float16
)

criterion = nn.CrossEntropyLoss()

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

results = []

# ============================================================
# BATCH TEST
# ============================================================

for batch_size in [1, 2, 4]:

    print("\n" + "-" * 70)
    print(
        f"Batch {batch_size} | "
        f"Sequence {SEQ_LEN}"
    )

    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    try:

        input_ids = torch.randint(
            0,
            50000,
            (batch_size, SEQ_LEN),
            device=DEVICE,
            dtype=torch.long
        )

        targets = torch.randint(
            0,
            NUM_LABELS,
            (batch_size, SEQ_LEN),
            device=DEVICE,
            dtype=torch.long
        )

        # ----------------------------------------------------
        # Warmup
        # ----------------------------------------------------

        for _ in range(2):

            hidden = a4_model.encode(
                input_ids
            )

            logits = ner_head(hidden)

            loss = criterion(
                logits.float().reshape(
                    -1,
                    NUM_LABELS
                ),
                targets.reshape(-1)
            )

            loss.backward()

            a4_model.zero_grad(
                set_to_none=True
            )

            ner_head.zero_grad(
                set_to_none=True
            )

        torch.cuda.synchronize()

        # ----------------------------------------------------
        # Timed run
        # ----------------------------------------------------

        steps = 3

        start = time.time()

        for _ in range(steps):

            hidden = a4_model.encode(
                input_ids
            )

            logits = ner_head(hidden)

            loss = criterion(
                logits.float().reshape(
                    -1,
                    NUM_LABELS
                ),
                targets.reshape(-1)
            )

            loss.backward()

            a4_model.zero_grad(
                set_to_none=True
            )

            ner_head.zero_grad(
                set_to_none=True
            )

        torch.cuda.synchronize()

        elapsed = time.time() - start

        total_tokens = (
            batch_size *
            SEQ_LEN *
            steps
        )

        tokens_sec = (
            total_tokens / elapsed
        )

        peak_vram = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        reserved_vram = (
            torch.cuda.max_memory_reserved()
            / 1024**3
        )

        print(
            f"Time       : {elapsed:.3f}s"
        )

        print(
            f"Tokens/sec : {tokens_sec:,.0f}"
        )

        print(
            f"Peak VRAM  : {peak_vram:.2f} GB"
        )

        print(
            f"Reserved   : {reserved_vram:.2f} GB"
        )

        results.append({
            "batch": batch_size,
            "tokens_sec": tokens_sec,
            "vram": peak_vram
        })

        del input_ids
        del targets
        del hidden
        del logits
        del loss

    except RuntimeError as e:

        if "out of memory" in str(e).lower():

            print("❌ CUDA OUT OF MEMORY")

            torch.cuda.empty_cache()
            gc.collect()

        else:
            raise


# ============================================================
# RESULTS
# ============================================================

torch.cuda.empty_cache()
gc.collect()

print("\n" + "=" * 70)
print("A4 ENCODER BENCHMARK RESULTS")
print("=" * 70)

for r in results:

    print(
        f"Batch {r['batch']:2d} | "
        f"{r['tokens_sec']:>10,.0f} tok/s | "
        f"Peak VRAM {r['vram']:.2f} GB"
    )

print("=" * 70)

In [ ]:
# ============================================================
# CUDA SMOKE TEST
# ============================================================

import torch

print("=" * 60)
print("CUDA SMOKE TEST")
print("=" * 60)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}:",
            torch.cuda.get_device_name(i)
        )

    x = torch.tensor(
        [1.0],
        device="cuda:0"
    )

    torch.cuda.synchronize()

    print(
        "CUDA smoke test:",
        x.item()
    )

    del x
    torch.cuda.empty_cache()

    print("\n✅ CUDA CONTEXT HEALTHY")

else:
    print("\n❌ CUDA NOT AVAILABLE")

print("=" * 60)

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

print("\nGPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

In [ ]:
# ============================================================
# FRESH GPU HEALTH TEST
# ============================================================

import torch

device = torch.device("cuda:0")

x = torch.randn(
    1024,
    1024,
    device=device
)

y = x @ x

torch.cuda.synchronize()

print("=" * 60)
print("GPU HEALTH TEST")
print("=" * 60)
print("GPU:", torch.cuda.get_device_name(0))
print("Result shape:", y.shape)
print("Result finite:", torch.isfinite(y).all().item())
print(
    "VRAM allocated:",
    f"{torch.cuda.memory_allocated()/1024**3:.3f} GB"
)

del x, y
torch.cuda.empty_cache()

print("\n✅ GPU CONTEXT HEALTHY")

In [ ]:
import torch

print("=" * 60)
print("FRESH GPU SESSION")
print("=" * 60)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)
print("Available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

# Actual CUDA allocation test
x = torch.tensor([1.0], device="cuda:0")
torch.cuda.synchronize()

print("\nCUDA allocation test:", x.item())

del x
torch.cuda.empty_cache()

print("✅ GPU CONTEXT HEALTHY")

In [ ]:
# ============================================================
# CELL 2 — RELOAD PHASE-2 DATA + TOKENIZER
# ============================================================

import os
import json
from pathlib import Path
from tokenizers import Tokenizer

print("=" * 70)
print("RELOADING PHASE-2 DATA")
print("=" * 70)

# ------------------------------------------------------------
# Kaggle input locations
# ------------------------------------------------------------

BASE = Path("/kaggle/input/datasets/aq1de3")

TOKENIZER_PATH = BASE / "tokenizer" / "tokenizer.json"

print("Tokenizer:", TOKENIZER_PATH)
print("Exists   :", TOKENIZER_PATH.exists())

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

print(
    "Tokenizer vocab:",
    tokenizer.get_vocab_size()
)

# ------------------------------------------------------------
# Find Phase-2 files
# ------------------------------------------------------------

matches = []

for root, dirs, files in os.walk("/kaggle/input"):

    for filename in files:

        if filename in {
            "fold1-text.json",
            "fold1-gold.json",
            "fold2-text.json",
            "fold2-gold.json",
            "fold3-text.json",
        }:

            matches.append(
                os.path.join(root, filename)
            )

print("\nFound files:")

for path in sorted(matches):
    print(" ", path)

assert len(matches) >= 5, (
    "Phase-2 files were not found. "
    "Send me the file paths shown in Kaggle."
)

# ------------------------------------------------------------
# Load JSON
# ------------------------------------------------------------

file_map = {
    os.path.basename(p): p
    for p in matches
}


def load_json(name):
    with open(
        file_map[name],
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)


fold1_text = load_json("fold1-text.json")
fold1_gold = load_json("fold1-gold.json")

fold2_text = load_json("fold2-text.json")
fold2_gold = load_json("fold2-gold.json")

fold3_test = load_json("fold3-text.json")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA RELOADED")
print("=" * 70)

print("Fold 1 text :", len(fold1_text))
print("Fold 1 gold :", len(fold1_gold))
print("Fold 2 text :", len(fold2_text))
print("Fold 2 gold :", len(fold2_gold))
print("Fold 3 test :", len(fold3_test))

assert len(fold1_text) == 35
assert len(fold1_gold) == 35
assert len(fold2_text) == 35
assert len(fold2_gold) == 35
assert len(fold3_test) == 35

print("\n✅ PHASE-2 DATA RESTORED")

In [ ]:
# ============================================================
# FIND PHASE-2 FILES IN THE FRESH KAGGLE SESSION
# ============================================================

import os

print("=" * 70)
print("SEARCHING /kaggle/input")
print("=" * 70)

targets = {
    "fold1-text.json",
    "fold1-gold.json",
    "fold2-text.json",
    "fold2-gold.json",
    "fold3-text.json",
    "labels.txt",
    "tokenizer.json",
}

found = []

for root, dirs, files in os.walk("/kaggle/input"):

    for filename in files:

        if filename in targets:

            path = os.path.join(
                root,
                filename
            )

            found.append(path)

            print(path)

print("\n" + "=" * 70)
print(f"TOTAL MATCHES: {len(found)}")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 3 — RELOAD OFFICIAL PHASE-2 DATA FROM HUGGING FACE
# ============================================================

import os
import json
import requests
from pathlib import Path

HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, (
    "HF_TOKEN is not available. "
    "Make sure your Kaggle secret is named HF_TOKEN."
)

headers = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

BASE_URL = (
    "https://huggingface.co/datasets/"
    "Exploration-Lab/CS-781-Capstone/resolve/main/"
)

files_to_load = {
    "fold1_text": "Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text": "Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold": "Phase_2_LNER/Train/fold2-gold.json",
    "fold3_test": "Phase_2_LNER/Test/fold3-text.json",
    "labels": "Phase_2_LNER/Train/labels.txt",
}

loaded = {}

for name, rel_path in files_to_load.items():

    url = BASE_URL + rel_path

    r = requests.get(
        url,
        headers=headers,
        timeout=120
    )

    r.raise_for_status()

    if rel_path.endswith(".txt"):
        loaded[name] = r.text
    else:
        loaded[name] = r.json()

    print(
        f"✓ {name:<12} "
        f"{len(r.content) / 1024:.1f} KB"
    )

fold1_text = loaded["fold1_text"]
fold1_gold = loaded["fold1_gold"]

fold2_text = loaded["fold2_text"]
fold2_gold = loaded["fold2_gold"]

fold3_test = loaded["fold3_test"]

labels = loaded["labels"].splitlines()

print("\n" + "=" * 70)
print("PHASE-2 DATA RELOADED")
print("=" * 70)

print("Fold-1 text :", len(fold1_text))
print("Fold-1 gold :", len(fold1_gold))
print("Fold-2 text :", len(fold2_text))
print("Fold-2 gold :", len(fold2_gold))
print("Fold-3 test :", len(fold3_test))
print("Labels      :", len(labels))

assert len(fold1_text) == 35
assert len(fold1_gold) == 35
assert len(fold2_text) == 35
assert len(fold2_gold) == 35
assert len(fold3_test) == 35
assert len(labels) == 12

print("\n✓ ALL PHASE-2 DATA CHECKS PASSED")

In [ ]:
import os

print("Environment variables containing HF:")
for k in os.environ:
    if "HF" in k.upper() or "HUGGING" in k.upper():
        print(" ", k)

print("\nKaggle input directories:")
import os
for x in os.listdir("/kaggle/input"):
    print(" ", x)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f in {
            "fold1-text.json",
            "fold1-gold.json",
            "fold2-text.json",
            "fold2-gold.json",
            "fold3-text.json",
            "labels.txt",
        }:
            print(os.path.join(root, f))

In [ ]:
# ============================================================
# CELL 3A — TEST OFFICIAL PHASE-2 DATA ACCESS
# ============================================================

import requests

url = (
    "https://huggingface.co/datasets/"
    "Exploration-Lab/CS-781-Capstone/resolve/main/"
    "Phase_2_LNER/Train/fold1-text.json"
)

r = requests.get(url, timeout=60)

print("HTTP STATUS:", r.status_code)
print("SIZE:", len(r.content), "bytes")

if r.status_code == 200:
    print("✓ OFFICIAL PHASE-2 DATA IS PUBLICLY ACCESSIBLE")
else:
    print("✗ Access failed")
    print(r.text[:500])

In [ ]:
# ============================================================
# CELL 3B — CHECK KAGGLE HF SECRET
# ============================================================

import os

HF_TOKEN = os.environ.get("HF_TOKEN")

print("HF_TOKEN available:", bool(HF_TOKEN))

if HF_TOKEN:
    print("✓ Hugging Face authentication token detected")
else:
    print("✗ HF_TOKEN still not available")
    

In [ ]:
# ============================================================
# CELL 3C — READ KAGGLE SECRET CORRECTLY
# ============================================================

import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

print("HF_TOKEN available:", bool(HF_TOKEN))

if HF_TOKEN:
    print("✓ Hugging Face token loaded from Kaggle Secrets")
else:
    print("✗ Token could not be loaded")

In [ ]:
# ============================================================
# CELL 3D — AUTHENTICATED PHASE-2 DATA LOAD
# ============================================================

import requests

BASE_URL = (
    "https://huggingface.co/datasets/"
    "Exploration-Lab/CS-781-Capstone/resolve/main/"
)

files_to_load = {
    "fold1_text": "Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text": "Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold": "Phase_2_LNER/Train/fold2-gold.json",
    "fold3_test": "Phase_2_LNER/Test/fold3-text.json",
    "labels": "Phase_2_LNER/Train/labels.txt",
}

headers = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

loaded = {}

print("=" * 70)
print("LOADING OFFICIAL PHASE-2 DATA")
print("=" * 70)

for name, rel_path in files_to_load.items():

    url = BASE_URL + rel_path

    r = requests.get(
        url,
        headers=headers,
        timeout=120
    )

    print(
        f"{name:<12} HTTP {r.status_code:<3} "
        f"{len(r.content)/1024:.1f} KB"
    )

    r.raise_for_status()

    if rel_path.endswith(".txt"):
        loaded[name] = r.text
    else:
        loaded[name] = r.json()

# Restore variables
fold1_text = loaded["fold1_text"]
fold1_gold = loaded["fold1_gold"]
fold2_text = loaded["fold2_text"]
fold2_gold = loaded["fold2_gold"]
fold3_test = loaded["fold3_test"]

labels = [
    x.strip()
    for x in loaded["labels"].splitlines()
    if x.strip()
]

print("\n" + "=" * 70)
print("VALIDATING")
print("=" * 70)

print("Fold-1 text :", len(fold1_text))
print("Fold-1 gold :", len(fold1_gold))
print("Fold-2 text :", len(fold2_text))
print("Fold-2 gold :", len(fold2_gold))
print("Fold-3 test :", len(fold3_test))
print("Labels      :", len(labels))

assert len(fold1_text) == 35
assert len(fold1_gold) == 35
assert len(fold2_text) == 35
assert len(fold2_gold) == 35
assert len(fold3_test) == 35
assert len(labels) == 12

print("\n✓ ALL 6 FILES LOADED")
print("✓ 70 TRAIN DOCUMENTS")
print("✓ 35 TEST DOCUMENTS")
print("✓ 12 ENTITY LABELS")
print("\nPHASE-2 DATA RECOVERY COMPLETE")

In [ ]:
# ============================================================
# CELL 4 — REBUILD PHASE-2 NER CHUNKS
# CPU ONLY
# ============================================================

from dataclasses import dataclass
from typing import List, Dict, Tuple

# ------------------------------------------------------------
# Configuration previously validated
# ------------------------------------------------------------

MAX_TOKENS = 1536
OVERLAP = 384
STRIDE = MAX_TOKENS - OVERLAP

LABELS = labels

label2id = {
    label: i
    for i, label in enumerate(LABELS)
}

# BIO labels:
# 0 = O
# 1..12 = B-label
# 13..24 = I-label

bio_labels = ["O"]

for label in LABELS:
    bio_labels.append("B-" + label)

for label in LABELS:
    bio_labels.append("I-" + label)

bio2id = {
    label: i
    for i, label in enumerate(bio_labels)
}


@dataclass
class NERChunk:
    doc_id: str
    chunk_id: int

    input_ids: List[int]
    offsets: List[Tuple[int, int]]

    bio_ids: List[int]

    gold_spans: List[Tuple[int, int, str]]

    char_start: int
    char_end: int


def get_doc_id(doc):
    """
    Handles the official JSON document representation
    without assuming a single field name.
    """
    for key in ["id", "doc_id", "document_id"]:
        if key in doc:
            return str(doc[key])

    raise KeyError(
        f"Could not find document ID. Keys: {list(doc.keys())}"
    )


def get_text(doc):
    """
    Extract document text.
    """
    for key in ["text", "document", "content"]:
        if key in doc:
            return doc[key]

    raise KeyError(
        f"Could not find text field. Keys: {list(doc.keys())}"
    )


def get_entities(gold):
    """
    Extract gold entities from the official gold object.
    """
    if isinstance(gold, list):
        return gold

    for key in ["entities", "annotations", "gold"]:
        if key in gold:
            return gold[key]

    raise KeyError(
        f"Could not find entity list. Keys: {list(gold.keys())}"
    )


def normalize_entity(entity):
    """
    Convert an official entity into:
        (start, end, label)

    Supports the previously observed official format.
    """

    if isinstance(entity, dict):

        start = entity.get(
            "start",
            entity.get("start_character")
        )

        end = entity.get(
            "end",
            entity.get("end_character")
        )

        label = entity.get(
            "label",
            entity.get("entity_type")
        )

        if start is None or end is None or label is None:
            raise ValueError(
                f"Cannot normalize entity: {entity}"
            )

        return int(start), int(end), str(label)

    if isinstance(entity, (list, tuple)) and len(entity) >= 3:
        return (
            int(entity[0]),
            int(entity[1]),
            str(entity[2])
        )

    raise ValueError(
        f"Unsupported entity format: {entity}"
    )


def tokenise_with_offsets(text):
    enc = tokenizer.encode(text)

    ids = enc.ids
    offsets = enc.offsets

    return ids, offsets


def build_chunks(text_obj, gold_obj):

    doc_id = get_doc_id(text_obj)
    text = get_text(text_obj)

    entities_raw = get_entities(gold_obj)

    entities = [
        normalize_entity(x)
        for x in entities_raw
    ]

    entities.sort(key=lambda x: (x[0], x[1]))

    ids, offsets = tokenise_with_offsets(text)

    chunks = []

    n_tokens = len(ids)

    start = 0
    chunk_id = 0

    while start < n_tokens:

        end = min(
            start + MAX_TOKENS,
            n_tokens
        )

        chunk_ids = ids[start:end]
        chunk_offsets = offsets[start:end]

        # Character range represented by this chunk
        valid_offsets = [
            x for x in chunk_offsets
            if x[1] > x[0]
        ]

        if valid_offsets:
            char_start = valid_offsets[0][0]
            char_end = valid_offsets[-1][1]
        else:
            char_start = 0
            char_end = 0

        # ----------------------------------------------------
        # Entities fully contained in this chunk
        # ----------------------------------------------------

        chunk_entities = []

        for ent_start, ent_end, label in entities:

            if (
                ent_start >= char_start
                and ent_end <= char_end
            ):
                chunk_entities.append(
                    (ent_start, ent_end, label)
                )

        # ----------------------------------------------------
        # BIO assignment
        # ----------------------------------------------------

        bio_ids = [
            bio2id["O"]
            for _ in chunk_ids
        ]

        for ent_start, ent_end, label in chunk_entities:

            covered = []

            for local_i, (tok_start, tok_end) in enumerate(
                chunk_offsets
            ):

                if tok_end <= tok_start:
                    continue

                # Token overlaps the entity span.
                if (
                    tok_start < ent_end
                    and tok_end > ent_start
                ):
                    covered.append(local_i)

            if not covered:
                continue

            first = covered[0]

            bio_ids[first] = bio2id[
                "B-" + label
            ]

            for idx in covered[1:]:
                bio_ids[idx] = bio2id[
                    "I-" + label
                ]

        chunks.append(
            NERChunk(
                doc_id=doc_id,
                chunk_id=chunk_id,
                input_ids=chunk_ids,
                offsets=chunk_offsets,
                bio_ids=bio_ids,
                gold_spans=chunk_entities,
                char_start=char_start,
                char_end=char_end,
            )
        )

        chunk_id += 1

        if end >= n_tokens:
            break

        start += STRIDE

    return chunks


# ============================================================
# BUILD FOLD 1
# ============================================================

fold1_chunks = []

for text_doc, gold_doc in zip(
    fold1_text,
    fold1_gold
):
    fold1_chunks.extend(
        build_chunks(
            text_doc,
            gold_doc
        )
    )


# ============================================================
# BUILD FOLD 2
# ============================================================

fold2_chunks = []

for text_doc, gold_doc in zip(
    fold2_text,
    fold2_gold
):
    fold2_chunks.extend(
        build_chunks(
            text_doc,
            gold_doc
        )
    )


train_chunks = fold1_chunks + fold2_chunks


# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("CHUNKING COMPLETE")
print("=" * 70)

print("Fold 1 chunks :", len(fold1_chunks))
print("Fold 2 chunks :", len(fold2_chunks))
print("Total chunks  :", len(train_chunks))

lengths = [
    len(x.input_ids)
    for x in train_chunks
]

print("\nToken statistics:")
print("Min    :", min(lengths))
print("Mean   :", sum(lengths) / len(lengths))
print("Max    :", max(lengths))

# ------------------------------------------------------------
# Entity containment
# ------------------------------------------------------------

total_gold = 0
contained_gold = 0

for text_doc, gold_doc in zip(
    fold1_text + fold2_text,
    fold1_gold + fold2_gold
):

    entities = get_entities(gold_doc)

    total_gold += len(entities)

    doc_chunks = [
        x for x in train_chunks
        if x.doc_id == get_doc_id(text_doc)
    ]

    for ent in entities:

        s, e, label = normalize_entity(ent)

        found = any(
            s >= c.char_start
            and e <= c.char_end
            for c in doc_chunks
        )

        contained_gold += int(found)


print("\nGold entities:")
print("Total     :", total_gold)
print("Contained :", contained_gold)

assert len(fold1_chunks) == 203
assert len(fold2_chunks) == 203
assert len(train_chunks) == 406

assert total_gold == 8393
assert contained_gold == 8393

print("\n✓ 406 CHUNKS VERIFIED")
print("✓ 8,393 ENTITIES VERIFIED")
print("✓ ALL GOLD SPANS CONTAINED")
print("\nCPU CHUNK PIPELINE READY")

In [ ]:
# ============================================================
# DIAGNOSTIC — INSPECT OFFICIAL PHASE-2 JSON STRUCTURE
# ============================================================

print("fold1_text type:", type(fold1_text))
print("fold1_gold type:", type(fold1_gold))

if isinstance(fold1_text, dict):
    print("\nfold1_text keys sample:")
    print(list(fold1_text.keys())[:3])

    first_key = next(iter(fold1_text))
    print("\nFirst key:", repr(first_key))
    print("First value type:", type(fold1_text[first_key]))
    print("First value preview:", repr(fold1_text[first_key])[:300])

elif isinstance(fold1_text, list):
    print("\nFirst item type:", type(fold1_text[0]))
    print("First item:", repr(fold1_text[0])[:500])

print("\n" + "=" * 70)

if isinstance(fold1_gold, dict):
    print("fold1_gold keys sample:")
    print(list(fold1_gold.keys())[:3])

    first_key = next(iter(fold1_gold))
    print("\nFirst gold key:", repr(first_key))
    print("First gold value type:", type(fold1_gold[first_key]))
    print("First gold value preview:", repr(fold1_gold[first_key])[:500])

elif isinstance(fold1_gold, list):
    print("First gold item type:", type(fold1_gold[0]))
    print("First gold item:", repr(fold1_gold[0])[:500])

In [ ]:
# ============================================================
# CELL 4 — CORRECT PHASE-2 CHUNK BUILDER
# Official JSON structure:
#   text: {doc_id: text}
#   gold: {doc_id: [[start,end,label,text], ...]}
# ============================================================

from dataclasses import dataclass
from typing import List, Tuple

MAX_TOKENS = 1536
OVERLAP = 384
STRIDE = MAX_TOKENS - OVERLAP

# ------------------------------------------------------------
# Label mappings
# ------------------------------------------------------------

LABELS = [x.strip() for x in labels if x.strip()]

label2id = {
    label: i
    for i, label in enumerate(LABELS)
}

bio_labels = ["O"]

for label in LABELS:
    bio_labels.append("B-" + label)

for label in LABELS:
    bio_labels.append("I-" + label)

bio2id = {
    label: i
    for i, label in enumerate(bio_labels)
}


@dataclass
class NERChunk:
    doc_id: str
    chunk_id: int

    input_ids: List[int]
    offsets: List[Tuple[int, int]]

    bio_ids: List[int]

    gold_spans: List[Tuple[int, int, str]]

    char_start: int
    char_end: int


# ------------------------------------------------------------
# Convert official gold format
# [start, end, label, entity_text]
# ------------------------------------------------------------

def normalize_gold(gold_list):

    result = []

    for item in gold_list:

        start = int(item[0])
        end = int(item[1])
        label = str(item[2])

        result.append(
            (start, end, label)
        )

    result.sort(
        key=lambda x: (x[0], x[1])
    )

    return result


# ------------------------------------------------------------
# Tokenize one document
# ------------------------------------------------------------

def tokenize_document(text):

    enc = tokenizer.encode(text)

    return (
        enc.ids,
        enc.offsets
    )


# ------------------------------------------------------------
# Build chunks for one document
# ------------------------------------------------------------

def build_document_chunks(
    doc_id,
    text,
    gold_list
):

    token_ids, token_offsets = tokenize_document(text)

    gold_spans = normalize_gold(gold_list)

    n_tokens = len(token_ids)

    chunks = []

    chunk_id = 0
    token_start = 0

    while token_start < n_tokens:

        token_end = min(
            token_start + MAX_TOKENS,
            n_tokens
        )

        ids = token_ids[
            token_start:token_end
        ]

        offsets = token_offsets[
            token_start:token_end
        ]

        # ----------------------------------------------------
        # Valid token offsets
        # ----------------------------------------------------

        valid_offsets = [
            (i, s, e)
            for i, (s, e) in enumerate(offsets)
            if e > s
        ]

        if valid_offsets:

            char_start = min(
                x[1] for x in valid_offsets
            )

            char_end = max(
                x[2] for x in valid_offsets
            )

        else:

            char_start = 0
            char_end = 0

        # ----------------------------------------------------
        # Find entities fully contained in this chunk
        # ----------------------------------------------------

        chunk_gold = []

        for ent_start, ent_end, label in gold_spans:

            if (
                ent_start >= char_start
                and ent_end <= char_end
            ):
                chunk_gold.append(
                    (
                        ent_start,
                        ent_end,
                        label
                    )
                )

        # ----------------------------------------------------
        # BIO labels
        # ----------------------------------------------------

        bio_ids = [
            bio2id["O"]
            for _ in ids
        ]

        for (
            ent_start,
            ent_end,
            label
        ) in chunk_gold:

            covered_tokens = []

            for local_idx, (
                tok_start,
                tok_end
            ) in enumerate(offsets):

                if tok_end <= tok_start:
                    continue

                # Character-span overlap
                if (
                    tok_start < ent_end
                    and tok_end > ent_start
                ):
                    covered_tokens.append(
                        local_idx
                    )

            if not covered_tokens:
                continue

            first = covered_tokens[0]

            bio_ids[first] = bio2id[
                "B-" + label
            ]

            for idx in covered_tokens[1:]:

                bio_ids[idx] = bio2id[
                    "I-" + label
                ]

        chunks.append(
            NERChunk(
                doc_id=str(doc_id),
                chunk_id=chunk_id,

                input_ids=list(ids),
                offsets=list(offsets),

                bio_ids=bio_ids,

                gold_spans=chunk_gold,

                char_start=char_start,
                char_end=char_end,
            )
        )

        chunk_id += 1

        if token_end >= n_tokens:
            break

        token_start += STRIDE

    return chunks


# ============================================================
# BUILD FOLD 1
# ============================================================

fold1_chunks = []

for doc_id, text in fold1_text.items():

    fold1_chunks.extend(
        build_document_chunks(
            doc_id,
            text,
            fold1_gold[doc_id]
        )
    )


# ============================================================
# BUILD FOLD 2
# ============================================================

fold2_chunks = []

for doc_id, text in fold2_text.items():

    fold2_chunks.extend(
        build_document_chunks(
            doc_id,
            text,
            fold2_gold[doc_id]
        )
    )


train_chunks = (
    fold1_chunks +
    fold2_chunks
)


# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("PHASE-2 CHUNKING COMPLETE")
print("=" * 70)

print(
    f"Fold 1 chunks : {len(fold1_chunks)}"
)

print(
    f"Fold 2 chunks : {len(fold2_chunks)}"
)

print(
    f"Total chunks  : {len(train_chunks)}"
)


# ------------------------------------------------------------
# Token statistics
# ------------------------------------------------------------

lengths = [
    len(c.input_ids)
    for c in train_chunks
]

print("\nTOKEN STATISTICS")
print("-" * 40)

print(
    f"Min    : {min(lengths)}"
)

print(
    f"Mean   : {sum(lengths) / len(lengths):.1f}"
)

print(
    f"Median : {sorted(lengths)[len(lengths)//2]}"
)

print(
    f"Max    : {max(lengths)}"
)


# ============================================================
# GOLD ENTITY VALIDATION
# ============================================================

total_gold = 0
contained_gold = 0

for gold_dict in [
    fold1_gold,
    fold2_gold
]:

    for doc_id, gold_list in gold_dict.items():

        entities = normalize_gold(
            gold_list
        )

        total_gold += len(entities)

        doc_chunks = [
            c for c in train_chunks
            if c.doc_id == str(doc_id)
        ]

        for (
            ent_start,
            ent_end,
            label
        ) in entities:

            found = any(
                ent_start >= c.char_start
                and ent_end <= c.char_end
                for c in doc_chunks
            )

            if found:
                contained_gold += 1


print("\nGOLD SPAN VALIDATION")
print("-" * 40)

print(
    f"Total gold entities : {total_gold}"
)

print(
    f"Contained in chunks : {contained_gold}"
)

print(
    f"Missing             : "
    f"{total_gold - contained_gold}"
)


# ============================================================
# ASSERTIONS
# ============================================================

assert len(fold1_chunks) == 203
assert len(fold2_chunks) == 203
assert len(train_chunks) == 406

assert total_gold == 8393
assert contained_gold == 8393

assert (
    total_gold - contained_gold
) == 0


print("\n" + "=" * 70)
print("✓ 406 CHUNKS VERIFIED")
print("✓ 8,393 GOLD ENTITIES VERIFIED")
print("✓ 0 MISSING GOLD SPANS")
print("✓ CHUNK PIPELINE READY")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 5 — A4 CHECKPOINT STRUCTURE
# CPU ONLY
# ============================================================

import torch
from pathlib import Path

A4_PATH = Path(
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)

assert A4_PATH.exists(), A4_PATH

ckpt = torch.load(
    A4_PATH,
    map_location="cpu",
    weights_only=False
)

print("=" * 70)
print("A4 CHECKPOINT")
print("=" * 70)

print("Path:", A4_PATH)
print("File size:",
      f"{A4_PATH.stat().st_size / 1024**2:.2f} MB")

print("\nCheckpoint type:", type(ckpt))

if isinstance(ckpt, dict):

    print("\nTop-level keys:")
    for k in ckpt.keys():
        print(" ", k)

    state = ckpt.get(
        "model_state_dict",
        ckpt.get("state_dict", ckpt)
    )

    print("\nState-dict tensors:", len(state))

    total_params = sum(
        v.numel()
        for v in state.values()
        if torch.is_tensor(v)
    )

    print(
        "Total checkpoint parameters:",
        f"{total_params:,}"
    )

    print("\nFirst 30 parameter keys:")
    for k in list(state.keys())[:30]:
        print(
            f"  {k:<60}",
            tuple(state[k].shape)
        )

else:
    raise TypeError(
        f"Unexpected checkpoint type: {type(ckpt)}"
    )

In [ ]:
# ============================================================
# CELL 6 — CORRECT A4 ENCODER RECONSTRUCTION
# EXACT CHECKPOINT KEY STRUCTURE
# CPU ONLY
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# A4 configuration
# ------------------------------------------------------------

VOCAB_SIZE = 50000
HIDDEN_SIZE = 512
NUM_HEADS = 8
NUM_ENCODER_LAYERS = 6
FFN_SIZE = 2048
MAX_ENCODER_LEN = 1024


# ------------------------------------------------------------
# Attention
# ------------------------------------------------------------

class A4SelfAttention(nn.Module):

    def __init__(self):
        super().__init__()

        self.q_proj = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE
        )

        self.k_proj = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE
        )

        self.v_proj = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE
        )

        self.out_proj = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE
        )

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # [B,T,512] -> [B,8,T,64]
        q = q.view(
            B, T,
            NUM_HEADS,
            HIDDEN_SIZE // NUM_HEADS
        ).transpose(1, 2)

        k = k.view(
            B, T,
            NUM_HEADS,
            HIDDEN_SIZE // NUM_HEADS
        ).transpose(1, 2)

        v = v.view(
            B, T,
            NUM_HEADS,
            HIDDEN_SIZE // NUM_HEADS
        ).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=0.0,
            is_causal=False
        )

        y = y.transpose(
            1, 2
        ).contiguous().view(
            B, T, C
        )

        return self.out_proj(y)


# ------------------------------------------------------------
# FFN
# IMPORTANT:
# checkpoint expects:
#   ffn.net.0.*
#   ffn.net.2.*
# ------------------------------------------------------------

class A4FFN(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                HIDDEN_SIZE,
                FFN_SIZE
            ),

            nn.GELU(),

            nn.Linear(
                FFN_SIZE,
                HIDDEN_SIZE
            )
        )

    def forward(self, x):
        return self.net(x)


# ------------------------------------------------------------
# Encoder layer
# ------------------------------------------------------------

class A4EncoderLayer(nn.Module):

    def __init__(self):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            HIDDEN_SIZE
        )

        self.attn = A4SelfAttention()

        self.norm2 = nn.LayerNorm(
            HIDDEN_SIZE
        )

        self.ffn = A4FFN()

    def forward(self, x):

        x = x + self.attn(
            self.norm1(x)
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ------------------------------------------------------------
# Encoder
# IMPORTANT:
# checkpoint expects encoder_pos.weight
# therefore nn.Embedding, NOT nn.Parameter
# ------------------------------------------------------------

class A4Encoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE
        )

        self.encoder_pos = nn.Embedding(
            MAX_ENCODER_LEN,
            HIDDEN_SIZE
        )

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer()
            for _ in range(NUM_ENCODER_LAYERS)
        ])

    def forward(self, input_ids):

        B, T = input_ids.shape

        if T > MAX_ENCODER_LEN:
            raise ValueError(
                f"T={T} exceeds {MAX_ENCODER_LEN}"
            )

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = self.token_embedding(
            input_ids
        )

        x = x + self.encoder_pos(
            positions
        ).unsqueeze(0)

        for layer in self.encoder_layers:
            x = layer(x)

        return x


# ============================================================
# EXTRACT ENCODER WEIGHTS
# ============================================================

state = ckpt["model_state_dict"]

encoder_state = {
    key: value
    for key, value in state.items()
    if (
        key.startswith("token_embedding.")
        or key.startswith("encoder_pos.")
        or key.startswith("encoder_layers.")
    )
}


# ============================================================
# BUILD + STRICT LOAD
# ============================================================

encoder = A4Encoder()

result = encoder.load_state_dict(
    encoder_state,
    strict=True
)

print("=" * 70)
print("A4 ENCODER STRICT LOAD")
print("=" * 70)

print("Missing keys   :", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)

assert len(result.missing_keys) == 0
assert len(result.unexpected_keys) == 0


# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

encoder_params = sum(
    p.numel()
    for p in encoder.parameters()
)

print(
    "Encoder parameters:",
    f"{encoder_params:,}"
)


# ============================================================
# CPU FORWARD SANITY TEST
# ============================================================

encoder.eval()

test_ids = torch.randint(
    0,
    VOCAB_SIZE,
    (1, 64),
    dtype=torch.long
)

with torch.inference_mode():

    output = encoder(
        test_ids
    )

print(
    "Input shape :",
    tuple(test_ids.shape)
)

print(
    "Output shape:",
    tuple(output.shape)
)

print(
    "Finite      :",
    bool(torch.isfinite(output).all())
)

assert output.shape == (
    1,
    64,
    HIDDEN_SIZE
)

assert torch.isfinite(output).all()

print("\n✓ STRICT LOAD SUCCESS")
print("✓ CPU FORWARD SUCCESS")
print("✓ OUTPUT FINITE")

In [ ]:
# ============================================================
# CELL 7 — A4 GPU SMOKE TEST
# ============================================================

import time
import torch

assert torch.cuda.is_available()

device = torch.device("cuda:0")

print("=" * 70)
print("A4 GPU SMOKE TEST")
print("=" * 70)

print("GPU:", torch.cuda.get_device_name(0))

# Move the already validated encoder
t0 = time.time()

encoder_gpu = encoder.to(device)

torch.cuda.synchronize()

print(
    f"Model transfer: {time.time() - t0:.2f} sec"
)

# Small safe test
test_ids_gpu = torch.randint(
    0,
    VOCAB_SIZE,
    (1, 64),
    dtype=torch.long,
    device=device
)

torch.cuda.synchronize()

t0 = time.time()

with torch.inference_mode():

    gpu_output = encoder_gpu(
        test_ids_gpu
    )

torch.cuda.synchronize()

elapsed = time.time() - t0

print(
    f"Forward time: {elapsed:.4f} sec"
)

print(
    "Output shape:",
    tuple(gpu_output.shape)
)

print(
    "Finite:",
    bool(torch.isfinite(gpu_output).all())
)

print(
    "VRAM allocated:",
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    "VRAM reserved:",
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

assert gpu_output.shape == (
    1,
    64,
    HIDDEN_SIZE
)

assert torch.isfinite(
    gpu_output
).all()

print("\n✓ A4 GPU FORWARD SUCCESS")
print("✓ CUDA CONTEXT HEALTHY")

In [ ]:
# ============================================================
# CELL 8 — PRE-TRAINING NER VALIDATION
# CPU ONLY
# ============================================================

print("=" * 70)
print("PHASE-2 PRE-TRAINING VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Every chunk has aligned lengths
# ------------------------------------------------------------

bad_lengths = []

for i, c in enumerate(train_chunks):

    if not (
        len(c.input_ids)
        == len(c.offsets)
        == len(c.bio_ids)
    ):
        bad_lengths.append(i)

print(
    "Chunk length mismatches:",
    len(bad_lengths)
)

assert len(bad_lengths) == 0


# ------------------------------------------------------------
# 2. BIO IDs are valid
# ------------------------------------------------------------

max_bio_id = len(bio_labels) - 1

invalid_bio = []

for i, c in enumerate(train_chunks):

    for x in c.bio_ids:

        if x < 0 or x > max_bio_id:
            invalid_bio.append(
                (i, x)
            )

print(
    "Invalid BIO IDs:",
    len(invalid_bio)
)

assert len(invalid_bio) == 0


# ------------------------------------------------------------
# 3. Count BIO labels
# ------------------------------------------------------------

bio_counts = {
    label: 0
    for label in bio_labels
}

for c in train_chunks:

    for x in c.bio_ids:

        bio_counts[
            bio_labels[x]
        ] += 1


print("\nBIO distribution:")

for label, count in bio_counts.items():

    if count > 0:
        print(
            f"{label:<15} {count:,}"
        )


# ------------------------------------------------------------
# 4. Verify every chunk's gold span is represented
# ------------------------------------------------------------

missing_chunk_spans = []

for c in train_chunks:

    for (
        ent_start,
        ent_end,
        label
    ) in c.gold_spans:

        covered = False

        for (
            tok_start,
            tok_end
        ) in c.offsets:

            if tok_end <= tok_start:
                continue

            if (
                tok_start < ent_end
                and tok_end > ent_start
            ):
                covered = True
                break

        if not covered:

            missing_chunk_spans.append(
                (
                    c.doc_id,
                    ent_start,
                    ent_end,
                    label
                )
            )


print(
    "\nGold spans without token coverage:",
    len(missing_chunk_spans)
)

assert len(missing_chunk_spans) == 0


# ------------------------------------------------------------
# 5. Check illegal BIO transitions
# ------------------------------------------------------------

illegal_transitions = []

for ci, c in enumerate(train_chunks):

    previous = "O"

    for token_idx, bio_id in enumerate(
        c.bio_ids
    ):

        current = bio_labels[bio_id]

        if current.startswith("I-"):

            current_type = current[2:]

            if previous not in (
                "B-" + current_type,
                "I-" + current_type
            ):

                illegal_transitions.append(
                    (
                        ci,
                        token_idx,
                        previous,
                        current
                    )
                )

        previous = current


print(
    "Illegal BIO transitions:",
    len(illegal_transitions)
)

assert len(illegal_transitions) == 0


print("\n" + "=" * 70)
print("✓ ALL PRE-TRAINING VALIDATIONS PASSED")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 9 — A4 + BIO NER MODEL
# GPU SMOKE TEST ONLY
# ============================================================

import torch
import torch.nn as nn

NUM_BIO_CLASSES = len(bio_labels)   # 25
PAD_LABEL_ID = -100


class A4NER(nn.Module):

    def __init__(
        self,
        encoder,
        num_labels,
        dropout=0.10
    ):
        super().__init__()

        self.encoder = encoder

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Linear(
            HIDDEN_SIZE,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask=None,
        labels=None
    ):

        hidden = self.encoder(
            input_ids
        )

        hidden = self.dropout(
            hidden
        )

        logits = self.classifier(
            hidden
        )

        loss = None

        if labels is not None:

            loss_fn = nn.CrossEntropyLoss(
                ignore_index=PAD_LABEL_ID
            )

            loss = loss_fn(
                logits.reshape(
                    -1,
                    logits.size(-1)
                ),
                labels.reshape(-1)
            )

        return logits, loss


# ============================================================
# CREATE MODEL
# ============================================================

model = A4NER(
    encoder=encoder_gpu,
    num_labels=NUM_BIO_CLASSES,
    dropout=0.10
)

model = model.to(device)


print("=" * 70)
print("A4 NER MODEL")
print("=" * 70)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Total parameters    : {total:,}"
)

print(
    f"Trainable parameters: {trainable:,}"
)

print(
    f"BIO classes         : {NUM_BIO_CLASSES}"
)


# ============================================================
# SMALL BATCH SMOKE TEST
# ============================================================

model.train()

BATCH_SIZE = 2
SEQ_LEN = 256

dummy_ids = torch.randint(
    0,
    VOCAB_SIZE,
    (BATCH_SIZE, SEQ_LEN),
    dtype=torch.long,
    device=device
)

dummy_labels = torch.randint(
    0,
    NUM_BIO_CLASSES,
    (BATCH_SIZE, SEQ_LEN),
    dtype=torch.long,
    device=device
)

# Ignore a small tail to verify masking
dummy_labels[:, -16:] = PAD_LABEL_ID


torch.cuda.synchronize()

logits, loss = model(
    dummy_ids,
    labels=dummy_labels
)

print(
    "\nLogits shape:",
    tuple(logits.shape)
)

print(
    "Loss:",
    float(loss)
)

assert logits.shape == (
    BATCH_SIZE,
    SEQ_LEN,
    NUM_BIO_CLASSES
)

assert torch.isfinite(loss)


# ============================================================
# BACKWARD TEST
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

optimizer.zero_grad(
    set_to_none=True
)

loss.backward()

# Check gradients
bad_grads = 0

for name, p in model.named_parameters():

    if p.grad is not None:

        if not torch.isfinite(
            p.grad
        ).all():

            bad_grads += 1

print(
    "Bad gradients:",
    bad_grads
)

assert bad_grads == 0

optimizer.step()

torch.cuda.synchronize()

print(
    "VRAM allocated:",
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print("\n" + "=" * 70)
print("✓ NER FORWARD PASS")
print("✓ LOSS COMPUTATION")
print("✓ BACKWARD PASS")
print("✓ OPTIMIZER STEP")
print("✓ NO NaN/INF GRADIENTS")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 10 — REAL PHASE-2 DATALOADER
# Dynamic padding + masks
# CPU preparation / GPU transfer happens per batch
# ============================================================

import random
import torch
from torch.utils.data import Dataset, DataLoader


# ============================================================
# DATASET
# ============================================================

class NERChunkDataset(Dataset):

    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        c = self.chunks[idx]

        return {
            "input_ids": c.input_ids,
            "bio_ids": c.bio_ids,
            "offsets": c.offsets,
            "doc_id": c.doc_id,
            "chunk_id": c.chunk_id,
        }


# ============================================================
# COLLATOR
# ============================================================

def ner_collate(batch):

    batch_size = len(batch)

    max_len = max(
        len(x["input_ids"])
        for x in batch
    )

    # --------------------------------------------------------
    # Allocate padded tensors
    # --------------------------------------------------------

    input_ids = torch.full(
        (batch_size, max_len),
        0,                         # tokenizer PAD = 0
        dtype=torch.long
    )

    labels_tensor = torch.full(
        (batch_size, max_len),
        -100,                      # ignored by CE
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (batch_size, max_len),
        dtype=torch.bool
    )

    # Keep offsets on CPU for later span decoding
    offsets = []
    doc_ids = []
    chunk_ids = []

    # --------------------------------------------------------
    # Fill batch
    # --------------------------------------------------------

    for i, item in enumerate(batch):

        n = len(item["input_ids"])

        input_ids[
            i,
            :n
        ] = torch.tensor(
            item["input_ids"],
            dtype=torch.long
        )

        labels_tensor[
            i,
            :n
        ] = torch.tensor(
            item["bio_ids"],
            dtype=torch.long
        )

        attention_mask[
            i,
            :n
        ] = True

        offsets.append(
            item["offsets"]
        )

        doc_ids.append(
            item["doc_id"]
        )

        chunk_ids.append(
            item["chunk_id"]
        )

    return {
        "input_ids": input_ids,
        "labels": labels_tensor,
        "attention_mask": attention_mask,
        "offsets": offsets,
        "doc_ids": doc_ids,
        "chunk_ids": chunk_ids,
    }


# ============================================================
# CREATE DATASET
# ============================================================

train_dataset = NERChunkDataset(
    train_chunks
)


# ============================================================
# STARTER BATCH SIZE
# ============================================================

BATCH_SIZE = 4


train_loader = DataLoader(
    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    collate_fn=ner_collate,

    num_workers=2,

    pin_memory=True,

    persistent_workers=True,

    drop_last=False
)


# ============================================================
# INSPECT ONE REAL BATCH
# ============================================================

batch = next(
    iter(train_loader)
)

print("=" * 70)
print("REAL DATA BATCH")
print("=" * 70)

print(
    "input_ids:",
    tuple(batch["input_ids"].shape)
)

print(
    "labels:",
    tuple(batch["labels"].shape)
)

print(
    "attention_mask:",
    tuple(batch["attention_mask"].shape)
)

print(
    "max sequence length:",
    batch["input_ids"].shape[1]
)

print(
    "valid tokens:",
    int(batch["attention_mask"].sum())
)

print(
    "padded tokens:",
    int(
        (~batch["attention_mask"]).sum()
    )
)

print(
    "documents:",
    batch["doc_ids"]
)

print(
    "chunks:",
    batch["chunk_ids"]
)


# ============================================================
# VALIDATION
# ============================================================

assert (
    batch["input_ids"].shape
    == batch["labels"].shape
)

assert (
    batch["input_ids"].shape
    == batch["attention_mask"].shape
)

# Padding positions must have -100 labels
padding_labels = batch["labels"][
    ~batch["attention_mask"]
]

assert torch.all(
    padding_labels == -100
)

# Real positions must not have -100
valid_labels = batch["labels"][
    batch["attention_mask"]
]

assert torch.all(
    valid_labels >= 0
)

print("\n✓ DYNAMIC PADDING CORRECT")
print("✓ ATTENTION MASK CORRECT")
print("✓ LABEL MASKING CORRECT")
print("✓ REAL PHASE-2 DATALOADER READY")

In [ ]:
# ============================================================
# CELL 11 — REAL-DATA FORWARD/BACKWARD SANITY TEST
# Only 3 batches
# ============================================================

import time
import torch

model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

print("=" * 70)
print("REAL-DATA TRAINING SANITY TEST")
print("=" * 70)

running_loss = 0.0

t0 = time.time()

for step, batch in enumerate(train_loader):

    if step >= 3:
        break

    input_ids = batch["input_ids"].to(
        device,
        non_blocking=True
    )

    labels_batch = batch["labels"].to(
        device,
        non_blocking=True
    )

    attention_mask = batch[
        "attention_mask"
    ].to(
        device,
        non_blocking=True
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    logits, loss = model(
        input_ids,
        attention_mask=attention_mask,
        labels=labels_batch
    )

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Non-finite loss at step {step}: {loss}"
        )

    loss.backward()

    # Gradient sanity
    bad_grad = False

    for p in model.parameters():

        if p.grad is not None:

            if not torch.isfinite(
                p.grad
            ).all():

                bad_grad = True
                break

    assert not bad_grad

    optimizer.step()

    running_loss += loss.item()

    print(
        f"Step {step + 1}/3 | "
        f"Loss {loss.item():.5f} | "
        f"Valid tokens "
        f"{int(attention_mask.sum())}"
    )


torch.cuda.synchronize()

elapsed = time.time() - t0

avg_loss = running_loss / 3

print("\n" + "=" * 70)
print(
    f"Average loss : {avg_loss:.5f}"
)
print(
    f"Time         : {elapsed:.2f} sec"
)
print(
    f"Speed        : {3/elapsed:.2f} batches/sec"
)
print(
    f"VRAM         : "
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)
print("=" * 70)

print("\n✓ REAL DATA FORWARD")
print("✓ REAL DATA BACKWARD")
print("✓ OPTIMIZER STEP")
print("✓ FINITE LOSS")
print("✓ FINITE GRADIENTS")

In [ ]:
# ============================================================
# CELL 11A — A4 CONTEXT-COMPATIBLE CHUNKING
# CPU ONLY
# ============================================================

A4_MAX_TOKENS = 1024

# Keep the same overlap proportion as the existing pipeline:
A4_OVERLAP = 256
A4_STRIDE = A4_MAX_TOKENS - A4_OVERLAP

print("=" * 70)
print("A4 CONTEXT CONSTRAINT")
print("=" * 70)

print("Original chunk length :", MAX_TOKENS)
print("A4 encoder maximum    :", A4_MAX_TOKENS)
print("A4 overlap            :", A4_OVERLAP)
print("A4 stride             :", A4_STRIDE)

In [ ]:
# ============================================================
# CELL 11B — A4-COMPATIBLE CHUNKS
# CPU ONLY
# ============================================================

A4_MAX_TOKENS = 1024
A4_OVERLAP = 256
A4_STRIDE = 768


def build_a4_chunks(doc_id, text, gold_list):

    token_ids, token_offsets = tokenize_document(text)

    gold_spans = normalize_gold(gold_list)

    n_tokens = len(token_ids)

    chunks = []
    chunk_id = 0
    token_start = 0

    while token_start < n_tokens:

        token_end = min(
            token_start + A4_MAX_TOKENS,
            n_tokens
        )

        ids = token_ids[token_start:token_end]
        offsets = token_offsets[token_start:token_end]

        valid_offsets = [
            (s, e)
            for s, e in offsets
            if e > s
        ]

        if valid_offsets:
            char_start = min(
                s for s, e in valid_offsets
            )
            char_end = max(
                e for s, e in valid_offsets
            )
        else:
            char_start = 0
            char_end = 0

        chunk_gold = []

        for ent_start, ent_end, label in gold_spans:

            if (
                ent_start >= char_start
                and ent_end <= char_end
            ):
                chunk_gold.append(
                    (ent_start, ent_end, label)
                )

        bio_ids = [
            bio2id["O"]
            for _ in ids
        ]

        for ent_start, ent_end, label in chunk_gold:

            covered = []

            for local_idx, (
                tok_start,
                tok_end
            ) in enumerate(offsets):

                if tok_end <= tok_start:
                    continue

                if (
                    tok_start < ent_end
                    and tok_end > ent_start
                ):
                    covered.append(local_idx)

            if not covered:
                continue

            bio_ids[covered[0]] = bio2id[
                "B-" + label
            ]

            for idx in covered[1:]:
                bio_ids[idx] = bio2id[
                    "I-" + label
                ]

        chunks.append(
            NERChunk(
                doc_id=str(doc_id),
                chunk_id=chunk_id,
                input_ids=list(ids),
                offsets=list(offsets),
                bio_ids=bio_ids,
                gold_spans=chunk_gold,
                char_start=char_start,
                char_end=char_end,
            )
        )

        chunk_id += 1

        if token_end >= n_tokens:
            break

        token_start += A4_STRIDE

    return chunks


# ============================================================
# BUILD A4 FOLD 1
# ============================================================

a4_fold1_chunks = []

for doc_id, text in fold1_text.items():

    a4_fold1_chunks.extend(
        build_a4_chunks(
            doc_id,
            text,
            fold1_gold[doc_id]
        )
    )


# ============================================================
# BUILD A4 FOLD 2
# ============================================================

a4_fold2_chunks = []

for doc_id, text in fold2_text.items():

    a4_fold2_chunks.extend(
        build_a4_chunks(
            doc_id,
            text,
            fold2_gold[doc_id]
        )
    )


a4_train_chunks = (
    a4_fold1_chunks +
    a4_fold2_chunks
)


# ============================================================
# VALIDATE
# ============================================================

lengths = [
    len(c.input_ids)
    for c in a4_train_chunks
]

total_gold = 0
contained_gold = 0

for gold_dict in (
    fold1_gold,
    fold2_gold
):

    for doc_id, gold_list in gold_dict.items():

        entities = normalize_gold(
            gold_list
        )

        total_gold += len(entities)

        doc_chunks = [
            c for c in a4_train_chunks
            if c.doc_id == str(doc_id)
        ]

        for (
            ent_start,
            ent_end,
            label
        ) in entities:

            if any(
                ent_start >= c.char_start
                and ent_end <= c.char_end
                for c in doc_chunks
            ):
                contained_gold += 1


print("=" * 70)
print("A4 CHUNKING COMPLETE")
print("=" * 70)

print(
    "Fold 1 chunks:",
    len(a4_fold1_chunks)
)

print(
    "Fold 2 chunks:",
    len(a4_fold2_chunks)
)

print(
    "Total A4 chunks:",
    len(a4_train_chunks)
)

print("\nToken statistics:")
print(
    "Min:",
    min(lengths)
)

print(
    "Mean:",
    f"{sum(lengths)/len(lengths):.1f}"
)

print(
    "Max:",
    max(lengths)
)

print("\nGold validation:")
print(
    "Total:",
    total_gold
)

print(
    "Contained:",
    contained_gold
)

print(
    "Missing:",
    total_gold - contained_gold
)

assert max(lengths) <= A4_MAX_TOKENS
assert total_gold == 8393
assert contained_gold == 8393

print("\n✓ ALL CHUNKS <= 1024 TOKENS")
print("✓ ALL 8,393 GOLD ENTITIES CONTAINED")
print("✓ A4 TRAINING DATA READY")

In [ ]:
# ============================================================
# CELL 12 — A4 REAL-DATA DATALOADER
# ============================================================

from torch.utils.data import Dataset, DataLoader


class A4NERDataset(Dataset):

    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        c = self.chunks[idx]

        return {
            "input_ids": c.input_ids,
            "labels": c.bio_ids,
            "offsets": c.offsets,
            "doc_id": c.doc_id,
            "chunk_id": c.chunk_id,
        }


def a4_collate(batch):

    batch_size = len(batch)

    max_len = max(
        len(x["input_ids"])
        for x in batch
    )

    input_ids = torch.zeros(
        (batch_size, max_len),
        dtype=torch.long
    )

    labels_batch = torch.full(
        (batch_size, max_len),
        -100,
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (batch_size, max_len),
        dtype=torch.bool
    )

    offsets = []
    doc_ids = []
    chunk_ids = []

    for i, item in enumerate(batch):

        n = len(item["input_ids"])

        input_ids[i, :n] = torch.tensor(
            item["input_ids"],
            dtype=torch.long
        )

        labels_batch[i, :n] = torch.tensor(
            item["labels"],
            dtype=torch.long
        )

        attention_mask[i, :n] = True

        offsets.append(item["offsets"])
        doc_ids.append(item["doc_id"])
        chunk_ids.append(item["chunk_id"])

    return {
        "input_ids": input_ids,
        "labels": labels_batch,
        "attention_mask": attention_mask,
        "offsets": offsets,
        "doc_ids": doc_ids,
        "chunk_ids": chunk_ids,
    }


a4_dataset = A4NERDataset(
    a4_train_chunks
)

BATCH_SIZE = 4

a4_loader = DataLoader(
    a4_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=a4_collate,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    drop_last=False
)


# ------------------------------------------------------------
# Inspect
# ------------------------------------------------------------

batch = next(iter(a4_loader))

print("=" * 70)
print("A4 REAL-DATA BATCH")
print("=" * 70)

print(
    "Input:",
    tuple(batch["input_ids"].shape)
)

print(
    "Labels:",
    tuple(batch["labels"].shape)
)

print(
    "Mask:",
    tuple(batch["attention_mask"].shape)
)

print(
    "Valid tokens:",
    int(batch["attention_mask"].sum())
)

print(
    "Padding:",
    int((~batch["attention_mask"]).sum())
)

assert (
    batch["input_ids"].shape
    == batch["labels"].shape
    == batch["attention_mask"].shape
)

assert batch["input_ids"].shape[1] <= 1024

assert torch.all(
    batch["labels"][
        ~batch["attention_mask"]
    ] == -100
)

print("\n✓ A4 DATALOADER READY")
print("✓ MAX SEQUENCE <= 1024")
print("✓ PADDING MASK CORRECT")

In [ ]:
# ============================================================
# CELL 13 — A4 MASKED ATTENTION
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class A4SelfAttention(nn.Module):

    def __init__(self, hidden_size=512, num_heads=8):
        super().__init__()

        assert hidden_size % num_heads == 0

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.k_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.v_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.out_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

    def forward(
        self,
        x,
        attention_mask=None
    ):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # ----------------------------------------------------
        # True = valid token
        # False = padding
        #
        # SDPA boolean mask:
        # True  -> attention allowed
        # False -> attention blocked
        # ----------------------------------------------------

        attn_mask = None

        if attention_mask is not None:

            attn_mask = (
                attention_mask
                .to(torch.bool)
                .unsqueeze(1)
                .unsqueeze(1)
            )

        x = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=0.0,
            is_causal=False
        )

        x = x.transpose(
            1, 2
        ).contiguous().view(
            B, T, C
        )

        return self.out_proj(x)


class A4EncoderLayer(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_heads=8,
        ffn_size=2048
    ):

        super().__init__()

        self.norm1 = nn.LayerNorm(
            hidden_size
        )

        self.attn = A4SelfAttention(
            hidden_size,
            num_heads
        )

        self.norm2 = nn.LayerNorm(
            hidden_size
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                hidden_size,
                ffn_size
            ),
            nn.GELU(),
            nn.Linear(
                ffn_size,
                hidden_size
            )
        )

    def forward(
        self,
        x,
        attention_mask=None
    ):

        x = x + self.attn(
            self.norm1(x),
            attention_mask
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


class A4Encoder(nn.Module):

    def __init__(
        self,
        vocab_size=50000,
        hidden_size=512,
        max_length=1024,
        num_layers=6,
        num_heads=8,
        ffn_size=2048
    ):

        super().__init__()

        self.hidden_size = hidden_size
        self.max_length = max_length

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.encoder_pos = nn.Embedding(
            max_length,
            hidden_size
        )

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(
                hidden_size,
                num_heads,
                ffn_size
            )
            for _ in range(num_layers)
        ])

    def forward(
        self,
        input_ids,
        attention_mask=None
    ):

        B, T = input_ids.shape

        if T > self.max_length:
            raise ValueError(
                f"T={T} exceeds {self.max_length}"
            )

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            +
            self.encoder_pos(positions)
                .unsqueeze(0)
        )

        for layer in self.encoder_layers:

            x = layer(
                x,
                attention_mask
            )

        return x

In [ ]:
# ============================================================
# CELL 14 — A4 NER MODEL
# ============================================================

class A4NER(nn.Module):

    def __init__(
        self,
        encoder,
        num_labels=25,
        dropout=0.10
    ):

        super().__init__()

        self.encoder = encoder

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Linear(
            encoder.hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask=None,
        labels=None
    ):

        hidden = self.encoder(
            input_ids,
            attention_mask
        )

        logits = self.classifier(
            self.dropout(hidden)
        )

        loss = None

        if labels is not None:

            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1),
                ignore_index=-100
            )

        return {
            "logits": logits,
            "loss": loss
        }

In [ ]:
# ============================================================
# CELL 15 — REAL GPU SANITY TEST
# ============================================================

import gc
import time

gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda:0")

# Recreate exact encoder architecture
a4_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)

checkpoint = torch.load(
    A4_CHECKPOINT_PATH,
    map_location="cpu"
)

state = checkpoint["model_state_dict"]

# Encoder-only weights
encoder_state = {
    k: v
    for k, v in state.items()
    if (
        k.startswith("token_embedding.")
        or k.startswith("encoder_pos.")
        or k.startswith("encoder_layers.")
    )
}

missing, unexpected = a4_encoder.load_state_dict(
    encoder_state,
    strict=True
)

print("Missing:", missing)
print("Unexpected:", unexpected)

assert len(missing) == 0
assert len(unexpected) == 0

model = A4NER(
    a4_encoder,
    num_labels=25,
    dropout=0.10
).to(device)

model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

# ------------------------------------------------------------
# Three real batches
# ------------------------------------------------------------

torch.cuda.reset_peak_memory_stats()

times = []
losses = []

for step, batch in enumerate(a4_loader):

    if step >= 3:
        break

    input_ids = batch["input_ids"].to(
        device,
        non_blocking=True
    )

    labels = batch["labels"].to(
        device,
        non_blocking=True
    )

    attention_mask = batch[
        "attention_mask"
    ].to(
        device,
        non_blocking=True
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    torch.cuda.synchronize()

    t0 = time.perf_counter()

    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = output["loss"]

    loss.backward()

    optimizer.step()

    torch.cuda.synchronize()

    elapsed = (
        time.perf_counter() - t0
    )

    times.append(elapsed)
    losses.append(
        float(loss.detach().cpu())
    )

    print(
        f"Step {step+1}/3 | "
        f"loss={losses[-1]:.4f} | "
        f"time={elapsed:.3f}s | "
        f"tokens={int(attention_mask.sum())}"
    )


peak_vram = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

avg_time = sum(times) / len(times)

tokens_per_step = sum(
    int(
        next(
            iter([batch["attention_mask"].sum()])
        )
    )
    for _ in [0]
)

print("\n" + "=" * 70)
print("A4 REAL GPU SANITY TEST COMPLETE")
print("=" * 70)

print(
    "Average step time:",
    f"{avg_time:.3f}s"
)

print(
    "Peak VRAM:",
    f"{peak_vram:.2f} GB"
)

print(
    "Losses:",
    [round(x, 4) for x in losses]
)

print("\n✓ FORWARD PASS")
print("✓ LOSS")
print("✓ BACKWARD PASS")
print("✓ OPTIMIZER STEP")
print("✓ MASKED ATTENTION")
print("✓ GPU HEALTH")

In [ ]:
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint:", A4_CHECKPOINT_PATH)

import os
print("Checkpoint exists:", os.path.exists(A4_CHECKPOINT_PATH))
print(
    "Checkpoint size:",
    f"{os.path.getsize(A4_CHECKPOINT_PATH)/1024**2:.1f} MB"
)

print("Starting checkpoint load...")

checkpoint = torch.load(
    A4_CHECKPOINT_PATH,
    map_location="cpu"
)

print("✓ Checkpoint loaded")
print("Keys:", list(checkpoint.keys()))

In [ ]:
A4_CHECKPOINT_PATH = "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"

import os
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("Checkpoint:", A4_CHECKPOINT_PATH)
print("Exists:", os.path.exists(A4_CHECKPOINT_PATH))

print(
    "Size:",
    f"{os.path.getsize(A4_CHECKPOINT_PATH)/1024**2:.1f} MB"
)

print("\nStarting checkpoint load...")

checkpoint = torch.load(
    A4_CHECKPOINT_PATH,
    map_location="cpu"
)

print("✓ Checkpoint loaded")
print("Keys:", list(checkpoint.keys()))

if "model_state_dict" in checkpoint:
    print(
        "State dict tensors:",
        len(checkpoint["model_state_dict"])
    )

In [ ]:
# ============================================================
# CELL 16 — A4 STRICT LOAD + REAL DATA GPU TEST
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import gc

A4_CHECKPOINT_PATH = "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"

device = torch.device("cuda:0")

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# 1. A4 MODEL
# ------------------------------------------------------------

a4_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)

checkpoint = torch.load(
    A4_CHECKPOINT_PATH,
    map_location="cpu"
)

state = checkpoint["model_state_dict"]

encoder_state = {
    k: v
    for k, v in state.items()
    if (
        k.startswith("token_embedding.")
        or k.startswith("encoder_pos.")
        or k.startswith("encoder_layers.")
    )
}

missing, unexpected = a4_encoder.load_state_dict(
    encoder_state,
    strict=True
)

print("=" * 70)
print("A4 CHECKPOINT VALIDATION")
print("=" * 70)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

assert not missing
assert not unexpected

print("✓ STRICT LOAD SUCCESS")


# ------------------------------------------------------------
# 2. NER MODEL
# ------------------------------------------------------------

model = A4NER(
    a4_encoder,
    num_labels=25,
    dropout=0.10
).to(device)

model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

print("✓ MODEL ON GPU")
print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)


# ------------------------------------------------------------
# 3. REAL DATA
# ------------------------------------------------------------

torch.cuda.reset_peak_memory_stats()

times = []
losses = []
token_counts = []

print("\n" + "=" * 70)
print("A4 REAL-DATA TRAINING SANITY TEST")
print("=" * 70)

for step, batch in enumerate(a4_loader):

    if step >= 3:
        break

    input_ids = batch["input_ids"].to(
        device,
        non_blocking=True
    )

    labels = batch["labels"].to(
        device,
        non_blocking=True
    )

    attention_mask = batch["attention_mask"].to(
        device,
        non_blocking=True
    )

    optimizer.zero_grad(set_to_none=True)

    torch.cuda.synchronize()
    start = time.perf_counter()

    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = output["loss"]

    loss.backward()

    optimizer.step()

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    valid_tokens = int(
        attention_mask.sum().item()
    )

    times.append(elapsed)
    losses.append(float(loss.detach().cpu()))
    token_counts.append(valid_tokens)

    print(
        f"Batch {step+1}/3 | "
        f"Loss {losses[-1]:.4f} | "
        f"Time {elapsed:.3f}s | "
        f"Valid tokens {valid_tokens:,}"
    )


# ------------------------------------------------------------
# 4. PERFORMANCE
# ------------------------------------------------------------

peak_vram = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

avg_time = sum(times) / len(times)
avg_tokens = sum(token_counts) / len(token_counts)

throughput = avg_tokens / avg_time

print("\n" + "=" * 70)
print("A4 GPU BENCHMARK")
print("=" * 70)

print(
    f"Average batch time : {avg_time:.3f} sec"
)

print(
    f"Average tokens     : {avg_tokens:,.0f}"
)

print(
    f"Throughput         : {throughput:,.0f} tokens/sec"
)

print(
    f"Peak VRAM          : {peak_vram:.2f} GB"
)

print(
    f"Losses             : "
    f"{[round(x,4) for x in losses]}"
)

print("\n✓ FORWARD")
print("✓ LOSS")
print("✓ BACKWARD")
print("✓ OPTIMIZER")
print("✓ ATTENTION MASK")
print("✓ GPU")
print("\nA4 REAL-DATA TEST PASSED")

In [ ]:
# ============================================================
# CELL 16B — STRICT LOAD TEST
# ============================================================

a4_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)

checkpoint = torch.load(
    A4_CHECKPOINT_PATH,
    map_location="cpu"
)

state = checkpoint["model_state_dict"]

encoder_state = {
    k: v
    for k, v in state.items()
    if (
        k.startswith("token_embedding.")
        or k.startswith("encoder_pos.")
        or k.startswith("encoder_layers.")
    )
}

missing, unexpected = a4_encoder.load_state_dict(
    encoder_state,
    strict=True
)

print("=" * 70)
print("A4 STRICT LOAD")
print("=" * 70)

print("Missing:", missing)
print("Unexpected:", unexpected)

assert not missing
assert not unexpected

print("✓ STRICT LOAD SUCCESS")
print(
    "Encoder parameters:",
    f"{sum(p.numel() for p in a4_encoder.parameters()):,}"
)

In [ ]:
# ============================================================
# CELL 16C — SAFE CHECKPOINT KEY REMAPPING
# ============================================================

import torch

A4_CHECKPOINT_PATH = "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"

checkpoint = torch.load(
    A4_CHECKPOINT_PATH,
    map_location="cpu"
)

state = checkpoint["model_state_dict"]

# Extract encoder
encoder_state = {
    k: v
    for k, v in state.items()
    if (
        k.startswith("token_embedding.")
        or k.startswith("encoder_pos.")
        or k.startswith("encoder_layers.")
    )
}

# ------------------------------------------------------------
# Remap ONLY FFN key names
# ------------------------------------------------------------

remapped_state = {}

for k, v in encoder_state.items():

    new_key = k.replace(
        ".ffn.net.0.",
        ".ffn.0."
    ).replace(
        ".ffn.net.2.",
        ".ffn.2."
    )

    remapped_state[new_key] = v


print("=" * 70)
print("A4 CHECKPOINT KEY REMAPPING")
print("=" * 70)

print(
    "Original encoder keys:",
    len(encoder_state)
)

print(
    "Remapped encoder keys:",
    len(remapped_state)
)

# Verify that only FFN naming changed
changed = [
    (a, b)
    for a, b in zip(
        encoder_state.keys(),
        remapped_state.keys()
    )
    if a != b
]

print(
    "Keys renamed:",
    len(changed)
)

print("\nExample:")
for old, new in changed[:4]:
    print(old)
    print(" ->", new)

# ------------------------------------------------------------
# Strict load
# ------------------------------------------------------------

missing, unexpected = a4_encoder.load_state_dict(
    remapped_state,
    strict=True
)

print("\n" + "=" * 70)
print("STRICT LOAD RESULT")
print("=" * 70)

print("Missing:", missing)
print("Unexpected:", unexpected)

assert len(missing) == 0
assert len(unexpected) == 0

print("\n✓ A4 CHECKPOINT LOADED SUCCESSFULLY")
print(
    "Parameters:",
    f"{sum(p.numel() for p in a4_encoder.parameters()):,}"
)

In [ ]:
# ============================================================
# CELL 17 — A4 REAL-DATA GPU SANITY TEST
# ============================================================

import time
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda:0")

# ------------------------------------------------------------
# Build NER model using the successfully loaded encoder
# ------------------------------------------------------------

model = A4NER(
    a4_encoder,
    num_labels=25,
    dropout=0.10
).to(device)

model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

print("=" * 70)
print("A4 REAL-DATA GPU TEST")
print("=" * 70)

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

# ------------------------------------------------------------
# Benchmark 3 real batches
# ------------------------------------------------------------

torch.cuda.reset_peak_memory_stats()

times = []
losses = []
tokens_list = []

for step, batch in enumerate(a4_loader):

    if step >= 3:
        break

    input_ids = batch["input_ids"].to(
        device,
        non_blocking=True
    )

    labels = batch["labels"].to(
        device,
        non_blocking=True
    )

    attention_mask = batch["attention_mask"].to(
        device,
        non_blocking=True
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    torch.cuda.synchronize()
    start = time.perf_counter()

    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = output["loss"]

    loss.backward()

    optimizer.step()

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    valid_tokens = int(
        attention_mask.sum().item()
    )

    times.append(elapsed)
    losses.append(
        float(loss.detach().cpu())
    )
    tokens_list.append(valid_tokens)

    print(
        f"Batch {step+1}/3 | "
        f"Loss={losses[-1]:.4f} | "
        f"Time={elapsed:.3f}s | "
        f"Valid={valid_tokens:,}"
    )


# ------------------------------------------------------------
# Performance
# ------------------------------------------------------------

avg_time = sum(times) / len(times)
avg_tokens = sum(tokens_list) / len(tokens_list)

throughput = avg_tokens / avg_time

peak_vram = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)

print(f"Average batch time : {avg_time:.3f} sec")
print(f"Average tokens     : {avg_tokens:,.0f}")
print(f"Throughput         : {throughput:,.0f} tokens/sec")
print(f"Peak VRAM          : {peak_vram:.2f} GB")

print(
    "Losses:",
    [round(x, 4) for x in losses]
)

print("\n✓ FORWARD PASS")
print("✓ LOSS")
print("✓ BACKWARD PASS")
print("✓ OPTIMIZER STEP")
print("✓ REAL DATA")
print("✓ MASKED ATTENTION")

print("\nA4 REAL-DATA TEST PASSED")

In [ ]:
# ============================================================
# CELL 18 — STRICT CHARACTER-SPAN DECODER
# ============================================================

# BIO label mapping
id2label = {
    0: "O",
    1: "B-APP",
    2: "B-RESP",
    3: "B-A.COUNSEL",
    4: "B-R.COUNSEL",
    5: "B-JUDGE",
    6: "B-WIT",
    7: "B-AUTH",
    8: "B-COURT",
    9: "B-STAT",
    10: "B-PREC",
    11: "B-DATE",
    12: "B-CASENO",
    13: "I-APP",
    14: "I-RESP",
    15: "I-A.COUNSEL",
    16: "I-R.COUNSEL",
    17: "I-JUDGE",
    18: "I-WIT",
    19: "I-AUTH",
    20: "I-COURT",
    21: "I-STAT",
    22: "I-PREC",
    23: "I-DATE",
    24: "I-CASENO",
}

ENTITY_LABELS = {
    "APP",
    "RESP",
    "A.COUNSEL",
    "R.COUNSEL",
    "JUDGE",
    "WIT",
    "AUTH",
    "COURT",
    "STAT",
    "PREC",
    "DATE",
    "CASENO",
}


def bio_to_token_spans(
    pred_ids,
    token_offsets,
    text
):
    """
    Convert BIO predictions into token-based spans.

    Returns:
        [(token_start, token_end, label), ...]
    """

    spans = []

    current_label = None
    current_start = None
    current_end = None

    for idx, pred_id in enumerate(pred_ids):

        tag = id2label[int(pred_id)]

        # Ignore special/empty offsets
        if idx >= len(token_offsets):
            break

        start, end = token_offsets[idx]

        # Invalid offset
        if end <= start:
            continue

        if tag == "O":

            if current_label is not None:
                spans.append(
                    (
                        current_start,
                        current_end,
                        current_label
                    )
                )

                current_label = None
                current_start = None
                current_end = None

            continue

        prefix, label = tag.split("-", 1)

        if prefix == "B":

            # Close previous entity
            if current_label is not None:
                spans.append(
                    (
                        current_start,
                        current_end,
                        current_label
                    )
                )

            current_label = label
            current_start = start
            current_end = end

        elif prefix == "I":

            # Correct continuation
            if (
                current_label == label
                and current_start is not None
            ):

                current_end = end

            else:

                # Illegal I -> treat as new entity
                if current_label is not None:
                    spans.append(
                        (
                            current_start,
                            current_end,
                            current_label
                        )
                    )

                current_label = label
                current_start = start
                current_end = end

    # Close final entity
    if current_label is not None:

        spans.append(
            (
                current_start,
                current_end,
                current_label
            )
        )

    return spans


print("✓ BIO decoder defined")

In [ ]:
# ============================================================
# CELL 19 — STRICT MACRO-F1
# ============================================================

from collections import defaultdict


def strict_macro_f1(
    predictions,
    gold
):
    """
    predictions:
        {
            doc_id: [
                (start, end, label),
                ...
            ]
        }

    gold:
        {
            doc_id: [
                [start, end, label, optional_text],
                ...
            ]
        }
    """

    labels = sorted(ENTITY_LABELS)

    metrics = {}

    for label in labels:

        pred_set = set()
        gold_set = set()

        for doc_id, spans in predictions.items():

            for start, end, lab in spans:

                if lab == label:
                    pred_set.add(
                        (doc_id, start, end)
                    )

        for doc_id, spans in gold.items():

            for span in spans:

                start = int(span[0])
                end = int(span[1])
                lab = span[2]

                if lab == label:
                    gold_set.add(
                        (doc_id, start, end)
                    )

        tp = len(
            pred_set & gold_set
        )

        fp = len(
            pred_set - gold_set
        )

        fn = len(
            gold_set - pred_set
        )

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall
            / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        metrics[label] = {
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

    macro_f1 = sum(
        m["f1"]
        for m in metrics.values()
    ) / len(metrics)

    macro_precision = sum(
        m["precision"]
        for m in metrics.values()
    ) / len(metrics)

    macro_recall = sum(
        m["recall"]
        for m in metrics.values()
    ) / len(metrics)

    return {
        "macro_f1": macro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "per_label": metrics,
    }


print("✓ Strict Macro-F1 evaluator defined")

In [ ]:
# ============================================================
# CELL 20 — GOLD BIO → CHARACTER SPAN ROUND-TRIP VALIDATION
# ============================================================

total_gold = 0
recovered_exact = 0
decoded_total = 0

examples = []

for chunk in a4_train_chunks:

    # NERChunk is an object, so use attributes
    doc_id = chunk.doc_id
    text = chunk.text
    offsets = chunk.offsets
    labels = chunk.labels

    chunk_start = chunk.char_start
    chunk_end = chunk.char_end

    # Get gold entities for this document
    if doc_id in fold1_gold:
        doc_gold = fold1_gold[doc_id]
    else:
        doc_gold = fold2_gold[doc_id]

    # --------------------------------------------------------
    # Gold entities fully contained in this chunk
    # Convert document coordinates → chunk coordinates
    # --------------------------------------------------------

    gold_in_chunk = []

    for span in doc_gold:

        gs = int(span[0])
        ge = int(span[1])
        gl = span[2]

        if gs >= chunk_start and ge <= chunk_end:

            gold_in_chunk.append(
                (
                    gs - chunk_start,
                    ge - chunk_start,
                    gl
                )
            )

    total_gold += len(gold_in_chunk)

    # --------------------------------------------------------
    # Decode GOLD BIO labels
    # --------------------------------------------------------

    decoded = bio_to_token_spans(
        labels,
        offsets,
        text
    )

    gold_set = set(gold_in_chunk)
    decoded_set = set(decoded)

    exact = gold_set & decoded_set

    recovered_exact += len(exact)
    decoded_total += len(decoded_set)

    # --------------------------------------------------------
    # Save examples of failures
    # --------------------------------------------------------

    missing = gold_set - decoded_set

    if missing and len(examples) < 10:

        examples.append({
            "doc_id": doc_id,
            "missing": list(missing)[:5],
            "decoded": list(decoded_set)[:5]
        })


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("=" * 70)
print("GOLD BIO → CHARACTER SPAN ROUND-TRIP")
print("=" * 70)

print(
    f"Gold entities       : {total_gold:,}"
)

print(
    f"Exact recovered     : {recovered_exact:,}"
)

print(
    f"Exact recovery      : "
    f"{100 * recovered_exact / total_gold:.2f}%"
)

print(
    f"Decoded spans       : {decoded_total:,}"
)

print("\nFirst mismatches:")

if not examples:

    print("✓ NO MISMATCHES")

else:

    for i, ex in enumerate(examples, 1):

        print(f"\n[{i}] Doc: {ex['doc_id']}")
        print("Missing:", ex["missing"])
        print("Decoded:", ex["decoded"])

In [ ]:
# ============================================================
# CELL 20A — INSPECT ACTUAL NERChunk STRUCTURE
# ============================================================

chunk = a4_train_chunks[0]

print("=" * 70)
print("NERChunk STRUCTURE")
print("=" * 70)

print("Type:")
print(type(chunk))

print("\nAttributes:")
print(vars(chunk))

print("\nAvailable fields:")
print(list(vars(chunk).keys()))

In [ ]:
# ============================================================
# CELL 20B — GOLD BIO ROUND-TRIP TEST
# ============================================================

total_gold = 0
exact_recovered = 0
decoded_total = 0

mismatch_examples = []

for chunk in a4_train_chunks:

    offsets = chunk.offsets
    gold_bio = chunk.bio_ids
    gold_spans = chunk.gold_spans

    # Decode the GOLD BIO sequence
    decoded_spans = bio_to_token_spans(
        gold_bio,
        offsets,
        None
    )

    gold_set = set(
        (
            int(s),
            int(e),
            label
        )
        for s, e, label in gold_spans
    )

    decoded_set = set(decoded_spans)

    total_gold += len(gold_set)

    exact_recovered += len(
        gold_set & decoded_set
    )

    decoded_total += len(decoded_set)

    missing = gold_set - decoded_set

    if missing and len(mismatch_examples) < 10:

        mismatch_examples.append({
            "doc_id": chunk.doc_id,
            "chunk_id": chunk.chunk_id,
            "missing": list(missing)[:5],
            "decoded": list(decoded_set)[:5]
        })


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

recovery = (
    100.0 * exact_recovered / total_gold
    if total_gold
    else 0.0
)

print("=" * 70)
print("GOLD BIO → CHARACTER SPAN ROUND-TRIP")
print("=" * 70)

print(
    f"Gold spans          : {total_gold:,}"
)

print(
    f"Decoded spans       : {decoded_total:,}"
)

print(
    f"Exact recovered     : {exact_recovered:,}"
)

print(
    f"Exact recovery      : {recovery:.2f}%"
)

print("\nMismatches:")

if not mismatch_examples:

    print("✓ NONE")

else:

    for ex in mismatch_examples:

        print(
            f"\nDoc {ex['doc_id']} | "
            f"Chunk {ex['chunk_id']}"
        )

        print(
            "Missing:",
            ex["missing"]
        )

        print(
            "Decoded:",
            ex["decoded"]
        )

In [ ]:
# ============================================================
# CELL 21 — CHECK CHUNK COORDINATE SYSTEM
# ============================================================

for idx in [0, 1, 2, 3, 4]:

    chunk = a4_train_chunks[idx]

    print("\n" + "=" * 60)
    print(
        f"Chunk {idx} | "
        f"Doc {chunk.doc_id} | "
        f"char_start={chunk.char_start} | "
        f"char_end={chunk.char_end}"
    )

    print("First offset:", chunk.offsets[0])
    print("Last offset :", chunk.offsets[-1])

    print("First gold spans:")

    for span in chunk.gold_spans[:3]:
        print(" ", span)

    print("First BIO entities:")

    bio_spans = bio_to_token_spans(
        chunk.bio_ids,
        chunk.offsets,
        None
    )

    for span in bio_spans[:3]:
        print(" ", span)

In [ ]:
# ============================================================
# CELL 22 — TEST CHARACTER BOUNDARY NORMALIZATION
# ============================================================

def normalize_span_boundaries(
    start,
    end,
    label,
    text
):
    """
    Remove whitespace accidentally included at
    the beginning/end of a predicted token span.
    """

    while start < end and text[start].isspace():
        start += 1

    while end > start and text[end - 1].isspace():
        end -= 1

    return start, end, label


total_gold = 0
exact_raw = 0
exact_normalized = 0

raw_examples = []
normalized_examples = []

for chunk in a4_train_chunks:

    doc_id = chunk.doc_id

    # Get ORIGINAL full document text
    if doc_id in fold1_text:
        full_text = fold1_text[doc_id]
    else:
        full_text = fold2_text[doc_id]

    gold_set = {
        (int(s), int(e), label)
        for s, e, label in chunk.gold_spans
    }

    decoded = bio_to_token_spans(
        chunk.bio_ids,
        chunk.offsets,
        None
    )

    raw_set = set(decoded)

    normalized_set = {
        normalize_span_boundaries(
            s, e, label, full_text
        )
        for s, e, label in decoded
    }

    total_gold += len(gold_set)

    exact_raw += len(
        gold_set & raw_set
    )

    exact_normalized += len(
        gold_set & normalized_set
    )

    # Keep examples where normalization helped
    if (
        len(gold_set & normalized_set)
        > len(gold_set & raw_set)
        and len(raw_examples) < 5
    ):
        raw_examples.append(
            (doc_id, gold_set, raw_set)
        )

    if (
        len(gold_set - normalized_set) > 0
        and len(normalized_examples) < 5
    ):
        normalized_examples.append(
            (doc_id, gold_set, normalized_set)
        )


print("=" * 70)
print("CHARACTER BOUNDARY NORMALIZATION TEST")
print("=" * 70)

print(
    f"Gold spans              : {total_gold:,}"
)

print(
    f"Raw exact recovery      : "
    f"{exact_raw:,} "
    f"({100*exact_raw/total_gold:.2f}%)"
)

print(
    f"Normalized exact        : "
    f"{exact_normalized:,} "
    f"({100*exact_normalized/total_gold:.2f}%)"
)

print(
    f"Improvement             : "
    f"{exact_normalized-exact_raw:,} spans"
)

print("\n✓ TEST COMPLETE")

In [ ]:
# ============================================================
# CELL 23 — ANALYZE REMAINING 206 BOUNDARY MISMATCHES
# ============================================================

from collections import Counter

remaining = []

for chunk in a4_train_chunks:

    doc_id = chunk.doc_id

    if doc_id in fold1_text:
        full_text = fold1_text[doc_id]
    else:
        full_text = fold2_text[doc_id]

    gold_set = {
        (int(s), int(e), label)
        for s, e, label in chunk.gold_spans
    }

    decoded = bio_to_token_spans(
        chunk.bio_ids,
        chunk.offsets,
        None
    )

    normalized_set = {
        normalize_span_boundaries(
            s, e, label, full_text
        )
        for s, e, label in decoded
    }

    missing = gold_set - normalized_set

    for span in missing:
        remaining.append(
            (doc_id, chunk.chunk_id, span, full_text)
        )


print("=" * 70)
print("REMAINING BOUNDARY MISMATCH ANALYSIS")
print("=" * 70)

print(f"Remaining mismatches: {len(remaining)}")

# ------------------------------------------------------------
# Classify mismatch type
# ------------------------------------------------------------

categories = Counter()

for doc_id, chunk_id, (gs, ge, label), text in remaining:

    # Find token(s) touching the gold boundary
    start_tokens = [
        (i, s, e, text[s:e])
        for i, (s, e) in enumerate(
            a4_train_chunks[
                next(
                    j for j, c in enumerate(a4_train_chunks)
                    if c.doc_id == doc_id
                    and c.chunk_id == chunk_id
                )
            ].offsets
        )
        if s <= gs < e
    ]

    end_tokens = [
        (i, s, e, text[s:e])
        for i, (s, e) in enumerate(
            a4_train_chunks[
                next(
                    j for j, c in enumerate(a4_train_chunks)
                    if c.doc_id == doc_id
                    and c.chunk_id == chunk_id
                )
            ].offsets
        )
        if s < ge <= e
    ]

    start_inside = len(start_tokens) > 0
    end_inside = len(end_tokens) > 0

    if start_inside and end_inside:
        categories["both boundaries inside token"] += 1
    elif start_inside:
        categories["start inside token"] += 1
    elif end_inside:
        categories["end inside token"] += 1
    else:
        categories["neither boundary inside token"] += 1


print("\nMismatch categories:")

for category, count in categories.most_common():
    print(
        f"{category:35s}: "
        f"{count:5d} "
        f"({100*count/len(remaining):6.2f}%)"
    )

# ------------------------------------------------------------
# Show first 15 concrete examples
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIRST 15 REMAINING EXAMPLES")
print("=" * 70)

for doc_id, chunk_id, (gs, ge, label), text in remaining[:15]:

    print(
        f"\nDoc={doc_id} | Chunk={chunk_id} | "
        f"{label} | gold=({gs},{ge})"
    )

    print(
        "Gold text:",
        repr(text[gs:ge])
    )

    print(
        "Context:",
        repr(
            text[
                max(0, gs-15):
                min(len(text), ge+15)
            ]
        )
    )

In [ ]:
# ============================================================
# CELL 24 — DOCUMENT-LEVEL SPAN CEILING
# ============================================================

from collections import defaultdict, Counter

doc_gold = defaultdict(set)
doc_decoded = defaultdict(set)

for chunk in a4_train_chunks:

    doc_id = chunk.doc_id

    if doc_id in fold1_text:
        full_text = fold1_text[doc_id]
    else:
        full_text = fold2_text[doc_id]

    # Gold document-level spans
    for s, e, label in chunk.gold_spans:
        doc_gold[doc_id].add(
            (int(s), int(e), label)
        )

    # BIO → token spans → normalized character spans
    decoded = bio_to_token_spans(
        chunk.bio_ids,
        chunk.offsets,
        None
    )

    for s, e, label in decoded:
        doc_decoded[doc_id].add(
            normalize_span_boundaries(
                s, e, label, full_text
            )
        )


# ------------------------------------------------------------
# Document-level totals
# ------------------------------------------------------------

total_gold = sum(
    len(v) for v in doc_gold.values()
)

total_recovered = sum(
    len(doc_gold[d] & doc_decoded[d])
    for d in doc_gold
)

print("=" * 70)
print("DOCUMENT-LEVEL SPAN CEILING")
print("=" * 70)

print(f"Documents             : {len(doc_gold)}")
print(f"Unique gold spans     : {total_gold:,}")
print(f"Recovered spans       : {total_recovered:,}")
print(
    f"Recovery              : "
    f"{100*total_recovered/total_gold:.2f}%"
)

# ------------------------------------------------------------
# Per-label recovery
# ------------------------------------------------------------

gold_by_label = Counter()
hit_by_label = Counter()

for doc_id in doc_gold:

    gold = doc_gold[doc_id]
    pred = doc_decoded.get(doc_id, set())

    for span in gold:
        label = span[2]
        gold_by_label[label] += 1

        if span in pred:
            hit_by_label[label] += 1


print("\nPer-label recovery:")

for label in ENTITY_LABELS:

    g = gold_by_label[label]
    h = hit_by_label[label]

    print(
        f"{label:10s} "
        f"{h:5d}/{g:5d} "
        f"({100*h/g if g else 0:6.2f}%)"
    )

# ------------------------------------------------------------
# Remaining unique spans
# ------------------------------------------------------------

remaining = []

for doc_id in doc_gold:

    missing = (
        doc_gold[doc_id]
        - doc_decoded.get(doc_id, set())
    )

    for span in missing:
        remaining.append(
            (doc_id, *span)
        )


print("\nRemaining unique spans:", len(remaining))

print("\nFirst 20:")
for row in remaining[:20]:
    print(row)

In [ ]:
# ============================================================
# CELL 25 — A4 NER EXPERIMENT 1: SMOKE TEST
# Full Fine-Tuning Baseline
# ============================================================

import os
import time
import torch
import torch.nn as nn
from torch.optim import AdamW

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_LABELS = 25
LR = 2e-5
WEIGHT_DECAY = 0.01

SMOKE_STEPS = 20

print("=" * 70)
print("A4 NER — EXPERIMENT 1 SMOKE TEST")
print("=" * 70)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU  :", torch.cuda.get_device_name(0))
    torch.cuda.reset_peak_memory_stats()


# ------------------------------------------------------------
# Build fresh model from the validated A4 encoder
# ------------------------------------------------------------

model = A4NER(
    encoder=encoder,
    num_labels=NUM_LABELS,
    dropout=0.10
).to(DEVICE)

# IMPORTANT:
# Full fine-tuning — every A4 encoder parameter remains trainable.

for p in model.parameters():
    p.requires_grad = True


trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.CrossEntropyLoss(
    ignore_index=-100
)


# ------------------------------------------------------------
# Train only 20 batches
# ------------------------------------------------------------

model.train()

losses = []
times = []

start_time = time.time()

for step, batch in enumerate(a4_loader):

    if step >= SMOKE_STEPS:
        break

    step_start = time.time()

    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    labels = batch["labels"].to(DEVICE)

    optimizer.zero_grad(set_to_none=True)

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    loss = criterion(
        logits.reshape(-1, NUM_LABELS),
        labels.reshape(-1)
    )

    loss.backward()

    optimizer.step()

    step_time = time.time() - step_start

    losses.append(loss.item())
    times.append(step_time)

    if (step + 1) % 5 == 0 or step == 0:
        valid_tokens = (
            labels != -100
        ).sum().item()

        print(
            f"Step {step+1:02d}/{SMOKE_STEPS} | "
            f"Loss {loss.item():.4f} | "
            f"Valid tokens {valid_tokens:,} | "
            f"Time {step_time:.3f}s"
        )


# ------------------------------------------------------------
# Smoke-test summary
# ------------------------------------------------------------

elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("SMOKE TEST SUMMARY")
print("=" * 70)

print(f"Steps completed : {len(losses)}")
print(f"Initial loss    : {losses[0]:.4f}")
print(f"Final loss      : {losses[-1]:.4f}")
print(f"Mean batch time : {sum(times)/len(times):.3f}s")
print(f"Total time      : {elapsed:.2f}s")

if torch.cuda.is_available():
    peak_vram = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak VRAM       : {peak_vram:.2f} GB")

print("\n✓ SMOKE TEST COMPLETE")

In [ ]:
# ============================================================
# CELL 25A — FIX A4NER WRAPPER
# ============================================================

class A4NER(nn.Module):
    def __init__(
        self,
        encoder,
        num_labels=25,
        dropout=0.10,
        hidden_size=512
    ):
        super().__init__()

        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask=None
    ):
        x = self.encoder(
            input_ids,
            attention_mask=attention_mask
        )

        x = self.dropout(x)

        return self.classifier(x)


print("✓ A4NER wrapper fixed")

In [ ]:
# ============================================================
# CELL 25B — CORRECT A4NER FOR CURRENT A4Encoder API
# ============================================================

class A4NER(nn.Module):
    def __init__(
        self,
        encoder,
        num_labels=25,
        dropout=0.10,
        hidden_size=512
    ):
        super().__init__()

        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask=None
    ):
        # Current validated A4Encoder accepts input_ids only.
        x = self.encoder(input_ids)

        x = self.dropout(x)

        return self.classifier(x)


print("✓ A4NER wrapper corrected")

In [ ]:
# ============================================================
# CELL 26 — MASK-COMPATIBLE A4 ENCODER
# ============================================================

class A4EncoderMasked(A4Encoder):
    def forward(self, input_ids, attention_mask=None):

        B, T = input_ids.shape

        # Token embeddings
        x = self.token_embedding(input_ids)

        # Positional embeddings
        positions = torch.arange(
            T,
            device=input_ids.device
        ).unsqueeze(0)

        x = x + self.encoder_pos(positions)

        # Convert padding mask to boolean attention mask
        if attention_mask is not None:
            key_mask = (
                attention_mask
                .bool()
                .unsqueeze(1)
                .unsqueeze(1)
            )
        else:
            key_mask = None

        # Encoder layers
        for layer in self.encoder_layers:
            x = layer(
                x,
                attention_mask=key_mask
            )

        return x


print("✓ Mask-compatible encoder class defined")

In [ ]:
# ============================================================
# CELL 26B — CHECK EXISTING LAYER API
# ============================================================

import inspect

print(
    "A4Encoder.forward:",
    inspect.signature(A4Encoder.forward)
)

print(
    "A4EncoderLayer.forward:",
    inspect.signature(A4EncoderLayer.forward)
)

print(
    "A4SelfAttention.forward:",
    inspect.signature(A4SelfAttention.forward)
)

In [ ]:
# ============================================================
# CELL 27 — RESTORE PROPER MASKED A4 ENCODER
# ============================================================

# Use the ORIGINAL validated A4 encoder.
# Do NOT use A4EncoderMasked from Cell 26.

model = A4NER(
    encoder=encoder,
    num_labels=25,
    dropout=0.10
).to(DEVICE)

# Full fine-tuning
for p in model.parameters():
    p.requires_grad = True


# ------------------------------------------------------------
# Verify parameter count
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("=" * 70)
print("A4 NER — MASKED FULL FINE-TUNING CHECK")
print("=" * 70)

print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


# ------------------------------------------------------------
# Real batch forward test WITH attention mask
# ------------------------------------------------------------

batch = next(iter(a4_loader))

input_ids = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

model.eval()

with torch.no_grad():

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

print("\nInput shape :", tuple(input_ids.shape))
print("Mask shape  :", tuple(attention_mask.shape))
print("Logits shape:", tuple(logits.shape))

print(
    "Finite logits:",
    torch.isfinite(logits).all().item()
)

print("\n✓ MASKED FORWARD PASS PASSED")


In [ ]:
# ============================================================
# CELL 28 — EXPERIMENT 1
# A4 FULL FINE-TUNING BASELINE
# ============================================================

import os
import time
import torch
import torch.nn as nn
from torch.optim import AdamW

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DEVICE = torch.device("cuda")

LR = 2e-5
WEIGHT_DECAY = 0.01

EPOCHS = 3

CHECKPOINT_DIR = "/kaggle/working/phase2_exp1"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("=" * 70)
print("PHASE 2 — EXPERIMENT 1")
print("A4 FULL FINE-TUNING BASELINE")
print("=" * 70)

print(f"Device       : {torch.cuda.get_device_name(0)}")
print(f"Epochs       : {EPOCHS}")
print(f"Learning rate: {LR}")
print(f"Weight decay : {WEIGHT_DECAY}")
print(f"Batch size   : {a4_loader.batch_size}")
print(f"Checkpoints  : {CHECKPOINT_DIR}")


# ------------------------------------------------------------
# Fresh model
# ------------------------------------------------------------

model = A4NER(
    encoder=encoder,
    num_labels=25,
    dropout=0.10
).to(DEVICE)

for p in model.parameters():
    p.requires_grad = True


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.CrossEntropyLoss(
    ignore_index=-100
)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

global_step = 0
total_steps = EPOCHS * len(a4_loader)

training_start = time.time()

epoch_times = []

for epoch in range(EPOCHS):

    model.train()

    epoch_start = time.time()
    epoch_loss = 0.0

    for batch_idx, batch in enumerate(a4_loader):

        step_start = time.time()

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        labels = batch["labels"].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(
            logits.reshape(-1, 25),
            labels.reshape(-1)
        )

        loss.backward()

        optimizer.step()

        loss_value = loss.item()

        epoch_loss += loss_value
        global_step += 1

        step_time = time.time() - step_start

        # ----------------------------------------------------
        # Progress / ETA
        # ----------------------------------------------------

        if (
            batch_idx == 0
            or (batch_idx + 1) % 25 == 0
            or batch_idx == len(a4_loader) - 1
        ):

            elapsed = time.time() - training_start

            avg_step = elapsed / global_step

            remaining_steps = (
                total_steps - global_step
            )

            eta_seconds = (
                remaining_steps * avg_step
            )

            progress = (
                100.0 *
                global_step /
                total_steps
            )

            print(
                f"Epoch {epoch+1}/{EPOCHS} | "
                f"Step {batch_idx+1}/{len(a4_loader)} | "
                f"Global {global_step}/{total_steps} | "
                f"Loss {loss_value:.4f} | "
                f"{progress:5.1f}% | "
                f"{step_time:.3f}s | "
                f"ETA {eta_seconds/60:.1f} min"
            )

    # --------------------------------------------------------
    # Epoch checkpoint
    # --------------------------------------------------------

    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)

    avg_loss = epoch_loss / len(a4_loader)

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"a4_ner_exp1_epoch{epoch+1}.pt"
    )

    torch.save(
        {
            "epoch": epoch + 1,
            "global_step": global_step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": avg_loss,
            "config": {
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
                "batch_size": a4_loader.batch_size,
                "epochs": EPOCHS,
                "num_labels": 25,
                "architecture": "A4_full_finetune_NER"
            }
        },
        checkpoint_path
    )

    print("\n" + "-" * 70)
    print(
        f"Epoch {epoch+1} COMPLETE | "
        f"Avg loss {avg_loss:.4f} | "
        f"Time {epoch_time/60:.2f} min"
    )
    print(
        f"Checkpoint saved: {checkpoint_path}"
    )
    print("-" * 70 + "\n")


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

total_time = time.time() - training_start

print("=" * 70)
print("EXPERIMENT 1 TRAINING COMPLETE")
print("=" * 70)

print(f"Total steps : {global_step}")
print(f"Total time  : {total_time/60:.2f} min")

if torch.cuda.is_available():
    peak_vram = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )
    print(f"Peak VRAM   : {peak_vram:.2f} GB")

print("\n✓ TRAINING COMPLETE")

In [ ]:
# ============================================================
# CELL 29 — PHASE 2 VALIDATION CHUNKS
# Fold 1 = validation
# Fold 2 = training
# ============================================================

# We already have a4_train_chunks containing both folds.
# Separate them using doc IDs.

fold1_ids = set(fold1_text.keys())
fold2_ids = set(fold2_text.keys())

a4_val_chunks = [
    c for c in a4_train_chunks
    if c.doc_id in fold1_ids
]

a4_train_only_chunks = [
    c for c in a4_train_chunks
    if c.doc_id in fold2_ids
]

print("=" * 70)
print("PHASE 2 FOLD SPLIT")
print("=" * 70)

print(f"Fold 1 validation documents : {len(fold1_ids)}")
print(f"Fold 1 validation chunks    : {len(a4_val_chunks)}")

print(f"Fold 2 training documents   : {len(fold2_ids)}")
print(f"Fold 2 training chunks      : {len(a4_train_only_chunks)}")

print("\n✓ VALIDATION SPLIT READY")

In [ ]:
# ============================================================
# CELL 30 — VALIDATION DATALOADER
# ============================================================

from torch.utils.data import DataLoader

a4_val_loader = DataLoader(
    a4_val_chunks,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=True
)

print("=" * 70)
print("VALIDATION LOADER")
print("=" * 70)

print("Batches:", len(a4_val_loader))
print("Batch size:", a4_val_loader.batch_size)

print("\n✓ VALIDATION LOADER READY")

In [ ]:
# ============================================================
# CELL 30A — VALIDATION COLLATE FUNCTION
# ============================================================

import torch
from torch.utils.data import DataLoader

def a4_collate_fn(batch):
    """
    Dynamic-padding collator for NER chunks.

    Each NERChunk contains:
      input_ids
      bio_ids
      offsets
      gold_spans
    """

    max_len = max(
        len(chunk.input_ids)
        for chunk in batch
    )

    batch_size = len(batch)

    input_ids = torch.zeros(
        (batch_size, max_len),
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (batch_size, max_len),
        dtype=torch.long
    )

    labels = torch.full(
        (batch_size, max_len),
        -100,
        dtype=torch.long
    )

    for i, chunk in enumerate(batch):

        n = len(chunk.input_ids)

        input_ids[i, :n] = torch.tensor(
            chunk.input_ids,
            dtype=torch.long
        )

        attention_mask[i, :n] = 1

        labels[i, :n] = torch.tensor(
            chunk.bio_ids,
            dtype=torch.long
        )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


# ------------------------------------------------------------
# Build validation loader
# ------------------------------------------------------------

a4_val_loader = DataLoader(
    a4_val_chunks,
    batch_size=4,
    shuffle=False,
    collate_fn=a4_collate_fn,
    pin_memory=True
)

print("=" * 70)
print("VALIDATION LOADER READY")
print("=" * 70)

print(
    f"Validation documents : {len(fold1_ids)}"
)

print(
    f"Validation chunks    : {len(a4_val_chunks)}"
)

print(
    f"Validation batches   : {len(a4_val_loader)}"
)

print(
    f"Batch size           : "
    f"{a4_val_loader.batch_size}"
)

# Quick shape check
batch = next(iter(a4_val_loader))

print(
    "\nBatch input shape    :",
    tuple(batch["input_ids"].shape)
)

print(
    "Batch mask shape     :",
    tuple(batch["attention_mask"].shape)
)

print(
    "Batch labels shape   :",
    tuple(batch["labels"].shape)
)

print("\n✓ VALIDATION LOADER PASSED")

In [ ]:
# ============================================================
# CELL 31 — EXPERIMENT 1 VALIDATION
# Epoch 1 vs Epoch 2 vs Epoch 3
# ============================================================

import os
import time
import torch
import torch.nn as nn
from collections import defaultdict

DEVICE = torch.device("cuda")

CHECKPOINT_DIR = "/kaggle/working/phase2_exp1"

checkpoint_paths = [
    os.path.join(
        CHECKPOINT_DIR,
        "a4_ner_exp1_epoch1.pt"
    ),
    os.path.join(
        CHECKPOINT_DIR,
        "a4_ner_exp1_epoch2.pt"
    ),
    os.path.join(
        CHECKPOINT_DIR,
        "a4_ner_exp1_epoch3.pt"
    ),
]


# ------------------------------------------------------------
# Gold document-level spans
# ------------------------------------------------------------

val_gold = defaultdict(set)

for chunk in a4_val_chunks:

    for s, e, label in chunk.gold_spans:

        val_gold[chunk.doc_id].add(
            (int(s), int(e), label)
        )


# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

def load_exp1_checkpoint(path):

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    model = A4NER(
        encoder=encoder,
        num_labels=25,
        dropout=0.10
    ).to(DEVICE)

    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )

    model.eval()

    return model


# ------------------------------------------------------------
# BIO → normalized character spans
# ------------------------------------------------------------

def decode_prediction_chunk(
    pred_ids,
    chunk
):

    decoded = bio_to_token_spans(
        pred_ids,
        chunk.offsets,
        None
    )

    doc_id = chunk.doc_id

    if doc_id in fold1_text:
        full_text = fold1_text[doc_id]
    else:
        full_text = fold2_text[doc_id]

    normalized = set()

    for s, e, label in decoded:

        s, e, label = normalize_span_boundaries(
            s,
            e,
            label,
            full_text
        )

        if s < e:
            normalized.add(
                (int(s), int(e), label)
            )

    return normalized


# ------------------------------------------------------------
# Strict Macro-F1
# ------------------------------------------------------------

def calculate_strict_metrics(
    predictions,
    gold
):

    per_label = {}

    for label in ENTITY_LABELS:

        gold_set = set()

        pred_set = set()

        for doc_id, spans in gold.items():

            for s, e, l in spans:

                if l == label:
                    gold_set.add(
                        (doc_id, s, e, l)
                    )

        for doc_id, spans in predictions.items():

            for s, e, l in spans:

                if l == label:
                    pred_set.add(
                        (doc_id, s, e, l)
                    )

        tp = len(
            gold_set & pred_set
        )

        fp = len(
            pred_set - gold_set
        )

        fn = len(
            gold_set - pred_set
        )

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall /
            (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        per_label[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": tp,
            "fp": fp,
            "fn": fn
        }

    macro_precision = sum(
        x["precision"]
        for x in per_label.values()
    ) / len(per_label)

    macro_recall = sum(
        x["recall"]
        for x in per_label.values()
    ) / len(per_label)

    macro_f1 = sum(
        x["f1"]
        for x in per_label.values()
    ) / len(per_label)

    return (
        macro_precision,
        macro_recall,
        macro_f1,
        per_label
    )


# ------------------------------------------------------------
# Evaluate one checkpoint
# ------------------------------------------------------------

def evaluate_checkpoint(path):

    model = load_exp1_checkpoint(path)

    predictions = defaultdict(set)

    start_time = time.time()

    with torch.no_grad():

        for batch_idx, batch in enumerate(
            a4_val_loader
        ):

            input_ids = batch[
                "input_ids"
            ].to(
                DEVICE,
                non_blocking=True
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                DEVICE,
                non_blocking=True
            )

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            pred_ids = (
                logits.argmax(dim=-1)
                .cpu()
                .tolist()
            )

            # Recover corresponding chunks
            start = batch_idx * a4_val_loader.batch_size

            end = min(
                start + len(pred_ids),
                len(a4_val_chunks)
            )

            for local_i, chunk_i in enumerate(
                range(start, end)
            ):

                chunk = a4_val_chunks[chunk_i]

                spans = decode_prediction_chunk(
                    pred_ids[local_i],
                    chunk
                )

                predictions[
                    chunk.doc_id
                ].update(spans)

    elapsed = time.time() - start_time

    metrics = calculate_strict_metrics(
        predictions,
        val_gold
    )

    del model

    torch.cuda.empty_cache()

    return metrics, elapsed


# ------------------------------------------------------------
# Evaluate all three checkpoints
# ------------------------------------------------------------

results = {}

print("=" * 70)
print("EXPERIMENT 1 — STRICT VALIDATION")
print("=" * 70)

for epoch in [1, 2, 3]:

    path = os.path.join(
        CHECKPOINT_DIR,
        f"a4_ner_exp1_epoch{epoch}.pt"
    )

    print(
        f"\nEvaluating Epoch {epoch}..."
    )

    metrics, elapsed = evaluate_checkpoint(
        path
    )

    precision, recall, f1, per_label = metrics

    results[epoch] = metrics

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"Strict Macro-F1 : {f1:.4f}"
    )

    print(
        f"Time      : {elapsed:.2f}s"
    )


# ------------------------------------------------------------
# Best checkpoint
# ------------------------------------------------------------

best_epoch = max(
    results,
    key=lambda e: results[e][2]
)

best_f1 = results[
    best_epoch
][2]

print("\n" + "=" * 70)
print("EXPERIMENT 1 RESULTS")
print("=" * 70)

for epoch in [1, 2, 3]:

    p, r, f1, _ = results[epoch]

    print(
        f"Epoch {epoch}: "
        f"P={p:.4f} | "
        f"R={r:.4f} | "
        f"F1={f1:.4f}"
    )

print(
    f"\n★ BEST CHECKPOINT: Epoch {best_epoch}"
)

print(
    f"★ BEST STRICT MACRO-F1: {best_f1:.4f}"
)

print("\n✓ VALIDATION COMPLETE")

In [ ]:
# ============================================================
# CELL 32 — EXPERIMENT 1 VALIDATION DIAGNOSTIC
# ============================================================

import torch
from collections import Counter

# ------------------------------------------------------------
# Load BEST checkpoint
# ------------------------------------------------------------

best_path = (
    "/kaggle/working/phase2_exp1/"
    "a4_ner_exp1_epoch3.pt"
)

checkpoint = torch.load(
    best_path,
    map_location="cpu",
    weights_only=False
)

diagnostic_model = A4NER(
    encoder=encoder,
    num_labels=25,
    dropout=0.10
).to(DEVICE)

diagnostic_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

diagnostic_model.eval()


# ------------------------------------------------------------
# Token-level statistics
# ------------------------------------------------------------

correct = 0
total = 0

gold_counter = Counter()
pred_counter = Counter()

illegal_transitions = 0
total_sequences = 0

all_prediction_examples = []

with torch.no_grad():

    for batch_idx, batch in enumerate(a4_val_loader):

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        labels = batch["labels"].to(
            DEVICE,
            non_blocking=True
        )

        logits = diagnostic_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = logits.argmax(dim=-1)

        valid = labels != -100

        correct += (
            (preds == labels) & valid
        ).sum().item()

        total += valid.sum().item()

        gold_valid = labels[valid].detach().cpu().tolist()
        pred_valid = preds[valid].detach().cpu().tolist()

        gold_counter.update(gold_valid)
        pred_counter.update(pred_valid)

        # ----------------------------------------------------
        # Check BIO transitions
        # ----------------------------------------------------

        for row in range(labels.size(0)):

            seq_len = int(
                attention_mask[row].sum().item()
            )

            seq = (
                preds[row, :seq_len]
                .detach()
                .cpu()
                .tolist()
            )

            previous = 0

            for current in seq:

                current_name = id2label[current]
                previous_name = id2label[previous]

                if current_name.startswith("I-"):

                    current_type = current_name[2:]

                    if previous_name not in (
                        f"B-{current_type}",
                        f"I-{current_type}"
                    ):
                        illegal_transitions += 1

                previous = current

            total_sequences += 1

            # Save first few examples
            if (
                len(all_prediction_examples) < 5
                and row == 0
            ):
                all_prediction_examples.append(
                    (
                        input_ids[row]
                        .detach()
                        .cpu()
                        .tolist(),
                        labels[row]
                        .detach()
                        .cpu()
                        .tolist(),
                        preds[row]
                        .detach()
                        .cpu()
                        .tolist(),
                        seq_len
                    )
                )


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

token_accuracy = (
    correct / total
    if total else 0
)

print("=" * 70)
print("EXPERIMENT 1 — TOKEN-LEVEL VALIDATION DIAGNOSTIC")
print("=" * 70)

print(
    f"Valid tokens       : {total:,}"
)

print(
    f"Correct tokens     : {correct:,}"
)

print(
    f"Token accuracy     : {token_accuracy:.4f}"
)

print(
    f"Illegal I- transitions : "
    f"{illegal_transitions:,}"
)

print(
    f"Validation sequences  : "
    f"{total_sequences:,}"
)


# ------------------------------------------------------------
# Predicted label distribution
# ------------------------------------------------------------

print("\nGold label distribution:")

for idx in range(25):

    count = gold_counter[idx]

    if count:
        print(
            f"{idx:2d} "
            f"{id2label[idx]:12s} "
            f"{count:8,}"
        )


print("\nPredicted label distribution:")

for idx in range(25):

    count = pred_counter[idx]

    if count:
        print(
            f"{idx:2d} "
            f"{id2label[idx]:12s} "
            f"{count:8,}"
        )


# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del diagnostic_model

torch.cuda.empty_cache()

print("\n✓ DIAGNOSTIC COMPLETE")

In [ ]:
# ============================================================
# CELL 33 — EXPERIMENT 2
# TRAINING-FOLD-ONLY CLASS WEIGHTS
# ============================================================

from collections import Counter
import torch

# ------------------------------------------------------------
# Count BIO labels ONLY in Fold 2 training data
# ------------------------------------------------------------

train_label_counts = Counter()

for chunk in a4_train_only_chunks:

    for label_id in chunk.bio_ids:
        train_label_counts[int(label_id)] += 1


print("=" * 70)
print("EXPERIMENT 2 — TRAINING LABEL DISTRIBUTION")
print("=" * 70)

total_train_tokens = sum(
    train_label_counts.values()
)

print(
    f"Training tokens: {total_train_tokens:,}"
)

for idx in range(25):

    count = train_label_counts[idx]

    print(
        f"{idx:2d} "
        f"{id2label[idx]:12s} "
        f"{count:8,} "
        f"{100*count/total_train_tokens:7.3f}%"
    )


# ------------------------------------------------------------
# Inverse-frequency weighting
#
# Weight = total / (num_classes * class_count)
#
# This is calculated ONLY from Fold 2.
# ------------------------------------------------------------

num_classes = 25

class_weights = torch.zeros(
    num_classes,
    dtype=torch.float32
)

for idx in range(num_classes):

    count = train_label_counts[idx]

    if count > 0:
        class_weights[idx] = (
            total_train_tokens /
            (num_classes * count)
        )

# ------------------------------------------------------------
# Normalize mean weight to 1
# ------------------------------------------------------------

class_weights = (
    class_weights /
    class_weights.mean()
)

print("\n" + "=" * 70)
print("CLASS WEIGHTS")
print("=" * 70)

for idx in range(num_classes):

    print(
        f"{idx:2d} "
        f"{id2label[idx]:12s} "
        f"weight={class_weights[idx].item():.4f}"
    )

print("\n✓ WEIGHTS CALCULATED FROM TRAINING FOLD ONLY")

In [ ]:
from torch.utils.data import DataLoader

a4_train_loader_exp2 = DataLoader(
    a4_train_only_chunks,
    batch_size=4,
    shuffle=True,
    collate_fn=a4_collate_fn,
    pin_memory=True
)

print(
    f"Training chunks: {len(a4_train_only_chunks)}"
)

print(
    f"Training batches: {len(a4_train_loader_exp2)}"
)

In [ ]:
# ============================================================
# CELL 34 — EXPERIMENT 2
# A4 + CLASS-WEIGHTED CROSS-ENTROPY
# ============================================================

import os
import time
import torch
import torch.nn as nn
from torch.optim import AdamW

DEVICE = torch.device("cuda")

LR = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 3

CHECKPOINT_DIR_2 = "/kaggle/working/phase2_exp2"
os.makedirs(CHECKPOINT_DIR_2, exist_ok=True)

print("=" * 70)
print("PHASE 2 — EXPERIMENT 2")
print("A4 FULL FINE-TUNING + CLASS-WEIGHTED CE")
print("=" * 70)

# ------------------------------------------------------------
# Fresh model
# ------------------------------------------------------------

model2 = A4NER(
    encoder=encoder,
    num_labels=25,
    dropout=0.10
).to(DEVICE)

for p in model2.parameters():
    p.requires_grad = True


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer2 = AdamW(
    model2.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# Move weights to GPU
loss_weights = class_weights.to(DEVICE)

criterion2 = nn.CrossEntropyLoss(
    weight=loss_weights,
    ignore_index=-100
)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

global_step = 0
total_steps = EPOCHS * len(a4_train_only_chunks) // 4

training_start = time.time()

for epoch in range(EPOCHS):

    model2.train()

    epoch_start = time.time()
    epoch_loss = 0.0

    for batch_idx, batch in enumerate(a4_loader):

        step_start = time.time()

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        labels = batch["labels"].to(
            DEVICE,
            non_blocking=True
        )

        optimizer2.zero_grad(set_to_none=True)

        logits = model2(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion2(
            logits.reshape(-1, 25),
            labels.reshape(-1)
        )

        loss.backward()

        optimizer2.step()

        loss_value = loss.item()

        epoch_loss += loss_value
        global_step += 1

        elapsed = time.time() - training_start
        avg_step = elapsed / global_step

        remaining_steps = (
            EPOCHS * len(a4_loader)
            - global_step
        )

        eta = remaining_steps * avg_step

        progress = (
            100.0 * global_step /
            (EPOCHS * len(a4_loader))
        )

        if (
            batch_idx == 0
            or (batch_idx + 1) % 25 == 0
            or batch_idx == len(a4_loader) - 1
        ):
            print(
                f"Epoch {epoch+1}/{EPOCHS} | "
                f"Step {batch_idx+1}/{len(a4_loader)} | "
                f"Global {global_step} | "
                f"Loss {loss_value:.4f} | "
                f"{progress:5.1f}% | "
                f"{step_time:.3f}s | "
                f"ETA {eta/60:.1f} min"
            )

    epoch_time = time.time() - epoch_start
    avg_loss = epoch_loss / len(a4_loader)

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR_2,
        f"a4_ner_exp2_epoch{epoch+1}.pt"
    )

    torch.save(
        {
            "epoch": epoch + 1,
            "global_step": global_step,
            "model_state_dict": model2.state_dict(),
            "optimizer_state_dict": optimizer2.state_dict(),
            "train_loss": avg_loss,
            "class_weights": class_weights.cpu(),
            "config": {
                "architecture": "A4_full_finetune_weighted_CE",
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
                "epochs": EPOCHS,
                "num_labels": 25
            }
        },
        checkpoint_path
    )

    print("\n" + "-" * 70)
    print(
        f"Epoch {epoch+1} COMPLETE | "
        f"Avg loss {avg_loss:.4f} | "
        f"Time {epoch_time/60:.2f} min"
    )
    print(
        f"Checkpoint saved: {checkpoint_path}"
    )
    print("-" * 70)


total_time = time.time() - training_start

print("\n" + "=" * 70)
print("EXPERIMENT 2 TRAINING COMPLETE")
print("=" * 70)

print(f"Total steps : {global_step}")
print(f"Total time  : {total_time/60:.2f} min")

print(
    f"Peak VRAM   : "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)

print("\n✓ EXPERIMENT 2 COMPLETE")

In [ ]:
# ============================================================
# CELL 35 — EXPERIMENT 2 STRICT VALIDATION
# ============================================================

import os
import time
import torch
from collections import defaultdict

EXP2_DIR = "/kaggle/working/phase2_exp2"

exp2_results = {}

print("=" * 70)
print("EXPERIMENT 2 — STRICT VALIDATION")
print("=" * 70)

for epoch in [1, 2, 3]:

    checkpoint_path = os.path.join(
        EXP2_DIR,
        f"a4_ner_exp2_epoch{epoch}.pt"
    )

    print(f"\nEvaluating Epoch {epoch}...")

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    model_eval = A4NER(
        encoder=encoder,
        num_labels=25,
        dropout=0.10
    ).to(DEVICE)

    model_eval.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )

    model_eval.eval()

    predictions = defaultdict(set)

    start_time = time.time()

    with torch.no_grad():

        for batch_idx, batch in enumerate(
            a4_val_loader
        ):

            input_ids = batch[
                "input_ids"
            ].to(
                DEVICE,
                non_blocking=True
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                DEVICE,
                non_blocking=True
            )

            logits = model_eval(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            pred_ids = (
                logits.argmax(dim=-1)
                .cpu()
                .tolist()
            )

            start = (
                batch_idx *
                a4_val_loader.batch_size
            )

            end = min(
                start + len(pred_ids),
                len(a4_val_chunks)
            )

            for local_i, chunk_i in enumerate(
                range(start, end)
            ):

                chunk = a4_val_chunks[chunk_i]

                spans = decode_prediction_chunk(
                    pred_ids[local_i],
                    chunk
                )

                predictions[
                    chunk.doc_id
                ].update(spans)

    elapsed = time.time() - start_time

    metrics = calculate_strict_metrics(
        predictions,
        val_gold
    )

    precision, recall, f1, per_label = metrics

    exp2_results[epoch] = metrics

    print(
        f"Precision       : {precision:.4f}"
    )

    print(
        f"Recall          : {recall:.4f}"
    )

    print(
        f"Strict Macro-F1 : {f1:.4f}"
    )

    print(
        f"Time            : {elapsed:.2f}s"
    )

    del model_eval
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

best_exp2_epoch = max(
    exp2_results,
    key=lambda e: exp2_results[e][2]
)

best_exp2_f1 = exp2_results[
    best_exp2_epoch
][2]

print("\n" + "=" * 70)
print("EXPERIMENT 2 RESULTS")
print("=" * 70)

for epoch in [1, 2, 3]:

    p, r, f1, _ = exp2_results[epoch]

    print(
        f"Epoch {epoch}: "
        f"P={p:.4f} | "
        f"R={r:.4f} | "
        f"F1={f1:.4f}"
    )

print(
    f"\n★ BEST EPOCH: {best_exp2_epoch}"
)

print(
    f"★ BEST STRICT MACRO-F1: "
    f"{best_exp2_f1:.4f}"
)

print("\n✓ EXPERIMENT 2 VALIDATION COMPLETE")

In [ ]:
# ============================================================
# CELL 36 — INSPECT A4 ATTENTION TARGETS
# ============================================================

print("=" * 70)
print("A4 LORA TARGET INSPECTION")
print("=" * 70)

for i, layer in enumerate(encoder.encoder_layers):

    print(f"\nEncoder layer {i}:")

    print(
        "q_proj:",
        type(layer.attn.q_proj).__name__,
        tuple(layer.attn.q_proj.weight.shape)
    )

    print(
        "k_proj:",
        type(layer.attn.k_proj).__name__,
        tuple(layer.attn.k_proj.weight.shape)
    )

    print(
        "v_proj:",
        type(layer.attn.v_proj).__name__,
        tuple(layer.attn.v_proj.weight.shape)
    )

    print(
        "out_proj:",
        type(layer.attn.out_proj).__name__,
        tuple(layer.attn.out_proj.weight.shape)
    )

print("\n✓ TARGET INSPECTION COMPLETE")

In [ ]:
# ============================================================
# CELL 37 — EXPERIMENT 3: LORA MODULE
# ============================================================

import math
import torch
import torch.nn as nn


class LoRALinear(nn.Module):
    """
    Frozen base Linear + trainable low-rank update.

    W' = W + scaling * B(A(x))
    """

    def __init__(
        self,
        base_layer,
        rank=8,
        alpha=16,
        dropout=0.05
    ):
        super().__init__()

        self.base = base_layer

        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        self.lora_dropout = nn.Dropout(dropout)

        self.lora_A = nn.Parameter(
            torch.empty(
                rank,
                base_layer.in_features
            )
        )

        self.lora_B = nn.Parameter(
            torch.zeros(
                base_layer.out_features,
                rank
            )
        )

        # Standard LoRA initialization
        nn.init.kaiming_uniform_(
            self.lora_A,
            a=math.sqrt(5)
        )

        # Start with zero update
        nn.init.zeros_(self.lora_B)

        # Freeze original A4 weights
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):

        base_output = self.base(x)

        lora_output = (
            self.lora_B @
            self.lora_A
        )

        lora_output = (
            self.lora_dropout(x)
            @ lora_output.T
        )

        return (
            base_output +
            self.scaling * lora_output
        )


print("✓ LoRALinear defined")

In [ ]:
# ============================================================
# CELL 38 — INJECT LORA INTO A4 Q/V PROJECTIONS
# ============================================================

import copy

# Start from the ORIGINAL validated A4 encoder.
# Deep copy prevents accidental modification of the
# checkpoint reconstruction used by previous experiments.

lora_encoder = copy.deepcopy(encoder)


# ------------------------------------------------------------
# Inject LoRA into Q and V
# ------------------------------------------------------------

for layer in lora_encoder.encoder_layers:

    layer.attn.q_proj = LoRALinear(
        layer.attn.q_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )

    layer.attn.v_proj = LoRALinear(
        layer.attn.v_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )


# ------------------------------------------------------------
# Freeze everything first
# ------------------------------------------------------------

for p in lora_encoder.parameters():
    p.requires_grad = False


# Re-enable only LoRA parameters
for module in lora_encoder.modules():

    if isinstance(module, LoRALinear):

        module.lora_A.requires_grad = True
        module.lora_B.requires_grad = True


# ------------------------------------------------------------
# NER model
# ------------------------------------------------------------

model3 = A4NER(
    encoder=lora_encoder,
    num_labels=25,
    dropout=0.10
).to(DEVICE)


# Classification head is trainable
for p in model3.classifier.parameters():
    p.requires_grad = True


# ------------------------------------------------------------
# Parameter statistics
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model3.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model3.parameters()
    if p.requires_grad
)

print("=" * 70)
print("EXPERIMENT 3 — LORA PARAMETER CHECK")
print("=" * 70)

print(
    f"Total parameters     : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params:,}"
)

print(
    f"Trainable percentage  : "
    f"{100*trainable_params/total_params:.2f}%"
)

print("\nTrainable parameter groups:")

for name, p in model3.named_parameters():

    if p.requires_grad:

        print(
            f"{name:70s} "
            f"{p.numel():,}"
        )

print("\n✓ LORA INJECTION COMPLETE")

In [ ]:
# ============================================================
# CELL 39 — EXPERIMENT 3 LORA SANITY TEST
# ============================================================

import torch
import torch.nn as nn

model3.train()

optimizer3_test = torch.optim.AdamW(
    [
        p for p in model3.parameters()
        if p.requires_grad
    ],
    lr=2e-4,
    weight_decay=0.01
)

criterion3_test = nn.CrossEntropyLoss(
    ignore_index=-100
)

batch = next(iter(a4_train_loader_exp2))

input_ids = batch["input_ids"].to(
    DEVICE,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    DEVICE,
    non_blocking=True
)

labels = batch["labels"].to(
    DEVICE,
    non_blocking=True
)

optimizer3_test.zero_grad(set_to_none=True)

logits = model3(
    input_ids=input_ids,
    attention_mask=attention_mask
)

loss = criterion3_test(
    logits.reshape(-1, 25),
    labels.reshape(-1)
)

loss.backward()

# ------------------------------------------------------------
# Check gradients
# ------------------------------------------------------------

lora_grad_norm = 0.0
lora_grad_count = 0

for name, p in model3.named_parameters():

    if (
        p.requires_grad
        and p.grad is not None
    ):

        lora_grad_norm += (
            p.grad.detach()
            .float()
            .norm()
            .item() ** 2
        )

        lora_grad_count += 1


lora_grad_norm = (
    lora_grad_norm ** 0.5
)


print("=" * 70)
print("EXPERIMENT 3 — LORA SANITY TEST")
print("=" * 70)

print(
    f"Loss               : {loss.item():.4f}"
)

print(
    f"Trainable tensors  : {lora_grad_count}"
)

print(
    f"Gradient norm      : "
    f"{lora_grad_norm:.6f}"
)

print(
    "Finite logits      :",
    torch.isfinite(logits).all().item()
)

print(
    "Finite gradients   :",
    all(
        p.grad is None or
        torch.isfinite(p.grad).all().item()
        for p in model3.parameters()
    )
)

print("\n✓ LORA BACKWARD TEST PASSED")

In [ ]:
# ============================================================
# CELL 40 — VERIFY CLEAN TRAIN/VALIDATION SPLIT
# ============================================================

print("=" * 70)
print("CLEAN PHASE-2 SPLIT VERIFICATION")
print("=" * 70)

print(f"Training chunks   : {len(a4_train_only_chunks)}")
print(f"Training batches  : {len(a4_train_loader_exp2)}")
print(f"Validation chunks : {len(a4_val_loader.dataset)}")
print(f"Validation batches : {len(a4_val_loader)}")

assert len(a4_train_only_chunks) == 306, \
    "Training split is not Fold 2!"

assert len(a4_train_loader_exp2) == 77, \
    "Training loader should contain ~77 batches!"

assert len(a4_val_loader.dataset) == 298, \
    "Validation split is not Fold 1!"

print("\n✓ TRAINING = FOLD 2")
print("✓ VALIDATION = FOLD 1")
print("✓ NO CHUNK-LEVEL SPLIT LEAKAGE")

In [ ]:
# ============================================================
# CELL 41 — EXPERIMENT 3: LORA TRAINING
# ============================================================

import os
import time
import torch
import torch.nn as nn

EXP3_DIR = "/kaggle/working/phase2_exp3"
os.makedirs(EXP3_DIR, exist_ok=True)

# ------------------------------------------------------------
# Optimizer: ONLY trainable LoRA + classifier parameters
# ------------------------------------------------------------

trainable_params = [
    p for p in model3.parameters()
    if p.requires_grad
]

optimizer3 = torch.optim.AdamW(
    trainable_params,
    lr=2e-4,
    weight_decay=0.01
)

criterion3 = nn.CrossEntropyLoss(
    ignore_index=-100
)

EPOCHS3 = 3

history3 = []

print("=" * 70)
print("EXPERIMENT 3 — A4 + LoRA")
print("=" * 70)

print(f"Train chunks       : {len(a4_train_only_chunks)}")
print(f"Train batches      : {len(a4_train_loader_exp2)}")
print(f"Validation chunks  : {len(a4_val_loader.dataset)}")
print(f"Trainable params   : {sum(p.numel() for p in trainable_params):,}")
print(f"Learning rate      : 2e-4")
print(f"Epochs             : {EPOCHS3}")
print("=" * 70)

total_start = time.time()

for epoch in range(1, EPOCHS3 + 1):

    model3.train()

    epoch_start = time.time()
    running_loss = 0.0
    valid_tokens = 0

    for step, batch in enumerate(
        a4_train_loader_exp2,
        start=1
    ):

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        labels = batch["labels"].to(
            DEVICE,
            non_blocking=True
        )

        optimizer3.zero_grad(set_to_none=True)

        logits = model3(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion3(
            logits.reshape(-1, 25),
            labels.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0
        )

        optimizer3.step()

        running_loss += loss.item()

        valid_tokens += (
            labels != -100
        ).sum().item()

        if step % 20 == 0 or step == len(a4_train_loader_exp2):

            elapsed = time.time() - epoch_start

            print(
                f"Epoch {epoch}/{EPOCHS3} | "
                f"Step {step:3d}/{len(a4_train_loader_exp2)} | "
                f"Loss {loss.item():.4f} | "
                f"Time {elapsed:.1f}s"
            )

    epoch_time = time.time() - epoch_start

    avg_loss = (
        running_loss /
        len(a4_train_loader_exp2)
    )

    history3.append({
        "epoch": epoch,
        "train_loss": avg_loss,
        "time_sec": epoch_time,
        "valid_tokens": valid_tokens
    })

    checkpoint_path = os.path.join(
        EXP3_DIR,
        f"a4_ner_exp3_lora_epoch{epoch}.pt"
    )

    torch.save(
        {
            "experiment": "A4_LoRA",
            "epoch": epoch,
            "model_state_dict": model3.state_dict(),
            "optimizer_state_dict": optimizer3.state_dict(),
            "train_loss": avg_loss,
            "history": history3,
            "train_chunks": len(a4_train_only_chunks),
            "val_chunks": len(a4_val_loader.dataset),
            "lora_rank": 8,
            "lora_alpha": 16,
            "lora_dropout": 0.05,
        },
        checkpoint_path
    )

    print(
        f"✓ Epoch {epoch} complete | "
        f"Avg loss {avg_loss:.4f} | "
        f"{epoch_time:.1f}s"
    )

total_time = time.time() - total_start

print("\n" + "=" * 70)
print("EXPERIMENT 3 TRAINING COMPLETE")
print("=" * 70)

for h in history3:
    print(
        f"Epoch {h['epoch']}: "
        f"loss={h['train_loss']:.4f}, "
        f"time={h['time_sec']:.1f}s"
    )

print(f"\nTotal training time: {total_time:.1f}s")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

print(f"\nCheckpoints saved in:")
print(EXP3_DIR)

In [ ]:
# ============================================================
# CELL 42 — EXPERIMENT 3: FOLD-1 STRICT EVALUATION
# ============================================================

import torch
from collections import defaultdict

print("=" * 70)
print("EXPERIMENT 3 — LORA STRICT EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Load final LoRA checkpoint
# ------------------------------------------------------------

ckpt_path = (
    "/kaggle/working/phase2_exp3/"
    "a4_ner_exp3_lora_epoch3.pt"
)

checkpoint = torch.load(
    ckpt_path,
    map_location=DEVICE,
    weights_only=False
)

model3.load_state_dict(
    checkpoint["model_state_dict"]
)

model3.eval()

print("Checkpoint loaded:")
print(ckpt_path)

# ------------------------------------------------------------
# Document-level prediction storage
# ------------------------------------------------------------

predictions3 = defaultdict(set)
gold3 = defaultdict(set)

total_chunks = 0
total_tokens = 0

with torch.no_grad():

    for batch in a4_val_loader:

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        logits = model3(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pred_ids = logits.argmax(
            dim=-1
        ).cpu().tolist()

        input_ids_cpu = batch["input_ids"].tolist()
        masks_cpu = batch["attention_mask"].tolist()

        batch_size = len(batch["doc_id"])

        for i in range(batch_size):

            doc_id = batch["doc_id"][i]
            offsets = batch["offsets"][i]

            valid_len = int(
                masks_cpu[i].sum()
            )

            pred = pred_ids[i][:valid_len]
            offsets_i = offsets[:valid_len]

            # BIO -> character spans
            spans = bio_to_token_spans(
                pred,
                offsets_i,
                text=fold1_text[doc_id]
            )

            # Normalize whitespace boundaries
            for start, end, label in spans:

                start, end = normalize_span_boundaries(
                    start,
                    end,
                    label,
                    fold1_text[doc_id]
                )

                if start < end:
                    predictions3[doc_id].add(
                        (start, end, label)
                    )

            # Gold spans
            for span in batch["gold_spans"][i]:
                gold3[doc_id].add(
                    tuple(span[:3])
                )

            total_chunks += 1
            total_tokens += valid_len

# ------------------------------------------------------------
# Official strict macro F1
# ------------------------------------------------------------

result3 = strict_macro_f1(
    predictions3,
    gold3
)

print("\n" + "=" * 70)
print("EXPERIMENT 3 — RESULTS")
print("=" * 70)

print(
    f"Strict Macro Precision : "
    f"{result3['macro_precision']:.4f}"
)

print(
    f"Strict Macro Recall    : "
    f"{result3['macro_recall']:.4f}"
)

print(
    f"Strict Macro F1        : "
    f"{result3['macro_f1']:.4f}"
)

print("\nPer-label results:")

for label in ENTITY_LABELS:

    p = result3["per_label"][label]["precision"]
    r = result3["per_label"][label]["recall"]
    f = result3["per_label"][label]["f1"]

    print(
        f"{label:12s} "
        f"P={p:.4f} "
        f"R={r:.4f} "
        f"F1={f:.4f}"
    )

print("\nEvaluation details:")
print(f"Validation chunks : {total_chunks}")
print(f"Validation tokens : {total_tokens}")

print("\n✓ Experiment 3 evaluation complete.")

In [ ]:
# ============================================================
# CELL 42 — EXPERIMENT 3 LORA FINAL EVALUATION
# ============================================================

import torch
from collections import defaultdict

print("=" * 70)
print("EXPERIMENT 3 — LORA STRICT EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

ckpt_path = (
    "/kaggle/working/phase2_exp3/"
    "a4_ner_exp3_lora_epoch3.pt"
)

checkpoint = torch.load(
    ckpt_path,
    map_location=DEVICE,
    weights_only=False
)

model3.load_state_dict(
    checkpoint["model_state_dict"]
)

model3.eval()

print("✓ LoRA checkpoint loaded")
print(f"Checkpoint: {ckpt_path}")

# ------------------------------------------------------------
# Prediction storage
# ------------------------------------------------------------

predictions3 = defaultdict(set)
gold3 = defaultdict(set)

total_chunks = 0
total_tokens = 0

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

with torch.no_grad():

    for batch_idx, batch in enumerate(a4_val_loader):

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        logits = model3(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pred_ids = logits.argmax(
            dim=-1
        ).cpu().tolist()

        masks_cpu = batch["attention_mask"].tolist()

        for i, doc_id in enumerate(batch["doc_id"]):

            valid_len = int(
                sum(masks_cpu[i])
            )

            pred = pred_ids[i][:valid_len]

            offsets_i = batch["offsets"][i][:valid_len]

            text = fold1_text[doc_id]

            # ------------------------------------------------
            # BIO -> character spans
            # ------------------------------------------------

            spans = bio_to_token_spans(
                pred,
                offsets_i,
                text=text
            )

            # ------------------------------------------------
            # Normalize whitespace boundaries
            # ------------------------------------------------

            for start, end, label in spans:

                start, end = normalize_span_boundaries(
                    start,
                    end,
                    label,
                    text
                )

                if start < end:

                    predictions3[doc_id].add(
                        (start, end, label)
                    )

            # ------------------------------------------------
            # Gold spans
            # ------------------------------------------------

            for span in batch["gold_spans"][i]:

                gold3[doc_id].add(
                    tuple(span[:3])
                )

            total_chunks += 1
            total_tokens += valid_len

        # Progress
        if (
            batch_idx + 1 == 1
            or (batch_idx + 1) % 20 == 0
            or batch_idx + 1 == len(a4_val_loader)
        ):
            print(
                f"Evaluated "
                f"{batch_idx + 1}/{len(a4_val_loader)} batches"
            )

# ------------------------------------------------------------
# Strict Macro-F1
# ------------------------------------------------------------

result3 = strict_macro_f1(
    predictions3,
    gold3
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXPERIMENT 3 — FINAL RESULTS")
print("=" * 70)

print(
    f"Strict Macro Precision : "
    f"{result3['macro_precision']:.4f}"
)

print(
    f"Strict Macro Recall    : "
    f"{result3['macro_recall']:.4f}"
)

print(
    f"Strict Macro F1        : "
    f"{result3['macro_f1']:.4f}"
)

print("\nPer-label Strict F1:")
print("-" * 70)

for label in ENTITY_LABELS:

    p = result3["per_label"][label]["precision"]
    r = result3["per_label"][label]["recall"]
    f = result3["per_label"][label]["f1"]

    print(
        f"{label:12s} "
        f"P={p:.4f} "
        f"R={r:.4f} "
        f"F1={f:.4f}"
    )

print("\n" + "-" * 70)

print(f"Validation chunks : {total_chunks}")
print(f"Validation tokens : {total_tokens}")

print(
    f"Predicted spans   : "
    f"{sum(len(v) for v in predictions3.values())}"
)

print(
    f"Gold spans        : "
    f"{sum(len(v) for v in gold3.values())}"
)

print("\n✓ EXPERIMENT 3 EVALUATION COMPLETE")

In [ ]:
# ============================================================
# CELL 42A — CORRECT NERChunk VALIDATION COLLATE
# ============================================================

def a4_eval_collate_fn(batch):

    # NERChunk objects -> attribute access
    max_len = max(
        len(x.input_ids)
        for x in batch
    )

    batch_size = len(batch)

    input_ids = torch.zeros(
        (batch_size, max_len),
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (batch_size, max_len),
        dtype=torch.long
    )

    labels = torch.full(
        (batch_size, max_len),
        -100,
        dtype=torch.long
    )

    doc_ids = []
    offsets = []
    gold_spans = []

    for i, item in enumerate(batch):

        n = len(item.input_ids)

        input_ids[i, :n] = torch.tensor(
            item.input_ids,
            dtype=torch.long
        )

        attention_mask[i, :n] = 1

        labels[i, :n] = torch.tensor(
            item.bio_ids,
            dtype=torch.long
        )

        doc_ids.append(item.doc_id)
        offsets.append(item.offsets)
        gold_spans.append(item.gold_spans)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "doc_id": doc_ids,
        "offsets": offsets,
        "gold_spans": gold_spans,
    }


# Rebuild validation loader
a4_val_loader = torch.utils.data.DataLoader(
    a4_val_chunks,
    batch_size=4,
    shuffle=False,
    collate_fn=a4_eval_collate_fn,
    pin_memory=True
)


# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print("=" * 70)
print("VALIDATION LOADER — FINAL VERIFICATION")
print("=" * 70)

test_batch = next(iter(a4_val_loader))

print("Keys:", list(test_batch.keys()))
print("Batch size:", len(test_batch["doc_id"]))
print("Input shape:", tuple(test_batch["input_ids"].shape))
print("First doc:", test_batch["doc_id"][0])
print("First offsets:", len(test_batch["offsets"][0]))
print("First gold spans:", len(test_batch["gold_spans"][0]))

assert "doc_id" in test_batch
assert "offsets" in test_batch
assert "gold_spans" in test_batch

assert len(test_batch["doc_id"]) == 4

print("\n✓ NERChunk attribute access correct")
print("✓ doc_id preserved")
print("✓ offsets preserved")
print("✓ gold_spans preserved")
print("✓ Validation loader ready")

In [ ]:
# ============================================================
# CELL 42A — DEFINITIVE VALIDATION LOADER FIX
# ============================================================

def a4_eval_collate_fn(batch):

    # NERChunk objects -> ATTRIBUTE access
    max_len = max(len(x.input_ids) for x in batch)

    batch_size = len(batch)

    input_ids = torch.zeros(
        (batch_size, max_len),
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (batch_size, max_len),
        dtype=torch.long
    )

    labels = torch.full(
        (batch_size, max_len),
        -100,
        dtype=torch.long
    )

    doc_ids = []
    offsets = []
    gold_spans = []

    for i, item in enumerate(batch):

        n = len(item.input_ids)

        input_ids[i, :n] = torch.tensor(
            item.input_ids,
            dtype=torch.long
        )

        attention_mask[i, :n] = 1

        labels[i, :n] = torch.tensor(
            item.bio_ids,
            dtype=torch.long
        )

        doc_ids.append(item.doc_id)
        offsets.append(item.offsets)
        gold_spans.append(item.gold_spans)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "doc_id": doc_ids,
        "offsets": offsets,
        "gold_spans": gold_spans,
    }


# IMPORTANT:
# Rebuild the DataLoader so it receives the NEW function.
a4_val_loader = torch.utils.data.DataLoader(
    a4_val_chunks,
    batch_size=4,
    shuffle=False,
    collate_fn=a4_eval_collate_fn,
    pin_memory=True
)


# ============================================================
# HARD VERIFICATION
# ============================================================

print("=" * 70)
print("DEFINITIVE VALIDATION LOADER VERIFICATION")
print("=" * 70)

print(
    "Active collate function:",
    a4_val_loader.collate_fn.__name__
)

test_batch = next(iter(a4_val_loader))

print("\nKeys:")
print(list(test_batch.keys()))

print("\nShapes:")
print("input_ids       :", tuple(test_batch["input_ids"].shape))
print("attention_mask  :", tuple(test_batch["attention_mask"].shape))
print("labels          :", tuple(test_batch["labels"].shape))

print("\nMetadata:")
print("Batch size      :", len(test_batch["doc_id"]))
print("First doc       :", test_batch["doc_id"][0])
print("First offsets   :", len(test_batch["offsets"][0]))
print("First gold spans:", len(test_batch["gold_spans"][0]))

assert a4_val_loader.collate_fn is a4_eval_collate_fn
assert "doc_id" in test_batch
assert "offsets" in test_batch
assert "gold_spans" in test_batch

print("\n" + "=" * 70)
print("✓ OLD COLLATE FUNCTION REMOVED")
print("✓ NEW ATTRIBUTE-BASED COLLATE FUNCTION ACTIVE")
print("✓ NERChunk HANDLING CORRECT")
print("✓ VALIDATION LOADER READY")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 42 — EXPERIMENT 3 LORA STRICT EVALUATION
# ============================================================

from collections import defaultdict

print("=" * 70)
print("EXPERIMENT 3 — A4 + LoRA STRICT EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Load Epoch-3 checkpoint
# ------------------------------------------------------------

ckpt_path = (
    "/kaggle/working/phase2_exp3/"
    "a4_ner_exp3_lora_epoch3.pt"
)

checkpoint = torch.load(
    ckpt_path,
    map_location=DEVICE,
    weights_only=False
)

model3.load_state_dict(
    checkpoint["model_state_dict"]
)

model3.eval()

print("✓ Checkpoint loaded")

# ------------------------------------------------------------
# Prediction / gold storage
# ------------------------------------------------------------

predictions3 = defaultdict(set)
gold3 = defaultdict(set)

total_chunks = 0
total_tokens = 0

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

with torch.no_grad():

    for batch_idx, batch in enumerate(a4_val_loader):

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        logits = model3(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pred_ids = logits.argmax(
            dim=-1
        ).cpu().tolist()

        masks_cpu = batch["attention_mask"].tolist()

        for i, doc_id in enumerate(batch["doc_id"]):

            valid_len = int(sum(masks_cpu[i]))

            pred = pred_ids[i][:valid_len]

            offsets_i = batch["offsets"][i][:valid_len]

            text = fold1_text[doc_id]

            # BIO -> character spans
            spans = bio_to_token_spans(
                pred,
                offsets_i,
                text=text
            )

            # Normalize boundaries
            for start, end, label in spans:

                # IMPORTANT: function returns 3 values
                start, end, label = normalize_span_boundaries(
                    start,
                    end,
                    label,
                    text
                )

                if start < end:

                    predictions3[doc_id].add(
                        (start, end, label)
                    )

            # Gold spans
            for span in batch["gold_spans"][i]:

                gold3[doc_id].add(
                    tuple(span[:3])
                )

            total_chunks += 1
            total_tokens += valid_len

        # Progress
        if (
            batch_idx == 0
            or (batch_idx + 1) % 20 == 0
            or batch_idx + 1 == len(a4_val_loader)
        ):

            print(
                f"Evaluated "
                f"{batch_idx + 1}/{len(a4_val_loader)} batches"
            )

# ------------------------------------------------------------
# Strict Macro-F1
# ------------------------------------------------------------

result3 = strict_macro_f1(
    predictions3,
    gold3
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXPERIMENT 3 — FINAL RESULTS")
print("=" * 70)

print(
    f"Strict Macro Precision : "
    f"{result3['macro_precision']:.4f}"
)

print(
    f"Strict Macro Recall    : "
    f"{result3['macro_recall']:.4f}"
)

print(
    f"Strict Macro F1        : "
    f"{result3['macro_f1']:.4f}"
)

print("\nPer-label Strict Results:")
print("-" * 70)

for label in ENTITY_LABELS:

    p = result3["per_label"][label]["precision"]
    r = result3["per_label"][label]["recall"]
    f = result3["per_label"][label]["f1"]

    print(
        f"{label:12s} "
        f"P={p:.4f} "
        f"R={r:.4f} "
        f"F1={f:.4f}"
    )

print("\n" + "-" * 70)

print(f"Validation chunks : {total_chunks}")
print(f"Validation tokens : {total_tokens}")

print(
    f"Predicted spans   : "
    f"{sum(len(v) for v in predictions3.values())}"
)

print(
    f"Gold spans        : "
    f"{sum(len(v) for v in gold3.values())}"
)

print("\n✓ EXPERIMENT 3 EVALUATION COMPLETE")

In [ ]:
# ============================================================
# CELL 43 — EXPERIMENT 4: LORMA
# ============================================================

import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 70)
print("EXPERIMENT 4 — A4 + LORMA")
print("=" * 70)


# ------------------------------------------------------------
# LORMA Linear Layer
# ------------------------------------------------------------
# Low-rank residual adaptation:
#
# W' = W + B @ A
#
# Base W remains frozen.
# A and B are trainable.
# ------------------------------------------------------------

class LORMALinear(nn.Module):

    def __init__(
        self,
        base_layer,
        rank=8,
        alpha=16,
        dropout=0.05
    ):
        super().__init__()

        self.base = base_layer

        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features

        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        self.dropout = nn.Dropout(dropout)

        # Trainable low-rank matrices
        self.lorma_A = nn.Parameter(
            torch.empty(
                rank,
                self.in_features
            )
        )

        self.lorma_B = nn.Parameter(
            torch.zeros(
                self.out_features,
                rank
            )
        )

        # Same initialization principle as low-rank adaptation:
        # A random, B initially zero.
        nn.init.kaiming_uniform_(
            self.lorma_A,
            a=5**0.5
        )

        nn.init.zeros_(
            self.lorma_B
        )

        # Freeze original layer
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):

        base_output = self.base(x)

        low_rank = (
            self.dropout(x)
            @ self.lorma_A.T
            @ self.lorma_B.T
        )

        return (
            base_output
            + self.scaling * low_rank
        )


# ------------------------------------------------------------
# Build independent LORMA encoder
# ------------------------------------------------------------

lorma_encoder = copy.deepcopy(
    encoder
)


# ------------------------------------------------------------
# Inject LORMA into Q and V projections
# ------------------------------------------------------------

lorma_targets = []

for layer_idx, layer in enumerate(
    lorma_encoder.layers
):

    layer.attn.q_proj = LORMALinear(
        layer.attn.q_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )

    layer.attn.v_proj = LORMALinear(
        layer.attn.v_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )

    lorma_targets.append(
        f"layer{layer_idx}.q_proj"
    )

    lorma_targets.append(
        f"layer{layer_idx}.v_proj"
    )


# ------------------------------------------------------------
# Build NER model
# ------------------------------------------------------------

model4 = A4NER(
    encoder=lorma_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(DEVICE)


# ------------------------------------------------------------
# Freeze everything first
# ------------------------------------------------------------

for p in model4.parameters():
    p.requires_grad = False


# Enable only LORMA parameters
for name, p in model4.named_parameters():

    if (
        "lorma_A" in name
        or "lorma_B" in name
        or "classifier" in name
    ):
        p.requires_grad = True


# ------------------------------------------------------------
# Parameter report
# ------------------------------------------------------------

total_params4 = sum(
    p.numel()
    for p in model4.parameters()
)

trainable_params4 = sum(
    p.numel()
    for p in model4.parameters()
    if p.requires_grad
)

print("\nLORMA target modules:")
for x in lorma_targets:
    print(" ", x)

print("\nParameter statistics:")
print(
    f"Total parameters    : {total_params4:,}"
)

print(
    f"Trainable parameters: {trainable_params4:,}"
)

print(
    f"Trainable percentage : "
    f"{100 * trainable_params4 / total_params4:.2f}%"
)

print("\n✓ LORMA model constructed")

In [ ]:
# ============================================================
# CELL 43A — VERIFY A4 ENCODER STRUCTURE
# ============================================================

print("=" * 70)
print("A4 ENCODER STRUCTURE")
print("=" * 70)

print("Encoder class:", type(encoder).__name__)

print("\nTop-level modules:")
for name, module in encoder.named_children():
    print(f"  {name}: {type(module).__name__}")

print("\nEncoder layer count:")
print(len(encoder.encoder_layers))

print("\nFirst encoder layer:")
print(type(encoder.encoder_layers[0]).__name__)

print("\nAttention modules:")
print(type(encoder.encoder_layers[0].attn).__name__)

print("\nAttention projections:")

for name, module in encoder.encoder_layers[0].attn.named_children():
    print(f"  {name}: {type(module).__name__}")

print("\n✓ A4 structure verified")

In [ ]:
# ============================================================
# CELL 44 — EXPERIMENT 4: A4 + LORMA MODEL
# ============================================================

import copy
import torch
import torch.nn as nn

print("=" * 70)
print("EXPERIMENT 4 — A4 + LORMA")
print("=" * 70)


# ------------------------------------------------------------
# LORMA linear adaptation layer
# ------------------------------------------------------------

class LORMALinear(nn.Module):

    def __init__(
        self,
        base_layer,
        rank=8,
        alpha=16,
        dropout=0.05
    ):
        super().__init__()

        self.base = base_layer

        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features

        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        self.lorma_dropout = nn.Dropout(dropout)

        self.lorma_A = nn.Parameter(
            torch.empty(
                rank,
                self.in_features
            )
        )

        self.lorma_B = nn.Parameter(
            torch.zeros(
                self.out_features,
                rank
            )
        )

        nn.init.kaiming_uniform_(
            self.lorma_A,
            a=5**0.5
        )

        nn.init.zeros_(
            self.lorma_B
        )

        # Freeze original projection
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):

        base_output = self.base(x)

        adapted = (
            self.lorma_dropout(x)
            @ self.lorma_A.T
            @ self.lorma_B.T
        )

        return (
            base_output
            + self.scaling * adapted
        )


# ------------------------------------------------------------
# Independent copy of validated A4 encoder
# ------------------------------------------------------------

lorma_encoder = copy.deepcopy(encoder)


# ------------------------------------------------------------
# Inject LORMA into Q and V projections
# ------------------------------------------------------------

for layer_idx, layer in enumerate(
    lorma_encoder.encoder_layers
):

    layer.attn.q_proj = LORMALinear(
        layer.attn.q_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )

    layer.attn.v_proj = LORMALinear(
        layer.attn.v_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )

print("\n✓ LORMA injected into:")
print("  6 × q_proj")
print("  6 × v_proj")


# ------------------------------------------------------------
# Build NER model
# ------------------------------------------------------------

model4 = A4NER(
    encoder=lorma_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(DEVICE)


# ------------------------------------------------------------
# Freeze entire model
# ------------------------------------------------------------

for p in model4.parameters():
    p.requires_grad = False


# ------------------------------------------------------------
# Enable only LORMA + classifier
# ------------------------------------------------------------

for name, p in model4.named_parameters():

    if (
        "lorma_A" in name
        or "lorma_B" in name
        or "classifier" in name
    ):
        p.requires_grad = True


# ------------------------------------------------------------
# Parameter statistics
# ------------------------------------------------------------

total_params4 = sum(
    p.numel()
    for p in model4.parameters()
)

trainable_params4 = sum(
    p.numel()
    for p in model4.parameters()
    if p.requires_grad
)

print("\n" + "=" * 70)
print("LORMA PARAMETER REPORT")
print("=" * 70)

print(
    f"Total parameters    : {total_params4:,}"
)

print(
    f"Trainable parameters: {trainable_params4:,}"
)

print(
    f"Trainable percentage : "
    f"{100 * trainable_params4 / total_params4:.2f}%"
)

print("\nTrainable tensors:")

for name, p in model4.named_parameters():

    if p.requires_grad:

        print(
            f"  {name:55s} "
            f"{tuple(p.shape)}"
        )

print("\n✓ LORMA model construction complete")

In [ ]:
# ============================================================
# CELL 45 — EXPERIMENT 4: LORMA SANITY TEST
# ============================================================

import torch
import torch.nn as nn

print("=" * 70)
print("EXPERIMENT 4 — LORMA SANITY TEST")
print("=" * 70)

model4.train()

optimizer4_test = torch.optim.AdamW(
    [
        p for p in model4.parameters()
        if p.requires_grad
    ],
    lr=2e-4,
    weight_decay=0.01
)

criterion4_test = nn.CrossEntropyLoss(
    ignore_index=-100
)

# Take one real Fold-2 batch
batch = next(iter(a4_train_loader_exp2))

input_ids = batch["input_ids"].to(
    DEVICE,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    DEVICE,
    non_blocking=True
)

labels = batch["labels"].to(
    DEVICE,
    non_blocking=True
)

optimizer4_test.zero_grad(set_to_none=True)

# Forward
logits = model4(
    input_ids=input_ids,
    attention_mask=attention_mask
)

# Loss
loss = criterion4_test(
    logits.reshape(-1, 25),
    labels.reshape(-1)
)

# Backward
loss.backward()

# Gradient norm over trainable parameters
grad_norm_sq = 0.0
trainable_gradients = 0

for p in model4.parameters():

    if p.requires_grad and p.grad is not None:

        grad = p.grad.detach()

        if not torch.isfinite(grad).all():
            raise RuntimeError(
                "Non-finite gradient detected!"
            )

        grad_norm_sq += (
            grad.norm().item() ** 2
        )

        trainable_gradients += 1

grad_norm = grad_norm_sq ** 0.5

# Finite logits
finite_logits = torch.isfinite(
    logits
).all().item()

finite_loss = torch.isfinite(
    loss
).item()

print(f"Loss               : {loss.item():.4f}")
print(f"Trainable tensors  : {trainable_gradients}")
print(f"Gradient norm      : {grad_norm:.6f}")
print(f"Finite logits      : {finite_logits}")
print(f"Finite loss        : {finite_loss}")

assert finite_logits
assert finite_loss
assert trainable_gradients == 26
assert grad_norm > 0

print("\n✓ LORMA FORWARD TEST PASSED")
print("✓ LORMA BACKWARD TEST PASSED")
print("✓ GRADIENT FLOW PASSED")

In [ ]:
# ============================================================
# CELL 46 — EXPERIMENT 4: A4 + LORMA TRAINING
# ============================================================

import os
import time

EXP4_DIR = "/kaggle/working/phase2_exp4"
os.makedirs(EXP4_DIR, exist_ok=True)

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

trainable_params4 = [
    p for p in model4.parameters()
    if p.requires_grad
]

optimizer4 = torch.optim.AdamW(
    trainable_params4,
    lr=2e-4,
    weight_decay=0.01
)

criterion4 = nn.CrossEntropyLoss(
    ignore_index=-100
)

EPOCHS4 = 3

history4 = []

print("=" * 70)
print("EXPERIMENT 4 — A4 + LORMA TRAINING")
print("=" * 70)

print(f"Train chunks       : {len(a4_train_only_chunks)}")
print(f"Train batches      : {len(a4_train_loader_exp2)}")
print(f"Validation chunks  : {len(a4_val_loader.dataset)}")
print(f"Trainable params   : {sum(p.numel() for p in trainable_params4):,}")
print(f"Learning rate      : 2e-4")
print(f"Epochs             : {EPOCHS4}")

total_start = time.time()

for epoch in range(1, EPOCHS4 + 1):

    model4.train()

    epoch_start = time.time()
    running_loss = 0.0
    valid_tokens = 0

    for step, batch in enumerate(
        a4_train_loader_exp2,
        start=1
    ):

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        labels = batch["labels"].to(
            DEVICE,
            non_blocking=True
        )

        optimizer4.zero_grad(set_to_none=True)

        logits = model4(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion4(
            logits.reshape(-1, 25),
            labels.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            trainable_params4,
            max_norm=1.0
        )

        optimizer4.step()

        running_loss += loss.item()

        valid_tokens += (
            labels != -100
        ).sum().item()

        if (
            step % 20 == 0
            or step == len(a4_train_loader_exp2)
        ):

            elapsed = time.time() - epoch_start

            print(
                f"Epoch {epoch}/{EPOCHS4} | "
                f"Step {step:3d}/{len(a4_train_loader_exp2)} | "
                f"Loss {loss.item():.4f} | "
                f"Time {elapsed:.1f}s"
            )

    epoch_time = time.time() - epoch_start

    avg_loss = (
        running_loss /
        len(a4_train_loader_exp2)
    )

    history4.append({
        "epoch": epoch,
        "train_loss": avg_loss,
        "time_sec": epoch_time,
        "valid_tokens": valid_tokens
    })

    checkpoint_path = os.path.join(
        EXP4_DIR,
        f"a4_ner_exp4_lorma_epoch{epoch}.pt"
    )

    torch.save(
        {
            "experiment": "A4_LORMA",
            "epoch": epoch,
            "model_state_dict": model4.state_dict(),
            "optimizer_state_dict": optimizer4.state_dict(),
            "train_loss": avg_loss,
            "history": history4,
            "train_chunks": len(a4_train_only_chunks),
            "val_chunks": len(a4_val_loader.dataset),
            "lorma_rank": 8,
            "lorma_alpha": 16,
            "lorma_dropout": 0.05,
        },
        checkpoint_path
    )

    print(
        f"✓ Epoch {epoch} complete | "
        f"Avg loss {avg_loss:.4f} | "
        f"{epoch_time:.1f}s"
    )

total_time = time.time() - total_start

print("\n" + "=" * 70)
print("EXPERIMENT 4 TRAINING COMPLETE")
print("=" * 70)

for h in history4:
    print(
        f"Epoch {h['epoch']}: "
        f"loss={h['train_loss']:.4f}, "
        f"time={h['time_sec']:.1f}s"
    )

print(f"\nTotal training time: {total_time:.1f}s")
print(
    f"Peak VRAM: "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)

print(f"\nCheckpoints saved in:")
print(EXP4_DIR)

In [ ]:
# ============================================================
# CELL 47 — EXPERIMENT 4 LORMA STRICT EVALUATION
# ============================================================

from collections import defaultdict

print("=" * 70)
print("EXPERIMENT 4 — A4 + LORMA STRICT EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Load Epoch-3 checkpoint
# ------------------------------------------------------------

ckpt_path = (
    "/kaggle/working/phase2_exp4/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

checkpoint = torch.load(
    ckpt_path,
    map_location=DEVICE,
    weights_only=False
)

model4.load_state_dict(
    checkpoint["model_state_dict"]
)

model4.eval()

print("✓ LORMA checkpoint loaded")

# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

predictions4 = defaultdict(set)
gold4 = defaultdict(set)

total_chunks = 0
total_tokens = 0

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

with torch.no_grad():

    for batch_idx, batch in enumerate(a4_val_loader):

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        logits = model4(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pred_ids = logits.argmax(
            dim=-1
        ).cpu().tolist()

        masks_cpu = batch["attention_mask"].tolist()

        for i, doc_id in enumerate(batch["doc_id"]):

            valid_len = int(
                sum(masks_cpu[i])
            )

            pred = pred_ids[i][:valid_len]

            offsets_i = batch["offsets"][i][:valid_len]

            text = fold1_text[doc_id]

            # BIO -> character spans
            spans = bio_to_token_spans(
                pred,
                offsets_i,
                text=text
            )

            # Normalize boundaries
            for start, end, label in spans:

                start, end, label = normalize_span_boundaries(
                    start,
                    end,
                    label,
                    text
                )

                if start < end:

                    predictions4[doc_id].add(
                        (start, end, label)
                    )

            # Gold
            for span in batch["gold_spans"][i]:

                gold4[doc_id].add(
                    tuple(span[:3])
                )

            total_chunks += 1
            total_tokens += valid_len

        if (
            batch_idx == 0
            or (batch_idx + 1) % 20 == 0
            or batch_idx + 1 == len(a4_val_loader)
        ):

            print(
                f"Evaluated "
                f"{batch_idx + 1}/{len(a4_val_loader)} batches"
            )

# ------------------------------------------------------------
# Strict Macro-F1
# ------------------------------------------------------------

result4 = strict_macro_f1(
    predictions4,
    gold4
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXPERIMENT 4 — FINAL RESULTS")
print("=" * 70)

print(
    f"Strict Macro Precision : "
    f"{result4['macro_precision']:.4f}"
)

print(
    f"Strict Macro Recall    : "
    f"{result4['macro_recall']:.4f}"
)

print(
    f"Strict Macro F1        : "
    f"{result4['macro_f1']:.4f}"
)

print("\nPer-label Strict Results:")
print("-" * 70)

for label in ENTITY_LABELS:

    p = result4["per_label"][label]["precision"]
    r = result4["per_label"][label]["recall"]
    f = result4["per_label"][label]["f1"]

    print(
        f"{label:12s} "
        f"P={p:.4f} "
        f"R={r:.4f} "
        f"F1={f:.4f}"
    )

print("\n" + "-" * 70)

print(f"Validation chunks : {total_chunks}")
print(f"Validation tokens : {total_tokens}")

print(
    f"Predicted spans   : "
    f"{sum(len(v) for v in predictions4.values())}"
)

print(
    f"Gold spans        : "
    f"{sum(len(v) for v in gold4.values())}"
)

print("\n✓ EXPERIMENT 4 EVALUATION COMPLETE")

In [ ]:
# ============================================================
# CELL 48 — EXPERIMENT 5: KNOWLEDGE DISTILLATION SETUP
# ============================================================

import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 70)
print("EXPERIMENT 5 — A4 + KNOWLEDGE DISTILLATION")
print("=" * 70)


# ============================================================
# 1. TEACHER
# ============================================================

teacher4 = copy.deepcopy(model4)

teacher4.eval()

for p in teacher4.parameters():
    p.requires_grad = False


print("✓ Teacher created from trained A4 + LORMA")


# ============================================================
# 2. STUDENT
# ============================================================
# Fresh A4 encoder from the original Phase-1 checkpoint.
# No LoRA/LORMA parameters in the student.
# ============================================================

student_encoder = copy.deepcopy(encoder)

model5 = A4NER(
    encoder=student_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(DEVICE)


# ============================================================
# Student trainable parameters
# ============================================================

for p in model5.parameters():
    p.requires_grad = True


student_total = sum(
    p.numel()
    for p in model5.parameters()
)

student_trainable = sum(
    p.numel()
    for p in model5.parameters()
    if p.requires_grad
)


print("\nStudent:")
print(
    f"Total parameters    : {student_total:,}"
)

print(
    f"Trainable parameters: {student_trainable:,}"
)


# ============================================================
# 3. KD CONFIGURATION
# ============================================================

KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5

print("\nKD configuration:")
print(
    f"Temperature T : {KD_TEMPERATURE}"
)

print(
    f"KD alpha      : {KD_ALPHA}"
)

print(
    "Hard-label weight : "
    f"{1.0 - KD_ALPHA}"
)


# ============================================================
# 4. KD LOSS
# ============================================================

def kd_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=2.0,
    alpha=0.5
):

    # Hard-label CE
    hard_loss = F.cross_entropy(
        student_logits.reshape(-1, 25),
        labels.reshape(-1),
        ignore_index=-100
    )

    # Soft teacher targets
    student_log_probs = F.log_softmax(
        student_logits / temperature,
        dim=-1
    )

    teacher_probs = F.softmax(
        teacher_logits / temperature,
        dim=-1
    )

    # Only valid (non-padding) tokens
    valid_mask = (
        labels != -100
    )

    student_log_probs = student_log_probs[
        valid_mask
    ]

    teacher_probs = teacher_probs[
        valid_mask
    ]

    soft_loss = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean"
    ) * (
        temperature ** 2
    )

    total_loss = (
        (1.0 - alpha) * hard_loss
        + alpha * soft_loss
    )

    return (
        total_loss,
        hard_loss,
        soft_loss
    )


print("\n✓ KD loss function ready")
print("\n✓ Teacher frozen")
print("✓ Student trainable")
print("✓ KD setup complete")

In [ ]:
# ============================================================
# CELL 49 — EXPERIMENT 5: KD SANITY TEST
# ============================================================

import torch

print("=" * 70)
print("EXPERIMENT 5 — KNOWLEDGE DISTILLATION SANITY TEST")
print("=" * 70)

student = model5
teacher = teacher4

student.train()
teacher.eval()

# ------------------------------------------------------------
# Optimizer for sanity test
# ------------------------------------------------------------

optimizer5_test = torch.optim.AdamW(
    [
        p for p in student.parameters()
        if p.requires_grad
    ],
    lr=2e-5,
    weight_decay=0.01
)

# ------------------------------------------------------------
# Real Fold-2 batch
# ------------------------------------------------------------

batch = next(iter(a4_train_loader_exp2))

input_ids = batch["input_ids"].to(
    DEVICE,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    DEVICE,
    non_blocking=True
)

labels = batch["labels"].to(
    DEVICE,
    non_blocking=True
)

optimizer5_test.zero_grad(set_to_none=True)

# ------------------------------------------------------------
# Teacher forward — NO gradients
# ------------------------------------------------------------

with torch.no_grad():

    teacher_logits = teacher(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

# ------------------------------------------------------------
# Student forward
# ------------------------------------------------------------

student_logits = student(
    input_ids=input_ids,
    attention_mask=attention_mask
)

# ------------------------------------------------------------
# KD loss
# ------------------------------------------------------------

total_loss, hard_loss, soft_loss = kd_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=KD_TEMPERATURE,
    alpha=KD_ALPHA
)

# ------------------------------------------------------------
# Backward
# ------------------------------------------------------------

total_loss.backward()

# ------------------------------------------------------------
# Gradient checks
# ------------------------------------------------------------

grad_norm_sq = 0.0
trainable_gradients = 0

for p in student.parameters():

    if p.requires_grad and p.grad is not None:

        grad = p.grad.detach()

        if not torch.isfinite(grad).all():
            raise RuntimeError(
                "Non-finite student gradient detected!"
            )

        grad_norm_sq += (
            grad.norm().item() ** 2
        )

        trainable_gradients += 1

grad_norm = grad_norm_sq ** 0.5

# ------------------------------------------------------------
# Finite checks
# ------------------------------------------------------------

finite_student_logits = torch.isfinite(
    student_logits
).all().item()

finite_teacher_logits = torch.isfinite(
    teacher_logits
).all().item()

finite_total_loss = torch.isfinite(
    total_loss
).item()

finite_hard_loss = torch.isfinite(
    hard_loss
).item()

finite_soft_loss = torch.isfinite(
    soft_loss
).item()

# ------------------------------------------------------------
# Verify teacher stayed frozen
# ------------------------------------------------------------

teacher_has_grad = any(
    p.grad is not None
    for p in teacher.parameters()
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print(f"Total KD loss      : {total_loss.item():.4f}")
print(f"Hard CE loss       : {hard_loss.item():.4f}")
print(f"Soft KD loss       : {soft_loss.item():.4f}")
print(f"Trainable tensors  : {trainable_gradients}")
print(f"Student grad norm  : {grad_norm:.6f}")
print(f"Student logits     : {finite_student_logits}")
print(f"Teacher logits     : {finite_teacher_logits}")
print(f"Total loss finite  : {finite_total_loss}")
print(f"Hard loss finite   : {finite_hard_loss}")
print(f"Soft loss finite   : {finite_soft_loss}")
print(f"Teacher gradients  : {teacher_has_grad}")

assert finite_student_logits
assert finite_teacher_logits
assert finite_total_loss
assert finite_hard_loss
assert finite_soft_loss

assert trainable_gradients > 0
assert grad_norm > 0

assert not teacher_has_grad

print("\n✓ KD FORWARD TEST PASSED")
print("✓ KD LOSS TEST PASSED")
print("✓ STUDENT BACKWARD TEST PASSED")
print("✓ TEACHER REMAINED FROZEN")

In [ ]:
# ============================================================
# CELL 50 — EXPERIMENT 5: A4 + KD TRAINING
# ============================================================

import os
import time

EXP5_DIR = "/kaggle/working/phase2_exp5"
os.makedirs(EXP5_DIR, exist_ok=True)

# ------------------------------------------------------------
# Student optimizer
# ------------------------------------------------------------

student = model5

optimizer5 = torch.optim.AdamW(
    [
        p for p in student.parameters()
        if p.requires_grad
    ],
    lr=2e-5,
    weight_decay=0.01
)

EPOCHS5 = 3

history5 = []

print("=" * 70)
print("EXPERIMENT 5 — A4 + KNOWLEDGE DISTILLATION TRAINING")
print("=" * 70)

print(f"Train chunks       : {len(a4_train_only_chunks)}")
print(f"Train batches      : {len(a4_train_loader_exp2)}")
print(f"Validation chunks  : {len(a4_val_loader.dataset)}")
print(f"Student parameters : {student_trainable:,}")
print(f"Learning rate      : 2e-5")
print(f"Temperature        : {KD_TEMPERATURE}")
print(f"KD alpha           : {KD_ALPHA}")
print(f"Epochs             : {EPOCHS5}")

total_start = time.time()

for epoch in range(1, EPOCHS5 + 1):

    student.train()
    teacher.eval()

    epoch_start = time.time()

    running_total = 0.0
    running_hard = 0.0
    running_soft = 0.0

    valid_tokens = 0

    for step, batch in enumerate(
        a4_train_loader_exp2,
        start=1
    ):

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        labels = batch["labels"].to(
            DEVICE,
            non_blocking=True
        )

        optimizer5.zero_grad(set_to_none=True)

        # ----------------------------------------------------
        # Teacher
        # ----------------------------------------------------

        with torch.no_grad():

            teacher_logits = teacher(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # ----------------------------------------------------
        # Student
        # ----------------------------------------------------

        student_logits = student(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # ----------------------------------------------------
        # KD loss
        # ----------------------------------------------------

        total_loss, hard_loss, soft_loss = kd_loss(
            student_logits,
            teacher_logits,
            labels,
            temperature=KD_TEMPERATURE,
            alpha=KD_ALPHA
        )

        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            max_norm=1.0
        )

        optimizer5.step()

        running_total += total_loss.item()
        running_hard += hard_loss.item()
        running_soft += soft_loss.item()

        valid_tokens += (
            labels != -100
        ).sum().item()

        if (
            step % 20 == 0
            or step == len(a4_train_loader_exp2)
        ):

            elapsed = time.time() - epoch_start

            print(
                f"Epoch {epoch}/{EPOCHS5} | "
                f"Step {step:3d}/{len(a4_train_loader_exp2)} | "
                f"Total {total_loss.item():.4f} | "
                f"Hard {hard_loss.item():.4f} | "
                f"Soft {soft_loss.item():.4f} | "
                f"Time {elapsed:.1f}s"
            )

    epoch_time = time.time() - epoch_start

    avg_total = (
        running_total /
        len(a4_train_loader_exp2)
    )

    avg_hard = (
        running_hard /
        len(a4_train_loader_exp2)
    )

    avg_soft = (
        running_soft /
        len(a4_train_loader_exp2)
    )

    history5.append({
        "epoch": epoch,
        "total_loss": avg_total,
        "hard_loss": avg_hard,
        "soft_loss": avg_soft,
        "time_sec": epoch_time,
        "valid_tokens": valid_tokens
    })

    checkpoint_path = os.path.join(
        EXP5_DIR,
        f"a4_ner_exp5_kd_epoch{epoch}.pt"
    )

    torch.save(
        {
            "experiment": "A4_KD",
            "epoch": epoch,
            "model_state_dict": student.state_dict(),
            "optimizer_state_dict": optimizer5.state_dict(),
            "total_loss": avg_total,
            "hard_loss": avg_hard,
            "soft_loss": avg_soft,
            "history": history5,
            "train_chunks": len(a4_train_only_chunks),
            "val_chunks": len(a4_val_loader.dataset),
            "temperature": KD_TEMPERATURE,
            "alpha": KD_ALPHA
        },
        checkpoint_path
    )

    print(
        f"✓ Epoch {epoch} complete | "
        f"Total={avg_total:.4f} | "
        f"Hard={avg_hard:.4f} | "
        f"Soft={avg_soft:.4f} | "
        f"{epoch_time:.1f}s"
    )

total_time = time.time() - total_start

print("\n" + "=" * 70)
print("EXPERIMENT 5 KD TRAINING COMPLETE")
print("=" * 70)

for h in history5:

    print(
        f"Epoch {h['epoch']}: "
        f"total={h['total_loss']:.4f}, "
        f"hard={h['hard_loss']:.4f}, "
        f"soft={h['soft_loss']:.4f}, "
        f"time={h['time_sec']:.1f}s"
    )

print(f"\nTotal training time: {total_time:.1f}s")

print(
    f"Peak VRAM: "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)

print("\nCheckpoints saved in:")
print(EXP5_DIR)

In [ ]:
# ============================================================
# EXPERIMENT 5 — KD CLEAN FOLD-1 EVALUATION
# ============================================================

import os
import torch

EXP5_DIR = "/kaggle/working/phase2_exp5"

# Find saved checkpoints
ckpts = sorted([
    os.path.join(EXP5_DIR, f)
    for f in os.listdir(EXP5_DIR)
    if f.endswith(".pt")
])

print("=" * 70)
print("EXPERIMENT 5 — A4 + KNOWLEDGE DISTILLATION EVALUATION")
print("=" * 70)
print("Checkpoints found:")
for p in ckpts:
    print(" ", os.path.basename(p))

print("\nValidation chunks :", len(a4_val_chunks))
print("Validation batches:", len(a4_val_loader))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results_exp5 = []

for ckpt_path in ckpts:

    print("\n" + "-" * 70)
    print("Evaluating:", os.path.basename(ckpt_path))

    # Fresh model with original A4 encoder
    eval_encoder = copy.deepcopy(encoder)
    eval_model = A4NER(
        eval_encoder,
        num_labels=25,
        dropout=0.10,
        hidden_size=512
    )

    state = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False
    )

    # Support either direct state_dict or checkpoint dictionary
    if isinstance(state, dict) and "model_state_dict" in state:
        eval_model.load_state_dict(state["model_state_dict"], strict=True)
    else:
        eval_model.load_state_dict(state, strict=True)

    eval_model = eval_model.to(device)
    eval_model.eval()

    predictions = {}
    gold = {}

    total_tokens = 0
    total_pred_spans = 0
    total_gold_spans = 0

    with torch.no_grad():

        for batch in a4_val_loader:

            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(
                device, non_blocking=True
            )

            logits = eval_model(
                input_ids,
                attention_mask=attention_mask
            )

            pred_ids = logits.argmax(dim=-1).cpu().tolist()

            B = len(batch["doc_id"])

            for i in range(B):

                doc_id = batch["doc_id"][i]
                offsets = batch["offsets"][i]
                text = fold1_text[doc_id]

                # Remove padding offsets using attention mask
                valid_len = int(
                    batch["attention_mask"][i].sum().item()
                )

                offsets_valid = offsets[:valid_len]
                pred_valid = pred_ids[i][:valid_len]

                spans = bio_to_token_spans(
                    pred_valid,
                    offsets_valid,
                    text
                )

                # Normalize boundaries for deterministic
                # whitespace recovery.
                norm_spans = []

                for start, end, label in spans:
                    start, end, label = normalize_span_boundaries(
                        start, end, label, text
                    )
                    norm_spans.append((start, end, label))

                predictions.setdefault(doc_id, [])
                predictions[doc_id].extend(norm_spans)

                # Gold spans are already document-global.
                gold.setdefault(doc_id, [])
                gold[doc_id].extend(batch["gold_spans"][i])

                total_tokens += valid_len
                total_pred_spans += len(norm_spans)
                total_gold_spans += len(batch["gold_spans"][i])

    # Deduplicate overlapping chunk predictions/gold
    for doc_id in predictions:
        predictions[doc_id] = list(set(predictions[doc_id]))

    for doc_id in gold:
        gold[doc_id] = list(set(gold[doc_id]))

    metrics = strict_macro_f1(
        predictions,
        gold
    )

    print("\nRESULT")
    print("-" * 70)

    if isinstance(metrics, dict):

        for k, v in metrics.items():
            if isinstance(v, float):
                print(f"{k:25s}: {v:.4f}")
            else:
                print(f"{k:25s}: {v}")

    results_exp5.append({
        "checkpoint": os.path.basename(ckpt_path),
        "metrics": metrics
    })

    del eval_model
    torch.cuda.empty_cache()


print("\n" + "=" * 70)
print("EXPERIMENT 5 KD EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# EXPERIMENT 6 — A1 + FULL FINE-TUNING
# CELL 1: CHECKPOINT INSPECTION
# ============================================================

import os
import copy
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

A1_CKPT = "/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt"

print("=" * 70)
print("EXPERIMENT 6 — A1 + FULL FINE-TUNING")
print("=" * 70)

print("Checkpoint:", A1_CKPT)
print("Exists:", os.path.exists(A1_CKPT))

ckpt = torch.load(
    A1_CKPT,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(ckpt))

if isinstance(ckpt, dict):
    print("Checkpoint keys:")
    for k in ckpt.keys():
        print(" ", k)

a1_state = ckpt["model_state_dict"]

print("\nA1 state tensors:", len(a1_state))
print("\nFirst checkpoint keys:")

for k in list(a1_state.keys())[:120]:
    print(f"{k:60s} {tuple(a1_state[k].shape)}")

print("\nA1 parameter count:",
      sum(v.numel() for v in a1_state.values()))

print("=" * 70)

In [ ]:
# ============================================================
# EXPERIMENT 6 — A1 + FULL FINE-TUNING
# CELL 2: EXACT A1 RECONSTRUCTION
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# A1 components
# ------------------------------------------------------------

class A1TransformerBlock(nn.Module):
    def __init__(
        self,
        hidden_size=512,
        num_heads=8,
        ffn_size=2048,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(hidden_size)

        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, ffn_size),
            nn.GELU(),
            nn.Linear(ffn_size, hidden_size)
        )

    def forward(self, x, attention_mask=None):

        # Pre-norm attention
        h = self.norm1(x)

        # Causal mask
        seq_len = h.size(1)

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=h.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=causal_mask,
            need_weights=False
        )

        x = x + attn_out

        # Pre-norm FFN
        x = x + self.ffn(self.norm2(x))

        return x


class A1LM(nn.Module):

    def __init__(
        self,
        vocab_size=50000,
        hidden_size=512,
        num_layers=8,
        num_heads=8,
        ffn_size=2048,
        max_seq_len=2048,
        dropout=0.1
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            hidden_size
        )

        self.blocks = nn.ModuleList([
            A1TransformerBlock(
                hidden_size=hidden_size,
                num_heads=num_heads,
                ffn_size=ffn_size,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(hidden_size)

        self.lm_head = nn.Linear(
            hidden_size,
            vocab_size,
            bias=False
        )

    def forward(
        self,
        input_ids,
        attention_mask=None,
        return_hidden=True
    ):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        ).unsqueeze(0)

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        for block in self.blocks:
            x = block(
                x,
                attention_mask=attention_mask
            )

        x = self.final_norm(x)

        if return_hidden:
            return x

        return self.lm_head(x)


# ------------------------------------------------------------
# Build and STRICTLY load Phase-1 A1 checkpoint
# ------------------------------------------------------------

a1_encoder = A1LM(
    vocab_size=50000,
    hidden_size=512,
    num_layers=8,
    num_heads=8,
    ffn_size=2048,
    max_seq_len=2048,
    dropout=0.1
)

missing, unexpected = a1_encoder.load_state_dict(
    a1_state,
    strict=False
)

print("=" * 70)
print("A1 RECONSTRUCTION")
print("=" * 70)

print("Missing keys   :", missing)
print("Unexpected keys:", unexpected)

a1_params = sum(
    p.numel() for p in a1_encoder.parameters()
)

print("Model parameters:", f"{a1_params:,}")
print("Checkpoint params:", f"{sum(v.numel() for v in a1_state.values()):,}")

assert a1_params == 77_468_672
assert len(missing) == 0
assert len(unexpected) == 0

print("\n✓ Parameter count EXACT")
print("✓ Missing keys = 0")
print("✓ Unexpected keys = 0")
print("✓ A1 architecture reconstructed successfully")

In [ ]:
# ============================================================
# EXPERIMENT 6 — A1 + FULL FINE-TUNING
# CELL 3: REAL-DATA GPU SMOKE TEST
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("A1 REAL-DATA GPU SMOKE TEST")
print("=" * 70)

print("Device:", device)

# Move A1 encoder to GPU
a1_encoder = a1_encoder.to(device)

# Use one real batch from the existing A4-compatible loader.
# The tokenizer/chunks are the same official Phase-2 data.
batch = next(iter(a4_train_loader_exp2))

input_ids = batch["input_ids"].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    device,
    non_blocking=True
)

print("Input shape :", tuple(input_ids.shape))
print("Valid tokens:", int(attention_mask.sum().item()))

# Forward pass
torch.cuda.reset_peak_memory_stats(device)

with torch.no_grad():

    start = time.time()

    hidden = a1_encoder(
        input_ids,
        attention_mask=attention_mask,
        return_hidden=True
    )

    elapsed = time.time() - start

print("\nOutput shape:", tuple(hidden.shape))
print("Forward time:", f"{elapsed:.3f}s")

print(
    "Finite output:",
    bool(torch.isfinite(hidden).all().item())
)

print(
    "Peak VRAM:",
    f"{torch.cuda.max_memory_allocated(device)/1024**3:.2f} GB"
)

assert hidden.shape[0] == input_ids.shape[0]
assert hidden.shape[1] == input_ids.shape[1]
assert hidden.shape[2] == 512
assert torch.isfinite(hidden).all()

print("\n✓ Real Phase-2 data forward passed")
print("✓ Shape correct")
print("✓ Output finite")
print("✓ A1 GPU smoke test PASSED")

# Cleanup
del hidden, input_ids, attention_mask, batch
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# EXPERIMENT 6 — A1 + FULL FINE-TUNING
# CELL 4: NER MODEL + BACKWARD SANITY TEST
# ============================================================

class A1NER(nn.Module):
    def __init__(
        self,
        encoder,
        num_labels=25,
        dropout=0.10,
        hidden_size=512
    ):
        super().__init__()

        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask=None
    ):
        x = self.encoder(
            input_ids,
            attention_mask=attention_mask,
            return_hidden=True
        )

        x = self.dropout(x)

        return self.classifier(x)


# ------------------------------------------------------------
# Build A1 NER
# ------------------------------------------------------------

model6 = A1NER(
    a1_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)

total_params = sum(
    p.numel() for p in model6.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model6.parameters()
    if p.requires_grad
)

print("=" * 70)
print("A1 NER SANITY TEST")
print("=" * 70)

print("Total parameters    :", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

assert total_params == 77_481_497
assert trainable_params == 77_481_497

# ------------------------------------------------------------
# Real training batch
# ------------------------------------------------------------

batch = next(iter(a4_train_loader_exp2))

input_ids = batch["input_ids"].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    device,
    non_blocking=True
)

labels = batch["labels"].to(
    device,
    non_blocking=True
)

optimizer6 = torch.optim.AdamW(
    model6.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

model6.train()

torch.cuda.reset_peak_memory_stats(device)

start = time.time()

logits = model6(
    input_ids,
    attention_mask=attention_mask
)

loss = F.cross_entropy(
    logits.view(-1, 25),
    labels.view(-1),
    ignore_index=-100
)

optimizer6.zero_grad(set_to_none=True)

loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    model6.parameters(),
    max_norm=1.0
)

optimizer6.step()

elapsed = time.time() - start

print("\nLoss        :", f"{loss.item():.4f}")
print("Logits      :", tuple(logits.shape))
print("Grad norm   :", f"{float(grad_norm):.4f}")
print("Time        :", f"{elapsed:.3f}s")
print(
    "Peak VRAM   :",
    f"{torch.cuda.max_memory_allocated(device)/1024**3:.2f} GB"
)

print(
    "Finite logits:",
    bool(torch.isfinite(logits).all().item())
)

print(
    "Finite loss:",
    bool(torch.isfinite(loss).item())
)

print(
    "Finite gradients:",
    all(
        p.grad is None or torch.isfinite(p.grad).all().item()
        for p in model6.parameters()
    )
)

assert logits.shape[-1] == 25
assert torch.isfinite(logits).all()
assert torch.isfinite(loss)
assert torch.isfinite(torch.tensor(float(grad_norm)))

print("\n✓ Forward passed")
print("✓ Loss passed")
print("✓ Backward passed")
print("✓ Optimizer step passed")
print("✓ Gradients finite")
print("✓ A1 NER sanity test PASSED")

# Cleanup batch tensors
del batch, input_ids, attention_mask, labels
del logits, loss
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# EXPERIMENT 6 — A1 + FULL FINE-TUNING
# CELL 5: CLEAN FOLD-2 TRAINING
# ============================================================

import os
import time
import torch

EXP6_DIR = "/kaggle/working/phase2_exp6"
os.makedirs(EXP6_DIR, exist_ok=True)

EPOCHS6 = 3
LR6 = 2e-5
WD6 = 0.01

optimizer6 = torch.optim.AdamW(
    model6.parameters(),
    lr=LR6,
    weight_decay=WD6
)

print("=" * 70)
print("EXPERIMENT 6 — A1 + FULL FINE-TUNING TRAINING")
print("=" * 70)

print("Train chunks       :", len(a4_train_only_chunks))
print("Train batches      :", len(a4_train_loader_exp2))
print("Validation chunks  :", len(a4_val_chunks))
print("Parameters         :", f"{trainable_params:,}")
print("Learning rate      :", LR6)
print("Weight decay       :", WD6)
print("Epochs             :", EPOCHS6)

epoch_results6 = []

overall_start = time.time()

for epoch in range(1, EPOCHS6 + 1):

    model6.train()

    epoch_start = time.time()

    running_loss = 0.0
    steps = 0

    for step, batch in enumerate(
        a4_train_loader_exp2,
        start=1
    ):

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        optimizer6.zero_grad(set_to_none=True)

        logits = model6(
            input_ids,
            attention_mask=attention_mask
        )

        loss = F.cross_entropy(
            logits.view(-1, 25),
            labels.view(-1),
            ignore_index=-100
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model6.parameters(),
            max_norm=1.0
        )

        optimizer6.step()

        running_loss += loss.item()
        steps += 1

        if step % 20 == 0 or step == len(a4_train_loader_exp2):

            elapsed = time.time() - epoch_start

            print(
                f"Epoch {epoch}/{EPOCHS6} | "
                f"Step {step:3d}/{len(a4_train_loader_exp2)} | "
                f"Loss {loss.item():.4f} | "
                f"Time {elapsed:.1f}s"
            )

        del input_ids, attention_mask, labels
        del logits, loss

    epoch_loss = running_loss / steps
    epoch_time = time.time() - epoch_start

    epoch_results6.append({
        "epoch": epoch,
        "loss": epoch_loss,
        "time": epoch_time
    })

    ckpt_path = os.path.join(
        EXP6_DIR,
        f"a1_ner_exp6_fullft_epoch{epoch}.pt"
    )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model6.state_dict(),
            "optimizer_state_dict": optimizer6.state_dict(),
            "train_loss": epoch_loss,
            "architecture": "A1",
            "technique": "full_finetuning",
        },
        ckpt_path
    )

    print(
        f"✓ Epoch {epoch} complete | "
        f"Loss={epoch_loss:.4f} | "
        f"{epoch_time:.1f}s"
    )

total_time6 = time.time() - overall_start

print("\n" + "=" * 70)
print("EXPERIMENT 6 A1 TRAINING COMPLETE")
print("=" * 70)

for r in epoch_results6:
    print(
        f"Epoch {r['epoch']}: "
        f"loss={r['loss']:.4f}, "
        f"time={r['time']:.1f}s"
    )

print(f"\nTotal training time: {total_time6:.1f}s")
print(
    "Peak VRAM:",
    f"{torch.cuda.max_memory_allocated(device)/1024**3:.2f} GB"
)

print("\nCheckpoints saved in:")
print(EXP6_DIR)

In [ ]:
# ============================================================
# EXPERIMENT 6 — A1 + FULL FINE-TUNING
# CELL 6: CLEAN FOLD-1 EVALUATION
# ============================================================

import os
import copy
import torch

EXP6_DIR = "/kaggle/working/phase2_exp6"

ckpts6 = sorted([
    os.path.join(EXP6_DIR, f)
    for f in os.listdir(EXP6_DIR)
    if f.endswith(".pt")
])

print("=" * 70)
print("EXPERIMENT 6 — A1 CLEAN FOLD-1 EVALUATION")
print("=" * 70)

print("Checkpoints found:")
for p in ckpts6:
    print(" ", os.path.basename(p))

print("\nValidation chunks :", len(a4_val_chunks))
print("Validation batches:", len(a4_val_loader))

results_exp6 = []

for ckpt_path in ckpts6:

    print("\n" + "-" * 70)
    print("Evaluating:", os.path.basename(ckpt_path))

    # Fresh exact A1 encoder
    eval_encoder = A1LM(
        vocab_size=50000,
        hidden_size=512,
        num_layers=8,
        num_heads=8,
        ffn_size=2048,
        max_seq_len=2048,
        dropout=0.1
    )

    eval_model = A1NER(
        eval_encoder,
        num_labels=25,
        dropout=0.10,
        hidden_size=512
    )

    state = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False
    )

    eval_model.load_state_dict(
        state["model_state_dict"],
        strict=True
    )

    eval_model = eval_model.to(device)
    eval_model.eval()

    predictions = {}
    gold = {}

    total_tokens = 0
    total_pred_spans = 0
    total_gold_spans = 0

    with torch.no_grad():

        for batch in a4_val_loader:

            input_ids = batch["input_ids"].to(
                device,
                non_blocking=True
            )

            attention_mask = batch["attention_mask"].to(
                device,
                non_blocking=True
            )

            logits = eval_model(
                input_ids,
                attention_mask=attention_mask
            )

            pred_ids = logits.argmax(
                dim=-1
            ).cpu().tolist()

            B = len(batch["doc_id"])

            for i in range(B):

                doc_id = batch["doc_id"][i]

                text = fold1_text[doc_id]

                valid_len = int(
                    batch["attention_mask"][i].sum().item()
                )

                offsets_valid = batch["offsets"][i][:valid_len]
                pred_valid = pred_ids[i][:valid_len]

                spans = bio_to_token_spans(
                    pred_valid,
                    offsets_valid,
                    text
                )

                norm_spans = []

                for start, end, label in spans:

                    start, end, label = normalize_span_boundaries(
                        start,
                        end,
                        label,
                        text
                    )

                    norm_spans.append(
                        (start, end, label)
                    )

                predictions.setdefault(
                    doc_id,
                    []
                )

                predictions[doc_id].extend(
                    norm_spans
                )

                gold.setdefault(
                    doc_id,
                    []
                )

                gold[doc_id].extend(
                    batch["gold_spans"][i]
                )

                total_tokens += valid_len
                total_pred_spans += len(norm_spans)
                total_gold_spans += len(
                    batch["gold_spans"][i]
                )

    # Deduplicate chunk-overlap predictions/gold
    for doc_id in predictions:
        predictions[doc_id] = list(
            set(predictions[doc_id])
        )

    for doc_id in gold:
        gold[doc_id] = list(
            set(gold[doc_id])
        )

    metrics = strict_macro_f1(
        predictions,
        gold
    )

    print("\nRESULT")
    print("-" * 70)

    for k, v in metrics.items():

        if isinstance(v, float):
            print(
                f"{k:25s}: {v:.4f}"
            )
        else:
            print(
                f"{k:25s}: {v}"
            )

    results_exp6.append({
        "checkpoint":
            os.path.basename(ckpt_path),
        "metrics": metrics
    })

    del eval_model
    del eval_encoder
    torch.cuda.empty_cache()


print("\n" + "=" * 70)
print("EXPERIMENT 6 A1 EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# A1 DIAGNOSTIC — PREDICTION DISTRIBUTION
# ============================================================

best_a1_ckpt = (
    "/kaggle/working/phase2_exp6/"
    "a1_ner_exp6_fullft_epoch3.pt"
)

eval_encoder = A1LM(
    vocab_size=50000,
    hidden_size=512,
    num_layers=8,
    num_heads=8,
    ffn_size=2048,
    max_seq_len=2048,
    dropout=0.1
)

eval_model = A1NER(
    eval_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
)

state = torch.load(
    best_a1_ckpt,
    map_location="cpu",
    weights_only=False
)

eval_model.load_state_dict(
    state["model_state_dict"],
    strict=True
)

eval_model = eval_model.to(device)
eval_model.eval()

print("=" * 70)
print("A1 PREDICTION DIAGNOSTIC")
print("=" * 70)

pred_class_counts = torch.zeros(
    25,
    dtype=torch.long
)

gold_class_counts = torch.zeros(
    25,
    dtype=torch.long
)

total_valid = 0

with torch.no_grad():

    for batch in a4_val_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        logits = eval_model(
            input_ids,
            attention_mask=attention_mask
        )

        pred_ids = logits.argmax(dim=-1).cpu()

        valid = labels != -100

        pred_valid = pred_ids[valid]
        gold_valid = labels[valid]

        pred_class_counts += torch.bincount(
            pred_valid,
            minlength=25
        )

        gold_class_counts += torch.bincount(
            gold_valid,
            minlength=25
        )

        total_valid += int(valid.sum())

print("\nTotal valid tokens:", total_valid)

print("\nPREDICTED class distribution")
print("-" * 70)

for i in range(25):
    pct = (
        100.0 * pred_class_counts[i].item()
        / total_valid
    )

    print(
        f"{i:2d} {id2label[i]:15s} "
        f"{pred_class_counts[i].item():8d} "
        f"{pct:7.3f}%"
    )

print("\nGOLD class distribution")
print("-" * 70)

for i in range(25):
    pct = (
        100.0 * gold_class_counts[i].item()
        / total_valid
    )

    print(
        f"{i:2d} {id2label[i]:15s} "
        f"{gold_class_counts[i].item():8d} "
        f"{pct:7.3f}%"
    )

print("\nTop predicted classes:")

top_values, top_indices = torch.topk(
    pred_class_counts,
    k=10
)

for value, idx in zip(top_values, top_indices):
    print(
        f"{id2label[idx.item()]:15s}: "
        f"{value.item():,}"
    )

print("=" * 70)

In [ ]:
# ============================================================
# A1 EXPERIMENT 7 — CLASS-WEIGHTED CE
# CELL 1: FOLD-2 CLASS WEIGHTS
# ============================================================

import torch

print("=" * 70)
print("EXPERIMENT 7 — A1 + CLASS-WEIGHTED CROSS ENTROPY")
print("=" * 70)

# Count labels ONLY from Fold 2
class_counts7 = torch.zeros(
    25,
    dtype=torch.long
)

for chunk in a4_train_only_chunks:

    labels = torch.tensor(
        chunk.bio_ids,
        dtype=torch.long
    )

    counts = torch.bincount(
        labels,
        minlength=25
    )

    class_counts7 += counts

print("Fold-2 label counts:")
print("-" * 70)

for i in range(25):
    print(
        f"{i:2d} {id2label[i]:15s} "
        f"{class_counts7[i].item():8d}"
    )

# ------------------------------------------------------------
# Inverse-frequency weights
# ------------------------------------------------------------

safe_counts = class_counts7.float().clamp_min(1)

weights7 = 1.0 / safe_counts

# Normalize mean weight = 1
weights7 = weights7 / weights7.mean()

# Prevent extreme numerical behavior
weights7 = weights7.clamp(
    min=0.05,
    max=5.0
)

print("\nClass weights:")
print("-" * 70)

for i in range(25):
    print(
        f"{i:2d} {id2label[i]:15s} "
        f"{weights7[i].item():.4f}"
    )

print("\nO weight:", f"{weights7[0].item():.4f}")
print(
    "Mean weight:",
    f"{weights7.mean().item():.4f}"
)

weights7 = weights7.to(device)

print("\n✓ Weights computed from Fold 2 only")
print("✓ No validation information used")

In [ ]:
# ============================================================
# EXPERIMENT 7 — A1 + CLASS-WEIGHTED CE
# CELL 2: FRESH MODEL + SANITY TEST
# ============================================================

import time
import torch
import torch.nn.functional as F

# ------------------------------------------------------------
# Fresh exact A1 encoder
# ------------------------------------------------------------

a1_encoder_w = A1LM(
    vocab_size=50000,
    hidden_size=512,
    num_layers=8,
    num_heads=8,
    ffn_size=2048,
    max_seq_len=2048,
    dropout=0.1
)

a1_encoder_w.load_state_dict(
    a1_state,
    strict=True
)

model7 = A1NER(
    a1_encoder_w,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)

total_params7 = sum(
    p.numel() for p in model7.parameters()
)

trainable_params7 = sum(
    p.numel()
    for p in model7.parameters()
    if p.requires_grad
)

print("=" * 70)
print("EXPERIMENT 7 — A1 WEIGHTED CE SANITY TEST")
print("=" * 70)

print(
    "Total parameters    :",
    f"{total_params7:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_params7:,}"
)

assert total_params7 == 77_481_497
assert trainable_params7 == 77_481_497

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer7 = torch.optim.AdamW(
    model7.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

# ------------------------------------------------------------
# Real Fold-2 batch
# ------------------------------------------------------------

batch = next(iter(a4_train_loader_exp2))

input_ids = batch["input_ids"].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    device,
    non_blocking=True
)

labels = batch["labels"].to(
    device,
    non_blocking=True
)

model7.train()

torch.cuda.reset_peak_memory_stats(device)

start = time.time()

logits = model7(
    input_ids,
    attention_mask=attention_mask
)

loss = F.cross_entropy(
    logits.view(-1, 25),
    labels.view(-1),
    weight=weights7,
    ignore_index=-100
)

optimizer7.zero_grad(set_to_none=True)

loss.backward()

grad_norm7 = torch.nn.utils.clip_grad_norm_(
    model7.parameters(),
    max_norm=1.0
)

optimizer7.step()

elapsed = time.time() - start

print("\nLoss      :", f"{loss.item():.4f}")
print("Logits    :", tuple(logits.shape))
print("Grad norm :", f"{float(grad_norm7):.4f}")
print("Time      :", f"{elapsed:.3f}s")

print(
    "Peak VRAM :",
    f"{torch.cuda.max_memory_allocated(device)/1024**3:.2f} GB"
)

print(
    "Finite logits:",
    bool(torch.isfinite(logits).all().item())
)

print(
    "Finite loss:",
    bool(torch.isfinite(loss).item())
)

print(
    "Finite gradients:",
    all(
        p.grad is None or
        torch.isfinite(p.grad).all().item()
        for p in model7.parameters()
    )
)

assert logits.shape[-1] == 25
assert torch.isfinite(logits).all()
assert torch.isfinite(loss)
assert torch.isfinite(
    torch.tensor(float(grad_norm7))
)

print("\n✓ Weighted CE forward passed")
print("✓ Weighted CE loss passed")
print("✓ Backward passed")
print("✓ Optimizer step passed")
print("✓ Gradients finite")
print("✓ A1 weighted-CE sanity test PASSED")

del batch, input_ids, attention_mask, labels
del logits, loss

torch.cuda.empty_cache()

In [ ]:
# ============================================================
# EXPERIMENT 7 — A1 + CLASS-WEIGHTED CE
# CELL 3: CLEAN FOLD-2 TRAINING
# ============================================================

import os
import time
import torch
import torch.nn.functional as F

EXP7_DIR = "/kaggle/working/phase2_exp7"
os.makedirs(EXP7_DIR, exist_ok=True)

EPOCHS7 = 3
LR7 = 2e-5
WD7 = 0.01

optimizer7 = torch.optim.AdamW(
    model7.parameters(),
    lr=LR7,
    weight_decay=WD7
)

print("=" * 70)
print("EXPERIMENT 7 — A1 + CLASS-WEIGHTED CE TRAINING")
print("=" * 70)

print("Train chunks       :", len(a4_train_only_chunks))
print("Train batches      :", len(a4_train_loader_exp2))
print("Validation chunks  :", len(a4_val_chunks))
print("Parameters         :", f"{trainable_params7:,}")
print("Learning rate      :", LR7)
print("Weight decay       :", WD7)
print("Epochs             :", EPOCHS7)

epoch_results7 = []

overall_start = time.time()

for epoch in range(1, EPOCHS7 + 1):

    model7.train()
    epoch_start = time.time()

    running_loss = 0.0
    steps = 0

    for step, batch in enumerate(
        a4_train_loader_exp2,
        start=1
    ):

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        optimizer7.zero_grad(set_to_none=True)

        logits = model7(
            input_ids,
            attention_mask=attention_mask
        )

        loss = F.cross_entropy(
            logits.view(-1, 25),
            labels.view(-1),
            weight=weights7,
            ignore_index=-100
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model7.parameters(),
            max_norm=1.0
        )

        optimizer7.step()

        running_loss += loss.item()
        steps += 1

        if step % 20 == 0 or step == len(a4_train_loader_exp2):

            elapsed = time.time() - epoch_start

            print(
                f"Epoch {epoch}/{EPOCHS7} | "
                f"Step {step:3d}/{len(a4_train_loader_exp2)} | "
                f"Loss {loss.item():.4f} | "
                f"Time {elapsed:.1f}s"
            )

        del input_ids, attention_mask, labels
        del logits, loss

    epoch_loss = running_loss / steps
    epoch_time = time.time() - epoch_start

    epoch_results7.append({
        "epoch": epoch,
        "loss": epoch_loss,
        "time": epoch_time
    })

    ckpt_path = os.path.join(
        EXP7_DIR,
        f"a1_ner_exp7_weighted_epoch{epoch}.pt"
    )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model7.state_dict(),
            "optimizer_state_dict": optimizer7.state_dict(),
            "train_loss": epoch_loss,
            "architecture": "A1",
            "technique": "class_weighted_ce",
            "class_weights": weights7.detach().cpu()
        },
        ckpt_path
    )

    print(
        f"✓ Epoch {epoch} complete | "
        f"Loss={epoch_loss:.4f} | "
        f"{epoch_time:.1f}s"
    )

total_time7 = time.time() - overall_start

print("\n" + "=" * 70)
print("EXPERIMENT 7 A1 WEIGHTED TRAINING COMPLETE")
print("=" * 70)

for r in epoch_results7:
    print(
        f"Epoch {r['epoch']}: "
        f"loss={r['loss']:.4f}, "
        f"time={r['time']:.1f}s"
    )

print(f"\nTotal training time: {total_time7:.1f}s")
print(
    "Peak VRAM:",
    f"{torch.cuda.max_memory_allocated(device)/1024**3:.2f} GB"
)

print("\nCheckpoints saved in:")
print(EXP7_DIR)

In [ ]:
# ============================================================
# EXPERIMENT 7 — A1 + CLASS-WEIGHTED CE
# CELL 4: CLEAN FOLD-1 EVALUATION
# ============================================================

import os
import torch

EXP7_DIR = "/kaggle/working/phase2_exp7"

ckpts7 = sorted([
    os.path.join(EXP7_DIR, f)
    for f in os.listdir(EXP7_DIR)
    if f.endswith(".pt")
])

print("=" * 70)
print("EXPERIMENT 7 — A1 WEIGHTED CE FOLD-1 EVALUATION")
print("=" * 70)

print("Checkpoints found:")
for p in ckpts7:
    print(" ", os.path.basename(p))

print("\nValidation chunks :", len(a4_val_chunks))
print("Validation batches:", len(a4_val_loader))

results_exp7 = []

for ckpt_path in ckpts7:

    print("\n" + "-" * 70)
    print("Evaluating:", os.path.basename(ckpt_path))

    # Fresh exact A1 model
    eval_encoder = A1LM(
        vocab_size=50000,
        hidden_size=512,
        num_layers=8,
        num_heads=8,
        ffn_size=2048,
        max_seq_len=2048,
        dropout=0.1
    )

    eval_model = A1NER(
        eval_encoder,
        num_labels=25,
        dropout=0.10,
        hidden_size=512
    )

    state = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False
    )

    eval_model.load_state_dict(
        state["model_state_dict"],
        strict=True
    )

    eval_model = eval_model.to(device)
    eval_model.eval()

    predictions = {}
    gold = {}

    with torch.no_grad():

        for batch in a4_val_loader:

            input_ids = batch["input_ids"].to(
                device,
                non_blocking=True
            )

            attention_mask = batch["attention_mask"].to(
                device,
                non_blocking=True
            )

            logits = eval_model(
                input_ids,
                attention_mask=attention_mask
            )

            pred_ids = logits.argmax(
                dim=-1
            ).cpu().tolist()

            B = len(batch["doc_id"])

            for i in range(B):

                doc_id = batch["doc_id"][i]
                text = fold1_text[doc_id]

                valid_len = int(
                    batch["attention_mask"][i].sum().item()
                )

                offsets_valid = batch["offsets"][i][:valid_len]
                pred_valid = pred_ids[i][:valid_len]

                spans = bio_to_token_spans(
                    pred_valid,
                    offsets_valid,
                    text
                )

                norm_spans = []

                for start, end, label in spans:

                    start, end, label = normalize_span_boundaries(
                        start,
                        end,
                        label,
                        text
                    )

                    norm_spans.append(
                        (start, end, label)
                    )

                predictions.setdefault(
                    doc_id,
                    []
                )

                predictions[doc_id].extend(
                    norm_spans
                )

                gold.setdefault(
                    doc_id,
                    []
                )

                gold[doc_id].extend(
                    batch["gold_spans"][i]
                )

    # Deduplicate overlapping chunks
    for doc_id in predictions:
        predictions[doc_id] = list(
            set(predictions[doc_id])
        )

    for doc_id in gold:
        gold[doc_id] = list(
            set(gold[doc_id])
        )

    metrics = strict_macro_f1(
        predictions,
        gold
    )

    print("\nRESULT")
    print("-" * 70)

    for k, v in metrics.items():

        if isinstance(v, float):
            print(
                f"{k:25s}: {v:.4f}"
            )
        else:
            print(
                f"{k:25s}: {v}"
            )

    results_exp7.append({
        "checkpoint":
            os.path.basename(ckpt_path),
        "metrics": metrics
    })

    del eval_model
    del eval_encoder
    torch.cuda.empty_cache()


print("\n" + "=" * 70)
print("EXPERIMENT 7 A1 WEIGHTED EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# EXPERIMENT 8 — A2 NER
# CELL 1: EXACT CHECKPOINT INSPECTION
# ============================================================

import os
import torch

A2_CKPT = "/kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt"

print("=" * 70)
print("EXPERIMENT 8 — A2 CHECKPOINT INSPECTION")
print("=" * 70)

print("Checkpoint:", A2_CKPT)
print("Exists:", os.path.exists(A2_CKPT))

ckpt_a2 = torch.load(
    A2_CKPT,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(ckpt_a2))

print("\nCheckpoint keys:")
for k in ckpt_a2.keys():
    print(" ", k)

a2_state = ckpt_a2["model_state_dict"]

print("\nA2 state tensors:", len(a2_state))

print("\nFirst checkpoint keys:")
for k in list(a2_state.keys())[:150]:
    print(
        f"{k:65s} "
        f"{tuple(a2_state[k].shape)}"
    )

a2_checkpoint_params = sum(
    v.numel()
    for v in a2_state.values()
)

print(
    "\nA2 checkpoint parameter count:",
    f"{a2_checkpoint_params:,}"
)

print(
    "Training step:",
    ckpt_a2.get("step")
)

print(
    "Training loss:",
    ckpt_a2.get("train_loss")
)

print("=" * 70)

In [ ]:
# ============================================================
# EXPERIMENT 8 — A2 NER
# CELL 2: EXACT A2 RECONSTRUCTION
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# A2 Grouped-Query Attention
# ------------------------------------------------------------

class A2Attention(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_q_heads=8,
        num_kv_heads=2,
        head_dim=64
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim

        assert hidden_size == num_q_heads * head_dim
        assert num_q_heads % num_kv_heads == 0

        self.q_proj = nn.Linear(
            hidden_size,
            num_q_heads * head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            hidden_size,
            num_kv_heads * head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            hidden_size,
            num_kv_heads * head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            num_q_heads * head_dim,
            hidden_size,
            bias=False
        )

    def forward(
        self,
        x,
        attention_mask=None
    ):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.num_q_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        # GQA: repeat K/V heads to match Q heads
        repeat_factor = (
            self.num_q_heads //
            self.num_kv_heads
        )

        k = k.repeat_interleave(
            repeat_factor,
            dim=1
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1
        )

        # Causal attention
        causal_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        if attention_mask is not None:

            key_mask = (
                ~attention_mask.to(torch.bool)
                .unsqueeze(1)
                .unsqueeze(2)
            )

            attn_mask = (
                causal_mask
                .unsqueeze(0)
                .unsqueeze(0)
                |
                key_mask
            )

        else:

            attn_mask = (
                causal_mask
                .unsqueeze(0)
                .unsqueeze(0)
            )

        attn = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=~attn_mask,
            dropout_p=0.0,
            is_causal=False
        )

        attn = attn.transpose(
            1,
            2
        ).contiguous()

        attn = attn.view(
            B,
            T,
            self.hidden_size
        )

        return self.out_proj(attn)


# ------------------------------------------------------------
# A2 Transformer Layer
# ------------------------------------------------------------

class A2Layer(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_q_heads=8,
        num_kv_heads=2,
        head_dim=64,
        ffn_size=2048
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            hidden_size
        )

        self.attention = A2Attention(
            hidden_size=hidden_size,
            num_q_heads=num_q_heads,
            num_kv_heads=num_kv_heads,
            head_dim=head_dim
        )

        self.norm2 = nn.LayerNorm(
            hidden_size
        )

        self.ffn = nn.ModuleDict({
            "fc1": nn.Linear(
                hidden_size,
                ffn_size
            ),
            "fc2": nn.Linear(
                ffn_size,
                hidden_size
            )
        })

    def forward(
        self,
        x,
        attention_mask=None
    ):

        x = x + self.attention(
            self.norm1(x),
            attention_mask=attention_mask
        )

        h = self.norm2(x)

        h = self.ffn["fc1"](h)
        h = F.gelu(h)
        h = self.ffn["fc2"](h)

        x = x + h

        return x


# ------------------------------------------------------------
# Exact A2 Encoder
# ------------------------------------------------------------

class A2LM(nn.Module):

    def __init__(
        self,
        vocab_size=50000,
        hidden_size=512,
        num_layers=9,
        num_q_heads=8,
        num_kv_heads=2,
        head_dim=64,
        ffn_size=2048,
        max_seq_len=2048
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            hidden_size
        )

        self.layers = nn.ModuleList([
            A2Layer(
                hidden_size=hidden_size,
                num_q_heads=num_q_heads,
                num_kv_heads=num_kv_heads,
                head_dim=head_dim,
                ffn_size=ffn_size
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(
            hidden_size
        )

        # Phase-1 checkpoint contains this separately.
        self.output_bias = nn.Parameter(
            torch.zeros(vocab_size)
        )

    def forward(
        self,
        input_ids,
        attention_mask=None,
        return_hidden=True
    ):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        ).unsqueeze(0)

        x = (
            self.token_embedding(input_ids)
            +
            self.position_embedding(positions)
        )

        for layer in self.layers:

            x = layer(
                x,
                attention_mask=attention_mask
            )

        x = self.final_norm(x)

        if return_hidden:
            return x

        # Phase-1 tied output projection
        logits = F.linear(
            x,
            self.token_embedding.weight,
            self.output_bias
        )

        return logits


# ------------------------------------------------------------
# Construct exact A2 model
# ------------------------------------------------------------

a2_encoder = A2LM(
    vocab_size=50000,
    hidden_size=512,
    num_layers=9,
    num_q_heads=8,
    num_kv_heads=2,
    head_dim=64,
    ffn_size=2048,
    max_seq_len=2048
)

missing_a2, unexpected_a2 = (
    a2_encoder.load_state_dict(
        a2_state,
        strict=False
    )
)

a2_params = sum(
    p.numel()
    for p in a2_encoder.parameters()
)

print("=" * 70)
print("A2 RECONSTRUCTION")
print("=" * 70)

print("Missing keys   :", missing_a2)
print("Unexpected keys:", unexpected_a2)

print(
    "Model parameters    :",
    f"{a2_params:,}"
)

print(
    "Checkpoint parameters:",
    f"{a2_checkpoint_params:,}"
)

assert a2_params == 51_513_680
assert len(missing_a2) == 0
assert len(unexpected_a2) == 0

print("\n✓ Parameter count EXACT")
print("✓ Missing keys = 0")
print("✓ Unexpected keys = 0")
print("✓ A2 GQA architecture reconstructed successfully")

In [ ]:
# ============================================================
# EXPERIMENT 8 — A2 NER
# CELL 3: REAL-DATA GPU SMOKE TEST
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

a2_encoder = a2_encoder.to(device)

print("=" * 70)
print("A2 REAL-DATA GPU SMOKE TEST")
print("=" * 70)

print("Device:", device)

batch = next(iter(a4_train_loader_exp2))

input_ids = batch["input_ids"].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    device,
    non_blocking=True
)

print("Input shape :", tuple(input_ids.shape))
print(
    "Valid tokens:",
    int(attention_mask.sum().item())
)

torch.cuda.reset_peak_memory_stats(device)

start = time.time()

with torch.no_grad():

    hidden = a2_encoder(
        input_ids,
        attention_mask=attention_mask,
        return_hidden=True
    )

elapsed = time.time() - start

print("\nOutput shape:", tuple(hidden.shape))
print("Forward time:", f"{elapsed:.3f}s")

print(
    "Finite output:",
    bool(torch.isfinite(hidden).all().item())
)

print(
    "Peak VRAM:",
    f"{torch.cuda.max_memory_allocated(device)/1024**3:.2f} GB"
)

assert hidden.shape[0] == input_ids.shape[0]
assert hidden.shape[1] == input_ids.shape[1]
assert hidden.shape[2] == 512
assert torch.isfinite(hidden).all()

print("\n✓ Real Phase-2 data forward passed")
print("✓ Shape correct")
print("✓ Output finite")
print("✓ A2 GPU smoke test PASSED")

del hidden
del input_ids
del attention_mask
del batch

torch.cuda.empty_cache()

In [ ]:
# ============================================================
# EXPERIMENT 8 — A2 NER
# CELL 4: NER MODEL + BACKWARD SANITY TEST
# ============================================================

class A2NER(nn.Module):

    def __init__(
        self,
        encoder,
        num_labels=25,
        dropout=0.10,
        hidden_size=512
    ):
        super().__init__()

        self.encoder = encoder

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Linear(
            hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask=None
    ):

        x = self.encoder(
            input_ids,
            attention_mask=attention_mask,
            return_hidden=True
        )

        x = self.dropout(x)

        return self.classifier(x)


# ------------------------------------------------------------
# Build A2 NER model
# ------------------------------------------------------------

model8 = A2NER(
    a2_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)

total_params8 = sum(
    p.numel()
    for p in model8.parameters()
)

trainable_params8 = sum(
    p.numel()
    for p in model8.parameters()
    if p.requires_grad
)

print("=" * 70)
print("A2 NER SANITY TEST")
print("=" * 70)

print(
    "Total parameters    :",
    f"{total_params8:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_params8:,}"
)

assert total_params8 == 51_526_497
assert trainable_params8 == 51_526_497


# ------------------------------------------------------------
# Real Fold-2 batch
# ------------------------------------------------------------

batch = next(
    iter(a4_train_loader_exp2)
)

input_ids = batch["input_ids"].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    device,
    non_blocking=True
)

labels = batch["labels"].to(
    device,
    non_blocking=True
)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer8 = torch.optim.AdamW(
    model8.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

model8.train()

torch.cuda.reset_peak_memory_stats(
    device
)

start = time.time()

logits = model8(
    input_ids,
    attention_mask=attention_mask
)

loss = F.cross_entropy(
    logits.view(-1, 25),
    labels.view(-1),
    ignore_index=-100
)

optimizer8.zero_grad(
    set_to_none=True
)

loss.backward()

grad_norm8 = torch.nn.utils.clip_grad_norm_(
    model8.parameters(),
    max_norm=1.0
)

optimizer8.step()

elapsed = time.time() - start


print("\nLoss      :", f"{loss.item():.4f}")
print(
    "Logits    :",
    tuple(logits.shape)
)
print(
    "Grad norm :",
    f"{float(grad_norm8):.4f}"
)
print(
    "Time      :",
    f"{elapsed:.3f}s"
)

print(
    "Peak VRAM :",
    f"{torch.cuda.max_memory_allocated(device)/1024**3:.2f} GB"
)

print(
    "Finite logits:",
    bool(
        torch.isfinite(logits).all().item()
    )
)

print(
    "Finite loss:",
    bool(
        torch.isfinite(loss).item()
    )
)

print(
    "Finite gradients:",
    all(
        p.grad is None or
        torch.isfinite(p.grad).all().item()
        for p in model8.parameters()
    )
)

assert logits.shape[-1] == 25
assert torch.isfinite(logits).all()
assert torch.isfinite(loss)
assert torch.isfinite(
    torch.tensor(float(grad_norm8))
)

print("\n✓ Forward passed")
print("✓ Loss passed")
print("✓ Backward passed")
print("✓ Optimizer step passed")
print("✓ Gradients finite")
print("✓ A2 NER sanity test PASSED")


# Cleanup
del batch
del input_ids
del attention_mask
del labels
del logits
del loss

torch.cuda.empty_cache()

In [ ]:
# ============================================================
# A2 NER — PARAMETER COUNT FIX + SANITY TEST
# ============================================================

print("=" * 70)
print("A2 NER SANITY TEST")
print("=" * 70)

print(
    "Total parameters    :",
    f"{total_params8:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_params8:,}"
)

# Exact expected count:
# A2 encoder  = 51,513,680
# NER head    = 512*25 + 25 = 12,825
# Total       = 51,526,505

assert total_params8 == 51_526_505
assert trainable_params8 == 51_526_505

print("\n✓ Parameter count EXACT")
print("✓ A2 encoder + NER head validated")

In [ ]:
# ============================================================
# EXPERIMENT 8 — A2 NER
# CELL 5: FORWARD + BACKWARD SANITY TEST
# ============================================================

batch = next(iter(a4_train_loader_exp2))

input_ids = batch["input_ids"].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    device,
    non_blocking=True
)

labels = batch["labels"].to(
    device,
    non_blocking=True
)

optimizer8 = torch.optim.AdamW(
    model8.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

model8.train()

torch.cuda.reset_peak_memory_stats(device)

start = time.time()

logits = model8(
    input_ids,
    attention_mask=attention_mask
)

loss = F.cross_entropy(
    logits.view(-1, 25),
    labels.view(-1),
    ignore_index=-100
)

optimizer8.zero_grad(
    set_to_none=True
)

loss.backward()

grad_norm8 = torch.nn.utils.clip_grad_norm_(
    model8.parameters(),
    max_norm=1.0
)

optimizer8.step()

elapsed = time.time() - start

print("=" * 70)
print("A2 NER FORWARD/BACKWARD SANITY TEST")
print("=" * 70)

print("Loss      :", f"{loss.item():.4f}")
print("Logits    :", tuple(logits.shape))
print("Grad norm :", f"{float(grad_norm8):.4f}")
print("Time      :", f"{elapsed:.3f}s")

print(
    "Peak VRAM :",
    f"{torch.cuda.max_memory_allocated(device)/1024**3:.2f} GB"
)

print(
    "Finite logits:",
    bool(torch.isfinite(logits).all().item())
)

print(
    "Finite loss:",
    bool(torch.isfinite(loss).item())
)

print(
    "Finite gradients:",
    all(
        p.grad is None or
        torch.isfinite(p.grad).all().item()
        for p in model8.parameters()
    )
)

assert logits.shape == (
    input_ids.shape[0],
    input_ids.shape[1],
    25
)

assert torch.isfinite(logits).all()
assert torch.isfinite(loss)
assert torch.isfinite(
    torch.tensor(float(grad_norm8))
)

print("\n✓ Forward passed")
print("✓ Loss passed")
print("✓ Backward passed")
print("✓ Optimizer step passed")
print("✓ Gradients finite")
print("✓ A2 NER sanity test PASSED")

del batch
del input_ids
del attention_mask
del labels
del logits
del loss

torch.cuda.empty_cache()

In [ ]:
# ============================================================
# EXPERIMENT 8 — A2 FULL FINE-TUNING
# CELL 6: CLEAN FOLD-2 TRAINING
# ============================================================

import os
import time

EXP8_DIR = "/kaggle/working/phase2_exp8"
os.makedirs(EXP8_DIR, exist_ok=True)

# Fresh A2 model from the original Phase-1 checkpoint
a2_encoder_clean = A2LM(
    vocab_size=50000,
    hidden_size=512,
    num_layers=9,
    num_q_heads=8,
    num_kv_heads=2,
    head_dim=64,
    ffn_size=2048,
    max_seq_len=2048
)

a2_encoder_clean.load_state_dict(
    a2_state,
    strict=True
)

model8 = A2NER(
    a2_encoder_clean,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)

optimizer8 = torch.optim.AdamW(
    model8.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

NUM_EPOCHS = 3

print("=" * 70)
print("EXPERIMENT 8 — A2 FULL FINE-TUNING")
print("=" * 70)

print("Training chunks :", len(a4_train_only_chunks))
print("Training batches:", len(a4_train_loader_exp2))
print("Validation docs :", len(fold1_ids))
print("Epochs           :", NUM_EPOCHS)
print("Learning rate    :", "2e-5")
print("Weight decay     :", "0.01")
print("Batch size       :", 4)

total_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):

    model8.train()

    epoch_start = time.time()
    running_loss = 0.0
    valid_tokens = 0

    for step, batch in enumerate(
        a4_train_loader_exp2,
        start=1
    ):

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        optimizer8.zero_grad(
            set_to_none=True
        )

        logits = model8(
            input_ids,
            attention_mask=attention_mask
        )

        loss = F.cross_entropy(
            logits.view(-1, 25),
            labels.view(-1),
            ignore_index=-100
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model8.parameters(),
            max_norm=1.0
        )

        optimizer8.step()

        n_valid = (
            labels != -100
        ).sum().item()

        running_loss += (
            loss.item() * n_valid
        )

        valid_tokens += n_valid

        del input_ids
        del attention_mask
        del labels
        del logits
        del loss

        if step % 25 == 0 or step == len(a4_train_loader_exp2):

            current_loss = (
                running_loss / valid_tokens
            )

            elapsed = (
                time.time() - epoch_start
            )

            print(
                f"Epoch {epoch}/{NUM_EPOCHS} | "
                f"Batch {step}/{len(a4_train_loader_exp2)} | "
                f"Loss {current_loss:.4f} | "
                f"Time {elapsed:.1f}s"
            )

    epoch_loss = (
        running_loss / valid_tokens
    )

    epoch_time = (
        time.time() - epoch_start
    )

    ckpt_path = (
        f"{EXP8_DIR}/"
        f"a2_ner_exp8_fullft_epoch{epoch}.pt"
    )

    torch.save(
        {
            "experiment": 8,
            "architecture": "A2",
            "technique": "Full Fine-Tuning",
            "epoch": epoch,
            "train_loss": epoch_loss,
            "model_state_dict": model8.state_dict(),
            "optimizer_state_dict": optimizer8.state_dict(),
        },
        ckpt_path
    )

    print(
        f"\n✓ Epoch {epoch} complete | "
        f"loss={epoch_loss:.4f} | "
        f"time={epoch_time:.1f}s"
    )

    print(
        f"✓ Checkpoint saved: {ckpt_path}"
    )

total_time = time.time() - total_start

print("\n" + "=" * 70)
print("A2 TRAINING COMPLETE")
print("=" * 70)

print(
    "Total training time:",
    f"{total_time:.1f}s"
)

print(
    "Total training time:",
    f"{total_time/60:.2f} min"
)

print("✓ Clean Fold-2 training finished")

In [ ]:
# ============================================================
# EXPERIMENT 8 — A2 FULL FT
# CELL 7: CLEAN FOLD-1 EVALUATION
# ============================================================

import os
import torch

print("=" * 70)
print("EXPERIMENT 8 — A2 FULL FT | CLEAN FOLD-1 EVALUATION")
print("=" * 70)

results_exp8 = []

for epoch in [1, 2, 3]:

    ckpt_path = (
        f"/kaggle/working/phase2_exp8/"
        f"a2_ner_exp8_fullft_epoch{epoch}.pt"
    )

    print(f"\n{'-' * 70}")
    print(f"Evaluating A2 Epoch {epoch}")
    print(f"{'-' * 70}")

    checkpoint = torch.load(
        ckpt_path,
        map_location=device,
        weights_only=False
    )

    model8.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )

    model8.eval()

    predictions = {}

    with torch.no_grad():

        for batch in a4_val_loader:

            input_ids = batch["input_ids"].to(
                device,
                non_blocking=True
            )

            attention_mask = batch["attention_mask"].to(
                device,
                non_blocking=True
            )

            logits = model8(
                input_ids,
                attention_mask=attention_mask
            )

            pred_ids = logits.argmax(
                dim=-1
            ).cpu().tolist()

            batch_offsets = batch["offsets"]
            batch_doc_ids = batch["doc_id"]

            for i, doc_id in enumerate(batch_doc_ids):

                spans = bio_to_token_spans(
                    pred_ids[i],
                    batch_offsets[i],
                    fold1_text[doc_id]
                )

                # Normalize boundaries
                clean_spans = []

                for start, end, label in spans:

                    start, end, label = (
                        normalize_span_boundaries(
                            start,
                            end,
                            label,
                            fold1_text[doc_id]
                        )
                    )

                    if end > start:
                        clean_spans.append(
                            (start, end, label)
                        )

                if doc_id not in predictions:
                    predictions[doc_id] = set()

                predictions[doc_id].update(
                    clean_spans
                )

            del input_ids
            del attention_mask
            del logits

    # Convert sets to lists
    predictions = {
        doc_id: list(spans)
        for doc_id, spans in predictions.items()
    }

    # Ensure every validation document exists
    for doc_id in fold1_ids:
        predictions.setdefault(
            doc_id,
            []
        )

    metrics = strict_macro_f1(
        predictions,
        fold1_gold
    )

    print(
        f"\nA2 Epoch {epoch}"
    )

    print(
        f"Strict Macro-F1 : {metrics['f1']:.4f}"
    )

    print(
        f"Strict Macro-P  : {metrics['precision']:.4f}"
    )

    print(
        f"Strict Macro-R  : {metrics['recall']:.4f}"
    )

    results_exp8.append({
        "epoch": epoch,
        "f1": metrics["f1"],
        "precision": metrics["precision"],
        "recall": metrics["recall"]
    })

    del checkpoint
    torch.cuda.empty_cache()


print("\n" + "=" * 70)
print("A2 EXPERIMENT 8 RESULTS")
print("=" * 70)

for r in results_exp8:

    print(
        f"Epoch {r['epoch']} | "
        f"F1={r['f1']:.4f} | "
        f"P={r['precision']:.4f} | "
        f"R={r['recall']:.4f}"
    )

best_exp8 = max(
    results_exp8,
    key=lambda x: x["f1"]
)

print("\nBEST A2 CHECKPOINT")
print(
    f"Epoch       : {best_exp8['epoch']}"
)
print(
    f"Strict F1   : {best_exp8['f1']:.4f}"
)
print(
    f"Precision   : {best_exp8['precision']:.4f}"
)
print(
    f"Recall      : {best_exp8['recall']:.4f}"
)

print("\n✓ Clean Fold-1 evaluation complete")

In [ ]:
# ============================================================
# A2 EVALUATION — KEY-SAFE VERSION
# ============================================================

print("\n" + "=" * 70)
print("A2 EXPERIMENT 8 RESULTS")
print("=" * 70)

print("Returned metric keys:", list(metrics.keys()))
print("Returned metrics:", metrics)

In [ ]:
# ============================================================
# A2 DIAGNOSTIC — PREDICTION DISTRIBUTION
# ============================================================

import collections
import torch

print("=" * 70)
print("A2 DIAGNOSTIC — FOLD-1 PREDICTION DISTRIBUTION")
print("=" * 70)

# Use best/last checkpoint for diagnostic
ckpt_path = (
    "/kaggle/working/phase2_exp8/"
    "a2_ner_exp8_fullft_epoch3.pt"
)

checkpoint = torch.load(
    ckpt_path,
    map_location=device,
    weights_only=False
)

model8.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model8.eval()

pred_counter = collections.Counter()
gold_counter = collections.Counter()

total_tokens = 0
total_pred_entities = 0
total_gold_entities = 0

# Inspect first few batches
with torch.no_grad():

    for batch_idx, batch in enumerate(a4_val_loader):

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        logits = model8(
            input_ids,
            attention_mask=attention_mask
        )

        pred_ids = logits.argmax(
            dim=-1
        )

        # Count predictions only on valid tokens
        valid_mask = labels != -100

        valid_preds = pred_ids[
            valid_mask
        ].detach().cpu().tolist()

        valid_gold = labels[
            valid_mask
        ].detach().cpu().tolist()

        pred_counter.update(valid_preds)
        gold_counter.update(valid_gold)

        # Decode entities from first 2 batches
        if batch_idx < 2:

            for i, doc_id in enumerate(
                batch["doc_id"]
            ):

                spans = bio_to_token_spans(
                    pred_ids[i].detach().cpu().tolist(),
                    batch["offsets"][i],
                    fold1_text[doc_id]
                )

                total_pred_entities += len(spans)

                if i < 2:
                    print(
                        f"\nBatch {batch_idx}, "
                        f"Doc {doc_id}"
                    )
                    print(
                        "Predicted entities:",
                        len(spans)
                    )

                    if spans:
                        print(
                            "First predictions:",
                            spans[:10]
                        )

        total_tokens += len(valid_preds)

        for label_id in valid_gold:
            if label_id != 0:
                total_gold_entities += 1

        del input_ids
        del attention_mask
        del labels
        del logits
        del pred_ids

print("\n" + "=" * 70)
print("TOKEN-LEVEL DISTRIBUTION")
print("=" * 70)

print(
    "Total valid tokens:",
    total_tokens
)

print("\nPredicted BIO IDs:")

for label_id, count in sorted(
    pred_counter.items()
):
    print(
        f"{label_id:2d} | "
        f"{id2label[label_id]:12s} | "
        f"{count:8d} | "
        f"{count / total_tokens * 100:7.3f}%"
    )

print("\nGold BIO IDs:")

for label_id, count in sorted(
    gold_counter.items()
):
    print(
        f"{label_id:2d} | "
        f"{id2label[label_id]:12s} | "
        f"{count:8d} | "
        f"{count / total_tokens * 100:7.3f}%"
    )

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    "Predicted non-O tokens:",
    total_tokens - pred_counter.get(0, 0)
)

print(
    "Gold non-O tokens:",
    total_tokens - gold_counter.get(0, 0)
)

print(
    "Predicted entities in inspected batches:",
    total_pred_entities
)

print(
    "Gold non-O token count:",
    total_gold_entities
)

print("\n✓ Diagnostic complete")

In [ ]:
# ============================================================
# A3 — CHECKPOINT INSPECTION
# ============================================================

import torch
import os

A3_PATH = "/kaggle/input/datasets/aq1de3/a3check/step_0001000.pt"

a3_ckpt = torch.load(
    A3_PATH,
    map_location="cpu",
    weights_only=False
)

print("=" * 70)
print("A3 CHECKPOINT INSPECTION")
print("=" * 70)

print("Checkpoint:", A3_PATH)
print("Top-level keys:", list(a3_ckpt.keys()))

if "model_state_dict" in a3_ckpt:
    a3_state = a3_ckpt["model_state_dict"]
elif "state_dict" in a3_ckpt:
    a3_state = a3_ckpt["state_dict"]
else:
    # Find the dictionary containing tensor parameters
    candidates = {
        k: v for k, v in a3_ckpt.items()
        if isinstance(v, dict)
        and any(torch.is_tensor(x) for x in v.values())
    }

    print("Candidate state dictionaries:", list(candidates.keys()))

    if len(candidates) == 1:
        a3_state = next(iter(candidates.values()))
    else:
        raise RuntimeError(
            "Could not uniquely identify A3 model state dict."
        )

print("\nNumber of tensors:", len(a3_state))

total_params = sum(
    v.numel()
    for v in a3_state.values()
    if torch.is_tensor(v)
)

print("Total parameters:", total_params)

print("\nCheckpoint metadata:")

for k, v in a3_ckpt.items():
    if not isinstance(v, dict):
        print(f"{k}: {v}")

print("\nFirst 40 state-dict keys:")

for k, v in list(a3_state.items())[:40]:
    print(
        f"{k:60s} "
        f"{tuple(v.shape)} "
        f"{v.dtype}"
    )

print("\n✓ A3 checkpoint inspection complete")

In [ ]:
# ============================================================
# A3 — EXACT ARCHITECTURE RECONSTRUCTION
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class A3FFN(nn.Module):
    """
    Checkpoint-compatible A3 FFN.

    State keys:
        ffn.0.weight
        ffn.0.bias
        ffn.3.weight
        ffn.3.bias
    """

    def __init__(self, hidden_size=512, ffn_size=2048, dropout=0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_size, hidden_size),
        )

    def forward(self, x):
        return self.net(x)


class A3Attention(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_heads=8,
        dropout=0.1
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        assert (
            hidden_size % num_heads == 0
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

    def forward(
        self,
        x,
        attention_mask=None,
        causal=True
    ):

        B, T, C = x.shape

        attn_mask = None

        if causal:

            attn_mask = torch.triu(
                torch.ones(
                    T,
                    T,
                    device=x.device,
                    dtype=torch.bool
                ),
                diagonal=1
            )

        key_padding_mask = None

        if attention_mask is not None:

            key_padding_mask = (
                attention_mask == 0
            )

        out, _ = self.attn(
            x,
            x,
            x,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=False
        )

        return out


class A3Layer(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_heads=8,
        ffn_size=2048,
        dropout=0.1
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            hidden_size
        )

        self.attn = A3Attention(
            hidden_size=hidden_size,
            num_heads=num_heads,
            dropout=dropout
        )

        self.ln2 = nn.LayerNorm(
            hidden_size
        )

        self.ffn = A3FFN(
            hidden_size=hidden_size,
            ffn_size=ffn_size,
            dropout=dropout
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
        attention_mask=None
    ):

        # Pre-norm attention residual
        x = x + self.dropout(
            self.attn(
                self.ln1(x),
                attention_mask=attention_mask
            )
        )

        # Pre-norm FFN residual
        x = x + self.dropout(
            self.ffn(
                self.ln2(x)
            )
        )

        return x


class A3LM(nn.Module):

    def __init__(
        self,
        vocab_size=50000,
        hidden_size=512,
        num_layers=8,
        num_heads=8,
        ffn_size=2048,
        max_seq_len=2048,
        dropout=0.1,
        prefix_ratio=0.5
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.max_seq_len = max_seq_len
        self.prefix_ratio = prefix_ratio

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            hidden_size
        )

        self.layers = nn.ModuleList([
            A3Layer(
                hidden_size=hidden_size,
                num_heads=num_heads,
                ffn_size=ffn_size,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(
            hidden_size
        )

        self.lm_head = nn.Linear(
            hidden_size,
            vocab_size,
            bias=False
        )

        # The checkpoint has tied token/lm weights.
        self.lm_head.weight = (
            self.token_embedding.weight
        )

    def forward(
        self,
        input_ids,
        attention_mask=None
    ):

        B, T = input_ids.shape

        assert T <= self.max_seq_len

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            +
            self.position_embedding(positions)
        )

        for layer in self.layers:

            x = layer(
                x,
                attention_mask=attention_mask
            )

        x = self.final_norm(x)

        return x


# ------------------------------------------------------------
# Instantiate
# ------------------------------------------------------------

a3_encoder = A3LM(
    vocab_size=50000,
    hidden_size=512,
    num_layers=8,
    num_heads=8,
    ffn_size=2048,
    max_seq_len=2048,
    dropout=0.1,
    prefix_ratio=0.5
)

# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

a3_model_params = sum(
    p.numel()
    for p in a3_encoder.parameters()
)

print("=" * 70)
print("A3 ARCHITECTURE")
print("=" * 70)

print(
    "Model parameters:",
    f"{a3_model_params:,}"
)

print(
    "Checkpoint parameters:",
    f"{total_params:,}"
)

print(
    "Difference:",
    a3_model_params - total_params
)

assert (
    a3_model_params == total_params
), (
    f"Parameter mismatch: "
    f"{a3_model_params} != {total_params}"
)

print(
    "\n✓ Parameter count EXACT"
)

# ------------------------------------------------------------
# Strict checkpoint loading
# ------------------------------------------------------------

missing, unexpected = (
    a3_encoder.load_state_dict(
        a3_state,
        strict=False
    )
)

print(
    "\nMissing keys:",
    missing
)

print(
    "Unexpected keys:",
    unexpected
)

assert len(missing) == 0
assert len(unexpected) == 0

print(
    "\n✓ Strict-compatible state loading"
)

print("\nA3 reconstruction PASSED")

In [ ]:
# ============================================================
# A3 — EXACT KEY MISMATCH DIAGNOSTIC
# ============================================================

print("=" * 70)
print("A3 CHECKPOINT KEY MISMATCH")
print("=" * 70)

missing, unexpected = a3_encoder.load_state_dict(
    a3_state,
    strict=False
)

print("\nMISSING KEYS:")
for k in missing:
    print("  ", k)

print("\nUNEXPECTED KEYS:")
for k in unexpected:
    print("  ", k)

print("\nCounts:")
print("Missing   :", len(missing))
print("Unexpected:", len(unexpected))

print("\n" + "=" * 70)
print("CHECKPOINT KEYS AROUND DIFFERENCES")
print("=" * 70)

# Show all A3 checkpoint keys after layer 3
for k in a3_state.keys():
    if (
        "layers.3" in k
        or "layers.4" in k
        or "layers.5" in k
        or "layers.6" in k
        or "layers.7" in k
        or "final" in k
        or "lm_head" in k
    ):
        print(k)

In [ ]:
# ============================================================
# A3 — CHECKPOINT-COMPATIBLE ARCHITECTURE
# ============================================================

class A3Attention(nn.MultiheadAttention):
    """
    Directly exposes the checkpoint keys:
        attn.in_proj_weight
        attn.in_proj_bias
        attn.out_proj.weight
        attn.out_proj.bias
    """

    def __init__(
        self,
        hidden_size=512,
        num_heads=8,
        dropout=0.1
    ):
        super().__init__(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )


class A3FFN(nn.Sequential):
    """
    Directly exposes:
        ffn.0.*
        ffn.3.*
    """

    def __init__(
        self,
        hidden_size=512,
        ffn_size=2048,
        dropout=0.1
    ):
        super().__init__(
            nn.Linear(hidden_size, ffn_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_size, hidden_size)
        )


class A3Layer(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        num_heads=8,
        ffn_size=2048,
        dropout=0.1
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            hidden_size
        )

        self.attn = A3Attention(
            hidden_size=hidden_size,
            num_heads=num_heads,
            dropout=dropout
        )

        self.ln2 = nn.LayerNorm(
            hidden_size
        )

        self.ffn = A3FFN(
            hidden_size=hidden_size,
            ffn_size=ffn_size,
            dropout=dropout
        )

    def forward(
        self,
        x,
        attention_mask=None
    ):

        B, T, C = x.shape

        # Causal attention mask
        causal_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # Padding mask
        key_padding_mask = None

        if attention_mask is not None:
            key_padding_mask = (
                attention_mask == 0
            )

        attn_out, _ = self.attn(
            self.ln1(x),
            self.ln1(x),
            self.ln1(x),
            attn_mask=causal_mask,
            key_padding_mask=key_padding_mask,
            need_weights=False
        )

        x = x + attn_out

        x = x + self.ffn(
            self.ln2(x)
        )

        return x


class A3LM(nn.Module):

    def __init__(
        self,
        vocab_size=50000,
        hidden_size=512,
        num_layers=8,
        num_heads=8,
        ffn_size=2048,
        max_seq_len=2048,
        dropout=0.1,
        prefix_ratio=0.5
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.max_seq_len = max_seq_len
        self.prefix_ratio = prefix_ratio

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            hidden_size
        )

        self.layers = nn.ModuleList([
            A3Layer(
                hidden_size=hidden_size,
                num_heads=num_heads,
                ffn_size=ffn_size,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        # CHECKPOINT EXPECTS ln_f
        self.ln_f = nn.LayerNorm(
            hidden_size
        )

        # CHECKPOINT EXPECTS lm_head.weight
        self.lm_head = nn.Linear(
            hidden_size,
            vocab_size,
            bias=False
        )

        # Phase-1 checkpoint uses tied embeddings.
        self.lm_head.weight = (
            self.token_embedding.weight
        )

    def forward(
        self,
        input_ids,
        attention_mask=None
    ):

        B, T = input_ids.shape

        assert T <= self.max_seq_len

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            +
            self.position_embedding(positions)
        )

        for layer in self.layers:

            x = layer(
                x,
                attention_mask=attention_mask
            )

        x = self.ln_f(x)

        return x


# ============================================================
# INSTANTIATE
# ============================================================

a3_encoder = A3LM(
    vocab_size=50000,
    hidden_size=512,
    num_layers=8,
    num_heads=8,
    ffn_size=2048,
    max_seq_len=2048,
    dropout=0.1,
    prefix_ratio=0.5
)


# ============================================================
# PARAMETER VALIDATION
# ============================================================

a3_model_params = sum(
    p.numel()
    for p in a3_encoder.parameters()
)

print("=" * 70)
print("A3 PARAMETER VALIDATION")
print("=" * 70)

print(
    "Model parameters     :",
    f"{a3_model_params:,}"
)

print(
    "Checkpoint parameters:",
    f"{total_params:,}"
)

print(
    "Difference           :",
    a3_model_params - total_params
)

assert (
    a3_model_params == total_params
)

print("✓ Parameter count EXACT")


# ============================================================
# STRICT LOAD
# ============================================================

missing, unexpected = (
    a3_encoder.load_state_dict(
        a3_state,
        strict=True
    )
)

print("✓ Strict checkpoint load PASSED")

print("\n" + "=" * 70)
print("A3 RECONSTRUCTION PASSED")
print("=" * 70)

In [ ]:
# ============================================================
# A3 — FINAL CHECKPOINT-COMPATIBLE RECONSTRUCTION
# ============================================================

class A3LM(nn.Module):

    def __init__(
        self,
        vocab_size=50000,
        hidden_size=512,
        num_layers=8,
        num_heads=8,
        ffn_size=2048,
        max_seq_len=2048,
        dropout=0.1,
        prefix_ratio=0.5
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.max_seq_len = max_seq_len
        self.prefix_ratio = prefix_ratio

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            hidden_size
        )

        self.layers = nn.ModuleList([
            A3Layer(
                hidden_size=hidden_size,
                num_heads=num_heads,
                ffn_size=ffn_size,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(
            hidden_size
        )

    def forward(
        self,
        input_ids,
        attention_mask=None
    ):

        B, T = input_ids.shape

        assert T <= self.max_seq_len

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            +
            self.position_embedding(positions)
        )

        for layer in self.layers:

            x = layer(
                x,
                attention_mask=attention_mask
            )

        x = self.ln_f(x)

        return x


# ============================================================
# INSTANTIATE
# ============================================================

a3_encoder = A3LM(
    vocab_size=50000,
    hidden_size=512,
    num_layers=8,
    num_heads=8,
    ffn_size=2048,
    max_seq_len=2048,
    dropout=0.1,
    prefix_ratio=0.5
)


# ============================================================
# PARAMETER COUNT
# ============================================================

a3_model_params = sum(
    p.numel()
    for p in a3_encoder.parameters()
)

print("=" * 70)
print("A3 FINAL VALIDATION")
print("=" * 70)

print(
    "Model parameters     :",
    f"{a3_model_params:,}"
)

print(
    "Checkpoint parameters:",
    f"{total_params:,}"
)

print(
    "Difference           :",
    a3_model_params - total_params
)

assert (
    a3_model_params == total_params
)

print("✓ Parameter count EXACT")


# ============================================================
# STRICT LOAD
# ============================================================

a3_encoder.load_state_dict(
    a3_state,
    strict=True
)

print("✓ Strict checkpoint load PASSED")

print("\n" + "=" * 70)
print("✓ A3 RECONSTRUCTION PASSED")
print("=" * 70)

In [ ]:
# ============================================================
# A3 — GPU FORWARD SMOKE TEST
# ============================================================

import time
import torch

print("=" * 70)
print("A3 GPU SMOKE TEST")
print("=" * 70)

a3_encoder = a3_encoder.to(device)
a3_encoder.eval()

batch = next(iter(a4_train_loader_exp2))

input_ids = batch["input_ids"][:1].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"][:1].to(
    device,
    non_blocking=True
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

# Warm-up
with torch.no_grad():
    _ = a3_encoder(
        input_ids,
        attention_mask=attention_mask
    )

torch.cuda.synchronize()

start = time.time()

with torch.no_grad():
    hidden = a3_encoder(
        input_ids,
        attention_mask=attention_mask
    )

torch.cuda.synchronize()

elapsed = time.time() - start

peak_vram = (
    torch.cuda.max_memory_allocated(device)
    / (1024 ** 3)
)

print("Input shape :", tuple(input_ids.shape))
print("Output shape:", tuple(hidden.shape))
print("Time        :", f"{elapsed:.3f}s")
print("Peak VRAM   :", f"{peak_vram:.2f} GB")
print("Finite      :", torch.isfinite(hidden).all().item())

assert hidden.shape == (
    1,
    input_ids.shape[1],
    512
)

assert torch.isfinite(hidden).all()

print("\n✓ A3 GPU forward passed")

del input_ids
del attention_mask
del hidden

torch.cuda.empty_cache()

In [ ]:
# ============================================================
# A3 — NER HEAD + FORWARD/BACKWARD SANITY TEST
# ============================================================

class A3NER(nn.Module):

    def __init__(
        self,
        encoder,
        hidden_size=512,
        num_labels=25,
        dropout=0.10
    ):
        super().__init__()

        self.encoder = encoder

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Linear(
            hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask=None
    ):

        hidden = self.encoder(
            input_ids,
            attention_mask=attention_mask
        )

        hidden = self.dropout(hidden)

        return self.classifier(hidden)


# ============================================================
# BUILD A3 NER
# ============================================================

model9 = A3NER(
    encoder=a3_encoder,
    hidden_size=512,
    num_labels=25,
    dropout=0.10
).to(device)


# ============================================================
# PARAMETER COUNT
# ============================================================

a3_total_params = sum(
    p.numel()
    for p in model9.parameters()
)

a3_trainable_params = sum(
    p.numel()
    for p in model9.parameters()
    if p.requires_grad
)

expected_ner_params = (
    51_868_672
    + (512 * 25)
    + 25
)

print("=" * 70)
print("A3 NER VALIDATION")
print("=" * 70)

print(
    "Total parameters     :",
    f"{a3_total_params:,}"
)

print(
    "Trainable parameters :",
    f"{a3_trainable_params:,}"
)

print(
    "Expected parameters  :",
    f"{expected_ner_params:,}"
)

assert (
    a3_total_params == expected_ner_params
)

assert (
    a3_trainable_params == expected_ner_params
)

print("✓ Parameter count EXACT")


# ============================================================
# FORWARD + BACKWARD
# ============================================================

batch = next(
    iter(a4_train_loader_exp2)
)

input_ids = batch["input_ids"].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    device,
    non_blocking=True
)

labels = batch["labels"].to(
    device,
    non_blocking=True
)

optimizer9 = torch.optim.AdamW(
    model9.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

model9.train()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

start = time.time()

logits = model9(
    input_ids,
    attention_mask=attention_mask
)

loss = F.cross_entropy(
    logits.view(-1, 25),
    labels.view(-1),
    ignore_index=-100
)

optimizer9.zero_grad(
    set_to_none=True
)

loss.backward()

grad_norm9 = (
    torch.nn.utils.clip_grad_norm_(
        model9.parameters(),
        max_norm=1.0
    )
)

optimizer9.step()

torch.cuda.synchronize()

elapsed = time.time() - start

peak_vram = (
    torch.cuda.max_memory_allocated(device)
    / (1024 ** 3)
)

finite_logits = torch.isfinite(
    logits
).all().item()

finite_loss = torch.isfinite(
    loss
).item()

finite_grads = all(
    p.grad is None
    or torch.isfinite(p.grad).all().item()
    for p in model9.parameters()
)


print("\n" + "=" * 70)
print("A3 NER FORWARD/BACKWARD SANITY TEST")
print("=" * 70)

print(
    f"Loss      : {loss.item():.4f}"
)

print(
    f"Logits    : {tuple(logits.shape)}"
)

print(
    f"Grad norm : {grad_norm9.item():.4f}"
)

print(
    f"Time      : {elapsed:.3f}s"
)

print(
    f"Peak VRAM : {peak_vram:.2f} GB"
)

print(
    "Finite logits:",
    finite_logits
)

print(
    "Finite loss:",
    finite_loss
)

print(
    "Finite gradients:",
    finite_grads
)

assert logits.shape == (
    input_ids.shape[0],
    input_ids.shape[1],
    25
)

assert finite_logits
assert finite_loss
assert finite_grads

print("\n✓ Forward passed")
print("✓ Loss passed")
print("✓ Backward passed")
print("✓ Optimizer step passed")
print("✓ Gradients finite")
print("✓ A3 NER sanity test PASSED")


# Cleanup
del input_ids
del attention_mask
del labels
del logits
del loss

torch.cuda.empty_cache()

In [ ]:
# ============================================================
# EXPERIMENT 9 — A3 FULL FINE-TUNING
# CLEAN FOLD-2 TRAINING
# ============================================================

import os
import time
import torch

EXP9_DIR = "/kaggle/working/phase2_exp9"
os.makedirs(EXP9_DIR, exist_ok=True)

# ------------------------------------------------------------
# IMPORTANT:
# Re-load the original Phase-1 A3 checkpoint.
# This prevents the sanity-test optimizer step from affecting
# the actual experiment.
# ------------------------------------------------------------

a3_encoder_exp9 = A3LM(
    vocab_size=50000,
    hidden_size=512,
    num_layers=8,
    num_heads=8,
    ffn_size=2048,
    max_seq_len=2048,
    dropout=0.1,
    prefix_ratio=0.5
)

a3_encoder_exp9.load_state_dict(
    a3_state,
    strict=True
)

model9 = A3NER(
    encoder=a3_encoder_exp9,
    hidden_size=512,
    num_labels=25,
    dropout=0.10
).to(device)

optimizer9 = torch.optim.AdamW(
    model9.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

NUM_EPOCHS = 3

print("=" * 70)
print("EXPERIMENT 9 — A3 FULL FINE-TUNING")
print("=" * 70)

print("Training chunks :", len(a4_train_only_chunks))
print("Training batches:", len(a4_train_loader_exp2))
print("Validation docs :", len(fold1_ids))
print("Epochs           :", NUM_EPOCHS)
print("Learning rate    :", "2e-5")
print("Weight decay     :", "0.01")
print("Batch size       :", 4)

total_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):

    model9.train()

    epoch_start = time.time()
    running_loss = 0.0
    valid_tokens = 0

    for step, batch in enumerate(
        a4_train_loader_exp2,
        start=1
    ):

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        optimizer9.zero_grad(
            set_to_none=True
        )

        logits = model9(
            input_ids,
            attention_mask=attention_mask
        )

        loss = F.cross_entropy(
            logits.view(-1, 25),
            labels.view(-1),
            ignore_index=-100
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model9.parameters(),
            max_norm=1.0
        )

        optimizer9.step()

        n_valid = (
            labels != -100
        ).sum().item()

        running_loss += (
            loss.item() * n_valid
        )

        valid_tokens += n_valid

        del input_ids
        del attention_mask
        del labels
        del logits
        del loss

        if (
            step % 25 == 0
            or step == len(a4_train_loader_exp2)
        ):

            current_loss = (
                running_loss / valid_tokens
            )

            elapsed = (
                time.time() - epoch_start
            )

            print(
                f"Epoch {epoch}/{NUM_EPOCHS} | "
                f"Batch {step}/{len(a4_train_loader_exp2)} | "
                f"Loss {current_loss:.4f} | "
                f"Time {elapsed:.1f}s"
            )

    epoch_loss = (
        running_loss / valid_tokens
    )

    epoch_time = (
        time.time() - epoch_start
    )

    ckpt_path = (
        f"{EXP9_DIR}/"
        f"a3_ner_exp9_fullft_epoch{epoch}.pt"
    )

    torch.save(
        {
            "experiment": 9,
            "architecture": "A3",
            "technique": "Full Fine-Tuning",
            "epoch": epoch,
            "train_loss": epoch_loss,
            "model_state_dict": model9.state_dict(),
            "optimizer_state_dict": optimizer9.state_dict(),
        },
        ckpt_path
    )

    print(
        f"\n✓ Epoch {epoch} complete | "
        f"loss={epoch_loss:.4f} | "
        f"time={epoch_time:.1f}s"
    )

    print(
        f"✓ Checkpoint saved: {ckpt_path}"
    )

total_time = time.time() - total_start

print("\n" + "=" * 70)
print("A3 TRAINING COMPLETE")
print("=" * 70)

print(
    "Total training time:",
    f"{total_time:.1f}s"
)

print(
    "Total training time:",
    f"{total_time/60:.2f} min"
)

print("✓ Clean Fold-2 training finished")

In [ ]:
# ============================================================
# EXPERIMENT 9 — A3 CLEAN FOLD-1 EVALUATION
# ============================================================

import torch

print("=" * 70)
print("EXPERIMENT 9 — A3 CLEAN FOLD-1 EVALUATION")
print("=" * 70)

results_exp9 = []

for epoch in [1, 2, 3]:

    ckpt_path = (
        f"/kaggle/working/phase2_exp9/"
        f"a3_ner_exp9_fullft_epoch{epoch}.pt"
    )

    print(f"\n{'-' * 70}")
    print(f"Evaluating A3 Epoch {epoch}")
    print(f"{'-' * 70}")

    checkpoint = torch.load(
        ckpt_path,
        map_location=device,
        weights_only=False
    )

    model9.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )

    model9.eval()

    predictions = {}

    with torch.no_grad():

        for batch in a4_val_loader:

            input_ids = batch["input_ids"].to(
                device,
                non_blocking=True
            )

            attention_mask = batch["attention_mask"].to(
                device,
                non_blocking=True
            )

            logits = model9(
                input_ids,
                attention_mask=attention_mask
            )

            pred_ids = logits.argmax(
                dim=-1
            ).cpu().tolist()

            for i, doc_id in enumerate(
                batch["doc_id"]
            ):

                spans = bio_to_token_spans(
                    pred_ids[i],
                    batch["offsets"][i],
                    fold1_text[doc_id]
                )

                clean_spans = []

                for start, end, label in spans:

                    start, end, label = (
                        normalize_span_boundaries(
                            start,
                            end,
                            label,
                            fold1_text[doc_id]
                        )
                    )

                    if end > start:
                        clean_spans.append(
                            (start, end, label)
                        )

                if doc_id not in predictions:
                    predictions[doc_id] = set()

                predictions[doc_id].update(
                    clean_spans
                )

            del input_ids
            del attention_mask
            del logits

    predictions = {
        doc_id: list(spans)
        for doc_id, spans in predictions.items()
    }

    for doc_id in fold1_ids:
        predictions.setdefault(
            doc_id,
            []
        )

    metrics = strict_macro_f1(
        predictions,
        fold1_gold
    )

    print(
        f"Strict Macro-F1 : "
        f"{metrics['macro_f1']:.4f}"
    )

    print(
        f"Strict Macro-P  : "
        f"{metrics['macro_precision']:.4f}"
    )

    print(
        f"Strict Macro-R  : "
        f"{metrics['macro_recall']:.4f}"
    )

    results_exp9.append({
        "epoch": epoch,
        "f1": metrics["macro_f1"],
        "precision": metrics["macro_precision"],
        "recall": metrics["macro_recall"],
        "per_label": metrics["per_label"]
    })

    del checkpoint
    torch.cuda.empty_cache()


print("\n" + "=" * 70)
print("A3 EXPERIMENT 9 RESULTS")
print("=" * 70)

for r in results_exp9:

    print(
        f"Epoch {r['epoch']} | "
        f"F1={r['f1']:.4f} | "
        f"P={r['precision']:.4f} | "
        f"R={r['recall']:.4f}"
    )

best_exp9 = max(
    results_exp9,
    key=lambda x: x["f1"]
)

print("\nBEST A3 CHECKPOINT")
print(
    f"Epoch     : {best_exp9['epoch']}"
)

print(
    f"Strict F1 : {best_exp9['f1']:.4f}"
)

print(
    f"Precision : {best_exp9['precision']:.4f}"
)

print(
    f"Recall    : {best_exp9['recall']:.4f}"
)

print("\n✓ A3 evaluation complete")

In [ ]:
# ============================================================
# EXPERIMENT 10 — A4 CLASS-AWARE KD
# CELL 1: FOLD-2 CLASS WEIGHTS
# ============================================================

from collections import Counter
import torch
import torch.nn.functional as F

print("=" * 70)
print("EXPERIMENT 10 — CLASS WEIGHT PREPARATION")
print("=" * 70)

label_counts_exp10 = Counter()

for chunk in a4_train_only_chunks:
    label_counts_exp10.update(
        int(x)
        for x in chunk.bio_ids
        if int(x) >= 0
    )

print("\nFold-2 BIO label counts:")

for label_id in range(25):
    print(
        f"{label_id:2d} | "
        f"{id2label[label_id]:12s} | "
        f"{label_counts_exp10[label_id]:8d}"
    )

# ------------------------------------------------------------
# Inverse-frequency weighting.
# Keep O down-weighted, but prevent extreme weights.
# ------------------------------------------------------------

counts = torch.tensor(
    [
        label_counts_exp10[i]
        for i in range(25)
    ],
    dtype=torch.float32
)

freq = counts / counts.sum()

weights_exp10 = 1.0 / torch.sqrt(
    freq + 1e-8
)

# Normalize around mean 1
weights_exp10 = (
    weights_exp10 /
    weights_exp10.mean()
)

# Avoid extreme weights
weights_exp10 = torch.clamp(
    weights_exp10,
    min=0.05,
    max=4.0
)

# Explicitly reduce O contribution
weights_exp10[0] = 0.05

# Renormalize non-O weights mildly
non_o_mean = weights_exp10[1:].mean()
weights_exp10[1:] = (
    weights_exp10[1:] /
    non_o_mean
)

print("\nClass weights:")

for label_id in range(25):
    print(
        f"{label_id:2d} | "
        f"{id2label[label_id]:12s} | "
        f"{weights_exp10[label_id]:.4f}"
    )

assert torch.isfinite(
    weights_exp10
).all()

assert weights_exp10.shape == (25,)

print("\n✓ Class weights created from Fold-2 only")

In [ ]:
# ============================================================
# EXPERIMENT 10 — CELL 2
# A4 LORMA TEACHER + FRESH A4 STUDENT
# ============================================================

import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

EXP10_DIR = "/kaggle/working/phase2_exp10"
os.makedirs(EXP10_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. LOAD EXP4 LORMA TEACHER
# ------------------------------------------------------------

teacher_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    num_layers=6,
    num_heads=8,
    ffn_size=2048,
    max_seq_len=1024,
    dropout=0.1
)

teacher_model = A4NER(
    teacher_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)

teacher_path = (
    "/kaggle/working/phase2_exp4/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

teacher_ckpt = torch.load(
    teacher_path,
    map_location=device,
    weights_only=False
)

teacher_model.load_state_dict(
    teacher_ckpt["model_state_dict"],
    strict=True
)

teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad = False

print("=" * 70)
print("EXP10 — TEACHER")
print("=" * 70)

print("Teacher:", teacher_path)
print("✓ Exp4 LORMA teacher loaded")
print("✓ Teacher frozen")


# ------------------------------------------------------------
# 2. FRESH A4 STUDENT
# ------------------------------------------------------------

student_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    num_layers=6,
    num_heads=8,
    ffn_size=2048,
    max_seq_len=1024,
    dropout=0.1
)

# IMPORTANT:
# Load original Phase-1 A4 checkpoint, not Exp4 weights.
student_encoder.load_state_dict(
    a4_state,
    strict=True
)

student_model = A4NER(
    student_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)

student_total = sum(
    p.numel()
    for p in student_model.parameters()
)

student_trainable = sum(
    p.numel()
    for p in student_model.parameters()
    if p.requires_grad
)

print("\n" + "=" * 70)
print("EXP10 — STUDENT")
print("=" * 70)

print(
    "Total parameters     :",
    f"{student_total:,}"
)

print(
    "Trainable parameters :",
    f"{student_trainable:,}"
)

assert student_total == 45_051_417
assert student_trainable == 45_051_417

print("✓ Fresh A4 student validated")


# ------------------------------------------------------------
# 3. CLASS WEIGHTS → GPU
# ------------------------------------------------------------

weights_exp10 = weights_exp10.to(device)

print(
    "\nClass weights device:",
    weights_exp10.device
)

print("\n✓ EXP10 teacher/student setup complete")

In [ ]:
# ============================================================
# EXPERIMENT 10 — CELL 2
# A4 LORMA TEACHER + FRESH A4 STUDENT
# ============================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F

EXP10_DIR = "/kaggle/working/phase2_exp10"
os.makedirs(EXP10_DIR, exist_ok=True)


# ============================================================
# 1. LOAD EXP4 LORMA TEACHER
# ============================================================

teacher_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    num_layers=6,
    num_heads=8,
    ffn_size=2048,
    dropout=0.1
)

teacher_model = A4NER(
    teacher_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)

teacher_path = (
    "/kaggle/working/phase2_exp4/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

teacher_ckpt = torch.load(
    teacher_path,
    map_location=device,
    weights_only=False
)

teacher_model.load_state_dict(
    teacher_ckpt["model_state_dict"],
    strict=True
)

teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad = False

print("=" * 70)
print("EXP10 — TEACHER")
print("=" * 70)

print("Teacher:", teacher_path)
print("✓ Exp4 LORMA teacher loaded")
print("✓ Teacher frozen")


# ============================================================
# 2. FRESH A4 STUDENT
# ============================================================

student_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    num_layers=6,
    num_heads=8,
    ffn_size=2048,
    dropout=0.1
)

# Load ORIGINAL Phase-1 A4 weights
student_encoder.load_state_dict(
    a4_state,
    strict=True
)

student_model = A4NER(
    student_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)

student_total = sum(
    p.numel()
    for p in student_model.parameters()
)

student_trainable = sum(
    p.numel()
    for p in student_model.parameters()
    if p.requires_grad
)

print("\n" + "=" * 70)
print("EXP10 — STUDENT")
print("=" * 70)

print(
    "Total parameters     :",
    f"{student_total:,}"
)

print(
    "Trainable parameters :",
    f"{student_trainable:,}"
)

assert student_total == 45_051_417
assert student_trainable == 45_051_417

print("✓ Fresh A4 student validated")


# ============================================================
# 3. CLASS WEIGHTS
# ============================================================

weights_exp10 = weights_exp10.to(device)

print(
    "\nClass weights device:",
    weights_exp10.device
)

print("\n✓ EXP10 teacher/student setup complete")

In [ ]:
import inspect

print(inspect.signature(A4Encoder))
print(inspect.signature(A4Encoder.__init__))

In [ ]:
# ============================================================
# EXPERIMENT 10 — CELL 2
# A4 LORMA TEACHER + FRESH A4 STUDENT
# ============================================================

import os
import torch

EXP10_DIR = "/kaggle/working/phase2_exp10"
os.makedirs(EXP10_DIR, exist_ok=True)


# ============================================================
# 1. LOAD EXP4 LORMA TEACHER
# ============================================================

teacher_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)

teacher_model = A4NER(
    teacher_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)

teacher_path = (
    "/kaggle/working/phase2_exp4/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

teacher_ckpt = torch.load(
    teacher_path,
    map_location=device,
    weights_only=False
)

teacher_model.load_state_dict(
    teacher_ckpt["model_state_dict"],
    strict=True
)

teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad = False

print("=" * 70)
print("EXP10 — TEACHER")
print("=" * 70)

print("Teacher:", teacher_path)
print("✓ Exp4 LORMA teacher loaded")
print("✓ Teacher frozen")


# ============================================================
# 2. FRESH A4 STUDENT
# ============================================================

student_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)

# ORIGINAL Phase-1 A4 weights
student_encoder.load_state_dict(
    a4_state,
    strict=True
)

student_model = A4NER(
    student_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)

student_total = sum(
    p.numel()
    for p in student_model.parameters()
)

student_trainable = sum(
    p.numel()
    for p in student_model.parameters()
    if p.requires_grad
)

print("\n" + "=" * 70)
print("EXP10 — STUDENT")
print("=" * 70)

print(
    "Total parameters     :",
    f"{student_total:,}"
)

print(
    "Trainable parameters :",
    f"{student_trainable:,}"
)

assert student_total == 45_051_417
assert student_trainable == 45_051_417

print("✓ Fresh A4 student validated")


# ============================================================
# 3. CLASS WEIGHTS
# ============================================================

weights_exp10 = weights_exp10.to(device)

print(
    "\nClass weights device:",
    weights_exp10.device
)

print("\n✓ EXP10 teacher/student setup complete")

In [ ]:
# ============================================================
# EXP10 — INSPECT EXISTING EXP4 LORMA TEACHER
# ============================================================

print("=" * 70)
print("EXP4 LORMA TEACHER CHECKPOINT INSPECTION")
print("=" * 70)

print("Checkpoint:")
print(teacher_path)

teacher_state = teacher_ckpt["model_state_dict"]

print("\nNumber of tensors:")
print(len(teacher_state))

print("\nTotal parameters:")
print(
    sum(
        v.numel()
        for v in teacher_state.values()
        if torch.is_tensor(v)
    )
)

print("\nFirst 80 checkpoint keys:")

for k, v in list(teacher_state.items())[:80]:
    print(
        f"{k:75s} "
        f"{tuple(v.shape)} "
        f"{v.dtype}"
    )

print("\n" + "=" * 70)
print("LORMA-SPECIFIC KEYS")
print("=" * 70)

lorma_keys = [
    k for k in teacher_state.keys()
    if "lorma" in k.lower()
]

print("Number:", len(lorma_keys))

for k in lorma_keys[:40]:
    print(k)

print("\n✓ Teacher checkpoint inspection complete")

In [ ]:
# ============================================================
# EXP10 — FIND EXISTING LORMA CLASS
# ============================================================

import inspect

print("=" * 70)
print("AVAILABLE LORMA CLASSES")
print("=" * 70)

for name in [
    "LORMA",
    "LORMAConfig",
    "LORMALinear",
    "LormaLinear",
    "A4LORMA",
    "LORMAWrapper",
]:
    obj = globals().get(name, None)

    if obj is not None:
        print(
            f"\n{name}:"
        )
        print(
            inspect.signature(obj)
            if callable(obj)
            else type(obj)
        )

In [ ]:
# Show classes/objects in the notebook whose names contain LORMA

print("\nObjects containing 'LORMA':")

for name, obj in globals().items():
    if "lorma" in name.lower():
        print(
            name,
            type(obj)
        )

In [ ]:
# ============================================================
# EXP10 — CELL 3
# REBUILD EXACT EXP4 LORMA TEACHER
# ============================================================

# Start from a fresh A4 encoder
teacher_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)

# ------------------------------------------------------------
# Apply the SAME LORMA configuration used in Exp4
# to q_proj and v_proj of all 6 encoder layers.
# ------------------------------------------------------------

for layer in teacher_encoder.encoder_layers:

    layer.attn.q_proj = LORMALinear(
        layer.attn.q_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )

    layer.attn.v_proj = LORMALinear(
        layer.attn.v_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )


# ------------------------------------------------------------
# Build NER teacher
# ------------------------------------------------------------

teacher_model = A4NER(
    teacher_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)


# ------------------------------------------------------------
# Load Exp4 checkpoint
# ------------------------------------------------------------

teacher_path = (
    "/kaggle/working/phase2_exp4/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

teacher_ckpt = torch.load(
    teacher_path,
    map_location=device,
    weights_only=False
)

teacher_model.load_state_dict(
    teacher_ckpt["model_state_dict"],
    strict=True
)


# ------------------------------------------------------------
# Freeze teacher
# ------------------------------------------------------------

teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad = False


# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

teacher_total = sum(
    p.numel()
    for p in teacher_model.parameters()
)

teacher_trainable = sum(
    p.numel()
    for p in teacher_model.parameters()
    if p.requires_grad
)

print("=" * 70)
print("EXP10 — LORMA TEACHER VALIDATION")
print("=" * 70)

print(
    "Teacher parameters:",
    f"{teacher_total:,}"
)

print(
    "Trainable parameters:",
    teacher_trainable
)

print(
    "Expected parameters:",
    "45,149,721"
)

assert teacher_total == 45_149_721
assert teacher_trainable == 0

print("✓ Exact Exp4 teacher reconstructed")
print("✓ Exp4 checkpoint loaded")
print("✓ Teacher completely frozen")

In [ ]:
# ============================================================
# EXP10 — EXACT EXP4 LORMA TEACHER
# FIX: CHECKPOINT-COMPATIBLE FFN NAMING
# ============================================================

import torch
import torch.nn as nn


# ------------------------------------------------------------
# Replace each A4 FFN with a wrapper exposing ffn.net.*
# ------------------------------------------------------------

class A4FFNCheckpoint(nn.Module):

    def __init__(
        self,
        hidden_size=512,
        ffn_size=2048
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_size),
            nn.GELU(),
            nn.Linear(ffn_size, hidden_size)
        )

    def forward(self, x):
        return self.net(x)


# ------------------------------------------------------------
# Fresh A4 encoder
# ------------------------------------------------------------

teacher_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)


# ------------------------------------------------------------
# Replace FFNs with checkpoint-compatible wrappers
# ------------------------------------------------------------

for layer in teacher_encoder.encoder_layers:

    layer.ffn = A4FFNCheckpoint(
        hidden_size=512,
        ffn_size=2048
    )


# ------------------------------------------------------------
# Apply exact Exp4 LORMA configuration
# ------------------------------------------------------------

for layer in teacher_encoder.encoder_layers:

    layer.attn.q_proj = LORMALinear(
        layer.attn.q_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )

    layer.attn.v_proj = LORMALinear(
        layer.attn.v_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )


# ------------------------------------------------------------
# Build teacher NER
# ------------------------------------------------------------

teacher_model = A4NER(
    teacher_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)


# ------------------------------------------------------------
# Load Exp4 checkpoint
# ------------------------------------------------------------

teacher_path = (
    "/kaggle/working/phase2_exp4/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

teacher_ckpt = torch.load(
    teacher_path,
    map_location=device,
    weights_only=False
)

teacher_model.load_state_dict(
    teacher_ckpt["model_state_dict"],
    strict=True
)


# ------------------------------------------------------------
# Freeze teacher
# ------------------------------------------------------------

teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad = False


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

teacher_total = sum(
    p.numel()
    for p in teacher_model.parameters()
)

teacher_trainable = sum(
    p.numel()
    for p in teacher_model.parameters()
    if p.requires_grad
)

print("=" * 70)
print("EXP10 — LORMA TEACHER VALIDATION")
print("=" * 70)

print(
    "Teacher parameters:",
    f"{teacher_total:,}"
)

print(
    "Trainable parameters:",
    teacher_trainable
)

print(
    "Expected parameters:",
    "45,149,721"
)

assert teacher_total == 45_149_721
assert teacher_trainable == 0

print("✓ Exact Exp4 teacher reconstructed")
print("✓ Exp4 checkpoint loaded")
print("✓ Teacher completely frozen")

In [ ]:
# ============================================================
# EXP10 — CELL 4
# FRESH A4 STUDENT + KD SANITY TEST
# ============================================================

# Fresh student from ORIGINAL Phase-1 A4
student_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)

student_encoder.load_state_dict(
    a4_state,
    strict=True
)

student_model = A4NER(
    student_encoder,
    num_labels=25,
    dropout=0.10,
    hidden_size=512
).to(device)

student_total = sum(
    p.numel()
    for p in student_model.parameters()
)

print("=" * 70)
print("EXP10 — STUDENT VALIDATION")
print("=" * 70)

print(
    "Student parameters:",
    f"{student_total:,}"
)

assert student_total == 45_051_417

print("✓ Fresh Phase-1 A4 student loaded")


# ============================================================
# KD SANITY BATCH
# ============================================================

batch = next(
    iter(a4_train_loader_exp2)
)

input_ids = batch["input_ids"].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    device,
    non_blocking=True
)

labels = batch["labels"].to(
    device,
    non_blocking=True
)

optimizer10 = torch.optim.AdamW(
    student_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

T = 2.0
ALPHA = 0.5

student_model.train()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

# ------------------------------------------------------------
# Teacher
# ------------------------------------------------------------

with torch.no_grad():

    teacher_logits = teacher_model(
        input_ids,
        attention_mask=attention_mask
    )


# ------------------------------------------------------------
# Student
# ------------------------------------------------------------

student_logits = student_model(
    input_ids,
    attention_mask=attention_mask
)


# ------------------------------------------------------------
# Weighted hard-label CE
# ------------------------------------------------------------

hard_loss = F.cross_entropy(
    student_logits.view(-1, 25),
    labels.view(-1),
    weight=weights_exp10,
    ignore_index=-100
)


# ------------------------------------------------------------
# Soft KD loss
# ------------------------------------------------------------

valid_mask = (
    labels.view(-1) != -100
)

s_logits = student_logits.view(-1, 25)[valid_mask]
t_logits = teacher_logits.view(-1, 25)[valid_mask]

soft_loss = F.kl_div(
    F.log_softmax(
        s_logits / T,
        dim=-1
    ),
    F.softmax(
        t_logits / T,
        dim=-1
    ),
    reduction="batchmean"
) * (T ** 2)


# ------------------------------------------------------------
# Combined loss
# ------------------------------------------------------------

total_loss = (
    ALPHA * hard_loss
    +
    (1.0 - ALPHA) * soft_loss
)

optimizer10.zero_grad(
    set_to_none=True
)

total_loss.backward()

grad_norm10 = torch.nn.utils.clip_grad_norm_(
    student_model.parameters(),
    max_norm=1.0
)

optimizer10.step()

torch.cuda.synchronize()

peak_vram = (
    torch.cuda.max_memory_allocated(device)
    / (1024 ** 3)
)

print("\n" + "=" * 70)
print("EXP10 — KD SANITY TEST")
print("=" * 70)

print(
    f"Hard CE loss : {hard_loss.item():.4f}"
)

print(
    f"Soft KD loss : {soft_loss.item():.4f}"
)

print(
    f"Total loss   : {total_loss.item():.4f}"
)

print(
    f"Grad norm    : {grad_norm10.item():.4f}"
)

print(
    f"Peak VRAM    : {peak_vram:.2f} GB"
)

print(
    "Finite student logits:",
    torch.isfinite(student_logits).all().item()
)

print(
    "Finite teacher logits:",
    torch.isfinite(teacher_logits).all().item()
)

print(
    "Finite total loss:",
    torch.isfinite(total_loss).item()
)

assert student_logits.shape == (
    input_ids.shape[0],
    input_ids.shape[1],
    25
)

assert teacher_logits.shape == student_logits.shape

assert torch.isfinite(
    student_logits
).all()

assert torch.isfinite(
    teacher_logits
).all()

assert torch.isfinite(
    total_loss
)

print("\n✓ Teacher forward passed")
print("✓ Student forward passed")
print("✓ Weighted CE passed")
print("✓ KD loss passed")
print("✓ Backward passed")
print("✓ Optimizer step passed")
print("✓ EXP10 sanity test PASSED")


# Cleanup
del input_ids
del attention_mask
del labels
del teacher_logits
del student_logits
del hard_loss
del soft_loss
del total_loss

torch.cuda.empty_cache()

In [ ]:
# ============================================================
# EXP10 — RESTORE ORIGINAL PHASE-1 A4 CHECKPOINT
# ============================================================

A4_PATH = (
    "/kaggle/input/datasets/aq1de3/a4check/"
    "a4_step_22000.pt"
)

a4_ckpt = torch.load(
    A4_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(a4_ckpt))

if isinstance(a4_ckpt, dict):
    print("Checkpoint keys:", list(a4_ckpt.keys()))

# Extract state dict
if "model_state_dict" in a4_ckpt:
    a4_state = a4_ckpt["model_state_dict"]
elif "state_dict" in a4_ckpt:
    a4_state = a4_ckpt["state_dict"]
elif "model" in a4_ckpt and isinstance(a4_ckpt["model"], dict):
    a4_state = a4_ckpt["model"]
else:
    a4_state = a4_ckpt

print("A4 tensors:", len(a4_state))

a4_params = sum(
    v.numel()
    for v in a4_state.values()
    if torch.is_tensor(v)
)

print(
    "A4 checkpoint parameters:",
    f"{a4_params:,}"
)

assert a4_params == 96_389_120

print("✓ Original Phase-1 A4 checkpoint restored")

In [ ]:
# ============================================================
# EXP10 — CORRECT FRESH A4 STUDENT RECONSTRUCTION
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

print("=" * 70)
print("EXP10 — CORRECT A4 STUDENT RECONSTRUCTION")
print("=" * 70)


# ------------------------------------------------------------
# 1. Checkpoint-compatible FFN wrapper
# ------------------------------------------------------------

class A4FFNCheckpoint(nn.Module):
    def __init__(self, hidden_size=512, ffn_size=2048):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_size),
            nn.GELU(),
            nn.Linear(ffn_size, hidden_size)
        )

    def forward(self, x):
        return self.net(x)


# ------------------------------------------------------------
# 2. Fresh A4 encoder
# ------------------------------------------------------------

student_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)


# ------------------------------------------------------------
# 3. Replace FFNs with checkpoint-compatible wrappers
# ------------------------------------------------------------

for layer in student_encoder.encoder_layers:
    layer.ffn = A4FFNCheckpoint(
        hidden_size=512,
        ffn_size=2048
    )


# ------------------------------------------------------------
# 4. Extract ONLY encoder weights from Phase-1 A4
# ------------------------------------------------------------

encoder_state = {}

for key, value in a4_state.items():

    if key.startswith("encoder."):
        new_key = key[len("encoder."):]
        encoder_state[new_key] = value

    elif key.startswith("encoder_layers."):
        encoder_state[key] = value

    elif key.startswith("encoder_pos."):
        encoder_state[key] = value

    elif key.startswith("token_embedding."):
        encoder_state[key] = value


print("Full checkpoint tensors :", len(a4_state))
print("Encoder tensors extracted:", len(encoder_state))


# ------------------------------------------------------------
# 5. Strict validation
# ------------------------------------------------------------

missing, unexpected = student_encoder.load_state_dict(
    encoder_state,
    strict=False
)

print("\nMissing keys:")
for k in missing:
    print(" ", k)

print("\nUnexpected keys:")
for k in unexpected:
    print(" ", k)

assert len(missing) == 0, "Encoder checkpoint is missing parameters!"
assert len(unexpected) == 0, "Unexpected encoder parameters found!"

print("\n✓ Encoder checkpoint loaded EXACTLY")


# ------------------------------------------------------------
# 6. Build NER student
# ------------------------------------------------------------

student_model = A4NER(
    encoder=student_encoder,
    num_labels=25
).to(device)


student_params = sum(
    p.numel()
    for p in student_model.parameters()
)

print(f"\nStudent parameters : {student_params:,}")
print("Expected           : 45,051,417")

assert student_params == 45_051_417

print("✓ Student parameter count EXACT")


# ------------------------------------------------------------
# 7. Optimizer
# ------------------------------------------------------------

optimizer10 = optim.AdamW(
    student_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

T = 2.0
ALPHA = 0.5


# ------------------------------------------------------------
# 8. Real training batch
# ------------------------------------------------------------

student_model.train()
teacher_model.eval()

batch = next(iter(a4_train_loader_exp2))

input_ids = batch["input_ids"].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    device,
    non_blocking=True
)

labels = batch["labels"].to(
    device,
    non_blocking=True
)

print("\nBatch:")
print("input_ids :", tuple(input_ids.shape))
print("labels    :", tuple(labels.shape))
print("valid tok :", int(attention_mask.sum().item()))


# ------------------------------------------------------------
# 9. Teacher forward
# ------------------------------------------------------------

with torch.no_grad():

    teacher_logits = teacher_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )


print(
    "Teacher logits:",
    tuple(teacher_logits.shape)
)


# ------------------------------------------------------------
# 10. Student forward
# ------------------------------------------------------------

student_logits = student_model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

print(
    "Student logits:",
    tuple(student_logits.shape)
)


# ------------------------------------------------------------
# 11. Weighted hard CE
# ------------------------------------------------------------

hard_loss = F.cross_entropy(
    student_logits.reshape(-1, 25),
    labels.reshape(-1),
    weight=weights_exp10,
    ignore_index=-100
)


# ------------------------------------------------------------
# 12. Soft KD
# ------------------------------------------------------------

valid_mask = (
    attention_mask.bool()
    & (labels != -100)
)

student_valid = student_logits[valid_mask]
teacher_valid = teacher_logits[valid_mask]

soft_loss = F.kl_div(
    F.log_softmax(
        student_valid / T,
        dim=-1
    ),
    F.softmax(
        teacher_valid / T,
        dim=-1
    ),
    reduction="batchmean"
) * (T ** 2)


# ------------------------------------------------------------
# 13. Combined loss
# ------------------------------------------------------------

total_loss = (
    ALPHA * hard_loss
    + (1.0 - ALPHA) * soft_loss
)

print("\nLosses:")
print(f"Hard CE : {hard_loss.item():.6f}")
print(f"Soft KD : {soft_loss.item():.6f}")
print(f"Total   : {total_loss.item():.6f}")


# ------------------------------------------------------------
# 14. Backward
# ------------------------------------------------------------

optimizer10.zero_grad(set_to_none=True)

total_loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    student_model.parameters(),
    max_norm=1.0
)

print(
    f"Grad norm: {float(grad_norm):.6f}"
)


# ------------------------------------------------------------
# 15. Optimizer step
# ------------------------------------------------------------

optimizer10.step()

print("✓ Optimizer step passed")


# ------------------------------------------------------------
# 16. Numerical checks
# ------------------------------------------------------------

checks = {
    "student_logits_finite":
        torch.isfinite(student_logits).all().item(),

    "teacher_logits_finite":
        torch.isfinite(teacher_logits).all().item(),

    "hard_loss_finite":
        torch.isfinite(hard_loss).item(),

    "soft_loss_finite":
        torch.isfinite(soft_loss).item(),

    "total_loss_finite":
        torch.isfinite(total_loss).item(),

    "grad_norm_finite":
        torch.isfinite(
            torch.as_tensor(grad_norm)
        ).item()
}

print("\nNumerical checks:")

for name, value in checks.items():
    print(f"{name:25s}: {value}")

assert all(checks.values())

if torch.cuda.is_available():

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    print(
        f"\nPeak VRAM: {peak_vram:.2f} GB"
    )


print("\n" + "=" * 70)
print("✓ EXP10 KD SANITY TEST PASSED")
print("=" * 70)

In [ ]:
# ------------------------------------------------------------
# Soft KD
# ------------------------------------------------------------

valid_mask = (
    attention_mask.bool()
    & (labels != -100)
)

student_valid = student_logits[valid_mask]
teacher_valid = teacher_logits[valid_mask]

soft_loss = F.kl_div(
    F.log_softmax(
        student_valid / T,
        dim=-1
    ),
    F.softmax(
        teacher_valid / T,
        dim=-1
    ),
    reduction="batchmean"
) * (T ** 2)

total_loss = (
    ALPHA * hard_loss
    + (1.0 - ALPHA) * soft_loss
)

print(f"Soft KD: {soft_loss.item():.6f}")
print(f"Total:   {total_loss.item():.6f}")

optimizer10.zero_grad(set_to_none=True)

total_loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    student_model.parameters(),
    max_norm=1.0
)

optimizer10.step()

print(f"Grad norm: {float(grad_norm):.6f}")
print("✓ Optimizer step passed")

assert torch.isfinite(student_logits).all()
assert torch.isfinite(teacher_logits).all()
assert torch.isfinite(total_loss)
assert torch.isfinite(torch.as_tensor(grad_norm))

print("\n" + "=" * 70)
print("✓ EXP10 KD SANITY TEST PASSED")
print("=" * 70)

if torch.cuda.is_available():
    print(
        f"Peak VRAM: "
        f"{torch.cuda.max_memory_allocated() / (1024**3):.2f} GB"
    )

In [ ]:
# ============================================================
# EXP10 — FRESH FORWARD + BACKWARD SANITY TEST
# ============================================================

# Make sure weights are on the same device
weights_exp10 = weights_exp10.to(device)

student_model.train()
teacher_model.eval()

# ------------------------------------------------------------
# Fresh teacher forward
# ------------------------------------------------------------

with torch.no_grad():
    teacher_logits = teacher_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

# ------------------------------------------------------------
# Fresh student forward
# ------------------------------------------------------------

student_logits = student_model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

# ------------------------------------------------------------
# Hard weighted CE
# ------------------------------------------------------------

hard_loss = F.cross_entropy(
    student_logits.reshape(-1, 25),
    labels.reshape(-1),
    weight=weights_exp10,
    ignore_index=-100
)

# ------------------------------------------------------------
# Soft KD
# ------------------------------------------------------------

valid_mask = (
    attention_mask.bool()
    & (labels != -100)
)

student_valid = student_logits[valid_mask]
teacher_valid = teacher_logits[valid_mask]

soft_loss = F.kl_div(
    F.log_softmax(student_valid / T, dim=-1),
    F.softmax(teacher_valid / T, dim=-1),
    reduction="batchmean"
) * (T ** 2)

# ------------------------------------------------------------
# Combined loss
# ------------------------------------------------------------

total_loss = (
    ALPHA * hard_loss
    + (1.0 - ALPHA) * soft_loss
)

print("=" * 70)
print("EXP10 — FRESH KD SANITY")
print("=" * 70)

print(f"Hard CE : {hard_loss.item():.6f}")
print(f"Soft KD : {soft_loss.item():.6f}")
print(f"Total   : {total_loss.item():.6f}")

print(
    "Student logits:",
    tuple(student_logits.shape)
)

print(
    "Teacher logits:",
    tuple(teacher_logits.shape)
)

# ------------------------------------------------------------
# Fresh backward — ONLY ONCE
# ------------------------------------------------------------

optimizer10.zero_grad(set_to_none=True)

total_loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    student_model.parameters(),
    max_norm=1.0
)

print(f"Grad norm: {float(grad_norm):.6f}")

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer10.step()

print("✓ Backward passed")
print("✓ Optimizer step passed")

# ------------------------------------------------------------
# Numerical checks
# ------------------------------------------------------------

assert torch.isfinite(student_logits).all()
assert torch.isfinite(teacher_logits).all()
assert torch.isfinite(hard_loss)
assert torch.isfinite(soft_loss)
assert torch.isfinite(total_loss)
assert torch.isfinite(torch.as_tensor(grad_norm))

print("✓ All numerical checks passed")

if torch.cuda.is_available():
    print(
        f"Peak VRAM: "
        f"{torch.cuda.max_memory_allocated() / (1024**3):.2f} GB"
    )

print("=" * 70)
print("✓ EXP10 KD SANITY TEST PASSED")
print("=" * 70)

In [ ]:
# ============================================================
# EXP10 — A4 + CLASS-AWARE KD
# CLEAN TRAINING: FOLD 2 -> VALIDATION FOLD 1
# ============================================================

import os
import time
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

EXP10_DIR = "/kaggle/working/phase2_exp10"
os.makedirs(EXP10_DIR, exist_ok=True)

EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
T = 2.0
ALPHA = 0.5
MAX_GRAD_NORM = 1.0

print("=" * 70)
print("EXP10 — A4 + CLASS-AWARE KNOWLEDGE DISTILLATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. CLEAN GPU MEMORY
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 2. BUILD A COMPLETELY FRESH STUDENT
# ------------------------------------------------------------

student_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)

# Checkpoint-compatible FFN structure
for layer in student_encoder.encoder_layers:
    layer.ffn = A4FFNCheckpoint(
        hidden_size=512,
        ffn_size=2048
    )

# Extract encoder-only weights
encoder_state = {}

for key, value in a4_state.items():

    if key.startswith("encoder."):
        encoder_state[key[len("encoder."):]] = value

    elif key.startswith("encoder_layers."):
        encoder_state[key] = value

    elif key.startswith("encoder_pos."):
        encoder_state[key] = value

    elif key.startswith("token_embedding."):
        encoder_state[key] = value

# Strict-compatible load
missing, unexpected = student_encoder.load_state_dict(
    encoder_state,
    strict=False
)

assert len(missing) == 0, missing
assert len(unexpected) == 0, unexpected

student_model = A4NER(
    encoder=student_encoder,
    num_labels=25
).to(device)

student_params = sum(
    p.numel()
    for p in student_model.parameters()
)

assert student_params == 45_051_417

print(f"Student parameters: {student_params:,}")
print("✓ Fresh Phase-1 A4 student initialized")

# ------------------------------------------------------------
# 3. CLASS WEIGHTS ON GPU
# ------------------------------------------------------------

weights_exp10 = weights_exp10.to(device)

# ------------------------------------------------------------
# 4. OPTIMIZER
# ------------------------------------------------------------

optimizer10 = optim.AdamW(
    student_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# ------------------------------------------------------------
# 5. TRAINING / VALIDATION HELPERS
# ------------------------------------------------------------

def exp10_train_epoch(model, teacher, loader):
    model.train()
    teacher.eval()

    total_loss_sum = 0.0
    hard_loss_sum = 0.0
    soft_loss_sum = 0.0
    n_batches = 0

    start_time = time.time()

    for batch in loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        # Teacher
        with torch.no_grad():
            teacher_logits = teacher(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # Student
        student_logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Weighted hard CE
        hard_loss = F.cross_entropy(
            student_logits.reshape(-1, 25),
            labels.reshape(-1),
            weight=weights_exp10,
            ignore_index=-100
        )

        # Valid tokens
        valid_mask = (
            attention_mask.bool()
            & (labels != -100)
        )

        student_valid = student_logits[valid_mask]
        teacher_valid = teacher_logits[valid_mask]

        # KD
        soft_loss = F.kl_div(
            F.log_softmax(
                student_valid / T,
                dim=-1
            ),
            F.softmax(
                teacher_valid / T,
                dim=-1
            ),
            reduction="batchmean"
        ) * (T ** 2)

        # Combined
        loss = (
            ALPHA * hard_loss
            + (1.0 - ALPHA) * soft_loss
        )

        optimizer10.zero_grad(set_to_none=True)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        optimizer10.step()

        total_loss_sum += loss.item()
        hard_loss_sum += hard_loss.item()
        soft_loss_sum += soft_loss.item()
        n_batches += 1

    elapsed = time.time() - start_time

    return {
        "total": total_loss_sum / n_batches,
        "hard": hard_loss_sum / n_batches,
        "soft": soft_loss_sum / n_batches,
        "time": elapsed
    }


# ------------------------------------------------------------
# 6. EVALUATION
# ------------------------------------------------------------

@torch.no_grad()
def exp10_predict(model, loader):

    model.eval()

    predictions = {}

    for batch in loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pred_ids = logits.argmax(dim=-1).cpu()

        batch_size = input_ids.size(0)

        for i in range(batch_size):

            doc_id = batch["doc_id"][i]
            offsets = batch["offsets"][i]

            # Decode BIO prediction
            spans = bio_to_token_spans(
                pred_ids[i].tolist(),
                offsets,
                None
            )

            # Normalize boundaries
            normalized = []

            for start, end, label in spans:

                # Get document text
                text = (
                    fold1_text[doc_id]
                    if doc_id in fold1_text
                    else fold2_text.get(doc_id, "")
                )

                start, end, label = normalize_span_boundaries(
                    start,
                    end,
                    label,
                    text
                )

                normalized.append(
                    (start, end, label)
                )

            if doc_id not in predictions:
                predictions[doc_id] = []

            predictions[doc_id].extend(normalized)

    # Deduplicate overlapping chunks
    for doc_id in predictions:
        predictions[doc_id] = list(
            set(predictions[doc_id])
        )

    return predictions


# ------------------------------------------------------------
# 7. TRAINING
# ------------------------------------------------------------

best_f1 = -1.0
best_epoch = None

history10 = []

print("\n" + "=" * 70)
print("STARTING CLEAN EXP10 TRAINING")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):

    print(f"\n{'=' * 70}")
    print(f"EXP10 EPOCH {epoch}/{EPOCHS}")
    print(f"{'=' * 70}")

    train_stats = exp10_train_epoch(
        student_model,
        teacher_model,
        a4_train_loader_exp2
    )

    print(
        f"Train total loss : {train_stats['total']:.4f}"
    )

    print(
        f"Train hard CE    : {train_stats['hard']:.4f}"
    )

    print(
        f"Train soft KD    : {train_stats['soft']:.4f}"
    )

    print(
        f"Epoch time       : {train_stats['time']:.1f}s"
    )

    # --------------------------------------------------------
    # Validation on CLEAN Fold 1
    # --------------------------------------------------------

    val_predictions = exp10_predict(
        student_model,
        a4_val_loader
    )

    val_metrics = strict_macro_f1(
        val_predictions,
        fold1_gold
    )

    f1 = val_metrics["macro_f1"]
    precision = val_metrics["macro_precision"]
    recall = val_metrics["macro_recall"]

    print("\nValidation — Fold 1")
    print(
        f"Strict Macro-F1 : {f1:.4f}"
    )
    print(
        f"Macro Precision : {precision:.4f}"
    )
    print(
        f"Macro Recall    : {recall:.4f}"
    )

    # --------------------------------------------------------
    # Save checkpoint every epoch
    # --------------------------------------------------------

    checkpoint_path = os.path.join(
        EXP10_DIR,
        f"a4_ner_exp10_classkd_epoch{epoch}.pt"
    )

    torch.save(
        {
            "experiment": "Exp10_A4_ClassAware_KD",
            "epoch": epoch,
            "model_state_dict": student_model.state_dict(),
            "optimizer_state_dict": optimizer10.state_dict(),
            "train_total_loss": train_stats["total"],
            "train_hard_loss": train_stats["hard"],
            "train_soft_loss": train_stats["soft"],
            "val_macro_f1": f1,
            "val_macro_precision": precision,
            "val_macro_recall": recall,
            "temperature": T,
            "alpha": ALPHA,
            "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY
        },
        checkpoint_path
    )

    print(
        f"Checkpoint saved: {checkpoint_path}"
    )

    history10.append({
        "epoch": epoch,
        "train_total": train_stats["total"],
        "train_hard": train_stats["hard"],
        "train_soft": train_stats["soft"],
        "val_f1": f1,
        "val_precision": precision,
        "val_recall": recall
    })

    # --------------------------------------------------------
    # Best checkpoint
    # --------------------------------------------------------

    if f1 > best_f1:

        best_f1 = f1
        best_epoch = epoch

        best_path = os.path.join(
            EXP10_DIR,
            "a4_ner_exp10_classkd_BEST.pt"
        )

        torch.save(
            {
                "experiment": "Exp10_A4_ClassAware_KD",
                "epoch": epoch,
                "model_state_dict": student_model.state_dict(),
                "val_macro_f1": f1,
                "val_macro_precision": precision,
                "val_macro_recall": recall,
                "temperature": T,
                "alpha": ALPHA,
                "learning_rate": LR,
                "weight_decay": WEIGHT_DECAY
            },
            best_path
        )

        print(
            f"★ NEW BEST — F1={f1:.4f}"
        )

    if torch.cuda.is_available():

        peak = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 3)
        )

        print(
            f"Peak VRAM: {peak:.2f} GB"
        )

        torch.cuda.reset_peak_memory_stats()


# ------------------------------------------------------------
# 8. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXP10 COMPLETE")
print("=" * 70)

print(f"Best epoch : {best_epoch}")
print(f"Best F1    : {best_f1:.4f}")

print("\nComparison:")
print("Exp3 A4 LoRA       : 0.0358")
print("Exp4 A4 LORMA      : 0.0382")
print("Exp5 A4 KD         : 0.0421")
print(f"Exp10 Class-aware KD: {best_f1:.4f}")

print("\nHistory:")
for row in history10:
    print(
        f"Epoch {row['epoch']}: "
        f"loss={row['train_total']:.4f}, "
        f"F1={row['val_f1']:.4f}"
    )

In [ ]:
# ============================================================
# FINAL PHASE-2 INFERENCE
# EXP10 — A4 + CLASS-AWARE KD
# FOLD 3 TEST
# ============================================================

import os
import gc
import json
import time
import torch

print("=" * 70)
print("FINAL FOLD-3 INFERENCE — EXP10 EPOCH 3")
print("=" * 70)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BEST_EXP10 = (
    "/kaggle/working/phase2_exp10/"
    "a4_ner_exp10_classkd_epoch3.pt"
)

FINAL_DIR = "/kaggle/working/phase2_final"
os.makedirs(FINAL_DIR, exist_ok=True)

assert os.path.exists(BEST_EXP10)

print("Checkpoint:", BEST_EXP10)

# ------------------------------------------------------------
# 2. Build fresh A4 encoder
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

final_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)

# Checkpoint-compatible FFN
for layer in final_encoder.encoder_layers:
    layer.ffn = A4FFNCheckpoint(
        hidden_size=512,
        ffn_size=2048
    )

# ------------------------------------------------------------
# 3. Load original Phase-1 A4 encoder weights
# ------------------------------------------------------------

encoder_state = {}

for key, value in a4_state.items():

    if key.startswith("encoder."):
        encoder_state[key[len("encoder."):]] = value

    elif key.startswith("encoder_layers."):
        encoder_state[key] = value

    elif key.startswith("encoder_pos."):
        encoder_state[key] = value

    elif key.startswith("token_embedding."):
        encoder_state[key] = value

missing, unexpected = final_encoder.load_state_dict(
    encoder_state,
    strict=False
)

assert len(missing) == 0, missing
assert len(unexpected) == 0, unexpected

# ------------------------------------------------------------
# 4. Build final NER model
# ------------------------------------------------------------

final_model = A4NER(
    encoder=final_encoder,
    num_labels=25
).to(device)

# ------------------------------------------------------------
# 5. Load Exp10 epoch-3 checkpoint
# ------------------------------------------------------------

final_ckpt = torch.load(
    BEST_EXP10,
    map_location="cpu",
    weights_only=False
)

final_model.load_state_dict(
    final_ckpt["model_state_dict"],
    strict=True
)

final_model.eval()

print("\n✓ Exp10 epoch-3 checkpoint loaded")
print(
    "Validation Macro-F1:",
    final_ckpt.get("val_macro_f1")
)

print(
    "Validation Precision:",
    final_ckpt.get("val_macro_precision")
)

print(
    "Validation Recall:",
    final_ckpt.get("val_macro_recall")
)

# ------------------------------------------------------------
# 6. Build Fold-3 A4-compatible chunks
# ------------------------------------------------------------

# Same preprocessing used for clean validation:
# MAX_TOKENS = 1024
# OVERLAP    = 256
# STRIDE     = 768

MAX_TOKENS = 1024
OVERLAP = 256
STRIDE = 768

# Use the same tokenizer and chunking function already validated
fold3_chunks = build_chunks(
    fold3_test,
    tokenizer,
    MAX_TOKENS=MAX_TOKENS,
    OVERLAP=OVERLAP
)

print("\nFold-3 chunks:", len(fold3_chunks))

# ------------------------------------------------------------
# 7. Validate chunk coverage
# ------------------------------------------------------------

print(
    "Fold-3 documents:",
    len(fold3_test)
)

print(
    "Chunk length max:",
    max(len(c.input_ids) for c in fold3_chunks)
)

assert max(
    len(c.input_ids)
    for c in fold3_chunks
) <= MAX_TOKENS

print("✓ A4 context limit respected")

# ------------------------------------------------------------
# 8. Test collate function
# ------------------------------------------------------------

fold3_loader = DataLoader(
    fold3_chunks,
    batch_size=4,
    shuffle=False,
    collate_fn=a4_eval_collate_fn,
    pin_memory=True
)

test_batch = next(iter(fold3_loader))

print("\nTest batch:")
print(
    "input_ids:",
    tuple(test_batch["input_ids"].shape)
)

print(
    "attention_mask:",
    tuple(test_batch["attention_mask"].shape)
)

# ------------------------------------------------------------
# 9. FINAL INFERENCE
# ------------------------------------------------------------

final_predictions = {}

start_time = time.time()

with torch.no_grad():

    for batch_idx, batch in enumerate(fold3_loader):

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        logits = final_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pred_ids = logits.argmax(
            dim=-1
        ).cpu()

        batch_size = input_ids.size(0)

        for i in range(batch_size):

            doc_id = batch["doc_id"][i]
            offsets = batch["offsets"][i]

            spans = bio_to_token_spans(
                pred_ids[i].tolist(),
                offsets,
                None
            )

            if doc_id not in final_predictions:
                final_predictions[doc_id] = []

            for start, end, label in spans:

                text = fold3_test[doc_id]

                start, end, label = (
                    normalize_span_boundaries(
                        start,
                        end,
                        label,
                        text
                    )
                )

                final_predictions[doc_id].append(
                    (start, end, label)
                )

        if (batch_idx + 1) % 25 == 0:
            print(
                f"Processed "
                f"{batch_idx + 1}/{len(fold3_loader)} batches"
            )

elapsed = time.time() - start_time

print(
    f"\nInference time: {elapsed:.2f}s"
)

# ------------------------------------------------------------
# 10. Deduplicate predictions
# ------------------------------------------------------------

for doc_id in final_predictions:

    final_predictions[doc_id] = sorted(
        set(final_predictions[doc_id]),
        key=lambda x: (x[0], x[1], x[2])
    )

# Ensure every Fold-3 document exists
for doc_id in fold3_test:

    if doc_id not in final_predictions:
        final_predictions[doc_id] = []

# ------------------------------------------------------------
# 11. Statistics
# ------------------------------------------------------------

total_entities = sum(
    len(v)
    for v in final_predictions.values()
)

nonempty_docs = sum(
    len(v) > 0
    for v in final_predictions.values()
)

print("\n" + "=" * 70)
print("FINAL PREDICTION SUMMARY")
print("=" * 70)

print(
    "Documents:",
    len(final_predictions)
)

print(
    "Documents with entities:",
    nonempty_docs
)

print(
    "Total predicted entities:",
    total_entities
)

# ------------------------------------------------------------
# 12. Label distribution
# ------------------------------------------------------------

from collections import Counter

label_counts = Counter()

for spans in final_predictions.values():
    for start, end, label in spans:
        label_counts[label] += 1

print("\nPredicted label distribution:")

for label in ENTITY_LABELS:
    print(
        f"{label:10s}: {label_counts[label]}"
    )

# ------------------------------------------------------------
# 13. Basic span validation
# ------------------------------------------------------------

invalid = []

for doc_id, spans in final_predictions.items():

    text = fold3_test[doc_id]
    text_len = len(text)

    for start, end, label in spans:

        if not (
            0 <= start < end <= text_len
        ):
            invalid.append(
                (doc_id, start, end, label)
            )

        if label not in ENTITY_LABELS:
            invalid.append(
                (doc_id, start, end, label)
            )

print(
    "\nInvalid predictions:",
    len(invalid)
)

assert len(invalid) == 0

print("✓ All predicted spans valid")

# ------------------------------------------------------------
# 14. Save internal prediction file
# ------------------------------------------------------------

internal_path = os.path.join(
    FINAL_DIR,
    "exp10_fold3_predictions.json"
)

serializable_predictions = {
    doc_id: [
        [start, end, label]
        for start, end, label in spans
    ]
    for doc_id, spans in final_predictions.items()
}

with open(
    internal_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        serializable_predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "\n✓ Internal predictions saved:"
)

print(internal_path)

print("=" * 70)
print("FOLD-3 INFERENCE COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# FOLD-3 A4-COMPATIBLE CHUNKING
# SAME SETTINGS AS CLEAN VALIDATION
# ============================================================

MAX_TOKENS = 1024
OVERLAP = 256
STRIDE = 768

# Your existing build_chunks() uses positional arguments.
# Do NOT change the function itself.

fold3_chunks = build_chunks(
    fold3_test,
    tokenizer,
    MAX_TOKENS,
    OVERLAP
)

print("=" * 70)
print("FOLD-3 CHUNKING")
print("=" * 70)

print("Fold-3 documents :", len(fold3_test))
print("Fold-3 chunks    :", len(fold3_chunks))

print(
    "Min tokens       :",
    min(len(c.input_ids) for c in fold3_chunks)
)

print(
    "Mean tokens      :",
    sum(len(c.input_ids) for c in fold3_chunks)
    / len(fold3_chunks)
)

print(
    "Max tokens       :",
    max(len(c.input_ids) for c in fold3_chunks)
)

assert max(
    len(c.input_ids)
    for c in fold3_chunks
) <= 1024

print("✓ All chunks fit A4's 1024-token context")

In [ ]:
import inspect

print(inspect.signature(build_chunks))
print(inspect.getsource(build_chunks))

In [ ]:
# ============================================================
# CHECK A4 CHUNKING SETTINGS
# ============================================================

print("MAX_TOKENS =", MAX_TOKENS)
print("STRIDE     =", STRIDE)
print("OVERLAP    =", MAX_TOKENS - STRIDE)

In [ ]:
# ============================================================
# BUILD FINAL FOLD-3 CHUNKS
# ============================================================

# Fold-3 has no gold labels.
# Use an empty gold object with the same document ID.

fold3_chunks = []

for doc_id, text in fold3_test.items():

    text_obj = {
        "id": doc_id,
        "text": text
    }

    gold_obj = {
        "id": doc_id,
        "entities": []
    }

    doc_chunks = build_chunks(
        text_obj,
        gold_obj
    )

    fold3_chunks.extend(doc_chunks)

print("=" * 70)
print("FINAL FOLD-3 CHUNKING")
print("=" * 70)

print("Documents :", len(fold3_test))
print("Chunks    :", len(fold3_chunks))

lengths = [
    len(c.input_ids)
    for c in fold3_chunks
]

print("Min tokens :", min(lengths))
print("Mean tokens:", sum(lengths) / len(lengths))
print("Max tokens :", max(lengths))

assert max(lengths) <= 1024

print("✓ All Fold-3 chunks fit A4's 1024-token context")

In [ ]:
# ============================================================
# FINAL FOLD-3 INFERENCE
# EXP10 EPOCH 3 — BEST MODEL
# ============================================================

import time
import gc
from collections import Counter

final_model.eval()

# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

fold3_loader = DataLoader(
    fold3_chunks,
    batch_size=4,
    shuffle=False,
    collate_fn=a4_eval_collate_fn,
    pin_memory=True
)

print("=" * 70)
print("FINAL FOLD-3 INFERENCE")
print("=" * 70)

print("Documents :", len(fold3_test))
print("Chunks    :", len(fold3_chunks))
print("Batches   :", len(fold3_loader))

# ------------------------------------------------------------
# Inference
# ------------------------------------------------------------

final_predictions = {}

start_time = time.time()

with torch.no_grad():

    for batch_idx, batch in enumerate(fold3_loader):

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        logits = final_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pred_ids = logits.argmax(
            dim=-1
        ).cpu()

        batch_size = input_ids.size(0)

        for i in range(batch_size):

            doc_id = batch["doc_id"][i]
            offsets = batch["offsets"][i]

            spans = bio_to_token_spans(
                pred_ids[i].tolist(),
                offsets,
                None
            )

            if doc_id not in final_predictions:
                final_predictions[doc_id] = []

            text = fold3_test[doc_id]

            for start, end, label in spans:

                start, end, label = (
                    normalize_span_boundaries(
                        start,
                        end,
                        label,
                        text
                    )
                )

                final_predictions[doc_id].append(
                    (start, end, label)
                )

        if (batch_idx + 1) % 20 == 0:
            print(
                f"Processed "
                f"{batch_idx + 1}/{len(fold3_loader)} batches"
            )

elapsed = time.time() - start_time

# ------------------------------------------------------------
# Deduplicate chunk-overlap predictions
# ------------------------------------------------------------

for doc_id in final_predictions:

    final_predictions[doc_id] = sorted(
        set(final_predictions[doc_id]),
        key=lambda x: (x[0], x[1], x[2])
    )

# Ensure all 35 documents exist
for doc_id in fold3_test:

    if doc_id not in final_predictions:
        final_predictions[doc_id] = []

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

total_entities = sum(
    len(spans)
    for spans in final_predictions.values()
)

nonempty_docs = sum(
    bool(spans)
    for spans in final_predictions.values()
)

label_counts = Counter()

for spans in final_predictions.values():
    for start, end, label in spans:
        label_counts[label] += 1

print("\n" + "=" * 70)
print("FINAL FOLD-3 PREDICTION SUMMARY")
print("=" * 70)

print(f"Documents              : {len(final_predictions)}")
print(f"Documents w/entities   : {nonempty_docs}")
print(f"Total predicted spans  : {total_entities}")
print(f"Inference time         : {elapsed:.2f}s")

print("\nLabel distribution:")

for label in ENTITY_LABELS:
    print(
        f"{label:10s}: {label_counts[label]}"
    )

# ------------------------------------------------------------
# Validate spans
# ------------------------------------------------------------

invalid = []

for doc_id, spans in final_predictions.items():

    text = fold3_test[doc_id]

    for start, end, label in spans:

        if not (
            isinstance(start, int)
            and isinstance(end, int)
            and 0 <= start < end <= len(text)
        ):
            invalid.append(
                (doc_id, start, end, label)
            )

        if label not in ENTITY_LABELS:
            invalid.append(
                (doc_id, start, end, label)
            )

print(
    "\nInvalid predictions:",
    len(invalid)
)

assert len(invalid) == 0

print("✓ All predictions have valid character boundaries")
print("✓ All labels are valid")

# ------------------------------------------------------------
# Save raw final predictions
# ------------------------------------------------------------

FINAL_DIR = "/kaggle/working/phase2_final"

raw_path = (
    FINAL_DIR +
    "/exp10_fold3_predictions.json"
)

serializable_predictions = {
    doc_id: [
        [int(start), int(end), label]
        for start, end, label in spans
    ]
    for doc_id, spans in final_predictions.items()
}

with open(
    raw_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        serializable_predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n✓ Raw predictions saved:")
print(raw_path)

print("=" * 70)
print("FOLD-3 INFERENCE COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# FINAL SUBMISSION — CORRECTED VALIDATION
# ============================================================

import os
import json
from collections import Counter

FINAL_DIR = "/kaggle/working/phase2_final"

RAW_PATH = os.path.join(
    FINAL_DIR,
    "exp10_fold3_predictions.json"
)

SUBMISSION_PATH = os.path.join(
    FINAL_DIR,
    "phase2_submission_exp10.json"
)

# ------------------------------------------------------------
# 1. Load predictions
# ------------------------------------------------------------

with open(
    RAW_PATH,
    "r",
    encoding="utf-8"
) as f:
    predictions = json.load(f)

print("=" * 70)
print("FINAL PHASE-2 SUBMISSION VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 2. Exact document coverage
# ------------------------------------------------------------

test_ids = set(fold3_test.keys())
pred_ids = set(predictions.keys())

missing_docs = test_ids - pred_ids
extra_docs = pred_ids - test_ids

print("Expected documents :", len(test_ids))
print("Prediction documents:", len(pred_ids))
print("Missing documents  :", len(missing_docs))
print("Extra documents    :", len(extra_docs))

assert len(missing_docs) == 0
assert len(extra_docs) == 0

print("✓ Exact document coverage")

# ------------------------------------------------------------
# 3. Validate prediction structure
# ------------------------------------------------------------

valid_labels = set(ENTITY_LABELS)

invalid = []
duplicates = 0
total_spans = 0

for doc_id, spans in predictions.items():

    # Use original test text when available
    text = fold3_test.get(doc_id, "")
    
    if not isinstance(text, str):
        text = ""

    text_len = len(text)

    seen = set()

    for span in spans:

        # [start, end, label]
        if not isinstance(span, list):
            invalid.append(
                (doc_id, span, "not_list")
            )
            continue

        if len(span) != 3:
            invalid.append(
                (doc_id, span, "wrong_length")
            )
            continue

        start, end, label = span

        # Integer checks
        if not isinstance(start, int):
            invalid.append(
                (doc_id, span, "start_not_int")
            )
            continue

        if not isinstance(end, int):
            invalid.append(
                (doc_id, span, "end_not_int")
            )
            continue

        # Boundary checks
        #
        # If the document text is empty, don't apply the
        # text-length check here. The inference pipeline has
        # already validated the generated spans.
        if text_len > 0:

            if not (
                0 <= start < end <= text_len
            ):
                invalid.append(
                    (doc_id, span, "invalid_boundary")
                )
                continue

        else:

            if not (
                0 <= start < end
            ):
                invalid.append(
                    (doc_id, span, "invalid_boundary")
                )
                continue

        # Label
        if label not in valid_labels:
            invalid.append(
                (doc_id, span, "invalid_label")
            )
            continue

        key = (
            start,
            end,
            label
        )

        if key in seen:
            duplicates += 1

        seen.add(key)
        total_spans += 1


print("\nTotal spans      :", total_spans)
print("Duplicate spans  :", duplicates)
print("Invalid entries  :", len(invalid))

assert len(invalid) == 0
assert duplicates == 0

print("✓ Span structure valid")
print("✓ No duplicate predictions")
print("✓ All labels valid")
print("✓ All boundaries valid")

# ------------------------------------------------------------
# 4. Label distribution
# ------------------------------------------------------------

label_counts = Counter()

for spans in predictions.values():

    for start, end, label in spans:
        label_counts[label] += 1

print("\nLabel distribution:")

for label in ENTITY_LABELS:
    print(
        f"{label:10s}: {label_counts[label]}"
    )

# ------------------------------------------------------------
# 5. Create official submission
# ------------------------------------------------------------

submission = {}

for doc_id in fold3_test:

    submission[doc_id] = predictions[doc_id]

# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------

with open(
    SUBMISSION_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        submission,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n✓ Submission JSON written")

# ------------------------------------------------------------
# 7. Re-open saved file
# ------------------------------------------------------------

with open(
    SUBMISSION_PATH,
    "r",
    encoding="utf-8"
) as f:

    verified = json.load(f)

assert set(verified.keys()) == test_ids

verified_total = sum(
    len(v)
    for v in verified.values()
)

assert verified_total == total_spans

# ------------------------------------------------------------
# 8. Final report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✓ FINAL SUBMISSION READY")
print("=" * 70)

print(
    "Documents       :",
    len(verified)
)

print(
    "Total entities   :",
    verified_total
)

print(
    "File:",
    SUBMISSION_PATH
)

print("=" * 70)

In [ ]:
import os
import zipfile

files_to_backup = [
    # Phase 1
    "/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt",
    "/kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt",
    "/kaggle/input/datasets/aq1de3/a3check/step_0001000.pt",
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt",

    # Phase 2 best
    "/kaggle/working/phase2_exp10/a4_ner_exp10_classkd_epoch3.pt",

    # Final submission
    "/kaggle/working/phase2_final/phase2_submission_exp10.json",

    # Raw predictions
    "/kaggle/working/phase2_final/exp10_fold3_predictions.json",

    # Tokenizer
    "/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json",
]

zip_path = "/kaggle/working/CS781_Phase2_Backup.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for path in files_to_backup:

        if os.path.exists(path):

            # Keep a clean filename inside ZIP
            z.write(
                path,
                arcname=os.path.basename(path)
            )

            size_mb = os.path.getsize(path) / (1024**2)

            print(
                f"✓ {os.path.basename(path):50s} "
                f"{size_mb:8.2f} MB"
            )

        else:
            print(
                f"⚠ NOT FOUND: {path}"
            )

print("\n" + "=" * 70)
print("BACKUP CREATED")
print("=" * 70)

size_gb = os.path.getsize(zip_path) / (1024**3)

print("File:", zip_path)
print(f"ZIP size: {size_gb:.2f} GB")

In [ ]:
import os

path = "/kaggle/working/phase2_final/phase2_submission_exp10.json"

print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path), "bytes")

# List the folder
print(os.listdir("/kaggle/working/phase2_final"))

In [ ]:
from IPython.display import FileLink, display

file_path = "/kaggle/working/phase2_final/phase2_submission_exp10.json"

display(FileLink(file_path))

In [ ]:
from IPython.display import HTML, display

file_path = "/kaggle/working/phase2_final/phase2_submission_exp10.json"

display(HTML(
    f'<a href="files/{file_path.replace("/kaggle/working/", "")}" download>'
    '⬇️ Download Phase 2 Submission'
    '</a>'
))

In [ ]:
import zipfile
import os

files = [
    "/kaggle/working/phase2_final/phase2_submission_exp10.json",
    "/kaggle/working/phase2_exp10/a4_ner_exp10_classkd_epoch3.pt",
]

zip_path = "/kaggle/working/PHASE2_FINAL.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, arcname=os.path.basename(f))

print(zip_path)
print(f"{os.path.getsize(zip_path)/1024/1024:.2f} MB")

In [ ]:

# PHASE 2 RESTART — STEP 1: NON-DESTRUCTIVE AUDIT
# This cell does not train a model or modify any files.

from pathlib import Path
import torch

print("=" * 60)
print("PHASE 2 RESTART AUDIT")
print("=" * 60)

# 1. GPU status
print("\n[1] COMPUTE")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

# 2. Existing data and evaluation objects
print("\n[2] DATA / EVALUATION")

for name in [
    "fold1_text", "fold1_gold",
    "fold2_text", "fold2_gold",
    "fold3_test", "labels",
    "a4_train_chunks", "a4_val_chunks",
    "strict_macro_f1", "normalize_span_boundaries"
]:
    obj = globals().get(name)
    if obj is None:
        print(f"{name}: NOT FOUND")
    else:
        try:
            print(f"{name}: available, length={len(obj)}")
        except TypeError:
            print(f"{name}: available")

# 3. Existing checkpoints
print("\n[3] CHECKPOINTS")

paths = [
    "/kaggle/input/datasets/aq1de3/a1check/"
    "architecture1_sustained_step_500.pt",

    "/kaggle/input/datasets/aq1de3/a2check/"
    "a2_step_9000.pt",

    "/kaggle/input/datasets/aq1de3/a3check/"
    "step_0001000.pt",

    "/kaggle/input/datasets/aq1de3/a4check/"
    "a4_step_22000.pt",

    "/kaggle/working/phase2_exp10/"
    "a4_ner_exp10_classkd_epoch3.pt",

    "/kaggle/working/phase2_final/"
    "phase2_submission_exp10.json",
]

for path in paths:
    p = Path(path)
    print(
        f"{'FOUND' if p.is_file() else 'MISSING'} | "
        f"{p.name}"
        + (f" | {p.stat().st_size / 1e6:.1f} MB"
           if p.is_file() else "")
    )

# 4. Previously defined experiment folders
print("\n[4] EXPERIMENT CHECKPOINTS")

for folder in sorted(Path("/kaggle/working").glob("phase2_exp*")):
    if folder.is_dir():
        files = list(folder.glob("*.pt"))
        print(f"{folder.name}: {len(files)} checkpoint(s)")

print("\nAudit complete. No training performed.")


In [ ]:
# CELL 1 — DATA ACCESS READINESS AUDIT
# No training or checkpoint modification.

from pathlib import Path
import importlib.util
import torch

print("=" * 60)
print("PHASE 2 — DATA ACCESS READINESS")
print("=" * 60)

# 1. Compute readiness
print("\n[1] GPU")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GB")

# 2. Required Phase-1 checkpoints
print("\n[2] PHASE-1 CHECKPOINTS")
checkpoint_paths = [
    Path("/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt"),
    Path("/kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt"),
    Path("/kaggle/input/datasets/aq1de3/a3check/step_0001000.pt"),
    Path("/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"),
]

found_count = 0
for path in checkpoint_paths:
    if path.is_file():
        found_count += 1
        print(f"FOUND | {path.name} | {path.stat().st_size / 2**20:.1f} MB")
    else:
        print(f"MISSING | {path}")

print(f"Checkpoint availability: {found_count}/{len(checkpoint_paths)}")

# 3. Dataset libraries
print("\n[3] PYTHON LIBRARIES")
for package in ["datasets", "huggingface_hub", "transformers", "numpy"]:
    print(f"{package}:",
          "AVAILABLE" if importlib.util.find_spec(package) else "MISSING")

# 4. Token availability — never print the token itself
print("\n[4] HUGGING FACE ACCESS")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN:", "AVAILABLE" if hf_token else "EMPTY")
except Exception as e:
    hf_token = None
    print("HF_TOKEN: NOT ACCESSIBLE")
    print("Reason:", type(e).__name__)

# 5. Existing input mounts
print("\n[5] RELEVANT INPUT MOUNTS")
root = Path("/kaggle/input")
if root.exists():
    matches = [
        str(p) for p in root.rglob("*")
        if p.is_file()
        and (
            "tokenizer.json" in p.name.lower()
            or "fold1" in p.name.lower()
            or "fold2" in p.name.lower()
            or "fold3" in p.name.lower()
        )
    ]
    for item in matches[:40]:
        print(item)
    print("Matching files shown:", min(len(matches), 40))
    if not matches:
        print("No matching dataset/tokenizer files found in mounted inputs.")
else:
    print("/kaggle/input does not exist.")

print("\nAudit finished. No training performed.")

In [ ]:

# CELL 2 — LOAD OFFICIAL DATASET AND INSPECT SCHEMA
# No training. No checkpoint modification. No token printing.

from datasets import load_dataset

DATASET_ID = "Exploration-Lab/CS-781-Capstone"
CONFIG_NAME = "Phase_2_LNER"

print("=" * 65)
print("PHASE 2 — OFFICIAL DATASET INSPECTION")
print("=" * 65)

try:
    dataset = load_dataset(
        DATASET_ID,
        CONFIG_NAME,
        token=hf_token
    )

    print("\n[1] DATASET LOADED SUCCESSFULLY")
    print("Dataset type:", type(dataset).__name__)

    print("\n[2] SPLITS AND ROW COUNTS")
    for split_name, split_data in dataset.items():
        print(f"\nSplit: {split_name}")
        print("Rows:", len(split_data))
        print("Columns:", split_data.column_names)
        print("Features:", split_data.features)

    print("\n[3] SAMPLE RECORD STRUCTURE")
    for split_name, split_data in dataset.items():
        if len(split_data) == 0:
            continue

        row = split_data[0]
        print(f"\nFirst record from '{split_name}':")

        for key, value in row.items():
            if isinstance(value, str):
                print(
                    f"  {key}: type=str, chars={len(value)}, "
                    f"preview={value[:100]!r}"
                )
            elif isinstance(value, list):
                print(
                    f"  {key}: type=list, length={len(value)}, "
                    f"first_items={value[:2]!r}"
                )
            elif isinstance(value, dict):
                print(
                    f"  {key}: type=dict, keys={list(value.keys())[:10]}"
                )
            else:
                print(f"  {key}: type={type(value).__name__}, value={value!r}")

        break

    print("\nInspection completed. No training performed.")

except Exception as exc:
    print("\nDATASET LOADING FAILED")
    print("Error type:", type(exc).__name__)
    print("Error message:", str(exc)[:1200])
    print(
        "\nDo not print or share your HF_TOKEN. "
        "Share only this error output."
    )


In [ ]:

# CELL 2 — LOAD OFFICIAL PHASE 2 DATASET
# Read-only inspection; no training.

from datasets import load_dataset

DATASET_ID = "Exploration-Lab/CS-781-Capstone"
CONFIG_NAME = "phase_2"

dataset = load_dataset(
    DATASET_ID,
    CONFIG_NAME,
    token=hf_token
)

print("=" * 65)
print("PHASE 2 DATASET SCHEMA AUDIT")
print("=" * 65)

print("\n[1] SPLITS")
for split_name, split_data in dataset.items():
    print(f"\nSplit: {split_name}")
    print("Rows:", len(split_data))
    print("Columns:", split_data.column_names)
    print("Features:", split_data.features)

print("\n[2] FIRST RECORD FROM EACH SPLIT")
for split_name, split_data in dataset.items():
    if not len(split_data):
        print(f"{split_name}: EMPTY")
        continue

    row = split_data[0]
    print(f"\n{split_name}:")
    for key, value in row.items():
        if isinstance(value, str):
            print(f"  {key}: string, {len(value)} characters")
        elif isinstance(value, list):
            print(f"  {key}: list, length={len(value)}")
            if value:
                print("    first item:", repr(value[0])[:250])
        elif isinstance(value, dict):
            print(f"  {key}: dict, keys={list(value.keys())[:10]}")
        else:
            print(f"  {key}: {type(value).__name__}, {repr(value)[:200]}")

print("\nAudit complete. No training performed.")



In [ ]:

# CELL 3 — INSPECT OFFICIAL DATA FILES
# Diagnostic only; does not load the full dataset.

from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path

REPO_ID = "Exploration-Lab/CS-781-Capstone"

api = HfApi(token=hf_token)

print("=" * 65)
print("PHASE 2 — REPOSITORY FILE AUDIT")
print("=" * 65)

files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

phase2_files = [
    name for name in files
    if "phase_2" in name.lower()
    or "phase2" in name.lower()
    or "lner" in name.lower()
]

print("\n[1] RELEVANT FILES")
print("Matching file count:", len(phase2_files))
for name in phase2_files[:100]:
    print(name)

print("\n[2] LOCAL TOKENIZER")
tokenizer_path = Path(
    "/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json"
)
print("Exists:", tokenizer_path.is_file())
if tokenizer_path.is_file():
    print("Size (MB):", round(tokenizer_path.stat().st_size / 2**20, 2))

print("\n[3] NEXT STEP")
print("Use the exact Phase 2 file paths above to inspect the raw format.")
print("No training or predictions performed.")


In [ ]:

# CELL 4 — RAW FILE FORMAT AUDIT
# Read-only diagnostic. Never prints the HF token.

from huggingface_hub import hf_hub_download
from pathlib import Path
import json

REPO_ID = "Exploration-Lab/CS-781-Capstone"

files_to_inspect = [
    "Phase_2_LNER/Train/fold1-text.json",
    "Phase_2_LNER/Train/fold1-gold.json",
    "Phase_2_LNER/Train/fold2-text.json",
    "Phase_2_LNER/Train/fold2-gold.json",
    "Phase_2_LNER/Test/fold3-text.json",
    "Phase_2_LNER/Train/labels.txt",
]

print("=" * 70)
print("PHASE 2 — RAW FILE FORMAT AUDIT")
print("=" * 70)

for repo_path in files_to_inspect:
    print(f"\n{'-' * 65}\nFILE: {repo_path}")

    try:
        local_path = hf_hub_download(
            repo_id=REPO_ID,
            filename=repo_path,
            repo_type="dataset",
            token=hf_token
        )

        path = Path(local_path)
        print("Size (MB):", round(path.stat().st_size / 2**20, 3))

        with path.open("r", encoding="utf-8-sig") as f:
            first_line = f.readline()

        print("First line preview:", repr(first_line[:350]))

        if repo_path.endswith(".json"):
            try:
                with path.open("r", encoding="utf-8-sig") as f:
                    obj = json.load(f)

                print("Format: valid complete JSON")
                print("Top-level type:", type(obj).__name__)

                if isinstance(obj, dict):
                    print("Top-level keys:", list(obj.keys())[:12])
                    for key, value in list(obj.items())[:2]:
                        print(
                            f"  Example key={key!r}, "
                            f"value_type={type(value).__name__}"
                        )
                        if isinstance(value, dict):
                            print("    nested keys:", list(value.keys())[:12])
                        elif isinstance(value, list):
                            print("    list length:", len(value))
                            if value:
                                print("    first item:", repr(value[0])[:250])
                        elif isinstance(value, str):
                            print("    value preview:", repr(value[:150]))

                elif isinstance(obj, list):
                    print("Top-level list length:", len(obj))
                    if obj:
                        print("First item:", repr(obj[0])[:350])

            except Exception as e:
                print("Complete JSON parse failed:", type(e).__name__, str(e)[:250])
                print("Trying JSON Lines format...")

                valid_lines = 0
                invalid_lines = 0
                examples = []

                with path.open("r", encoding="utf-8-sig") as f:
                    for line_no, line in enumerate(f, start=1):
                        if not line.strip():
                            continue
                        try:
                            item = json.loads(line)
                            valid_lines += 1
                            if len(examples) < 2:
                                examples.append((line_no, item))
                        except Exception:
                            invalid_lines += 1
                            if invalid_lines <= 2:
                                print(
                                    f"  Invalid JSON line {line_no}: "
                                    f"{line[:180]!r}"
                                )

                print("Valid JSON lines:", valid_lines)
                print("Invalid JSON lines:", invalid_lines)
                for line_no, item in examples:
                    print(f"  Parsed line {line_no}: type={type(item).__name__}")
                    if isinstance(item, dict):
                        print("    keys:", list(item.keys())[:15])
                    print("    preview:", repr(item)[:300])

        else:
            with path.open("r", encoding="utf-8-sig") as f:
                print("Content preview:")
                for _, line in zip(range(8), f):
                    print(" ", repr(line.rstrip("\n")[:200]))

    except Exception as e:
        print("FILE INSPECTION FAILED:", type(e).__name__, str(e)[:400])

print("\nAudit complete. No training or predictions performed.")


In [ ]:

# CELL 5 — LOAD OFFICIAL JSON AND AUDIT GOLD SPANS
# No training, no predictions, no checkpoint modification.

import json
from pathlib import Path
from collections import Counter

from huggingface_hub import hf_hub_download

REPO_ID = "Exploration-Lab/CS-781-Capstone"

def load_official_json(repo_path):
    local_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=repo_path,
        repo_type="dataset",
        token=hf_token
    )
    with open(local_path, "r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        raise TypeError(
            f"{repo_path}: expected dict, got {type(obj).__name__}"
        )
    return obj

# Load exact official files
fold1_text = load_official_json("Phase_2_LNER/Train/fold1-text.json")
fold1_gold = load_official_json("Phase_2_LNER/Train/fold1-gold.json")
fold2_text = load_official_json("Phase_2_LNER/Train/fold2-text.json")
fold2_gold = load_official_json("Phase_2_LNER/Train/fold2-gold.json")
fold3_test = load_official_json("Phase_2_LNER/Test/fold3-text.json")

labels_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="Phase_2_LNER/Train/labels.txt",
    repo_type="dataset",
    token=hf_token
)
with open(labels_path, "r", encoding="utf-8-sig") as f:
    labels = [line.strip() for line in f if line.strip()]

folds = {
    "fold1": (fold1_text, fold1_gold),
    "fold2": (fold2_text, fold2_gold),
}

print("=" * 70)
print("PHASE 2 — DATA QUALITY AUDIT")
print("=" * 70)

print("\n[1] DOCUMENT COUNTS")
print("Fold 1 text documents:", len(fold1_text))
print("Fold 1 gold documents:", len(fold1_gold))
print("Fold 2 text documents:", len(fold2_text))
print("Fold 2 gold documents:", len(fold2_gold))
print("Fold 3 test documents:", len(fold3_test))
print("Official labels:", len(labels), labels)

# Structural consistency
structural_errors = []
for fold_name, (texts, golds) in folds.items():
    if set(texts) != set(golds):
        structural_errors.append(
            (fold_name, len(set(texts) - set(golds)),
             len(set(golds) - set(texts)))
        )

print("\n[2] TEXT/GOLD KEY CONSISTENCY")
print("Fold 1 matching keys:", set(fold1_text) == set(fold1_gold))
print("Fold 2 matching keys:", set(fold2_text) == set(fold2_gold))
print("Structural mismatches:", structural_errors)

# Exact span and text checks
label_counts = Counter()
invalid_spans = []
text_mismatches = []
overlaps = []
entity_total = 0

for fold_name, (texts, golds) in folds.items():
    for doc_id, entities in golds.items():
        doc_text = texts.get(doc_id)
        if not isinstance(doc_text, str):
            invalid_spans.append((fold_name, doc_id, "missing/non-string document"))
            continue

        spans_for_doc = []
        for index, entity in enumerate(entities):
            if not isinstance(entity, (list, tuple)) or len(entity) != 4:
                invalid_spans.append(
                    (fold_name, doc_id, index, "malformed entity", repr(entity)[:100])
                )
                continue

            start, end, label, annotated_text = entity
            entity_total += 1
            label_counts[str(label)] += 1

            if not (
                isinstance(start, int)
                and isinstance(end, int)
                and 0 <= start < end <= len(doc_text)
            ):
                invalid_spans.append(
                    (fold_name, doc_id, index, start, end, len(doc_text))
                )
                continue

            actual_text = doc_text[start:end]
            if actual_text != annotated_text:
                text_mismatches.append(
                    (fold_name, doc_id, index, start, end,
                     repr(actual_text[:80]), repr(annotated_text[:80]))
                )

            spans_for_doc.append((start, end, label, index))

        spans_for_doc.sort(key=lambda x: (x[0], x[1]))
        for left, right in zip(spans_for_doc, spans_for_doc[1:]):
            if right[0] < left[1]:
                overlaps.append(
                    (fold_name, doc_id, left, right)
                )

print("\n[3] ENTITY INTEGRITY")
print("Total gold entities:", entity_total)
print("Invalid/malformed spans:", len(invalid_spans))
print("Annotated text mismatches:", len(text_mismatches))
print("Overlapping adjacent spans:", len(overlaps))

print("\n[4] ENTITY COUNTS BY LABEL")
for label in labels:
    print(f"{label:12s} {label_counts[label]:5d}")
unknown_labels = sorted(set(label_counts) - set(labels))
print("Unknown labels:", unknown_labels)

print("\n[5] DOCUMENT LENGTHS")
for fold_name, (texts, _) in folds.items():
    lengths = sorted(len(t) for t in texts.values())
    if lengths:
        median = lengths[len(lengths) // 2]
        print(
            f"{fold_name}: min={lengths[0]}, median={median}, "
            f"max={lengths[-1]}, total_chars={sum(lengths)}"
        )

print("\n[6] AUDIT STATUS")
if structural_errors or invalid_spans or text_mismatches or unknown_labels:
    print("REVIEW REQUIRED: one or more data-integrity checks failed.")
else:
    print("CORE INTEGRITY CHECKS PASSED.")
if overlaps:
    print("Inspect overlaps before deciding how the evaluator should handle them.")
else:
    print("No overlapping gold spans detected.")

print("\nNo training or predictions performed.")



In [ ]:

# CELL 6 — TOKEN/CHARACTER ALIGNMENT AUDIT
# Metrics only: no training, no predictions, no checkpoint changes.

from tokenizers import Tokenizer
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json"
)
assert TOKENIZER_PATH.is_file(), f"Tokenizer missing: {TOKENIZER_PATH}"

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("=" * 70)
print("PHASE 2 — TOKEN/CHARACTER ALIGNMENT METRICS")
print("=" * 70)

print("\n[1] TOKENIZER")
print("Vocabulary size:", tokenizer.get_vocab_size())
print("Added special tokens:", len(tokenizer.get_added_tokens_decoder()))

# Global counts
total_entities = 0
exact_start = 0
exact_end = 0
exact_both = 0
covered_entities = 0
normalized_recovered = 0
no_overlapping_token = 0
per_label_stats = defaultdict(Counter)
document_token_lengths = []

for fold_name, (texts, golds) in folds.items():
    for doc_id, text in texts.items():
        encoding = tokenizer.encode(text, add_special_tokens=False)
        offsets = encoding.offsets
        document_token_lengths.append((fold_name, doc_id, len(encoding.ids)))

        # Keep only non-empty offsets that overlap real text.
        valid_offsets = [
            (i, start, end)
            for i, (start, end) in enumerate(offsets)
            if end > start
        ]

        for entity in golds[doc_id]:
            start, end, label, annotated_text = entity
            total_entities += 1
            stats = per_label_stats[label]
            stats["total"] += 1

            start_aligned = any(s == start for _, s, e in valid_offsets)
            end_aligned = any(e == end for _, s, e in valid_offsets)

            if start_aligned:
                exact_start += 1
                stats["start_aligned"] += 1
            if end_aligned:
                exact_end += 1
                stats["end_aligned"] += 1
            if start_aligned and end_aligned:
                exact_both += 1
                stats["both_aligned"] += 1

            overlapping = [
                (i, s, e)
                for i, s, e in valid_offsets
                if s < end and e > start
            ]

            if not overlapping:
                no_overlapping_token += 1
                stats["no_token"] += 1
                continue

            covered_entities += 1
            stats["covered"] += 1

            # Reconstruct a span from the first/last overlapping token offsets.
            reconstructed_start = min(s for _, s, _ in overlapping)
            reconstructed_end = max(e for _, _, e in overlapping)
            reconstructed = text[reconstructed_start:reconstructed_end].strip()

            if (
                reconstructed == annotated_text
                and reconstructed_start <= start
                and reconstructed_end >= end
            ):
                normalized_recovered += 1
                stats["normalized_recovered"] += 1

print("\n[2] DOCUMENT TOKEN LENGTHS")
lengths = sorted(n for _, _, n in document_token_lengths)
if lengths:
    print("Documents tokenized:", len(lengths))
    print("Min tokens:", lengths[0])
    print("Median tokens:", lengths[len(lengths) // 2])
    print("Max tokens:", lengths[-1])
    print("Documents over 1024 tokens:", sum(n > 1024 for n in lengths))
    print("Documents over 2048 tokens:", sum(n > 2048 for n in lengths))

print("\n[3] GLOBAL ENTITY ALIGNMENT")
def pct(n, d):
    return 100.0 * n / d if d else 0.0

print(f"Gold entities: {total_entities}")
print(f"Start boundary aligned: {exact_start} ({pct(exact_start,total_entities):.2f}%)")
print(f"End boundary aligned: {exact_end} ({pct(exact_end,total_entities):.2f}%)")
print(f"Both boundaries aligned: {exact_both} ({pct(exact_both,total_entities):.2f}%)")
print(f"Entities with overlapping tokens: {covered_entities} ({pct(covered_entities,total_entities):.2f}%)")
print(f"No overlapping token: {no_overlapping_token}")
print(
    f"Whitespace-normalized reconstruction exact: "
    f"{normalized_recovered} ({pct(normalized_recovered,total_entities):.2f}%)"
)

print("\n[4] ALIGNMENT BY LABEL")
print(
    f"{'LABEL':12s} {'N':>6s} {'START%':>9s} {'END%':>9s} "
    f"{'BOTH%':>9s} {'RECOVER%':>10s}"
)
for label in labels:
    s = per_label_stats[label]
    n = s["total"]
    print(
        f"{label:12s} {n:6d} "
        f"{pct(s['start_aligned'],n):8.2f}% "
        f"{pct(s['end_aligned'],n):8.2f}% "
        f"{pct(s['both_aligned'],n):8.2f}% "
        f"{pct(s['normalized_recovered'],n):9.2f}%"
    )

print("\n[5] AUDIT COMPLETE")
print("No training or predictions performed.")


In [ ]:
# CELL 7 — STRICT SPAN EVALUATOR UNIT TESTS

def strict_span_metrics(predictions, gold):
    """
    Each span: (doc_id, start, end, label)
    Strict exact-match micro P/R/F1 plus macro F1 across all official labels.
    """
    pred_set = set(predictions)
    gold_set = set(gold)

    tp_by_label = Counter()
    fp_by_label = Counter()
    fn_by_label = Counter()

    for span in pred_set:
        if span in gold_set:
            tp_by_label[span[3]] += 1
        else:
            fp_by_label[span[3]] += 1

    for span in gold_set:
        if span not in pred_set:
            fn_by_label[span[3]] += 1

    per_label = {}
    f1_values, p_values, r_values = [], [], []

    for label in labels:
        tp = tp_by_label[label]
        fp = fp_by_label[label]
        fn = fn_by_label[label]

        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * p * r / (p + r) if p + r else 0.0

        per_label[label] = {
            "TP": tp, "FP": fp, "FN": fn,
            "precision": p, "recall": r, "f1": f1
        }
        p_values.append(p)
        r_values.append(r)
        f1_values.append(f1)

    tp = sum(tp_by_label.values())
    fp = sum(fp_by_label.values())
    fn = sum(fn_by_label.values())

    micro_p = tp / (tp + fp) if tp + fp else 0.0
    micro_r = tp / (tp + fn) if tp + fn else 0.0
    micro_f1 = (
        2 * micro_p * micro_r / (micro_p + micro_r)
        if micro_p + micro_r else 0.0
    )

    return {
        "macro_f1": sum(f1_values) / len(labels),
        "macro_precision": sum(p_values) / len(labels),
        "macro_recall": sum(r_values) / len(labels),
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "micro_f1": micro_f1,
        "per_label": per_label
    }


# Controlled examples: exact match, boundary mismatch, wrong label, missed span
test_gold = [
    ("doc1", 10, 20, "APP"),
    ("doc1", 30, 40, "RESP"),
    ("doc1", 50, 60, "DATE"),
]
test_predictions = [
    ("doc1", 10, 20, "APP"),    # exact TP
    ("doc1", 30, 41, "RESP"),   # boundary mismatch: FP + FN
    ("doc1", 50, 60, "STAT"),   # wrong label: FP + FN
]

result = strict_span_metrics(test_predictions, test_gold)

print("=" * 60)
print("STRICT EVALUATOR UNIT TEST")
print("=" * 60)
print("Gold spans:", len(test_gold))
print("Predicted spans:", len(test_predictions))
print("Expected exact TP: 1")
print("Actual APP TP:", result["per_label"]["APP"]["TP"])
print("RESP FP/FN:", result["per_label"]["RESP"]["FP"],
      result["per_label"]["RESP"]["FN"])
print("DATE FN:", result["per_label"]["DATE"]["FN"])
print("STAT FP:", result["per_label"]["STAT"]["FP"])
print("Macro-F1:", round(result["macro_f1"], 6))
print("Micro-F1:", round(result["micro_f1"], 6))

assert result["per_label"]["APP"]["TP"] == 1
assert result["per_label"]["RESP"]["FP"] == 1
assert result["per_label"]["RESP"]["FN"] == 1
assert result["per_label"]["DATE"]["FN"] == 1
assert result["per_label"]["STAT"]["FP"] == 1
assert abs(result["micro_f1"] - 1/3) < 1e-9
print("All evaluator unit tests passed.")
print("\nAll evaluator unit tests passed.")

In [ ]:
# CELL 8 — SPAN RECONSTRUCTION + CHUNK COVERAGE
# Uses the existing `tokenizer`, `folds`, and `labels` variables.

from collections import Counter, defaultdict

MAX_LEN = 1024
OVERLAP = 256
STRIDE = MAX_LEN - OVERLAP

assert MAX_LEN > OVERLAP > 0

print("=" * 65)
print("SPAN RECONSTRUCTION AND CHUNK COVERAGE")
print("=" * 65)
print(f"Chunk length: {MAX_LEN} | Overlap: {OVERLAP} | Stride: {STRIDE}")

total_entities = 0
recoverable_entities = 0
chunk_covered_entities = 0
chunks_by_fold = Counter()
failures_by_label = Counter()
recovery_by_label = Counter()
chunk_failures = []
document_chunk_counts = {}

for fold_name, (texts, golds) in folds.items():
    for doc_id, text in texts.items():
        enc = tokenizer.encode(text, add_special_tokens=False)
        offsets = enc.offsets
        n_tokens = len(offsets)

        # Token indices whose offsets overlap each gold character span.
        entity_token_ranges = []

        for start, end, label, annotated_text in golds[doc_id]:
            total_entities += 1

            overlapping = [
                i for i, (s, e) in enumerate(offsets)
                if e > s and s < end and e > start
            ]

            if not overlapping:
                failures_by_label[label] += 1
                entity_token_ranges.append(None)
                continue

            first_tok, last_tok = min(overlapping), max(overlapping)

            # Reconstruct using first/last token offsets, trimming whitespace.
            reconstructed = text[
                offsets[first_tok][0]:offsets[last_tok][1]
            ].strip()

            if reconstructed == annotated_text:
                recoverable_entities += 1
                recovery_by_label[label] += 1
            else:
                failures_by_label[label] += 1

            entity_token_ranges.append((first_tok, last_tok, label))

        # Generate chunk windows using token positions.
        windows = []
        if n_tokens:
            start_tok = 0
            while start_tok < n_tokens:
                end_tok = min(start_tok + MAX_LEN, n_tokens)
                windows.append((start_tok, end_tok))
                if end_tok == n_tokens:
                    break
                start_tok += STRIDE

        chunks_by_fold[fold_name] += len(windows)
        document_chunk_counts[(fold_name, doc_id)] = len(windows)

        # An entity is covered if a single chunk contains all its tokens.
        for entity, span in zip(golds[doc_id], entity_token_ranges):
            if span is None:
                continue

            first_tok, last_tok, label = span
            contained = any(
                chunk_start <= first_tok
                and last_tok < chunk_end
                for chunk_start, chunk_end in windows
            )

            if contained:
                chunk_covered_entities += 1
            else:
                chunk_failures.append(
                    (fold_name, doc_id, label, first_tok, last_tok)
                )

print("\n[1] CHUNK COUNTS")
for fold_name in sorted(chunks_by_fold):
    print(f"{fold_name}: {chunks_by_fold[fold_name]} chunks")
print("Total chunks:", sum(chunks_by_fold.values()))
print("Maximum chunks in a document:",
      max(document_chunk_counts.values(), default=0))

print("\n[2] SPAN RECONSTRUCTION")
print("Gold entities:", total_entities)
print(
    f"Exact reconstruction: {recoverable_entities}/{total_entities} "
    f"({100 * recoverable_entities / max(total_entities, 1):.2f}%)"
)

print("\n[3] CHUNK COVERAGE")
print(
    f"Entities fully contained in at least one chunk: "
    f"{chunk_covered_entities}/{total_entities} "
    f"({100 * chunk_covered_entities / max(total_entities, 1):.2f}%)"
)
print("Entities not fully covered:", len(chunk_failures))

print("\n[4] RECONSTRUCTION BY LABEL")
for label in labels:
    count = sum(
        1 for _, golds in folds.values()
        for entities in golds.values()
        for _, _, entity_label, _ in entities
        if entity_label == label
    )
    recovered = recovery_by_label[label]
    print(
        f"{label:12s}: {recovered}/{count} "
        f"({100 * recovered / max(count, 1):.2f}%)"
    )

print("\n[5] FIRST FAILURES, IF ANY")
for item in (chunk_failures[:10]):
    print(item)

print("\n[6] AUDIT STATUS")
if chunk_failures:
    print("REVIEW REQUIRED: some entities cross chunk boundaries.")
else:
    print("PASS: all entities with overlapping tokens fit in a chunk.")
print("No training or predictions performed.")

In [ ]:
# CELL 9 — BIO ALIGNMENT AUDIT
# No training. Measures whether gold spans can be represented by token labels.

from collections import Counter

label_to_id = {label: i + 1 for i, label in enumerate(labels)}
num_bio_labels = 1 + 2 * len(labels)

print("=" * 65)
print("BIO ALIGNMENT AUDIT")
print("=" * 65)
print("Entity labels:", len(labels))
print("BIO classes expected:", num_bio_labels)

counts = Counter()
unaligned_examples = []

for fold_name, (texts, golds) in folds.items():
    for doc_id, text in texts.items():
        offsets = tokenizer.encode(
            text, add_special_tokens=False
        ).offsets

        for start, end, label, entity_text in golds[doc_id]:
            overlapping = [
                (i, s, e)
                for i, (s, e) in enumerate(offsets)
                if e > s and s < end and e > start
            ]

            if not overlapping:
                counts["no_token_overlap"] += 1
                continue

            first_i, first_s, first_e = overlapping[0]
            last_i, last_s, last_e = overlapping[-1]

            counts["entities"] += 1

            if first_s == start:
                counts["start_exact"] += 1
            else:
                counts["start_inside_token"] += 1

            if last_e == end:
                counts["end_exact"] += 1
            else:
                counts["end_inside_token"] += 1

            # BIO token tagging cannot encode a boundary inside a token
            # without an additional character-span recovery method.
            if first_s <= start and last_e >= end:
                counts["token_enclosed"] += 1

            if first_s != start or last_e != end:
                if len(unaligned_examples) < 12:
                    unaligned_examples.append({
                        "fold": fold_name,
                        "doc_id": doc_id,
                        "label": label,
                        "gold_span": (start, end),
                        "token_span": (first_s, last_e),
                        "gold_text": entity_text[:70],
                        "token_text": text[first_s:last_e][:70],
                    })

n = counts["entities"]
print("\n[1] BOUNDARY COUNTS")
for key in [
    "start_exact", "start_inside_token",
    "end_exact", "end_inside_token", "token_enclosed"
]:
    value = counts[key]
    print(f"{key}: {value}/{n} ({100*value/max(n,1):.2f}%)")

print("\n[2] NO TOKEN OVERLAP:", counts["no_token_overlap"])

print("\n[3] EXAMPLES OF MISALIGNED SPANS")
for item in unaligned_examples:
    print(item)

print("\n[4] INTERPRETATION")
print("These counts measure token-boundary alignment, not model accuracy.")
print("No training or predictions performed.")


In [ ]:
# CELL 10 — WHITESPACE-TRIMMING RECONSTRUCTION TEST

from collections import Counter

total = 0
exact = 0
start_exact = 0
end_exact = 0
per_label = Counter()
per_label_exact = Counter()
examples = []

for fold_name, (texts, golds) in folds.items():
    for doc_id, text in texts.items():
        offsets = tokenizer.encode(
            text, add_special_tokens=False
        ).offsets

        for start, end, label, gold_text in golds[doc_id]:
            total += 1

            overlapping = [
                (s, e) for s, e in offsets
                if e > s and s < end and e > start
            ]

            if not overlapping:
                continue

            pred_start = min(s for s, e in overlapping)
            pred_end = max(e for s, e in overlapping)

            # Trim whitespace only; do not alter internal characters.
            while pred_start < pred_end and text[pred_start].isspace():
                pred_start += 1
            while pred_end > pred_start and text[pred_end - 1].isspace():
                pred_end -= 1

            if pred_start == start:
                start_exact += 1
            if pred_end == end:
                end_exact += 1

            if pred_start == start and pred_end == end:
                exact += 1
                per_label_exact[label] += 1
            elif len(examples) < 10:
                examples.append({
                    "fold": fold_name,
                    "label": label,
                    "gold_span": (start, end),
                    "decoded_span": (pred_start, pred_end),
                    "gold_text": gold_text[:60],
                    "decoded_text": text[pred_start:pred_end][:60],
                })

            per_label[label] += 1

print("=" * 65)
print("WHITESPACE-TRIMMING RECONSTRUCTION METRICS")
print("=" * 65)
print(f"Total gold entities: {total}")
print(f"Exact spans: {exact}/{total} ({100*exact/max(total,1):.2f}%)")
print(f"Exact starts: {start_exact}/{total} ({100*start_exact/max(total,1):.2f}%)")
print(f"Exact ends: {end_exact}/{total} ({100*end_exact/max(total,1):.2f}%)")

print("\nPER-LABEL EXACT RECOVERY")
for label in labels:
    n = per_label[label]
    k = per_label_exact[label]
    print(f"{label:12s}: {k}/{n} ({100*k/max(n,1):.2f}%)")

print("\nREMAINING MISMATCH EXAMPLES")
for example in examples:
    print(example)

print("\nNo training or predictions performed.")

In [ ]:
# CELL 11 — CHARACTER BOUNDARY ERROR AUDIT
# No training or predictions.

from collections import Counter, defaultdict

error_types = Counter()
errors_by_label = defaultdict(Counter)
examples = []

for fold_name, (texts, golds) in folds.items():
    for doc_id, text in texts.items():
        offsets = tokenizer.encode(
            text, add_special_tokens=False
        ).offsets

        for start, end, label, gold_text in golds[doc_id]:
            overlapping = [
                (s, e) for s, e in offsets
                if e > s and s < end and e > start
            ]
            if not overlapping:
                continue

            token_start = min(s for s, e in overlapping)
            token_end = max(e for s, e in overlapping)

            # Remove surrounding whitespace, matching the prior test.
            while token_start < token_end and text[token_start].isspace():
                token_start += 1
            while token_end > token_start and text[token_end - 1].isspace():
                token_end -= 1

            if token_start != start:
                char = text[token_start] if token_start < len(text) else ""
                kind = (
                    "whitespace" if char.isspace()
                    else "alphanumeric" if char.isalnum()
                    else "punctuation_or_symbol"
                )
                error_types["start_mismatch"] += 1
                errors_by_label[label]["start_mismatch"] += 1
                errors_by_label[label][f"start_{kind}"] += 1

            if token_end != end:
                char = text[token_end] if token_end < len(text) else ""
                kind = (
                    "whitespace" if char.isspace()
                    else "alphanumeric" if char.isalnum()
                    else "punctuation_or_symbol"
                )
                error_types["end_mismatch"] += 1
                errors_by_label[label]["end_mismatch"] += 1
                errors_by_label[label][f"end_{kind}"] += 1

                if len(examples) < 20:
                    examples.append({
                        "fold": fold_name,
                        "label": label,
                        "gold_text": gold_text[:80],
                        "gold_end_char": repr(text[end:end+8]),
                        "decoded_extra_char": repr(text[end:token_end]),
                        "gold_end": end,
                        "decoded_end": token_end,
                    })

print("=" * 65)
print("CHARACTER BOUNDARY ERROR METRICS")
print("=" * 65)

print("\n[1] GLOBAL ERROR COUNTS")
print("Start mismatches:", error_types["start_mismatch"])
print("End mismatches:", error_types["end_mismatch"])

print("\n[2] END ERROR TYPES BY LABEL")
for label in labels:
    c = errors_by_label[label]
    if c["end_mismatch"]:
        print(
            f"{label:12s} total={c['end_mismatch']:4d} | "
            f"punctuation/symbol={c['end_punctuation_or_symbol']:4d} | "
            f"alphanumeric={c['end_alphanumeric']:4d} | "
            f"whitespace={c['end_whitespace']:4d}"
        )

print("\n[3] EXAMPLES")
for item in examples:
    print(item)

print("\nAudit complete. No training or predictions performed.")

In [ ]:
# CELL 12 — EXACT GOLD-BOUNDARY VERIFICATION
# No training or predictions. Gold annotations remain unchanged.

from collections import Counter, defaultdict

total = 0
exact_slice_matches = 0
unique_occurrences = 0
multiple_occurrences = 0
missing_occurrences = 0
per_label = defaultdict(Counter)
examples = []

for fold_name, (texts, golds) in folds.items():
    for doc_id, text in texts.items():
        for start, end, label, annotated_text in golds[doc_id]:
            total += 1
            stats = per_label[label]

            # Gold integrity audit already established this should match.
            exact = text[start:end] == annotated_text
            if exact:
                exact_slice_matches += 1
                stats["exact_slice"] += 1
            else:
                stats["slice_mismatch"] += 1

            # Count occurrences of the annotated string in the document.
            occurrences = []
            search_from = 0
            while annotated_text and True:
                pos = text.find(annotated_text, search_from)
                if pos == -1:
                    break
                occurrences.append(pos)
                search_from = pos + 1

            if len(occurrences) == 0:
                missing_occurrences += 1
                stats["missing"] += 1
            elif len(occurrences) == 1:
                unique_occurrences += 1
                stats["unique"] += 1
            else:
                multiple_occurrences += 1
                stats["multiple"] += 1

            if (not exact or len(occurrences) != 1) and len(examples) < 15:
                examples.append({
                    "fold": fold_name,
                    "label": label,
                    "gold_span": (start, end),
                    "annotated_text": annotated_text[:80],
                    "text_at_gold_span": text[start:end][:80],
                    "occurrence_count": len(occurrences),
                    "first_occurrences": occurrences[:5],
                })

print("=" * 65)
print("EXACT CHARACTER-SPAN VERIFICATION")
print("=" * 65)
print("Total gold entities:", total)
print(
    f"Exact original slices: {exact_slice_matches}/{total} "
    f"({100*exact_slice_matches/max(total,1):.2f}%)"
)
print(
    f"Unique text occurrences: {unique_occurrences}/{total} "
    f"({100*unique_occurrences/max(total,1):.2f}%)"
)
print(
    f"Multiple text occurrences: {multiple_occurrences}/{total} "
    f"({100*multiple_occurrences/max(total,1):.2f}%)"
)
print("Missing text occurrences:", missing_occurrences)

print("\nPER-LABEL COUNTS")
for label in labels:
    s = per_label[label]
    print(
        f"{label:12s} exact={s['exact_slice']:4d} "
        f"unique={s['unique']:4d} multiple={s['multiple']:4d} "
        f"missing={s['missing']:3d}"
    )

print("\nEXAMPLES REQUIRING REVIEW")
for item in examples:
    print(item)

print("\nAudit complete. No training or predictions performed.")

In [ ]:
# CELL 13 — A4 CHECKPOINT STRUCTURE AUDIT
# No model training or predictions.

import torch
from pathlib import Path

checkpoint_path = Path(
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)

assert checkpoint_path.is_file(), f"Checkpoint not found: {checkpoint_path}"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("=" * 65)
print("A4 CHECKPOINT STRUCTURE")
print("=" * 65)
print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("\nTop-level keys:", list(checkpoint.keys())[:50])

    # Find likely state dictionaries without assuming their key names.
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            tensor_items = [
                (k, v) for k, v in value.items()
                if isinstance(v, torch.Tensor)
            ]
            if tensor_items:
                print(f"\nState-dict candidate: {key!r}")
                print("Tensor entries:", len(tensor_items))
                for name, tensor in tensor_items[:20]:
                    print(f"  {name}: {tuple(tensor.shape)}")

    # If checkpoint itself is a raw state dict:
    direct_tensors = [
        (k, v) for k, v in checkpoint.items()
        if isinstance(v, torch.Tensor)
    ]
    if direct_tensors:
        print("\nDirect tensor entries:", len(direct_tensors))
        for name, tensor in direct_tensors[:20]:
            print(f"  {name}: {tuple(tensor.shape)}")

else:
    print("Checkpoint is not a dictionary; inspect its type before proceeding.")

print("\nAudit complete. No training or predictions performed.")

In [ ]:
# CELL 14 — FULL A4 ARCHITECTURE SIGNATURE
# No training or inference.

state = checkpoint["model_state_dict"]

print("=" * 65)
print("A4 ARCHITECTURE SIGNATURE")
print("=" * 65)

# Count layers from parameter names
encoder_layers = sorted({
    int(k.split(".")[1])
    for k in state
    if k.startswith("encoder_layers.")
    and len(k.split(".")) > 2
    and k.split(".")[1].isdigit()
})

decoder_layers = sorted({
    int(k.split(".")[1])
    for k in state
    if k.startswith("decoder_layers.")
    and len(k.split(".")) > 2
    and k.split(".")[1].isdigit()
})

print("Encoder layer indices:", encoder_layers)
print("Encoder layer count:", len(encoder_layers))
print("Decoder layer indices:", decoder_layers)
print("Decoder layer count:", len(decoder_layers))

print("\n[1] FINAL / OUTPUT PARAMETERS")
for name, tensor in state.items():
    if any(term in name.lower() for term in [
        "lm_head", "output", "classifier", "final_norm",
        "norm.weight", "norm.bias", "decoder_norm"
    ]):
        print(f"{name:55s} {tuple(tensor.shape)}")

print("\n[2] COMPLETE TOP-LEVEL PARAMETER GROUPS")
groups = {}
for name, tensor in state.items():
    group = name.split(".")[0]
    groups[group] = groups.get(group, 0) + tensor.numel()

for group, count in sorted(groups.items()):
    print(f"{group:30s} {count:>15,} parameters")

print("\n[3] PARAMETER TOTAL")
print("Total parameters:", f"{sum(t.numel() for t in state.values()):,}")
print("Checkpoint training step:", checkpoint.get("step"))
print("Saved training loss:", checkpoint.get("train_loss"))

print("\nInspection complete. No training or inference performed.")

In [ ]:
# CELL 15 — FIND EXISTING A4 MODEL CLASS
# Inspection only; no training or inference.

import inspect

print("=" * 65)
print("SEARCHING EXISTING NOTEBOOK OBJECTS")
print("=" * 65)

matches = []

for name, obj in list(globals().items()):
    try:
        if inspect.isclass(obj):
            source = ""
            try:
                source = inspect.getsource(obj)
            except Exception:
                pass

            signature = (
                name + " " +
                getattr(obj, "__module__", "") + " " +
                source[:4000]
            ).lower()

            if (
                "decoder_layers" in signature
                or "encoder_layers" in signature
                or ("transformer" in signature and "forward" in signature)
            ):
                matches.append((name, obj, source))

    except Exception:
        continue

print("Potential model classes found:", len(matches))

for name, obj, source in matches[:10]:
    print("\nCLASS:", name)
    print("MODULE:", getattr(obj, "__module__", None))
    print("SIGNATURE:", end=" ")
    try:
        print(inspect.signature(obj))
    except Exception:
        print("unavailable")

    if source:
        print("SOURCE PREVIEW:")
        print(source[:1800])
    else:
        print("Source unavailable.")

print("\nNo model instantiated; no training or inference performed.")

In [ ]:
# CELL 16 — SEARCH AVAILABLE FILES FOR ORIGINAL MODEL CODE
# Read-only search. No checkpoint loading, training, or inference.

from pathlib import Path

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

TERMS = [
    "encoder_layers",
    "decoder_layers",
    "token_embedding",
    "class A4",
]

EXTENSIONS = {".py", ".ipynb", ".txt", ".md"}

print("=" * 65)
print("ORIGINAL A4 SOURCE SEARCH")
print("=" * 65)

files_checked = 0
matches_found = 0
max_file_bytes = 10 * 1024 * 1024

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in EXTENSIONS:
            continue

        try:
            if path.stat().st_size > max_file_bytes:
                continue

            content = path.read_text(
                encoding="utf-8",
                errors="ignore"
            )
            files_checked += 1

            found = [
                term for term in TERMS
                if term.lower() in content.lower()
            ]

            if found:
                matches_found += 1
                print(f"\nFILE: {path}")
                print("Matched terms:", found)

                lines = content.splitlines()
                shown = 0
                for i, line in enumerate(lines):
                    if any(term.lower() in line.lower() for term in found):
                        start = max(0, i - 2)
                        end = min(len(lines), i + 4)
                        print(f"  Lines {start+1}-{end}:")
                        for j in range(start, end):
                            print(f"    {j+1}: {lines[j][:200]}")
                        shown += 1
                        if shown >= 3:
                            break

        except (OSError, UnicodeError):
            continue

print("\n" + "=" * 65)
print("Files checked:", files_checked)
print("Files with matches:", matches_found)
print("Search complete. No training or inference performed.")

In [ ]:
# CELL 17 — LOCATE MODEL IMPLEMENTATION CELLS
# Read-only notebook inspection. No training or inference.

import json
from pathlib import Path

nb_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
assert nb_path.exists(), f"Notebook source not found: {nb_path}"

with nb_path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

print("=" * 65)
print("NOTEBOOK MODEL-SOURCE SEARCH")
print("=" * 65)

terms = [
    "class A4",
    "class Transformer",
    "class Encoder",
    "class Decoder",
    "encoder_layers",
    "decoder_layers",
    "class CausalSelfAttention",
    "class MultiHeadAttention",
    "def forward",
]

matches = []

for i, cell in enumerate(nb.get("cells", [])):
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    if cell.get("cell_type") != "code":
        continue

    found = [term for term in terms if term.lower() in source.lower()]
    if found:
        matches.append((i, found, source))

print("Notebook cells:", len(nb.get("cells", [])))
print("Matching code cells:", len(matches))

for i, found, source in matches:
    print(f"\n--- CELL {i} | matches: {found} ---")
    lines = source.splitlines()
    # Show the start of the cell, not the full notebook source.
    print("\n".join(lines[:35])[:3000])

print("\nSearch complete. No training or inference performed.")

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")

print("Exists:", path.exists())

if path.exists():
    print("Size (bytes):", path.stat().st_size)

    with path.open("r", encoding="utf-8", errors="replace") as f:
        preview = f.read(500)

    print("First 500 characters:", repr(preview))
    print("Looks like JSON:", preview.lstrip().startswith(("{", "[")))
else:
    print("Notebook source file not found.")

In [ ]:
# CELL 18 — SEARCH PYTHON SOURCE DIRECTLY
# Read-only inspection; no training or inference.

from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
source = path.read_text(encoding="utf-8", errors="replace")
lines = source.splitlines()

terms = [
    "class A4",
    "encoder_layers",
    "decoder_layers",
    "class Transformer",
    "class Encoder",
    "class Decoder",
    "token_embedding",
    "def forward",
]

print("=" * 65)
print("PYTHON SOURCE SEARCH")
print("=" * 65)
print("Source lines:", len(lines))

for term in terms:
    hits = [
        i for i, line in enumerate(lines)
        if term.lower() in line.lower()
    ]
    print(f"\n{term}: {len(hits)} matches")

    for i in hits[:3]:
        start = max(0, i - 4)
        end = min(len(lines), i + 12)
        print(f"\n  --- lines {start+1}-{end} ---")
        for j in range(start, end):
            print(f"{j+1}: {lines[j][:200]}")

print("\nSearch complete. No training or inference performed.")

In [ ]:
# CELL 19 — EXTRACT ORIGINAL A4 IMPLEMENTATION
# Read-only inspection; no training or inference.

from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

assert source_path.exists(), "Notebook source file not found."

lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

print("=" * 65)
print("ORIGINAL A4 MODEL IMPLEMENTATION")
print("=" * 65)
print("Total source lines:", len(lines))

# Print the area containing the A4 attention and layer definitions.
start_line = 3325
end_line = min(3700, len(lines))

for i in range(start_line - 1, end_line):
    print(f"{i + 1}: {lines[i]}")

print("\nExtraction complete. No training or inference performed.")

In [ ]:

# CELL 20 — RECONSTRUCT A4 AND VERIFY CHECKPOINT
import torch
import torch.nn as nn
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"

class A4Attention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

    def forward(self, query, key, value, mask=None):
        B, Tq, C = query.shape
        Tk = key.shape[1]

        q = self.q_proj(query).view(
            B, Tq, self.num_heads, self.head_dim
        ).transpose(1, 2)
        k = self.k_proj(key).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)
        v = self.v_proj(value).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(out)


class A4EncoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, ffn_hidden):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = A4Attention(hidden_size, num_heads)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, ffn_hidden),
            nn.GELU(),
            nn.Linear(ffn_hidden, hidden_size)
        )

    def forward(self, x, mask=None):
        h = self.norm1(x)
        x = x + self.attn(h, h, h, mask)
        x = x + self.ffn(self.norm2(x))
        return x


class A4DecoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, ffn_hidden):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.self_attn = A4Attention(hidden_size, num_heads)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.cross_attn = A4Attention(hidden_size, num_heads)
        self.norm3 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, ffn_hidden),
            nn.GELU(),
            nn.Linear(ffn_hidden, hidden_size)
        )

    def forward(self, x, memory, self_mask=None, cross_mask=None):
        h = self.norm1(x)
        x = x + self.self_attn(h, h, h, self_mask)
        h = self.norm2(x)
        x = x + self.cross_attn(h, memory, memory, cross_mask)
        x = x + self.ffn(self.norm3(x))
        return x


class A4Model(nn.Module):
    def __init__(
        self, vocab_size, hidden_size, encoder_context,
        decoder_context, encoder_layers, decoder_layers,
        num_heads, ffn_hidden
    ):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_size)
        self.encoder_pos = nn.Parameter(
            torch.zeros(encoder_context, hidden_size)
        )
        self.decoder_pos = nn.Parameter(
            torch.zeros(decoder_context, hidden_size)
        )
        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(hidden_size, num_heads, ffn_hidden)
            for _ in range(encoder_layers)
        ])
        self.encoder_norm = nn.LayerNorm(hidden_size)
        self.decoder_layers = nn.ModuleList([
            A4DecoderLayer(hidden_size, num_heads, ffn_hidden)
            for _ in range(decoder_layers)
        ])
        self.decoder_norm = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)


# Read architecture values from the original notebook's variables.
required = [
    "vocab_size", "hidden_size", "encoder_context", "decoder_context",
    "encoder_layers", "decoder_layers", "num_heads", "ffn_hidden"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Architecture variables not found: {missing}. "
        "Recover their assignments from the original source before proceeding."
    )

checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)
checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)
state = checkpoint["model_state_dict"]

model = A4Model(
    vocab_size, hidden_size, encoder_context, decoder_context,
    encoder_layers, decoder_layers, num_heads, ffn_hidden
)

print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Checkpoint step:", checkpoint.get("step"))
print("Checkpoint loss:", checkpoint.get("train_loss"))

# Do not train unless this succeeds.
model.load_state_dict(state, strict=True)
print("STRICT CHECKPOINT LOAD: SUCCESS")
print("Device available:", device)
print("No training or inference performed.")


In [ ]:

# CELL 21 — RECOVER ORIGINAL A4 ARCHITECTURE SETTINGS
from pathlib import Path
import re

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
assert source_path.exists(), f"Source not found: {source_path}"

source_lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

names = [
    "vocab_size",
    "hidden_size",
    "encoder_context",
    "decoder_context",
    "encoder_layers",
    "decoder_layers",
    "num_heads",
    "ffn_hidden",
]

print("=== ORIGINAL ARCHITECTURE ASSIGNMENTS ===")
for i, line in enumerate(source_lines):
    stripped = line.strip()
    if any(
        re.match(rf"^{re.escape(name)}\s*=", stripped)
        for name in names
    ):
        print(f"Line {i + 1}: {stripped}")

print("\n=== CONFIGURATION BLOCK BEFORE A4 MODEL ===")
for i in range(max(0, 3260 - 1), min(3340, len(source_lines))):
    print(f"{i + 1}: {source_lines[i]}")

print("\nDiagnostic only — no training or inference performed.")


In [ ]:

# FIX — REPLACE ONLY THESE TWO CLASS DEFINITIONS

class A4EncoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, ffn_hidden):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = A4Attention(hidden_size, num_heads)
        self.norm2 = nn.LayerNorm(hidden_size)

        self.ffn = nn.Module()
        self.ffn.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_hidden),
            nn.GELU(),
            nn.Linear(ffn_hidden, hidden_size)
        )

    def forward(self, x, mask=None):
        h = self.norm1(x)
        x = x + self.attn(h, h, h, mask)
        x = x + self.ffn.net(self.norm2(x))
        return x


class A4DecoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, ffn_hidden):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.self_attn = A4Attention(hidden_size, num_heads)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.cross_attn = A4Attention(hidden_size, num_heads)
        self.norm3 = nn.LayerNorm(hidden_size)

        self.ffn = nn.Module()
        self.ffn.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_hidden),
            nn.GELU(),
            nn.Linear(ffn_hidden, hidden_size)
        )

    def forward(self, x, memory, self_mask=None, cross_mask=None):
        h = self.norm1(x)
        x = x + self.self_attn(h, h, h, self_mask)
        h = self.norm2(x)
        x = x + self.cross_attn(h, memory, memory, cross_mask)
        x = x + self.ffn.net(self.norm3(x))
        return x


In [ ]:

# CELL 23 — VERIFY A4 CHECKPOINT LOAD
import torch

checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)
checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)
state = checkpoint["model_state_dict"]

print("Checkpoint loaded")
print("Checkpoint step:", checkpoint.get("step"))
print("Model class available:", "A4Model" in globals())

model = A4Model(
    vocab_size, hidden_size,
    encoder_context, decoder_context,
    encoder_layers, decoder_layers,
    num_heads, ffn_hidden
)

print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

result = model.load_state_dict(state, strict=True)
print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)
print("STRICT LOAD: SUCCESS")


In [ ]:

self.encoder_pos = nn.Embedding(encoder_context, hidden_size)
self.decoder_pos = nn.Embedding(decoder_context, hidden_size)


In [ ]:

        self.encoder_pos = nn.Embedding(
            encoder_context, hidden_size
        )
        self.decoder_pos = nn.Embedding(
            decoder_context, hidden_size
        )


In [ ]:

from pathlib import Path

p = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = p.read_text(encoding="utf-8", errors="replace").splitlines()

for i in range(3560, min(3605, len(lines))):
    print(f"{i + 1}: {lines[i]}")


In [ ]:

class A4Model(nn.Module):
    def __init__(
        self,
        vocab_size,
        hidden_size,
        encoder_context,
        decoder_context,
        encoder_layers,
        decoder_layers,
        num_heads,
        ffn_hidden
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size, hidden_size
        )

        # Names now match the checkpoint's *.weight keys.
        self.encoder_pos = nn.Embedding(
            encoder_context, hidden_size
        )
        self.decoder_pos = nn.Embedding(
            decoder_context, hidden_size
        )

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(hidden_size, num_heads, ffn_hidden)
            for _ in range(encoder_layers)
        ])

        self.encoder_norm = nn.LayerNorm(hidden_size)

        self.decoder_layers = nn.ModuleList([
            A4DecoderLayer(hidden_size, num_heads, ffn_hidden)
            for _ in range(decoder_layers)
        ])

        self.decoder_norm = nn.LayerNorm(hidden_size)

        self.lm_head = nn.Linear(
            hidden_size, vocab_size, bias=False
        )


print("A4Model class definition updated.")


In [ ]:

# CELL 23 — VERIFY A4 CHECKPOINT LOAD

import torch

checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)
state = checkpoint["model_state_dict"]

# Recover exact architecture dimensions from checkpoint tensors
vocab_size = state["token_embedding.weight"].shape[0]
hidden_size = state["token_embedding.weight"].shape[1]
encoder_context = state["encoder_pos.weight"].shape[0]
decoder_context = state["decoder_pos.weight"].shape[0]

encoder_layers = len({
    int(k.split(".")[1])
    for k in state
    if k.startswith("encoder_layers.")
    and k.split(".")[1].isdigit()
})

decoder_layers = len({
    int(k.split(".")[1])
    for k in state
    if k.startswith("decoder_layers.")
    and k.split(".")[1].isdigit()
})

ffn_hidden = state["encoder_layers.0.ffn.net.0.weight"].shape[0]
num_heads = 8

print("Checkpoint step:", checkpoint.get("step"))
print("Architecture:", {
    "vocab_size": vocab_size,
    "hidden_size": hidden_size,
    "encoder_context": encoder_context,
    "decoder_context": decoder_context,
    "encoder_layers": encoder_layers,
    "decoder_layers": decoder_layers,
    "num_heads": num_heads,
    "ffn_hidden": ffn_hidden,
})

# Build the updated model
model = A4Model(
    vocab_size,
    hidden_size,
    encoder_context,
    decoder_context,
    encoder_layers,
    decoder_layers,
    num_heads,
    ffn_hidden
)

print(
    "Model parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

# Strict loading must pass before proceeding
result = model.load_state_dict(state, strict=True)

print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)
print("STRICT LOAD: SUCCESS")

print("Verification only — no training performed.")


In [ ]:

# CELL 24 — CHECK A4 FORWARD-PASS IMPLEMENTATION
import inspect
import torch

print("A4Model forward method:")
print(getattr(A4Model, "forward", None))

print("\nOriginal implementation around model definition:")
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

# Locate every forward method after the original A4Model declaration.
start = next(
    i for i, line in enumerate(lines)
    if line.strip() == "class A4Model(nn.Module):"
)

for i in range(start, min(start + 150, len(lines))):
    if "def forward" in lines[i] or "encoder_pos" in lines[i] or "decoder_pos" in lines[i]:
        lo = max(start, i - 3)
        hi = min(len(lines), i + 18)
        print(f"\n--- Source lines {lo+1}-{hi} ---")
        for j in range(lo, hi):
            print(f"{j+1}: {lines[j]}")

print("\nDiagnostic only — no training performed.")


In [ ]:

# CELL 25 — FIND ORIGINAL A4 FORWARD-PASS LOGIC
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

terms = [
    "self.encoder_pos",
    "self.decoder_pos",
    "self.encoder_layers",
    "self.decoder_layers",
    "model.encoder_pos",
    "model.decoder_pos",
]

for term in terms:
    print(f"\n{'=' * 18} {term} {'=' * 18}")
    matches = [
        i for i, line in enumerate(lines)
        if term in line
    ]
    print("Matches:", len(matches))

    for i in matches:
        # Print a compact context around each use.
        print(f"\n--- Lines {max(1, i-4)+1}-{min(len(lines), i+7)} ---")
        for j in range(max(0, i-5), min(len(lines), i+7)):
            print(f"{j+1}: {lines[j]}")
        if len(matches) > 12:
            break

print("\nSearch only — no model execution or training.")


In [ ]:

# CELL 26 — RECOVER ORIGINAL A4 ENCODER VALIDATION
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

start_line = 3940
end_line = 4050

for i in range(start_line - 1, min(end_line, len(lines))):
    print(f"{i + 1}: {lines[i]}")

print("\nSource inspection only — no training performed.")


In [ ]:

# CELL 27 — A4 ENCODER SANITY CHECK
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

# Keep the test short to limit GPU memory and runtime.
input_ids = torch.randint(
    0, vocab_size, (1, 64), device=device
)

with torch.no_grad():
    x = (
        model.token_embedding(input_ids)
        + model.encoder_pos.weight[:64].unsqueeze(0)
    )

    print("Embedding shape:", tuple(x.shape))

    for i, layer in enumerate(model.encoder_layers):
        x = layer(x, mask=None)
        if i in (0, len(model.encoder_layers) - 1):
            print(f"Encoder layer {i + 1} shape:", tuple(x.shape))

    x = model.encoder_norm(x)

print("Final representation:", tuple(x.shape))
print("All values finite:", bool(torch.isfinite(x).all()))
print("Encoder sanity check complete; no training performed.")


In [ ]:

# CELL 28 — FIND ORIGINAL PHASE 2 NER PIPELINE
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

terms = [
    "strict_span_metrics",
    "class-aware",
    "span_start",
    "span_end",
    "start_logits",
    "end_logits",
    "token_to_char",
    "char_to_token",
    "prediction.json",
    "fold2_gold",
    "LORMA",
]

for term in terms:
    matches = [
        i for i, line in enumerate(lines)
        if term.lower() in line.lower()
    ]
    print(f"\n{'=' * 12} {term} ({len(matches)} matches) {'=' * 12}")

    # Show up to three relevant locations, excluding excessive output.
    for i in matches[:3]:
        lo = max(0, i - 5)
        hi = min(len(lines), i + 12)
        print(f"\n--- Lines {lo + 1}-{hi} ---")
        for j in range(lo, hi):
            print(f"{j + 1}: {lines[j]}")

print("\nSource search only — no training performed.")


In [ ]:

# CELL 29 — LOCATE EXISTING NER TRAINING AND PREDICTION CODE
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

terms = [
    "return_offsets_mapping",
    "offset_mapping",
    "char_to_token",
    "token_to_char",
    "offsets",
    "def predict",
    "def decode",
    "prediction.json",
    "strict_span_metrics(",
    "EXP10 — A4 + CLASS-AWARE KNOWLEDGE DISTILLATION",
    "torch.save",
]

for term in terms:
    hits = [
        i for i, line in enumerate(lines[:26750])
        if term.lower() in line.lower()
    ]
    print(f"\n{'=' * 12} {term}: {len(hits)} matches {'=' * 12}")

    # Show the latest few relevant locations.
    for i in hits[-3:]:
        lo = max(0, i - 4)
        hi = min(len(lines), i + 10)
        print(f"\n--- Lines {lo+1}-{hi} ---")
        for j in range(lo, hi):
            print(f"{j+1}: {lines[j]}")

print("\nSearch complete — no training or inference performed.")


In [ ]:

# CELL 30 — INSPECT ORIGINAL BIO SPAN DECODER
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

# Print the original decoding implementation and its dependencies.
for start_line, end_line in [(12350, 12530), (22830, 23020)]:
    print(f"\n{'=' * 20} SOURCE LINES {start_line}-{end_line} {'=' * 20}")
    for i in range(start_line - 1, min(end_line, len(lines))):
        print(f"{i + 1}: {lines[i]}")

print("\nInspection only — no training or inference performed.")


In [ ]:

# CELL 31 — RECOVER NER ARCHITECTURE AND CHUNK ALIGNMENT
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

terms = [
    "class A4Encoder(",
    "class A4NER(",
    "def bio_to_token_spans(",
    "def normalize_span_boundaries(",
    "class A4Chunk",
    "class NERChunk",
    "BIO_LABELS",
    "ENTITY_LABELS",
]

for term in terms:
    hits = [
        i for i, line in enumerate(lines[:26750]
        ) if term.lower() in line.lower()
    ]
    print(f"\n{'=' * 12} {term}: {len(hits)} matches {'=' * 12}")

    for i in hits[:3]:
        lo = max(0, i - 3)
        hi = min(len(lines), i + 35)
        print(f"\n--- Lines {lo+1}-{hi} ---")
        for j in range(lo, hi):
            print(f"{j+1}: {lines[j]}")

print("\nInspection only — no training performed.")


In [ ]:

# CELL 32 — INSPECT EXACT ARCHITECTURE AND LABEL ALIGNMENT
from pathlib import Path

source_path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
src = source_path.read_text(encoding="utf-8", errors="replace").splitlines()

ranges = [
    (8129, 8205),   # A4Encoder implementation
    (8565, 8625),   # A4NER implementation
    (4124, 4195),   # Character spans -> BIO labels
    (10677, 10765), # BIO predictions -> character spans
    (22870, 22965), # Exp10 encoder/checkpoint setup
]

for start, end in ranges:
    print(f"\n{'='*24} LINES {start}-{end} {'='*24}")
    for i in range(start - 1, min(end, len(src))):
        print(f"{i+1}: {src[i]}")

print("\nInspection only — no training performed.")


In [ ]:

# CELL 33 — CHECK CURRENT OBJECTS AND GOLD-LABEL ALIGNMENT
import torch

print("=== Runtime objects ===")
for name in [
    "A4Encoder", "A4NER", "A4FFNCheckpoint",
    "a4_state", "state", "tokenizer",
    "fold1_text", "fold1_gold", "fold2_text", "fold2_gold",
    "a4_train_chunks", "train_chunks", "LABEL2ID", "ID2LABEL",
]:
    obj = globals().get(name, None)
    print(f"{name}: {'AVAILABLE' if obj is not None else 'missing'}")

print("\n=== Current label mapping ===")
if "LABEL2ID" in globals():
    print("Number of BIO labels:", len(LABEL2ID))
    print("First labels:", list(LABEL2ID.items())[:9])
    print("Last labels:", list(LABEL2ID.items())[-6:])

print("\n=== Encoder constructor ===")
import inspect
if "A4Encoder" in globals():
    print(inspect.signature(A4Encoder))
    print("forward:", inspect.signature(A4Encoder.forward))

print("\n=== NER constructor ===")
if "A4NER" in globals():
    print(inspect.signature(A4NER))
    print("forward:", inspect.signature(A4NER.forward))

print("\n=== Checkpoint state keys ===")
ckpt_state = globals().get("a4_state", globals().get("state", None))
if isinstance(ckpt_state, dict):
    keys = list(ckpt_state.keys())
    print("Tensor entries:", len(keys))
    print("Sample keys:", keys[:12])
    print("Encoder FFN keys:", [
        k for k in keys
        if "encoder_layers.0.ffn" in k
    ][:12])

print("\n=== Chunk label consistency ===")
chunks = globals().get(
    "a4_train_chunks",
    globals().get("train_chunks", None)
)
if chunks is not None:
    print("Chunk count:", len(chunks))
    first = chunks[0] if len(chunks) else None
    if first is not None:
        print("First chunk type:", type(first).__name__)
        print("First chunk fields:", getattr(first, "__dict__", {}).keys())
        for field in ["input_ids", "bio_ids", "bio_labels",
                      "token_offsets", "offsets"]:
            value = getattr(first, field, None)
            if value is not None:
                print(f"{field}: length={len(value)}")
else:
    print("No recognized chunk list found.")

print("\nDiagnostic only — no training performed.")


In [ ]:

# CELL 34 — LOCATE AND VERIFY A4 CHECKPOINT
from pathlib import Path
import torch

print("=== Candidate checkpoint files ===")
roots = [Path("/kaggle/input"), Path("/kaggle/working")]
candidates = []

for root in roots:
    if root.exists():
        for p in root.rglob("*.pt"):
            if any(x in p.name.lower() for x in ["a4", "22000"]):
                candidates.append(p)

for p in candidates:
    print(p)

print("\n=== Existing checkpoint object ===")
print("state type:", type(state).__name__)
print("state entries:", len(state) if isinstance(state, dict) else "N/A")

if isinstance(state, dict):
    print("Top-level keys:", list(state.keys())[:20])

    if all(torch.is_tensor(v) for v in state.values()):
        sd = state
    elif isinstance(state.get("model_state_dict"), dict):
        sd = state["model_state_dict"]
    elif isinstance(state.get("state_dict"), dict):
        sd = state["state_dict"]
    else:
        sd = None

    if sd is not None:
        print("\nTensor count:", len(sd))
        for key in [
            "token_embedding.weight",
            "encoder_pos.weight",
            "encoder_layers.0.ffn.net.0.weight",
            "encoder_layers.0.ffn.net.2.weight",
            "decoder_pos.weight",
        ]:
            if key in sd:
                print(key, tuple(sd[key].shape))
            else:
                print(key, "NOT FOUND")
    else:
        print("Could not identify a state dict in the current object.")

print("\nInspection only — no training performed.")


In [ ]:

# CELL 35 — VERIFY REQUIRED MODULE DEFINITIONS
import inspect
import torch
from pathlib import Path

for name in [
    "A4Attention",
    "A4EncoderLayer",
    "A4FFNCheckpoint",
    "A4FFN",
]:
    obj = globals().get(name)
    print(f"\n{name}: {'AVAILABLE' if obj is not None else 'MISSING'}")
    if obj is not None and inspect.isclass(obj):
        try:
            print("Constructor:", inspect.signature(obj))
            print("Forward:", inspect.signature(obj.forward))
        except Exception as e:
            print("Inspection error:", e)

print("\n=== First encoder layer checkpoint keys ===")
for key, value in state.items():
    if key.startswith("encoder_layers.0."):
        print(f"{key:55s} {tuple(value.shape)}")

print("\n=== Encoder checkpoint key count ===")
encoder_keys = [
    k for k in state
    if k.startswith(("token_embedding.", "encoder_pos.", "encoder_layers."))
]
print("Encoder tensors:", len(encoder_keys))
print("Total checkpoint tensors:", len(state))

print("\nInspection only — no training performed.")


In [ ]:

# CELL 36 — BUILD ENCODER AND STRICTLY LOAD A4 WEIGHTS
import torch
import torch.nn as nn

class A4FFNCheckpoint(nn.Module):
    def __init__(self, hidden_size=512, ffn_size=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_size),
            nn.GELU(),
            nn.Linear(ffn_size, hidden_size),
        )

    def forward(self, x):
        return self.net(x)


class A4EncoderForNER(nn.Module):
    def __init__(
        self,
        vocab_size=50000,
        hidden_size=512,
        max_position=1024,
        num_layers=6,
        num_heads=8,
        ffn_size=2048,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.token_embedding = nn.Embedding(vocab_size, hidden_size)
        self.encoder_pos = nn.Embedding(max_position, hidden_size)

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(hidden_size, num_heads, ffn_size)
            for _ in range(num_layers)
        ])

        # Ensure checkpoint-compatible FFN key names.
        for layer in self.encoder_layers:
            layer.ffn = A4FFNCheckpoint(hidden_size, ffn_size)

    def forward(self, input_ids, attention_mask=None):
        batch_size, seq_len = input_ids.shape
        if seq_len > self.encoder_pos.num_embeddings:
            raise ValueError(f"Sequence length {seq_len} exceeds 1024")

        positions = torch.arange(seq_len, device=input_ids.device)
        x = self.token_embedding(input_ids)
        x = x + self.encoder_pos(positions).unsqueeze(0)

        # Match the Phase-1 encoder's no-mask forward path.
        for layer in self.encoder_layers:
            x = layer(x, mask=None)

        return x


# Extract only the pretrained encoder parameters.
encoder_state = {
    key: value
    for key, value in state.items()
    if key.startswith((
        "token_embedding.",
        "encoder_pos.",
        "encoder_layers.",
    ))
}

encoder = A4EncoderForNER()

# This must succeed with no missing or unexpected keys.
load_result = encoder.load_state_dict(encoder_state, strict=True)
print("Strict checkpoint load:", load_result)
print("Encoder parameters:", f"{sum(p.numel() for p in encoder.parameters()):,}")

# Dummy forward-pass test only.
encoder.eval()
with torch.no_grad():
    dummy_ids = torch.randint(0, 50000, (1, 64))
    dummy_hidden = encoder(dummy_ids)

print("Dummy input shape:", tuple(dummy_ids.shape))
print("Encoder output shape:", tuple(dummy_hidden.shape))
print("All output values finite:", torch.isfinite(dummy_hidden).all().item())
print("Training performed: NO")


In [ ]:

# CELL 37 — VERIFY TOKENIZER AND GOLD ANNOTATION FORMAT
print("=== Tokenizer ===")
print("Tokenizer type:", type(tokenizer).__name__)

try:
    print("Tokenizer vocab size:", tokenizer.get_vocab_size())
except Exception:
    try:
        print("Tokenizer vocab size:", tokenizer.vocab_size)
    except Exception as e:
        print("Could not read vocab size:", e)

print("\n=== Fold 1 sample ===")
print("Document IDs:", list(fold1_text.keys())[:3])

sample_id = next(iter(fold1_text))
sample_text = fold1_text[sample_id]
sample_gold = fold1_gold[sample_id]

print("Sample document ID:", sample_id)
print("Text type:", type(sample_text).__name__)
print("Text length:", len(sample_text))
print("Text preview:", repr(sample_text[:300]))
print("Gold annotation type:", type(sample_gold).__name__)
print("Gold annotation count:", len(sample_gold))
print("Gold examples:", sample_gold[:5])

print("\n=== Tokenizer offset test ===")
encoded = tokenizer.encode(sample_text[:500])
print("Token count:", len(encoded.ids))
print("First token IDs:", encoded.ids[:12])
print("First offsets:", encoded.offsets[:12])

print("\nInspection only — no training performed.")


In [ ]:

# CELL 38 — AUDIT CHARACTER-TO-TOKEN ALIGNMENT
from collections import Counter

def audit_alignment(texts, gold_by_doc, tokenizer, max_docs=None):
    stats = Counter()
    examples = []

    doc_ids = list(texts.keys())
    if max_docs is not None:
        doc_ids = doc_ids[:max_docs]

    for doc_id in doc_ids:
        text = texts[doc_id]
        enc = tokenizer.encode(text)
        offsets = enc.offsets

        for span in gold_by_doc[doc_id]:
            start, end, label = int(span[0]), int(span[1]), str(span[2])

            if not (0 <= start < end <= len(text)):
                stats["invalid_gold_spans"] += 1
                continue

            stats["gold_spans"] += 1
            covered = [
                (i, ts, te)
                for i, (ts, te) in enumerate(offsets)
                if te > ts and ts < end and te > start
            ]

            if not covered:
                stats["no_overlapping_token"] += 1
                if len(examples) < 8:
                    examples.append((doc_id, start, end, label, text[start:end], []))
                continue

            first_start = covered[0][1]
            last_end = covered[-1][2]

            if first_start == start:
                stats["exact_start_boundary"] += 1
            if last_end == end:
                stats["exact_end_boundary"] += 1
            if first_start == start and last_end == end:
                stats["exact_both_boundaries"] += 1

            # Tokens whose offsets extend outside the gold span.
            overhang = any(ts < start or te > end for _, ts, te in covered)
            if overhang:
                stats["span_has_overhanging_token"] += 1
                if len(examples) < 8:
                    examples.append((
                        doc_id, start, end, label, text[start:end],
                        [(i, ts, te, text[ts:te]) for i, ts, te in covered]
                    ))

    total = stats["gold_spans"]
    print("Documents checked:", len(doc_ids))
    print("Gold spans checked:", total)
    print("Invalid gold spans:", stats["invalid_gold_spans"])
    print("Spans with no overlapping token:", stats["no_overlapping_token"])

    for key, title in [
        ("exact_start_boundary", "Exact start boundary"),
        ("exact_end_boundary", "Exact end boundary"),
        ("exact_both_boundaries", "Both boundaries exact"),
        ("span_has_overhanging_token", "At least one token extends outside span"),
    ]:
        count = stats[key]
        print(f"{title}: {count}/{total} ({100*count/total:.2f}%)" if total else f"{title}: 0")

    print("\nExamples of boundary issues:")
    for item in examples:
        print(item)

    return stats

stats_fold1 = audit_alignment(fold1_text, fold1_gold, tokenizer)
stats_fold2 = audit_alignment(fold2_text, fold2_gold, tokenizer)

print("\nAudit only — no labels or model weights changed.")


In [ ]:

# CELL 39 — LOCATE EXISTING CHUNKING AND LABEL FUNCTIONS
from pathlib import Path

source_path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
src = source_path.read_text(encoding="utf-8", errors="replace").splitlines()

terms = [
    "def char_spans_to_bio(",
    "def bio_to_char_spans(",
    "def build_gold_token_labels(",
    "def make_chunks(",
    "def create_chunks(",
    "def build_chunks(",
    "def calculate_strict_metrics(",
    "def strict_span_metrics(",
]

for term in terms:
    hits = [
        i for i, line in enumerate(src)
        if term.lower() in line.lower()
    ]
    print(f"\n{'='*15} {term}: {len(hits)} matches {'='*15}")
    for i in hits[:4]:
        lo = max(0, i - 2)
        hi = min(len(src), i + 32)
        print(f"\n--- Lines {lo+1}-{hi} ---")
        for j in range(lo, hi):
            print(f"{j+1}: {src[j]}")

print("\nInspection only — no training performed.")


In [ ]:

# CELL 40 — INSPECT CHUNKING IMPLEMENTATION
from pathlib import Path

source_path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
src = source_path.read_text(encoding="utf-8", errors="replace").splitlines()

ranges = [
    (7080, 7255),   # Chunk dataclass and build_chunks
    (7255, 7350),   # Remaining chunk logic/helpers
    (4176, 4255),   # BIO predictions -> character spans
    (4514, 4575),   # Alternative gold-label builder
]

for start, end in ranges:
    print(f"\n{'='*18} SOURCE LINES {start}-{end} {'='*18}")
    for i in range(start - 1, min(end, len(src))):
        print(f"{i+1}: {src[i]}")

print("\n=== Current notebook constants ===")
for name in [
    "MAX_TOKENS", "MAX_LENGTH", "MAX_ENCODER_LEN",
    "STRIDE", "OVERLAP", "ENTITY_LABELS", "labels",
]:
    print(f"{name}:", globals().get(name, "<not defined in current session>"))

print("\nInspection only — no training performed.")


In [ ]:

# CELL 41 — BUILD CONSISTENT NER CHUNKS (NO TRAINING)
from collections import Counter
from dataclasses import dataclass
import torch

ENTITY_LABELS = list(labels)
BIO_LABELS = ["O"] + [
    tag
    for entity in ENTITY_LABELS
    for tag in (f"B-{entity}", f"I-{entity}")
]
LABEL2ID = {tag: i for i, tag in enumerate(BIO_LABELS)}
ID2LABEL = {i: tag for tag, i in LABEL2ID.items()}

MAX_TOKENS = 1024
STRIDE = 768
IGNORE_INDEX = -100

@dataclass
class NERChunkV2:
    doc_id: str
    chunk_id: int
    input_ids: list
    offsets: list
    bio_ids: list
    gold_spans: list

def build_document_chunks(doc_id, text, gold_spans):
    enc = tokenizer.encode(text)
    ids = enc.ids
    offsets = enc.offsets
    n = len(ids)

    # Assign BIO labels globally before slicing into overlapping chunks.
    global_labels = [LABEL2ID["O"]] * n
    entity_token_ranges = []

    for raw in gold_spans:
        start, end, entity = int(raw[0]), int(raw[1]), str(raw[2])
        if entity not in ENTITY_LABELS:
            raise ValueError(f"Unknown label {entity!r} in {doc_id}")

        covered = [
            i for i, (ts, te) in enumerate(offsets)
            if te > ts and ts < end and te > start
        ]
        if not covered:
            raise ValueError(f"Entity has no tokens: {doc_id}, {raw}")

        entity_token_ranges.append((covered[0], covered[-1], start, end, entity))
        global_labels[covered[0]] = LABEL2ID[f"B-{entity}"]
        for i in covered[1:]:
            global_labels[i] = LABEL2ID[f"I-{entity}"]

    chunks = []
    entity_covered_by_chunk = Counter()
    chunk_id = 0
    start_idx = 0

    while start_idx < n:
        end_idx = min(start_idx + MAX_TOKENS, n)
        chunk_ids = ids[start_idx:end_idx]
        chunk_offsets = offsets[start_idx:end_idx]
        chunk_labels = global_labels[start_idx:end_idx].copy()

        contained_spans = []
        for first, last, s, e, entity in entity_token_ranges:
            # Count only chunks containing the complete token sequence.
            if first >= start_idx and last < end_idx:
                contained_spans.append((s, e, entity))
                entity_covered_by_chunk[(s, e, entity)] += 1
            elif first < end_idx and last >= start_idx:
                # Do not teach the model that a clipped entity is O.
                lo = max(first, start_idx) - start_idx
                hi = min(last, end_idx - 1) - start_idx
                for j in range(lo, hi + 1):
                    chunk_labels[j] = IGNORE_INDEX

        chunks.append(NERChunkV2(
            doc_id=str(doc_id),
            chunk_id=chunk_id,
            input_ids=chunk_ids,
            offsets=chunk_offsets,
            bio_ids=chunk_labels,
            gold_spans=contained_spans,
        ))

        chunk_id += 1
        if end_idx >= n:
            break
        start_idx += STRIDE

    missing = [
        (s, e, entity)
        for _, _, s, e, entity in entity_token_ranges
        if entity_covered_by_chunk[(s, e, entity)] == 0
    ]
    return chunks, missing

def build_fold_chunks(text_map, gold_map):
    if set(text_map) != set(gold_map):
        raise ValueError("Text/gold document IDs do not match.")

    chunks = []
    missing_entities = []
    for doc_id, text in text_map.items():
        doc_chunks, missing = build_document_chunks(
            doc_id, text, gold_map[doc_id]
        )
        chunks.extend(doc_chunks)
        missing_entities.extend((doc_id, *x) for x in missing)

    return chunks, missing_entities

fold1_chunks_v2, missing_fold1 = build_fold_chunks(fold1_text, fold1_gold)
fold2_chunks_v2, missing_fold2 = build_fold_chunks(fold2_text, fold2_gold)

print("BIO classes:", len(BIO_LABELS))
print("Fold 1 documents/chunks:", len(fold1_text), len(fold1_chunks_v2))
print("Fold 2 documents/chunks:", len(fold2_text), len(fold2_chunks_v2))
print("Fold 1 entities not fully represented:", len(missing_fold1))
print("Fold 2 entities not fully represented:", len(missing_fold2))

all_chunks = fold1_chunks_v2 + fold2_chunks_v2
print("Total chunks:", len(all_chunks))
print("Max chunk length:", max(len(c.input_ids) for c in all_chunks))
print("Min chunk length:", min(len(c.input_ids) for c in all_chunks))
print("Chunks with ignored boundary tokens:",
      sum(any(y == IGNORE_INDEX for y in c.bio_ids) for c in all_chunks))

assert len(BIO_LABELS) == 25
assert all(len(c.input_ids) == len(c.bio_ids) == len(c.offsets)
           for c in all_chunks)
assert not missing_fold1 and not missing_fold2, (
    missing_fold1[:5], missing_fold2[:5]
)

print("\nChunk construction and structural checks passed.")
print("Training performed: NO")


In [ ]:

# CELL 42 — BIO ROUND-TRIP VALIDATION (NO TRAINING)

from collections import Counter

def decode_chunk_gold(chunk):
    """Decode token BIO labels into character spans; exclude ignored tokens."""
    valid_indices = [
        i for i, label_id in enumerate(chunk.bio_ids)
        if label_id != IGNORE_INDEX
    ]

    spans = []
    active_label = None
    active_start = None
    active_end = None

    def flush():
        nonlocal active_label, active_start, active_end
        if active_label is not None:
            s, e = active_start, active_end
            while s < e and chunk.offsets[0] is None:
                break
            spans.append((s, e, active_label))
        active_label = active_start = active_end = None

    for i in valid_indices:
        tag = ID2LABEL[chunk.bio_ids[i]]
        start, end = chunk.offsets[i]

        if tag == "O":
            flush()
            continue

        prefix, entity = tag.split("-", 1)

        if prefix == "B" or active_label != entity:
            flush()
            active_label = entity
            active_start, active_end = start, end
        else:
            active_end = end

    flush()

    # Remove tokenizer-added whitespace at span boundaries.
    text = None
    return spans

# Validate labels and offsets structurally, without training.
bad_lengths = []
invalid_ids = Counter()
ignored = 0
nonempty = 0

for chunk in all_chunks:
    if not (len(chunk.input_ids) == len(chunk.offsets) == len(chunk.bio_ids)):
        bad_lengths.append((chunk.doc_id, chunk.chunk_id))
    for y in chunk.bio_ids:
        if y == IGNORE_INDEX:
            ignored += 1
        elif y < 0 or y >= len(BIO_LABELS):
            invalid_ids[y] += 1
        else:
            nonempty += 1

print("Chunks checked:", len(all_chunks))
print("Length mismatches:", len(bad_lengths))
print("Invalid BIO IDs:", dict(invalid_ids))
print("Ignored token labels:", ignored)
print("Valid supervised token labels:", nonempty)

assert not bad_lengths
assert not invalid_ids
print("BIO label integrity checks passed.")
print("Training performed: NO")


In [ ]:

# CELL 43 — GOLD SPAN ROUND-TRIP CHECK (NO TRAINING)

from collections import Counter, defaultdict

def encode_gold_to_bio(text, gold_spans):
    enc = tokenizer.encode(text)
    offsets = enc.offsets
    token_labels = ["O"] * len(offsets)

    for raw in gold_spans:
        s, e, entity = int(raw[0]), int(raw[1]), str(raw[2])
        covered = [
            i for i, (ts, te) in enumerate(offsets)
            if te > ts and ts < e and te > s
        ]
        if not covered:
            raise ValueError(f"Entity has no token coverage: {raw}")

        token_labels[covered[0]] = f"B-{entity}"
        for i in covered[1:]:
            token_labels[i] = f"I-{entity}"

    return offsets, token_labels


def decode_bio_to_spans(text, offsets, token_labels):
    decoded = []
    active_label = None
    active_start = active_end = None

    def flush():
        nonlocal active_label, active_start, active_end
        if active_label is not None:
            s, e = active_start, active_end
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            if s < e:
                decoded.append((s, e, active_label))
        active_label = active_start = active_end = None

    for (start, end), tag in zip(offsets, token_labels):
        if tag == "O":
            flush()
            continue

        prefix, entity = tag.split("-", 1)
        if prefix == "B" or active_label != entity:
            flush()
            active_label = entity
            active_start, active_end = start, end
        else:
            active_end = end

    flush()
    return decoded


counts = defaultdict(Counter)
examples = []

for split_name, text_map, gold_map in [
    ("fold1", fold1_text, fold1_gold),
    ("fold2", fold2_text, fold2_gold),
]:
    for doc_id, text in text_map.items():
        offsets, bio = encode_gold_to_bio(text, gold_map[doc_id])
        decoded = decode_bio_to_spans(text, offsets, bio)

        gold_set = {
            (int(x[0]), int(x[1]), str(x[2]))
            for x in gold_map[doc_id]
        }
        decoded_set = set(decoded)

        counts[split_name]["gold"] += len(gold_set)
        counts[split_name]["decoded"] += len(decoded_set)
        counts[split_name]["exact"] += len(gold_set & decoded_set)
        counts[split_name]["missing"] += len(gold_set - decoded_set)
        counts[split_name]["extra"] += len(decoded_set - gold_set)

        if gold_set - decoded_set and len(examples) < 10:
            examples.append({
                "split": split_name,
                "doc_id": doc_id,
                "missing": list(gold_set - decoded_set)[:3],
            })

for split_name in ("fold1", "fold2"):
    c = counts[split_name]
    recall = c["exact"] / c["gold"] if c["gold"] else 0
    print(
        f"{split_name}: gold={c['gold']}, decoded={c['decoded']}, "
        f"exact={c['exact']}, missing={c['missing']}, "
        f"extra={c['extra']}, exact_recall={recall:.4%}"
    )

print("Mismatch examples:", examples[:5])
print("Training performed: NO")


In [ ]:

# CELL 44 — INSPECT SPAN MISMATCHES (NO TRAINING)

def inspect_span_mismatch(split_name, doc_id, gold_span):
    text_map = fold1_text if split_name == "fold1" else fold2_text
    gold_map = fold1_gold if split_name == "fold1" else fold2_gold

    text = text_map[doc_id]
    offsets, bio = encode_gold_to_bio(text, gold_map[doc_id])
    decoded = decode_bio_to_spans(text, offsets, bio)

    s, e, entity = gold_span
    same_label = [
        d for d in decoded
        if d[2] == entity and not (d[1] <= s or d[0] >= e)
    ]

    print(f"\n{split_name} | doc={doc_id} | label={entity}")
    print("Gold:", (s, e, entity), repr(text[s:e]))
    print("Overlapping decoded spans:")
    for d in same_label:
        print(" ", d, repr(text[d[0]:d[1]]))

    print("Nearby token offsets:")
    for i, (ts, te) in enumerate(offsets):
        if te > s - 8 and ts < e + 8:
            print(
                f"  token={i:5d} offsets=({ts},{te}) "
                f"tag={bio[i]:12s} text={text[ts:te]!r}"
            )

for example in examples[:3]:
    for span in example["missing"][:2]:
        inspect_span_mismatch(
            example["split"],
            example["doc_id"],
            span
        )

print("\nTraining performed: NO")


In [ ]:

# CELL 45 — TOKEN BOUNDARY RECOVERY DIAGNOSTIC (NO TRAINING)

from collections import Counter

def boundary_recovery_stats(text_map, gold_map):
    stats = Counter()
    examples = []

    for doc_id, text in text_map.items():
        offsets = tokenizer.encode(text).offsets

        for raw in gold_map[doc_id]:
            s, e, label = int(raw[0]), int(raw[1]), str(raw[2])

            overlapping = [
                (i, ts, te)
                for i, (ts, te) in enumerate(offsets)
                if te > ts and ts < e and te > s
            ]

            stats["entities"] += 1
            if not overlapping:
                stats["no_overlapping_tokens"] += 1
                continue

            first_i, first_s, first_e = overlapping[0]
            last_i, last_s, last_e = overlapping[-1]

            start_exact = first_s == s
            end_exact = last_e == e

            stats["start_exact"] += start_exact
            stats["end_exact"] += end_exact
            stats["both_exact"] += start_exact and end_exact

            if not (start_exact and end_exact) and len(examples) < 12:
                examples.append({
                    "doc_id": doc_id,
                    "label": label,
                    "gold": (s, e),
                    "first_token": (first_s, first_e, text[first_s:first_e]),
                    "last_token": (last_s, last_e, text[last_s:last_e]),
                    "gold_text": text[s:e],
                })

    return stats, examples


for split_name, text_map, gold_map in [
    ("fold1", fold1_text, fold1_gold),
    ("fold2", fold2_text, fold2_gold),
]:
    stats, samples = boundary_recovery_stats(text_map, gold_map)
    n = stats["entities"]
    print(f"\n{split_name}: {n} entities")
    print("Exact start:", f"{stats['start_exact']}/{n}")
    print("Exact end:", f"{stats['end_exact']}/{n}")
    print("Both exact:", f"{stats['both_exact']}/{n}")
    print("Examples of non-exact boundaries:")
    for sample in samples[:6]:
        print(sample)

print("\nTraining performed: NO")


In [ ]:

# CELL 46 — CLASSIFY TOKEN/GOLD BOUNDARY OVERHANG (NO TRAINING)

from collections import Counter, defaultdict

def classify_boundary_overhang(text_map, gold_map):
    stats = Counter()
    by_label = defaultdict(Counter)
    examples = []

    for doc_id, text in text_map.items():
        offsets = tokenizer.encode(text).offsets

        for raw in gold_map[doc_id]:
            s, e, label = int(raw[0]), int(raw[1]), str(raw[2])

            covered = [
                (i, ts, te)
                for i, (ts, te) in enumerate(offsets)
                if te > ts and ts < e and te > s
            ]

            if not covered:
                stats["no_token"] += 1
                continue

            _, first_s, first_e = covered[0]
            _, last_s, last_e = covered[-1]

            left_extra = text[first_s:s] if first_s < s else ""
            right_extra = text[e:last_e] if last_e > e else ""

            if not left_extra and not right_extra:
                kind = "exact"
            elif (not left_extra or left_extra.isspace()) and (
                not right_extra or right_extra.isspace()
            ):
                kind = "whitespace_only"
            else:
                kind = "non_whitespace_crossing"

            stats[kind] += 1
            by_label[label][kind] += 1

            if kind == "non_whitespace_crossing" and len(examples) < 12:
                examples.append({
                    "doc_id": doc_id,
                    "label": label,
                    "gold_text": text[s:e],
                    "left_extra": left_extra,
                    "right_extra": right_extra,
                    "left_token": text[first_s:first_e],
                    "right_token": text[last_s:last_e],
                })

    return stats, by_label, examples


for split_name, text_map, gold_map in [
    ("fold1", fold1_text, fold1_gold),
    ("fold2", fold2_text, fold2_gold),
]:
    stats, by_label, examples = classify_boundary_overhang(
        text_map, gold_map
    )

    total = sum(stats.values())
    print(f"\n{split_name}: {total} entities")
    for category in [
        "exact", "whitespace_only", "non_whitespace_crossing", "no_token"
    ]:
        print(f"{category}: {stats[category]}")

    print("Non-whitespace crossing examples:")
    for ex in examples[:6]:
        print(ex)

print("\nTraining performed: NO")


In [ ]:
from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

terms = [
    "a4_ner_exp10_classkd_epoch3.pt",
    "a4_ner_exp10",
    "classkd",
]

matches = []

for root in roots:
    for path in root.rglob("*"):
        if path.is_file() and any(term.lower() in path.name.lower() for term in terms):
            matches.append(path)

print("Matching checkpoints:")
for path in matches:
    print(path, f"({path.stat().st_size / (1024**2):.1f} MB)")

if not matches:
    print("Experiment 10 checkpoint not found by filename.")

In [ ]:
from pathlib import Path

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\n{'='*60}\n{root}")
    files = sorted(root.rglob("*.pt"))

    if not files:
        print("No .pt checkpoints found.")
        continue

    for p in files:
        print(f"{p}  |  {p.stat().st_size / (1024**2):.1f} MB")

    print("Total checkpoints:", len(files))

In [ ]:
import json
from pathlib import Path

notebook_path = Path("/kaggle/working/phase-2.ipynb")

if not notebook_path.exists():
    candidates = list(Path("/kaggle/working").glob("*.ipynb"))
    print("Notebook not found at expected path.")
    print("Available notebooks:", candidates)
else:
    nb = json.loads(notebook_path.read_text(encoding="utf-8"))

    terms = [
        "exp10",
        "classkd",
        "a4_ner_exp10_classkd_epoch3.pt",
        "submission_p2.zip",
        "prediction.json",
    ]

    for i, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))
        outputs = json.dumps(cell.get("outputs", []), ensure_ascii=False)
        combined = source + "\n" + outputs

        found = [term for term in terms if term.lower() in combined.lower()]
        if found:
            print(f"\n--- Cell {i} | Matches: {found} ---")
            print(source[:5000])

In [ ]:

from pathlib import Path

for root in [Path("/kaggle/working"), Path("/kaggle/input"), Path("/kaggle/temp")]:
    print(f"\nSearching {root}")
    if root.exists():
        files = list(root.rglob("*.ipynb"))
        for p in files[:30]:
            print(p)
        if not files:
            print("No notebooks found.")

In [ ]:
from pathlib import Path
import zipfile
import json

print("=== SEARCHING FOR EXISTING SUBMISSIONS ===")

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path("/kaggle/temp"),
]

for root in roots:
    if not root.exists():
        continue

    for pattern in ["submission*.zip", "prediction.json",
                    "*exp10*.pt", "*classkd*.pt"]:
        matches = list(root.rglob(pattern))
        for p in matches[:30]:
            print(f"{p} | {p.stat().st_size:,} bytes")

print("\n=== CHECK KNOWN-SCORING ZIP ===")
p = Path("/kaggle/working/submission_p2.zip")

if p.exists():
    with zipfile.ZipFile(p) as z:
        print("ZIP contents:", z.namelist())
        print("Has prediction.json:",
              "prediction.json" in z.namelist())
else:
    print("submission_p2.zip is not at the expected path.")

In [ ]:
from pathlib import Path

for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    print(f"\n{root}:")
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() in {
                ".zip", ".json", ".ipynb", ".pt", ".pth"
            }:
                print(p)

In [ ]:
from pathlib import Path

matches = list(
    Path("/kaggle/input").rglob("a4_ner_exp4_lorma_epoch3.pt")
)

if matches:
    print("Checkpoint found:", matches[0])
    print(f"Size: {matches[0].stat().st_size / 1024**2:.1f} MB")
else:
    print("Checkpoint not found. Check that the dataset is attached.")

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("a4_ner_exp4_lorma_epoch3.pt"):
    print("FOUND:", p)
    print(f"Size: {p.stat().st_size / 1024**2:.1f} MB")

In [ ]:
from pathlib import Path
import torch

CKPT_PATH = Path(
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/a4_ner_exp4_lorma_epoch3.pt"
)

assert CKPT_PATH.exists(), f"Checkpoint missing: {CKPT_PATH}"

checkpoint = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint loaded successfully.")
print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Available keys:", list(checkpoint.keys()))
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: {len(value)} entries")
        elif isinstance(value, (int, float, str)):
            print(f"{key}: {value}")

In [ ]:
state = checkpoint["model_state_dict"]

print("Total parameter entries:", len(state))
print("\nFirst 30 parameter names and shapes:")

for i, (name, value) in enumerate(state.items()):
    if i >= 30:
        break
    if hasattr(value, "shape"):
        print(f"{name}: {tuple(value.shape)}")
    else:
        print(f"{name}: {type(value)}")

print("\nLORMA configuration:")
print("Rank:", checkpoint["lorma_rank"])
print("Alpha:", checkpoint["lorma_alpha"])
print("Dropout:", checkpoint["lorma_dropout"])

In [ ]:
import inspect
import torch.nn as nn

print("Existing model classes:")
for name, obj in list(globals().items()):
    if isinstance(obj, type) and (
        "Encoder" in name or "LORMA" in name.upper()
        or "NER" in name.upper() or "Model" in name
    ):
        try:
            print(f"\n{name}")
            print(inspect.getfile(obj))
            print(inspect.signature(obj))
        except (TypeError, OSError, ValueError):
            print(name, "(built-in or dynamically defined)")

print("\nCheckpoint head / non-encoder parameters:")
for name, tensor in checkpoint["model_state_dict"].items():
    if not name.startswith("encoder."):
        print(name, tuple(tensor.shape))

In [ ]:
from pathlib import Path
import json

roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

terms = [
    "class LORMA",
    "lorma_A",
    "class TransformerEncoder",
    "encoder_layers",
    "classifier",
    "A4_LORMA",
]

found = False

for root in roots:
    for path in root.rglob("*.ipynb"):
        try:
            nb = json.loads(path.read_text(encoding="utf-8"))
            for i, cell in enumerate(nb.get("cells", [])):
                source = "".join(cell.get("source", []))
                matches = [term for term in terms if term.lower() in source.lower()]
                if matches:
                    found = True
                    print(f"\nFILE: {path} | CELL: {i} | MATCHES: {matches}")
                    print(source[:5000])
        except Exception:
            pass

if not found:
    print("No matching notebook code found in Kaggle input/working.")

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path

# --------------------------------------------------
# 1. Load Experiment 4 checkpoint
# --------------------------------------------------
CKPT_PATH = Path(
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/a4_ner_exp4_lorma_epoch3.pt"
)
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
sd = ckpt["model_state_dict"]

H = sd["encoder.token_embedding.weight"].shape[1]
V = sd["encoder.token_embedding.weight"].shape[0]
MAX_POS = sd["encoder.encoder_pos.weight"].shape[0]
N_LAYERS = len({
    int(k.split(".")[2])
    for k in sd
    if k.startswith("encoder.encoder_layers.")
    and k.split(".")[2].isdigit()
})
FFN = sd["encoder.encoder_layers.0.ffn.net.0.weight"].shape[0]
NUM_LABELS = sd["classifier.weight"].shape[0]

print({
    "vocab": V, "hidden": H, "max_positions": MAX_POS,
    "encoder_layers": N_LAYERS, "ffn_size": FFN,
    "labels": NUM_LABELS
})

# --------------------------------------------------
# 2. Reconstruct A4 encoder layers
# --------------------------------------------------
class LORMALinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        self.lorma_A = nn.Parameter(torch.empty(rank, base.in_features))
        self.lorma_B = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scaling = alpha / rank
        self.lorma_dropout = nn.Dropout(dropout)

    def forward(self, x):
        return (
            self.base(x)
            + self.scaling
            * (self.lorma_dropout(x) @ self.lorma_A.T @ self.lorma_B.T)
        )

class A4Attention(nn.Module):
    def __init__(self, hidden, heads=8):
        super().__init__()
        self.q_proj = nn.Linear(hidden, hidden)
        self.k_proj = nn.Linear(hidden, hidden)
        self.v_proj = nn.Linear(hidden, hidden)
        self.out_proj = nn.Linear(hidden, hidden)
        self.num_heads = heads
        self.head_dim = hidden // heads

    def forward(self, query, key, value, mask=None):
        B, Tq, C = query.shape
        Tk = key.shape[1]

        q = self.q_proj(query).view(B, Tq, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(key).view(B, Tk, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(value).view(B, Tk, self.num_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        out = torch.softmax(scores, dim=-1) @ v
        out = out.transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(out)

class A4FFN(nn.Module):
    def __init__(self, hidden, ffn):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, ffn),
            nn.GELU(),
            nn.Linear(ffn, hidden)
        )

    def forward(self, x):
        return self.net(x)

class A4EncoderLayer(nn.Module):
    def __init__(self, hidden, ffn):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden)
        self.attn = A4Attention(hidden, 8)
        self.norm2 = nn.LayerNorm(hidden)
        self.ffn = A4FFN(hidden, ffn)

    def forward(self, x, mask=None):
        h = self.norm1(x)
        x = x + self.attn(h, h, h, mask)
        return x + self.ffn(self.norm2(x))

class A4EncoderForNER(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_size = H
        self.token_embedding = nn.Embedding(V, H)
        self.encoder_pos = nn.Embedding(MAX_POS, H)
        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(H, FFN) for _ in range(N_LAYERS)
        ])
        self.encoder_norm = nn.LayerNorm(H)

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        if T > MAX_POS:
            raise ValueError(f"Sequence length {T} exceeds {MAX_POS}")
        pos = torch.arange(T, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.encoder_pos(pos).unsqueeze(0)
        for layer in self.encoder_layers:
            x = layer(x, mask=None)
        return self.encoder_norm(x)

# --------------------------------------------------
# 3. Insert LORMA into Q/V projections
# --------------------------------------------------
encoder = A4EncoderForNER()

for layer in encoder.encoder_layers:
    layer.attn.q_proj = LORMALinear(layer.attn.q_proj, rank=8, alpha=16, dropout=0.05)
    layer.attn.v_proj = LORMALinear(layer.attn.v_proj, rank=8, alpha=16, dropout=0.05)

class A4NER(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.10)
        self.classifier = nn.Linear(H, NUM_LABELS)

    def forward(self, input_ids, attention_mask=None):
        return self.classifier(self.dropout(self.encoder(input_ids)))

model4 = A4NER()

# --------------------------------------------------
# 4. Strictly load every checkpoint parameter
# --------------------------------------------------
result = model4.load_state_dict(sd, strict=True)
print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)

assert not result.missing_keys and not result.unexpected_keys

# --------------------------------------------------
# 5. Small inference smoke test
# --------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model4 = model4.to(device).eval()

with torch.no_grad():
    dummy = torch.zeros((1, 32), dtype=torch.long, device=device)
    logits = model4(dummy)

print("Logits shape:", tuple(logits.shape))
print("Finite logits:", bool(torch.isfinite(logits).all()))
print("STRICT LOAD + SMOKE TEST PASSED")

In [ ]:
sd = ckpt["model_state_dict"]

print("Encoder-level keys:")
for key in sd:
    if key.startswith("encoder.") and key.count(".") <= 2:
        value = sd[key]
        print(key, tuple(value.shape) if hasattr(value, "shape") else type(value))

print("\nFinal 25 checkpoint keys:")
for key in list(sd.keys())[-25:]:
    value = sd[key]
    print(key, tuple(value.shape) if hasattr(value, "shape") else type(value))

In [ ]:
result = model4.load_state_dict(
    ckpt["model_state_dict"],
    strict=True
)

print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)

assert not result.missing_keys
assert not result.unexpected_keys

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model4 = model4.to(device).eval()

with torch.no_grad():
    dummy = torch.zeros((1, 32), dtype=torch.long, device=device)
    logits = model4(dummy)

print("Logits shape:", tuple(logits.shape))
print("Finite logits:", bool(torch.isfinite(logits).all()))

In [ ]:
result = model4.load_state_dict(
    ckpt["model_state_dict"],
    strict=True
)

print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)

In [ ]:
print("Model class:", type(model4).__name__)
print("Encoder class:", type(model4.encoder).__name__)

print("\nNormalization modules:")
for name, module in model4.named_modules():
    if "norm" in name.lower():
        print(name, type(module).__name__)

print("\nExpected but absent checkpoint keys:")
checkpoint_keys = set(ckpt["model_state_dict"].keys())
model_keys = set(model4.state_dict().keys())

for key in sorted(model_keys - checkpoint_keys):
    print(key)

In [ ]:
# Remove the normalization module that is not present in the checkpoint
if hasattr(model4.encoder, "encoder_norm"):
    delattr(model4.encoder, "encoder_norm")

# Replace the encoder forward method so it does not use encoder_norm
def encoder_forward(input_ids, attention_mask=None):
    B, T = input_ids.shape
    if T > MAX_POS:
        raise ValueError(f"Sequence length {T} exceeds {MAX_POS}")

    positions = torch.arange(T, device=input_ids.device)
    x = (
        model4.encoder.token_embedding(input_ids)
        + model4.encoder.encoder_pos(positions).unsqueeze(0)
    )

    for layer in model4.encoder.encoder_layers:
        x = layer(x, mask=None)

    return x

import types
model4.encoder.forward = types.MethodType(
    encoder_forward, model4.encoder
)

result = model4.load_state_dict(
    ckpt["model_state_dict"], strict=True
)

print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model4 = model4.to(device).eval()

with torch.no_grad():
    dummy = torch.zeros((1, 32), dtype=torch.long, device=device)
    logits = model4(dummy)

print("Device:", device)
print("Logits shape:", tuple(logits.shape))
print("All logits finite:", bool(torch.isfinite(logits).all()))
print("Predicted tag IDs:", logits.argmax(dim=-1)[0, :10].tolist())

assert logits.shape == (1, 32, 25)
assert torch.isfinite(logits).all()

print("Inference smoke test passed.")

In [ ]:
import types
import torch

def encoder_forward(self, input_ids, attention_mask=None):
    B, T = input_ids.shape

    if T > MAX_POS:
        raise ValueError(f"Sequence length {T} exceeds {MAX_POS}")

    positions = torch.arange(T, device=input_ids.device)
    x = (
        self.token_embedding(input_ids)
        + self.encoder_pos(positions).unsqueeze(0)
    )

    for layer in self.encoder_layers:
        x = layer(x, mask=None)

    return x

# Patch the existing encoder instance
model4.encoder.forward = types.MethodType(
    encoder_forward, model4.encoder
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model4 = model4.to(device).eval()

with torch.no_grad():
    dummy = torch.zeros((1, 32), dtype=torch.long, device=device)
    logits = model4(dummy)

print("Device:", device)
print("Logits shape:", tuple(logits.shape))
print("All logits finite:", bool(torch.isfinite(logits).all()))

assert logits.shape == (1, 32, 25)
assert torch.isfinite(logits).all()

print("Inference smoke test passed.")

In [ ]:
from pathlib import Path
import json

base = Path("/kaggle/input/datasets/aq1de3")

test_path = base / "fold3-text/fold3-text.json"
labels_path = base / "labels/labels.txt"
tokenizer_path = base / "tokenizer/tokenizer.json"

for p in [test_path, labels_path, tokenizer_path]:
    print(f"{p.name}: exists={p.exists()}")
    if p.exists():
        print(f"  size={p.stat().st_size / 1024:.1f} KB")

with open(test_path, encoding="utf-8") as f:
    test_data = json.load(f)

with open(labels_path, encoding="utf-8") as f:
    label_lines = [line.strip() for line in f if line.strip()]

print("\nTest document count:", len(test_data))
print("First document ID:", next(iter(test_data)))
print("Checkpoint output classes:", model4.classifier.out_features)
print("Label file entries:", len(label_lines))
print("First 30 labels:", label_lines[:30])

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("Top-level Kaggle inputs:")
for p in root.iterdir():
    print("DIR " if p.is_dir() else "FILE", p)

print("\nFiles matching test data, labels, or tokenizer:")
for p in root.rglob("*"):
    if p.is_file() and (
        "fold3-text" in p.name.lower()
        or p.name.lower() in {"labels.txt", "tokenizer.json"}
    ):
        print(p)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input/datasets/aq1de3")

print("Does dataset directory exist?", root.exists())

if root.exists():
    for p in sorted(root.rglob("*")):
        if p.is_file():
            print(f"{p}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("JSON files:")
for p in root.rglob("*.json"):
    print(p)

print("\nText/label files:")
for p in root.rglob("*"):
    if p.is_file() and (
        p.suffix.lower() in {".txt", ".csv"}
        or "label" in p.name.lower()
        or "fold3" in p.name.lower()
    ):
        print(p)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

for pattern in [
    "fold3-text.json",
    "labels.txt",
    "fold1-text.json",
    "fold1-gold.json",
    "fold2-text.json",
    "fold2-gold.json",
]:
    matches = list(root.rglob(pattern))
    print(f"\n{pattern}:")
    for p in matches:
        print(" ", p)
    if not matches:
        print("  NOT FOUND")

In [ ]:
import json
from pathlib import Path
from tokenizers import Tokenizer

TEST_PATH = Path("/kaggle/input/datasets/aq1de3/fold3-te/fold3-text.json")
LABEL_PATH = Path("/kaggle/input/datasets/aq1de3/labelses/labels.txt")
FOLD1_TEXT = Path("/kaggle/input/datasets/aq1de3/fold1-tex/fold1-text.json")
FOLD1_GOLD = Path("/kaggle/input/datasets/aq1de3/fold1-gold1/fold1-gold.json")
FOLD2_TEXT = Path("/kaggle/input/datasets/aq1de3/fold2-tex/fold2-text.json")
FOLD2_GOLD = Path("/kaggle/input/datasets/aq1de3/fold2-go/fold2-gold.json")
TOKENIZER_PATH = Path("/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json")

for p in [TEST_PATH, LABEL_PATH, FOLD1_TEXT, FOLD1_GOLD,
          FOLD2_TEXT, FOLD2_GOLD, TOKENIZER_PATH]:
    assert p.exists(), f"Missing file: {p}"

def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

test_data = read_json(TEST_PATH)
fold1_text = read_json(FOLD1_TEXT)
fold1_gold = read_json(FOLD1_GOLD)
fold2_text = read_json(FOLD2_TEXT)
fold2_gold = read_json(FOLD2_GOLD)

with open(LABEL_PATH, encoding="utf-8") as f:
    labels = [line.strip() for line in f if line.strip()]

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("Test documents:", len(test_data))
print("Fold 1 text/gold:", len(fold1_text), len(fold1_gold))
print("Fold 2 text/gold:", len(fold2_text), len(fold2_gold))
print("Number of model classes:", model4.classifier.out_features)
print("Number of labels:", len(labels))
print("Labels:", labels)
print("Tokenizer vocabulary size:", tokenizer.get_vocab_size())

first_id = next(iter(test_data))
print("\nFirst test document ID:", first_id)
print("First test record type:", type(test_data[first_id]).__name__)
print("First test record preview:")
print(str(test_data[first_id])[:1200])

assert len(labels) == model4.classifier.out_features, (
    "Label count does not match classifier output size; inspect the mapping "
    "before decoding predictions."
)

In [ ]:
print("Classifier output classes:", model4.classifier.out_features)
print("Labels loaded:", len(labels))

print("\nLabels, with their current indices:")
for i, label in enumerate(labels):
    print(f"{i:2d}: {label!r}")

print("\nRaw labels.txt contents:")
with open(LABEL_PATH, encoding="utf-8") as f:
    print(repr(f.read()[:3000]))

print("\nTest documents:", len(test_data))
first_id = next(iter(test_data))
print("First test ID:", first_id)
print("First test record:")
print(str(test_data[first_id])[:1500])

print("\nCheckpoint classifier:")
print("Weight shape:", tuple(ckpt["model_state_dict"]["classifier.weight"].shape))
print("Bias shape:", tuple(ckpt["model_state_dict"]["classifier.bias"].shape))

In [ ]:
# Inspect one annotated training example
train_id = next(iter(fold1_text))
train_text = fold1_text[train_id]
train_gold = fold1_gold.get(train_id, fold1_gold.get(str(train_id), None))

print("Training document ID:", train_id)
print("Text type:", type(train_text).__name__)
print("Text preview:", str(train_text)[:300])
print("Gold annotation type:", type(train_gold).__name__)
print("Gold annotations preview:", str(train_gold)[:10] if isinstance(train_gold, list) else train_gold)

# Inspect tokenizer offsets on a short sample
sample_text = str(train_text)
encoding = tokenizer.encode(sample_text[:500], add_special_tokens=False)

print("\nSample token count:", len(encoding.ids))
print("First 10 token IDs:", encoding.ids[:10])
print("First 10 character offsets:", encoding.offsets[:10])

for token_id, (start, end) in list(zip(encoding.ids, encoding.offsets))[:10]:
    print(
        f"token={token_id:6d}, offset=({start:3d},{end:3d}), "
        f"text={sample_text[start:end]!r}"
    )

In [ ]:
# Inspect annotations and chunk settings
train_id = next(iter(fold1_text))
train_text = fold1_text[train_id]
train_gold = fold1_gold.get(train_id, fold1_gold.get(str(train_id), []))

print("First 5 gold spans:")
for span in train_gold[:5]:
    start, end, label = int(span[0]), int(span[1]), str(span[2])
    print({
        "span": span,
        "text_at_offsets": train_text[start:end],
        "valid_bounds": 0 <= start < end <= len(train_text),
        "label_in_official_labels": label in labels
    })

print("\nTokenizer vocabulary size:", tokenizer.get_vocab_size())
print("Model maximum position embeddings:", MAX_POS)
print("Classifier output classes:", model4.classifier.out_features)

# Inspect the training notebook constants if they exist in this session
for name in ["MAX_TOKENS", "MAX_LENGTH", "STRIDE", "OVERLAP"]:
    print(f"{name} =", globals().get(name, "not defined"))

In [ ]:
import torch
from collections import Counter

MAX_TOKENS = 768
OVERLAP = 256
STEP = MAX_TOKENS - OVERLAP

# Check token lengths before full inference
test_lengths = {}
for doc_id, text in test_data.items():
    test_lengths[str(doc_id)] = len(
        tokenizer.encode(text, add_special_tokens=False).ids
    )

print("Test documents:", len(test_lengths))
print("Total tokens:", sum(test_lengths.values()))
print("Max token length:", max(test_lengths.values()))
print("Median token length:", sorted(test_lengths.values())[len(test_lengths)//2])
print("Estimated chunks:", sum(
    max(1, (n - OVERLAP + STEP - 1) // STEP)
    for n in test_lengths.values()
))

# Smoke-test one real chunk from the first test document
doc_id = next(iter(test_data))
text = test_data[doc_id]
enc = tokenizer.encode(text, add_special_tokens=False)

input_ids = enc.ids[:MAX_TOKENS]
offsets = enc.offsets[:MAX_TOKENS]

device = next(model4.parameters()).device
x = torch.tensor([input_ids], dtype=torch.long, device=device)

with torch.no_grad():
    logits = model4(x)

pred_ids = logits.argmax(dim=-1)[0].cpu().tolist()

print("\nReal document ID:", doc_id)
print("Original characters:", len(text))
print("Token count:", len(enc.ids))
print("Chunk length:", len(input_ids))
print("Logits shape:", tuple(logits.shape))
print("Predicted tag counts:", Counter(ID2LABEL[i] for i in pred_ids))
print("First 10 token predictions:")
for token_id, (start, end), pred in list(zip(input_ids, offsets, pred_ids))[:10]:
    print(ID2LABEL[pred], repr(text[start:end]))

assert logits.shape == (1, len(input_ids), 25)
assert torch.isfinite(logits).all()

In [ ]:
ENTITY_LABELS = labels.copy()

BIO_LABELS = ["O"]
for entity in ENTITY_LABELS:
    BIO_LABELS.extend([f"B-{entity}", f"I-{entity}"])

LABEL2ID = {label: i for i, label in enumerate(BIO_LABELS)}
ID2LABEL = {i: label for i, label in enumerate(BIO_LABELS)}

assert len(BIO_LABELS) == model4.classifier.out_features == 25

print("BIO labels initialized:", len(BIO_LABELS))
print(BIO_LABELS)


In [ ]:
def bio_to_char_spans(text, offsets, pred_ids):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    def close_entity():
        nonlocal current_label, current_start, current_end
        if current_label is not None and current_start is not None:
            if 0 <= current_start < current_end <= len(text):
                spans.append([
                    current_start,
                    current_end,
                    current_label
                ])
        current_label = None
        current_start = None
        current_end = None

    for (start, end), pred_id in zip(offsets, pred_ids):
        if start == end:
            continue

        tag = ID2LABEL[int(pred_id)]

        if tag == "O":
            close_entity()
            continue

        prefix, entity_type = tag.split("-", 1)

        if prefix == "B":
            close_entity()
            current_label = entity_type
            current_start = start
            current_end = end

        elif prefix == "I":
            if current_label == entity_type and current_start is not None:
                current_end = end
            else:
                close_entity()
                current_label = entity_type
                current_start = start
                current_end = end

    close_entity()
    return spans


# Decode the predictions already generated for the first chunk
chunk_spans = bio_to_char_spans(text, offsets, pred_ids)

print("Decoded spans:", len(chunk_spans))
print("\nFirst 20 decoded spans:")
for start, end, label in chunk_spans[:20]:
    print({
        "start": start,
        "end": end,
        "label": label,
        "text": text[start:end]
    })

assert all(
    0 <= s < e <= len(text) and text[s:e]
    for s, e, _ in chunk_spans
)
print("\nCharacter-boundary validation passed.")

In [ ]:
from collections import Counter

train_id = next(iter(fold1_text))
train_text = fold1_text[train_id]
train_gold = fold1_gold.get(train_id, fold1_gold.get(str(train_id), []))

enc = tokenizer.encode(train_text, add_special_tokens=False)
train_ids = enc.ids
train_offsets = enc.offsets

# Re-create gold BIO labels using the same offset-overlap rule
gold_bio = ["O"] * len(train_offsets)

for span in train_gold:
    start, end, entity_type = int(span[0]), int(span[1]), str(span[2])
    covered = [
        i for i, (ts, te) in enumerate(train_offsets)
        if te > start and ts < end
    ]
    if covered:
        gold_bio[covered[0]] = f"B-{entity_type}"
        for i in covered[1:]:
            gold_bio[i] = f"I-{entity_type}"

# Predict on the first chunk
n = min(MAX_TOKENS, len(train_ids))
x = torch.tensor([train_ids[:n]], dtype=torch.long, device=device)

model4.eval()
with torch.no_grad():
    train_logits = model4(x)

train_pred_ids = train_logits.argmax(-1)[0].cpu().tolist()
gold_chunk = gold_bio[:n]

print("Gold tag counts:", Counter(gold_chunk))
print("Predicted tag counts:", Counter(ID2LABEL[i] for i in train_pred_ids))

print("\nFirst 30 gold/prediction comparisons:")
for i in range(min(30, n)):
    s, e = train_offsets[i]
    print(
        repr(train_text[s:e]),
        "| gold:", gold_chunk[i],
        "| predicted:", ID2LABEL[train_pred_ids[i]]
    )

In [ ]:
from sklearn.metrics import classification_report, f1_score

gold_ids = [LABEL2ID[tag] for tag in gold_chunk]
pred_ids_eval = [int(i) for i in train_pred_ids]

print("Token accuracy:", sum(
    g == p for g, p in zip(gold_ids, pred_ids_eval)
) / len(gold_ids))

print("\nClassification report:")
print(classification_report(
    gold_ids,
    pred_ids_eval,
    labels=list(range(len(BIO_LABELS))),
    target_names=BIO_LABELS,
    zero_division=0,
    digits=4
))

gold_entity = [i for i, tag in enumerate(gold_chunk) if tag != "O"]
pred_entity = [i for i, tag in enumerate(
    [ID2LABEL[p] for p in pred_ids_eval]
) if tag != "O"]

print("Gold entity-token count:", len(gold_entity))
print("Predicted entity-token count:", len(pred_entity))
print("Exact BIO token macro-F1:", f1_score(
    gold_ids,
    pred_ids_eval,
    labels=list(range(1, len(BIO_LABELS))),
    average="macro",
    zero_division=0
))

In [ ]:
print("Checkpoint metadata:")
for key, value in ckpt.items():
    if key != "model_state_dict" and key != "optimizer_state_dict":
        print(f"{key}: {value}")

state = ckpt["model_state_dict"]

print("\nClassifier weight shape:", tuple(state["classifier.weight"].shape))
print("Classifier bias shape:", tuple(state["classifier.bias"].shape))

print("\nClassifier bias by current BIO label:")
bias = state["classifier.bias"].detach().cpu()
for i, value in enumerate(bias):
    print(f"{i:2d} {ID2LABEL[i]:12s} {value.item(): .4f}")

print("\nCurrent model class ordering:")
for i, label in enumerate(BIO_LABELS):
    print(i, label)

In [ ]:
from collections import Counter
import torch

def make_gold_bio(text, spans, offsets):
    tags = ["O"] * len(offsets)
    for span in spans:
        start, end, label = int(span[0]), int(span[1]), str(span[2])
        covered = [
            i for i, (s, e) in enumerate(offsets)
            if e > start and s < end
        ]
        if covered:
            tags[covered[0]] = f"B-{label}"
            for i in covered[1:]:
                tags[i] = f"I-{label}"
    return tags

total_gold_entities = 0
total_pred_entities = 0
total_correct_tokens = 0
total_tokens = 0
predicted_counts = Counter()
gold_counts = Counter()

# Check up to 5 documents, first 768 tokens each
for doc_id in list(fold1_text.keys())[:5]:
    txt = fold1_text[doc_id]
    spans = fold1_gold.get(doc_id, fold1_gold.get(str(doc_id), []))
    encoded = tokenizer.encode(txt, add_special_tokens=False)
    n = min(768, len(encoded.ids))
    if n == 0:
        continue

    x = torch.tensor([encoded.ids[:n]], dtype=torch.long, device=device)
    with torch.no_grad():
        logits = model4(x)
    pred = [ID2LABEL[int(i)] for i in logits.argmax(-1)[0].cpu().tolist()]
    gold = make_gold_bio(txt, spans, encoded.offsets[:n])

    total_correct_tokens += sum(a == b for a, b in zip(gold, pred))
    total_tokens += n
    predicted_counts.update(pred)
    gold_counts.update(gold)

print("Documents checked:", min(5, len(fold1_text)))
print("Token accuracy:", round(total_correct_tokens / max(1, total_tokens), 4))
print("Gold BIO counts:", gold_counts)
print("Predicted BIO counts:", predicted_counts)
print("Gold entity tokens:", sum(v for k, v in gold_counts.items() if k != "O"))
print("Predicted entity tokens:", sum(v for k, v in predicted_counts.items() if k != "O"))

In [ ]:
import inspect
import torch

print("Encoder class:", type(model4.encoder).__name__)
print("Encoder layer class:", type(model4.encoder.encoder_layers[0]).__name__)
print("Attention class:", type(model4.encoder.encoder_layers[0].attn).__name__)
print("FFN class:", type(model4.encoder.encoder_layers[0].ffn).__name__)

print("\nEncoder forward:")
print(inspect.getsource(type(model4.encoder).__class__) if False else
      inspect.getsource(model4.encoder.encoder_layers[0].forward))

print("\nAttention forward:")
print(inspect.getsource(model4.encoder.encoder_layers[0].attn.forward))

print("\nLORMA module details:")
for name, module in model4.named_modules():
    if hasattr(module, "lorma_A") and hasattr(module, "lorma_B"):
        print("Module:", name)
        print("Rank:", tuple(module.lorma_A.shape), tuple(module.lorma_B.shape))
        print("Scale:", getattr(module, "alpha", "alpha attribute not found"))
        print("Dropout:", getattr(module, "dropout", "dropout attribute not found"))
        break

In [ ]:
import inspect
import torch

# Inspect the actual LORMA module implementation
lorma = model4.encoder.encoder_layers[0].attn.q_proj

print("LORMA module type:", type(lorma))
print("\nModule attributes:")
print(lorma)

print("\nStored parameters:")
for name, param in lorma.named_parameters():
    print(name, tuple(param.shape))

print("\nStored buffers:")
for name, buf in lorma.named_buffers():
    print(name, tuple(buf.shape))

print("\nModule configuration:")
for attr in [
    "rank", "r", "alpha", "scaling", "scale",
    "lora_alpha", "lorma_alpha", "dropout_p"
]:
    if hasattr(lorma, attr):
        print(attr, getattr(lorma, attr))

print("\nForward method:")
try:
    print(inspect.getsource(type(lorma).forward))
except (OSError, TypeError):
    print("Source unavailable; forward method:", lorma.forward)

In [ ]:
import torch
from collections import Counter

model4.eval()

# Use the first training document
doc_id = next(iter(fold1_text))
text = fold1_text[doc_id]
gold_spans = fold1_gold[doc_id]

# Tokenize the document
enc = tokenizer(
    text,
    add_special_tokens=False,
    return_offsets_mapping=True,
    truncation=False
)

input_ids = enc["input_ids"]
offsets = enc["offset_mapping"]

print("Document:", doc_id)
print("Tokens:", len(input_ids))
print("Gold annotations:", len(gold_spans))

# Inspect predictions on a short prefix
N = min(768, len(input_ids))
x = torch.tensor([input_ids[:N]], dtype=torch.long)

with torch.no_grad():
    logits = model4(x)
    probs = torch.softmax(logits, dim=-1)
    pred = logits.argmax(dim=-1)[0].tolist()

print("\nPredicted label counts:")
print(Counter(pred))

print("\nTop predicted labels:")
for label_id, count in Counter(pred).most_common(10):
    print(label_id, count)

print("\nAverage maximum probability:", round(probs.max(dim=-1).values.mean().item(), 4))
print("Mean probability assigned to O:", round(probs[0, :, 0].mean().item(), 4))

print("\nFirst 30 token predictions:")
for i in range(min(30, N)):
    s, e = offsets[i]
    print(
        repr(text[s:e]),
        "=>",
        pred[i],
        "confidence=",
        round(probs[0, i, pred[i]].item(), 3)
    )

In [ ]:
import torch
from collections import Counter

model4.eval()

# First training document
doc_id = next(iter(fold1_text))
text = fold1_text[doc_id]
gold_spans = fold1_gold[doc_id]

# Correct API for tokenizers.Tokenizer
enc = tokenizer.encode(text, add_special_tokens=False)

input_ids = enc.ids
offsets = enc.offsets

print("Document:", doc_id)
print("Tokens:", len(input_ids))
print("Gold annotations:", len(gold_spans))

# Predict on the first 768 tokens
N = min(768, len(input_ids))
x = torch.tensor([input_ids[:N]], dtype=torch.long)

with torch.no_grad():
    logits = model4(x)
    probs = torch.softmax(logits, dim=-1)
    pred = logits.argmax(dim=-1)[0].tolist()

print("\nPredicted label counts:")
print(Counter(pred))

print("\nTop predicted labels:")
for label_id, count in Counter(pred).most_common(10):
    print(label_id, count)

print(
    "\nAverage maximum probability:",
    round(probs.max(dim=-1).values.mean().item(), 4)
)
print(
    "Mean probability assigned to O:",
    round(probs[0, :, 0].mean().item(), 4)
)

print("\nFirst 30 token predictions:")
for i in range(min(30, N)):
    s, e = offsets[i]
    print(
        repr(text[s:e]),
        "=>", pred[i],
        "confidence=", round(probs[0, i, pred[i]].item(), 3)
    )

In [ ]:
from collections import Counter

# Convert character-level gold spans into token-level BIO labels.
# Expected gold span format: [start, end, label, ...]
id_to_bio = ["O"]
for label in labels:
    id_to_bio.extend([f"B-{label}", f"I-{label}"])

bio_to_id = {tag: i for i, tag in enumerate(id_to_bio)}

gold_bio = ["O"] * len(input_ids)

for span in gold_spans:
    start, end, label = span[:3]

    covered = [
        i for i, (ts, te) in enumerate(offsets)
        if te > start and ts < end
    ]

    if not covered:
        continue

    gold_bio[covered[0]] = f"B-{label}"
    for i in covered[1:]:
        gold_bio[i] = f"I-{label}"

gold_prefix = gold_bio[:N]
pred_bio = [id_to_bio[i] for i in pred]

print("Gold BIO counts:", Counter(gold_prefix))
print("\nPredicted BIO counts:", Counter(pred_bio))

gold_entities = sum(tag != "O" for tag in gold_prefix)
pred_entities = sum(tag != "O" for tag in pred_bio)
correct_entities = sum(
    g == p and g != "O"
    for g, p in zip(gold_prefix, pred_bio)
)

print("\nGold entity tokens:", gold_entities)
print("Predicted entity tokens:", pred_entities)
print("Correct entity-tagged tokens:", correct_entities)

print("\nGold vs predicted for the first 100 tokens:")
for i in range(min(100, N)):
    if gold_prefix[i] != "O" or pred_bio[i] != "O":
        s, e = offsets[i]
        print(
            repr(text[s:e]),
            "| gold:", gold_prefix[i],
            "| pred:", pred_bio[i]
        )

In [ ]:
import torch

print("Checkpoint keys:")
print(list(ckpt.keys()))

print("\nCheckpoint metadata:")
for key, value in ckpt.items():
    if key != "model_state_dict":
        if isinstance(value, dict):
            print(f"\n{key}:")
            for k, v in value.items():
                print(f"  {k}: {v}")
        else:
            print(f"{key}: {value}")

print("\nModel configuration attributes:")
for obj_name, obj in [
    ("model4", model4),
    ("encoder", model4.encoder),
    ("first_layer", model4.encoder.encoder_layers[0]),
    ("attention", model4.encoder.encoder_layers[0].attn),
]:
    print(f"\n{obj_name}:")
    for attr in [
        "num_labels", "num_heads", "head_dim",
        "vocab_size", "max_positions", "max_seq_len",
        "hidden_size", "d_model", "dropout",
    ]:
        if hasattr(obj, attr):
            print(f"  {attr} = {getattr(obj, attr)}")

In [ ]:
# Check which parameters are marked trainable in the reconstructed model
trainable = [
    (name, tuple(param.shape))
    for name, param in model4.named_parameters()
    if param.requires_grad
]

print("Number of trainable parameter tensors:", len(trainable))
for name, shape in trainable:
    print(f"{name:65s} {shape}")

print("\nCheckpoint optimizer state count:",
      len(ckpt["optimizer_state_dict"]["state"]))

print("\nClassifier shape:",
      tuple(model4.classifier.weight.shape),
      tuple(model4.classifier.bias.shape))

In [ ]:
import torch

phase1_path = (
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)

phase1_ckpt = torch.load(
    phase1_path,
    map_location="cpu",
    weights_only=False
)

print("Phase 1 checkpoint keys:", list(phase1_ckpt.keys()))

# Identify the model state dictionary
if "model_state_dict" in phase1_ckpt:
    phase1_state = phase1_ckpt["model_state_dict"]
elif "state_dict" in phase1_ckpt:
    phase1_state = phase1_ckpt["state_dict"]
else:
    phase1_state = phase1_ckpt

phase2_state = ckpt["model_state_dict"]

print("\nComparing matching parameter tensors:")
unchanged = []
changed = []
missing = []

for name, p2 in phase2_state.items():
    if name not in phase1_state:
        missing.append(name)
        continue

    p1 = phase1_state[name]

    if p1.shape != p2.shape:
        print("SHAPE MISMATCH:", name, tuple(p1.shape), tuple(p2.shape))
        continue

    same = torch.equal(p1.detach().cpu(), p2.detach().cpu())

    if same:
        unchanged.append(name)
    else:
        changed.append(name)

print("\nUnchanged tensors:", len(unchanged))
print("Changed tensors:", len(changed))
print("Missing from Phase 1:", len(missing))

print("\nChanged parameter names:")
for name in changed:
    print(name)

In [ ]:
phase1_state = phase1_ckpt["model_state_dict"]
phase2_state = ckpt["model_state_dict"]

print("Phase 1 tensor count:", len(phase1_state))
print("Phase 2 tensor count:", len(phase2_state))

print("\nFirst 35 Phase 1 keys:")
for name, value in list(phase1_state.items())[:35]:
    print(name, tuple(value.shape))

print("\nFirst 35 Phase 2 keys:")
for name, value in list(phase2_state.items())[:35]:
    print(name, tuple(value.shape))

print("\nPhase 1 key prefixes:")
from collections import Counter
print(Counter(name.split(".")[0] for name in phase1_state))

print("\nPhase 2 key prefixes:")
print(Counter(name.split(".")[0] for name in phase2_state))

In [ ]:
import torch

p1 = phase1_ckpt["model_state_dict"]
p2 = ckpt["model_state_dict"]

# Map Phase 1 encoder names to Phase 2 names
def phase2_name_for_phase1(name):
    if name.startswith("token_embedding."):
        return "encoder." + name
    if name.startswith("encoder_pos."):
        return "encoder." + name
    if name.startswith("encoder_layers."):
        return "encoder." + name
    if name.startswith("encoder_norm."):
        return "encoder." + name
    return None

matched = []
mismatched = []
missing = []

for name, w1 in p1.items():
    target = phase2_name_for_phase1(name)
    if target is None:
        continue

    # q_proj/v_proj weights are stored under .base in Phase 2
    if ".attn.q_proj.weight" in target:
        target = target.replace(".attn.q_proj.weight", ".attn.q_proj.base.weight")
    elif ".attn.q_proj.bias" in target:
        target = target.replace(".attn.q_proj.bias", ".attn.q_proj.base.bias")
    elif ".attn.v_proj.weight" in target:
        target = target.replace(".attn.v_proj.weight", ".attn.v_proj.base.weight")
    elif ".attn.v_proj.bias" in target:
        target = target.replace(".attn.v_proj.bias", ".attn.v_proj.base.bias")

    if target not in p2:
        missing.append((name, target))
        continue

    w2 = p2[target].detach().cpu()
    w1 = w1.detach().cpu()

    if w1.shape != w2.shape:
        mismatched.append((name, target, tuple(w1.shape), tuple(w2.shape)))
    else:
        max_diff = (w1.float() - w2.float()).abs().max().item()
        matched.append((name, target, max_diff))

print("Matched tensors:", len(matched))
print("Missing tensors:", len(missing))
print("Shape mismatches:", len(mismatched))

if matched:
    exact = sum(diff == 0 for _, _, diff in matched)
    print("Exactly equal tensors:", exact)
    print("Different tensors:", len(matched) - exact)
    print("Maximum absolute difference:", max(diff for _, _, diff in matched))

print("\nFirst 20 differences:")
for source, target, diff in matched:
    if diff > 0:
        print(f"{source} -> {target}: max_abs_diff={diff:.6g}")
        if sum(1 for _, _, d in matched if d > 0) > 20:
            break

print("\nFirst 10 missing mappings:")
for item in missing[:10]:
    print(item)

In [ ]:
import torch

p1 = phase1_ckpt["model_state_dict"]
p2 = ckpt["model_state_dict"]

def map_phase1_to_phase2(name):
    if name.startswith(("token_embedding.", "encoder_pos.", "encoder_layers.")):
        target = "encoder." + name
    else:
        return None

    for proj in ("q_proj", "v_proj"):
        target = target.replace(
            f".attn.{proj}.weight",
            f".attn.{proj}.base.weight"
        )
        target = target.replace(
            f".attn.{proj}.bias",
            f".attn.{proj}.base.bias"
        )
    return target

groups = {}

for name, w1 in p1.items():
    target = map_phase1_to_phase2(name)
    if target is None or target not in p2:
        continue

    w2 = p2[target].detach().cpu()
    w1 = w1.detach().cpu()

    if w1.shape != w2.shape:
        continue

    diff = (w1.float() - w2.float()).abs()
    group = name.split(".")[0]

    groups.setdefault(group, []).append({
        "name": name,
        "mean": diff.mean().item(),
        "max": diff.max().item(),
        "exact": torch.equal(w1, w2)
    })

for group, entries in groups.items():
    print(f"\n{group}: {len(entries)} tensors")
    print("  Exact matches:", sum(x["exact"] for x in entries))
    print("  Mean of tensor-level mean differences:",
          sum(x["mean"] for x in entries) / len(entries))
    print("  Largest absolute difference:",
          max(x["max"] for x in entries))

print("\nLargest individual differences:")
all_entries = [x for entries in groups.values() for x in entries]
for item in sorted(all_entries, key=lambda x: x["mean"], reverse=True)[:15]:
    print(
        item["name"],
        "mean_abs_diff=", round(item["mean"], 7),
        "max_abs_diff=", round(item["max"], 7)
    )

In [ ]:
from collections import Counter
import torch

doc_id = next(iter(fold1_text))
text = fold1_text[doc_id]
gold_spans = fold1_gold[doc_id]

enc = tokenizer.encode(text, add_special_tokens=False)
ids = enc.ids
offsets = enc.offsets

# Use the same first 768 tokens as the earlier diagnostic
n = min(768, len(ids))
input_ids = torch.tensor([ids[:n]], dtype=torch.long)

model4.eval()
with torch.no_grad():
    logits = model4(input_ids)
    probs = torch.softmax(logits, dim=-1)
    pred = logits.argmax(dim=-1)[0].cpu().tolist()

pred_counts = Counter(pred)
confidence = probs.max(dim=-1).values[0].cpu()

print("Document:", doc_id)
print("Tokens examined:", n)
print("Predicted label counts:", pred_counts)
print("Mean maximum probability:", round(confidence.mean().item(), 4))
print("Mean probability of O:", round(probs[0, :, 0].mean().item(), 4))

# Show the highest-confidence non-O predictions
print("\nNon-O predictions:")
shown = 0
for i, label_id in enumerate(pred):
    if label_id == 0:
        continue
    start, end = offsets[i]
    print({
        "token_index": i,
        "label_id": label_id,
        "label": (
            "O" if label_id == 0 else
            f"{'B' if label_id % 2 else 'I'}-"
            f"{labels[(label_id - 1) // 2]}"
        ),
        "text": text[start:end],
        "confidence": round(confidence[i].item(), 4)
    })
    shown += 1
    if shown >= 20:
        break

In [ ]:
# Inspect the Phase 2 checkpoint's classifier
print("Classifier shape:", ckpt["model_state_dict"]["classifier.weight"].shape)
print("Number of labels:", len(labels) * 2 + 1)

print("\nExpected label mapping:")
id_to_label = ["O"]
for label in labels:
    id_to_label.extend([f"B-{label}", f"I-{label}"])

for i, label in enumerate(id_to_label):
    print(i, "->", label)

print("\nClassifier bias statistics:")
bias = ckpt["model_state_dict"]["classifier.bias"].detach().cpu()
for i, label in enumerate(id_to_label):
    print(f"{i:2d} {label:12s} bias={bias[i].item(): .4f}")

print("\nTop classifier biases:")
for i in torch.argsort(bias, descending=True)[:10].tolist():
    print(id_to_label[i], round(bias[i].item(), 4))

In [ ]:
print("Checkpoint metadata:")
for key in ("experiment", "epoch", "train_loss", "train_chunks", "val_chunks"):
    print(f"{key}: {ckpt.get(key, 'not stored')}")

print("\nClassifier weight shape:",
      tuple(ckpt["model_state_dict"]["classifier.weight"].shape))

print("Classifier bias shape:",
      tuple(ckpt["model_state_dict"]["classifier.bias"].shape))

print("Model output shape:", tuple(logits.shape))
print("Predicted IDs within range:",
      min(pred) >= 0 and max(pred) < len(id_to_label))

In [ ]:
from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input/datasets/aq1de3"),
]

keywords = [
    "LORMA",
    "lorma_A",
    "CrossEntropyLoss",
    "B-APP",
    "build_chunks_for_document",
    "A4NER",
]

for root in roots:
    print(f"\n=== Searching {root} ===")
    if not root.exists():
        continue

    found = 0
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix not in {".py", ".ipynb", ".md", ".txt"}:
            continue

        try:
            if path.stat().st_size > 20_000_000:
                continue
            content = path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue

        hits = [word for word in keywords if word.lower() in content.lower()]
        if hits:
            print(path, "->", ", ".join(hits))
            found += 1
            if found >= 30:
                print("(showing first 30 matching files)")
                break

In [ ]:
import json
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
notebook = json.loads(path.read_text(encoding="utf-8"))

keywords = [
    "class A4NER",
    "class A4EncoderForNER",
    "class LORMALinear",
    "def forward",
    "CrossEntropyLoss",
    "ENTITY_LABELS",
    "BIO_LABELS",
    "build_chunks_for_document",
    "lorma_A",
    "optimizer",
]

for i, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    matched = [k for k in keywords if k.lower() in source.lower()]

    if matched:
        print(f"\n{'=' * 25} CELL {i} {'=' * 25}")
        print("Matched:", ", ".join(matched))
        print(source[:7000])
        

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")

print("Exists:", path.exists())

if path.exists():
    print("Size:", path.stat().st_size, "bytes")
    print("First 500 characters:")
    print(repr(path.read_text(encoding="utf-8", errors="replace")[:500]))

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
source = path.read_text(encoding="utf-8")

keywords = [
    "class A4NER",
    "class A4EncoderForNER",
    "class LORMALinear",
    "CrossEntropyLoss",
    "ENTITY_LABELS",
    "BIO_LABELS",
    "build_chunks_for_document",
    "lorma_A",
]

lines = source.splitlines()
matches = []

for i, line in enumerate(lines):
    if any(k.lower() in line.lower() for k in keywords):
        matches.append(i)

# Merge nearby matches to avoid printing the same code repeatedly
ranges = []
for i in matches:
    start, end = max(0, i - 5), min(len(lines), i + 35)
    if ranges and start <= ranges[-1][1]:
        ranges[-1] = (ranges[-1][0], max(ranges[-1][1], end))
    else:
        ranges.append((start, end))

for start, end in ranges:
    print(f"\n{'=' * 20} LINES {start + 1}-{end} {'=' * 20}")
    print("\n".join(lines[start:end]))

In [ ]:
from collections import Counter

# Mapping A: interleaved B/I labels
interleaved_labels = ["O"]
for label in labels:
    interleaved_labels.extend([f"B-{label}", f"I-{label}"])

# Mapping B: all B labels, then all I labels
grouped_labels = (
    ["O"]
    + [f"B-{label}" for label in labels]
    + [f"I-{label}" for label in labels]
)

def make_gold_bio_ids(text, spans, offsets, label_order):
    label_to_id = {label: i for i, label in enumerate(label_order)}
    tags = ["O"] * len(offsets)

    for span in spans:
        start, end, entity = int(span[0]), int(span[1]), str(span[2])
        covered = [
            i for i, (s, e) in enumerate(offsets)
            if e > start and s < end
        ]
        if covered:
            tags[covered[0]] = f"B-{entity}"
            for i in covered[1:]:
                tags[i] = f"I-{entity}"

    return [label_to_id[tag] for tag in tags]

n = len(pred)
gold_interleaved = make_gold_bio_ids(
    text, gold_spans, offsets[:n], interleaved_labels
)
gold_grouped = make_gold_bio_ids(
    text, gold_spans, offsets[:n], grouped_labels
)

print("Tokens examined:", n)
print("Gold entity tokens:", sum(x != 0 for x in gold_grouped))
print("Predicted non-O tokens:", sum(x != 0 for x in pred))

print("\nToken accuracy against gold BIO IDs:")
print("Interleaved mapping:", round(
    sum(p == g for p, g in zip(pred, gold_interleaved)) / n, 4
))
print("Grouped mapping:", round(
    sum(p == g for p, g in zip(pred, gold_grouped)) / n, 4
))

print("\nPredictions interpreted with grouped mapping:")
for i, pred_id in enumerate(pred):
    if pred_id != 0:
        s, e = offsets[i]
        print(
            f"token={i:3d} | {grouped_labels[pred_id]:12s} "
            f"| {text[s:e]!r}"
        )

In [ ]:
# CELL — VERIFY THE TRAINING LABEL ORDER

for name in [
    "BIO_LABELS", "bio_labels", "ID2LABEL",
    "LABEL2ID", "ENTITY_LABELS"
]:
    if name in globals():
        value = globals()[name]
        print(f"\n{name}:")
        print(value)

print("\nCheckpoint metadata:")
for key, value in ckpt.items():
    if any(word in key.lower() for word in ["label", "class", "experiment", "epoch"]):
        if not isinstance(value, dict):
            print(key, "=", value)

print("\nClassifier output dimension:",
      ckpt["model_state_dict"]["classifier.weight"].shape[0])

print("\nCurrent model label count:", len(BIO_LABELS))

In [ ]:
from collections import Counter
import torch

doc_id = next(iter(fold1_text))
text = fold1_text[doc_id]
gold_spans = fold1_gold[doc_id]

enc = tokenizer.encode(text, add_special_tokens=False)
ids = enc.ids
offsets = enc.offsets

# Evaluate the first 768 tokens
n = min(768, len(ids))
input_ids = torch.tensor([ids[:n]], dtype=torch.long)

model4.eval()
with torch.no_grad():
    logits = model4(input_ids)
    pred = logits.argmax(dim=-1)[0].cpu().tolist()

# Construct gold BIO labels using the confirmed interleaved mapping
gold_tags = ["O"] * n

for span in gold_spans:
    start, end, entity = int(span[0]), int(span[1]), str(span[2])
    covered = [
        i for i, (s, e) in enumerate(offsets[:n])
        if e > start and s < end
    ]
    if covered:
        gold_tags[covered[0]] = f"B-{entity}"
        for i in covered[1:]:
            gold_tags[i] = f"I-{entity}"

gold_ids = [LABEL2ID[tag] for tag in gold_tags]

# Token-level metrics on entity tags only
gold_entity = {i for i, x in enumerate(gold_ids) if x != 0}
pred_entity = {i for i, x in enumerate(pred) if x != 0}
correct_entity_tags = sum(
    pred[i] == gold_ids[i] and gold_ids[i] != 0
    for i in range(n)
)

print("Document:", doc_id)
print("Tokens evaluated:", n)
print("Gold entity tokens:", len(gold_entity))
print("Predicted entity tokens:", len(pred_entity))
print("Correct entity-tagged tokens:", correct_entity_tags)

print("\nExact token-tag precision:",
      round(correct_entity_tags / max(1, len(pred_entity)), 4))
print("Exact token-tag recall:",
      round(correct_entity_tags / max(1, len(gold_entity)), 4))

print("\nGold entity tag counts:",
      Counter(gold_tags).most_common(15))
print("Predicted tag counts:",
      Counter(BIO_LABELS[x] for x in pred).most_common(15))

In [ ]:
import torch

p1 = phase1_ckpt["model_state_dict"]
p2 = ckpt["model_state_dict"]

def phase2_name_for_phase1(name):
    if name.startswith(("token_embedding.", "encoder_pos.", "encoder_layers.", "encoder_norm.")):
        target = "encoder." + name
    else:
        return None

    # Phase 2 wraps q_proj and v_proj in LORMA modules.
    for projection in ("q_proj", "v_proj"):
        target = target.replace(
            f".attn.{projection}.weight",
            f".attn.{projection}.base.weight"
        )
        target = target.replace(
            f".attn.{projection}.bias",
            f".attn.{projection}.base.bias"
        )

    return target

matched = []
missing = []
shape_mismatch = []

for name, w1 in p1.items():
    target = phase2_name_for_phase1(name)
    if target is None:
        continue

    if target not in p2:
        missing.append((name, target))
        continue

    w2 = p2[target].detach().cpu()
    w1 = w1.detach().cpu()

    if w1.shape != w2.shape:
        shape_mismatch.append((name, tuple(w1.shape), tuple(w2.shape)))
        continue

    max_diff = (w1.float() - w2.float()).abs().max().item()
    matched.append((name, max_diff))

exact = sum(diff == 0 for _, diff in matched)

print("Comparable tensors:", len(matched))
print("Exactly identical tensors:", exact)
print("Different tensors:", len(matched) - exact)
print("Missing target tensors:", len(missing))
print("Shape mismatches:", len(shape_mismatch))
print("\nFirst 15 differences:")
for name, diff in [(n, d) for n, d in matched if d != 0][:15]:
    print(name, "max_abs_diff =", diff)

print("\nFirst 10 missing mappings:", missing[:10])

In [ ]:
opt = ckpt["optimizer_state_dict"]

print("Optimizer type/config:")
print("Parameter groups:", len(opt["param_groups"]))
print("Optimizer state entries:", len(opt["state"]))

for i, group in enumerate(opt["param_groups"]):
    print(f"\nGroup {i}")
    print("Number of parameter IDs:", len(group["params"]))
    for key in ["lr", "weight_decay", "betas", "eps"]:
        if key in group:
            print(f"{key}: {group[key]}")

print("\nTrainable parameter counts in the current model:")
trainable = [
    (name, param.numel())
    for name, param in model4.named_parameters()
    if param.requires_grad
]
print("Trainable tensors:", len(trainable))
print("Trainable parameters:", sum(n for _, n in trainable))

print("\nParameters that appear to be adapters or classifier:")
for name, n in trainable:
    if "lorma_" in name or "classifier" in name:
        print(name, n)

In [ ]:
import torch

state = ckpt["model_state_dict"]
opt_state = ckpt["optimizer_state_dict"]["state"]

print("Optimizer state entries:", len(opt_state))
print("LORMA tensors:", sum("lorma_" in k for k in state))
print("Classifier tensors:", sum(k.startswith("classifier.") for k in state))

print("\nAdapter statistics:")
for name, tensor in state.items():
    if "lorma_" in name:
        t = tensor.detach().float()
        print(
            name,
            "| shape:", tuple(t.shape),
            "| std:", round(t.std().item(), 6),
            "| abs mean:", round(t.abs().mean().item(), 6),
            "| nonzero:", torch.count_nonzero(t).item()
        )

print("\nClassifier statistics:")
for name in ["classifier.weight", "classifier.bias"]:
    t = state[name].detach().float()
    print(
        name,
        "| shape:", tuple(t.shape),
        "| std:", round(t.std().item(), 6),
        "| abs mean:", round(t.abs().mean().item(), 6)
    )

In [ ]:
import inspect

keywords = [
    "A4Attention",
    "A4EncoderForNER",
    "A4NER",
    "LORMALinear",
    "predict",
    "decode",
    "validation",
    "evaluate",
    "build_chunks_for_document",
]

for name, obj in list(globals().items()):
    if not any(k.lower() in name.lower() for k in keywords):
        continue

    if inspect.isfunction(obj) or inspect.isclass(obj):
        try:
            source = inspect.getsource(obj)
            print(f"\n{'='*30}\n{name}\n{'='*30}")
            print(source[:6000])
        except (OSError, TypeError):
            print(f"\n{name}: source unavailable")

In [ ]:
import json
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")

print("Notebook source exists:", path.exists())

if not path.exists():
    print("Source file unavailable at this path.")
else:
    nb = json.loads(path.read_text(encoding="utf-8"))

    keywords = [
        "class A4EncoderForNER",
        "class A4NER",
        "class LORMALinear",
        "model4 =",
        "model4.load_state_dict",
        "torch.save",
        "optimizer4",
        "lorma_encoder",
        "train_loss",
        "encoder_norm",
    ]

    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        hits = [k for k in keywords if k.lower() in source.lower()]

        if hits:
            print(f"\n{'='*25} CELL {i} {'='*25}")
            print("Matches:", ", ".join(hits))
            print(source[:9000])

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")

print("Exists:", path.exists())

if path.exists():
    print("Size:", path.stat().st_size, "bytes")
    raw = path.read_text(encoding="utf-8", errors="replace")
    print("First 500 characters:")
    print(repr(raw[:500]))
else:
    print("Notebook source file not found.")

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
source = path.read_text(encoding="utf-8", errors="replace")
lines = source.splitlines()

keywords = [
    "class LORMALinear",
    "class A4EncoderForNER",
    "class A4NER",
    "model4 = A4NER",
    "model4.load_state_dict",
    "torch.save",
    "encoder_norm",
    "CELL 43",
]

matches = [
    i for i, line in enumerate(lines)
    if any(k.lower() in line.lower() for k in keywords)
]

print("Source lines:", len(lines))
print("Matching locations:", len(matches))

# Print each matching location with surrounding context
printed_until = -1

for i in matches:
    start = max(0, i - 8)
    end = min(len(lines), i + 45)

    if start <= printed_until:
        continue

    print(f"\n{'='*25} LINES {start+1}-{end} {'='*25}")
    print("\n".join(lines[start:end]))
    printed_until = end

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

ranges = [
    (8120, 8210),    # Original encoder definition and setup
    (13880, 14010),  # Experiment 3 encoder/checkpoint setup
    (14800, 14970),  # Experiment 4 LORMA construction
    (15000, 15160),  # Repeated Experiment 4 construction
]

for start, end in ranges:
    print(f"\n{'='*24} SOURCE LINES {start}-{end} {'='*24}")
    print("\n".join(
        f"{i+1}: {lines[i]}"
        for i in range(start - 1, min(end, len(lines)))
    ))

print("\n--- Encoder initialization references ---")
for i, line in enumerate(lines):
    if any(term in line.lower() for term in [
        "encoder.load_state_dict",
        "encoder_state",
        "load_state_dict(",
        "copy.deepcopy(encoder)",
    ]):
        if i < 16000:
            print(f"{i+1}: {line}")

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

ranges = [
    (9880, 9940),
    (10120, 10180),
    (10320, 10475),
    (13570, 13640),
    (14020, 14085),
    (14180, 14245),
    (14600, 14675),
]

for start, end in ranges:
    print(f"\n{'='*20} LINES {start}-{end} {'='*20}")
    print("\n".join(
        f"{i+1}: {lines[i]}"
        for i in range(start - 1, min(end, len(lines)))
    ))

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

for start, end in [(15540, 15680), (15680, 15820)]:
    print(f"\n{'='*20} LINES {start}-{end} {'='*20}")
    print("\n".join(
        f"{i+1}: {lines[i]}"
        for i in range(start - 1, min(end, len(lines)))
    ))

In [ ]:
import inspect
import torch

print("=" * 70)
print("LORMA VALIDATION PIPELINE AUDIT")
print("=" * 70)

# 1. Check the actual decoding functions
for name in [
    "bio_to_token_spans",
    "normalize_span_boundaries",
    "strict_macro_f1",
]:
    fn = globals().get(name)
    print(f"\n--- {name} ---")
    if callable(fn):
        try:
            print(inspect.getsource(fn))
        except (OSError, TypeError):
            print("Source unavailable; callable exists:", fn)
    else:
        print("FUNCTION NOT FOUND")

# 2. Check the active label mapping
print("\n--- BIO LABEL MAPPING ---")
print("BIO_LABELS:", BIO_LABELS)
print("ID2LABEL:", ID2LABEL)

assert len(BIO_LABELS) == 25
assert BIO_LABELS[0] == "O"

# 3. Check validation batch fields and shapes
batch = next(iter(a4_val_loader))
print("\n--- VALIDATION BATCH ---")
for key, value in batch.items():
    if torch.is_tensor(value):
        print(key, tuple(value.shape), value.dtype)
    elif isinstance(value, list):
        print(key, "list length =", len(value))
        if value:
            print(" first item type =", type(value[0]).__name__)
            print(" first item preview =", repr(value[0])[:250])
    else:
        print(key, type(value).__name__)

# 4. Check whether offsets and gold spans are document-relative
print("\n--- OFFSET / GOLD SPAN CHECK ---")
doc_id = batch["doc_id"][0]
offsets = batch["offsets"][0]
gold_spans = batch["gold_spans"][0]
text = fold1_text[doc_id]

print("Document ID:", doc_id)
print("Document character length:", len(text))
print("Chunk token count:", len(offsets))
print("Gold spans in this chunk:", len(gold_spans))
print("First offsets:", offsets[:8])
print("First gold spans:", gold_spans[:5])

for span in gold_spans[:5]:
    start, end, label = int(span[0]), int(span[1]), str(span[2])
    print(
        repr(text[start:end]),
        "| span =", (start, end, label),
        "| valid bounds =", 0 <= start < end <= len(text)
    )

print("\nAudit complete. No training or submission files changed.")

In [ ]:

import inspect
import torch

print("=" * 70)
print("LORMA PIPELINE AUDIT — SAFE MODE")
print("=" * 70)

# 1. Inspect decoding and metric functions
for name in [
    "bio_to_token_spans",
    "normalize_span_boundaries",
    "strict_macro_f1",
]:
    fn = globals().get(name)
    print(f"\n--- {name} ---")

    if callable(fn):
        try:
            print(inspect.getsource(fn))
        except (OSError, TypeError):
            print("Function exists, but source is unavailable.")
    else:
        print("Not defined in this session.")

# 2. Inspect label mapping
print("\n--- LABEL MAPPING ---")

if "BIO_LABELS" in globals():
    print("BIO_LABELS:", BIO_LABELS)
    assert len(BIO_LABELS) == 25, "Unexpected number of BIO labels"
    assert BIO_LABELS[0] == "O"

if "ID2LABEL" in globals():
    print("ID2LABEL:", ID2LABEL)

# 3. Find available validation objects without assuming their names
print("\n--- AVAILABLE VALIDATION OBJECTS ---")

for name in [
    "a4_val_loader",
    "a4_val_chunks",
    "a4_val_dataset",
    "a4_eval_collate_fn",
    "fold1_text",
    "fold1_gold",
    "model4",
    "checkpoint",
    "ckpt",
]:
    print(f"{name}: {'available' if name in globals() else 'not available'}")

# 4. Inspect checkpoint metadata if available
print("\n--- LORMA CHECKPOINT METADATA ---")

for name in ["checkpoint", "ckpt"]:
    obj = globals().get(name)
    if isinstance(obj, dict) and "model_state_dict" in obj:
        print("Checkpoint variable:", name)
        for key in [
            "experiment", "epoch", "train_loss",
            "train_chunks", "val_chunks",
            "lorma_rank", "lorma_alpha", "lorma_dropout",
        ]:
            if key in obj:
                print(f"{key}: {obj[key]}")
        break
else:
    print("No loaded LORMA checkpoint dictionary found.")

print("\nSafe audit complete. No training or submission files changed.")


In [ ]:

from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

ranges = [
    (10677, 10775),  # BIO decoder
    (11208, 11226),  # Boundary normalization
    (14350, 14615),  # Validation chunks / collate / loader
]

for start, end in ranges:
    print(f"\n{'='*20} SOURCE LINES {start}-{end} {'='*20}")
    print("\n".join(
        f"{i+1}: {lines[i]}"
        for i in range(start - 1, min(end, len(lines)))
    ))

# Locate the strict metric definition
print("\n--- STRICT METRIC DEFINITION LOCATIONS ---")
for i, line in enumerate(lines):
    if "def strict_macro_f1" in line:
        print(f"Definition at source line {i+1}")
        lo = max(0, i - 3)
        hi = min(len(lines), i + 65)
        print("\n".join(
            f"{j+1}: {lines[j]}" for j in range(lo, hi)
        ))

print("\nInspection only — no training or inference performed.")


In [ ]:

from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

terms = [
    "class NERChunk",
    "def build_chunks_for_document",
    "a4_val_chunks =",
    "a4_train_only_chunks =",
    "MAX_TOKENS =",
    "OVERLAP =",
    "STRIDE =",
]

for term in terms:
    hits = [
        i for i, line in enumerate(lines)
        if term.lower() in line.lower()
    ]

    print(f"\n{'='*15} {term}: {len(hits)} matches {'='*15}")

    # Show the first few definitions, avoiding huge output.
    for i in hits[:3]:
        lo = max(0, i - 4)
        hi = min(len(lines), i + 35)
        print(f"\n--- Source lines {lo+1}-{hi} ---")
        print("\n".join(
            f"{j+1}: {lines[j]}" for j in range(lo, hi)
        ))

print("\nInspection only — no training or inference performed.")


In [ ]:

from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

for start, end in [
    (7509, 7650),    # Phase 2 NERChunk and chunk construction
    (12180, 12230),  # Fold split
    (10794, 10885),  # Original strict span metric
]:
    print(f"\n{'='*18} SOURCE LINES {start}-{end} {'='*18}")
    print("\n".join(
        f"{i+1}: {lines[i]}"
        for i in range(start - 1, min(end, len(lines)))
    ))

print("\nInspection only — no training or inference performed.")


In [ ]:
from pathlib import Path
import torch

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

# 1. Finish reading the original chunk builder and training setup.
for start, end in [
    (7645, 7715),
    (13880, 14010),
    (14820, 14970),
    (15000, 15160),
]:
    print(f"\n{'='*20} SOURCE LINES {start}-{end} {'='*20}")
    print("\n".join(
        f"{i+1}: {lines[i]}"
        for i in range(start - 1, min(end, len(lines)))
    ))

# 2. Find every MAX_TOKENS assignment and chunk-builder call.
print("\n" + "="*20 + " CONFIGURATION REFERENCES " + "="*20)
for i, line in enumerate(lines):
    if (
        "MAX_TOKENS =" in line
        or "a4_train_chunks =" in line
        or "build_document_chunks(" in line
    ):
        print(f"{i+1}: {line.strip()}")

# 3. Inspect the live runtime, not just the saved source.
print("\n" + "="*20 + " LIVE RUNTIME " + "="*20)
print("MAX_TOKENS:", globals().get("MAX_TOKENS", "<undefined>"))
print("OVERLAP:", globals().get("OVERLAP", "<undefined>"))
print("STRIDE:", globals().get("STRIDE", "<undefined>"))

if "model4" in globals():
    pos = model4.encoder.encoder_pos.weight
    print("Model positional embedding shape:", tuple(pos.shape))
    print("Model positional capacity:", pos.shape[0])

if "ckpt" in globals():
    state = ckpt.get("model_state_dict", {})
    for key in [
        "encoder.encoder_pos.weight",
        "encoder_pos.weight",
    ]:
        if key in state:
            print("Checkpoint positional embedding:", key,
                  tuple(state[key].shape))

# 4. Check existing chunks, if they are still in memory.
for name in ["a4_train_chunks", "a4_val_chunks"]:
    chunks = globals().get(name)
    if chunks is not None:
        lengths = [len(c.input_ids) for c in chunks]
        print(
            f"{name}: count={len(chunks)}, "
            f"max_length={max(lengths, default=0)}, "
            f"min_length={min(lengths, default=0)}"
        )

print("\nInspection complete; no training or inference performed.")

In [ ]:
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

for start, end in [
    (9100, 9365),    # A4-specific chunk configuration and construction
    (7720, 7770),    # Earlier builder call sites
    (27340, 27490),  # Later reconstruction builder
    (15040, 15220),  # LORMA model construction
]:
    print(f"\n{'='*18} SOURCE LINES {start}-{end} {'='*18}")
    print("\n".join(
        f"{i+1}: {lines[i]}"
        for i in range(start - 1, min(end, len(lines)))
    ))

print("\n" + "="*18 + " CURRENT CHUNK OBJECTS " + "="*18)
for name in ["a4_train_chunks", "a4_val_chunks", "a4_train_only_chunks"]:
    chunks = globals().get(name)
    if chunks is None:
        print(f"{name}: not present")
        continue

    lengths = [len(c.input_ids) for c in chunks]
    print(
        f"{name}: count={len(chunks)}, "
        f"max_length={max(lengths, default=0)}, "
        f"min_length={min(lengths, default=0)}"
    )

print("\nInspection only — no training or inference performed.")

In [ ]:
# Rebuild A4 validation chunks using the original A4 configuration.
# No training or inference.

A4_MAX_TOKENS = 1024
A4_OVERLAP = 256
A4_STRIDE = A4_MAX_TOKENS - A4_OVERLAP

# Reuse the notebook's NERChunk, tokenizer, normalize_gold,
# and bio2id definitions.
assert A4_STRIDE == 768
assert model4.encoder.encoder_pos.weight.shape[0] == A4_MAX_TOKENS

def build_a4_validation_chunks(doc_id, text, gold_list):
    token_ids, token_offsets = tokenize_document(text)
    gold_spans = normalize_gold(gold_list)

    chunks = []
    token_start = 0
    chunk_id = 0
    n_tokens = len(token_ids)

    while token_start < n_tokens:
        token_end = min(token_start + A4_MAX_TOKENS, n_tokens)
        ids = token_ids[token_start:token_end]
        offsets = token_offsets[token_start:token_end]

        valid_offsets = [(s, e) for s, e in offsets if e > s]
        if valid_offsets:
            char_start = min(s for s, e in valid_offsets)
            char_end = max(e for s, e in valid_offsets)
        else:
            char_start = char_end = 0

        chunk_gold = [
            (s, e, lab)
            for s, e, lab in gold_spans
            if s >= char_start and e <= char_end
        ]

        bio_ids = [bio2id["O"]] * len(ids)

        for ent_start, ent_end, label in chunk_gold:
            covered = [
                i for i, (ts, te) in enumerate(offsets)
                if te > ts and ts < ent_end and te > ent_start
            ]
            if not covered:
                continue

            bio_ids[covered[0]] = bio2id["B-" + label]
            for i in covered[1:]:
                bio_ids[i] = bio2id["I-" + label]

        chunks.append(NERChunk(
            doc_id=str(doc_id),
            chunk_id=chunk_id,
            input_ids=list(ids),
            offsets=list(offsets),
            bio_ids=bio_ids,
            gold_spans=chunk_gold,
            char_start=char_start,
            char_end=char_end,
        ))

        chunk_id += 1
        if token_end >= n_tokens:
            break
        token_start += A4_STRIDE

    return chunks

a4_val_chunks = []
for doc_id, text in fold1_text.items():
    a4_val_chunks.extend(
        build_a4_validation_chunks(
            doc_id, text, fold1_gold[doc_id]
        )
    )

lengths = [len(c.input_ids) for c in a4_val_chunks]

print("Fold 1 validation documents:", len(fold1_text))
print("Fold 1 validation chunks:", len(a4_val_chunks))
print("Maximum chunk length:", max(lengths, default=0))
print("Minimum chunk length:", min(lengths, default=0))
print("All chunk lengths within model capacity:",
      all(0 < n <= A4_MAX_TOKENS for n in lengths))
print("Input/offset/label lengths match:",
      all(
          len(c.input_ids) == len(c.offsets) == len(c.bio_ids)
          for c in a4_val_chunks
      ))

assert a4_val_chunks
assert max(lengths) <= A4_MAX_TOKENS
assert all(
    len(c.input_ids) == len(c.offsets) == len(c.bio_ids)
    for c in a4_val_chunks
)

In [ ]:
# Rebuild Fold 1 validation chunks — self-contained tokenizer call.
# No training or inference.

A4_MAX_TOKENS = 1024
A4_OVERLAP = 256
A4_STRIDE = A4_MAX_TOKENS - A4_OVERLAP

assert model4.encoder.encoder_pos.weight.shape[0] == A4_MAX_TOKENS

def build_a4_validation_chunks(doc_id, text, gold_list):
    enc = tokenizer.encode(text)
    token_ids = enc.ids
    token_offsets = enc.offsets
    gold_spans = normalize_gold(gold_list)

    chunks = []
    token_start = 0
    chunk_id = 0
    n_tokens = len(token_ids)

    while token_start < n_tokens:
        token_end = min(token_start + A4_MAX_TOKENS, n_tokens)
        ids = token_ids[token_start:token_end]
        offsets = token_offsets[token_start:token_end]

        valid_offsets = [(s, e) for s, e in offsets if e > s]
        if valid_offsets:
            char_start = min(s for s, e in valid_offsets)
            char_end = max(e for s, e in valid_offsets)
        else:
            char_start = char_end = 0

        chunk_gold = [
            (s, e, label)
            for s, e, label in gold_spans
            if s >= char_start and e <= char_end
        ]

        bio_ids = [bio2id["O"]] * len(ids)

        for ent_start, ent_end, label in chunk_gold:
            covered = [
                i for i, (ts, te) in enumerate(offsets)
                if te > ts and ts < ent_end and te > ent_start
            ]
            if not covered:
                continue

            bio_ids[covered[0]] = bio2id["B-" + label]
            for i in covered[1:]:
                bio_ids[i] = bio2id["I-" + label]

        chunks.append(NERChunk(
            doc_id=str(doc_id),
            chunk_id=chunk_id,
            input_ids=list(ids),
            offsets=list(offsets),
            bio_ids=bio_ids,
            gold_spans=chunk_gold,
            char_start=char_start,
            char_end=char_end,
        ))

        chunk_id += 1
        if token_end >= n_tokens:
            break
        token_start += A4_STRIDE

    return chunks

a4_val_chunks = []
for doc_id, text in fold1_text.items():
    a4_val_chunks.extend(
        build_a4_validation_chunks(doc_id, text, fold1_gold[doc_id])
    )

lengths = [len(c.input_ids) for c in a4_val_chunks]

print("Fold 1 documents:", len(fold1_text))
print("Validation chunks:", len(a4_val_chunks))
print("Maximum chunk length:", max(lengths, default=0))
print("Minimum chunk length:", min(lengths, default=0))
print("All chunks fit model:", all(0 < n <= A4_MAX_TOKENS for n in lengths))
print("Input/offset/label lengths match:",
      all(len(c.input_ids) == len(c.offsets) == len(c.bio_ids)
          for c in a4_val_chunks))

assert a4_val_chunks
assert max(lengths) <= A4_MAX_TOKENS
assert all(len(c.input_ids) == len(c.offsets) == len(c.bio_ids)
           for c in a4_val_chunks)

In [ ]:
# Self-contained Fold 1 validation chunk reconstruction.
# No training or inference.

A4_MAX_TOKENS = 1024
A4_OVERLAP = 256
A4_STRIDE = A4_MAX_TOKENS - A4_OVERLAP

def normalize_gold_local(gold_list):
    result = []
    for item in gold_list:
        result.append((int(item[0]), int(item[1]), str(item[2])))
    return sorted(result, key=lambda x: (x[0], x[1]))

def build_a4_validation_chunks(doc_id, text, gold_list):
    enc = tokenizer.encode(text)
    token_ids, token_offsets = enc.ids, enc.offsets
    gold_spans = normalize_gold_local(gold_list)

    chunks = []
    token_start = 0
    chunk_id = 0
    n_tokens = len(token_ids)

    while token_start < n_tokens:
        token_end = min(token_start + A4_MAX_TOKENS, n_tokens)
        ids = token_ids[token_start:token_end]
        offsets = token_offsets[token_start:token_end]

        valid_offsets = [(s, e) for s, e in offsets if e > s]
        char_start = min((s for s, e in valid_offsets), default=0)
        char_end = max((e for s, e in valid_offsets), default=0)

        chunk_gold = [
            (s, e, label)
            for s, e, label in gold_spans
            if s >= char_start and e <= char_end
        ]

        bio_ids = [bio2id["O"]] * len(ids)

        for ent_start, ent_end, label in chunk_gold:
            covered = [
                i for i, (ts, te) in enumerate(offsets)
                if te > ts and ts < ent_end and te > ent_start
            ]
            if not covered:
                continue

            bio_ids[covered[0]] = bio2id["B-" + label]
            for i in covered[1:]:
                bio_ids[i] = bio2id["I-" + label]

        chunks.append(NERChunk(
            doc_id=str(doc_id),
            chunk_id=chunk_id,
            input_ids=list(ids),
            offsets=list(offsets),
            bio_ids=bio_ids,
            gold_spans=chunk_gold,
            char_start=char_start,
            char_end=char_end,
        ))

        chunk_id += 1
        if token_end >= n_tokens:
            break
        token_start += A4_STRIDE

    return chunks

a4_val_chunks = []
for doc_id, text in fold1_text.items():
    a4_val_chunks.extend(
        build_a4_validation_chunks(doc_id, text, fold1_gold[doc_id])
    )

lengths = [len(c.input_ids) for c in a4_val_chunks]

print("Fold 1 documents:", len(fold1_text))
print("Validation chunks:", len(a4_val_chunks))
print("Maximum chunk length:", max(lengths, default=0))
print("Minimum chunk length:", min(lengths, default=0))
print("All chunks fit model:", all(0 < n <= A4_MAX_TOKENS for n in lengths))
print("Input/offset/label lengths match:",
      all(len(c.input_ids) == len(c.offsets) == len(c.bio_ids)
          for c in a4_val_chunks))

assert a4_val_chunks
assert max(lengths) <= A4_MAX_TOKENS
assert all(len(c.input_ids) == len(c.offsets) == len(c.bio_ids)
           for c in a4_val_chunks)

In [ ]:
# Rebuild Fold 1 validation chunks with local definitions.
# No training or inference.

from dataclasses import dataclass

A4_MAX_TOKENS = 1024
A4_OVERLAP = 256
A4_STRIDE = A4_MAX_TOKENS - A4_OVERLAP

ENTITY_LABELS_LOCAL = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL",
    "JUDGE", "WIT", "AUTH", "COURT",
    "STAT", "PREC", "DATE", "CASENO"
]

BIO_LABELS_LOCAL = ["O"] + [
    tag
    for entity in ENTITY_LABELS_LOCAL
    for tag in (f"B-{entity}", f"I-{entity}")
]
bio2id_local = {tag: i for i, tag in enumerate(BIO_LABELS_LOCAL)}

@dataclass
class ValidationChunkLocal:
    doc_id: str
    chunk_id: int
    input_ids: list
    offsets: list
    bio_ids: list
    gold_spans: list
    char_start: int
    char_end: int

def normalize_gold_local(gold_list):
    result = [
        (int(item[0]), int(item[1]), str(item[2]))
        for item in gold_list
    ]
    result.sort(key=lambda x: (x[0], x[1]))
    return result

def build_a4_validation_chunks(doc_id, text, gold_list):
    enc = tokenizer.encode(text)
    token_ids, token_offsets = enc.ids, enc.offsets
    gold_spans = normalize_gold_local(gold_list)

    chunks = []
    token_start = 0
    chunk_id = 0
    n_tokens = len(token_ids)

    while token_start < n_tokens:
        token_end = min(token_start + A4_MAX_TOKENS, n_tokens)
        ids = token_ids[token_start:token_end]
        offsets = token_offsets[token_start:token_end]

        valid_offsets = [(s, e) for s, e in offsets if e > s]
        char_start = min((s for s, e in valid_offsets), default=0)
        char_end = max((e for s, e in valid_offsets), default=0)

        chunk_gold = [
            (s, e, label)
            for s, e, label in gold_spans
            if s >= char_start and e <= char_end
        ]

        bio_ids = [bio2id_local["O"]] * len(ids)

        for ent_start, ent_end, label in chunk_gold:
            if label not in ENTITY_LABELS_LOCAL:
                raise ValueError(f"Unknown entity label: {label}")

            covered = [
                i for i, (ts, te) in enumerate(offsets)
                if te > ts and ts < ent_end and te > ent_start
            ]
            if not covered:
                continue

            bio_ids[covered[0]] = bio2id_local[f"B-{label}"]
            for i in covered[1:]:
                bio_ids[i] = bio2id_local[f"I-{label}"]

        chunks.append(ValidationChunkLocal(
            doc_id=str(doc_id),
            chunk_id=chunk_id,
            input_ids=list(ids),
            offsets=list(offsets),
            bio_ids=bio_ids,
            gold_spans=chunk_gold,
            char_start=char_start,
            char_end=char_end,
        ))

        chunk_id += 1
        if token_end >= n_tokens:
            break
        token_start += A4_STRIDE

    return chunks

a4_val_chunks = []
for doc_id, text in fold1_text.items():
    a4_val_chunks.extend(
        build_a4_validation_chunks(doc_id, text, fold1_gold[doc_id])
    )

lengths = [len(c.input_ids) for c in a4_val_chunks]

print("BIO label count:", len(BIO_LABELS_LOCAL))
print("Fold 1 documents:", len(fold1_text))
print("Validation chunks:", len(a4_val_chunks))
print("Maximum chunk length:", max(lengths, default=0))
print("Minimum chunk length:", min(lengths, default=0))
print("All chunks fit 1024 positions:", all(0 < n <= 1024 for n in lengths))
print("Input, offset and BIO lengths match:",
      all(len(c.input_ids) == len(c.offsets) == len(c.bio_ids)
          for c in a4_val_chunks))

assert len(BIO_LABELS_LOCAL) == 25
assert a4_val_chunks
assert max(lengths) <= A4_MAX_TOKENS
assert all(len(c.input_ids) == len(c.offsets) == len(c.bio_ids)
           for c in a4_val_chunks)

In [ ]:
import torch

model4.eval()
device = next(model4.parameters()).device
sample = a4_val_chunks[0]

input_ids = torch.tensor(
    [sample.input_ids],
    dtype=torch.long,
    device=device
)

with torch.inference_mode():
    logits = model4(input_ids=input_ids)

print("Device:", device)
print("Input shape:", tuple(input_ids.shape))
print("Logits shape:", tuple(logits.shape))
print("All logits finite:", bool(torch.isfinite(logits).all()))
print("Predicted BIO IDs:", logits.argmax(dim=-1)[0, :20].tolist())
print("Expected output classes:", len(BIO_LABELS_LOCAL))

assert logits.shape == (1, len(sample.input_ids), 25)
assert torch.isfinite(logits).all()

print("\nInference smoke test passed.")

In [ ]:
from collections import Counter
import torch

model4.eval()
device = next(model4.parameters()).device

pred_counts = Counter()
gold_counts = Counter()
correct = 0
total = 0
confidence_sum = 0.0
entity_pred_tokens = 0
entity_gold_tokens = 0

N_DIAGNOSTIC_CHUNKS = min(8, len(a4_val_chunks))

with torch.inference_mode():
    for chunk in a4_val_chunks[:N_DIAGNOSTIC_CHUNKS]:
        x = torch.tensor(
            [chunk.input_ids],
            dtype=torch.long,
            device=device
        )

        logits = model4(input_ids=x)
        probs = torch.softmax(logits, dim=-1)
        pred_ids = logits.argmax(dim=-1)[0].cpu().tolist()

        pred_counts.update(pred_ids)
        gold_counts.update(chunk.bio_ids)

        correct += sum(
            p == g for p, g in zip(pred_ids, chunk.bio_ids)
        )
        total += len(chunk.bio_ids)

        confidence_sum += probs.max(dim=-1).values.sum().item()
        entity_pred_tokens += sum(p != 0 for p in pred_ids)
        entity_gold_tokens += sum(g != 0 for g in chunk.bio_ids)

def readable_counts(counter):
    return {
        BIO_LABELS_LOCAL[i]: counter.get(i, 0)
        for i in range(len(BIO_LABELS_LOCAL))
        if counter.get(i, 0) > 0
    }

print("Diagnostic chunks:", N_DIAGNOSTIC_CHUNKS)
print("Tokens compared:", total)
print("Token accuracy:", round(correct / total, 4) if total else None)
print("Mean maximum softmax probability:",
      round(confidence_sum / total, 4) if total else None)
print("Predicted entity-tag tokens:", entity_pred_tokens)
print("Gold entity-tag tokens:", entity_gold_tokens)

print("\nPredicted BIO tag counts:")
print(readable_counts(pred_counts))

print("\nGold BIO tag counts:")
print(readable_counts(gold_counts))

In [ ]:
import torch

print("=" * 60)
print("CHECKPOINT AND LABEL ALIGNMENT")
print("=" * 60)

print("Model class:", type(model4).__name__)
print("Encoder class:", type(model4.encoder).__name__)
print("Positional embedding:", tuple(model4.encoder.encoder_pos.weight.shape))
print("Classifier weight:", tuple(model4.classifier.weight.shape))
print("Classifier bias:", tuple(model4.classifier.bias.shape))
print("Number of local BIO labels:", len(BIO_LABELS_LOCAL))

print("\nBIO ID mapping:")
for i, label in enumerate(BIO_LABELS_LOCAL):
    print(f"{i:2d}: {label}")

if "ckpt" in globals():
    print("\nCheckpoint metadata:")
    for key in ["experiment", "epoch", "train_loss", "train_chunks",
                "val_chunks", "lorma_rank", "lorma_alpha", "lorma_dropout"]:
        print(f"{key}: {ckpt.get(key, '<not stored>')}")

    state = ckpt.get("model_state_dict", {})
    print("Checkpoint classifier shape:",
          tuple(state["classifier.weight"].shape)
          if "classifier.weight" in state else "<missing>")

    print("Checkpoint encoder position shape:",
          tuple(state["encoder.encoder_pos.weight"].shape)
          if "encoder.encoder_pos.weight" in state else "<missing>")

print("\nLive model parameter device:",
      next(model4.parameters()).device)
print("Live model in eval mode:", not model4.training)
print("\nInspection only — no weights changed.")

In [ ]:
import torch
from collections import Counter

state = ckpt["model_state_dict"]
live_state = model4.state_dict()

print("=" * 60)
print("LOADED WEIGHT VERIFICATION")
print("=" * 60)

missing = sorted(set(state) - set(live_state))
unexpected = sorted(set(live_state) - set(state))
print("Checkpoint keys missing from live model:", missing[:10], "count:", len(missing))
print("Extra live model keys:", unexpected[:10], "count:", len(unexpected))

mismatches = []
for key, expected in state.items():
    if key not in live_state:
        continue
    actual = live_state[key].detach().cpu()
    expected = expected.detach().cpu()
    if actual.shape != expected.shape or not torch.equal(actual, expected):
        mismatches.append(key)

print("Tensor values not exactly equal to checkpoint:", len(mismatches))
print("First mismatches:", mismatches[:15])

adapter_keys = [
    k for k in state
    if "lorma_A" in k or "lorma_B" in k
]
print("\nAdapter tensors:", len(adapter_keys))

for key in adapter_keys[:8]:
    t = live_state[key].detach().float().cpu()
    print(
        key,
        "| shape:", tuple(t.shape),
        "| abs mean:", round(t.abs().mean().item(), 7),
        "| std:", round(t.std().item(), 7),
        "| nonzero:", int(torch.count_nonzero(t).item())
    )

print("\nClassifier weight std:",
      round(model4.classifier.weight.detach().float().std().item(), 6))
print("Classifier bias std:",
      round(model4.classifier.bias.detach().float().std().item(), 6))

assert not missing, "Checkpoint keys are missing from live model."
assert not unexpected, "Live model has unexpected state keys."
assert not mismatches, "Live model weights differ from checkpoint."

print("\nCheckpoint/live model weights match exactly.")

In [ ]:
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

terms = [
    "class A4Attention",
    "class A4EncoderLayer",
    "class A4EncoderForNER",
    "class A4NER",
]

for term in terms:
    hits = [
        i for i, line in enumerate(lines)
        if term in line
    ]
    print(f"\n{'='*65}\n{term}: {len(hits)} occurrence(s)")
    for i in hits:
        start = max(0, i - 2)
        end = min(len(lines), i + 65)
        print(f"\n--- Source lines {start+1}-{end} ---")
        print("\n".join(
            f"{j+1}: {lines[j]}"
            for j in range(start, end)
        ))

print("\nInspection only — no model weights changed.")

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

# Search for the original LORMA experiment setup and training references.
terms = [
    "lorma_encoder = copy.deepcopy(encoder)",
    "model4 = A4NER",
    "experiment = \"A4_LORMA\"",
    "'A4_LORMA'",
    '"A4_LORMA"',
    "a4_train_chunks =",
    "A4_MAX_TOKENS",
]

for term in terms:
    hits = [i for i, line in enumerate(lines) if term in line]
    print(f"\n{'='*65}\n{term}: {len(hits)} hit(s)")
    for i in hits[-5:]:
        start = max(0, i - 18)
        end = min(len(lines), i + 35)
        print(f"\n--- Lines {start+1}-{end} ---")
        print("\n".join(
            f"{j+1}: {lines[j]}" for j in range(start, end)
        ))

print("\nInspection only — no training or inference performed.")

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

# Inspect the original LORMA setup and encoder initialization.
for start, end in [
    (13880, 14010),
    (14820, 14955),
    (15120, 15230),
    (30759, 30885),
]:
    print(f"\n{'='*22} SOURCE LINES {start}-{end} {'='*22}")
    print("\n".join(
        f"{i+1}: {lines[i]}"
        for i in range(start - 1, min(end, len(lines)))
    ))

print("\nInspection only — no weights changed.")

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

terms = [
    "a4_step_22000.pt",
    "architecture1_sustained_step_500.pt",
    "load_state_dict",
    "encoder = A4Encoder",
    "encoder = A4EncoderForNER",
    "encoder.load_state_dict",
    "torch.load",
    "Validated A4 encoder",
]

for term in terms:
    hits = [i for i, line in enumerate(lines) if term.lower() in line.lower()]
    print(f"\n{'='*70}\n{term}: {len(hits)} matches")

    # Show the first few matches, which are more likely to be original setup.
    for i in hits[:5]:
        start = max(0, i - 12)
        end = min(len(lines), i + 24)
        print(f"\n--- Lines {start+1}-{end} ---")
        print("\n".join(
            f"{j+1}: {lines[j]}" for j in range(start, end)
        ))

print("\nInspection only — no weights changed.")

In [ ]:
import torch

phase1_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)

phase1_ckpt = torch.load(
    phase1_path,
    map_location="cpu",
    weights_only=False
)
phase1_state = phase1_ckpt["model_state_dict"]
phase2_state = model4.state_dict()

# Compare only corresponding, unchanged encoder tensors.
# For LORMA-wrapped q/v layers, compare against their base weights.
comparisons = []

for key, phase1_tensor in phase1_state.items():
    if not key.startswith((
        "token_embedding.",
        "encoder_pos.",
        "encoder_layers."
    )):
        continue

    if key not in phase2_state:
        # Map the original q/v projection weights to LORMA base weights.
        if ".attn.q_proj." in key:
            phase2_key = key.replace(
                ".attn.q_proj.", ".attn.q_proj.base."
            )
        elif ".attn.v_proj." in key:
            phase2_key = key.replace(
                ".attn.v_proj.", ".attn.v_proj.base."
            )
        else:
            continue
    else:
        phase2_key = key

    if phase2_key not in phase2_state:
        continue

    a = phase1_tensor.detach().cpu()
    b = phase2_state[phase2_key].detach().cpu()

    if a.shape != b.shape:
        continue

    comparisons.append({
        "key": key,
        "exact": torch.equal(a, b),
        "max_abs_diff": (a.float() - b.float()).abs().max().item(),
    })

print("Comparable tensors:", len(comparisons))
print("Exactly identical:", sum(x["exact"] for x in comparisons))
print("Different:", sum(not x["exact"] for x in comparisons))

print("\nLargest differences:")
for item in sorted(
    comparisons,
    key=lambda x: x["max_abs_diff"],
    reverse=True
)[:12]:
    print(
        item["key"],
        "| exact:", item["exact"],
        "| max_abs_diff:", f'{item["max_abs_diff"]:.8g}'
    )

print("\nNo weights changed; comparison only.")

In [ ]:
import torch

phase1_path = "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"

phase1_ckpt = torch.load(
    phase1_path,
    map_location="cpu",
    weights_only=False
)
phase1_state = phase1_ckpt["model_state_dict"]
phase2_state = model4.state_dict()

comparisons = []
unmatched = []

for key, a in phase1_state.items():
    if not key.startswith((
        "token_embedding.",
        "encoder_pos.",
        "encoder_layers."
    )):
        continue

    # Phase 2 stores the encoder under model4.encoder
    phase2_key = "encoder." + key

    # LORMA wraps q_proj and v_proj in a module named "base"
    if ".attn.q_proj." in key:
        phase2_key = "encoder." + key.replace(
            ".attn.q_proj.", ".attn.q_proj.base."
        )
    elif ".attn.v_proj." in key:
        phase2_key = "encoder." + key.replace(
            ".attn.v_proj.", ".attn.v_proj.base."
        )

    if phase2_key not in phase2_state:
        unmatched.append((key, phase2_key))
        continue

    b = phase2_state[phase2_key]

    if a.shape != b.shape:
        unmatched.append((key, phase2_key, tuple(a.shape), tuple(b.shape)))
        continue

    a = a.detach().cpu()
    b = b.detach().cpu()

    comparisons.append({
        "key": key,
        "exact": torch.equal(a, b),
        "max_abs_diff": (a.float() - b.float()).abs().max().item()
    })

print("Comparable tensors:", len(comparisons))
print("Exactly identical:", sum(x["exact"] for x in comparisons))
print("Different:", sum(not x["exact"] for x in comparisons))
print("Unmatched keys:", len(unmatched))

print("\nLargest differences:")
for item in sorted(
    comparisons,
    key=lambda x: x["max_abs_diff"],
    reverse=True
)[:10]:
    print(item)

print("\nFirst unmatched keys:")
for item in unmatched[:10]:
    print(item)

In [ ]:
import torch

phase1_path = "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
phase2_path = "/kaggle/input/datasets/aq1de3/a4-ner-exp4-lorma-epoch3/a4_ner_exp4_lorma_epoch3.pt"

p1 = torch.load(phase1_path, map_location="cpu", weights_only=False)
p2 = torch.load(phase2_path, map_location="cpu", weights_only=False)

print("PHASE 1 CHECKPOINT")
print("Top-level keys:", list(p1.keys()))
print("Metadata:", {
    k: v for k, v in p1.items()
    if not isinstance(v, (dict, torch.Tensor))
})

print("\nPHASE 2 CHECKPOINT")
print("Top-level keys:", list(p2.keys()))
print("Metadata:", {
    k: v for k, v in p2.items()
    if not isinstance(v, (dict, torch.Tensor))
})

print("\nPhase 1 state-dict sample:")
print(list(p1["model_state_dict"].keys())[:20])

print("\nPhase 2 state-dict sample:")
print(list(p2["model_state_dict"].keys())[:30])

print("\nPhase 2 training history:")
print(p2.get("history", "No history field found"))

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

terms = [
    "lorma_encoder = copy.deepcopy(encoder)",
    "model4 = A4NER",
    "requires_grad",
    "optimizer4",
    "A4_CHECKPOINT_PATH",
    "encoder.load_state_dict",
    "a4_step_22000.pt",
]

for term in terms:
    hits = [i for i, line in enumerate(lines) if term in line]
    print(f"\n{'=' * 25}\n{term}: {len(hits)} matches")

    for i in hits[-3:]:
        start = max(0, i - 12)
        end = min(len(lines), i + 20)
        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

ranges = [
    (9880, 9940),    # Earlier A4 encoder initialization
    (10374, 10475),  # Phase 1 checkpoint loading and key remapping
    (13570, 13640),  # Validated encoder setup for NER
    (14820, 15010),  # LORMA definition and model construction
    (15010, 15120),  # Trainable parameters and optimizer setup
    (15120, 15510),  # LORMA training and checkpoint saving
]

for start, end in ranges:
    print(f"\n{'=' * 25} LINES {start}-{end} {'=' * 25}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1}: {lines[i]}")

In [ ]:
import torch

p1 = torch.load(
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt",
    map_location="cpu",
    weights_only=False
)["model_state_dict"]

p2 = torch.load(
    "/kaggle/input/datasets/aq1de3/a4-ner-exp4-lorma-epoch3/a4_ner_exp4_lorma_epoch3.pt",
    map_location="cpu",
    weights_only=False
)["model_state_dict"]

def phase2_key_for(k):
    if ".attn.q_proj." in k:
        k = k.replace(".attn.q_proj.", ".attn.q_proj.base.")
    elif ".attn.v_proj." in k:
        k = k.replace(".attn.v_proj.", ".attn.v_proj.base.")
    return "encoder." + k

def compare_states(reference, candidate, prefix=""):
    results = []
    for k, a in reference.items():
        if not k.startswith((
            "token_embedding.", "encoder_pos.", "encoder_layers."
        )):
            continue

        target = prefix + k
        if target not in candidate:
            target = phase2_key_for(k)

        if target not in candidate:
            continue

        b = candidate[target].detach().cpu()
        a = a.detach().cpu()

        if a.shape == b.shape:
            results.append((
                k,
                torch.equal(a, b),
                (a.float() - b.float()).abs().max().item()
            ))

    print("Comparable:", len(results))
    print("Identical:", sum(x[1] for x in results))
    print("Different:", sum(not x[1] for x in results))
    if results:
        print("Largest difference:", max(results, key=lambda x: x[2]))

print("A. Phase 1 checkpoint vs saved Phase 2 model")
compare_states(p1, p2)

if "encoder" in globals():
    print("\nB. Current encoder vs saved Phase 2 model")
    current_encoder_state = encoder.state_dict()
    compare_states(current_encoder_state, p2, prefix="encoder.")
else:
    print("\nNo variable named `encoder` exists in the current session.")

In [ ]:
import torch
from pathlib import Path

checkpoint_paths = {
    "A1": "/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt",
    "A2": "/kaggle/input/datasets/aq1de3/a2check/a2_step_9000.pt",
    "A3": "/kaggle/input/datasets/aq1de3/a3check/step_0001000.pt",
    "A4": "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt",
}

current = {
    k: v.detach().cpu()
    for k, v in encoder.state_dict().items()
}

for name, path in checkpoint_paths.items():
    if not Path(path).exists():
        print(f"{name}: checkpoint not found")
        continue

    obj = torch.load(path, map_location="cpu", weights_only=False)
    state = obj.get("model_state_dict", obj)

    matches, identical, compared = 0, 0, 0

    for k, a in current.items():
        if k not in state or state[k].shape != a.shape:
            continue

        compared += 1
        if torch.equal(a, state[k].cpu()):
            identical += 1

    print(
        f"{name}: comparable={compared}, "
        f"identical={identical}, "
        f"different={compared-identical}"
    )

In [ ]:
import torch

path = "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
state = torch.load(path, map_location="cpu", weights_only=False)["model_state_dict"]
current = {k: v.detach().cpu() for k, v in encoder.state_dict().items()}

# Align the Phase 1 FFN naming with the current encoder naming.
mapped = {}
for k, v in state.items():
    if not k.startswith(("token_embedding.", "encoder_pos.", "encoder_layers.")):
        continue
    new_k = k.replace(".ffn.net.0.", ".ffn.0.").replace(".ffn.net.2.", ".ffn.2.")
    mapped[new_k] = v.detach().cpu()

common = [k for k in current if k in mapped and current[k].shape == mapped[k].shape]
missing = [k for k in current if k not in mapped]
shape_mismatch = [k for k in current if k in mapped and current[k].shape != mapped[k].shape]
identical = [k for k in common if torch.equal(current[k], mapped[k])]

print("Current encoder tensors:", len(current))
print("Phase 1 encoder tensors after mapping:", len(mapped))
print("Comparable tensors:", len(common))
print("Exactly identical:", len(identical))
print("Different:", len(common) - len(identical))
print("Missing from Phase 1 mapping:", len(missing))
print("Shape mismatches:", len(shape_mismatch))
print("\nFirst missing keys:", missing[:10])
print("\nFirst mismatches:", shape_mismatch[:10])

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

terms = [
    "encoder = A4Encoder(",
    "encoder = a4_encoder",
    "encoder.load_state_dict",
    "encoder.load_state_dict(",
    "encoder = copy.deepcopy",
    "encoder = torch.load",
    "encoder.load_state_dict(encoder_state",
    "A4_CHECKPOINT_PATH",
]

for term in terms:
    hits = [i for i, line in enumerate(lines) if term in line]
    print(f"\n{'=' * 18} {term} ({len(hits)} matches) {'=' * 18}")
    for i in hits:
        if i < 14820:  # Only initialization code before Experiment 4
            start = max(0, i - 8)
            end = min(len(lines), i + 14)
            print(f"\n--- Lines {start+1}-{end} ---")
            for j in range(start, end):
                print(f"{j+1}: {lines[j]}")

In [ ]:
import torch

path = "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
checkpoint = torch.load(path, map_location="cpu", weights_only=False)
state = checkpoint["model_state_dict"]

# Extract original A4 encoder weights and adapt FFN key names
encoder_state = {
    k: v.detach().cpu()
    for k, v in state.items()
    if (
        k.startswith("token_embedding.")
        or k.startswith("encoder_pos.")
        or k.startswith("encoder_layers.")
    )
}

remapped_state = {
    k.replace(".ffn.net.0.", ".ffn.0.")
     .replace(".ffn.net.2.", ".ffn.2."): v
    for k, v in encoder_state.items()
}

current = {
    k: v.detach().cpu()
    for k, v in encoder.state_dict().items()
}

missing = [k for k in current if k not in remapped_state]
unexpected = [k for k in remapped_state if k not in current]
shape_mismatch = [
    k for k in current.keys() & remapped_state.keys()
    if current[k].shape != remapped_state[k].shape
]

comparable = [
    k for k in current.keys() & remapped_state.keys()
    if current[k].shape == remapped_state[k].shape
]

identical = [
    k for k in comparable
    if torch.equal(current[k], remapped_state[k])
]

print("Original encoder tensors:", len(remapped_state))
print("Current encoder tensors:", len(current))
print("Comparable:", len(comparable))
print("Exactly identical:", len(identical))
print("Different:", len(comparable) - len(identical))
print("Missing:", missing[:10], "count =", len(missing))
print("Unexpected:", unexpected[:10], "count =", len(unexpected))
print("Shape mismatches:", shape_mismatch[:10], "count =", len(shape_mismatch))

In [ ]:
try:
    import torch
    from pathlib import Path

    path = Path(
        "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
    )

    print("1. Checkpoint exists:", path.exists())
    print("2. Encoder exists:", "encoder" in globals())

    if path.exists():
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=False
        )
        print("3. Checkpoint keys:", list(checkpoint.keys()))

        state = checkpoint.get("model_state_dict", {})
        print("4. Checkpoint tensor count:", len(state))

    if "encoder" in globals():
        current = encoder.state_dict()
        print("5. Current encoder tensor count:", len(current))

    print("DIAGNOSTIC FINISHED")

except Exception as e:
    print("ERROR:", type(e).__name__, str(e))

In [ ]:
print("CELL EXECUTION WORKS")
print("Encoder exists:", "encoder" in globals())
print("PyTorch loaded:", "torch" in globals())

In [ ]:
from pathlib import Path
import zipfile

for p in Path("/kaggle/working").rglob("phase2_exp10_recovered.zip"):
    print("Found:", p)
    print("Size (MB):", round(p.stat().st_size / 1024**2, 2))

    with zipfile.ZipFile(p) as z:
        print("ZIP contents:")
        for name in z.namelist():
            print(" -", name)

        bad_file = z.testzip()
        print("ZIP integrity:", "OK" if bad_file is None else bad_file)

In [1]:
print("KAGGLE WORKS")

KAGGLE WORKS


In [ ]:
from pathlib import Path

print("Working directory:", Path.cwd())

for p in Path("/kaggle/working").glob("*"):
    print(p.name, "| directory" if p.is_dir() else f"| {p.stat().st_size:,} bytes")

In [ ]:
from pathlib import Path

root = Path("/kaggle/working")

matches = []
for p in root.rglob("*"):
    if p.is_file() and (
        p.suffix.lower() == ".zip"
        or p.name.lower() == "prediction.json"
    ):
        matches.append(p)

print("Files found:", len(matches))
for p in matches:
    print(p, "|", round(p.stat().st_size / 1024**2, 2), "MB")

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("Input directory exists:", root.exists())

for p in root.iterdir():
    print("\nDataset:", p.name)
    if p.is_dir():
        for child in list(p.iterdir())[:20]:
            print("  ", child.name)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input/datasets/aq1de3")

print("Searching:", root)

count = 0
for p in root.rglob("*"):
    if p.is_file() and (
        p.suffix.lower() == ".zip"
        or p.name.lower() == "prediction.json"
    ):
        print(
            p,
            "|",
            round(p.stat().st_size / 1024**2, 2),
            "MB"
        )
        count += 1

print("\nTotal matching files:", count)

In [ ]:
names = [
    "model4",
    "ckpt",
    "encoder",
    "tokenizer",
    "fold1_text",
    "fold1_gold",
    "fold2_text",
    "fold2_gold",
    "test_data",
    "labels",
    "a4_val_chunks",
]

for name in names:
    exists = name in globals()
    obj = globals().get(name)
    print(f"{name:15} exists={exists}", end="")
    if exists:
        try:
            print(f" | type={type(obj).__name__} | len={len(obj)}")
        except Exception:
            print(f" | type={type(obj).__name__}")
    else:
        print()

In [ ]:
from pathlib import Path

root = Path("/kaggle/input/datasets/aq1de3")

for p in sorted(root.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(root)} | {p.stat().st_size / 1024**2:.2f} MB")

In [ ]:
import json
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/aq1de3")

fold1_text = json.loads(
    (ROOT / "fold1-tex/fold1-text.json").read_text()
)
fold1_gold = json.loads(
    (ROOT / "fold1-gold1/fold1-gold.json").read_text()
)

fold2_text = json.loads(
    (ROOT / "fold2-te/fold2-text.json").read_text()
)
fold2_gold = json.loads(
    (ROOT / "fold2-go/fold2-gold.json").read_text()
)

print("Fold 1 text documents:", len(fold1_text))
print("Fold 1 gold documents:", len(fold1_gold))
print("Fold 2 text documents:", len(fold2_text))
print("Fold 2 gold documents:", len(fold2_gold))

print("\nFold 1 sample ID:", next(iter(fold1_text)))
print("Fold 1 sample text type:", type(next(iter(fold1_text.values()))).__name__)
print("Fold 1 sample gold type:", type(next(iter(fold1_gold.values()))).__name__)

In [ ]:
def inspect_dataset(texts, gold, name):
    text_ids = set(texts)
    gold_ids = set(gold)

    print(f"\n{'=' * 45}")
    print(name)
    print(f"{'=' * 45}")
    print("Text IDs:", len(text_ids))
    print("Gold IDs:", len(gold_ids))
    print("Missing gold IDs:", len(text_ids - gold_ids))
    print("Extra gold IDs:", len(gold_ids - text_ids))

    malformed = 0
    out_of_bounds = 0
    total_spans = 0

    for doc_id in text_ids & gold_ids:
        text = texts[doc_id]
        annotations = gold[doc_id]

        for ann in annotations:
            total_spans += 1

            if not isinstance(ann, (list, tuple)) or len(ann) != 3:
                malformed += 1
                continue

            start, end, label = ann

            if (
                not isinstance(start, int)
                or not isinstance(end, int)
                or start < 0
                or end <= start
                or end > len(text)
            ):
                out_of_bounds += 1

    print("Total gold spans:", total_spans)
    print("Malformed annotations:", malformed)
    print("Invalid/out-of-bounds spans:", out_of_bounds)

inspect_dataset(fold1_text, fold1_gold, "FOLD 1")
inspect_dataset(fold2_text, fold2_gold, "FOLD 2")

In [ ]:
doc_id = next(iter(fold1_gold))
gold = fold1_gold[doc_id]

print("Document ID:", doc_id)
print("Gold container type:", type(gold).__name__)
print("Number of items:", len(gold))
print("\nFirst 3 annotation items:")

for i, item in enumerate(gold[:3]):
    print(f"\nItem {i}")
    print("Type:", type(item).__name__)
    print("Value:", repr(item)[:1000])

In [ ]:
def inspect_dataset_v2(texts, gold, name):
    text_ids = set(texts)
    gold_ids = set(gold)

    malformed = 0
    out_of_bounds = 0
    text_mismatches = 0
    total_spans = 0
    label_counts = {}

    for doc_id in text_ids & gold_ids:
        text = texts[doc_id]

        for ann in gold[doc_id]:
            total_spans += 1

            if not isinstance(ann, (list, tuple)) or len(ann) != 4:
                malformed += 1
                continue

            start, end, label, entity_text = ann

            if (
                not isinstance(start, int)
                or not isinstance(end, int)
                or start < 0
                or end <= start
                or end > len(text)
            ):
                out_of_bounds += 1
                continue

            if text[start:end] != entity_text:
                text_mismatches += 1

            label_counts[label] = label_counts.get(label, 0) + 1

    print(f"\n{name}")
    print("Document ID mismatches:", len(text_ids ^ gold_ids))
    print("Total gold spans:", total_spans)
    print("Malformed annotations:", malformed)
    print("Invalid/out-of-bounds spans:", out_of_bounds)
    print("Entity text mismatches:", text_mismatches)
    print("Labels:", label_counts)

inspect_dataset_v2(fold1_text, fold1_gold, "FOLD 1")
inspect_dataset_v2(fold2_text, fold2_gold, "FOLD 2")

In [ ]:
from pathlib import Path
from collections import Counter

ROOT = Path("/kaggle/input/datasets/aq1de3")
label_path = ROOT / "labelses/labels.txt"

print("Raw labels.txt:")
print(repr(label_path.read_text()))

labels_from_file = [
    line.strip()
    for line in label_path.read_text().splitlines()
    if line.strip()
]

gold_labels = set()
for annotations in list(fold1_gold.values()) + list(fold2_gold.values()):
    for ann in annotations:
        if isinstance(ann, (list, tuple)) and len(ann) == 4:
            gold_labels.add(ann[2])

print("\nLabels from file:", labels_from_file)
print("Unique gold labels:", sorted(gold_labels))
print("Gold labels missing from file:", sorted(gold_labels - set(labels_from_file)))
print("Unused labels from file:", sorted(set(labels_from_file) - gold_labels))

In [ ]:
from tokenizers import Tokenizer
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/aq1de3")
tokenizer = Tokenizer.from_file(str(ROOT / "tokenizer/tokenizer.json"))

MAX_TOKENS = 1024
STRIDE = 768

def gold_bio_roundtrip(text, annotations):
    enc = tokenizer.encode(text)
    offsets = enc.offsets
    n = len(offsets)

    reconstructed = []

    for start_idx in range(0, n, STRIDE):
        end_idx = min(start_idx + MAX_TOKENS, n)
        chunk_offsets = offsets[start_idx:end_idx]
        tags = ["O"] * len(chunk_offsets)

        for start, end, label, entity_text in annotations:
            covered = [
                i for i, (ts, te) in enumerate(chunk_offsets)
                if te > start and ts < end
            ]

            if not covered:
                continue

            # Only assign an entity if the entire span is inside this chunk.
            first_start = chunk_offsets[covered[0]][0]
            last_end = chunk_offsets[covered[-1]][1]

            if first_start > start or last_end < end:
                continue

            tags[covered[0]] = f"B-{label}"
            for i in covered[1:]:
                tags[i] = f"I-{label}"

        # Decode BIO tags back to character spans.
        i = 0
        while i < len(tags):
            if not tags[i].startswith("B-"):
                i += 1
                continue

            label = tags[i][2:]
            char_start = chunk_offsets[i][0]
            char_end = chunk_offsets[i][1]
            j = i + 1

            while (
                j < len(tags)
                and tags[j] == f"I-{label}"
            ):
                char_end = chunk_offsets[j][1]
                j += 1

            reconstructed.append((char_start, char_end, label))
            i = j

        if end_idx == n:
            break

    return reconstructed

# Use a small sample first.
sample_ids = list(fold1_text)[:5]
gold_total = 0
recovered_total = 0
exact_matches = 0

for doc_id in sample_ids:
    gold_spans = {
        (start, end, label)
        for start, end, label, _ in fold1_gold[doc_id]
    }
    predicted_spans = set(
        gold_bio_roundtrip(fold1_text[doc_id], fold1_gold[doc_id])
    )

    gold_total += len(gold_spans)
    recovered_total += len(predicted_spans)
    exact_matches += len(gold_spans & predicted_spans)

    print(
        doc_id,
        "| gold:", len(gold_spans),
        "| reconstructed:", len(predicted_spans),
        "| exact:", len(gold_spans & predicted_spans),
        "| missing:", len(gold_spans - predicted_spans)
    )

print("\nTOTAL")
print("Gold spans:", gold_total)
print("Reconstructed spans:", recovered_total)
print("Exact matches:", exact_matches)
print("Gold-span recovery:", round(exact_matches / gold_total, 4) if gold_total else 0)

In [ ]:
doc_id = "115651329"
text = fold1_text[doc_id]
annotations = fold1_gold[doc_id]

# Inspect the first gold entity
start, end, label, entity_text = annotations[0]
enc = tokenizer.encode(text)

print("Gold span:", (start, end, label, entity_text))
print("Actual text slice:", repr(text[start:end]))

print("\nOverlapping tokens:")
for i, (ts, te) in enumerate(enc.offsets):
    if te > start and ts < end:
        print(
            "token_index =", i,
            "| offset =", (ts, te),
            "| token_text =", repr(text[ts:te])
        )

print("\nGold start/end:", start, end)

In [ ]:
def boundary_diagnostic(texts, gold, sample_size=5):
    total = 0
    start_mismatch = 0
    end_mismatch = 0
    both_mismatch = 0

    for doc_id in list(texts)[:sample_size]:
        text = texts[doc_id]
        offsets = tokenizer.encode(text).offsets

        for start, end, label, entity_text in gold[doc_id]:
            covered = [
                (ts, te) for ts, te in offsets
                if te > start and ts < end
            ]

            if not covered:
                continue

            total += 1
            sm = covered[0][0] != start
            em = covered[-1][1] != end

            start_mismatch += sm
            end_mismatch += em
            both_mismatch += sm and em

    print("Gold spans examined:", total)
    print("Start boundary mismatches:", start_mismatch)
    print("End boundary mismatches:", end_mismatch)
    print("Both boundaries mismatched:", both_mismatch)
    print("Exact token-boundary coverage:", total - start_mismatch - end_mismatch + both_mismatch)

boundary_diagnostic(fold1_text, fold1_gold)

In [ ]:
print("Tokenizer configuration:")
print(tokenizer)

print("\nTokenizer settings:")
print("Pre-tokenizer:", tokenizer.pre_tokenizer)
print("Normalizer:", tokenizer.normalizer)
print("Post-processor:", tokenizer.post_processor)

doc_id = "115651329"
text = fold1_text[doc_id]
enc = tokenizer.encode(text)

print("\nFirst 40 tokens with offsets:")
for i, (token, (start, end)) in enumerate(
    zip(enc.tokens, enc.offsets)
):
    print(
        f"{i:3} | {start:5}-{end:5} | "
        f"{repr(text[start:end])} | token={repr(token)}"
    )

In [ ]:
def gold_to_bio_test(annotations, offsets):
    tags = ["O"] * len(offsets)

    for start, end, label, *_ in annotations:
        covered = [
            i for i, (ts, te) in enumerate(offsets)
            if te > start and ts < end
        ]

        if not covered:
            continue

        tags[covered[0]] = f"B-{label}"
        for i in covered[1:]:
            tags[i] = f"I-{label}"

    return tags


def bio_to_spans_trimmed(text, offsets, tags):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    def close():
        nonlocal current_label, current_start, current_end

        if current_label is not None:
            start, end = current_start, current_end

            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1

            if start < end:
                spans.append((start, end, current_label))

        current_label = current_start = current_end = None

    for (ts, te), tag in zip(offsets, tags):
        if ts == te:
            continue

        if tag == "O":
            close()
            continue

        prefix, label = tag.split("-", 1)

        if prefix == "B" or current_label != label:
            close()
            current_label = label
            current_start = ts
            current_end = te
        else:
            current_end = te

    close()
    return spans


total = exact = 0

for doc_id in list(fold1_text)[:5]:
    text = fold1_text[doc_id]
    offsets = tokenizer.encode(text).offsets
    gold = fold1_gold[doc_id]

    tags = gold_to_bio_test(gold, offsets)
    reconstructed = set(bio_to_spans_trimmed(text, offsets, tags))
    gold_set = {(s, e, label) for s, e, label, *_ in gold}

    matches = len(gold_set & reconstructed)
    total += len(gold_set)
    exact += matches

    print(
        doc_id,
        "| gold:", len(gold_set),
        "| reconstructed:", len(reconstructed),
        "| exact:", matches,
        "| missed:", len(gold_set - reconstructed)
    )

print("\nOverall exact-span recovery:", round(exact / total, 4))

In [ ]:
from pathlib import Path
import torch

CHECKPOINT_PATH = Path(
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

print("Checkpoint exists:", CHECKPOINT_PATH.exists())

if not CHECKPOINT_PATH.exists():
    print("ERROR: Checkpoint path not found.")
else:
    print("Checkpoint size (MiB):",
          round(CHECKPOINT_PATH.stat().st_size / (1024 ** 2), 2))

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False
    )

    print("\nCheckpoint type:", type(checkpoint).__name__)

    if isinstance(checkpoint, dict):
        print("Top-level keys:", list(checkpoint.keys()))

        for key, value in checkpoint.items():
            if key not in ("model_state_dict", "state_dict",
                           "optimizer_state_dict"):
                print(f"{key}: {value}")

        state = (
            checkpoint.get("model_state_dict")
            or checkpoint.get("state_dict")
        )

        if state is None and all(
            isinstance(v, torch.Tensor) for v in checkpoint.values()
        ):
            state = checkpoint

        if state is not None:
            print("\nNumber of parameter tensors:", len(state))
            print("First 12 parameter names and shapes:")
            for i, (name, tensor) in enumerate(state.items()):
                if i == 12:
                    break
                print(f"{name}: {tuple(tensor.shape)}")
        else:
            print("\nNo recognizable model state dictionary found.")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

for name in [
    "A4NER",
    "A4EncoderForNER",
    "LORMALinear",
    "fold1_text",
    "fold1_gold",
    "tokenizer",
    "BIO_LABELS",
    "ID2LABEL",
    "bio_to_char_spans",
]:
    print(f"{name}: {'available' if name in globals() else 'MISSING'}")

In [ ]:
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/"
    "__notebook_source__.ipynb"
)

print("Source exists:", source_path.exists())

if source_path.exists():
    source = source_path.read_text(errors="replace")
    print("Source size:", len(source))
    print("A4EncoderForNER occurrences:",
          source.count("class A4EncoderForNER"))
    print("A4NER occurrences:",
          source.count("class A4NER"))
    print("LORMALinear occurrences:",
          source.count("class LORMALinear"))
else:
    print("Notebook source is unavailable at this path.")

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path

# Exact dimensions verified from the checkpoint
V = 50000
H = 512
MAX_POS = 1024
FFN = 2048
N_LAYERS = 6
NUM_LABELS = 25

class LORMALinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        self.lorma_A = nn.Parameter(torch.empty(rank, base.in_features))
        self.lorma_B = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scaling = alpha / rank
        self.lorma_dropout = nn.Dropout(dropout)

    def forward(self, x):
        return (
            self.base(x)
            + self.scaling
            * (self.lorma_dropout(x) @ self.lorma_A.T @ self.lorma_B.T)
        )


class A4Attention(nn.Module):
    def __init__(self, hidden, heads=8):
        super().__init__()
        self.q_proj = nn.Linear(hidden, hidden)
        self.k_proj = nn.Linear(hidden, hidden)
        self.v_proj = nn.Linear(hidden, hidden)
        self.out_proj = nn.Linear(hidden, hidden)
        self.num_heads = heads
        self.head_dim = hidden // heads

    def forward(self, query, key, value, mask=None):
        B, Tq, C = query.shape
        Tk = key.shape[1]

        q = self.q_proj(query).view(
            B, Tq, self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = self.k_proj(key).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        v = self.v_proj(value).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        out = torch.softmax(scores, dim=-1) @ v
        out = out.transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(out)


class A4FFN(nn.Module):
    def __init__(self, hidden, ffn):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, ffn),
            nn.GELU(),
            nn.Linear(ffn, hidden),
        )

    def forward(self, x):
        return self.net(x)


class A4EncoderLayer(nn.Module):
    def __init__(self, hidden, ffn):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden)
        self.attn = A4Attention(hidden, heads=8)
        self.norm2 = nn.LayerNorm(hidden)
        self.ffn = A4FFN(hidden, ffn)

    def forward(self, x, mask=None):
        h = self.norm1(x)
        x = x + self.attn(h, h, h, mask)
        return x + self.ffn(self.norm2(x))


class A4EncoderForNER(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_size = H
        self.token_embedding = nn.Embedding(V, H)
        self.encoder_pos = nn.Embedding(MAX_POS, H)
        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(H, FFN) for _ in range(N_LAYERS)
        ])

        # Intentionally no final encoder_norm:
        # the verified checkpoint does not contain it.

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        if T > MAX_POS:
            raise ValueError(f"Sequence length {T} exceeds {MAX_POS}")

        pos = torch.arange(T, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.encoder_pos(pos).unsqueeze(0)

        for layer in self.encoder_layers:
            x = layer(x, mask=None)

        return x


class A4NER(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = A4EncoderForNER()

        for layer in self.encoder.encoder_layers:
            layer.attn.q_proj = LORMALinear(
                layer.attn.q_proj, rank=8, alpha=16, dropout=0.05
            )
            layer.attn.v_proj = LORMALinear(
                layer.attn.v_proj, rank=8, alpha=16, dropout=0.05
            )

        self.dropout = nn.Dropout(0.10)
        self.classifier = nn.Linear(H, NUM_LABELS)

    def forward(self, input_ids, attention_mask=None):
        return self.classifier(self.dropout(self.encoder(input_ids)))


checkpoint_path = Path(
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

ckpt = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)

model4 = A4NER()
load_result = model4.load_state_dict(
    ckpt["model_state_dict"], strict=True
)
model4.eval()

print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)
assert not load_result.missing_keys
assert not load_result.unexpected_keys

with torch.no_grad():
    dummy = torch.zeros((1, 32), dtype=torch.long)
    logits = model4(dummy)

print("Logits shape:", tuple(logits.shape))
print("All logits finite:", bool(torch.isfinite(logits).all()))
print("Checkpoint epoch:", ckpt["epoch"])
print("Checkpoint train loss:", ckpt["train_loss"])
print("MODEL LOAD AND SMOKE TEST PASSED")

In [ ]:
import numpy as np
from collections import defaultdict
from tokenizers import Tokenizer

# Reconstruct the original BIO label order.
ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]

BIO_LABELS = ["O"]
for label in ENTITY_LABELS:
    BIO_LABELS.extend([f"B-{label}", f"I-{label}"])

LABEL2ID = {label: i for i, label in enumerate(BIO_LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

assert len(BIO_LABELS) == model4.classifier.out_features == 25

def bio_to_char_spans(text, offsets, pred_ids):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    def close_entity():
        nonlocal current_label, current_start, current_end

        if current_label is not None:
            start, end = current_start, current_end

            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1

            if start < end:
                spans.append((start, end, current_label))

        current_label = current_start = current_end = None

    for (start, end), pred_id in zip(offsets, pred_ids):
        if start == end:
            continue

        tag = ID2LABEL[int(pred_id)]

        if tag == "O":
            close_entity()
            continue

        prefix, entity_type = tag.split("-", 1)

        if prefix == "B" or current_label != entity_type:
            close_entity()
            current_label = entity_type
            current_start = start
            current_end = end
        else:
            current_end = end

    close_entity()
    return spans


# Confirm that the tokenizer object can supply offsets.
print("Tokenizer type:", type(tokenizer).__name__)
print("Validation documents:", len(fold1_text))

device = torch.device("cpu")
model4 = model4.to(device).eval()

per_label = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
total_gold = total_pred = total_exact = 0
predicted_token_count = gold_token_count = 0

for doc_num, (doc_id, text) in enumerate(fold1_text.items(), 1):
    annotations = fold1_gold[doc_id]

    encoding = tokenizer.encode(text, add_special_tokens=False)
    input_ids = encoding.ids
    offsets = encoding.offsets

    # Match the original chunking: 1024 tokens, stride 768.
    doc_pred_ids = [0] * len(input_ids)
    doc_pred_conf = [float("-inf")] * len(input_ids)

    for chunk_start in range(0, len(input_ids), 768):
        chunk_end = min(chunk_start + 1024, len(input_ids))
        chunk_ids = input_ids[chunk_start:chunk_end]

        if not chunk_ids:
            break

        x = torch.tensor([chunk_ids], dtype=torch.long)

        with torch.no_grad():
            logits = model4(x)[0]
            probs = torch.softmax(logits, dim=-1)
            conf, pred = probs.max(dim=-1)

        for j, (p, c) in enumerate(zip(pred.tolist(), conf.tolist())):
            global_idx = chunk_start + j
            if c > doc_pred_conf[global_idx]:
                doc_pred_ids[global_idx] = p
                doc_pred_conf[global_idx] = c

        if chunk_end == len(input_ids):
            break

    pred_spans = set(
        bio_to_char_spans(text, offsets, doc_pred_ids)
    )
    gold_spans = {
        (int(s), int(e), str(label))
        for s, e, label, *_ in annotations
    }

    total_gold += len(gold_spans)
    total_pred += len(pred_spans)
    total_exact += len(gold_spans & pred_spans)

    for label in ENTITY_LABELS:
        gold_label = {s for s in gold_spans if s[2] == label}
        pred_label = {s for s in pred_spans if s[2] == label}
        per_label[label]["tp"] += len(gold_label & pred_label)
        per_label[label]["fp"] += len(pred_label - gold_label)
        per_label[label]["fn"] += len(gold_label - pred_label)

    if doc_num % 5 == 0 or doc_num == len(fold1_text):
        print(f"Processed {doc_num}/{len(fold1_text)} documents")

print("\n===== STRICT SPAN VALIDATION =====")
print("Gold spans:", total_gold)
print("Predicted spans:", total_pred)
print("Exact matches:", total_exact)

precision = total_exact / total_pred if total_pred else 0.0
recall = total_exact / total_gold if total_gold else 0.0
micro_f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0.0
)

label_f1s = []
print("\nPer-label results:")
for label in ENTITY_LABELS:
    m = per_label[label]
    p = m["tp"] / (m["tp"] + m["fp"]) if m["tp"] + m["fp"] else 0.0
    r = m["tp"] / (m["tp"] + m["fn"]) if m["tp"] + m["fn"] else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    label_f1s.append(f1)
    print(
        f"{label:12s} P={p:.4f} R={r:.4f} F1={f1:.4f} "
        f"TP={m['tp']} FP={m['fp']} FN={m['fn']}"
    )

print("\nMicro precision:", round(precision, 4))
print("Micro recall:   ", round(recall, 4))
print("Micro F1:       ", round(micro_f1, 4))
print("Strict Macro-F1:", round(sum(label_f1s) / len(label_f1s), 4))

In [ ]:
from pathlib import Path
from tokenizers import Tokenizer

TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json"
)

print("Tokenizer file exists:", TOKENIZER_PATH.exists())

assert TOKENIZER_PATH.exists(), (
    f"Tokenizer not found: {TOKENIZER_PATH}"
)

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("Tokenizer loaded:", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.get_vocab_size())

# Quick offset test
sample_text = next(iter(fold1_text.values()))
encoding = tokenizer.encode(sample_text, add_special_tokens=False)

print("Sample token count:", len(encoding.ids))
print("Sample offset count:", len(encoding.offsets))
print("Tokenizer initialization PASSED")

In [ ]:
import json
from pathlib import Path
from tokenizers import Tokenizer

ROOT = Path("/kaggle/input/datasets/aq1de3")

TOKENIZER_PATH = ROOT / "tokenizer" / "tokenizer.json"
FOLD1_TEXT_PATH = ROOT / "fold1-tex" / "fold1-text.json"
FOLD1_GOLD_PATH = ROOT / "fold1-gold1" / "fold1-gold.json"

for path in [TOKENIZER_PATH, FOLD1_TEXT_PATH, FOLD1_GOLD_PATH]:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")
    assert path.exists(), f"File not found: {path}"

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

with open(FOLD1_TEXT_PATH, "r", encoding="utf-8") as f:
    fold1_text = json.load(f)

with open(FOLD1_GOLD_PATH, "r", encoding="utf-8") as f:
    fold1_gold = json.load(f)

print("\nTokenizer vocabulary:", tokenizer.get_vocab_size())
print("Fold 1 text documents:", len(fold1_text))
print("Fold 1 gold documents:", len(fold1_gold))
print("Matching document IDs:", set(fold1_text) == set(fold1_gold))

sample_id = next(iter(fold1_text))
encoding = tokenizer.encode(
    fold1_text[sample_id],
    add_special_tokens=False
)

print("Sample document:", sample_id)
print("Sample token count:", len(encoding.ids))
print("Sample offset count:", len(encoding.offsets))

assert set(fold1_text) == set(fold1_gold)
assert len(encoding.ids) == len(encoding.offsets)
assert "model4" in globals(), "The model is missing; reload it before validation."

print("\nVALIDATION DATA READY")

In [ ]:

import torch
from collections import defaultdict

# --------------------------------------------------
# 1. Labels — preserve the trained classifier's order
# --------------------------------------------------
ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]

BIO_LABELS = ["O"]
for label in ENTITY_LABELS:
    BIO_LABELS.extend([f"B-{label}", f"I-{label}"])

ID2LABEL = {i: label for i, label in enumerate(BIO_LABELS)}

assert model4.classifier.out_features == len(BIO_LABELS) == 25
assert set(fold1_text) == set(fold1_gold)

# --------------------------------------------------
# 2. BIO tags -> character spans
# --------------------------------------------------
def bio_to_char_spans(text, offsets, pred_ids):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    def close_entity():
        nonlocal current_label, current_start, current_end

        if current_label is not None:
            start, end = current_start, current_end

            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1

            if start < end:
                spans.append((start, end, current_label))

        current_label = None
        current_start = None
        current_end = None

    for (start, end), pred_id in zip(offsets, pred_ids):
        if start == end:
            continue

        tag = ID2LABEL[int(pred_id)]

        if tag == "O":
            close_entity()
            continue

        prefix, entity_type = tag.split("-", 1)

        if prefix == "B" or current_label != entity_type:
            close_entity()
            current_label = entity_type
            current_start = start
            current_end = end
        else:
            current_end = end

    close_entity()
    return spans


# --------------------------------------------------
# 3. Evaluate complete documents
# --------------------------------------------------
device = next(model4.parameters()).device
model4.eval()

per_label = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
total_gold = 0
total_pred = 0
total_exact = 0

MAX_TOKENS = 1024
STRIDE = 768

for doc_num, (doc_id, text) in enumerate(fold1_text.items(), 1):
    encoding = tokenizer.encode(text, add_special_tokens=False)
    input_ids = encoding.ids
    offsets = encoding.offsets

    # Aggregate overlapping chunk predictions.
    # Average probabilities for tokens appearing in multiple chunks.
    prob_sum = [None] * len(input_ids)
    prob_count = [0] * len(input_ids)

    for chunk_start in range(0, len(input_ids), STRIDE):
        chunk_end = min(chunk_start + MAX_TOKENS, len(input_ids))
        chunk_ids = input_ids[chunk_start:chunk_end]

        if not chunk_ids:
            break

        x = torch.tensor(
            [chunk_ids], dtype=torch.long, device=device
        )

        with torch.inference_mode():
            logits = model4(x)[0]
            probs = torch.softmax(logits, dim=-1).cpu()

        for j in range(len(chunk_ids)):
            idx = chunk_start + j
            if prob_sum[idx] is None:
                prob_sum[idx] = probs[j].clone()
            else:
                prob_sum[idx] += probs[j]
            prob_count[idx] += 1

        if chunk_end >= len(input_ids):
            break

    pred_ids = [
        int((prob_sum[i] / prob_count[i]).argmax().item())
        for i in range(len(input_ids))
    ]

    predicted = set(bio_to_char_spans(text, offsets, pred_ids))
    gold = {
        (int(span[0]), int(span[1]), str(span[2]))
        for span in fold1_gold[doc_id]
    }

    total_gold += len(gold)
    total_pred += len(predicted)
    total_exact += len(gold & predicted)

    for label in ENTITY_LABELS:
        g = {s for s in gold if s[2] == label}
        p = {s for s in predicted if s[2] == label}

        per_label[label]["tp"] += len(g & p)
        per_label[label]["fp"] += len(p - g)
        per_label[label]["fn"] += len(g - p)

    if doc_num % 5 == 0 or doc_num == len(fold1_text):
        print(f"Processed {doc_num}/{len(fold1_text)} documents")


# --------------------------------------------------
# 4. Strict span-level metrics
# --------------------------------------------------
print("\n========== STRICT SPAN VALIDATION ==========")
print("Gold spans:     ", total_gold)
print("Predicted spans:", total_pred)
print("Exact matches:  ", total_exact)

micro_precision = total_exact / total_pred if total_pred else 0.0
micro_recall = total_exact / total_gold if total_gold else 0.0
micro_f1 = (
    2 * micro_precision * micro_recall /
    (micro_precision + micro_recall)
    if micro_precision + micro_recall else 0.0
)

label_f1s = []

print("\nPer-label metrics:")
print(f"{'LABEL':12s} {'PRECISION':>10s} {'RECALL':>10s} "
      f"{'F1':>10s} {'TP':>6s} {'FP':>6s} {'FN':>6s}")

for label in ENTITY_LABELS:
    m = per_label[label]
    tp, fp, fn = m["tp"], m["fp"], m["fn"]

    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0

    label_f1s.append(f1)

    print(f"{label:12s} {p:10.4f} {r:10.4f} {f1:10.4f} "
          f"{tp:6d} {fp:6d} {fn:6d}")

strict_macro_f1 = sum(label_f1s) / len(ENTITY_LABELS)

print("\nMicro precision:", round(micro_precision, 4))
print("Micro recall:   ", round(micro_recall, 4))
print("Micro F1:       ", round(micro_f1, 4))
print("Strict Macro-F1:", round(strict_macro_f1, 4))
print("\nValidation complete. No checkpoint or submission was modified.")


In [ ]:

from collections import Counter
import torch, json, os

# 1. Inspect checkpoint metadata and classifier
ckpt_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/a4_ner_exp4_lorma_epoch3.pt"
)
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

print("=== CHECKPOINT ===")
print("Metadata keys:", [k for k in ckpt.keys() if k != "model_state_dict"])
for k, v in ckpt.items():
    if k != "model_state_dict":
        print(k, "=", str(v)[:500])

print("\n=== LABEL MAPPING ===")
print("Number of output classes:", model4.classifier.out_features)
print("ID2LABEL:", list(enumerate(ID2LABEL)))

# 2. Inspect the first document and its gold BIO tags
doc_id = next(iter(fold1_text))
doc = fold1_text[doc_id]
gold_doc = fold1_gold[doc_id]

print("\n=== INPUT STRUCTURE ===")
print("Document ID:", doc_id)
print("Document type:", type(doc).__name__)
if isinstance(doc, dict):
    print("Document keys:", list(doc.keys()))
    for k, v in doc.items():
        if isinstance(v, (str, list, tuple)):
            print(k, "type:", type(v).__name__,
                  "length:", len(v), "sample:", str(v[:3])[:250])

print("\nGold annotation type:", type(gold_doc).__name__)
print("Gold annotation sample:", str(gold_doc[:5])[:1000])

# 3. Inspect raw model predictions on the first 8 chunks.
# Uses the same tokenizer and model already loaded in your notebook.
if isinstance(doc, dict):
    text = doc.get("text", doc.get("tokens", None))
else:
    text = doc

if isinstance(text, str):
    enc = tokenizer.encode(text)
    token_ids = enc.ids
    offsets = enc.offsets
elif isinstance(text, list) and text and isinstance(text[0], str):
    # If source stores pre-tokenized words, tokenize each word separately.
    encodings = [tokenizer.encode(w, add_special_tokens=False) for w in text]
    token_ids = [i for e in encodings for i in e.ids]
    offsets = []
    print("Input appears pre-tokenized; flattened subword count:", len(token_ids))
else:
    token_ids = None
    print("Could not infer raw text field automatically; inspect input keys above.")

if token_ids is not None:
    pred_counter = Counter()
    confidence_values = []
    model4.eval()

    with torch.no_grad():
        for start in range(0, min(len(token_ids), 8 * 512), 512):
            ids = torch.tensor([token_ids[start:start+512]], dtype=torch.long)
            logits = model4(ids)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]
            if isinstance(logits, dict):
                logits = logits.get("logits", next(iter(logits.values())))

            probs = torch.softmax(logits, dim=-1)
            pred = probs.argmax(-1)[0].tolist()
            pred_counter.update(pred)
            confidence_values.extend(probs.max(-1).values[0].tolist())

    print("\n=== PREDICTIONS ON FIRST CHUNKS ===")
    print("Predicted class counts:")
    for idx, count in sorted(pred_counter.items()):
        label = ID2LABEL[idx] if idx < len(ID2LABEL) else "UNKNOWN"
        print(f"{idx:2d} {label:12s} {count}")

    print("Mean max probability:",
          round(sum(confidence_values) / max(1, len(confidence_values)), 4))


In [ ]:

from collections import Counter
import torch

# Rebuild the BIO mapping explicitly from labels.txt.
labels_path = "/kaggle/input/datasets/aq1de3/labelses/labels.txt"

with open(labels_path, "r", encoding="utf-8") as f:
    entity_labels = [line.strip() for line in f if line.strip()]

BIO_LABELS = ["O"]
for label in entity_labels:
    BIO_LABELS.extend([f"B-{label}", f"I-{label}"])

LABEL2ID = {label: i for i, label in enumerate(BIO_LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

print("Labels file order:", entity_labels)
print("BIO labels:", BIO_LABELS)
print("Model output classes:", model4.classifier.out_features)
assert len(BIO_LABELS) == model4.classifier.out_features, (
    "Number of BIO labels does not match classifier output size!"
)

# Use the first document.
doc_id = next(iter(fold1_text))
text = fold1_text[doc_id]
gold_spans = fold1_gold[doc_id]

enc = tokenizer.encode(text)
token_ids = enc.ids
offsets = enc.offsets

# Convert gold character spans into token-level BIO IDs.
gold_bio = [LABEL2ID["O"]] * len(token_ids)

for start, end, label, entity_text in gold_spans:
    if label not in LABEL2ID:
        print("Unknown gold label:", label)
        continue

    covered = [
        i for i, (ts, te) in enumerate(offsets)
        if te > ts and te > start and ts < end
    ]
    if not covered:
        continue

    gold_bio[covered[0]] = LABEL2ID[f"B-{label}"]
    for i in covered[1:]:
        gold_bio[i] = LABEL2ID[f"I-{label}"]

# Run inference using the same tokenizer output, with no training or updates.
pred_bio = []
model4.eval()

with torch.no_grad():
    for start in range(0, len(token_ids), 512):
        ids = torch.tensor(
            [token_ids[start:start + 512]], dtype=torch.long
        )
        logits = model4(ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]
        if isinstance(logits, dict):
            logits = logits["logits"]

        pred_bio.extend(logits.argmax(-1)[0].cpu().tolist())

pred_bio = pred_bio[:len(gold_bio)]

print("\nToken counts:", len(token_ids), len(gold_bio), len(pred_bio))
print("Gold BIO distribution:")
for idx, count in sorted(Counter(gold_bio).items()):
    print(f"{idx:2d} {ID2LABEL[idx]:12s} {count}")

print("\nPredicted BIO distribution:")
for idx, count in sorted(Counter(pred_bio).items()):
    print(f"{idx:2d} {ID2LABEL.get(idx, 'UNKNOWN'):12s} {count}")

print("\nGold vs predicted token accuracy:",
      sum(g == p for g, p in zip(gold_bio, pred_bio)) / max(1, len(gold_bio)))

print("\nGold/predicted confusion counts (non-O classes only):")
pairs = Counter(
    (ID2LABEL[g], ID2LABEL.get(p, str(p)))
    for g, p in zip(gold_bio, pred_bio)
    if g != 0 or p != 0
)
for (g, p), n in pairs.most_common(30):
    print(f"Gold={g:12s} Pred={p:12s} Count={n}")


In [ ]:

from collections import Counter
import torch

# Correct BIO label mapping
BIO_LABELS = ["O"]
for label in entity_labels:
    BIO_LABELS.extend([f"B-{label}", f"I-{label}"])

LABEL2ID = {label: i for i, label in enumerate(BIO_LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

doc_id = next(iter(fold1_text))
text = fold1_text[doc_id]
gold_spans = fold1_gold[doc_id]

enc = tokenizer.encode(text)
token_ids = enc.ids
offsets = enc.offsets

# Correctly create gold BIO tags from character spans.
gold_bio = [LABEL2ID["O"]] * len(token_ids)
unknown_labels = Counter()

for start, end, label, entity_text in gold_spans:
    if label not in entity_labels:
        unknown_labels[label] += 1
        continue

    covered = [
        i for i, (ts, te) in enumerate(offsets)
        if te > ts and te > start and ts < end
    ]
    if not covered:
        continue

    gold_bio[covered[0]] = LABEL2ID[f"B-{label}"]
    for i in covered[1:]:
        gold_bio[i] = LABEL2ID[f"I-{label}"]

# Model predictions
pred_bio = []
model4.eval()

with torch.no_grad():
    for start in range(0, len(token_ids), 512):
        ids = torch.tensor(
            [token_ids[start:start + 512]], dtype=torch.long
        )
        logits = model4(ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]
        if isinstance(logits, dict):
            logits = logits["logits"]
        pred_bio.extend(logits.argmax(-1)[0].cpu().tolist())

pred_bio = pred_bio[:len(gold_bio)]

correct = sum(g == p for g, p in zip(gold_bio, pred_bio))
gold_entities = sum(x != 0 for x in gold_bio)
pred_entities = sum(x != 0 for x in pred_bio)

print("Document:", doc_id)
print("Tokens:", len(token_ids))
print("Gold spans:", len(gold_spans))
print("Gold non-O tokens:", gold_entities)
print("Predicted non-O tokens:", pred_entities)
print("Unknown gold labels:", dict(unknown_labels))
print("Token accuracy:", round(correct / max(1, len(gold_bio)), 4))

print("\nGold BIO distribution:")
for idx, n in sorted(Counter(gold_bio).items()):
    print(f"{idx:2d} {ID2LABEL[idx]:12s} {n}")

print("\nPredicted BIO distribution:")
for idx, n in sorted(Counter(pred_bio).items()):
    print(f"{idx:2d} {ID2LABEL[idx]:12s} {n}")

# Entity-token recall: what fraction of gold entity tokens receive
# any non-O prediction? This is diagnostic, not strict span F1.
gold_entity_positions = [i for i, g in enumerate(gold_bio) if g != 0]
covered = sum(pred_bio[i] != 0 for i in gold_entity_positions)
print("\nGold entity tokens predicted as non-O:",
      f"{covered}/{len(gold_entity_positions)}")


In [ ]:

# Check whether the original training/validation chunks still exist.
print("a4_train_chunks exists:", "a4_train_chunks" in globals())
print("a4_val_chunks exists:", "a4_val_chunks" in globals())

if "a4_train_chunks" in globals() and a4_train_chunks:
    c = a4_train_chunks[0]
    print("\nTRAIN CHUNK")
    print("input_ids length:", len(c.input_ids))
    print("offsets length:", len(c.offsets))
    print("bio_ids length:", len(c.bio_ids))
    print("first 40 input IDs:", c.input_ids[:40])
    print("first 40 BIO IDs:", c.bio_ids[:40])

if "a4_val_chunks" in globals() and a4_val_chunks:
    c = a4_val_chunks[0]
    print("\nVALIDATION CHUNK")
    print("input_ids length:", len(c.input_ids))
    print("offsets length:", len(c.offsets))
    print("bio_ids length:", len(c.bio_ids))
    print("first 40 input IDs:", c.input_ids[:40])
    print("first 40 BIO IDs:", c.bio_ids[:40])

print("\nCurrent model input call:")
print("model4 forward signature:", __import__("inspect").signature(model4.forward))


In [ ]:

import torch
from collections import Counter

doc_id = next(iter(fold1_text))
text = fold1_text[doc_id]
gold_spans = fold1_gold[doc_id]

enc = tokenizer.encode(text)
ids = enc.ids[:512]
offsets = enc.offsets[:512]

# Rebuild the BIO mapping.
BIO_LABELS = ["O"]
for label in entity_labels:
    BIO_LABELS += [f"B-{label}", f"I-{label}"]
LABEL2ID = {label: i for i, label in enumerate(BIO_LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

gold = [0] * len(ids)
for start, end, label, entity_text in gold_spans:
    covered = [
        i for i, (ts, te) in enumerate(offsets)
        if te > ts and ts < end and te > start
    ]
    if not covered or label not in entity_labels:
        continue
    gold[covered[0]] = LABEL2ID[f"B-{label}"]
    for i in covered[1:]:
        gold[i] = LABEL2ID[f"I-{label}"]

model4.eval()
with torch.no_grad():
    input_ids = torch.tensor([ids], dtype=torch.long)
    logits = model4(input_ids=input_ids)
    pred = logits.argmax(-1)[0].tolist()

print("First 512 tokens")
print("Gold distribution:",
      {ID2LABEL[k]: v for k, v in Counter(gold).items()})
print("Prediction distribution:",
      {ID2LABEL[k]: v for k, v in Counter(pred).items()})

gold_entities = [i for i, label in enumerate(gold) if label != 0]
print("Gold entity tokens:", len(gold_entities))
print("Gold entity tokens predicted non-O:",
      sum(pred[i] != 0 for i in gold_entities))

print("\nFirst 30 positions: token ID, gold label, predicted label")
for i in range(min(30, len(ids))):
    print(ids[i], ID2LABEL[gold[i]], ID2LABEL[pred[i]])


In [ ]:

import inspect

for name in ["LORMALinear", "A4Attention", "A4EncoderLayer",
             "A4EncoderForNER", "A4NER"]:
    obj = globals().get(name)
    print("\n" + "=" * 65)
    print(name, "exists:", obj is not None)
    if obj is not None:
        try:
            print(inspect.getsource(obj))
        except Exception as e:
            print("Source unavailable:", repr(e))

print("\nModel parameter summary:")
print("Total parameters:", sum(p.numel() for p in model4.parameters()))
print("Trainable parameters:", sum(
    p.numel() for p in model4.parameters() if p.requires_grad
))
print("Device:", next(model4.parameters()).device)
print("Training mode:", model4.training)


In [ ]:

from collections import defaultdict

groups = defaultdict(lambda: [0, 0])
print("Parameter name | Shape | Trainable")

for name, p in model4.named_parameters():
    group = (
        "LORMA adapter" if "lorma_" in name.lower()
        else "Classifier" if "classifier" in name.lower()
        else "Encoder/base" 
    )
    groups[group][0] += p.numel()
    if p.requires_grad:
        groups[group][1] += p.numel()
    if "lorma_" in name.lower() or "classifier" in name.lower():
        print(name, tuple(p.shape), p.requires_grad)

print("\n=== PARAMETER COUNTS ===")
for group, (total, trainable) in groups.items():
    print(
        f"{group:16s} total={total:,} "
        f"trainable={trainable:,} frozen={total-trainable:,}"
    )

print("\n=== LORMA MODULES ===")
for name, module in model4.named_modules():
    if isinstance(module, LORMALinear):
        print(
            name,
            "rank =", tuple(module.lorma_A.shape),
            "alpha/scaling =", getattr(module, "scaling", "not exposed"),
            "dropout =", getattr(module, "dropout", "not exposed")
        )

print("\nAudit only — no parameters or checkpoint changed.")


In [ ]:

import torch
from pathlib import Path

lorma_path = Path(
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/a4_ner_exp4_lorma_epoch3.pt"
)
base_path = Path(
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)

lorma_ckpt = torch.load(
    lorma_path, map_location="cpu", weights_only=False
)
base_ckpt = torch.load(
    base_path, map_location="cpu", weights_only=False
)

print("=== TOP-LEVEL KEYS ===")
print("LORMA:", list(lorma_ckpt.keys()))
print("Phase 1 A4:", list(base_ckpt.keys()))

def find_state_dict(obj):
    if not isinstance(obj, dict):
        return None
    for key in ("model_state_dict", "state_dict", "model", "encoder_state_dict"):
        if key in obj and isinstance(obj[key], dict):
            return obj[key]
    # If this is already a raw state dict, check for tensor values.
    if obj and all(torch.is_tensor(v) for v in obj.values()):
        return obj
    return None

lorma_sd = find_state_dict(lorma_ckpt)
base_sd = find_state_dict(base_ckpt)

print("\nState dict found:", lorma_sd is not None, base_sd is not None)

if lorma_sd is None or base_sd is None:
    print("\nCould not automatically locate both state dictionaries.")
    print("Phase 1 sample keys:", list(base_ckpt.keys())[:30])
else:
    print("LORMA state keys:", len(lorma_sd))
    print("Base state keys:", len(base_sd))
    print("\nPhase 1 sample state keys:")
    for k in list(base_sd.keys())[:25]:
        v = base_sd[k]
        print(k, tuple(v.shape) if torch.is_tensor(v) else type(v).__name__)

    # Compare only identically named tensors with identical shapes.
    common = []
    mismatched_shapes = []
    for k, a in lorma_sd.items():
        if k in base_sd and torch.is_tensor(a) and torch.is_tensor(base_sd[k]):
            b = base_sd[k]
            if a.shape == b.shape:
                common.append((k, a, b))
            else:
                mismatched_shapes.append((k, tuple(a.shape), tuple(b.shape)))

    print("\nSame-name, same-shape tensors:", len(common))
    print("Same-name tensors with different shapes:", len(mismatched_shapes))

    if mismatched_shapes:
        print("Shape mismatch examples:", mismatched_shapes[:10])

    exact = []
    different = []
    for k, a, b in common:
        if torch.equal(a, b):
            exact.append(k)
        else:
            different.append(k)

    print("Exactly equal tensors:", len(exact))
    print("Different tensors:", len(different))
    print("\nDifferent tensor examples:")
    for k in different[:30]:
        print(k)

    print("\nExact-match examples:")
    for k in exact[:20]:
        print(k)

print("\nComparison only — no checkpoint or model changed.")


In [ ]:

import torch

base_sd = base_ckpt["model_state_dict"]
lorma_sd = lorma_ckpt["model_state_dict"]

comparisons = []

for key, lora_tensor in lorma_sd.items():
    mapped = key

    # LORMA model wraps encoder parameters in "encoder."
    if mapped.startswith("encoder."):
        mapped = mapped[len("encoder."):]

    # Adapter projections store original linear weights under "base".
    mapped = mapped.replace(".q_proj.base.", ".q_proj.")
    mapped = mapped.replace(".v_proj.base.", ".v_proj.")

    if mapped in base_sd and torch.is_tensor(lora_tensor):
        base_tensor = base_sd[mapped]
        if base_tensor.shape == lora_tensor.shape:
            comparisons.append((
                key,
                mapped,
                torch.equal(lora_tensor, base_tensor),
                (lora_tensor.float() - base_tensor.float()).abs().max().item()
            ))

print("Comparable tensors:", len(comparisons))
print("Exactly equal:", sum(x[2] for x in comparisons))
print("Different:", sum(not x[2] for x in comparisons))

print("\nComparison details:")
for lora_key, base_key, equal, max_diff in comparisons:
    print(
        f"{lora_key} -> {base_key} | "
        f"equal={equal} | max_abs_diff={max_diff:.8g}"
    )


In [ ]:

from collections import defaultdict
import torch

groups = defaultdict(list)

for lora_key, base_key, equal, max_diff in comparisons:
    if ".lorma_A" in lora_key or ".lorma_B" in lora_key:
        category = "LORMA adapter"
    elif "classifier" in lora_key:
        category = "Classifier"
    elif ".q_proj.base." in lora_key or ".v_proj.base." in lora_key:
        category = "Q/V base projection"
    elif ".attn." in lora_key:
        category = "Other attention parameters"
    elif ".ffn." in lora_key:
        category = "Feed-forward parameters"
    elif ".norm" in lora_key:
        category = "Normalization parameters"
    else:
        category = "Embeddings / positional parameters"

    groups[category].append(max_diff)

print("=== Difference summary ===")
for category, diffs in groups.items():
    print(
        f"{category:32s} | "
        f"tensors={len(diffs):3d} | "
        f"max_diff={max(diffs):.8g} | "
        f"mean_max_diff={sum(diffs)/len(diffs):.8g}"
    )

print("\n=== Checkpoint training metadata ===")
for key in ("experiment", "epoch", "train_loss",
            "train_chunks", "val_chunks",
            "lorma_rank", "lorma_alpha"):
    print(f"{key}: {lorma_ckpt.get(key)}")


In [ ]:

# Inspect optimizer state and current trainable parameters.
# This does not modify model weights.

state = lorma_ckpt["model_state_dict"]
optimizer_state = lorma_ckpt["optimizer_state_dict"]

print("=== Checkpoint structure ===")
print("LORMA adapter tensors:",
      sum("lorma_A" in k or "lorma_B" in k for k in state))
print("Classifier tensors:",
      sum(k.startswith("classifier.") for k in state))
print("Optimizer state entries:",
      len(optimizer_state.get("state", {})))

print("\n=== Current model trainability ===")
trainable = [
    (name, param.numel())
    for name, param in model4.named_parameters()
    if param.requires_grad
]
frozen = [
    (name, param.numel())
    for name, param in model4.named_parameters()
    if not param.requires_grad
]

print("Trainable tensors:", len(trainable))
print("Trainable parameters:", sum(n for _, n in trainable))
print("Frozen tensors:", len(frozen))
print("Frozen parameters:", sum(n for _, n in frozen))

print("\nTrainable parameter names (first 30):")
for name, _ in trainable[:30]:
    print(name)


In [ ]:

from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

print("Exists:", source_path.exists())

if source_path.exists():
    print("Size (bytes):", source_path.stat().st_size)

    with source_path.open("rb") as f:
        beginning = f.read(500)

    print("\nFirst 500 bytes:")
    print(repr(beginning))

    if source_path.stat().st_size == 0:
        print("\nDIAGNOSIS: The file is empty.")
    elif beginning.lstrip().startswith((b"{", b"[")):
        print("\nDIAGNOSIS: Looks like JSON; it may be truncated or malformed.")
    else:
        print("\nDIAGNOSIS: The file does not start like a JSON notebook.")


In [ ]:

from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

terms = [
    "A4_CHECKPOINT_PATH",
    "a4_step_22000.pt",
    "lorma_encoder = copy.deepcopy(encoder)",
    "Freeze entire model",
    "optimizer4 =",
]

for term in terms:
    hits = [
        i for i, line in enumerate(lines)
        if term.lower() in line.lower()
    ]

    print(f"\n{'=' * 18} {term} ({len(hits)} matches) {'=' * 18}")

    # Print up to the first 3 matches and their surrounding context.
    for i in hits[:3]:
        start = max(0, i - 15)
        end = min(len(lines), i + 25)

        print(f"\n--- Source lines {start + 1}-{end} ---")
        print("\n".join(
            f"{j + 1}: {lines[j]}"
            for j in range(start, end)
        ))


In [ ]:

from pathlib import Path

lines = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
).read_text(encoding="utf-8", errors="replace").splitlines()

# Show the LORMA setup and the preceding encoder initialization.
target = next(
    i for i, line in enumerate(lines)
    if line.strip() == "lorma_encoder = copy.deepcopy(encoder)"
)

print("=" * 80)
print("CODE BEFORE LORMA INITIALIZATION")
print("=" * 80)

start = max(0, target - 180)
end = min(len(lines), target + 35)

print("\n".join(
    f"{i + 1}: {lines[i]}"
    for i in range(start, end)
))

print("\n" + "=" * 80)
print("ALL ENCODER INITIALIZATION REFERENCES")
print("=" * 80)

for i, line in enumerate(lines):
    if any(term in line for term in [
        "encoder = A4Encoder(",
        "encoder.load_state_dict(",
        "encoder = a4_encoder",
        "encoder = copy.deepcopy(",
    ]):
        print(f"{i + 1}: {line.strip()}")


In [ ]:

from pathlib import Path

lines = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
).read_text(encoding="utf-8", errors="replace").splitlines()

ranges = [
    (8170, 8235),
    (10290, 10375),
    (14865, 14935),
    (27110, 27175),
]

for start, end in ranges:
    print("\n" + "=" * 90)
    print(f"SOURCE LINES {start}-{end}")
    print("=" * 90)

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1}: {lines[i]}")


In [ ]:

import torch
from pathlib import Path

p1_path = Path(
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)
p2_path = Path(
    "/kaggle/input/datasets/aq1de3/a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

p1 = torch.load(p1_path, map_location="cpu", weights_only=False)
p2 = torch.load(p2_path, map_location="cpu", weights_only=False)

s1 = p1["model_state_dict"]
s2 = p2["model_state_dict"]

print("Phase 1 checkpoint:", p1_path.exists())
print("LORMA checkpoint:", p2_path.exists())
print("LORMA metadata:")
for k in ["experiment", "epoch", "train_loss", "train_chunks",
          "val_chunks", "lorma_rank", "lorma_alpha", "lorma_dropout"]:
    print(f"  {k}: {p2.get(k)}")

print("\nLORMA adapter tensors:")
adapters = [
    (k, v) for k, v in s2.items()
    if "lorma_A" in k or "lorma_B" in k
]
print("Count:", len(adapters))
for k, v in adapters[:8]:
    print(" ", k, tuple(v.shape))

print("\nClassifier tensors:")
for k, v in s2.items():
    if "classifier" in k:
        print(" ", k, tuple(v.shape))

# Compare corresponding pretrained base tensors.
comparisons = []
for k2, v2 in s2.items():
    k1 = k2.removeprefix("encoder.")
    k1 = k1.replace(".q_proj.base.", ".q_proj.")
    k1 = k1.replace(".v_proj.base.", ".v_proj.")

    if k1 in s1 and s1[k1].shape == v2.shape:
        if any(x in k2 for x in [
            "token_embedding.", "encoder_pos.",
            "encoder_layers."
        ]) and ".lorma_" not in k2:
            diff = (s1[k1].float() - v2.float()).abs()
            comparisons.append((k2, diff.max().item(), diff.mean().item()))

print("\nMatched base tensors:", len(comparisons))
if comparisons:
    print("Exact matches:", sum(mx == 0 for _, mx, _ in comparisons))
    print("Different tensors:", sum(mx > 0 for _, mx, _ in comparisons))
    print("Largest absolute difference:",
          max(mx for _, mx, _ in comparisons))
    print("Mean of tensor-wise mean differences:",
          sum(mn for _, _, mn in comparisons) / len(comparisons))

print("\nDiagnostic only — no files or model weights changed.")


In [ ]:

import torch
from collections import Counter

print("Model:", type(model4).__name__)
print("Device:", next(model4.parameters()).device)
print("Tokenizer:", type(tokenizer).__name__)
print("Number of validation documents:", len(fold1_text))

# Inspect the first document and its gold annotations.
doc = fold1_text[0]
gold = fold1_gold[0]

print("\nDocument keys:", list(doc.keys()) if isinstance(doc, dict) else type(doc))
print("Gold type:", type(gold).__name__)
print("Gold sample:", gold[:5] if isinstance(gold, list) else gold)

# Confirm the model's output dimensions and label distribution.
model4.eval()
device = next(model4.parameters()).device

if isinstance(doc, dict):
    text = doc.get("text", "")
    doc_id = doc.get("id", doc.get("doc_id", "unknown"))
else:
    text = str(doc)
    doc_id = "unknown"

enc = tokenizer.encode(text)
ids = enc.ids
print("\nDocument ID:", doc_id)
print("Text chars:", len(text))
print("Tokenizer tokens:", len(ids))

# Test a short window to avoid unnecessary CPU load.
window = ids[:256]
input_ids = torch.tensor([window], dtype=torch.long, device=device)
attention_mask = torch.ones_like(input_ids)

with torch.no_grad():
    logits = model4(input_ids, attention_mask=attention_mask)
    probs = logits.softmax(dim=-1)
    pred = logits.argmax(dim=-1)[0].cpu().tolist()

print("\nLogits shape:", tuple(logits.shape))
print("All logits finite:", torch.isfinite(logits).all().item())
print("Predicted class distribution (first 256 tokens):")
print(Counter(pred))
print("Top class probabilities (first 10 tokens):")
print(probs[0, :10].max(dim=-1).values.cpu().tolist())


In [ ]:

from collections import Counter

print("fold1_text type:", type(fold1_text).__name__)
print("fold1_gold type:", type(fold1_gold).__name__)

print("\nText dataset:")
if isinstance(fold1_text, dict):
    print("Number of entries:", len(fold1_text))
    print("First keys:", list(fold1_text.keys())[:5])
    first_key = next(iter(fold1_text))
    print("First key:", repr(first_key))
    print("First value type:", type(fold1_text[first_key]).__name__)
    print("First value preview:", repr(fold1_text[first_key])[:500])
else:
    print("Number of entries:", len(fold1_text))
    print("First entry:", repr(fold1_text[0])[:500])

print("\nGold dataset:")
if isinstance(fold1_gold, dict):
    print("Number of entries:", len(fold1_gold))
    print("First keys:", list(fold1_gold.keys())[:5])
    first_gold_key = next(iter(fold1_gold))
    print("First key:", repr(first_gold_key))
    print("First value type:", type(fold1_gold[first_gold_key]).__name__)
    print("First value preview:", repr(fold1_gold[first_gold_key])[:500])
else:
    print("Number of entries:", len(fold1_gold))
    print("First entry:", repr(fold1_gold[0])[:500])


In [ ]:

import torch
from collections import Counter

doc_id = "115651329"
text = fold1_text[doc_id]
gold = fold1_gold[doc_id]

print("Document ID:", doc_id)
print("Text length:", len(text))
print("Gold spans:", len(gold))
print("Gold examples:", gold[:5])

# Tokenize using the existing tokenizer.
enc = tokenizer.encode(text)
ids = enc.ids
offsets = enc.offsets

print("\nToken count:", len(ids))
print("Offset count:", len(offsets))
print("First 5 token offsets:", offsets[:5])

# Run the current saved LORMA model on a short window.
model4.eval()
device = next(model4.parameters()).device

window_size = min(256, len(ids))
input_ids = torch.tensor(
    [ids[:window_size]], dtype=torch.long, device=device
)
attention_mask = torch.ones_like(input_ids)

with torch.no_grad():
    logits = model4(input_ids, attention_mask=attention_mask)
    probs = logits.softmax(dim=-1)
    predictions = logits.argmax(dim=-1)[0].cpu().tolist()

print("\nLogits shape:", tuple(logits.shape))
print("Finite logits:", torch.isfinite(logits).all().item())
print("Predicted label IDs:", Counter(predictions))
print("Top probabilities for first 10 tokens:",
      probs[0, :10].max(dim=-1).values.cpu().tolist())

# Check that the expected label mapping is available.
print("\nID2LABEL exists:", "ID2LABEL" in globals())
if "ID2LABEL" in globals():
    print("Label mapping sample:",
          list(ID2LABEL.items())[:8])


In [ ]:

import torch
from collections import Counter

doc_id = "115651329"
text = fold1_text[doc_id]
gold = fold1_gold[doc_id]

enc = tokenizer.encode(text)
ids = enc.ids
offsets = enc.offsets

model4.eval()
device = next(model4.parameters()).device

# Use the same 1024-token windows and 768-token stride.
max_len = 1024
stride = 768
all_logits = torch.zeros(len(ids), 25)
counts = torch.zeros(len(ids))

with torch.no_grad():
    for start in range(0, len(ids), stride):
        end = min(start + max_len, len(ids))
        chunk_ids = torch.tensor(
            [ids[start:end]], dtype=torch.long, device=device
        )
        mask = torch.ones_like(chunk_ids)

        out = model4(chunk_ids, attention_mask=mask)[0].cpu()
        all_logits[start:end] += out
        counts[start:end] += 1

        if end == len(ids):
            break

all_logits /= counts.clamp_min(1).unsqueeze(-1)
pred_ids = all_logits.argmax(dim=-1).tolist()
pred_labels = [ID2LABEL[i] for i in pred_ids]

# Decode BIO predictions into character spans.
pred_spans = []
active_label = None
active_start = None
active_end = None

for i, label in enumerate(pred_labels):
    token_start, token_end = offsets[i]

    if token_end <= token_start:
        continue

    if label == "O":
        if active_label is not None:
            pred_spans.append(
                [active_start, active_end, active_label,
                 text[active_start:active_end]]
            )
            active_label = None
        continue

    prefix, entity_type = label.split("-", 1)

    if prefix == "B" or active_label != entity_type:
        if active_label is not None:
            pred_spans.append(
                [active_start, active_end, active_label,
                 text[active_start:active_end]]
            )
        active_label = entity_type
        active_start = token_start
        active_end = token_end
    else:
        active_end = token_end

if active_label is not None:
    pred_spans.append(
        [active_start, active_end, active_label,
         text[active_start:active_end]]
    )

def normalize_span(span):
    start, end, label, _ = span
    return (start, end, label)

gold_set = {normalize_span(x) for x in gold}
pred_set = {normalize_span(x) for x in pred_spans}
tp = len(gold_set & pred_set)

print("Gold spans:", len(gold_set))
print("Predicted spans:", len(pred_set))
print("Exact span matches:", tp)
print("False positives:", len(pred_set - gold_set))
print("False negatives:", len(gold_set - pred_set))

print("\nGold examples:")
for x in gold[:12]:
    print(x)

print("\nPredicted examples:")
for x in pred_spans[:20]:
    print(x)

print("\nPredicted BIO label counts:")
print(Counter(pred_labels))


In [ ]:

doc_id = "115651329"
text = fold1_text[doc_id]
gold = fold1_gold[doc_id]

enc = tokenizer.encode(text)
offsets = enc.offsets
tokens = enc.tokens

print("Document:", doc_id)
print("Text length:", len(text))
print("Token count:", len(offsets))

for start, end, label, entity_text in gold[:12]:
    overlapping = [
        (i, offsets[i], tokens[i])
        for i, (ts, te) in enumerate(offsets)
        if te > start and ts < end
    ]

    exact_start = any(ts == start for ts, te in offsets)
    exact_end = any(te == end for ts, te in offsets)

    print("\n" + "=" * 65)
    print("Gold:", repr(entity_text), label, (start, end))
    print("Text at offsets:", repr(text[start:end]))
    print("Exact token start:", exact_start)
    print("Exact token end:", exact_end)
    print("Overlapping tokens:")
    for item in overlapping[:20]:
        print(" ", item)


In [ ]:

from collections import Counter

doc_id = "115651329"
text = fold1_text[doc_id]
gold = fold1_gold[doc_id]

enc = tokenizer.encode(text)
offsets = enc.offsets
tokens = enc.tokens

label_to_id = {v: k for k, v in ID2LABEL.items()}
bio_ids = [label_to_id["O"]] * len(offsets)

# Recreate the likely overlap-based BIO assignment.
for start, end, entity_type, entity_text in gold:
    hit = [
        i for i, (ts, te) in enumerate(offsets)
        if te > start and ts < end
    ]
    if not hit:
        continue

    bio_ids[hit[0]] = label_to_id[f"B-{entity_type}"]
    for i in hit[1:]:
        bio_ids[i] = label_to_id[f"I-{entity_type}"]

print("Gold entities:", len(gold))
print("Generated BIO counts:",
      Counter(ID2LABEL[i] for i in bio_ids))

print("\nFirst 12 entities: token labels and boundaries")
for start, end, entity_type, entity_text in gold[:12]:
    hit = [
        i for i, (ts, te) in enumerate(offsets)
        if te > start and ts < end
    ]
    print("\nEntity:", repr(entity_text), entity_type, (start, end))
    for i in hit:
        print(
            f"  token={i:4d} offset={offsets[i]} "
            f"text={tokens[i]!r} label={ID2LABEL[bio_ids[i]]}"
        )


In [ ]:

from pathlib import Path

lines = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
).read_text(encoding="utf-8", errors="replace").splitlines()

terms = [
    "def build_a4_chunks",
    "a4_train_only_chunks",
    "ignore_index=-100",
    "label_ids",
    "entity_start",
    "token_start",
    "char_start",
    "BIO",
]

for term in terms:
    hits = [
        i for i, line in enumerate(lines)
        if term.lower() in line.lower()
    ]

    print(f"\n{'=' * 25}\n{term}: {len(hits)} matches")

    for i in hits[:4]:
        start = max(0, i - 12)
        end = min(len(lines), i + 28)

        print(f"\n--- Source lines {start + 1}-{end} ---")
        print("\n".join(
            f"{j + 1}: {lines[j]}"
            for j in range(start, end)
        ))


In [ ]:

from pathlib import Path

lines = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
).read_text(encoding="utf-8", errors="replace").splitlines()

# Print the original A4 chunk-building function.
start, end = 9175, 9275

print("=" * 80)
print("ORIGINAL A4 CHUNK BUILDER")
print("=" * 80)
print("\n".join(
    f"{i + 1}: {lines[i]}"
    for i in range(start - 1, min(end, len(lines)))
))

# Find the definitions of the helper functions it calls.
for term in ["def tokenize_document", "def normalize_gold", "def build_gold_token_labels"]:
    hits = [
        i for i, line in enumerate(lines)
        if term in line
    ]

    print(f"\n{'=' * 25}\n{term}: {len(hits)} occurrence(s)")

    for i in hits[:3]:
        a = max(0, i - 3)
        b = min(len(lines), i + 45)
        print("\n".join(
            f"{j + 1}: {lines[j]}"
            for j in range(a, b)
        ))


In [ ]:

import torch

p2 = torch.load(
    "/kaggle/input/datasets/aq1de3/a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt",
    map_location="cpu",
    weights_only=False
)

state = p2["model_state_dict"]

print("=" * 70)
print("LABEL MAPPING")
print("=" * 70)

print("Number of labels:", len(ID2LABEL))
for i in range(len(ID2LABEL)):
    print(i, "->", ID2LABEL[i])

print("\n" + "=" * 70)
print("CURRENT MODEL CLASSIFIER")
print("=" * 70)

print("Weight shape:", tuple(model4.classifier.weight.shape))
print("Bias shape:", tuple(model4.classifier.bias.shape))

print("\n" + "=" * 70)
print("SAVED CHECKPOINT CLASSIFIER")
print("=" * 70)

for key in ["classifier.weight", "classifier.bias"]:
    value = state.get(key)
    print(key, None if value is None else tuple(value.shape))

if "classifier.weight" in state:
    print(
        "Saved classifier finite:",
        torch.isfinite(state["classifier.weight"]).all().item()
    )

print("\n" + "=" * 70)
print("CURRENT VS SAVED CLASSIFIER")
print("=" * 70)

for key, current in [
    ("classifier.weight", model4.classifier.weight.detach().cpu()),
    ("classifier.bias", model4.classifier.bias.detach().cpu()),
]:
    saved = state.get(key)
    if saved is None:
        print(key, "MISSING FROM CHECKPOINT")
    elif saved.shape != current.shape:
        print(key, "SHAPE MISMATCH")
    else:
        print(
            key,
            "exactly equal:", torch.equal(current, saved),
            "max abs difference:",
            (current.float() - saved.float()).abs().max().item()
        )


In [ ]:

import torch

state = p2["model_state_dict"]

print("=" * 65)
print("CHECKPOINT AND MODEL STATE")
print("=" * 65)

print("Checkpoint keys:", list(p2.keys()))

# Compare every checkpoint tensor against the loaded model.
current_state = model4.state_dict()
missing = [k for k in state if k not in current_state]
shape_mismatch = [
    (k, tuple(state[k].shape), tuple(current_state[k].shape))
    for k in state
    if k in current_state and state[k].shape != current_state[k].shape
]

print("Checkpoint tensors:", len(state))
print("Model tensors:", len(current_state))
print("Missing model keys:", missing[:20], "count =", len(missing))
print("Shape mismatches:", shape_mismatch[:20],
      "count =", len(shape_mismatch))

# Find LORMA parameters in both the checkpoint and loaded model.
lorma_keys = [
    k for k in state
    if "lorma_A" in k or "lorma_B" in k
]

print("\nLORMA parameter tensors:", len(lorma_keys))
for k in lorma_keys[:12]:
    print(
        k,
        "shape =", tuple(state[k].shape),
        "matches loaded model =",
        k in current_state and torch.equal(
            state[k].cpu(), current_state[k].detach().cpu()
        )
    )

print("\nAll checkpoint tensors match loaded model:",
      not missing and not shape_mismatch and
      all(
          torch.equal(state[k].cpu(), current_state[k].detach().cpu())
          for k in state if k in current_state
      ))


In [ ]:

import torch
from collections import Counter

model4.eval()

doc_id = "115651329"
text = fold1_text[doc_id]
token_ids, token_offsets = tokenize_document(text)

device = next(model4.parameters()).device
input_ids = torch.tensor(
    token_ids[:256], dtype=torch.long
).unsqueeze(0).to(device)

print("Model device:", device)
print("Model training mode:", model4.training)
print("Input shape:", tuple(input_ids.shape))

# Normal inference with LORMA enabled.
with torch.no_grad():
    out = model4(input_ids)
    logits = out.logits if hasattr(out, "logits") else out
    probs = torch.softmax(logits, dim=-1)
    pred = logits.argmax(dim=-1)[0]

print("\nLogits shape:", tuple(logits.shape))
print("All logits finite:", torch.isfinite(logits).all().item())
print("Predicted labels:", Counter(pred.cpu().tolist()))
print("Top class probabilities, first 20 tokens:")
print(probs[0, :20].max(dim=-1).values.cpu().tolist())

# Inspect how strongly the trained LORMA branches contribute.
print("\nLORMA adapter norms:")
for name, module in model4.named_modules():
    if hasattr(module, "lorma_A") and hasattr(module, "lorma_B"):
        with torch.no_grad():
            a = module.lorma_A
            b = module.lorma_B
            delta_norm = torch.linalg.matrix_norm(b @ a).item()
            base_norm = torch.linalg.matrix_norm(
                module.base.weight
            ).item()
        print(
            name,
            "||B @ A|| =", round(delta_norm, 5),
            "||base weight|| =", round(base_norm, 5)
        )


In [ ]:

import torch
from collections import Counter

# Recreate the missing helper function.
def tokenize_document(text):
    enc = tokenizer.encode(text)
    return enc.ids, enc.offsets

model4.eval()

doc_id = "115651329"
text = fold1_text[doc_id]
token_ids, token_offsets = tokenize_document(text)

device = next(model4.parameters()).device
input_ids = torch.tensor(
    token_ids[:256], dtype=torch.long
).unsqueeze(0).to(device)

print("Model device:", device)
print("Model training mode:", model4.training)
print("Input shape:", tuple(input_ids.shape))

with torch.no_grad():
    out = model4(input_ids)
    logits = out.logits if hasattr(out, "logits") else out
    probs = torch.softmax(logits, dim=-1)
    pred = logits.argmax(dim=-1)[0]

print("\nLogits shape:", tuple(logits.shape))
print("All logits finite:", torch.isfinite(logits).all().item())
print("Predicted labels:", Counter(pred.cpu().tolist()))
print("Maximum class probabilities, first 20 tokens:")
print(probs[0, :20].max(dim=-1).values.cpu().tolist())

print("\nLORMA adapter norms:")
for name, module in model4.named_modules():
    if hasattr(module, "lorma_A") and hasattr(module, "lorma_B"):
        with torch.no_grad():
            delta_norm = torch.linalg.matrix_norm(
                module.lorma_B @ module.lorma_A
            ).item()
            base_norm = torch.linalg.matrix_norm(
                module.base.weight
            ).item()

        print(
            name,
            "||B @ A|| =", round(delta_norm, 5),
            "||base weight|| =", round(base_norm, 5)
        )


In [ ]:

import torch
from collections import Counter

# Reconstruct the original A4 validation chunking and BIO labels.
MAX_TOKENS = 1024
STRIDE = 768

labels = [
    "O",
    "B-APP", "I-APP",
    "B-RESP", "I-RESP",
    "B-A.COUNSEL", "I-A.COUNSEL",
    "B-R.COUNSEL", "I-R.COUNSEL",
    "B-JUDGE", "I-JUDGE",
    "B-WIT", "I-WIT",
    "B-AUTH", "I-AUTH",
    "B-COURT", "I-COURT",
    "B-STAT", "I-STAT",
    "B-PREC", "I-PREC",
    "B-DATE", "I-DATE",
    "B-CASENO", "I-CASENO"
]
label2id = {label: i for i, label in enumerate(labels)}

model4.eval()
device = next(model4.parameters()).device

total_loss = 0.0
total_tokens = 0
correct_tokens = 0
gold_entity_tokens = 0
pred_entity_tokens = 0
matched_entity_tokens = 0
gold_counts = Counter()
pred_counts = Counter()
chunk_count = 0

with torch.no_grad():
    for doc_id, text in fold1_text.items():
        gold_list = fold1_gold[doc_id]
        enc = tokenizer.encode(text)
        ids_all, offsets_all = enc.ids, enc.offsets

        gold_spans = sorted(
            [(int(x[0]), int(x[1]), str(x[2])) for x in gold_list],
            key=lambda x: (x[0], x[1])
        )

        start = 0
        while start < len(ids_all):
            end = min(start + MAX_TOKENS, len(ids_all))
            ids = ids_all[start:end]
            offsets = offsets_all[start:end]

            valid = [(s, e) for s, e in offsets if e > s]
            char_start = min((s for s, e in valid), default=0)
            char_end = max((e for s, e in valid), default=0)

            chunk_gold = [
                (s, e, lab) for s, e, lab in gold_spans
                if s >= char_start and e <= char_end
            ]

            y = [0] * len(ids)
            for es, ee, lab in chunk_gold:
                covered = [
                    i for i, (ts, te) in enumerate(offsets)
                    if te > ts and ts < ee and te > es
                ]
                if not covered:
                    continue
                y[covered[0]] = label2id["B-" + lab]
                for i in covered[1:]:
                    y[i] = label2id["I-" + lab]

            x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
            target = torch.tensor(y, dtype=torch.long).to(device)

            out = model4(x)
            logits = out.logits if hasattr(out, "logits") else out
            logits = logits[0]

            loss = torch.nn.functional.cross_entropy(
                logits, target, reduction="sum"
            )
            total_loss += loss.item()
            total_tokens += len(y)

            pred = logits.argmax(dim=-1).cpu().tolist()

            for gold_id, pred_id in zip(y, pred):
                gold_counts[gold_id] += 1
                pred_counts[pred_id] += 1

                if gold_id == pred_id:
                    correct_tokens += 1
                if gold_id != 0:
                    gold_entity_tokens += 1
                if pred_id != 0:
                    pred_entity_tokens += 1
                if gold_id != 0 and gold_id == pred_id:
                    matched_entity_tokens += 1

            chunk_count += 1
            if end == len(ids_all):
                break
            start += STRIDE

print("Validation documents:", len(fold1_text))
print("Validation chunks:", chunk_count)
print("Tokens evaluated (includes chunk overlaps):", total_tokens)
print("Mean token cross-entropy:", round(total_loss / max(total_tokens, 1), 5))
print("Token accuracy:", round(correct_tokens / max(total_tokens, 1), 5))
print("Gold entity tokens:", gold_entity_tokens)
print("Predicted entity tokens:", pred_entity_tokens)
print("Correctly predicted entity tokens:", matched_entity_tokens)
print("Entity-token recall:", round(
    matched_entity_tokens / max(gold_entity_tokens, 1), 5
))
print("Predicted non-O fraction:", round(
    pred_entity_tokens / max(total_tokens, 1), 5
))

print("\nMost common gold labels:")
print([(label2id and labels[k], v) for k, v in gold_counts.most_common(12)])

print("\nMost common predicted labels:")
print([(labels[k], v) for k, v in pred_counts.most_common(12)])


In [ ]:

from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
src = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

terms = [
    "bio2id =",
    "ID2LABEL",
    "lorma_encoder = copy.deepcopy(encoder)",
    "A4_LORMA",
    "a4_ner_exp4_lorma_epoch3.pt",
    "lorma_rank",
]

for term in terms:
    hits = [
        i for i, line in enumerate(src)
        if term.lower() in line.lower()
    ]

    print("\n" + "=" * 75)
    print(f"{term}: {len(hits)} occurrence(s)")
    print("=" * 75)

    # Show up to three matches, including surrounding code.
    for i in hits[:3]:
        lo = max(0, i - 8)
        hi = min(len(src), i + 18)
        print(f"\n--- Source lines {lo+1}-{hi} ---")
        print("\n".join(
            f"{j+1}: {src[j]}"
            for j in range(lo, hi)
        ))


In [ ]:

import torch
from collections import Counter

# Preserve the entity-label order used by the original LABELS list.
ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]

# This is the original grouped order used by bio2id.
BIO_LABELS_GROUPED = (
    ["O"]
    + [f"B-{label}" for label in ENTITY_LABELS]
    + [f"I-{label}" for label in ENTITY_LABELS]
)

LABEL2ID_GROUPED = {
    label: i for i, label in enumerate(BIO_LABELS_GROUPED)
}
ID2LABEL_GROUPED = {
    i: label for label, i in LABEL2ID_GROUPED.items()
}

print("Corrected original label mapping:")
for i, label in enumerate(BIO_LABELS_GROUPED):
    print(i, "->", label)

assert len(BIO_LABELS_GROUPED) == model4.classifier.out_features == 25

# Validation with the corrected numeric IDs.
model4.eval()
device = next(model4.parameters()).device

total_loss = 0.0
total_tokens = 0
correct_tokens = 0
gold_entity_tokens = 0
pred_entity_tokens = 0
matched_entity_tokens = 0
gold_counts = Counter()
pred_counts = Counter()
chunk_count = 0

with torch.no_grad():
    for doc_id, text in fold1_text.items():
        gold_list = fold1_gold[doc_id]
        enc = tokenizer.encode(text)
        ids_all, offsets_all = enc.ids, enc.offsets

        gold_spans = sorted(
            [(int(x[0]), int(x[1]), str(x[2])) for x in gold_list],
            key=lambda x: (x[0], x[1])
        )

        start = 0
        while start < len(ids_all):
            end = min(start + 1024, len(ids_all))
            ids = ids_all[start:end]
            offsets = offsets_all[start:end]

            valid = [(s, e) for s, e in offsets if e > s]
            char_start = min((s for s, e in valid), default=0)
            char_end = max((e for s, e in valid), default=0)

            chunk_gold = [
                (s, e, lab) for s, e, lab in gold_spans
                if s >= char_start and e <= char_end
            ]

            y = [0] * len(ids)
            for es, ee, lab in chunk_gold:
                covered = [
                    i for i, (ts, te) in enumerate(offsets)
                    if te > ts and ts < ee and te > es
                ]
                if not covered:
                    continue
                y[covered[0]] = LABEL2ID_GROUPED["B-" + lab]
                for i in covered[1:]:
                    y[i] = LABEL2ID_GROUPED["I-" + lab]

            x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
            target = torch.tensor(y, dtype=torch.long).to(device)

            out = model4(x)
            logits = out.logits if hasattr(out, "logits") else out
            logits = logits[0]

            total_loss += torch.nn.functional.cross_entropy(
                logits, target, reduction="sum"
            ).item()
            total_tokens += len(y)

            pred = logits.argmax(dim=-1).cpu().tolist()

            for gold_id, pred_id in zip(y, pred):
                gold_counts[gold_id] += 1
                pred_counts[pred_id] += 1
                correct_tokens += int(gold_id == pred_id)
                gold_entity_tokens += int(gold_id != 0)
                pred_entity_tokens += int(pred_id != 0)
                matched_entity_tokens += int(
                    gold_id != 0 and gold_id == pred_id
                )

            chunk_count += 1
            if end == len(ids_all):
                break
            start += 768

print("\n=== VALIDATION WITH CORRECTED LABEL IDS ===")
print("Documents:", len(fold1_text))
print("Chunks:", chunk_count)
print("Mean token cross-entropy:", round(total_loss / max(total_tokens, 1), 5))
print("Token accuracy:", round(correct_tokens / max(total_tokens, 1), 5))
print("Gold entity tokens:", gold_entity_tokens)
print("Predicted entity tokens:", pred_entity_tokens)
print("Correct entity tokens:", matched_entity_tokens)
print("Entity-token recall:", round(
    matched_entity_tokens / max(gold_entity_tokens, 1), 5
))
print("Predicted non-O fraction:", round(
    pred_entity_tokens / max(total_tokens, 1), 5
))
print("Predicted labels:", [
    (ID2LABEL_GROUPED[k], v)
    for k, v in pred_counts.most_common(15)
])


In [ ]:

from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

if not source_path.exists():
    print("Source file not found:", source_path)
    print("Files in /kaggle/working:")
    for p in Path("/kaggle/working").rglob("*"):
        if p.is_file() and p.suffix in {".py", ".ipynb"}:
            print(p)
else:
    lines = source_path.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()

    keywords = (
        "bio2id",
        "id2bio",
        "BIO_LABELS",
        "bio_labels",
        "LORMA",
        "lorma_encoder",
    )

    matches = [
        i for i, line in enumerate(lines)
        if any(k.lower() in line.lower() for k in keywords)
    ]

    print("Source lines:", len(lines))
    print("Matching lines:", len(matches))

    # Show mapping-related lines first, with nearby context.
    shown = set()
    for i in matches:
        if not any(
            k.lower() in lines[i].lower()
            for k in ("bio2id", "id2bio", "BIO_LABELS", "bio_labels")
        ):
            continue

        start = max(0, i - 5)
        end = min(len(lines), i + 9)

        if any(j in shown for j in range(start, end)):
            continue

        print(f"\n--- Source lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")
            shown.add(j)


In [ ]:

from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

for start, end in [(7065, 7085), (15100, 15180), (15465, 15496)]:
    print(f"\n{'=' * 20} Lines {start}-{end} {'=' * 20}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1}: {lines[i]}")


In [ ]:

import torch
from collections import Counter

ENTITY_LABELS_CHECK = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL",
    "JUDGE", "WIT", "AUTH", "COURT",
    "STAT", "PREC", "DATE", "CASENO"
]

BIO_LABELS_CHECK = (
    ["O"]
    + [f"B-{x}" for x in ENTITY_LABELS_CHECK]
    + [f"I-{x}" for x in ENTITY_LABELS_CHECK]
)
ID2LABEL_CHECK = dict(enumerate(BIO_LABELS_CHECK))

MAX_LEN = 1024
STRIDE = 768
device = next(model4.parameters()).device
model4.eval()

def decode_spans(text, offsets, tag_ids):
    spans = []
    active_label = None
    start = end = None

    def finish():
        if active_label is not None and start is not None and end is not None:
            s, e = start, end
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            if s < e:
                spans.append((s, e, active_label))

    for (ts, te), tag_id in zip(offsets, tag_ids):
        if te <= ts:
            continue

        tag = ID2LABEL_CHECK.get(int(tag_id), "O")
        if tag == "O":
            finish()
            active_label = None
            start = end = None
            continue

        prefix, label = tag.split("-", 1)

        if prefix == "B" or label != active_label:
            finish()
            active_label = label
            start, end = ts, te
        else:
            end = te

    finish()
    return spans

tp = fp = fn = 0
gold_by_label = Counter()
pred_by_label = Counter()
tp_by_label = Counter()

with torch.inference_mode():
    for doc_id, text in fold1_text.items():
        enc = tokenizer.encode(text)
        ids = enc.ids
        offsets = enc.offsets

        # Accumulate logits for overlapping windows.
        sums = torch.zeros(
            (len(ids), len(BIO_LABELS_CHECK)),
            dtype=torch.float32
        )
        counts = torch.zeros(len(ids), dtype=torch.float32)

        for start_idx in range(0, len(ids), STRIDE):
            end_idx = min(start_idx + MAX_LEN, len(ids))
            chunk_ids = ids[start_idx:end_idx]

            if not chunk_ids:
                break

            x = torch.tensor(
                [chunk_ids], dtype=torch.long, device=device
            )
            mask = torch.ones_like(x)

            logits = model4(x, attention_mask=mask)
            if isinstance(logits, dict):
                logits = logits["logits"]
            elif hasattr(logits, "logits"):
                logits = logits.logits
            elif isinstance(logits, (tuple, list)):
                logits = logits[0]

            logits = logits[0].detach().float().cpu()
            sums[start_idx:end_idx] += logits
            counts[start_idx:end_idx] += 1

            if end_idx == len(ids):
                break

        avg_logits = sums / counts.clamp_min(1).unsqueeze(1)
        pred_ids = avg_logits.argmax(dim=-1).tolist()

        predicted = set(decode_spans(text, offsets, pred_ids))
        gold = set(
            (int(s), int(e), str(label))
            for s, e, label, *rest in fold1_gold[doc_id]
        )

        tp += len(predicted & gold)
        fp += len(predicted - gold)
        fn += len(gold - predicted)

        for s, e, label in gold:
            gold_by_label[label] += 1
        for s, e, label in predicted:
            pred_by_label[label] += 1
        for s, e, label in predicted & gold:
            tp_by_label[label] += 1

all_labels = ENTITY_LABELS_CHECK
f1_values = []

print("=== STRICT SPAN VALIDATION: FOLD 1 ===")
print("Documents:", len(fold1_text))
print("TP:", tp, "FP:", fp, "FN:", fn)

micro_p = tp / max(1, tp + fp)
micro_r = tp / max(1, tp + fn)
micro_f1 = 2 * micro_p * micro_r / max(1e-12, micro_p + micro_r)

for label in all_labels:
    p = tp_by_label[label] / max(1, pred_by_label[label])
    r = tp_by_label[label] / max(1, gold_by_label[label])
    f1 = 2 * p * r / max(1e-12, p + r)
    f1_values.append(f1)
    print(
        f"{label:12s} gold={gold_by_label[label]:4d} "
        f"pred={pred_by_label[label]:4d} "
        f"TP={tp_by_label[label]:4d} "
        f"P={p:.4f} R={r:.4f} F1={f1:.4f}"
    )

print(f"\nMicro precision: {micro_p:.5f}")
print(f"Micro recall:    {micro_r:.5f}")
print(f"Micro F1:        {micro_f1:.5f}")
print(f"Strict Macro-F1: {sum(f1_values)/len(f1_values):.5f}")


In [ ]:

import torch
from pathlib import Path

p1_path = Path(
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)
p2_path = Path(
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/a4_ner_exp4_lorma_epoch3.pt"
)

p1 = torch.load(p1_path, map_location="cpu", weights_only=False)
p2 = torch.load(p2_path, map_location="cpu", weights_only=False)

def get_state(obj):
    if isinstance(obj, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj

s1 = get_state(p1)
s2 = get_state(p2)

print("Phase 1 top-level keys:",
      list(p1.keys()) if isinstance(p1, dict) else type(p1))
print("Phase 2 experiment:", p2.get("experiment"))
print("Phase 2 epoch:", p2.get("epoch"))

print("\nPhase 1 sample keys:", list(s1.keys())[:12])
print("Phase 2 sample keys:", list(s2.keys())[:12])

print("\nPhase 1 tensors:", sum(torch.is_tensor(v) for v in s1.values()))
print("Phase 2 tensors:", sum(torch.is_tensor(v) for v in s2.values()))

# Compare identical keys and shapes only.
same = different = missing = 0
diffs = []

for k, v2 in s2.items():
    if not torch.is_tensor(v2):
        continue
    if k not in s1 or not torch.is_tensor(s1[k]):
        missing += 1
        continue
    v1 = s1[k]
    if v1.shape != v2.shape:
        continue
    same += 1
    delta = (v1.float() - v2.float()).abs().max().item()
    if delta == 0:
        different += 0
    else:
        different += 1
        diffs.append((delta, k))

print("\nSame-key, same-shape tensors:", same)
print("Non-identical among them:", different)
print("Missing keys:", missing)
print("\nLargest differences:")
for delta, key in sorted(diffs, reverse=True)[:15]:
    print(f"{key}: max_abs_diff={delta:.6g}")


In [ ]:

import torch

def phase2_to_phase1_key(k):
    # Remove the encoder prefix.
    if not k.startswith("encoder."):
        return None

    k = k[len("encoder."):]

    # Unwrap LORMA's frozen base projections.
    k = k.replace(".q_proj.base.", ".q_proj.")
    k = k.replace(".v_proj.base.", ".v_proj.")

    return k

matched = []
missing = []
shape_mismatch = []
different = []
exact = []

for k2, v2 in s2.items():
    if not torch.is_tensor(v2):
        continue

    k1 = phase2_to_phase1_key(k2)
    if k1 is None or ".lorma_A" in k2 or ".lorma_B" in k2:
        continue

    if k1 not in s1 or not torch.is_tensor(s1[k1]):
        missing.append((k2, k1))
        continue

    v1 = s1[k1]
    if v1.shape != v2.shape:
        shape_mismatch.append((k2, k1, tuple(v1.shape), tuple(v2.shape)))
        continue

    max_diff = (v1.float() - v2.float()).abs().max().item()
    matched.append((k2, k1, max_diff))

    if max_diff == 0:
        exact.append(k2)
    else:
        different.append((max_diff, k2, k1))

print("Comparable base tensors:", len(matched))
print("Exactly equal:", len(exact))
print("Different:", len(different))
print("Missing Phase 1 keys:", len(missing))
print("Shape mismatches:", len(shape_mismatch))

print("\nLargest differences:")
for diff, k2, k1 in sorted(different, reverse=True)[:15]:
    print(f"{diff:.6g} | Phase 2: {k2} | Phase 1: {k1}")

print("\nExamples of missing keys:")
for item in missing[:10]:
    print(item)


In [ ]:

# Diagnostic only: no weights are changed.

def normalize_encoder_key(k):
    if k.startswith("encoder."):
        k = k[len("encoder."):]
    k = k.replace(".q_proj.base.", ".q_proj.")
    k = k.replace(".v_proj.base.", ".v_proj.")
    return k

def compare_states(reference, candidate, name):
    exact = 0
    different = 0
    missing = 0
    errors = []

    for key, ref_value in reference.items():
        if not torch.is_tensor(ref_value):
            continue

        candidates = [
            (k, v) for k, v in candidate.items()
            if torch.is_tensor(v)
            and normalize_encoder_key(k) == normalize_encoder_key(key)
        ]

        if not candidates:
            missing += 1
            continue

        candidate_key, value = candidates[0]

        if ref_value.shape != value.shape:
            continue

        delta = (ref_value.float() - value.float()).abs().max().item()

        if delta == 0:
            exact += 1
        else:
            different += 1
            if len(errors) < 5:
                errors.append((key, candidate_key, delta))

    print(f"\n{name}")
    print("Exact matches:", exact)
    print("Different:", different)
    print("Missing:", missing)
    print("Examples:", errors)

active_encoder_state = encoder.state_dict()

compare_states(
    s1,
    active_encoder_state,
    "Phase 1 checkpoint vs current encoder"
)

compare_states(
    s1,
    s2,
    "Phase 1 checkpoint vs Phase 2 checkpoint"
)


In [ ]:

import torch

# Reuse s1 and s2 if they still exist; otherwise reload them.
from pathlib import Path

p1_path = Path(
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)
p2_path = Path(
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/a4_ner_exp4_lorma_epoch3.pt"
)

if "s1" not in globals():
    p1 = torch.load(p1_path, map_location="cpu", weights_only=False)
    s1 = p1["model_state_dict"]

if "s2" not in globals():
    p2 = torch.load(p2_path, map_location="cpu", weights_only=False)
    s2 = p2["model_state_dict"]

def phase2_to_phase1_key(k):
    if not k.startswith("encoder."):
        return None
    k = k[len("encoder."):]
    k = k.replace(".q_proj.base.", ".q_proj.")
    k = k.replace(".v_proj.base.", ".v_proj.")
    return k

matched, exact, different = 0, 0, 0
missing, shape_mismatch = [], []
largest = []

for k2, v2 in s2.items():
    if not torch.is_tensor(v2):
        continue
    if ".lorma_A" in k2 or ".lorma_B" in k2:
        continue

    k1 = phase2_to_phase1_key(k2)
    if k1 is None:
        continue
    if k1 not in s1:
        missing.append((k2, k1))
        continue

    v1 = s1[k1]
    if not torch.is_tensor(v1):
        continue
    if v1.shape != v2.shape:
        shape_mismatch.append((k1, tuple(v1.shape), tuple(v2.shape)))
        continue

    delta = (v1.float() - v2.float()).abs().max().item()
    matched += 1
    if delta == 0:
        exact += 1
    else:
        different += 1
        largest.append((delta, k1))

print("Comparable base tensors:", matched)
print("Exactly equal:", exact)
print("Different:", different)
print("Missing:", len(missing))
print("Shape mismatches:", len(shape_mismatch))
print("\nLargest differences:")
for delta, key in sorted(largest, reverse=True)[:10]:
    print(f"{key}: {delta:.6g}")


In [ ]:

from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = source_path.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

terms = [
    "encoder.load_state_dict",
    "A4_CHECKPOINT_PATH",
    "a4_step_22000.pt",
    "lorma_encoder = copy.deepcopy(encoder)",
]

for term in terms:
    hits = [
        i for i, line in enumerate(lines)
        if term.lower() in line.lower()
    ]

    print(f"\n{'=' * 65}\n{term}: {len(hits)} occurrence(s)")

    for i in hits[-3:]:
        start = max(0, i - 12)
        end = min(len(lines), i + 20)
        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")


In [ ]:

from pathlib import Path

p = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = p.read_text(encoding="utf-8", errors="replace").splitlines()

# Original LORMA initialization is around line 15128.
for start, end in [(14860, 14940), (15080, 15170)]:
    print(f"\n{'='*20} Lines {start}-{end} {'='*20}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")


In [ ]:

from pathlib import Path

path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

terms = [
    "encoder.load_state_dict(",
    "encoder = A4Encoder(",
    "A4_CHECKPOINT_PATH =",
    "checkpoint[\"model_state_dict\"]",
    "encoder_state",
]

for term in terms:
    hits = [
        i for i, line in enumerate(lines[:14820])
        if term.lower() in line.lower()
    ]

    print(f"\n{'='*60}\n{term}: {len(hits)} matches")

    for i in hits[-4:]:
        start = max(0, i - 12)
        end = min(14820, i + 16)
        print(f"\n--- Lines {start+1}-{end} ---")
        for j in range(start, end):
            print(f"{j+1}: {lines[j]}")


In [ ]:

from pathlib import Path

p = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = p.read_text(encoding="utf-8", errors="replace").splitlines()

# Inspect the final encoder/checkpoint setup before LORMA begins.
terms = (
    "encoder =",
    "encoder.load_state_dict",
    "A4_CHECKPOINT_PATH",
    "torch.load(",
    "lorma_encoder = copy.deepcopy",
)

hits = []
for i, line in enumerate(lines[:14860]):
    if any(term in line for term in terms):
        hits.append(i)

for i in hits[-18:]:
    start = max(0, i - 5)
    end = min(len(lines), i + 9)
    print(f"\n--- Source lines {start+1}-{end} ---")
    for j in range(start, end):
        print(f"{j+1}: {lines[j]}")


In [ ]:

import torch
from pathlib import Path

a4_path = Path(
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)
lorma_path = Path(
    "/kaggle/input/datasets/aq1de3/a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

a4_ckpt = torch.load(
    a4_path, map_location="cpu", weights_only=False
)
lorma_ckpt = torch.load(
    lorma_path, map_location="cpu", weights_only=False
)

a4_state = a4_ckpt["model_state_dict"]
lorma_state = lorma_ckpt["model_state_dict"]

# Match LORMA's frozen base projection weights to Phase 1 keys.
matched, missing, mismatched = 0, [], []
max_diffs = []

for key, value in lorma_state.items():
    if key.startswith("encoder."):
        original_key = key[len("encoder."):]

        for projection in ("q_proj", "v_proj"):
            marker = f".{projection}.base."
            if marker in original_key:
                original_key = original_key.replace(
                    marker, f".{projection}."
                )
                break

        if original_key in a4_state:
            base = a4_state[original_key]
            if base.shape == value.shape:
                matched += 1
                diff = (base.float() - value.float()).abs().max().item()
                max_diffs.append((diff, key))
            else:
                mismatched.append((key, tuple(value.shape), tuple(base.shape)))
        elif ".lorma_A" not in key and ".lorma_B" not in key:
            missing.append((key, original_key))

adapters = [
    (key, value.float().abs().max().item())
    for key, value in lorma_state.items()
    if ".lorma_A" in key or ".lorma_B" in key
]

print("Phase 1 checkpoint:", a4_path.exists())
print("LORMA checkpoint:", lorma_path.exists())
print("Matched base tensors:", matched)
print("Missing base keys:", len(missing))
print("Shape mismatches:", len(mismatched))
print("Exactly equal matched tensors:",
      sum(diff == 0 for diff, _ in max_diffs))
print("Largest base-weight differences:")
for diff, key in sorted(max_diffs, reverse=True)[:10]:
    print(f"{diff:.8g}  {key}")

print("\nAdapter tensors:", len(adapters))
print("Adapters with nonzero values:",
      sum(max_abs > 0 for _, max_abs in adapters))
print("First adapter values:")
for key, max_abs in adapters[:8]:
    print(f"{key}: max_abs={max_abs:.8g}")


In [ ]:

from pathlib import Path

p = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = p.read_text(encoding="utf-8", errors="replace").splitlines()

for start, end in [(12340, 12620), (12800, 13200), (15158, 15280)]:
    print(f"\n{'='*25} Lines {start}-{end} {'='*25}")
    for i in range(start - 1, min(end, len(lines))):
        line = lines[i]
        if any(term in line.lower() for term in [
            "requires_grad", "optimizer", "parameters()",
            "copy.deepcopy", "encoder =", "loss.backward",
            "optimizer.step", "model1", "model2", "model4",
            "lorma_encoder"
        ]):
            lo, hi = max(start - 1, i - 2), min(len(lines), i + 3)
            print(f"\n--- around line {i+1} ---")
            for j in range(lo, hi):
                print(f"{j+1}: {lines[j]}")


In [ ]:

from pathlib import Path

p = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = p.read_text(encoding="utf-8", errors="replace").splitlines()

for start, end in [(13060, 13165), (13560, 13610), (15120, 15155)]:
    print(f"\n{'='*20} Lines {start}-{end} {'='*20}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")


In [ ]:

import torch
from pathlib import Path

phase1_path = Path(
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)
exp2_dir = Path("/kaggle/working/phase2_exp2")

print("Experiment 2 directory exists:", exp2_dir.exists())

if exp2_dir.exists():
    for p in sorted(exp2_dir.glob("*.pt")):
        print(p.name, f"{p.stat().st_size / 1024**2:.1f} MiB")

phase1 = torch.load(
    phase1_path, map_location="cpu", weights_only=False
)["model_state_dict"]

if exp2_dir.exists():
    for p in sorted(exp2_dir.glob("*.pt")):
        ckpt = torch.load(
            p, map_location="cpu", weights_only=False
        )
        state = ckpt.get("model_state_dict", ckpt)

        keys = [
            k for k in state
            if k in phase1 and state[k].shape == phase1[k].shape
        ]

        if keys:
            diffs = [
                (state[k].float() - phase1[k].float()).abs().max().item()
                for k in keys
            ]
            print(
                f"\n{p.name}: matched={len(keys)}, "
                f"exactly_equal={sum(d == 0 for d in diffs)}, "
                f"mean_max_abs_diff={sum(diffs)/len(diffs):.8g}, "
                f"largest_diff={max(diffs):.8g}"
            )


In [ ]:

import torch
import json
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import precision_recall_fscore_support

# --------------------------------------------------
# 1. Paths and device
# --------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LORMA_PATH = Path(
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt"
)
FOLD1_TEXT_PATH = Path(
    "/kaggle/input/datasets/aq1de3/fold1-tex/fold1-text.json"
)
FOLD1_GOLD_PATH = Path(
    "/kaggle/input/datasets/aq1de3/fold1-gold1/fold1-gold.json"
)

assert LORMA_PATH.exists(), f"Missing checkpoint: {LORMA_PATH}"
assert FOLD1_TEXT_PATH.exists(), f"Missing file: {FOLD1_TEXT_PATH}"
assert FOLD1_GOLD_PATH.exists(), f"Missing file: {FOLD1_GOLD_PATH}"

print("Device:", DEVICE)

# --------------------------------------------------
# 2. Load data
# --------------------------------------------------
def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

fold1_text = read_json(FOLD1_TEXT_PATH)
fold1_gold = read_json(FOLD1_GOLD_PATH)

print("Fold 1 text documents:", len(fold1_text))
print("Fold 1 gold documents:", len(fold1_gold))

# --------------------------------------------------
# 3. Validate model dependencies
# --------------------------------------------------
required = ["A4NER", "A4EncoderForNER", "tokenizer", "decode_spans"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the earlier model/tokenizer/decoder cells first. "
        f"Missing: {missing}"
    )

# The architecture must match the saved checkpoint.
checkpoint = torch.load(
    LORMA_PATH, map_location="cpu", weights_only=False
)
state = checkpoint["model_state_dict"]

# Reuse the already reconstructed LORMA model only if it exists.
if "model4" not in globals():
    raise RuntimeError(
        "model4 is not defined. Run the earlier cell that reconstructs "
        "A4NER and injects LORMA into Q/V projections before evaluating."
    )

model4.load_state_dict(state, strict=True)
model4.to(DEVICE)
model4.eval()

print("Checkpoint loaded:", checkpoint.get("experiment", "LORMA"))
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))

# --------------------------------------------------
# 4. Inspect data schema before inference
# --------------------------------------------------
print("\nText sample type:", type(fold1_text).__name__)
print("Gold sample type:", type(fold1_gold).__name__)

if isinstance(fold1_text, dict):
    print("Text sample:", next(iter(fold1_text.items())))
else:
    print("Text sample:", fold1_text[0])

if isinstance(fold1_gold, dict):
    print("Gold sample:", next(iter(fold1_gold.items())))
else:
    print("Gold sample:", fold1_gold[0])

print(
    "\nModel loaded successfully. Verify the displayed data schema "
    "matches your existing inference function before running evaluation."
)


In [ ]:

import inspect

print("Tokenizer class:", type(tokenizer))
print("Tokenizer supports offset mappings:",
      hasattr(tokenizer, "encode"))

print("\nDecoder signature:")
print(inspect.signature(decode_spans))

print("\nModel forward signature:")
print(inspect.signature(model4.forward))

print("\nFirst document ID:", next(iter(fold1_text)))
print("First gold document ID:", next(iter(fold1_gold)))

print("\nGold span example:")
first_id = next(iter(fold1_gold))
print(fold1_gold[first_id][:5])

print("\nExisting inference helpers:")
for name in [
    "chunk_document",
    "make_chunks",
    "tokenize_document",
    "decode_spans",
    "bio2id",
    "id2bio",
    "LABELS",
]:
    if name in globals():
        obj = globals()[name]
        print(name, "->", type(obj).__name__)
        if callable(obj):
            try:
                print("  signature:", inspect.signature(obj))
            except (TypeError, ValueError):
                pass


In [ ]:

import torch
import numpy as np
from collections import defaultdict

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------
MAX_LEN = 1024
STRIDE = 768
DEVICE = next(model4.parameters()).device

# Grouped BIO mapping used by the original notebook:
# 0=O, 1..12=B-label, 13..24=I-label.
if "LABELS" in globals():
    labels = list(LABELS)
else:
    with open(
        "/kaggle/input/datasets/aq1de3/labelses/labels.txt",
        "r", encoding="utf-8"
    ) as f:
        labels = [x.strip() for x in f if x.strip()]

assert len(labels) == 12, f"Expected 12 entity labels, got {len(labels)}"

bio_labels = ["O"] + [f"B-{x}" for x in labels] + [f"I-{x}" for x in labels]
assert len(bio_labels) == 25

# --------------------------------------------------
# 2. Inference, averaging logits over overlapping chunks
# --------------------------------------------------
model4.eval()
all_pred = {}
all_gold = {}

with torch.inference_mode():
    for doc_i, (doc_id, text) in enumerate(fold1_text.items(), start=1):
        encoded = tokenize_document(text)

        # Expect token IDs and character offsets from your helper.
        if isinstance(encoded, dict):
            token_ids = encoded.get("input_ids")
            offsets = encoded.get("offsets", encoded.get("offset_mapping"))
        elif isinstance(encoded, (tuple, list)) and len(encoded) >= 2:
            token_ids, offsets = encoded[0], encoded[1]
        else:
            raise TypeError(
                "Unexpected tokenize_document output. "
                f"Got {type(encoded)}; inspect tokenize_document(text)."
            )

        if token_ids and isinstance(token_ids[0], list):
            token_ids = token_ids[0]
        if offsets and isinstance(offsets[0], list) and len(offsets[0]) == 2:
            pass
        elif offsets and isinstance(offsets[0], list):
            offsets = offsets[0]

        n = len(token_ids)
        if len(offsets) != n:
            raise ValueError(
                f"{doc_id}: {n} tokens but {len(offsets)} offsets"
            )

        # Accumulate per-token logits across overlapping windows.
        logit_sum = np.zeros((n, 25), dtype=np.float32)
        logit_count = np.zeros(n, dtype=np.float32)

        starts = list(range(0, max(1, n), STRIDE))
        for start in starts:
            end = min(start + MAX_LEN, n)
            if start >= n:
                break

            ids = torch.tensor(
                [token_ids[start:end]], dtype=torch.long, device=DEVICE
            )
            mask = torch.ones_like(ids)

            logits = model4(input_ids=ids, attention_mask=mask)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]
            logits = logits[0].float().cpu().numpy()

            logit_sum[start:end] += logits
            logit_count[start:end] += 1

            if end == n:
                break

        averaged = logit_sum / np.maximum(logit_count[:, None], 1.0)
        tag_ids = averaged.argmax(axis=-1).tolist()

        pred_spans = decode_spans(text, offsets, tag_ids)
        all_pred[str(doc_id)] = {
            (int(s), int(e), str(label))
            for s, e, label, *rest in pred_spans
        }

        gold_spans = fold1_gold[str(doc_id)]
        all_gold[str(doc_id)] = {
            (int(s), int(e), str(label))
            for s, e, label, *rest in gold_spans
        }

        if doc_i % 5 == 0 or doc_i == len(fold1_text):
            print(f"Processed {doc_i}/{len(fold1_text)} documents")

# --------------------------------------------------
# 3. Strict span metrics
# --------------------------------------------------
print("\nSTRICT SPAN-LEVEL RESULTS")
print("=" * 78)
print(f"{'Label':<14} {'Gold':>7} {'Pred':>7} {'TP':>7} {'Prec':>9} {'Recall':>9} {'F1':>9}")

per_label_f1 = []
total_tp = total_fp = total_fn = 0

for label in labels:
    gold_set = {
        (doc_id, s, e, lab)
        for doc_id, spans in all_gold.items()
        for s, e, lab in spans if lab == label
    }
    pred_set = {
        (doc_id, s, e, lab)
        for doc_id, spans in all_pred.items()
        for s, e, lab in spans if lab == label
    }

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    total_tp += tp
    total_fp += fp
    total_fn += fn
    per_label_f1.append(f1)

    print(
        f"{label:<14} {len(gold_set):>7} {len(pred_set):>7} "
        f"{tp:>7} {precision:>9.4f} {recall:>9.4f} {f1:>9.4f}"
    )

micro_p = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0
micro_r = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0.0
micro_f1 = (
    2 * micro_p * micro_r / (micro_p + micro_r)
    if micro_p + micro_r else 0.0
)
macro_f1 = float(np.mean(per_label_f1))

print("=" * 78)
print("Total TP:", total_tp, "FP:", total_fp, "FN:", total_fn)
print(f"Micro precision: {micro_p:.6f}")
print(f"Micro recall:    {micro_r:.6f}")
print(f"Micro F1:        {micro_f1:.6f}")
print(f"Macro F1:        {macro_f1:.6f}")
print("\nExisting best submission score: 0.0515 (unchanged)")



In [ ]:

from collections import Counter, defaultdict

pred_count = Counter()
gold_count = Counter()
exact_matches = 0
text_mismatches = []
same_label_overlaps = 0
wrong_label_overlaps = 0
boundary_near_matches = 0

for doc_id, text in fold1_text.items():
    doc_id = str(doc_id)
    preds = all_pred[doc_id]
    golds = all_gold[doc_id]

    for s, e, label in preds:
        pred_count[label] += 1
        if not (0 <= s < e <= len(text)):
            text_mismatches.append((doc_id, s, e, label, "INVALID_OFFSETS"))
        elif not text[s:e].strip():
            text_mismatches.append((doc_id, s, e, label, repr(text[s:e])))

    for s, e, label in golds:
        gold_count[label] += 1

    exact_matches += len(preds & golds)

    for ps, pe, pl in preds:
        for gs, ge, gl in golds:
            if max(ps, gs) < min(pe, ge):
                if pl == gl:
                    same_label_overlaps += 1
                    if (ps, pe) != (gs, ge):
                        boundary_near_matches += 1
                else:
                    wrong_label_overlaps += 1

print("Prediction counts:", pred_count)
print("\nGold counts:", gold_count)
print("\nExact span matches:", exact_matches)
print("Same-label overlaps:", same_label_overlaps)
print("Same-label overlaps with different boundaries:", boundary_near_matches)
print("Different-label overlaps:", wrong_label_overlaps)
print("Invalid/empty prediction offsets:", len(text_mismatches))

if text_mismatches:
    print("\nFirst offset problems:")
    for row in text_mismatches[:10]:
        print(row)

# Show examples for labels with zero exact matches.
print("\n--- DATE / PREC / CASENO examples ---")
for target in ["DATE", "PREC", "CASENO"]:
    print(f"\n{target}:")
    shown = 0

    for doc_id, text in fold1_text.items():
        doc_id = str(doc_id)
        preds = sorted(
            (s, e, lab) for s, e, lab in all_pred[doc_id]
            if lab == target
        )
        golds = sorted(
            (s, e, lab) for s, e, lab in all_gold[doc_id]
            if lab == target
        )

        for s, e, lab in preds[:3]:
            overlapping_gold = [
                (gs, ge, gl, text[gs:ge])
                for gs, ge, gl in all_gold[doc_id]
                if max(s, gs) < min(e, ge)
            ]
            print(
                "PRED:", (s, e, lab), repr(text[s:e]),
                "\n  overlapping gold:", overlapping_gold[:2]
            )
            shown += 1
            if shown >= 5:
                break
        if shown >= 5:
            break

    if shown == 0:
        print("No predictions for this label.")


In [ ]:

import torch

doc_id = "115651329"
text = fold1_text[doc_id]
gold_spans = fold1_gold[doc_id]
device = next(model4.parameters()).device

# Tokenize using the existing helper.
encoded = tokenize_document(text)

if isinstance(encoded, dict):
    token_ids = encoded.get("input_ids")
    offsets = encoded.get("offsets", encoded.get("offset_mapping"))
else:
    token_ids, offsets = encoded[:2]

if token_ids and isinstance(token_ids[0], list):
    token_ids = token_ids[0]
if offsets and isinstance(offsets[0], list) and len(offsets[0]) != 2:
    offsets = offsets[0]

# Average logits for overlapping windows.
n = len(token_ids)
logit_sum = torch.zeros((n, 25), dtype=torch.float32)
logit_count = torch.zeros(n, dtype=torch.float32)

model4.eval()
with torch.inference_mode():
    for start in range(0, n, 768):
        end = min(start + 1024, n)
        ids = torch.tensor(
            [token_ids[start:end]], dtype=torch.long, device=device
        )
        mask = torch.ones_like(ids)
        out = model4(input_ids=ids, attention_mask=mask)
        if isinstance(out, (tuple, list)):
            out = out[0]
        logits = out[0].float().cpu()
        logit_sum[start:end] += logits
        logit_count[start:end] += 1
        if end == n:
            break

pred_ids = (logit_sum / logit_count.clamp_min(1)[:, None]).argmax(-1).tolist()

labels = list(LABELS) if "LABELS" in globals() else [
    x.strip() for x in open(
        "/kaggle/input/datasets/aq1de3/labelses/labels.txt",
        encoding="utf-8"
    ) if x.strip()
]
bio_labels = ["O"] + [f"B-{x}" for x in labels] + [f"I-{x}" for x in labels]

# Print token-level context around the first gold span for each target class.
for target in ["DATE", "PREC", "CASENO"]:
    match = next(
        (span for span in gold_spans if span[2] == target),
        None
    )
    if match is None:
        print(f"\nNo {target} gold span in this document.")
        continue

    gs, ge, gl, gold_text = match
    indices = [
        i for i, (s, e) in enumerate(offsets)
        if e > gs and s < ge
    ]
    if not indices:
        print(f"\nNo tokens overlap gold {target}: {gold_text!r}")
        continue

    lo = max(0, min(indices) - 4)
    hi = min(n, max(indices) + 5)

    print(f"\n{'='*72}")
    print(f"GOLD {target}: [{gs}, {ge}) {gold_text!r}")
    print(f"{'Tok':>5} {'Offsets':>15} {'Predicted BIO':<18} Token text")
    for i in range(lo, hi):
        s, e = offsets[i]
        token_text = text[s:e] if e > s else ""
        print(
            f"{i:5d} [{s:6d},{e:6d}) "
            f"{bio_labels[pred_ids[i]]:<18} {token_text!r}"
        )


In [ ]:

import torch
import numpy as np
from collections import defaultdict

device = next(model4.parameters()).device
model4.eval()

labels = list(LABELS) if "LABELS" in globals() else [
    x.strip() for x in open(
        "/kaggle/input/datasets/aq1de3/labelses/labels.txt",
        encoding="utf-8"
    ) if x.strip()
]
label_to_b = {lab: i + 1 for i, lab in enumerate(labels)}
label_to_i = {lab: i + 13 for i, lab in enumerate(labels)}

# Accumulate statistics for each gold label.
stats = defaultdict(lambda: {
    "tokens": 0, "gold_tag_prob_sum": 0.0,
    "o_prob_sum": 0.0, "gold_tag_wins": 0
})

with torch.inference_mode():
    for doc_id, text in fold1_text.items():
        doc_id = str(doc_id)
        encoded = tokenize_document(text)

        if isinstance(encoded, dict):
            token_ids = encoded.get("input_ids")
            offsets = encoded.get("offsets", encoded.get("offset_mapping"))
        else:
            token_ids, offsets = encoded[:2]

        if token_ids and isinstance(token_ids[0], list):
            token_ids = token_ids[0]
        if offsets and isinstance(offsets[0], list) and len(offsets[0]) != 2:
            offsets = offsets[0]

        n = len(token_ids)
        sums = torch.zeros((n, 25), dtype=torch.float32)
        counts = torch.zeros(n, dtype=torch.float32)

        for start in range(0, n, 768):
            end = min(start + 1024, n)
            ids = torch.tensor(
                [token_ids[start:end]], dtype=torch.long, device=device
            )
            mask = torch.ones_like(ids)
            out = model4(input_ids=ids, attention_mask=mask)
            if isinstance(out, (tuple, list)):
                out = out[0]
            sums[start:end] += out[0].float().cpu()
            counts[start:end] += 1
            if end == n:
                break

        probs = torch.softmax(sums / counts.clamp_min(1)[:, None], dim=-1)

        # Assign each token a gold BIO label from overlapping gold spans.
        gold_tags = [0] * n
        for s, e, lab, *_ in fold1_gold[doc_id]:
            if lab not in label_to_b:
                continue
            token_indices = [
                i for i, (ts, te) in enumerate(offsets)
                if te > s and ts < e
            ]
            for j, idx in enumerate(token_indices):
                gold_tags[idx] = (
                    label_to_b[lab] if j == 0 else label_to_i[lab]
                )

        for idx, tag in enumerate(gold_tags):
            if tag == 0:
                continue
            lab = labels[(tag - 1) % 12]
            p = probs[idx]
            stats[lab]["tokens"] += 1
            stats[lab]["gold_tag_prob_sum"] += p[tag].item()
            stats[lab]["o_prob_sum"] += p[0].item()
            stats[lab]["gold_tag_wins"] += int(p.argmax().item() == tag)

print(f"{'Label':<12} {'Tokens':>8} {'Gold-tag P':>12} {'O P':>10} {'Gold wins':>11}")
print("-" * 58)
for lab in labels:
    s = stats[lab]
    if not s["tokens"]:
        continue
    n = s["tokens"]
    print(
        f"{lab:<12} {n:>8} "
        f"{s['gold_tag_prob_sum']/n:>12.4f} "
        f"{s['o_prob_sum']/n:>10.4f} "
        f"{s['gold_tag_wins']/n:>10.1%}"
    )


In [ ]:

import torch
from collections import Counter, defaultdict

device = next(model4.parameters()).device
model4.eval()

labels = list(LABELS) if "LABELS" in globals() else [
    x.strip()
    for x in open(
        "/kaggle/input/datasets/aq1de3/labelses/labels.txt",
        encoding="utf-8"
    )
    if x.strip()
]
bio_labels = ["O"] + [f"B-{x}" for x in labels] + [f"I-{x}" for x in labels]

gold_entity_confusion = defaultdict(Counter)
gold_entity_totals = Counter()

with torch.inference_mode():
    for doc_id, text in fold1_text.items():
        doc_id = str(doc_id)
        encoded = tokenize_document(text)

        if isinstance(encoded, dict):
            token_ids = encoded.get("input_ids")
            offsets = encoded.get("offsets", encoded.get("offset_mapping"))
        else:
            token_ids, offsets = encoded[:2]

        if token_ids and isinstance(token_ids[0], list):
            token_ids = token_ids[0]
        if offsets and isinstance(offsets[0], list) and len(offsets[0]) != 2:
            offsets = offsets[0]

        n = len(token_ids)
        sums = torch.zeros((n, 25), dtype=torch.float32)
        counts = torch.zeros(n, dtype=torch.float32)

        for start in range(0, n, 768):
            end = min(start + 1024, n)
            ids = torch.tensor(
                [token_ids[start:end]], dtype=torch.long, device=device
            )
            mask = torch.ones_like(ids)
            out = model4(input_ids=ids, attention_mask=mask)
            if isinstance(out, (tuple, list)):
                out = out[0]
            sums[start:end] += out[0].float().cpu()
            counts[start:end] += 1
            if end == n:
                break

        pred = (sums / counts.clamp_min(1)[:, None]).argmax(-1).tolist()

        gold_tags = [0] * n
        for s, e, lab, *_ in fold1_gold[doc_id]:
            if lab not in labels:
                continue
            indices = [
                i for i, (ts, te) in enumerate(offsets)
                if te > s and ts < e
            ]
            for j, idx in enumerate(indices):
                gold_tags[idx] = (
                    1 + labels.index(lab)
                    if j == 0
                    else 13 + labels.index(lab)
                )

        for g, p in zip(gold_tags, pred):
            if g == 0:
                continue
            gold_label = labels[(g - 1) % 12]
            gold_entity_confusion[gold_label][bio_labels[p]] += 1
            gold_entity_totals[gold_label] += 1

print("Most common predictions on gold entity tokens")
print("=" * 75)

for lab in labels:
    counts = gold_entity_confusion[lab]
    total = gold_entity_totals[lab]
    if total == 0:
        continue
    print(f"\n{lab} (gold entity tokens={total})")
    for predicted_tag, count in counts.most_common(5):
        print(f"  {predicted_tag:<18} {count:>7} ({count/total:>6.1%})")

print("\nBIO label order used:")
print(bio_labels)


In [ ]:
import torch
from collections import Counter

device = next(model4.parameters()).device
labels = list(LABELS) if "LABELS" in globals() else [
    x.strip()
    for x in open(
        "/kaggle/input/datasets/aq1de3/labelses/labels.txt",
        encoding="utf-8"
    )
    if x.strip()
]

bio_labels = (
    ["O"]
    + [f"B-{x}" for x in labels]
    + [f"I-{x}" for x in labels]
)

model4.eval()
total_loss = 0.0
total_tokens = 0
correct = 0
tag_counts = Counter()
gold_counts = Counter()

with torch.inference_mode():
    for doc_id, text in fold1_text.items():
        doc_id = str(doc_id)
        encoded = tokenize_document(text)

        if isinstance(encoded, dict):
            token_ids = encoded.get("input_ids")
            offsets = encoded.get("offsets", encoded.get("offset_mapping"))
        else:
            token_ids, offsets = encoded[:2]

        if token_ids and isinstance(token_ids[0], list):
            token_ids = token_ids[0]

        n = len(token_ids)
        gold_tags = [0] * n

        # Reconstruct BIO labels from gold character spans.
        for s, e, lab, *_ in fold1_gold[doc_id]:
            if lab not in labels:
                continue

            indices = [
                i for i, (ts, te) in enumerate(offsets)
                if te > s and ts < e
            ]

            for j, idx in enumerate(indices):
                gold_tags[idx] = (
                    1 + labels.index(lab)
                    if j == 0
                    else 13 + labels.index(lab)
                )

        # Average logits from overlapping windows.
        sums = torch.zeros((n, 25), dtype=torch.float32)
        counts = torch.zeros(n, dtype=torch.float32)

        for start in range(0, n, 768):
            end = min(start + 1024, n)
            ids = torch.tensor(
                [token_ids[start:end]],
                dtype=torch.long,
                device=device
            )
            mask = torch.ones_like(ids)

            out = model4(input_ids=ids, attention_mask=mask)
            if isinstance(out, (tuple, list)):
                out = out[0]

            sums[start:end] += out[0].float().cpu()
            counts[start:end] += 1

            if end == n:
                break

        logits = sums / counts.clamp_min(1)[:, None]
        targets = torch.tensor(gold_tags, dtype=torch.long)

        losses = torch.nn.functional.cross_entropy(
            logits, targets, reduction="none"
        )
        total_loss += losses.sum().item()
        total_tokens += n

        pred = logits.argmax(dim=-1).tolist()
        correct += sum(p == g for p, g in zip(pred, gold_tags))

        tag_counts.update(pred)
        gold_counts.update(gold_tags)

print("Fold 1 reconstructed-label diagnostic")
print(f"Mean token cross-entropy: {total_loss / max(total_tokens, 1):.4f}")
print(f"Token accuracy:           {correct / max(total_tokens, 1):.4%}")

print("\nMost common predicted tags:")
for tag_id, count in tag_counts.most_common(10):
    print(f"{bio_labels[tag_id]:15s} {count:8d} ({count / total_tokens:.2%})")

print("\nGold tag distribution:")
for tag_id, count in sorted(gold_counts.items()):
    print(f"{bio_labels[tag_id]:15s} {count:8d} ({count / total_tokens:.2%})")

In [ ]:
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

if not source_path.exists():
    print("Source file not found:", source_path)
else:
    lines = source_path.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()

    keywords = [
        "offset_mapping",
        "bio2id",
        "B-" + "{label}",
        "char_to_token",
        "tokenize_document",
        "train_chunks",
        "gold_spans",
    ]

    matches = []
    for i, line in enumerate(lines):
        if any(k.lower() in line.lower() for k in keywords):
            matches.append(i)

    print("Found", len(matches), "matching source lines.")
    shown = set()

    for i in matches:
        lo, hi = max(0, i - 5), min(len(lines), i + 8)
        if any(j in shown for j in range(lo, hi)):
            continue
        print(f"\n--- Source lines {lo+1}-{hi} ---")
        for j in range(lo, hi):
            print(f"{j+1:6d}: {lines[j]}")
            shown.add(j)

In [ ]:
import torch

checkpoint_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint metadata:")
for key, value in ckpt.items():
    if key != "model_state_dict" and key != "optimizer_state_dict":
        print(f"{key}: {value}")

state = ckpt["model_state_dict"]

print("\nClassifier tensors:")
for key, value in state.items():
    if "classifier" in key.lower():
        print(key, tuple(value.shape))

print("\nLORMA tensors:")
lorma_keys = [
    key for key in state
    if "lorma_A" in key or "lorma_B" in key
]
print("Count:", len(lorma_keys))
for key in lorma_keys[:8]:
    print(key, tuple(state[key].shape))

print("\nExpected classes:", len(bio_labels))
print("Classifier output classes:", state["classifier.weight"].shape[0])

In [ ]:
import torch
from collections import Counter

device = next(model4.parameters()).device
model4.eval()

# Prefer the exact training split used by the LORMA experiment.
if "a4_train_only_chunks" in globals():
    diagnostic_chunks = a4_train_only_chunks
elif "a4_train_chunks" in globals() and "fold2_text" in globals():
    fold2_ids = {str(k) for k in fold2_text}
    diagnostic_chunks = [
        c for c in a4_train_chunks
        if str(c.doc_id) in fold2_ids
    ]
else:
    raise RuntimeError(
        "Training chunks are unavailable. Please rerun the notebook cell "
        "that builds a4_train_only_chunks."
    )

print("Training chunks available:", len(diagnostic_chunks))

gold_counts = Counter()
pred_counts = Counter()
correct = 0
total = 0
loss_sum = 0.0

with torch.inference_mode():
    for chunk in diagnostic_chunks:
        ids_list = chunk.input_ids
        gold_list = chunk.bio_ids

        ids = torch.tensor(
            [ids_list], dtype=torch.long, device=device
        )
        targets = torch.tensor(
            gold_list, dtype=torch.long, device=device
        )

        out = model4(
            input_ids=ids,
            attention_mask=torch.ones_like(ids)
        )
        if isinstance(out, (tuple, list)):
            out = out[0]

        logits = out[0].float()
        n = min(logits.shape[0], targets.shape[0])
        logits = logits[:n]
        targets = targets[:n]

        loss_sum += torch.nn.functional.cross_entropy(
            logits, targets, reduction="sum"
        ).item()

        preds = logits.argmax(-1)
        correct += (preds == targets).sum().item()
        total += n

        gold_counts.update(targets.cpu().tolist())
        pred_counts.update(preds.cpu().tolist())

print(f"\nTraining-token cross-entropy: {loss_sum / max(total, 1):.4f}")
print(f"Training-token accuracy:     {correct / max(total, 1):.2%}")
print(f"Gold entity-token share:     {sum(v for k,v in gold_counts.items() if k != 0) / max(total,1):.2%}")
print(f"Predicted entity-token share:{sum(v for k,v in pred_counts.items() if k != 0) / max(total,1):.2%}")

print("\nMost common predicted tags:")
for tag_id, count in pred_counts.most_common(12):
    print(f"{bio_labels[tag_id]:15s} {count:9d} ({count / max(total,1):.2%})")

print("\nMost common gold tags:")
for tag_id, count in gold_counts.most_common(12):
    print(f"{bio_labels[tag_id]:15s} {count:9d} ({count / max(total,1):.2%})")

In [ ]:
print("Available data and chunk variables:")
for name in sorted(globals()):
    if any(k in name.lower() for k in [
        "fold1", "fold2", "chunk", "train", "gold",
        "tokenizer", "bio_labels", "model4"
    ]):
        obj = globals()[name]
        try:
            info = f"type={type(obj).__name__}, len={len(obj)}"
        except Exception:
            info = f"type={type(obj).__name__}"
        print(f"{name}: {info}")

In [ ]:
from pathlib import Path

root = Path("/kaggle/input/datasets/aq1de3")

for p in [
    root / "fold2-tex/fold2-text.json",
    root / "fold2-go/fold2-gold.json",
    root / "fold2-te/fold2-text.json",
    root / "fold3-te/fold3-text.json",
]:
    print(f"{p}: {'EXISTS' if p.exists() else 'NOT FOUND'}")

In [ ]:
import json
from pathlib import Path

root = Path("/kaggle/input/datasets/aq1de3")

text_path = root / "fold2-tex/fold2-text.json"
gold_path = root / "fold2-go/fold2-gold.json"

with open(text_path, "r", encoding="utf-8") as f:
    fold2_text_check = json.load(f)

with open(gold_path, "r", encoding="utf-8") as f:
    fold2_gold_check = json.load(f)

print("Text type:", type(fold2_text_check).__name__)
print("Gold type:", type(fold2_gold_check).__name__)

print("\nText sample:")
print(repr(list(fold2_text_check.items())[:1] if isinstance(fold2_text_check, dict)
           else fold2_text_check[:1]))

print("\nGold sample:")
print(repr(list(fold2_gold_check.items())[:1] if isinstance(fold2_gold_check, dict)
           else fold2_gold_check[:1]))

In [ ]:
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

print("Source exists:", source_path.exists())

if source_path.exists():
    source = source_path.read_text(
        encoding="utf-8", errors="replace"
    )

    for term in [
        "a4_train_only_chunks",
        "train_chunks",
        "chunk_size",
        "BIO_LABELS_GROUPED",
        "build_ner_chunks",
    ]:
        print(f"\n--- {term} ---")
        positions = []
        start = 0
        while True:
            pos = source.find(term, start)
            if pos < 0:
                break
            positions.append(pos)
            start = pos + len(term)

        print("Occurrences:", len(positions))
        for pos in positions[:3]:
            print(source[max(0, pos-250):pos+500])

In [ ]:
import inspect

for name in [
    "build_chunks",
    "get_entities",
    "tokenize_document",
    "collate_fn",
]:
    obj = globals().get(name)
    print(f"\n{name}:",
          "AVAILABLE" if obj is not None else "MISSING")

    if obj is not None and callable(obj):
        try:
            print(inspect.getsource(obj)[:2500])
        except Exception as e:
            print("Source unavailable:", e)

print("\nFold 2 document counts:")
print("Text:", len(fold2_text_check))
print("Gold:", len(fold2_gold_check))
print("Matching IDs:",
      len(set(fold2_text_check) & set(fold2_gold_check)))

In [ ]:
from pathlib import Path
import re

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
source = source_path.read_text(encoding="utf-8", errors="replace")

# Locate definitions and usages of the original data-building functions.
patterns = [
    r"def\s+build_chunks\s*\(",
    r"def\s+get_entities\s*\(",
    r"def\s+collate_fn\s*\(",
    r"class\s+\w*Chunk\w*",
]

for pattern in patterns:
    matches = list(re.finditer(pattern, source))
    print(f"\n{'=' * 65}\n{pattern}: {len(matches)} matches")

    for match in matches[:3]:
        start = max(0, match.start() - 300)
        end = min(len(source), match.start() + 3500)
        print(f"\n--- Source position {match.start()} ---")
        print(source[start:end])

In [ ]:
import json
from pathlib import Path
from collections import Counter
from dataclasses import dataclass

# --------------------------------------------------
# 1. Restore the original chunking configuration
# --------------------------------------------------
MAX_TOKENS = 1024
STRIDE = 768

ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL",
    "JUDGE", "WIT", "AUTH", "COURT",
    "STAT", "PREC", "DATE", "CASENO"
]

BIO_LABELS_GROUPED = (
    ["O"]
    + [f"B-{label}" for label in ENTITY_LABELS]
    + [f"I-{label}" for label in ENTITY_LABELS]
)
bio2id = {label: i for i, label in enumerate(BIO_LABELS_GROUPED)}

assert len(BIO_LABELS_GROUPED) == model4.classifier.out_features

@dataclass
class NERChunk:
    doc_id: str
    chunk_id: int
    input_ids: list
    offsets: list
    bio_ids: list
    gold_spans: list
    char_start: int
    char_end: int

# --------------------------------------------------
# 2. Normalize the official gold format
#    [start, end, label, entity_text]
# --------------------------------------------------
def normalize_gold(gold_list):
    result = []
    for item in gold_list:
        result.append((int(item[0]), int(item[1]), str(item[2])))
    return sorted(result, key=lambda x: (x[0], x[1]))

# --------------------------------------------------
# 3. Rebuild chunks using the original logic
# --------------------------------------------------
def rebuild_chunks(doc_id, text, gold_list):
    ids, offsets = tokenize_document(text)
    entities = normalize_gold(gold_list)

    chunks = []
    start = 0
    chunk_id = 0

    while start < len(ids):
        end = min(start + MAX_TOKENS, len(ids))
        chunk_ids = ids[start:end]
        chunk_offsets = offsets[start:end]

        valid_offsets = [
            (s, e) for s, e in chunk_offsets if e > s
        ]
        if valid_offsets:
            char_start = valid_offsets[0][0]
            char_end = valid_offsets[-1][1]
        else:
            char_start = char_end = 0

        chunk_gold = [
            (s, e, label)
            for s, e, label in entities
            if s >= char_start and e <= char_end
        ]

        bio_ids = [bio2id["O"]] * len(chunk_ids)

        for ent_start, ent_end, label in chunk_gold:
            covered = [
                i for i, (ts, te) in enumerate(chunk_offsets)
                if te > ts and ts < ent_end and te > ent_start
            ]
            if not covered:
                continue

            bio_ids[covered[0]] = bio2id[f"B-{label}"]
            for i in covered[1:]:
                bio_ids[i] = bio2id[f"I-{label}"]

        chunks.append(
            NERChunk(
                doc_id=str(doc_id),
                chunk_id=chunk_id,
                input_ids=chunk_ids,
                offsets=chunk_offsets,
                bio_ids=bio_ids,
                gold_spans=chunk_gold,
                char_start=char_start,
                char_end=char_end,
            )
        )

        chunk_id += 1
        if end >= len(ids):
            break
        start += STRIDE

    return chunks

# --------------------------------------------------
# 4. Rebuild Fold 2 training chunks
# --------------------------------------------------
fold2_text_data = fold2_text_check
fold2_gold_data = fold2_gold_check

a4_train_only_chunks = []

for doc_id, text in fold2_text_data.items():
    a4_train_only_chunks.extend(
        rebuild_chunks(
            doc_id,
            text,
            fold2_gold_data[doc_id]
        )
    )

# --------------------------------------------------
# 5. Verify the rebuilt dataset
# --------------------------------------------------
label_counts = Counter()
for chunk in a4_train_only_chunks:
    label_counts.update(chunk.bio_ids)

print("Fold 2 documents:", len(fold2_text_data))
print("Rebuilt training chunks:", len(a4_train_only_chunks))
print("Total chunk tokens:", sum(len(c.input_ids) for c in a4_train_only_chunks))
print("Gold entity spans in chunks:", sum(len(c.gold_spans) for c in a4_train_only_chunks))
print("Entity-tagged tokens:", sum(v for k, v in label_counts.items() if k != 0))
print("O-tagged tokens:", label_counts[0])
print("BIO mapping matches model:", len(bio2id) == model4.classifier.out_features)
print("Data preparation complete; no training performed.")

In [ ]:
import torch
import torch.nn.functional as F
from collections import Counter

model4.eval()
device = next(model4.parameters()).device

gold_counts = Counter()
pred_counts = Counter()
correct = 0
total = 0
loss_sum = 0.0

with torch.inference_mode():
    for chunk in a4_train_only_chunks:
        ids = torch.tensor(
            chunk.input_ids, dtype=torch.long, device=device
        ).unsqueeze(0)

        targets = torch.tensor(
            chunk.bio_ids, dtype=torch.long, device=device
        )

        output = model4(
            input_ids=ids,
            attention_mask=torch.ones_like(ids)
        )

        if isinstance(output, (tuple, list)):
            output = output[0]

        logits = output[0].float()
        n = min(logits.shape[0], targets.shape[0])
        logits = logits[:n]
        targets = targets[:n]

        loss_sum += F.cross_entropy(
            logits, targets, reduction="sum"
        ).item()

        preds = logits.argmax(dim=-1)
        correct += (preds == targets).sum().item()
        total += n

        gold_counts.update(targets.cpu().tolist())
        pred_counts.update(preds.cpu().tolist())

print(f"Device: {device}")
print(f"Training tokens evaluated: {total:,}")
print(f"Token cross-entropy: {loss_sum / max(total, 1):.4f}")
print(f"Token accuracy: {correct / max(total, 1):.2%}")
print(
    "Gold entity-token share: "
    f"{sum(v for k, v in gold_counts.items() if k != 0) / max(total, 1):.2%}"
)
print(
    "Predicted entity-token share: "
    f"{sum(v for k, v in pred_counts.items() if k != 0) / max(total, 1):.2%}"
)

print("\nPredicted tag distribution:")
for tag_id, count in pred_counts.most_common(15):
    print(
        f"{BIO_LABELS_GROUPED[tag_id]:15s} "
        f"{count:9,d} ({count / max(total, 1):.2%})"
    )

In [ ]:
from pathlib import Path
import re

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
source = source_path.read_text(encoding="utf-8", errors="replace")

terms = [
    "CrossEntropyLoss",
    "cross_entropy",
    "optimizer.step()",
    "lorma_A",
    "requires_grad",
    "IGNORE_INDEX",
    "train_loss",
]

for term in terms:
    hits = [
        m.start()
        for m in re.finditer(re.escape(term), source, flags=re.IGNORECASE)
    ]
    print(f"\n{'=' * 65}\n{term}: {len(hits)} occurrences")

    # Show a few occurrences with surrounding context.
    for pos in hits[:4]:
        print("\n---")
        print(source[max(0, pos - 500): min(len(source), pos + 1000)])

In [ ]:
import torch
import torch.nn.functional as F
from collections import Counter

model4.eval()
device = next(model4.parameters()).device
num_labels = len(BIO_LABELS_GROUPED)

gold_counts = Counter()
pred_counts = Counter()
true_positive = Counter()
probability_sum = torch.zeros(num_labels, dtype=torch.float64)
target_probability_sum = Counter()
total = 0

with torch.inference_mode():
    for chunk in a4_train_only_chunks:
        ids = torch.tensor(
            chunk.input_ids, dtype=torch.long, device=device
        ).unsqueeze(0)

        logits = model4(
            input_ids=ids,
            attention_mask=torch.ones_like(ids)
        )
        if isinstance(logits, (tuple, list)):
            logits = logits[0]
        logits = logits[0].float()

        targets = torch.tensor(
            chunk.bio_ids, dtype=torch.long, device=device
        )
        n = min(logits.shape[0], targets.numel())
        logits, targets = logits[:n], targets[:n]

        probs = torch.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)

        probability_sum += probs.sum(dim=0).cpu().double()
        total += n

        gold_counts.update(targets.cpu().tolist())
        pred_counts.update(preds.cpu().tolist())

        for tag_id in range(num_labels):
            mask = targets == tag_id
            if mask.any():
                target_probability_sum[tag_id] += (
                    probs[mask, tag_id].sum().item()
                )

        for gold_id, pred_id in zip(
            targets.cpu().tolist(), preds.cpu().tolist()
        ):
            if gold_id == pred_id:
                true_positive[gold_id] += 1

print(f"Tokens evaluated: {total:,}")
print(f"Mean predicted probability for O: {probability_sum[0].item()/total:.4f}")

print("\nPer-class results:")
print(f"{'Tag':15s} {'Gold':>8s} {'Pred':>8s} {'Recall':>9s} {'Mean P(gold)':>13s}")

for tag_id, tag in enumerate(BIO_LABELS_GROUPED):
    gold_n = gold_counts[tag_id]
    pred_n = pred_counts[tag_id]
    recall = true_positive[tag_id] / gold_n if gold_n else 0.0
    mean_gold_prob = (
        target_probability_sum[tag_id] / gold_n if gold_n else 0.0
    )
    if gold_n or pred_n:
        print(
            f"{tag:15s} {gold_n:8,d} {pred_n:8,d} "
            f"{recall:8.2%} {mean_gold_prob:12.4f}"
        )

print("\nClassifier bias values:")
bias = model4.classifier.bias.detach().float().cpu()
for i in range(num_labels):
    print(f"{BIO_LABELS_GROUPED[i]:15s} {bias[i].item(): .4f}")

In [ ]:
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)
source = source_path.read_text(encoding="utf-8", errors="replace")

# Find the original LORMA experiment section.
for term in [
    "EXPERIMENT 4 — LORMA",
    "A4_LORMA",
    "lorma_train_loader",
    "train_chunks",
    "lorma_checkpoint",
]:
    positions = []
    start = 0
    while True:
        pos = source.lower().find(term.lower(), start)
        if pos == -1:
            break
        positions.append(pos)
        start = pos + len(term)

    print(f"\n{'=' * 60}\n{term}: {len(positions)} matches")
    for pos in positions[-3:]:
        print(source[max(0, pos-500):min(len(source), pos+1800)])

In [ ]:
from pathlib import Path
import re

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

patterns = [
    r"lorma_encoder\s*=\s*copy\.deepcopy",
    r"model4\s*=\s*A4NER",
    r"experiment\s*=\s*[\"']A4_LORMA",
    r"def\s+train",
    r"for\s+epoch\s+in",
    r"CrossEntropyLoss",
    r"optimizer4\s*=\s*torch\.optim",
    r"lorma_A",
    r"a4_ner_exp4_lorma_epoch3\.pt",
]

for pattern in patterns:
    hits = [
        i for i, line in enumerate(lines)
        if re.search(pattern, line, flags=re.IGNORECASE)
    ]
    print(f"\n{'='*70}\n{pattern}: {len(hits)} matches")

    # Show the earliest relevant match and latest match,
    # without dumping the entire notebook source.
    selected = sorted(set(([hits[0], hits[-1]] if hits else [])))

    for i in selected:
        lo, hi = max(0, i - 10), min(len(lines), i + 24)
        print(f"\n--- Source lines {lo+1}-{hi} ---")
        print("\n".join(f"{j+1}: {lines[j]}" for j in range(lo, hi)))

In [ ]:
from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

for start, end in [
    (13880, 14010),  # Encoder construction and checkpoint loading
    (14945, 15010),  # Freeze/unfreeze and model setup
    (15355, 15530),  # LORMA optimizer, training loop and checkpoint saving
]:
    print(f"\n{'='*25} LINES {start}-{end} {'='*25}")
    print("\n".join(
        f"{i}: {lines[i-1]}"
        for i in range(start, min(end, len(lines)) + 1)
    ))

In [ ]:
from pathlib import Path
import re

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

patterns = [
    r"^\s*encoder\s*=",
    r"encoder\.load_state_dict",
    r"load_state_dict.*encoder",
    r"A4_CHECKPOINT_PATH",
    r"a4_step_22000\.pt",
    r"torch\.load\(",
    r"lorma_encoder\s*=\s*copy\.deepcopy",
]

hits = set()
for pattern in patterns:
    for i, line in enumerate(lines):
        if re.search(pattern, line, flags=re.IGNORECASE):
            # Exclude references that only appear in later diagnostic cells.
            if i < 15500:
                hits.add(i)

for i in sorted(hits):
    lo, hi = max(0, i - 8), min(len(lines), i + 16)
    print(f"\n--- Source lines {lo+1}-{hi} ---")
    print("\n".join(f"{j+1}: {lines[j]}" for j in range(lo, hi)))

In [ ]:
import torch

a4_path = "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
ckpt = torch.load(a4_path, map_location="cpu", weights_only=False)
a4_state = ckpt["model_state_dict"]

live_state = encoder.state_dict()

# Match direct keys or the checkpoint's encoder-prefixed keys.
pairs = []
for key, value in live_state.items():
    candidates = [key, f"encoder.{key}"]
    match = next(
        (candidate for candidate in candidates
         if candidate in a4_state
         and a4_state[candidate].shape == value.shape),
        None
    )
    if match is not None:
        pairs.append((key, match, value.detach().cpu(), a4_state[match]))

print("Matched tensors:", len(pairs))

exact = 0
different = 0
max_diff = 0.0

for live_key, ckpt_key, live, original in pairs:
    diff = (live.float() - original.float()).abs().max().item()
    if diff == 0:
        exact += 1
    else:
        different += 1
        max_diff = max(max_diff, diff)

print("Exactly equal:", exact)
print("Different:", different)
print("Maximum absolute difference:", max_diff)

print("\nFirst 15 differences:")
shown = 0
for live_key, ckpt_key, live, original in pairs:
    diff = (live.float() - original.float()).abs().max().item()
    if diff > 0:
        print(f"{live_key:55s} max_abs_diff={diff:.8f}")
        shown += 1
        if shown >= 15:
            break

In [ ]:
import torch

a4_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)
lorma_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

a4 = torch.load(a4_path, map_location="cpu", weights_only=False)
lorma = torch.load(lorma_path, map_location="cpu", weights_only=False)

a4_state = a4["model_state_dict"]
lorma_state = lorma["model_state_dict"]

print("Phase 1 A4 checkpoint:", len(a4_state), "tensors")
print("Phase 2 LORMA checkpoint:", len(lorma_state), "tensors")

# Compare corresponding frozen base weights.
pairs = []
for key, value in a4_state.items():
    if key in lorma_state and value.shape == lorma_state[key].shape:
        if key.startswith(("token_embedding.", "encoder_pos.", "encoder_layers.")):
            pairs.append((key, value, lorma_state[key]))

exact = 0
different = 0
max_diff = 0.0

for key, original, adapted in pairs:
    diff = (original.float() - adapted.float()).abs().max().item()
    if diff == 0:
        exact += 1
    else:
        different += 1
        max_diff = max(max_diff, diff)

print("\nComparable base tensors:", len(pairs))
print("Exactly equal:", exact)
print("Different:", different)
print("Maximum absolute difference:", max_diff)

print("\nFirst 10 differences:")
shown = 0
for key, original, adapted in pairs:
    diff = (original.float() - adapted.float()).abs().max().item()
    if diff > 0:
        print(f"{key}: {diff:.8f}")
        shown += 1
        if shown == 10:
            break

In [ ]:
print("===== PHASE 1 A4: FIRST 35 KEYS =====")
for key, value in list(a4_state.items())[:35]:
    print(f"{key:65s} {tuple(value.shape)}")

print("\n===== PHASE 2 LORMA: FIRST 35 KEYS =====")
for key, value in list(lorma_state.items())[:35]:
    print(f"{key:65s} {tuple(value.shape)}")

print("\n===== CHECKPOINT METADATA =====")
print("Phase 1 keys:", list(a4.keys()))
print("Phase 2 keys:", list(lorma.keys()))

In [ ]:
import torch

a4_state = a4["model_state_dict"]
lorma_state = lorma["model_state_dict"]

def find_lorma_key(phase1_key):
    # Phase 1 encoder keys become encoder.<key> in Phase 2.
    direct_key = "encoder." + phase1_key

    # Q/V projections are wrapped by LORMALinear, so their
    # original weights are stored under .base.weight/.base.bias.
    for projection in (".attn.q_proj.", ".attn.v_proj."):
        if projection in phase1_key:
            return direct_key.replace(
                projection, projection + "base."
            )

    # Other encoder parameters retain their names after prefixing.
    return direct_key

matched = []
missing = []
shape_mismatch = []

for key, original in a4_state.items():
    # Only compare encoder parameters; skip decoder and any head.
    if key.startswith("decoder_") or key.startswith("classifier."):
        continue

    candidate = find_lorma_key(key)

    if candidate not in lorma_state:
        missing.append((key, candidate))
        continue

    adapted = lorma_state[candidate]

    if original.shape != adapted.shape:
        shape_mismatch.append(
            (key, tuple(original.shape), tuple(adapted.shape))
        )
        continue

    matched.append((key, candidate, original, adapted))

exact = 0
different = 0
max_diff = 0.0
sum_sq = 0.0
num_values = 0

for original_key, adapted_key, original, adapted in matched:
    delta = original.float() - adapted.float()
    diff = delta.abs().max().item()

    if diff == 0:
        exact += 1
    else:
        different += 1
        max_diff = max(max_diff, diff)

    sum_sq += delta.square().sum().item()
    num_values += delta.numel()

print("Matched tensors:", len(matched))
print("Exactly equal:", exact)
print("Different:", different)
print("Missing:", len(missing))
print("Shape mismatches:", len(shape_mismatch))
print("Maximum absolute difference:", max_diff)
print(
    "Overall RMS difference:",
    (sum_sq / num_values) ** 0.5 if num_values else None
)

print("\nFirst 15 changed tensors:")
shown = 0
for original_key, adapted_key, original, adapted in matched:
    diff = (original.float() - adapted.float()).abs().max().item()
    if diff > 0:
        print(f"{original_key}: max_abs_diff={diff:.8f}")
        shown += 1
        if shown >= 15:
            break

print("\nFirst 10 missing mappings:")
for item in missing[:10]:
    print(item)

In [ ]:
from collections import defaultdict

layer_stats = defaultdict(lambda: {"tensors": 0, "changed": 0, "sq": 0.0, "n": 0, "max": 0.0})

for original_key, adapted_key, original, adapted in matched:
    delta = original.float() - adapted.float()
    stats = layer_stats[
        original_key.split(".")[1]
        if original_key.startswith("encoder_layers.")
        else original_key.split(".")[0]
    ]
    stats["tensors"] += 1
    stats["changed"] += int(delta.abs().max().item() > 0)
    stats["sq"] += delta.square().sum().item()
    stats["n"] += delta.numel()
    stats["max"] = max(stats["max"], delta.abs().max().item())

print(f"{'Component':25s} {'Tensors':>8s} {'Changed':>8s} {'RMS diff':>12s} {'Max diff':>12s}")
for name, s in layer_stats.items():
    rms = (s["sq"] / s["n"]) ** 0.5
    print(f"{name:25s} {s['tensors']:8d} {s['changed']:8d} {rms:12.7f} {s['max']:12.7f}")

In [ ]:
from pathlib import Path

source_path = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
)

if not source_path.exists():
    print("Notebook source not found at:", source_path)
    print("Please locate the source file in /kaggle/working.")
else:
    source = source_path.read_text(errors="replace")
    lines = source.splitlines()

    terms = [
        "lorma_encoder = copy.deepcopy(encoder)",
        "a4_step_22000.pt",
        "a4_encoder.load_state_dict",
        "A4_LORMA",
    ]

    for term in terms:
        hits = [
            i for i, line in enumerate(lines)
            if term.lower() in line.lower()
        ]
        print(f"\n{'=' * 20} {term} {'=' * 20}")
        if not hits:
            print("No exact match found.")
            continue

        for i in hits[-3:]:
            start = max(0, i - 20)
            end = min(len(lines), i + 31)
            print(f"\n--- Source lines {start + 1}-{end} ---")
            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = p.read_text(encoding="utf-8", errors="replace").splitlines()

for start, end in [(9850, 9950), (11870, 11980), (14870, 14960), (15100, 15180)]:
    print(f"\n{'='*25} SOURCE LINES {start}-{end} {'='*25}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")

In [ ]:

from pathlib import Path

p = Path(
    "/kaggle/working/.virtual_documents/"
    "__notebook_source__.ipynb"
)
lines = p.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

keywords = [
    "optimizer.step()",
    "encoder.load_state_dict",
    "a4_encoder.load_state_dict",
    "torch.save(",
    "model4.load_state_dict",
    "copy.deepcopy(encoder)",
    "FULL FINE-TUNING",
    "LORMA",
]

for keyword in keywords:
    hits = [
        i for i, line in enumerate(lines)
        if keyword.lower() in line.lower()
    ]
    print(f"\n{'='*18} {keyword}: {len(hits)} matches {'='*18}")

    for i in hits[-8:]:
        start = max(0, i - 5)
        end = min(len(lines), i + 9)
        print(f"\n--- Lines {start+1}-{end} ---")
        for j in range(start, end):
            print(f"{j+1}: {lines[j]}")


In [ ]:

from pathlib import Path

p = Path(
    "/kaggle/working/.virtual_documents/"
    "__notebook_source__.ipynb"
)
lines = p.read_text(
    encoding="utf-8", errors="replace"
).splitlines()

# Find the original LORMA setup.
targets = [
    i for i, line in enumerate(lines)
    if line.strip() == "lorma_encoder = copy.deepcopy(encoder)"
]

print("LORMA setup locations:", [i + 1 for i in targets])

# Inspect the first two actual occurrences.
for target in targets[:2]:
    start = max(0, target - 220)
    end = min(len(lines), target + 100)

    print(f"\n{'=' * 25}")
    print(f"LINES {start + 1}-{end}")
    print("=" * 25)

    for i in range(start, end):
        print(f"{i + 1}: {lines[i]}")


In [ ]:

import torch
from pathlib import Path

phase1_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)
lorma_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

a4 = torch.load(
    phase1_path, map_location="cpu", weights_only=False
)
lor = torch.load(
    lorma_path, map_location="cpu", weights_only=False
)

a4_state = a4["model_state_dict"]
lor_state = lor["model_state_dict"]

def mapped_key(k):
    # Phase 1 encoder keys become encoder.<key> in Phase 2.
    key = "encoder." + k
    for proj in (".attn.q_proj.", ".attn.v_proj."):
        if proj in k:
            return key.replace(proj, proj + "base.")
    return key

matched = []
missing = []

for key, value in a4_state.items():
    if key.startswith(("classifier.", "decoder_")):
        continue

    target = mapped_key(key)
    if target not in lor_state:
        missing.append((key, target))
    elif value.shape == lor_state[target].shape:
        matched.append((key, target, value, lor_state[target]))

equal = sum(
    torch.equal(original, saved)
    for _, _, original, saved in matched
)
different = len(matched) - equal

print("Matched encoder tensors:", len(matched))
print("Exactly equal tensors:", equal)
print("Different tensors:", different)
print("Missing mappings:", len(missing))
print("Examples of differences:")
for key, _, original, saved in [
    item for item in matched
    if not torch.equal(item[2], item[3])
][:8]:
    print(
        key,
        "max_abs_diff=",
        (original - saved).abs().max().item()
    )


In [ ]:

import torch

a4_state = torch.load(
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt",
    map_location="cpu",
    weights_only=False
)["model_state_dict"]

lor_state = torch.load(
    "/kaggle/input/datasets/aq1de3/a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt",
    map_location="cpu",
    weights_only=False
)["model_state_dict"]

def phase1_to_lorma_key(key):
    target = "encoder." + key
    for proj in (".attn.q_proj.", ".attn.v_proj."):
        if proj in key:
            target = target.replace(proj, proj + "base.")
            break
    return target

# Separate Q/V base weights from other encoder weights.
groups = {"Q/V base": [], "Other encoder": []}

for key, original in a4_state.items():
    if key.startswith(("classifier.", "decoder_")):
        continue

    target = phase1_to_lorma_key(key)
    if target not in lor_state or original.shape != lor_state[target].shape:
        continue

    group = (
        "Q/V base"
        if ".attn.q_proj." in key or ".attn.v_proj." in key
        else "Other encoder"
    )
    saved = lor_state[target]
    groups[group].append((
        key,
        torch.equal(original, saved),
        (original - saved).abs().max().item()
    ))

for group, rows in groups.items():
    print(f"\n{group}: {len(rows)} tensors")
    print("Exactly equal:", sum(equal for _, equal, _ in rows))
    print("Different:", sum(not equal for _, equal, _ in rows))
    print("Largest absolute difference:",
          max((diff for _, _, diff in rows), default=0.0))
    for key, equal, diff in rows[:4]:
        print(f"  {key}: equal={equal}, max_diff={diff:.6g}")


In [ ]:

import torch

phase1_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)

phase1 = torch.load(
    phase1_path,
    map_location="cpu",
    weights_only=False
)["model_state_dict"]

# Use the current encoder object from the notebook.
current = encoder.state_dict()

matched = []
missing = []

for key, value in phase1.items():
    if key.startswith(("classifier.", "decoder_")):
        continue

    if key not in current:
        missing.append(key)
    elif value.shape == current[key].shape:
        matched.append((key, value, current[key]))

print("Matched tensors:", len(matched))
print("Missing tensors:", len(missing))
print("Exactly equal:", sum(
    torch.equal(a, b) for _, a, b in matched
))
print("Different:", sum(
    not torch.equal(a, b) for _, a, b in matched
))

for key, original, live in [
    row for row in matched
    if not torch.equal(row[1], row[2])
][:8]:
    print(
        key,
        "max_abs_diff=",
        (original - live).abs().max().item()
    )


In [ ]:

import torch

print("Available model variables:")

for name in [
    "model4", "model", "a4_encoder",
    "lorma_encoder", "student", "student_model"
]:
    obj = globals().get(name)

    if obj is not None and hasattr(obj, "state_dict"):
        print(f"{name}: {type(obj).__name__}")
        if hasattr(obj, "encoder"):
            print("  contains .encoder")
        if hasattr(obj, "encoder_layers"):
            print("  contains .encoder_layers")

print("\nCheckpoint comparison candidates:")

for name in ["model4", "model", "a4_encoder",
             "lorma_encoder", "student", "student_model"]:
    obj = globals().get(name)
    if obj is None or not hasattr(obj, "state_dict"):
        continue

    sd = obj.state_dict()
    print(f"{name}: {len(sd)} state-dict entries")


In [ ]:

import torch

phase1_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)

phase1 = torch.load(
    phase1_path,
    map_location="cpu",
    weights_only=False
)["model_state_dict"]

live = model4.state_dict()

def to_live_key(key):
    target = "encoder." + key

    for projection in (
        ".attn.q_proj.", ".attn.v_proj."
    ):
        if projection in key:
            target = target.replace(
                projection, projection + "base."
            )
            break

    return target

matched, missing = [], []

for key, original in phase1.items():
    if key.startswith(("classifier.", "decoder_")):
        continue

    target = to_live_key(key)

    if target not in live:
        missing.append((key, target))
    elif original.shape == live[target].shape:
        matched.append((key, original, live[target]))

print("Matched:", len(matched))
print("Missing:", len(missing))
print("Exactly equal:", sum(
    torch.equal(a, b) for _, a, b in matched
))
print("Different:", sum(
    not torch.equal(a, b) for _, a, b in matched
))

for key, original, current in [
    row for row in matched
    if not torch.equal(row[1], row[2])
][:8]:
    print(
        key,
        "max_abs_diff =",
        (original - current).abs().max().item()
    )


In [ ]:

import torch

phase1_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)

phase1 = torch.load(
    phase1_path,
    map_location="cpu",
    weights_only=False
)["model_state_dict"]

live = model4.state_dict()

def to_live_key(key):
    target = "encoder." + key
    for projection in (".attn.q_proj.", ".attn.v_proj."):
        if projection in key:
            target = target.replace(
                projection, projection + "base."
            )
            break
    return target

matched, missing = [], []

for key, original in phase1.items():
    if key.startswith(("classifier.", "decoder_")):
        continue

    target = to_live_key(key)

    if target not in live:
        missing.append((key, target))
    elif original.shape == live[target].shape:
        # Move both tensors to CPU for comparison.
        matched.append((
            key,
            original.detach().cpu(),
            live[target].detach().cpu()
        ))

equal_count = sum(
    torch.equal(original, current)
    for _, original, current in matched
)

print("Matched tensors:", len(matched))
print("Missing mappings:", len(missing))
print("Exactly equal:", equal_count)
print("Different:", len(matched) - equal_count)

for key, original, current in [
    row for row in matched
    if not torch.equal(row[1], row[2])
][:8]:
    print(
        key,
        "max_abs_diff =",
        (original - current).abs().max().item()
    )


In [ ]:

import torch

phase1_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)
lorma_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt"
)

p1 = torch.load(
    phase1_path, map_location="cpu", weights_only=False
)
lor = torch.load(
    lorma_path, map_location="cpu", weights_only=False
)

print("PHASE 1 CHECKPOINT")
print("Top-level keys:", list(p1.keys()))
print("Model state entries:", len(p1["model_state_dict"]))

print("\nLORMA CHECKPOINT")
for key, value in lor.items():
    if key != "model_state_dict" and key != "optimizer_state_dict":
        print(f"{key}: {value}")

# Verify the frozen projection weights and report differences.
a4_state = p1["model_state_dict"]
lor_state = lor["model_state_dict"]

def mapped_key(key):
    target = "encoder." + key
    for proj in (".attn.q_proj.", ".attn.v_proj."):
        if proj in key:
            target = target.replace(proj, proj + "base.")
            break
    return target

for prefix in (
    "encoder_layers.0.attn.q_proj.",
    "encoder_layers.0.attn.v_proj.",
    "encoder_layers.0.attn.k_proj.",
):
    print(f"\n{prefix}")
    for key in a4_state:
        if key.startswith(prefix):
            target = mapped_key(key)
            if target in lor_state:
                original = a4_state[key].float()
                saved = lor_state[target].float()
                print(
                    key.rsplit(".", 1)[-1],
                    "max_diff =", (original - saved).abs().max().item(),
                    "mean_diff =", (original - saved).abs().mean().item()
                )


In [ ]:

import torch

p1 = torch.load(
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt",
    map_location="cpu",
    weights_only=False
)["model_state_dict"]

lor = torch.load(
    "/kaggle/input/datasets/aq1de3/a4-ner-exp4-lorma-epoch3/"
    "a4_ner_exp4_lorma_epoch3.pt",
    map_location="cpu",
    weights_only=False
)["model_state_dict"]

def map_key(k):
    target = "encoder." + k
    for proj in (".attn.q_proj.", ".attn.v_proj."):
        if proj in k:
            target = target.replace(proj, proj + "base.")
            break
    return target

rows = []

for key, original in p1.items():
    if key.startswith(("classifier.", "decoder_")):
        continue

    target = map_key(key)
    if target not in lor or original.shape != lor[target].shape:
        continue

    diff = (
        original.float() - lor[target].float()
    ).abs()

    rows.append((
        key,
        torch.equal(original, lor[target]),
        diff.max().item(),
        diff.mean().item()
    ))

print("Compared tensors:", len(rows))
print("Exactly equal:", sum(r[1] for r in rows))
print("Different:", sum(not r[1] for r in rows))

print("\nLargest differences:")
for key, equal, max_diff, mean_diff in sorted(
    rows, key=lambda r: r[2], reverse=True
)[:15]:
    print(
        f"{key}: max={max_diff:.6g}, "
        f"mean={mean_diff:.6g}, equal={equal}"
    )


In [ ]:

import torch

# Reuse the A4Encoder class already defined in your notebook.
clean_encoder = A4Encoder(
    vocab_size=50000,
    hidden_size=512,
    max_length=1024,
    num_layers=6,
    num_heads=8,
    ffn_size=2048
)

checkpoint = torch.load(
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt",
    map_location="cpu",
    weights_only=False
)

state = checkpoint["model_state_dict"]

encoder_state = {
    key: value
    for key, value in state.items()
    if key.startswith((
        "token_embedding.",
        "encoder_pos.",
        "encoder_layers."
    ))
}

result = clean_encoder.load_state_dict(
    encoder_state,
    strict=True
)

print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)
print("Encoder device:", next(clean_encoder.parameters()).device)
print("Encoder tensors:", len(clean_encoder.state_dict()))

assert not result.missing_keys
assert not result.unexpected_keys

print("Clean Phase 1 encoder loaded successfully.")


In [ ]:

import inspect

for name in [
    "A4Encoder",
    "A4EncoderForNER",
    "A4EncoderLayer",
    "A4SelfAttention",
    "A4NER",
    "LORMALinear"
]:
    obj = globals().get(name)
    print(f"\n{name}:",
          type(obj).__name__ if obj is not None else "NOT DEFINED")

    if obj is not None and inspect.isclass(obj):
        print("  Module:", obj.__module__)
        print("  Constructor:", inspect.signature(obj))


In [ ]:

import inspect

print("=" * 70)
print("A4EncoderForNER SOURCE")
print("=" * 70)
print(inspect.getsource(A4EncoderForNER))

print("\n" + "=" * 70)
print("A4NER SOURCE")
print("=" * 70)
print(inspect.getsource(A4NER))

print("\n" + "=" * 70)
print("A4EncoderForNER ATTRIBUTES")
print("=" * 70)
print([
    name for name in dir(A4EncoderForNER)
    if not name.startswith("__")
])


In [ ]:

import torch

print("=== Existing model ===")
print("model4 exists:", "model4" in globals())

print("\n=== Model architecture ===")
print(model4)

print("\n=== Encoder attributes ===")
print([
    name for name in dir(model4.encoder)
    if not name.startswith("__")
])

print("\n=== First 40 parameter names and shapes ===")
for i, (name, param) in enumerate(model4.named_parameters()):
    if i >= 40:
        break
    print(f"{name:65s} {tuple(param.shape)}")

print("\n=== Checkpoint keys ===")
ckpt_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)
ckpt = torch.load(
    ckpt_path, map_location="cpu", weights_only=False
)
state = ckpt["model_state_dict"]

print("Checkpoint entries:", len(state))
for i, (name, tensor) in enumerate(state.items()):
    if i >= 35:
        break
    print(f"{name:65s} {tuple(tensor.shape)}")


In [ ]:

import copy
import torch

device = next(model4.parameters()).device

# Create a separate copy of the existing encoder architecture.
clean_encoder = copy.deepcopy(model4.encoder).cpu()

# Load the original Phase 1 checkpoint.
ckpt_path = (
    "/kaggle/input/datasets/aq1de3/"
    "a4check/a4_step_22000.pt"
)
ckpt = torch.load(
    ckpt_path, map_location="cpu", weights_only=False
)
phase1 = ckpt["model_state_dict"]

# Transfer only the original encoder parameters.
target = clean_encoder.state_dict()
loaded, skipped = [], []

with torch.no_grad():
    for key, value in phase1.items():
        if not key.startswith(
            ("token_embedding.", "encoder_pos.", "encoder_layers.")
        ):
            continue

        # Q/V are LORMA wrappers in the copied encoder.
        candidates = [key]
        for projection in ("q_proj", "v_proj"):
            marker = f".attn.{projection}."
            if marker in key:
                candidates.append(
                    key.replace(marker, marker + "base.")
                )

        matched = False
        for dest_key in candidates:
            if (
                dest_key in target
                and target[dest_key].shape == value.shape
            ):
                target[dest_key] = value.detach().clone()
                loaded.append((key, dest_key))
                matched = True
                break

        if not matched:
            skipped.append((key, tuple(value.shape)))

result = clean_encoder.load_state_dict(target, strict=True)
clean_encoder = clean_encoder.to(device)

print("Mapped checkpoint tensors:", len(loaded))
print("Unmatched encoder tensors:", skipped)
print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)

assert not skipped, f"Some encoder tensors were not mapped: {skipped}"
assert not result.missing_keys
assert not result.unexpected_keys

# Verify representative weights against the source checkpoint.
checks = [
    ("token_embedding.weight", "token_embedding.weight"),
    ("encoder_pos.weight", "encoder_pos.weight"),
    (
        "encoder_layers.0.attn.q_proj.weight",
        "encoder_layers.0.attn.q_proj.base.weight",
    ),
    (
        "encoder_layers.0.attn.v_proj.weight",
        "encoder_layers.0.attn.v_proj.base.weight",
    ),
    (
        "encoder_layers.0.ffn.net.0.weight",
        "encoder_layers.0.ffn.net.0.weight",
    ),
]

for source_key, target_key in checks:
    source = phase1[source_key].to(device)
    actual = clean_encoder.state_dict()[target_key]
    print(
        source_key,
        "| exact match:",
        torch.equal(source, actual),
        "| max difference:",
        (source - actual).abs().max().item(),
    )

print("\nClean encoder verification completed.")


In [ ]:

# Inspect available variables without changing the existing models.
for name in [
    "model4", "clean_encoder", "tokenizer",
    "labels", "label2id", "id2label",
    "train_chunks", "val_chunks",
    "train_dataset", "val_dataset",
    "MAX_TOKENS", "STRIDE"
]:
    if name not in globals():
        print(f"{name}: NOT DEFINED")
        continue

    obj = globals()[name]
    try:
        length = len(obj)
    except (TypeError, AttributeError):
        length = "N/A"

    print(
        f"{name}: type={type(obj).__name__}, "
        f"length={length}"
    )

print("\n=== Existing model device ===")
print(next(model4.parameters()).device)

print("\n=== Clean encoder device ===")
print(next(clean_encoder.parameters()).device)


In [ ]:

import os
import json

paths = {
    "Fold 1 text": "/kaggle/input/datasets/aq1de3/fold1-tex/fold1-text.json",
    "Fold 1 gold": "/kaggle/input/datasets/aq1de3/fold1-gold1/fold1-gold.json",
    "Fold 2 text": "/kaggle/input/datasets/aq1de3/fold2-tex/fold2-text.json",
    "Fold 2 gold": "/kaggle/input/datasets/aq1de3/fold2-go/fold2-gold.json",
    "Labels": "/kaggle/input/datasets/aq1de3/labelses/labels.txt",
}

for name, path in paths.items():
    print(f"{name}: {'FOUND' if os.path.isfile(path) else 'MISSING'}")
    print(" ", path)

print("\nMAX_TOKENS =", MAX_TOKENS)
print("STRIDE =", STRIDE)
print("Label count =", len(labels))
print("Label mapping =", label2id)

# Inspect the JSON schema without modifying anything.
for name in ["Fold 1 text", "Fold 1 gold", "Fold 2 text", "Fold 2 gold"]:
    path = paths[name]
    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        print(f"\n{name}: type={type(obj).__name__}")
        if isinstance(obj, dict):
            print("Keys:", list(obj.keys())[:10])
        elif isinstance(obj, list):
            print("Document count:", len(obj))
            if obj:
                print("First item type:", type(obj[0]).__name__)
                print("First item preview:", repr(obj[0])[:500])


In [ ]:

import json

fold1_text_path = (
    "/kaggle/input/datasets/aq1de3/"
    "fold1-tex/fold1-text.json"
)
fold1_gold_path = (
    "/kaggle/input/datasets/aq1de3/"
    "fold1-gold1/fold1-gold.json"
)
fold2_text_path = (
    "/kaggle/input/datasets/aq1de3/"
    "fold2-tex/fold2-text.json"
)
fold2_gold_path = (
    "/kaggle/input/datasets/aq1de3/"
    "fold2-go/fold2-gold.json"
)

with open(fold1_text_path, encoding="utf-8") as f:
    fold1_text = json.load(f)
with open(fold1_gold_path, encoding="utf-8") as f:
    fold1_gold = json.load(f)
with open(fold2_text_path, encoding="utf-8") as f:
    fold2_text = json.load(f)
with open(fold2_gold_path, encoding="utf-8") as f:
    fold2_gold = json.load(f)

for fold_name, texts, golds in [
    ("Fold 1", fold1_text, fold1_gold),
    ("Fold 2", fold2_text, fold2_gold),
]:
    print(f"\n{'=' * 60}\n{fold_name}")
    print("Text documents:", len(texts))
    print("Gold documents:", len(golds))
    print("Matching document IDs:", set(texts) == set(golds))

    doc_id = next(iter(texts))
    text = texts[doc_id]
    spans = golds[doc_id]

    print("\nExample document ID:", doc_id)
    print("Text type:", type(text).__name__)
    print("Text preview:", repr(text)[:600])
    print("Gold annotation type:", type(spans).__name__)
    print("Gold annotation preview:", repr(spans)[:1000])


In [ ]:

import torch
from collections import Counter

# Keep the existing tokenizer and label mapping.
MAX_TOKENS = 1024
STRIDE = 768

def make_chunks(texts, annotations, tokenizer, label2id,
                max_tokens=1024, stride=768):
    chunks = []
    stats = Counter()
    bad_examples = []

    for doc_id, text in texts.items():
        entities = annotations.get(doc_id, [])

        # Validate annotation boundaries and entity text.
        valid_entities = []
        for start, end, label, entity_text in entities:
            if not (0 <= start < end <= len(text)):
                stats["invalid_offsets"] += 1
                if len(bad_examples) < 5:
                    bad_examples.append(
                        (doc_id, start, end, label, entity_text)
                    )
                continue

            if text[start:end] != entity_text:
                stats["text_mismatch"] += 1
                if len(bad_examples) < 5:
                    bad_examples.append(
                        (doc_id, start, end, label, entity_text,
                         text[start:end])
                    )
                continue

            if label not in label2id:
                stats["unknown_labels"] += 1
                continue

            valid_entities.append((start, end, label, entity_text))

        # Offsets preserve the relationship between text and annotations.
        encoded = tokenizer.encode(text)
        ids = encoded.ids
        offsets = encoded.offsets

        # The tokenizer may return special-token offsets; inspect and
        # exclude special tokens using their zero-length offsets.
        usable = [
            i for i, (start, end) in enumerate(offsets)
            if end > start
        ]

        stats["documents"] += 1
        stats["tokens"] += len(ids)

        # Build chunks on token indices.
        step = stride
        for chunk_start in range(0, len(ids), step):
            chunk_end = min(chunk_start + max_tokens, len(ids))
            if chunk_start >= len(ids):
                break

            chunk_ids = ids[chunk_start:chunk_end]
            chunk_offsets = offsets[chunk_start:chunk_end]
            token_labels = [label2id["O"]] * len(chunk_ids)

            # Assign BIO labels when token offsets overlap an entity.
            for start, end, label, _ in valid_entities:
                token_indices = [
                    j for j, (ts, te) in enumerate(chunk_offsets)
                    if te > ts and ts >= start and te <= end
                ]

                # Avoid assigning partial entities at chunk boundaries.
                if not token_indices:
                    continue

                first = token_indices[0]
                last = token_indices[-1]

                # Only assign an entity when its complete character span
                # is contained in this chunk's non-special token offsets.
                if (
                    chunk_offsets[first][0] != start
                    or chunk_offsets[last][1] != end
                ):
                    continue

                token_labels[first] = label2id[f"B-{label}"]
                for j in token_indices[1:]:
                    token_labels[j] = label2id[f"I-{label}"]

            chunks.append({
                "input_ids": chunk_ids,
                "labels": token_labels,
                "doc_id": doc_id,
                "chunk_start": chunk_start,
            })
            stats["chunks"] += 1

            if chunk_end == len(ids):
                break

    return chunks, stats, bad_examples


train_chunks, train_stats, train_bad = make_chunks(
    fold2_text, fold2_gold, tokenizer, label2id,
    MAX_TOKENS, STRIDE
)

val_chunks, val_stats, val_bad = make_chunks(
    fold1_text, fold1_gold, tokenizer, label2id,
    MAX_TOKENS, STRIDE
)

print("TRAIN STATS:", dict(train_stats))
print("VALIDATION STATS:", dict(val_stats))
print("\nTRAIN OFFSET ISSUES:", train_bad)
print("\nVALIDATION OFFSET ISSUES:", val_bad)

def summarize_labels(chunks):
    counts = Counter()
    for chunk in chunks:
        counts.update(chunk["labels"])
    print("O tokens:", counts[label2id["O"]])
    print("Entity tokens:", sum(
        n for idx, n in counts.items()
        if idx != label2id["O"]
    ))
    print("Total labeled tokens:", sum(counts.values()))
    print("Non-O labels:")
    for name, idx in label2id.items():
        if name != "O":
            print(f"  {name}: {counts[idx]}")

print("\n=== TRAIN LABEL DISTRIBUTION ===")
summarize_labels(train_chunks)
print("\n=== VALIDATION LABEL DISTRIBUTION ===")
summarize_labels(val_chunks)


In [ ]:

from collections import Counter

def make_chunks_fixed(texts, annotations, tokenizer, label2id,
                      max_tokens=1024, stride=768):
    chunks = []
    stats = Counter()
    bad_examples = []

    for doc_id, text in texts.items():
        entities = []

        for start, end, entity_type, entity_text in annotations[doc_id]:
            if not (0 <= start < end <= len(text)):
                stats["invalid_offsets"] += 1
                continue

            if text[start:end] != entity_text:
                stats["text_mismatch"] += 1
                if len(bad_examples) < 5:
                    bad_examples.append(
                        (doc_id, start, end, entity_type, entity_text,
                         text[start:end])
                    )
                continue

            if f"B-{entity_type}" not in label2id:
                stats["unknown_labels"] += 1
                continue

            entities.append((start, end, entity_type, entity_text))

        # Keep special-token offsets, if any, so IDs and labels stay aligned.
        encoded = tokenizer.encode(text)
        ids = encoded.ids
        offsets = encoded.offsets

        stats["documents"] += 1
        stats["tokens_before_chunking"] += len(ids)
        stats["gold_entities"] += len(entities)

        for chunk_start in range(0, len(ids), stride):
            chunk_end = min(chunk_start + max_tokens, len(ids))
            chunk_ids = ids[chunk_start:chunk_end]
            chunk_offsets = offsets[chunk_start:chunk_end]
            token_labels = [label2id["O"]] * len(chunk_ids)

            for start, end, entity_type, _ in entities:
                overlapping = [
                    j for j, (ts, te) in enumerate(chunk_offsets)
                    if te > ts and ts < end and te > start
                ]

                if not overlapping:
                    continue

                first, last = overlapping[0], overlapping[-1]

                # Assign only if the complete entity is represented by
                # token offsets in this chunk.
                if (
                    chunk_offsets[first][0] != start
                    or chunk_offsets[last][1] != end
                ):
                    continue

                # Avoid overwriting another entity's token labels.
                if any(token_labels[j] != label2id["O"]
                       for j in overlapping):
                    stats["overlapping_entities_skipped"] += 1
                    continue

                token_labels[first] = label2id[f"B-{entity_type}"]
                for j in overlapping[1:]:
                    token_labels[j] = label2id[f"I-{entity_type}"]

                stats["entities_labeled"] += 1

            chunks.append({
                "input_ids": chunk_ids,
                "labels": token_labels,
                "doc_id": doc_id,
                "chunk_start": chunk_start,
            })
            stats["chunks"] += 1

            if chunk_end == len(ids):
                break

    return chunks, stats, bad_examples


train_chunks, train_stats, train_bad = make_chunks_fixed(
    fold2_text, fold2_gold, tokenizer, label2id,
    MAX_TOKENS, STRIDE
)

val_chunks, val_stats, val_bad = make_chunks_fixed(
    fold1_text, fold1_gold, tokenizer, label2id,
    MAX_TOKENS, STRIDE
)

def report_chunks(name, chunks, stats, bad):
    counts = Counter(
        label
        for chunk in chunks
        for label in chunk["labels"]
    )
    print(f"\n=== {name} ===")
    print("Stats:", dict(stats))
    print("Offset mismatch examples:", bad)
    print("Chunk count:", len(chunks))
    print("O tokens:", counts[label2id["O"]])
    print("Entity tokens:", sum(
        n for idx, n in counts.items() if idx != label2id["O"]
    ))
    print("BIO label counts:")
    for label, idx in label2id.items():
        print(f"  {label}: {counts[idx]}")

report_chunks("TRAIN", train_chunks, train_stats, train_bad)
report_chunks("VALIDATION", val_chunks, val_stats, val_bad)


In [ ]:

from collections import Counter

def diagnose_alignment(texts, annotations, tokenizer):
    stats = Counter()
    examples = []

    for doc_id, text in texts.items():
        offsets = tokenizer.encode(text).offsets
        entities = annotations[doc_id]

        for start, end, entity_type, entity_text in entities:
            stats["gold_entities"] += 1

            # Tokens whose character offsets overlap the entity.
            overlapping = [
                (i, ts, te)
                for i, (ts, te) in enumerate(offsets)
                if te > ts and ts < end and te > start
            ]

            if not overlapping:
                stats["no_overlapping_tokens"] += 1
                continue

            first = overlapping[0]
            last = overlapping[-1]

            if first[1] == start and last[2] == end:
                stats["exact_boundary_alignment"] += 1
            else:
                stats["boundary_mismatch"] += 1
                if len(examples) < 12:
                    examples.append({
                        "doc_id": doc_id,
                        "entity": entity_text,
                        "type": entity_type,
                        "gold_span": (start, end),
                        "first_token_offset": (first[1], first[2]),
                        "last_token_offset": (last[1], last[2]),
                        "token_count": len(overlapping),
                        "overlap_preview": overlapping[:3],
                    })

    return stats, examples


train_align_stats, train_align_examples = diagnose_alignment(
    fold2_text, fold2_gold, tokenizer
)
val_align_stats, val_align_examples = diagnose_alignment(
    fold1_text, fold1_gold, tokenizer
)

print("TRAIN ALIGNMENT:", dict(train_align_stats))
print("\nVALIDATION ALIGNMENT:", dict(val_align_stats))
print("\nTRAIN EXAMPLES:")
for item in train_align_examples:
    print(item)
print("\nVALIDATION EXAMPLES:")
for item in val_align_examples:
    print(item)


In [ ]:

from collections import Counter

def test_offset_shifts(texts, annotations, tokenizer):
    results = Counter()
    examples = []

    for doc_id, text in texts.items():
        offsets = tokenizer.encode(text).offsets

        for start, end, entity_type, entity_text in annotations[doc_id]:
            # Confirm the annotation text itself is correct.
            if text[start:end] != entity_text:
                results["gold_text_mismatch"] += 1
                continue

            overlapping = [
                (ts, te)
                for ts, te in offsets
                if te > start and ts < end
            ]

            if not overlapping:
                results["no_overlap"] += 1
                continue

            token_start = overlapping[0][0]
            token_end = overlapping[-1][1]

            results["original_exact"] += int(
                token_start == start and token_end == end
            )

            # Test small shifts only as a diagnostic.
            for shift in (-1, 0, 1):
                results[f"start_shift_{shift}"] += int(
                    token_start == start + shift
                    and token_end == end
                )

            if len(examples) < 10:
                examples.append({
                    "entity": entity_text,
                    "gold": (start, end),
                    "token_span": (token_start, token_end),
                    "text_before": text[max(0, start-3):start],
                    "text_at_gold": text[start:end],
                    "text_at_token_start": text[token_start:token_end],
                })

    return results, examples


for name, texts, golds in [
    ("Fold 2", fold2_text, fold2_gold),
    ("Fold 1", fold1_text, fold1_gold),
]:
    stats, examples = test_offset_shifts(texts, golds, tokenizer)
    print(f"\n=== {name} ===")
    print(dict(stats))
    for example in examples:
        print(example)


In [ ]:

from collections import Counter

def make_chunks_aligned(texts, annotations, tokenizer, label2id,
                        max_tokens=1024, stride=768):
    chunks = []
    stats = Counter()

    for doc_id, text in texts.items():
        encoded = tokenizer.encode(text)
        ids = encoded.ids
        offsets = encoded.offsets
        entities = annotations[doc_id]

        stats["documents"] += 1
        stats["tokens_before_chunking"] += len(ids)
        stats["gold_entities"] += len(entities)

        # Map every token to the gold entity it overlaps.
        token_tags = [None] * len(ids)

        for start, end, entity_type, entity_text in entities:
            if text[start:end] != entity_text:
                stats["gold_text_mismatch"] += 1
                continue

            overlapping = [
                i for i, (ts, te) in enumerate(offsets)
                if te > ts and ts < end and te > start
            ]
            if not overlapping:
                stats["entities_without_tokens"] += 1
                continue

            first, last = overlapping[0], overlapping[-1]

            # If a token extends beyond the entity's right boundary,
            # don't silently assign a partial entity.
            if offsets[last][1] > end:
                stats["right_boundary_crossing"] += 1
                continue

            # Tokenizers may attach whitespace to the first token.
            # Accept a start offset before the gold span only if the
            # extra characters are whitespace.
            prefix = text[offsets[first][0]:start]
            if prefix.strip():
                stats["nonspace_left_boundary_crossing"] += 1
                continue

            if any(token_tags[i] is not None for i in overlapping):
                stats["overlap_conflicts"] += 1
                continue

            token_tags[first] = f"B-{entity_type}"
            for i in overlapping[1:]:
                token_tags[i] = f"I-{entity_type}"

            stats["entities_aligned"] += 1

        for chunk_start in range(0, len(ids), stride):
            chunk_end = min(chunk_start + max_tokens, len(ids))
            chunk_ids = ids[chunk_start:chunk_end]
            chunk_tags = token_tags[chunk_start:chunk_end]

            labels = [
                label2id[tag] if tag is not None else label2id["O"]
                for tag in chunk_tags
            ]

            chunks.append({
                "input_ids": chunk_ids,
                "labels": labels,
                "doc_id": doc_id,
                "chunk_start": chunk_start,
            })
            stats["chunks"] += 1

            if chunk_end == len(ids):
                break

    return chunks, stats


train_chunks, train_stats = make_chunks_aligned(
    fold2_text, fold2_gold, tokenizer, label2id,
    MAX_TOKENS, STRIDE
)
val_chunks, val_stats = make_chunks_aligned(
    fold1_text, fold1_gold, tokenizer, label2id,
    MAX_TOKENS, STRIDE
)

for name, chunks, stats in [
    ("TRAIN", train_chunks, train_stats),
    ("VALIDATION", val_chunks, val_stats),
]:
    counts = Counter(
        tag for chunk in chunks for tag in chunk["labels"]
    )
    print(f"\n=== {name} ===")
    print("Stats:", dict(stats))
    print("Entity tokens:", sum(
        n for idx, n in counts.items() if idx != label2id["O"]
    ))
    print("O tokens:", counts[label2id["O"]])
    print("BIO label counts:")
    for label, idx in label2id.items():
        print(f"  {label}: {counts[idx]}")


In [ ]:

import torch
from collections import Counter, defaultdict

def check_chunk_overlap(chunks):
    by_doc = defaultdict(list)
    for chunk in chunks:
        start = chunk["chunk_start"]
        end = start + len(chunk["input_ids"])
        by_doc[chunk["doc_id"]].append((start, end))

    total_overlap = 0
    docs_with_overlap = 0

    for doc_id, spans in by_doc.items():
        spans.sort()
        overlap = sum(
            max(0, min(a_end, b_end) - max(a_start, b_start))
            for i, (a_start, a_end) in enumerate(spans)
            for b_start, b_end in spans[i + 1:]
        )
        total_overlap += overlap
        docs_with_overlap += int(overlap > 0)

    return total_overlap, docs_with_overlap


for name, chunks in [
    ("TRAIN", train_chunks),
    ("VALIDATION", val_chunks),
]:
    overlap_tokens, overlap_docs = check_chunk_overlap(chunks)
    print(
        f"{name}: chunks={len(chunks)}, "
        f"overlap token occurrences={overlap_tokens}, "
        f"documents with overlap={overlap_docs}"
    )

print("\n=== Model forward-pass check ===")
model4.eval()
sample = train_chunks[0]
input_ids = torch.tensor(
    sample["input_ids"][:128],
    dtype=torch.long,
    device=next(model4.parameters()).device
).unsqueeze(0)

with torch.no_grad():
    output = model4(input_ids)

if isinstance(output, dict):
    print("Output type: dict")
    print("Keys:", list(output.keys()))
    for key, value in output.items():
        if torch.is_tensor(value):
            print(key, tuple(value.shape))
elif torch.is_tensor(output):
    print("Output type: tensor")
    print("Shape:", tuple(output.shape))
else:
    print("Output type:", type(output))
    print(repr(output)[:500])


In [ ]:

from collections import Counter

def make_nonoverlap_chunks(chunks, max_tokens=1024):
    # Reconstruct each document's token and label sequence.
    documents = {}

    for chunk in chunks:
        doc_id = chunk["doc_id"]
        start = chunk["chunk_start"]
        ids = chunk["input_ids"]
        labels_ = chunk["labels"]

        if doc_id not in documents:
            documents[doc_id] = {}

        for offset, (token_id, label_id) in enumerate(zip(ids, labels_)):
            pos = start + offset
            # Overlapping copies should have the same gold label.
            if pos in documents[doc_id]:
                old_id, old_label = documents[doc_id][pos]
                if old_id != token_id:
                    raise ValueError(
                        f"Token mismatch for {doc_id} at {pos}"
                    )
                if old_label != label_id:
                    raise ValueError(
                        f"Label mismatch for {doc_id} at {pos}: "
                        f"{old_label} vs {label_id}"
                    )
            else:
                documents[doc_id][pos] = (token_id, label_id)

    rebuilt = []
    for doc_id, positions in documents.items():
        ordered = sorted(positions.items())
        token_ids = [pair[1][0] for pair in ordered]
        labels_ = [pair[1][1] for pair in ordered]

        for start in range(0, len(token_ids), max_tokens):
            end = min(start + max_tokens, len(token_ids))
            rebuilt.append({
                "input_ids": token_ids[start:end],
                "labels": labels_[start:end],
                "doc_id": doc_id,
                "chunk_start": start,
            })

    return rebuilt


train_chunks_nonoverlap = make_nonoverlap_chunks(train_chunks)
val_chunks_nonoverlap = make_nonoverlap_chunks(val_chunks)

def chunk_summary(name, chunks):
    counts = Counter(
        label for chunk in chunks for label in chunk["labels"]
    )
    total = sum(counts.values())
    entities = total - counts[label2id["O"]]

    print(f"\n{name}")
    print("Chunks:", len(chunks))
    print("Tokens:", total)
    print("Entity tokens:", entities)
    print("O tokens:", counts[label2id["O"]])
    print("Entity token fraction:", round(entities / max(total, 1), 4))

chunk_summary("TRAIN NON-OVERLAP", train_chunks_nonoverlap)
chunk_summary("VALIDATION NON-OVERLAP", val_chunks_nonoverlap)


In [ ]:

from collections import defaultdict

def reconstruct_positions(chunks):
    docs = defaultdict(dict)

    for chunk in chunks:
        doc_id = chunk["doc_id"]
        start = chunk["chunk_start"]

        for offset, (token_id, label_id) in enumerate(
            zip(chunk["input_ids"], chunk["labels"])
        ):
            pos = start + offset
            previous = docs[doc_id].get(pos)

            if previous is not None and previous != (token_id, label_id):
                raise ValueError(
                    f"Conflicting copies: doc={doc_id}, pos={pos}, "
                    f"old={previous}, new={(token_id, label_id)}"
                )

            docs[doc_id][pos] = (token_id, label_id)

    return docs


for name, original, rebuilt in [
    ("TRAIN", train_chunks, train_chunks_nonoverlap),
    ("VALIDATION", val_chunks, val_chunks_nonoverlap),
]:
    old_docs = reconstruct_positions(original)
    new_docs = reconstruct_positions(rebuilt)

    print(f"\n=== {name} ===")
    print("Documents preserved:", set(old_docs) == set(new_docs))

    token_mismatches = 0
    label_mismatches = 0
    position_mismatches = 0

    for doc_id in old_docs:
        old_positions = old_docs[doc_id]
        new_positions = new_docs[doc_id]

        if set(old_positions) != set(new_positions):
            position_mismatches += len(
                set(old_positions).symmetric_difference(new_positions)
            )

        for pos in set(old_positions) & set(new_positions):
            old_token, old_label = old_positions[pos]
            new_token, new_label = new_positions[pos]
            token_mismatches += old_token != new_token
            label_mismatches += old_label != new_label

    print("Position mismatches:", position_mismatches)
    print("Token mismatches:", token_mismatches)
    print("Label mismatches:", label_mismatches)

    assert position_mismatches == 0
    assert token_mismatches == 0
    assert label_mismatches == 0

print("\nLabel preservation checks passed.")


In [ ]:

import copy
import torch
import torch.nn as nn
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Start from the clean, verified Phase 1 encoder.
fresh_encoder = copy.deepcopy(clean_encoder)

# Wrap Q/V projections with LORMA adapters.
# The existing A4Attention and A4EncoderLayer classes are retained.
for layer in fresh_encoder.encoder_layers:
    for proj_name in ("q_proj", "v_proj"):
        attention = layer.attn
        base = getattr(attention, proj_name)

        if isinstance(base, LORMALinear):
            continue

        setattr(
            attention,
            proj_name,
            LORMALinear(base, rank=8, alpha=16, dropout=0.05)
        )

# Construct a fresh NER model without modifying model4.
fresh_model = A4NER(encoder=fresh_encoder, num_labels=25, dropout=0.10)
fresh_model = fresh_model.to(device)

# Freeze base encoder parameters; train LORMA adapters and classifier.
for name, param in fresh_model.named_parameters():
    param.requires_grad = (
        "lorma_A" in name
        or "lorma_B" in name
        or name.startswith("classifier.")
    )

# Compute smoothed inverse-frequency weights from the training labels.
counts = Counter(
    label
    for chunk in train_chunks_nonoverlap
    for label in chunk["labels"]
)

frequencies = torch.tensor(
    [counts[i] for i in range(25)],
    dtype=torch.float32
)

# Smooth the inverse frequency to avoid excessively large weights.
weights = (frequencies.sum() / (25 * frequencies.clamp_min(1))).sqrt()
weights = weights / weights.mean()
weights[label2id["O"]] = min(weights[label2id["O"]].item(), 1.0)
weights = weights.to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

trainable = sum(p.numel() for p in fresh_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in fresh_model.parameters())

print("Device:", device)
print("Model output classes:", 25)
print("Trainable parameters:", f"{trainable:,}")
print("Total parameters:", f"{total:,}")
print("Trainable percentage:", round(100 * trainable / total, 2), "%")
print("Weighted loss:", criterion)
print("\nClass weights:")
for label, idx in label2id.items():
    print(f"{label:15s} count={counts[idx]:7d} weight={weights[idx].item():.3f}")

assert fresh_model is not model4
assert trainable > 0
print("\nFresh LORMA model configured successfully.")


In [ ]:

import inspect

print("=== A4NER constructor signature ===")
print(inspect.signature(A4NER.__init__))

print("\n=== Existing model instance attributes ===")
print(vars(model4).keys())

print("\n=== Existing model components ===")
for name, module in model4.named_children():
    print(name, "->", type(module).__name__)

print("\n=== Encoder assignment check ===")
print("Encoder type:", type(clean_encoder).__name__)
print("Encoder layers:", len(clean_encoder.encoder_layers))

print("\n=== LORMA constructor ===")
print(inspect.signature(LORMALinear.__init__))


In [ ]:

import copy
import torch
import torch.nn as nn
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build a fresh model using the constructor supported by your notebook.
fresh_model = A4NER()

# Install the independently verified Phase 1 encoder.
fresh_model.encoder = copy.deepcopy(clean_encoder)

# Wrap Q/V projections with LORMA adapters.
for layer in fresh_model.encoder.encoder_layers:
    for proj_name in ("q_proj", "v_proj"):
        base = getattr(layer.attn, proj_name)
        if not isinstance(base, LORMALinear):
            setattr(
                layer.attn,
                proj_name,
                LORMALinear(
                    base,
                    rank=8,
                    alpha=16,
                    dropout=0.05
                )
            )

fresh_model = fresh_model.to(device)

# Train adapters and the classification head only.
for name, param in fresh_model.named_parameters():
    param.requires_grad = (
        "lorma_A" in name
        or "lorma_B" in name
        or name.startswith("classifier.")
    )

# Calculate smoothed class weights from training labels.
counts = Counter(
    label
    for chunk in train_chunks_nonoverlap
    for label in chunk["labels"]
)

frequencies = torch.tensor(
    [counts[i] for i in range(25)],
    dtype=torch.float32
)
weights = (
    frequencies.sum() /
    (25 * frequencies.clamp_min(1))
).sqrt()
weights = weights / weights.mean()
weights[label2id["O"]] = min(
    weights[label2id["O"]].item(), 1.0
)
weights = weights.to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

trainable = sum(
    p.numel() for p in fresh_model.parameters()
    if p.requires_grad
)
total = sum(p.numel() for p in fresh_model.parameters())

print("Device:", device)
print("Trainable parameters:", f"{trainable:,}")
print("Total parameters:", f"{total:,}")
print("Trainable percentage:", round(100 * trainable / total, 2))
print("Encoder layers:", len(fresh_model.encoder.encoder_layers))
print("Classifier:", fresh_model.classifier)
print("Weighted loss:", criterion)

assert fresh_model is not model4
assert trainable > 0

# Test the output shape.
fresh_model.eval()
sample_ids = torch.tensor(
    train_chunks_nonoverlap[0]["input_ids"][:128],
    dtype=torch.long,
    device=device
).unsqueeze(0)

with torch.no_grad():
    logits = fresh_model(sample_ids)

print("Logits shape:", tuple(logits.shape))
assert logits.shape == (1, sample_ids.shape[1], 25)

print("Fresh LORMA model configured successfully.")


In [ ]:

import os
import copy
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

# Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

BATCH_SIZE = 2
EPOCHS = 3
LEARNING_RATE = 2e-4
SAVE_PATH = "/kaggle/working/a4_lorma_weighted_best.pt"

def collate_chunks(batch):
    input_tensors = [
        torch.tensor(x["input_ids"], dtype=torch.long)
        for x in batch
    ]
    label_tensors = [
        torch.tensor(x["labels"], dtype=torch.long)
        for x in batch
    ]

    input_ids = pad_sequence(
        input_tensors,
        batch_first=True,
        padding_value=0
    )
    labels_batch = pad_sequence(
        label_tensors,
        batch_first=True,
        padding_value=-100
    )

    # Mark padded token positions with the ignore index.
    lengths = torch.tensor([len(x) for x in input_tensors])
    mask = torch.arange(input_ids.size(1))[None, :] < lengths[:, None]
    labels_batch[~mask] = -100

    return input_ids, labels_batch


train_loader = DataLoader(
    train_chunks_nonoverlap,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_chunks
)
val_loader = DataLoader(
    val_chunks_nonoverlap,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_chunks
)

optimizer = torch.optim.AdamW(
    [p for p in fresh_model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=0.01
)

best_val_loss = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    fresh_model.train()
    train_loss_sum = 0.0
    train_tokens = 0

    for input_ids, target_ids in train_loader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = fresh_model(input_ids)

        loss = criterion(
            logits.reshape(-1, 25),
            target_ids.reshape(-1)
        )

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss: {loss.item()}")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in fresh_model.parameters() if p.requires_grad],
            max_norm=1.0
        )
        optimizer.step()

        valid_count = (target_ids != -100).sum().item()
        train_loss_sum += loss.item() * valid_count
        train_tokens += valid_count

    train_loss = train_loss_sum / max(train_tokens, 1)

    fresh_model.eval()
    val_loss_sum = 0.0
    val_tokens = 0
    correct = 0
    predicted_entity_tokens = 0
    gold_entity_tokens = 0

    with torch.no_grad():
        for input_ids, target_ids in val_loader:
            input_ids = input_ids.to(device)
            target_ids = target_ids.to(device)

            logits = fresh_model(input_ids)
            loss = criterion(
                logits.reshape(-1, 25),
                target_ids.reshape(-1)
            )

            valid = target_ids != -100
            predictions = logits.argmax(dim=-1)

            valid_count = valid.sum().item()
            val_loss_sum += loss.item() * valid_count
            val_tokens += valid_count
            correct += (
                (predictions == target_ids) & valid
            ).sum().item()
            predicted_entity_tokens += (
                (predictions != label2id["O"]) & valid
            ).sum().item()
            gold_entity_tokens += (
                (target_ids != label2id["O"]) & valid
            ).sum().item()

    val_loss = val_loss_sum / max(val_tokens, 1)
    val_accuracy = correct / max(val_tokens, 1)
    pred_fraction = predicted_entity_tokens / max(val_tokens, 1)
    gold_fraction = gold_entity_tokens / max(val_tokens, 1)

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "predicted_entity_fraction": pred_fraction,
        "gold_entity_fraction": gold_fraction,
    }
    history.append(record)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_accuracy={val_accuracy:.4f} | "
        f"pred_entity_frac={pred_fraction:.4f} | "
        f"gold_entity_frac={gold_fraction:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "model_state_dict": fresh_model.state_dict(),
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "history": history,
                "label2id": label2id,
            },
            SAVE_PATH
        )
        print("  Saved new best checkpoint:", SAVE_PATH)

print("\nTraining complete.")
print("Best validation loss:", best_val_loss)
print("Checkpoint exists:", os.path.isfile(SAVE_PATH))
print("History:", history)


In [ ]:

import os
import torch

checkpoint_path = "/kaggle/working/a4_lorma_weighted_best.pt"

assert os.path.isfile(checkpoint_path), "Checkpoint file not found."

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

result = fresh_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

fresh_model.eval()

print("Checkpoint epoch:", checkpoint["epoch"])
print("Saved validation loss:", checkpoint["val_loss"])
print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)
print("Model device:", next(fresh_model.parameters()).device)
print("Checkpoint reload successful.")


In [ ]:

from collections import Counter

def decode_bio_spans(pred_ids, offsets, id2label, text):
    """Convert BIO token predictions into character-level entity spans."""
    entities = []
    active_type = None
    active_start = None
    active_end = None

    def close_entity():
        nonlocal active_type, active_start, active_end
        if active_type is not None:
            entities.append(
                (active_start, active_end, active_type)
            )
        active_type = None
        active_start = None
        active_end = None

    for pred_id, (start, end) in zip(pred_ids, offsets):
        if end <= start:
            close_entity()
            continue

        tag = id2label.get(int(pred_id), "O")
        if tag == "O":
            close_entity()
            continue

        prefix, entity_type = tag.split("-", 1)

        if prefix == "B":
            close_entity()
            active_type = entity_type
            active_start = start
            active_end = end
        elif prefix == "I":
            if active_type == entity_type:
                active_end = end
            else:
                close_entity()
                active_type = entity_type
                active_start = start
                active_end = end
        else:
            close_entity()

    close_entity()
    return entities


id2label = {idx: label for label, idx in label2id.items()}

def evaluate_strict(model, texts, golds, tokenizer, label2id, id2label,
                    max_tokens=1024, stride=768):
    model.eval()
    device = next(model.parameters()).device
    per_type = Counter()
    totals = Counter()

    with torch.no_grad():
        for doc_id, text in texts.items():
            encoded = tokenizer.encode(text)
            token_ids = encoded.ids
            offsets = encoded.offsets

            # Predict each token once. Average logits where windows overlap.
            logit_sum = [None] * len(token_ids)
            logit_count = [0] * len(token_ids)

            for chunk_start in range(0, len(token_ids), stride):
                chunk_end = min(chunk_start + max_tokens, len(token_ids))
                ids = torch.tensor(
                    token_ids[chunk_start:chunk_end],
                    dtype=torch.long,
                    device=device
                ).unsqueeze(0)

                logits = model(ids)[0].float().cpu()
                for local_i in range(chunk_end - chunk_start):
                    global_i = chunk_start + local_i
                    if logit_sum[global_i] is None:
                        logit_sum[global_i] = logits[local_i]
                    else:
                        logit_sum[global_i] += logits[local_i]
                    logit_count[global_i] += 1

                if chunk_end == len(token_ids):
                    break

            predicted_ids = [
                int((logit_sum[i] / logit_count[i]).argmax().item())
                for i in range(len(token_ids))
            ]

            predicted = set(
                decode_bio_spans(
                    predicted_ids, offsets, id2label, text
                )
            )
            gold = {
                (start, end, entity_type)
                for start, end, entity_type, _ in golds[doc_id]
            }

            for entity in predicted:
                per_type[(entity[2], "pred")] += 1
            for entity in gold:
                per_type[(entity[2], "gold")] += 1

            for entity in predicted & gold:
                per_type[(entity[2], "tp")] += 1

            totals["pred"] += len(predicted)
            totals["gold"] += len(gold)
            totals["tp"] += len(predicted & gold)

    all_types = list(labels)
    results = {}
    f1_values = []

    for entity_type in all_types:
        tp = per_type[(entity_type, "tp")]
        pred = per_type[(entity_type, "pred")]
        gold = per_type[(entity_type, "gold")]

        precision = tp / pred if pred else 0.0
        recall = tp / gold if gold else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        results[entity_type] = {
            "tp": tp, "pred": pred, "gold": gold,
            "precision": precision, "recall": recall, "f1": f1
        }
        f1_values.append(f1)

    tp, pred, gold = totals["tp"], totals["pred"], totals["gold"]
    micro_p = tp / pred if pred else 0.0
    micro_r = tp / gold if gold else 0.0
    micro_f1 = (
        2 * micro_p * micro_r / (micro_p + micro_r)
        if micro_p + micro_r else 0.0
    )

    print("=== STRICT ENTITY-LEVEL RESULTS ===")
    print("Micro precision:", round(micro_p, 4))
    print("Micro recall:", round(micro_r, 4))
    print("Micro F1:", round(micro_f1, 4))
    print("Macro F1 (12 classes):", round(sum(f1_values) / len(f1_values), 4))
    print("Predicted entities:", pred)
    print("Gold entities:", gold)
    print("\nPer-class metrics:")
    for entity_type, values in results.items():
        print(
            f"{entity_type:12s} "
            f"P={values['precision']:.3f} "
            f"R={values['recall']:.3f} "
            f"F1={values['f1']:.3f} "
            f"TP={values['tp']} "
            f"Pred={values['pred']} "
            f"Gold={values['gold']}"
        )

    return results, {
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "micro_f1": micro_f1,
        "macro_f1": sum(f1_values) / len(f1_values),
    }


strict_results, strict_summary = evaluate_strict(
    fresh_model,
    fold1_text,
    fold1_gold,
    tokenizer,
    label2id,
    id2label,
    max_tokens=MAX_TOKENS,
    stride=STRIDE
)


In [ ]:

def inspect_span_alignment(
    model, doc_id, text, gold_annotations,
    tokenizer, label2id, id2label,
    max_tokens=1024, stride=768, max_examples=15
):
    device = next(model.parameters()).device
    encoded = tokenizer.encode(text)
    ids, offsets = encoded.ids, encoded.offsets

    logit_sum = [None] * len(ids)
    logit_count = [0] * len(ids)

    model.eval()
    with torch.no_grad():
        for chunk_start in range(0, len(ids), stride):
            chunk_end = min(chunk_start + max_tokens, len(ids))
            input_ids = torch.tensor(
                ids[chunk_start:chunk_end],
                dtype=torch.long,
                device=device
            ).unsqueeze(0)

            logits = model(input_ids)[0].float().cpu()

            for local_i in range(chunk_end - chunk_start):
                i = chunk_start + local_i
                if logit_sum[i] is None:
                    logit_sum[i] = logits[local_i].clone()
                else:
                    logit_sum[i] += logits[local_i]
                logit_count[i] += 1

            if chunk_end == len(ids):
                break

    pred_ids = [
        int((logit_sum[i] / logit_count[i]).argmax())
        for i in range(len(ids))
    ]

    predicted = decode_bio_spans(
        pred_ids, offsets, id2label, text
    )

    gold = [
        (start, end, entity_type, entity_text)
        for start, end, entity_type, entity_text
        in gold_annotations
    ]

    print("Document:", doc_id)
    print("\nFIRST PREDICTED ENTITIES:")
    for start, end, label in predicted[:max_examples]:
        print(
            (start, end, label),
            repr(text[start:end])
        )

    print("\nFIRST GOLD ENTITIES:")
    for start, end, label, entity_text in gold[:max_examples]:
        print(
            (start, end, label),
            repr(entity_text),
            "| text slice:", repr(text[start:end])
        )

    print("\nNEAREST GOLD SPAN FOR EACH FIRST PREDICTION:")
    for ps, pe, pl in predicted[:max_examples]:
        nearest = min(
            gold,
            key=lambda g: abs(g[0] - ps) + abs(g[1] - pe)
        )
        gs, ge, gl, gt = nearest
        print(
            "PRED:", (ps, pe, pl), repr(text[ps:pe]),
            "| NEAREST GOLD:", (gs, ge, gl), repr(gt)
        )


example_id = next(iter(fold1_text))
inspect_span_alignment(
    fresh_model,
    example_id,
    fold1_text[example_id],
    fold1_gold[example_id],
    tokenizer,
    label2id,
    id2label
)


In [ ]:

import os

source_path = (
    "/kaggle/working/.virtual_documents/"
    "__notebook_source__.ipynb"
)

print("Source file exists:", os.path.isfile(source_path))

if os.path.isfile(source_path):
    with open(source_path, "r", encoding="utf-8") as f:
        source = f.read()

    print("Source characters:", len(source))

    terms = [
        "def build_ner_chunks",
        "def create_ner",
        "A4NER(",
        "lorma_encoder",
        "tokenizer.encode",
        "offsets",
        "token_labels",
        "word_ids",
    ]

    for term in terms:
        positions = []
        start = 0
        while True:
            pos = source.find(term, start)
            if pos == -1:
                break
            positions.append(pos)
            start = pos + len(term)

        print(f"\n{term}: {len(positions)} occurrences")
        for pos in positions[-3:]:
            print(source[max(0, pos-250):pos+700])
            print("-" * 50)
else:
    print("Please locate the notebook source file in /kaggle/working.")


In [ ]:

from pathlib import Path

path = Path("/kaggle/working/.virtual_documents/__notebook_source__.ipynb")
lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

terms = [
    "def build_a4_chunks",
    "def build_a4_validation_chunks",
    "def make_chunks_fixed",
    "class A4NER",
    "class A4EncoderForNER",
    "EXPERIMENT 4",
    "lorma_encoder = copy.deepcopy(encoder)",
]

for term in terms:
    hits = [i for i, line in enumerate(lines) if term in line]
    print(f"\n{'='*70}\n{term}: {len(hits)} occurrence(s)")

    # Print the earliest relevant implementation, not later debug snippets.
    for i in hits[:2]:
        start = max(0, i - 8)
        end = min(len(lines), i + 65)
        print(f"\n--- Source lines {start+1}-{end} ---")
        print("\n".join(
            f"{j+1}: {lines[j]}"
            for j in range(start, end)
        ))


In [ ]:

from pathlib import Path

lines = Path(
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb"
).read_text(encoding="utf-8", errors="replace").splitlines()

# Original A4 chunk builder, including its complete label-assignment logic.
for start, end, title in [
    (9175, 9275, "ORIGINAL A4 CHUNK BUILDER"),
    (9120, 9168, "TOKENIZER AND GOLD HELPERS"),
    (9275, 9345, "ORIGINAL CHUNK CREATION AND COUNTS"),
    (14880, 15030, "ORIGINAL LORMA TRAINING SETUP"),
    (15158, 15310, "ORIGINAL LORMA MODEL AND TRAINING"),
]:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    print("\n".join(
        f"{i+1}: {lines[i]}"
        for i in range(start - 1, min(end, len(lines)))
    ))


In [ ]:

import torch
from collections import Counter

print("=" * 65)
print("CHECKPOINT AND LABEL-MAPPING AUDIT")
print("=" * 65)

# 1. Verify the exact label mapping used during training.
print("\nBIO label mapping:")
for label, idx in sorted(label2id.items(), key=lambda x: x[1]):
    print(f"{idx:2d}: {label}")

# 2. Check the checkpoint metadata.
ckpt_path = "/kaggle/working/a4_lorma_weighted_best.pt"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

print("\nCheckpoint keys:", list(ckpt.keys()))
print("Saved epoch:", ckpt.get("epoch"))
print("Saved validation loss:", ckpt.get("val_loss"))

saved_mapping = ckpt.get("label2id")
if saved_mapping is not None:
    print("Saved mapping matches current mapping:",
          saved_mapping == label2id)

# 3. Check the model's output class dimension.
fresh_model.eval()
device = next(fresh_model.parameters()).device
sample_ids = torch.tensor(
    tokenizer.encode(fold1_text[next(iter(fold1_text))]).ids[:128],
    dtype=torch.long,
    device=device
).unsqueeze(0)

with torch.no_grad():
    sample_logits = fresh_model(sample_ids)

print("\nLogit shape:", tuple(sample_logits.shape))
print("All logits finite:", bool(torch.isfinite(sample_logits).all()))

# 4. Check predictions on the first 128 tokens.
pred_ids = sample_logits[0].argmax(-1).cpu().tolist()
print("Predicted label distribution:",
      Counter(id2label[i] for i in pred_ids))

# 5. Check that the loaded weights match the saved checkpoint.
state = ckpt.get("model_state_dict", ckpt.get("state_dict"))
if state is not None:
    current_state = fresh_model.state_dict()
    matching = [
        k for k in state
        if k in current_state
        and state[k].shape == current_state[k].shape
        and torch.equal(state[k], current_state[k].detach().cpu())
    ]
    print("Exactly matching tensors:", len(matching), "/", len(state))
else:
    print("Could not identify the checkpoint state-dict key.")


In [ ]:

import torch

phase1_path = (
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)

phase1 = torch.load(
    phase1_path, map_location="cpu", weights_only=False
)

print("Phase 1 checkpoint type:", type(phase1).__name__)
if isinstance(phase1, dict):
    print("Phase 1 checkpoint keys:", list(phase1.keys())[:25])

def extract_state(obj):
    if not isinstance(obj, dict):
        return obj.state_dict()
    for key in (
        "model_state_dict", "state_dict", "model", "encoder_state_dict"
    ):
        if key in obj and isinstance(obj[key], dict):
            return obj[key]
    return obj

phase1_state = extract_state(phase1)
clean_state = clean_encoder.state_dict()

print("\nPhase 1 tensors:", len(phase1_state))
print("clean_encoder tensors:", len(clean_state))

# Compare using exact names first, then report likely matching names.
common = set(phase1_state) & set(clean_state)
print("Common parameter names:", len(common))

shape_matches = [
    k for k in common
    if hasattr(phase1_state[k], "shape")
    and phase1_state[k].shape == clean_state[k].shape
]
print("Common names with matching shapes:", len(shape_matches))

exact = [
    k for k in shape_matches
    if torch.equal(
        phase1_state[k].detach().cpu(),
        clean_state[k].detach().cpu()
    )
]
print("Exactly equal tensors:", len(exact))

print("\nSample Phase 1 names:")
for k in list(phase1_state)[:15]:
    print(" ", k, tuple(phase1_state[k].shape)
          if hasattr(phase1_state[k], "shape") else type(phase1_state[k]))

print("\nSample clean encoder names:")
for k in list(clean_state)[:15]:
    print(" ", k, tuple(clean_state[k].shape))

print("\nEncoder embedding rows:",
      clean_encoder.token_embedding.weight.shape[0])
print("Tokenizer vocabulary size:", tokenizer.get_vocab_size())


In [ ]:

import torch

p1 = phase1["model_state_dict"]
cs = clean_encoder.state_dict()

base_checks = []
missing = []

for key, value in cs.items():
    if ".base." in key:
        original_key = key.replace(".base.", ".")
        if original_key not in p1:
            missing.append((key, original_key))
            continue

        original = p1[original_key].detach().cpu()
        current = value.detach().cpu()

        if original.shape == current.shape:
            diff = (original.float() - current.float()).abs()
            base_checks.append({
                "wrapped_key": key,
                "original_key": original_key,
                "exact": torch.equal(original, current),
                "max_diff": diff.max().item(),
                "mean_diff": diff.mean().item()
            })
        else:
            missing.append((key, original_key, "shape mismatch"))

print("Wrapped base tensors compared:", len(base_checks))
print("Missing or incompatible mappings:", len(missing))
print("Exact matches:", sum(x["exact"] for x in base_checks))
print("Non-exact matches:", sum(not x["exact"] for x in base_checks))

for item in base_checks:
    if not item["exact"]:
        print(
            "\nMISMATCH:", item["wrapped_key"],
            "\n  Original:", item["original_key"],
            "\n  Max absolute difference:", item["max_diff"],
            "\n  Mean absolute difference:", item["mean_diff"]
        )

if missing:
    print("\nMissing examples:", missing[:10])

# Verify that LORMA B is not still all zero.
for key, value in cs.items():
    if "lorma_B" in key:
        print(
            key,
            "| abs mean:", value.float().abs().mean().item(),
            "| abs max:", value.float().abs().max().item()
        )


In [ ]:

import torch
from collections import Counter

doc_id = next(iter(fold1_text))
text = fold1_text[doc_id]
gold = fold1_gold[doc_id]

enc = tokenizer.encode(text)
ids = enc.ids
offsets = enc.offsets

# Run inference in chunks using the same maximum length and stride.
fresh_model.eval()
device = next(fresh_model.parameters()).device
logit_sum = torch.zeros(len(ids), 25)
logit_count = torch.zeros(len(ids), 1)

with torch.no_grad():
    for start in range(0, len(ids), 768):
        end = min(start + 1024, len(ids))
        x = torch.tensor(ids[start:end], dtype=torch.long,
                         device=device).unsqueeze(0)
        logits = fresh_model(x)[0].detach().cpu()
        logit_sum[start:end] += logits
        logit_count[start:end] += 1
        if end == len(ids):
            break

avg_logits = logit_sum / logit_count.clamp_min(1)
pred_ids = avg_logits.argmax(-1).tolist()

# Recreate gold BIO labels using the same overlap rule as the source builder.
gold_ids = [label2id["O"]] * len(ids)
for item in gold:
    s, e, label = item[:3]
    covered = [
        i for i, (ts, te) in enumerate(offsets)
        if te > ts and ts < e and te > s
    ]
    if covered:
        gold_ids[covered[0]] = label2id["B-" + label]
        for i in covered[1:]:
            gold_ids[i] = label2id["I-" + label]

print("Document:", doc_id, "| characters:", len(text), "| tokens:", len(ids))
print("Gold spans:", len(gold))
print("Gold BIO distribution:",
      Counter(id2label[i] for i in gold_ids))
print("Predicted BIO distribution:",
      Counter(id2label[i] for i in pred_ids))

print("\nFirst 100 tokens: token | gold | predicted")
for i in range(min(100, len(ids))):
    s, e = offsets[i]
    token_text = text[s:e].replace("\n", "\\n")
    print(f"{i:3d} | {token_text!r:28.28s} | "
          f"{id2label[gold_ids[i]]:12s} | {id2label[pred_ids[i]]}")


In [ ]:

# Inspect token offsets around gold entities in the current document.
examples = 0
left_whitespace_cases = 0
right_boundary_cases = 0

for item in gold:
    s, e, label = item[:3]
    covered = [
        i for i, (ts, te) in enumerate(offsets)
        if te > ts and ts < e and te > s
    ]
    if not covered:
        continue

    first = covered[0]
    last = covered[-1]
    ts, te = offsets[first]
    ls, le = offsets[last]

    if ts < s and text[ts:s].isspace():
        left_whitespace_cases += 1
    if le > e and text[e:le].isspace():
        right_boundary_cases += 1

    if examples < 12:
        print(f"\nGold: {label} [{s}, {e}] {text[s:e]!r}")
        print(f" First token: [{ts}, {te}] {text[ts:te]!r}")
        print(f" Last token:  [{ls}, {le}] {text[ls:le]!r}")
        examples += 1

print("\nGold entities inspected:", len(gold))
print("Leading-whitespace offset cases:", left_whitespace_cases)
print("Trailing-whitespace offset cases:", right_boundary_cases)


In [ ]:

# Compare the existing decoder with a whitespace-trimmed version.
# Uses the current document's pred_ids, offsets, text, and gold.

raw_spans = decode_bio_spans(pred_ids, offsets, id2label, text)

def span_key(span):
    # Support decoder output containing (start, end, label)
    # or (start, end, label, text).
    return (int(span[0]), int(span[1]), str(span[2]))

raw_keys = {span_key(s) for s in raw_spans}

trimmed_keys = set()
for span in raw_spans:
    s, e, label = span_key(span)
    while s < e and text[s].isspace():
        s += 1
    while e > s and text[e - 1].isspace():
        e -= 1
    if e > s:
        trimmed_keys.add((s, e, label))

gold_keys = {
    (int(item[0]), int(item[1]), str(item[2]))
    for item in gold
}

def report(name, predicted, expected):
    tp = len(predicted & expected)
    fp = len(predicted - expected)
    fn = len(expected - predicted)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    print(
        f"{name}: predicted={len(predicted)}, gold={len(expected)}, "
        f"TP={tp}, FP={fp}, FN={fn}, "
        f"P={precision:.4f}, R={recall:.4f}, F1={f1:.4f}"
    )

report("Original spans", raw_keys, gold_keys)
report("Trimmed spans", trimmed_keys, gold_keys)

print("\nFirst few corrected spans:")
for s, e, label in list(trimmed_keys)[:15]:
    print(label, (s, e), repr(text[s:e]))


In [ ]:

import inspect
from collections import Counter

print("=" * 65)
print("1. CURRENT DECODER SOURCE")
print("=" * 65)

try:
    print(inspect.getsource(decode_bio_spans))
except Exception as exc:
    print("Could not retrieve source:", repr(exc))

print("\n" + "=" * 65)
print("2. BIO TRANSITION AUDIT")
print("=" * 65)

def transition_audit(ids):
    invalid_i = 0
    entity_tokens = 0
    previous_type = None

    for idx in ids:
        label = id2label[int(idx)]
        if label == "O":
            previous_type = None
            continue

        prefix, entity_type = label.split("-", 1)
        entity_tokens += 1

        if prefix == "B":
            previous_type = entity_type
        elif prefix == "I":
            if previous_type != entity_type:
                invalid_i += 1
            previous_type = entity_type

    return invalid_i, entity_tokens

invalid_i, entity_tokens = transition_audit(pred_ids)
print("Predicted entity tokens:", entity_tokens)
print("Invalid/orphan I transitions:", invalid_i)

print("\n" + "=" * 65)
print("3. PREDICTED SPAN START TYPES")
print("=" * 65)

start_types = Counter()
for s, e, label in trimmed_keys:
    start_types[label] += 1

for label, count in start_types.most_common():
    print(f"{label:12s} {count:4d}")

print("\n" + "=" * 65)
print("4. GOLD VS PREDICTED TOKEN LABELS")
print("=" * 65)

print("Gold non-O tokens:", sum(i != label2id["O"] for i in gold_ids))
print("Predicted non-O tokens:", sum(i != label2id["O"] for i in pred_ids))
print("Gold B labels:", sum(id2label[i].startswith("B-") for i in gold_ids))
print("Predicted B labels:", sum(id2label[i].startswith("B-") for i in pred_ids))
print("Gold I labels:", sum(id2label[i].startswith("I-") for i in gold_ids))
print("Predicted I labels:", sum(id2label[i].startswith("I-") for i in pred_ids))


In [ ]:

def decode_spans_variant(pred_ids, offsets, id2label, text, orphan_i="start"):
    entities = []
    active_type = None
    active_start = None
    active_end = None

    def close_entity():
        nonlocal active_type, active_start, active_end
        if active_type is not None:
            entities.append((active_start, active_end, active_type))
        active_type = active_start = active_end = None

    for pred_id, (start, end) in zip(pred_ids, offsets):
        if end <= start:
            close_entity()
            continue

        tag = id2label.get(int(pred_id), "O")
        if tag == "O":
            close_entity()
            continue

        prefix, entity_type = tag.split("-", 1)

        if prefix == "B":
            close_entity()
            active_type, active_start, active_end = entity_type, start, end

        elif prefix == "I":
            if active_type == entity_type:
                active_end = end
            else:
                close_entity()
                if orphan_i == "start":
                    active_type, active_start, active_end = entity_type, start, end
                # orphan_i == "ignore": skip this token

    close_entity()

    # Trim whitespace from character boundaries.
    result = set()
    for s, e, label in entities:
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        if e > s:
            result.add((s, e, label))
    return result

gold_keys = {
    (int(item[0]), int(item[1]), str(item[2]))
    for item in gold
}

def score_spans(name, predicted, expected):
    tp = len(predicted & expected)
    fp = len(predicted - expected)
    fn = len(expected - predicted)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    print(f"{name:24s} spans={len(predicted):4d} "
          f"TP={tp:3d} FP={fp:3d} FN={fn:3d} "
          f"P={p:.4f} R={r:.4f} F1={f1:.4f}")

for strategy in ("start", "ignore"):
    pred_spans = decode_spans_variant(
        pred_ids, offsets, id2label, text, orphan_i=strategy
    )
    score_spans(f"Orphan I: {strategy}", pred_spans, gold_keys)


In [ ]:

import torch
from collections import Counter

def decode_eval(pred_ids, offsets, text, ignore_orphan_i=False):
    spans = []
    active_type = active_start = active_end = None

    def close():
        nonlocal active_type, active_start, active_end
        if active_type is not None:
            spans.append((active_start, active_end, active_type))
        active_type = active_start = active_end = None

    for pid, (s, e) in zip(pred_ids, offsets):
        if e <= s:
            close()
            continue

        tag = id2label[int(pid)]
        if tag == "O":
            close()
            continue

        prefix, typ = tag.split("-", 1)
        if prefix == "B":
            close()
            active_type, active_start, active_end = typ, s, e
        elif prefix == "I":
            if active_type == typ:
                active_end = e
            else:
                close()
                if not ignore_orphan_i:
                    active_type, active_start, active_end = typ, s, e

    close()

    # Trim whitespace from predicted character boundaries.
    result = set()
    for s, e, typ in spans:
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        if e > s:
            result.add((s, e, typ))
    return result


def predict_doc(text):
    ids = tokenizer.encode(text).ids
    offsets = tokenizer.encode(text).offsets
    logit_sum = torch.zeros(len(ids), 25)
    counts = torch.zeros(len(ids), 1)

    fresh_model.eval()
    device = next(fresh_model.parameters()).device

    with torch.no_grad():
        for start in range(0, len(ids), 768):
            end = min(start + 1024, len(ids))
            x = torch.tensor(
                ids[start:end], dtype=torch.long, device=device
            ).unsqueeze(0)
            logits = fresh_model(x)[0].detach().cpu()
            logit_sum[start:end] += logits
            counts[start:end] += 1
            if end == len(ids):
                break

    predictions = (logit_sum / counts.clamp_min(1)).argmax(-1).tolist()
    return predictions, offsets


def entity_set(gold_items):
    return {
        (int(item[0]), int(item[1]), str(item[2]))
        for item in gold_items
    }


def calc_metrics(predicted, expected):
    tp = len(predicted & expected)
    fp = len(predicted - expected)
    fn = len(expected - predicted)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return tp, fp, fn, p, r, f1


# Fold 1 is the held-out validation fold in the current setup.
totals = {
    "start": [0, 0, 0],
    "ignore": [0, 0, 0]
}
doc_f1 = {"start": [], "ignore": []}
pred_counts = Counter()

for doc_id, text in fold1_text.items():
    gold_set = entity_set(fold1_gold[doc_id])
    pred_ids, offsets = predict_doc(text)

    for mode, ignore_i in (("start", False), ("ignore", True)):
        pred_set = decode_eval(
            pred_ids, offsets, text, ignore_orphan_i=ignore_i
        )
        tp, fp, fn, p, r, f1 = calc_metrics(pred_set, gold_set)
        totals[mode][0] += tp
        totals[mode][1] += fp
        totals[mode][2] += fn
        doc_f1[mode].append(f1)

        if mode == "ignore":
            pred_counts["predicted_spans"] += len(pred_set)
            pred_counts["gold_spans"] += len(gold_set)

for mode in ("start", "ignore"):
    tp, fp, fn = totals[mode]
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0

    print(f"\nDecoder mode: {mode}")
    print(f"Micro TP={tp}, FP={fp}, FN={fn}")
    print(f"Micro precision={p:.4f}, recall={r:.4f}, F1={f1:.4f}")
    print(f"Mean per-document F1={sum(doc_f1[mode])/len(doc_f1[mode]):.4f}")

print("\nDocuments evaluated:", len(fold1_text))
print("Ignore-mode span totals:", dict(pred_counts))


In [ ]:

import torch

# Cache validation logits once so threshold comparisons are cheap.
validation_cache = []

fresh_model.eval()
device = next(fresh_model.parameters()).device

for doc_id, text in fold1_text.items():
    encoded = tokenizer.encode(text)
    ids, offsets = encoded.ids, encoded.offsets

    logit_sum = torch.zeros(len(ids), 25)
    counts = torch.zeros(len(ids), 1)

    with torch.no_grad():
        for start in range(0, len(ids), 768):
            end = min(start + 1024, len(ids))
            x = torch.tensor(
                ids[start:end], dtype=torch.long, device=device
            ).unsqueeze(0)
            logits = fresh_model(x)[0].detach().cpu()
            logit_sum[start:end] += logits
            counts[start:end] += 1
            if end == len(ids):
                break

    avg_logits = logit_sum / counts.clamp_min(1)
    probs = avg_logits.softmax(-1)

    validation_cache.append({
        "doc_id": doc_id,
        "text": text,
        "offsets": offsets,
        "probs": probs,
        "gold": entity_set(fold1_gold[doc_id])
    })

print("Cached documents:", len(validation_cache))
print("Now sweeping confidence thresholds...")

# A token is retained as an entity only when:
# 1) its best non-O class probability reaches threshold, and
# 2) that class beats O by at least the same margin.
# This is an inference heuristic, not a retraining method.

thresholds = [0.00, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]

results = []

for threshold in thresholds:
    total_tp = total_fp = total_fn = 0

    for item in validation_cache:
        probs = item["probs"]
        pred_ids = probs.argmax(-1).tolist()
        adjusted = []

        for i, pred in enumerate(pred_ids):
            if pred == label2id["O"]:
                adjusted.append(pred)
                continue

            entity_prob = probs[i, pred].item()
            o_prob = probs[i, label2id["O"]].item()

            if entity_prob < threshold or entity_prob - o_prob < threshold:
                adjusted.append(label2id["O"])
            else:
                adjusted.append(pred)

        spans = decode_eval(
            adjusted, item["offsets"], item["text"],
            ignore_orphan_i=True
        )
        tp, fp, fn, _, _, _ = calc_metrics(spans, item["gold"])
        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0
    recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

    results.append((threshold, total_tp, total_fp, total_fn, precision, recall, f1))

print("\n threshold | TP   FP    FN    precision recall   micro-F1")
for t, tp, fp, fn, p, r, f1 in results:
    print(f"   {t:5.2f}   | {tp:4d} {fp:5d} {fn:5d}    {p:.4f}  {r:.4f}   {f1:.4f}")

best = max(results, key=lambda x: x[-1])
print("\nBest threshold on this validation fold:", best[0])
print("Best validation micro-F1:", round(best[-1], 4))


In [ ]:

# Independent threshold sweep:
# p_min = minimum probability of the predicted entity class
# margin_min = minimum (P(entity class) - P(O))

p_thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
margin_thresholds = [0.00, 0.03, 0.05, 0.10, 0.15, 0.20]

results_2d = []

for p_min in p_thresholds:
    for margin_min in margin_thresholds:
        total_tp = total_fp = total_fn = 0

        for item in validation_cache:
            probs = item["probs"]
            pred_ids = probs.argmax(-1).tolist()
            adjusted = []

            for i, pred in enumerate(pred_ids):
                if pred == label2id["O"]:
                    adjusted.append(pred)
                    continue

                entity_prob = probs[i, pred].item()
                o_prob = probs[i, label2id["O"]].item()
                margin = entity_prob - o_prob

                if entity_prob < p_min or margin < margin_min:
                    adjusted.append(label2id["O"])
                else:
                    adjusted.append(pred)

            spans = decode_eval(
                adjusted, item["offsets"], item["text"],
                ignore_orphan_i=True
            )
            tp, fp, fn, _, _, _ = calc_metrics(spans, item["gold"])
            total_tp += tp
            total_fp += fp
            total_fn += fn

        precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0
        recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

        results_2d.append({
            "p_min": p_min, "margin_min": margin_min,
            "tp": total_tp, "fp": total_fp, "fn": total_fn,
            "precision": precision, "recall": recall, "f1": f1
        })

results_2d.sort(key=lambda r: r["f1"], reverse=True)

print("Top 12 threshold combinations by micro-F1:")
print("p_min margin | TP   FP    FN    Precision Recall  F1")
for r in results_2d[:12]:
    print(
        f"{r['p_min']:5.2f} {r['margin_min']:6.2f} | "
        f"{r['tp']:3d} {r['fp']:5d} {r['fn']:5d} "
        f"{r['precision']:.4f}   {r['recall']:.4f}  {r['f1']:.4f}"
    )

print("\nBest combination:", results_2d[0])


In [ ]:

from collections import defaultdict
import numpy as np

P_MIN = 0.40
MARGIN_MIN = 0.03

class_counts = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

for item in validation_cache:
    text = item["text"]
    offsets = item["offsets"]
    probs = np.asarray(item["probs"])
    gold = set(tuple(x) for x in item["gold"])

    pred_ids = probs.argmax(axis=1)
    o_prob = probs[:, label2id["O"]]

    # Suppress low-confidence entity tokens.
    for i, pred_id in enumerate(pred_ids):
        if pred_id == label2id["O"]:
            continue
        entity_prob = probs[i, pred_id]
        margin = entity_prob - o_prob[i]
        if entity_prob < P_MIN or margin < MARGIN_MIN:
            pred_ids[i] = label2id["O"]

    predicted = set(
        tuple(x) for x in
        decode_eval(pred_ids, offsets, text, ignore_orphan_i=True)
    )

    for span in gold | predicted:
        entity_type = span[2]
        if span in gold and span in predicted:
            class_counts[entity_type]["tp"] += 1
        elif span in predicted:
            class_counts[entity_type]["fp"] += 1
        else:
            class_counts[entity_type]["fn"] += 1

print(f"{'Entity':<12} {'TP':>5} {'FP':>6} {'FN':>6} "
      f"{'Precision':>10} {'Recall':>9} {'F1':>9}")

for entity_type, c in sorted(class_counts.items()):
    tp, fp, fn = c["tp"], c["fp"], c["fn"]
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall else 0.0)
    print(f"{entity_type:<12} {tp:>5} {fp:>6} {fn:>6} "
          f"{precision:>10.4f} {recall:>9.4f} {f1:>9.4f}")


In [ ]:

import torch

ckpt_path = "/kaggle/working/a4_lorma_weighted_best.pt"
checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)

saved_label2id = checkpoint.get("label2id")
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("\nSaved checkpoint label2id:")
print(saved_label2id)

print("\nCurrent runtime label2id:")
print(label2id)

if saved_label2id is None:
    print("\nWARNING: Checkpoint has no saved label mapping.")
elif saved_label2id == label2id:
    print("\nPASS: Label mappings match exactly.")
else:
    print("\nWARNING: Label mappings differ.")
    print("Saved labels by ID:")
    print(sorted(saved_label2id.items(), key=lambda x: x[1]))
    print("Runtime labels by ID:")
    print(sorted(label2id.items(), key=lambda x: x[1]))

print("\nClassifier output size:",
      fresh_model.classifier.out_features)
print("Number of runtime labels:", len(label2id))


In [ ]:

from collections import Counter, defaultdict
import numpy as np

# Compare raw argmax predictions with BIO labels reconstructed
# from the validation gold spans.
gold_counts = Counter()
pred_counts = Counter()
token_correct = Counter()
token_total = Counter()
entity_type_stats = defaultdict(Counter)

for item in validation_cache:
    offsets = item["offsets"]
    probs = np.asarray(item["probs"])
    gold_spans = item["gold"]

    gold_ids = np.zeros(len(offsets), dtype=np.int64)
    occupied = np.zeros(len(offsets), dtype=bool)

    # Assign B-/I- tags using token/entity character overlap.
    for start, end, entity_type in sorted(gold_spans):
        matching = []
        for i, (ts, te) in enumerate(offsets):
            if te > ts and ts < end and te > start:
                # Avoid tokens that extend beyond the entity's right edge.
                if te <= end:
                    matching.append(i)

        if not matching:
            continue

        b_id = label2id.get(f"B-{entity_type}")
        i_id = label2id.get(f"I-{entity_type}")
        if b_id is None or i_id is None:
            continue

        # Avoid overwriting an already assigned token.
        matching = [i for i in matching if not occupied[i]]
        if not matching:
            continue

        for j, idx in enumerate(matching):
            gold_ids[idx] = b_id if j == 0 else i_id
            occupied[idx] = True

    pred_ids = probs.argmax(axis=1)

    for g, p in zip(gold_ids, pred_ids):
        gold_tag = id2label[int(g)]
        pred_tag = id2label[int(p)]
        gold_counts[gold_tag] += 1
        pred_counts[pred_tag] += 1
        token_total[gold_tag] += 1
        token_correct[gold_tag] += int(g == p)

        if gold_tag != "O":
            entity_type = gold_tag.split("-", 1)[1]
            entity_type_stats[entity_type]["gold_tokens"] += 1
            entity_type_stats[entity_type]["correct_tokens"] += int(g == p)

print("RAW PREDICTION COUNTS (before confidence filtering)")
print(f"{'BIO label':<15} {'Gold':>9} {'Predicted':>11} {'Accuracy':>10}")

for tag in sorted(set(gold_counts) | set(pred_counts)):
    acc = token_correct[tag] / token_total[tag] if token_total[tag] else 0
    print(f"{tag:<15} {gold_counts[tag]:>9} {pred_counts[tag]:>11} {acc:>10.4f}")

print("\nENTITY-TYPE TOKEN ACCURACY")
print(f"{'Entity':<12} {'Gold tokens':>12} {'Correct':>9} {'Accuracy':>10}")

for entity_type, stats in sorted(entity_type_stats.items()):
    n = stats["gold_tokens"]
    c = stats["correct_tokens"]
    print(f"{entity_type:<12} {n:>12} {c:>9} {c/n if n else 0:>10.4f}")


In [ ]:

import torch
import numpy as np
from collections import Counter

fresh_model.eval()

def get_chunk_value(chunk, key):
    if key in chunk:
        return chunk[key]
    raise KeyError(
        f"Missing {key!r}. Available keys: {list(chunk.keys())}"
    )

# First inspect the actual training-chunk structure.
print("Training chunk keys:", list(train_chunks_nonoverlap[0].keys()))

# Training token-level accuracy using stored gold labels.
train_gold_counts = Counter()
train_pred_counts = Counter()
train_correct = Counter()

with torch.no_grad():
    for chunk in train_chunks_nonoverlap:
        input_ids = torch.as_tensor(
            get_chunk_value(chunk, "input_ids"),
            dtype=torch.long, device=device
        ).unsqueeze(0)

        labels = torch.as_tensor(
            get_chunk_value(chunk, "labels"),
            dtype=torch.long
        ).reshape(-1).cpu().numpy()

        logits = fresh_model(input_ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        preds = logits.argmax(dim=-1).squeeze(0).cpu().numpy()

        n = min(len(labels), len(preds))
        for g, p in zip(labels[:n], preds[:n]):
            if g < 0:
                continue
            gt = id2label[int(g)]
            pt = id2label[int(p)]
            train_gold_counts[gt] += 1
            train_pred_counts[pt] += 1
            train_correct[gt] += int(g == p)

print("\nTRAINING TOKEN ACCURACY")
print(f"{'Label':<15} {'Gold':>9} {'Pred':>9} {'Accuracy':>10}")
for tag in sorted(train_gold_counts):
    n = train_gold_counts[tag]
    print(f"{tag:<15} {n:>9} {train_pred_counts[tag]:>9} "
          f"{train_correct[tag]/n:>10.4f}")

print("\nTRAINING ENTITY-TYPE ACCURACY")
for entity_type in sorted({
    tag.split("-", 1)[1]
    for tag in train_gold_counts if tag != "O"
}):
    tags = [f"B-{entity_type}", f"I-{entity_type}"]
    n = sum(train_gold_counts[t] for t in tags)
    correct = sum(train_correct[t] for t in tags)
    print(f"{entity_type:<12} tokens={n:>6} "
          f"accuracy={correct/n if n else 0:.4f}")


In [ ]:

# Inspect token IDs, decoded token strings, and their training labels.
chunk = next(
    c for c in train_chunks_nonoverlap
    if any(int(x) > 0 for x in c["labels"])
)

ids = [int(x) for x in chunk["input_ids"]]
labels = [int(x) for x in chunk["labels"]]

print("Document:", chunk["doc_id"])
print("Chunk start:", chunk["chunk_start"])
print("Chunk length:", len(ids), len(labels))

def token_text(token_id):
    # Support common tokenizer APIs.
    if hasattr(tokenizer, "id_to_token"):
        result = tokenizer.id_to_token(int(token_id))
        if result is not None:
            return result
    if hasattr(tokenizer, "decode"):
        return tokenizer.decode([int(token_id)])
    return f"<id:{token_id}>"

print(f"\n{'Pos':>4} {'Token ID':>8} {'Token':<24} {'Label':<15}")
print("-" * 58)

for i, (token_id, label_id) in enumerate(zip(ids, labels)):
    if label_id > 0:
        lo = max(0, i - 3)
        hi = min(len(ids), i + 4)
        print(f"\n--- Entity context at token {i} ---")
        for j in range(lo, hi):
            token = token_text(ids[j]).replace("\n", "\\n")
            tag = id2label.get(int(labels[j]), str(labels[j]))
            marker = "  <--" if j == i else ""
            print(f"{j:>4} {ids[j]:>8} {token[:24]:<24} {tag:<15}{marker}")


In [ ]:

# Diagnostic only — does not train or modify the checkpoint.
import torch

embedding = fresh_model.encoder.token_embedding
embedding_size = embedding.num_embeddings

# Tokenizer vocabulary size across common tokenizer APIs.
if hasattr(tokenizer, "get_vocab"):
    vocab_size = len(tokenizer.get_vocab())
elif hasattr(tokenizer, "vocab_size"):
    vocab_size = tokenizer.vocab_size
else:
    vocab_size = None

all_ids = [
    int(token_id)
    for chunk in train_chunks_nonoverlap
    for token_id in chunk["input_ids"]
]

print("Tokenizer vocabulary size:", vocab_size)
print("Encoder embedding rows:", embedding_size)
print("Training token ID range:", min(all_ids), "to", max(all_ids))
print("Embedding-compatible ID range:", 0, "to", embedding_size - 1)

print("\nChecks:")
print("All training IDs fit embedding table:",
      min(all_ids) >= 0 and max(all_ids) < embedding_size)
if vocab_size is not None:
    print("Tokenizer vocab size equals embedding rows:",
          vocab_size == embedding_size)

print("\nTrainable parameters:")
trainable = []
for name, param in fresh_model.named_parameters():
    if param.requires_grad:
        trainable.append((name, param.numel()))

for name, count in trainable:
    print(f"{name:<65} {count:>10,}")

print("\nTotal trainable parameters:", sum(n for _, n in trainable))


In [ ]:

import torch

phase1_path = (
    "/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt"
)
phase2_path = "/kaggle/working/a4_lorma_weighted_best.pt"

def inspect_checkpoint(path, title):
    ckpt = torch.load(
        path, map_location="cpu", weights_only=False
    )
    print(f"\n{'=' * 65}\n{title}\n{'=' * 65}")
    print("Top-level type:", type(ckpt).__name__)

    if isinstance(ckpt, dict):
        print("Top-level keys:", list(ckpt.keys())[:30])

        state = None
        for key in ("model_state_dict", "state_dict", "model", "encoder_state_dict"):
            if key in ckpt and isinstance(ckpt[key], dict):
                state = ckpt[key]
                print("Selected state dictionary:", key)
                break

        if state is None and all(
            isinstance(k, str) and torch.is_tensor(v)
            for k, v in ckpt.items()
        ):
            state = ckpt
            print("Checkpoint itself appears to be a state dictionary.")

        if state is not None:
            print("Tensor entries:", len(state))
            for key, value in list(state.items())[:15]:
                print(f"  {key:<65} {tuple(value.shape)}")
        else:
            print("No standard state dictionary key found.")

    return ckpt

phase1_ckpt = inspect_checkpoint(phase1_path, "PHASE 1 A4 CHECKPOINT")
phase2_ckpt = inspect_checkpoint(phase2_path, "PHASE 2 LORMA CHECKPOINT")

print("\nCurrent encoder state examples:")
for key, value in list(fresh_model.encoder.state_dict().items())[:20]:
    print(f"  {key:<65} {tuple(value.shape)}")


In [ ]:

import torch

phase1_state = phase1_ckpt["model_state_dict"]
phase2_state = phase2_ckpt["model_state_dict"]

def phase1_to_phase2_key(key):
    # Phase 2 stores the encoder under encoder.
    if not (
        key.startswith("token_embedding.")
        or key.startswith("encoder_pos.")
        or key.startswith("encoder_layers.")
        or key.startswith("encoder_norm.")
    ):
        return None

    mapped = "encoder." + key

    # LoRMA wraps the original linear projection as ".base".
    for proj in ("q_proj", "v_proj"):
        mapped = mapped.replace(
            f".attn.{proj}.weight",
            f".attn.{proj}.base.weight"
        )
        mapped = mapped.replace(
            f".attn.{proj}.bias",
            f".attn.{proj}.base.bias"
        )

    return mapped

matched = []
missing = []
for k1, t1 in phase1_state.items():
    k2 = phase1_to_phase2_key(k1)
    if k2 is None:
        continue
    if k2 not in phase2_state:
        missing.append((k1, k2))
        continue
    t2 = phase2_state[k2]
    if t1.shape == t2.shape:
        diff = (t1.float() - t2.float()).abs()
        matched.append({
            "phase1_key": k1,
            "phase2_key": k2,
            "max_abs_diff": diff.max().item(),
            "mean_abs_diff": diff.mean().item(),
            "exact": torch.equal(t1, t2)
        })
    else:
        missing.append((k1, f"{k2} (shape mismatch)"))

print("Matched encoder tensors:", len(matched))
print("Unmatched encoder tensors:", len(missing))
print("Exact matches:", sum(x["exact"] for x in matched))
print("Non-exact matches:", sum(not x["exact"] for x in matched))

if matched:
    print("Largest max-absolute difference:",
          max(x["max_abs_diff"] for x in matched))
    print("\nFirst 20 comparisons:")
    for x in matched[:20]:
        print(
            f'{x["phase1_key"]:<50} '
            f'exact={str(x["exact"]):<5} '
            f'max_diff={x["max_abs_diff"]:.8g}'
        )

if missing:
    print("\nFirst unmatched tensors:")
    for a, b in missing[:20]:
        print(a, "->", b)


In [ ]:

import inspect

print("Phase 1 final normalization tensors:")
for k in ("encoder_norm.weight", "encoder_norm.bias"):
    print(k, tuple(phase1_state[k].shape))

print("\nDoes Phase 2 checkpoint contain final norm?")
for k in phase2_state:
    if "norm" in k.lower() and (
        "encoder_norm" in k.lower() or k.endswith("norm.weight")
        or k.endswith("norm.bias")
    ):
        print(k, tuple(phase2_state[k].shape))

print("\nEncoder class:", type(fresh_model.encoder).__name__)
print("\nEncoder forward implementation:")
try:
    print(inspect.getsource(type(fresh_model.encoder).forward))
except (OSError, TypeError) as e:
    print("Source unavailable:", e)

print("\nEncoder attributes related to normalization:")
for name in dir(fresh_model.encoder):
    if "norm" in name.lower():
        print(name, getattr(fresh_model.encoder, name))


In [ ]:

import copy
import torch
import torch.nn as nn

# Preserve the current best model and checkpoint.
repaired_model = copy.deepcopy(fresh_model)

# Restore the missing final normalization layer.
repaired_model.encoder.encoder_norm = nn.LayerNorm(512)

with torch.no_grad():
    repaired_model.encoder.encoder_norm.weight.copy_(
        phase1_state["encoder_norm.weight"]
    )
    repaired_model.encoder.encoder_norm.bias.copy_(
        phase1_state["encoder_norm.bias"]
    )

repaired_model = repaired_model.to(device)

# Keep the same LoRMA + classifier adaptation strategy.
for name, param in repaired_model.named_parameters():
    param.requires_grad = (
        "lorma_A" in name
        or "lorma_B" in name
        or name.startswith("classifier.")
    )

# The restored normalization layer is pretrained and frozen.
repaired_model.encoder.encoder_norm.weight.requires_grad = False
repaired_model.encoder.encoder_norm.bias.requires_grad = False

print("Final norm present:",
      hasattr(repaired_model.encoder, "encoder_norm"))
print("Norm weight matches Phase 1:",
      torch.equal(
          repaired_model.encoder.encoder_norm.weight.detach().cpu(),
          phase1_state["encoder_norm.weight"]
      ))
print("Norm bias matches Phase 1:",
      torch.equal(
          repaired_model.encoder.encoder_norm.bias.detach().cpu(),
          phase1_state["encoder_norm.bias"]
      ))
print("Trainable parameters:",
      sum(p.numel() for p in repaired_model.parameters()
          if p.requires_grad))
print("Previous best checkpoint remains:",
      "/kaggle/working/a4_lorma_weighted_best.pt")


In [ ]:

import types

def repaired_encoder_forward(self, input_ids, attention_mask=None):
    B, T = input_ids.shape
    if T > MAX_POS:
        raise ValueError(f"Sequence length {T} exceeds {MAX_POS}")

    pos = torch.arange(T, device=input_ids.device)
    x = self.token_embedding(input_ids) + self.encoder_pos(pos).unsqueeze(0)

    for layer in self.encoder_layers:
        x = layer(x, mask=None)

    # Restore the final normalization used by the Phase 1 encoder.
    x = self.encoder_norm(x)
    return x

repaired_model.encoder.forward = types.MethodType(
    repaired_encoder_forward, repaired_model.encoder
)

# Quick forward-pass sanity check.
sample_ids = torch.as_tensor(
    train_chunks_nonoverlap[0]["input_ids"][:128],
    dtype=torch.long, device=device
).unsqueeze(0)

repaired_model.eval()
with torch.no_grad():
    sample_logits = repaired_model(sample_ids)

if isinstance(sample_logits, (tuple, list)):
    sample_logits = sample_logits[0]

print("Logits shape:", tuple(sample_logits.shape))
print("Finite logits:", bool(torch.isfinite(sample_logits).all()))
print("Final norm used:", hasattr(repaired_model.encoder, "encoder_norm"))


In [ ]:

import copy
import torch
import torch.nn as nn
import types

# 1. Start from the repaired encoder, preserving its pretrained base weights
corrected_encoder = copy.deepcopy(repaired_model.encoder)

# 2. Replace old trained LoRMA adapters with fresh adapters
for layer in corrected_encoder.encoder_layers:
    for proj_name in ("q_proj", "v_proj"):
        proj = getattr(layer.attn, proj_name)

        if isinstance(proj, LORMALinear):
            base_linear = copy.deepcopy(proj.base)
        else:
            base_linear = copy.deepcopy(proj)

        setattr(
            layer.attn,
            proj_name,
            LORMALinear(
                base_linear,
                rank=8,
                alpha=16,
                dropout=0.05
            )
        )

# 3. Create a fresh NER classifier
corrected_model = A4NER()
corrected_model.encoder = corrected_encoder

# 4. Restore the corrected encoder forward pass
def corrected_encoder_forward(self, input_ids, attention_mask=None):
    B, T = input_ids.shape

    if T > MAX_POS:
        raise ValueError(f"Sequence length {T} exceeds MAX_POS={MAX_POS}")

    pos = torch.arange(T, device=input_ids.device)
    x = self.token_embedding(input_ids) + self.encoder_pos(pos).unsqueeze(0)

    for layer in self.encoder_layers:
        x = layer(x, mask=None)

    x = self.encoder_norm(x)
    return x

corrected_model.encoder.forward = types.MethodType(
    corrected_encoder_forward, corrected_model.encoder
)

# 5. Freeze the pretrained encoder; train only LoRMA and classifier
for name, param in corrected_model.named_parameters():
    param.requires_grad = (
        "lorma_A" in name
        or "lorma_B" in name
        or name.startswith("classifier.")
    )

corrected_model = corrected_model.to(device)

trainable = sum(p.numel() for p in corrected_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in corrected_model.parameters())

print("Trainable parameters:", f"{trainable:,}")
print("Total parameters:", f"{total:,}")
print("Final normalization:", hasattr(corrected_model.encoder, "encoder_norm"))
print("Classifier is fresh:", corrected_model.classifier is not repaired_model.classifier)
print("Model device:", next(corrected_model.parameters()).device)

# Sanity check
corrected_model.eval()
sample_ids = torch.as_tensor(
    train_chunks_nonoverlap[0]["input_ids"][:128],
    dtype=torch.long,
    device=device
).unsqueeze(0)

with torch.no_grad():
    sample_logits = corrected_model(sample_ids)

if isinstance(sample_logits, (tuple, list)):
    sample_logits = sample_logits[0]

print("Logits shape:", tuple(sample_logits.shape))
print("Finite logits:", bool(torch.isfinite(sample_logits).all()))


In [ ]:

import torch

corrected_model.eval()

sample_ids = torch.as_tensor(
    train_chunks_nonoverlap[0]["input_ids"][:128],
    dtype=torch.long,
    device=device
).unsqueeze(0)

enc = corrected_model.encoder

def report_tensor(name, x):
    if not torch.is_tensor(x):
        print(f"{name}: not a tensor")
        return
    finite = torch.isfinite(x)
    print(
        f"{name}: shape={tuple(x.shape)}, "
        f"finite={finite.all().item()}, "
        f"nan={torch.isnan(x).sum().item()}, "
        f"inf={torch.isinf(x).sum().item()}"
    )
    if finite.any():
        print(
            f"  finite range: "
            f"{x[finite].min().item():.6g} to "
            f"{x[finite].max().item():.6g}"
        )

with torch.no_grad():
    x = enc.token_embedding(sample_ids)
    report_tensor("Token embeddings", x)

    pos = torch.arange(sample_ids.shape[1], device=device)
    p = enc.encoder_pos(pos).unsqueeze(0)
    report_tensor("Position embeddings", p)

    x = x + p
    report_tensor("Embedding sum", x)

    for i, layer in enumerate(enc.encoder_layers):
        x = layer(x, mask=None)
        report_tensor(f"After encoder layer {i + 1}", x)
        if not torch.isfinite(x).all():
            print("First non-finite activation found; stopping layer inspection.")
            break

    if torch.isfinite(x).all():
        report_tensor("Final norm input", x)
        x = enc.encoder_norm(x)
        report_tensor("After final norm", x)

        logits = corrected_model.classifier(x)
        report_tensor("Classifier logits", logits)


In [ ]:

import torch
import torch.nn as nn

layer = corrected_model.encoder.encoder_layers[0]
attn = layer.attn

print("Attention module:", type(attn).__name__)

for name in ("q_proj", "k_proj", "v_proj", "out_proj"):
    if hasattr(attn, name):
        proj = getattr(attn, name)
        print(f"\n{name}: {type(proj).__name__}")

        if isinstance(proj, LORMALinear):
            print("  Base:", type(proj.base).__name__)
            print("  Rank:", proj.rank if hasattr(proj, "rank") else "not exposed")
            print("  Alpha:", proj.alpha if hasattr(proj, "alpha") else "not exposed")

            for pname, p in proj.named_parameters():
                finite = torch.isfinite(p).all().item()
                print(
                    f"  {pname}: shape={tuple(p.shape)}, "
                    f"finite={finite}, "
                    f"min={p.min().item():.6g}, "
                    f"max={p.max().item():.6g}"
                )
        elif isinstance(proj, nn.Module):
            for pname, p in proj.named_parameters():
                print(
                    f"  {pname}: shape={tuple(p.shape)}, "
                    f"finite={torch.isfinite(p).all().item()}"
                )

print("\n--- Check adapter outputs on finite input ---")

x = (
    corrected_model.encoder.token_embedding(sample_ids)
    + corrected_model.encoder.encoder_pos(
        torch.arange(sample_ids.shape[1], device=device)
    ).unsqueeze(0)
)

for name in ("q_proj", "k_proj", "v_proj", "out_proj"):
    if hasattr(attn, name):
        proj = getattr(attn, name)
        try:
            with torch.no_grad():
                if name == "out_proj":
                    # Test independently using the model's hidden dimension.
                    inp = torch.zeros(
                        1, 8, 512, device=device,
                        dtype=x.dtype
                    )
                else:
                    inp = x
                y = proj(inp)
            print(
                name,
                "shape=", tuple(y.shape),
                "finite=", torch.isfinite(y).all().item()
            )
        except Exception as e:
            print(name, "ERROR:", repr(e))


In [ ]:

import inspect
import torch

print("LORMALinear constructor:")
print(inspect.getsource(LORMALinear.__init__))

print("\nLORMALinear forward:")
print(inspect.getsource(LORMALinear.forward))

print("\nCurrent adapter NaN check:")
for i, layer in enumerate(corrected_model.encoder.encoder_layers):
    for name in ("q_proj", "v_proj"):
        proj = getattr(layer.attn, name)
        print(
            f"Layer {i+1} {name}: "
            f"A finite={torch.isfinite(proj.lorma_A).all().item()}, "
            f"B finite={torch.isfinite(proj.lorma_B).all().item()}"
        )


In [ ]:

import torch
import torch.nn as nn

# Initialize every fresh adapter in the corrected model.
for layer in corrected_model.encoder.encoder_layers:
    for proj_name in ("q_proj", "v_proj"):
        proj = getattr(layer.attn, proj_name)

        with torch.no_grad():
            nn.init.kaiming_uniform_(proj.lorma_A, a=5**0.5)
            nn.init.zeros_(proj.lorma_B)

# Verify all adapter parameters are finite.
all_finite = True

for i, layer in enumerate(corrected_model.encoder.encoder_layers):
    for name in ("q_proj", "v_proj"):
        proj = getattr(layer.attn, name)
        a_ok = torch.isfinite(proj.lorma_A).all().item()
        b_ok = torch.isfinite(proj.lorma_B).all().item()
        all_finite &= a_ok and b_ok
        print(f"Layer {i+1} {name}: A finite={a_ok}, B finite={b_ok}")

print("\nAll adapters finite:", all_finite)

# Check model output again.
corrected_model.eval()

with torch.no_grad():
    logits = corrected_model(sample_ids)

if isinstance(logits, (tuple, list)):
    logits = logits[0]

print("Logits shape:", tuple(logits.shape))
print("Logits finite:", torch.isfinite(logits).all().item())


In [ ]:

print("Train chunks:", len(train_chunks_nonoverlap))
print("Validation chunks:", len(val_chunks_nonoverlap))
print("First train chunk keys:", train_chunks_nonoverlap[0].keys())
print("First validation chunk keys:", val_chunks_nonoverlap[0].keys())

print("\nTrain input length:",
      len(train_chunks_nonoverlap[0]["input_ids"]))
print("Train label length:",
      len(train_chunks_nonoverlap[0]["labels"]))

print("\nCriterion:", criterion)
print("Weights shape:", tuple(weights.shape))
print("Weights finite:", torch.isfinite(weights).all().item())

# Check that the model has no NaN/Inf parameters.
bad_parameters = [
    name for name, p in corrected_model.named_parameters()
    if not torch.isfinite(p).all().item()
]
print("Non-finite parameters:", bad_parameters)

# Confirm the old best checkpoint is still present.
import os
best_path = "/kaggle/working/a4_lorma_weighted_best.pt"
print("Previous best checkpoint exists:", os.path.exists(best_path))


In [ ]:

import copy
import torch
import torch.nn as nn
from torch.optim import AdamW

# Keep the previous best checkpoint untouched.
candidate_path = "/kaggle/working/a4_lorma_normfix_candidate.pt"

optimizer = AdamW(
    [p for p in corrected_model.parameters() if p.requires_grad],
    lr=2e-4,
    weight_decay=0.01
)

batch_size = 2
epochs = 3

def make_batch(chunks, start, end, device):
    ids = torch.tensor(
        [c["input_ids"] for c in chunks[start:end]],
        dtype=torch.long, device=device
    )
    labels = torch.tensor(
        [c["labels"] for c in chunks[start:end]],
        dtype=torch.long, device=device
    )
    return ids, labels

best_candidate_val_loss = float("inf")
candidate_history = []

for epoch in range(1, epochs + 1):
    corrected_model.train()
    total_train_loss = 0.0
    train_batches = 0

    for start in range(0, len(train_chunks_nonoverlap), batch_size):
        ids, labels = make_batch(
            train_chunks_nonoverlap, start,
            min(start + batch_size, len(train_chunks_nonoverlap)),
            device
        )

        optimizer.zero_grad(set_to_none=True)
        logits = corrected_model(ids)

        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            labels.reshape(-1)
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch}, batch {start}"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in corrected_model.parameters() if p.requires_grad],
            max_norm=1.0
        )
        optimizer.step()

        total_train_loss += loss.item()
        train_batches += 1

    corrected_model.eval()
    total_val_loss = 0.0
    val_batches = 0

    with torch.no_grad():
        for start in range(0, len(val_chunks_nonoverlap), batch_size):
            ids, labels = make_batch(
                val_chunks_nonoverlap, start,
                min(start + batch_size, len(val_chunks_nonoverlap)),
                device
            )

            logits = corrected_model(ids)

            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            loss = criterion(
                logits.reshape(-1, logits.shape[-1]),
                labels.reshape(-1)
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite validation loss at epoch {epoch}, batch {start}"
                )

            total_val_loss += loss.item()
            val_batches += 1

    train_loss = total_train_loss / train_batches
    val_loss = total_val_loss / val_batches

    candidate_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(
        f"Epoch {epoch}/{epochs} | "
        f"train loss={train_loss:.4f} | "
        f"validation loss={val_loss:.4f}"
    )

    if val_loss < best_candidate_val_loss:
        best_candidate_val_loss = val_loss
        torch.save({
            "model_state_dict": corrected_model.state_dict(),
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "history": candidate_history,
            "label2id": label2id
        }, candidate_path)
        print("  Saved new candidate best:", candidate_path)

print("\nCandidate training history:")
for row in candidate_history:
    print(row)

print("\nBest candidate validation loss:", best_candidate_val_loss)
print("Candidate checkpoint exists:", __import__("os").path.exists(candidate_path))


In [ ]:

import os
import copy
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.nn.utils.rnn import pad_sequence

# ============================================================
# 1. Configuration
# ============================================================
candidate_path = "/kaggle/working/a4_lorma_normfix_candidate.pt"

batch_size = 2
epochs = 3
learning_rate = 2e-4
weight_decay = 0.01

assert len(train_chunks_nonoverlap) > 0
assert len(val_chunks_nonoverlap) > 0

# ============================================================
# 2. Verify model parameters before training
# ============================================================
corrected_model = corrected_model.to(device)

bad_parameters = [
    name for name, p in corrected_model.named_parameters()
    if not torch.isfinite(p).all().item()
]
if bad_parameters:
    raise RuntimeError(f"Non-finite model parameters: {bad_parameters}")

trainable_params = [
    p for p in corrected_model.parameters() if p.requires_grad
]
if not trainable_params:
    raise RuntimeError("No trainable parameters found.")

print("Train chunks:", len(train_chunks_nonoverlap))
print("Validation chunks:", len(val_chunks_nonoverlap))
print("Trainable parameters:", sum(p.numel() for p in trainable_params))

# ============================================================
# 3. Variable-length batch creation
# ============================================================
def make_batch(chunks, start, end, device):
    batch = chunks[start:end]

    ids_list = [
        torch.as_tensor(c["input_ids"], dtype=torch.long)
        for c in batch
    ]
    labels_list = [
        torch.as_tensor(c["labels"], dtype=torch.long)
        for c in batch
    ]

    ids = pad_sequence(
        ids_list,
        batch_first=True,
        padding_value=0
    ).to(device)

    labels = pad_sequence(
        labels_list,
        batch_first=True,
        padding_value=-100
    ).to(device)

    return ids, labels

# ============================================================
# 4. Weighted loss and optimizer
# ============================================================
criterion = nn.CrossEntropyLoss(
    weight=weights.to(device),
    ignore_index=-100
)

optimizer = AdamW(
    trainable_params,
    lr=learning_rate,
    weight_decay=weight_decay
)

# ============================================================
# 5. Training and validation
# ============================================================
best_candidate_val_loss = float("inf")
candidate_history = []

for epoch in range(1, epochs + 1):

    # -------------------- Training --------------------
    corrected_model.train()
    train_loss_sum = 0.0
    train_batches = 0

    for start in range(0, len(train_chunks_nonoverlap), batch_size):
        end = min(start + batch_size, len(train_chunks_nonoverlap))
        ids, labels = make_batch(
            train_chunks_nonoverlap, start, end, device
        )

        optimizer.zero_grad(set_to_none=True)

        logits = corrected_model(ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            labels.reshape(-1)
        )

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch}, batch {start}"
            )

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_params, max_norm=1.0
        )
        if not torch.isfinite(grad_norm).item():
            raise RuntimeError(
                f"Non-finite gradient norm at epoch {epoch}, batch {start}"
            )

        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    avg_train_loss = train_loss_sum / max(train_batches, 1)

    # -------------------- Validation --------------------
    corrected_model.eval()
    val_loss_sum = 0.0
    val_batches = 0

    with torch.no_grad():
        for start in range(0, len(val_chunks_nonoverlap), batch_size):
            end = min(start + batch_size, len(val_chunks_nonoverlap))
            ids, labels = make_batch(
                val_chunks_nonoverlap, start, end, device
            )

            logits = corrected_model(ids)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            loss = criterion(
                logits.reshape(-1, logits.shape[-1]),
                labels.reshape(-1)
            )

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"Non-finite validation loss at epoch {epoch}, batch {start}"
                )

            val_loss_sum += loss.item()
            val_batches += 1

    avg_val_loss = val_loss_sum / max(val_batches, 1)

    record = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss
    }
    candidate_history.append(record)

    print(
        f"Epoch {epoch}/{epochs} | "
        f"train loss={avg_train_loss:.4f} | "
        f"val loss={avg_val_loss:.4f}"
    )

    # Save candidate separately; preserve the existing best checkpoint.
    if avg_val_loss < best_candidate_val_loss:
        best_candidate_val_loss = avg_val_loss

        torch.save(
            {
                "model_state_dict": corrected_model.state_dict(),
                "epoch": epoch,
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss,
                "history": copy.deepcopy(candidate_history),
                "label2id": label2id
            },
            candidate_path
        )

        print("  Saved candidate best:", candidate_path)

# ============================================================
# 6. Final report
# ============================================================
print("\nTraining complete.")
print("Candidate checkpoint exists:", os.path.exists(candidate_path))
print("Best candidate validation loss:", best_candidate_val_loss)
print("Candidate history:")

for row in candidate_history:
    print(row)

print(
    "\nOriginal best checkpoint preserved:",
    os.path.exists("/kaggle/working/a4_lorma_weighted_best.pt")
)


In [ ]:

import inspect

print("decode_eval available:", "decode_eval" in globals())
if "decode_eval" in globals():
    print(inspect.signature(decode_eval))

print("Validation documents:", len(val_docs) if "val_docs" in globals() else "val_docs not found")
print("Validation chunks:", len(val_chunks_nonoverlap))
print("Candidate checkpoint exists:", __import__("os").path.exists(
    "/kaggle/working/a4_lorma_normfix_candidate.pt"
))


In [ ]:

import inspect

# Find document/chunk-related variables currently available.
for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if any(term in name.lower() for term in (
        "val", "valid", "gold", "doc", "chunk", "offset", "token"
    )):
        if isinstance(obj, (list, tuple, dict)):
            print(
                f"{name}: type={type(obj).__name__}, "
                f"length={len(obj)}"
            )

# Inspect evaluation-related function signatures.
for name in (
    "decode_eval",
    "make_chunks_aligned",
    "build_a4_chunks",
    "evaluate_model",
    "eval_model"
):
    obj = globals().get(name)
    if callable(obj):
        try:
            print(f"\n{name}{inspect.signature(obj)}")
        except Exception:
            print(f"\n{name}: callable")


In [ ]:

import torch
import inspect

print("Validation cache entries:", len(validation_cache))
print("Cache entry type:", type(validation_cache[0]).__name__)

if isinstance(validation_cache[0], dict):
    print("Cache keys:", validation_cache[0].keys())
    for k, v in validation_cache[0].items():
        try:
            print(
                f"  {k}: type={type(v).__name__}, "
                f"length={len(v)}"
            )
        except TypeError:
            print(f"  {k}: type={type(v).__name__}, value={v}")
else:
    print("First entry:", repr(validation_cache[0])[:1500])

print("\nGold dictionary examples:")
print("fold2_gold keys:", list(fold2_gold.keys())[:3])
print("fold2_gold first value type:", type(next(iter(fold2_gold.values()))).__name__)
print("fold2_gold first value:", repr(next(iter(fold2_gold.values())))[:1000])

print("\nDecoder source:")
print(inspect.getsource(decode_eval))


In [ ]:

import torch
import os
from collections import defaultdict

candidate_path = "/kaggle/working/a4_lorma_normfix_candidate.pt"
assert os.path.exists(candidate_path), "Candidate checkpoint not found."

# Load the candidate weights.
ckpt = torch.load(candidate_path, map_location=device, weights_only=False)
corrected_model.load_state_dict(ckpt["model_state_dict"], strict=True)
corrected_model.to(device).eval()

print("Evaluating candidate epoch:", ckpt["epoch"])
print("Candidate validation loss:", ckpt["val_loss"])

# Rebuild per-document token probabilities from validation chunks.
# Chunks are non-overlapping, so each token position should occur once.
doc_probs = defaultdict(dict)

with torch.no_grad():
    for chunk in val_chunks_nonoverlap:
        ids = torch.as_tensor(
            chunk["input_ids"], dtype=torch.long, device=device
        ).unsqueeze(0)

        logits = corrected_model(ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        probs = torch.softmax(logits[0].float(), dim=-1).cpu()

        doc_id = str(chunk["doc_id"])
        chunk_start = int(chunk["chunk_start"])

        # These are token-position probabilities, not character offsets.
        for j, prob in enumerate(probs):
            absolute_pos = chunk_start + j
            if absolute_pos not in doc_probs[doc_id]:
                doc_probs[doc_id][absolute_pos] = prob

print("Documents with predictions:", len(doc_probs))
print("Probability entries:", sum(len(v) for v in doc_probs.values()))

# Inspect the validation cache to recover document-specific token offsets.
cache_by_id = {str(x["doc_id"]): x for x in validation_cache}
print("Cached document IDs:", len(cache_by_id))
print("Example cached ID:", next(iter(cache_by_id)))
print("Example predicted ID:", next(iter(doc_probs)))

# Verify that token positions align before scoring.
for doc_id, positions in list(doc_probs.items())[:3]:
    print(
        doc_id,
        "predicted positions:", len(positions),
        "cached offsets:", len(cache_by_id[doc_id]["offsets"])
        if doc_id in cache_by_id else "NO MATCH"
    )


In [ ]:

from collections import Counter

# Build candidate predictions and gold sets for each validation document.
candidate_eval_docs = []

for doc_id, cache in cache_by_id.items():
    if doc_id not in doc_probs:
        continue

    text = cache["text"]
    offsets = cache["offsets"]
    gold = set(cache["gold"])

    # Ensure every cached token has a corresponding probability vector.
    probs_by_pos = doc_probs[doc_id]
    if len(probs_by_pos) != len(offsets):
        raise ValueError(
            f"Position mismatch for {doc_id}: "
            f"{len(probs_by_pos)} predictions vs {len(offsets)} offsets"
        )

    probs = torch.stack([probs_by_pos[i] for i in range(len(offsets))])
    candidate_eval_docs.append({
        "doc_id": doc_id,
        "text": text,
        "offsets": offsets,
        "probs": probs,
        "gold": gold
    })

print("Documents evaluated:", len(candidate_eval_docs))
print("Gold spans:", sum(len(d["gold"]) for d in candidate_eval_docs))

# Thresholding:
# 1. Select the most likely entity class.
# 2. Require minimum entity probability p_min.
# 3. Require entity probability minus O probability >= margin_min.
def predict_spans_for_doc(doc, p_min, margin_min):
    probs = doc["probs"]
    pred_ids = probs.argmax(dim=-1).tolist()

    filtered_ids = []
    for i, pred_id in enumerate(pred_ids):
        if pred_id == label2id["O"]:
            filtered_ids.append(pred_id)
            continue

        entity_prob = float(probs[i, pred_id])
        o_prob = float(probs[i, label2id["O"]])

        if entity_prob < p_min or entity_prob - o_prob < margin_min:
            filtered_ids.append(label2id["O"])
        else:
            filtered_ids.append(pred_id)

    return decode_eval(
        filtered_ids,
        doc["offsets"],
        doc["text"],
        ignore_orphan_i=True
    )

# Sweep thresholds and calculate exact-span micro-F1.
threshold_results = []
gold_total = sum(len(d["gold"]) for d in candidate_eval_docs)

for p_min in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    for margin_min in [0.00, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15]:
        tp = fp = fn = 0

        for doc in candidate_eval_docs:
            pred = predict_spans_for_doc(doc, p_min, margin_min)
            gold = doc["gold"]

            tp += len(pred & gold)
            fp += len(pred - gold)
            fn += len(gold - pred)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        threshold_results.append({
            "p_min": p_min,
            "margin_min": margin_min,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "micro_f1": f1
        })

threshold_results.sort(key=lambda x: x["micro_f1"], reverse=True)

print("\nTop 10 candidate settings:")
for r in threshold_results[:10]:
    print(
        f"p_min={r['p_min']:.2f}, "
        f"margin={r['margin_min']:.2f}, "
        f"TP={r['tp']}, FP={r['fp']}, FN={r['fn']}, "
        f"P={r['precision']:.4f}, R={r['recall']:.4f}, "
        f"micro-F1={r['micro_f1']:.4f}"
    )

best_candidate_result = threshold_results[0]
print("\nBest candidate validation micro-F1:",
      round(best_candidate_result["micro_f1"], 4))


In [ ]:

import torch
import os

paths = {
    "previous_best": "/kaggle/working/a4_lorma_weighted_best.pt",
    "normfix_candidate": "/kaggle/working/a4_lorma_normfix_candidate.pt",
}

for name, path in paths.items():
    print(f"\n--- {name} ---")
    print("Exists:", os.path.exists(path))
    if os.path.exists(path):
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        print("Epoch:", ckpt.get("epoch"))
        print("Train loss:", ckpt.get("train_loss"))
        print("Validation loss:", ckpt.get("val_loss"))
        print("History:", ckpt.get("history"))


In [ ]:

import os

print("Test file exists:",
      os.path.exists("/kaggle/input/datasets/aq1de3/fold3-te/fold3-text.json"))

print("Labels file exists:",
      os.path.exists("/kaggle/input/datasets/aq1de3/labelses/labels.txt"))

print("Best checkpoint exists:",
      os.path.exists("/kaggle/working/a4_lorma_weighted_best.pt"))

print("Existing submission-related variables:")
for name in globals():
    if any(k in name.lower() for k in ("submission", "submit", "prediction", "pred_file")):
        print(name)


In [ ]:

import json
import os

test_path = "/kaggle/input/datasets/aq1de3/fold3-te/fold3-text.json"
labels_path = "/kaggle/input/datasets/aq1de3/labelses/labels.txt"
best_path = "/kaggle/working/a4_lorma_weighted_best.pt"

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

with open(labels_path, "r", encoding="utf-8") as f:
    label_lines = f.read().splitlines()

print("Test data type:", type(test_data).__name__)
print("Test document count:", len(test_data))
print("Label count:", len(label_lines))
print("Label examples:", label_lines[:10])

if isinstance(test_data, dict):
    first_id = next(iter(test_data))
    print("First document ID:", first_id)
    print("First document value type:", type(test_data[first_id]).__name__)
    print("First document value:", repr(test_data[first_id])[:1000])
elif isinstance(test_data, list) and test_data:
    print("First document:", repr(test_data[0])[:1000])

print("\nPredictions variable:")
print("Type:", type(predictions).__name__)
try:
    print("Length:", len(predictions))
    print("First item:", repr(predictions[0])[:500])
except Exception:
    print(repr(predictions)[:500])

print("\nBest checkpoint exists:", os.path.exists(best_path))


In [ ]:

import inspect

for name, obj in list(globals().items()):
    if callable(obj) and any(
        term in name.lower()
        for term in ("predict", "submit", "export", "write", "infer")
    ):
        try:
            print(f"\n{name}{inspect.signature(obj)}")
        except Exception:
            print(f"\n{name}: {type(obj).__name__}")

print("\nRelevant model and tokenizer variables:")
for name in (
    "tokenizer", "test_data", "id2label", "label2id",
    "MAX_POS", "MAX_TOKENS", "STRIDE", "predictions"
):
    if name in globals():
        obj = globals()[name]
        try:
            print(name, "type=", type(obj).__name__, "length=", len(obj))
        except Exception:
            print(name, "type=", type(obj).__name__)


In [ ]:

import inspect

print("predict_doc source:\n")
print(inspect.getsource(predict_doc))

print("\nTokenizer methods available:")
print("encode:", hasattr(tokenizer, "encode"))
print("decode:", hasattr(tokenizer, "decode"))
print("get_vocab_size:", hasattr(tokenizer, "get_vocab_size"))

print("\nTest sample:")
test_id = next(iter(test_data))
test_text = test_data[test_id]
print("Document ID:", test_id)
print("Text length:", len(test_text))
print("Text preview:", repr(test_text[:250]))


In [ ]:

import os
import json
import torch

best_path = "/kaggle/working/a4_lorma_weighted_best.pt"
assert os.path.exists(best_path), f"Checkpoint not found: {best_path}"

# Load the previous best weights into the model used by predict_doc.
ckpt = torch.load(best_path, map_location=device, weights_only=False)
fresh_model.load_state_dict(ckpt["model_state_dict"], strict=True)
fresh_model.to(device)
fresh_model.eval()

print("Loaded checkpoint:", best_path)
print("Checkpoint epoch:", ckpt.get("epoch"))
print("Validation loss:", ckpt.get("val_loss"))

# Predict every test document.
submission_predictions = {}

for i, (doc_id, text) in enumerate(test_data.items(), start=1):
    pred_ids, offsets = predict_doc(text)

    if len(pred_ids) != len(offsets):
        raise ValueError(
            f"Token/offset mismatch for document {doc_id}: "
            f"{len(pred_ids)} predictions vs {len(offsets)} offsets"
        )

    submission_predictions[str(doc_id)] = {
        "predictions": pred_ids,
        "offsets": offsets
    }

    print(
        f"{i}/{len(test_data)} | document={doc_id} | "
        f"characters={len(text)} | tokens={len(pred_ids)}"
    )

# Save raw predictions separately; this is not yet the final competition format.
raw_path = "/kaggle/working/best_model_test_predictions.json"

with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(submission_predictions, f)

print("\nSaved raw predictions:", raw_path)
print("Documents:", len(submission_predictions))
print("File size (MB):", round(os.path.getsize(raw_path) / 1e6, 2))


In [ ]:

import json
import os

path = "/kaggle/working/best_model_test_predictions.json"

with open(path, "r", encoding="utf-8") as f:
    saved = json.load(f)

print("File exists:", os.path.exists(path))
print("Documents saved:", len(saved))
print("Expected documents:", len(test_data))
print("Sample document ID:", next(iter(saved)))
print("Sample prediction keys:", list(next(iter(saved.values())).keys()))


In [ ]:

import os
import shutil
import json

source_path = "/kaggle/working/best_model_test_predictions.json"
submission_dir = "/kaggle/working/submission"
destination_path = os.path.join(
    submission_dir, "best_model_test_predictions.json"
)

# Create the submission folder and copy the verified predictions.
os.makedirs(submission_dir, exist_ok=True)

if not os.path.exists(source_path):
    raise FileNotFoundError(f"Predictions file not found: {source_path}")

shutil.copy2(source_path, destination_path)

# Verify the saved file.
with open(destination_path, "r", encoding="utf-8") as f:
    saved_predictions = json.load(f)

assert len(saved_predictions) == 35, (
    f"Expected 35 documents, found {len(saved_predictions)}"
)

print("Submission folder:", submission_dir)
print("Saved file:", destination_path)
print("Documents:", len(saved_predictions))
print("File size (MB):", round(os.path.getsize(destination_path) / 1e6, 2))
print("Verification: PASSED")


In [ ]:

import os
import zipfile

submission_dir = "/kaggle/working/submission"
source_file = "/kaggle/working/best_model_test_predictions.json"
destination_file = os.path.join(
    submission_dir, "best_model_test_predictions.json"
)
zip_path = "/kaggle/working/submission.zip"

os.makedirs(submission_dir, exist_ok=True)

if not os.path.exists(source_file):
    raise FileNotFoundError(f"Missing predictions file: {source_file}")

# Ensure the submission folder contains the predictions.
import shutil
shutil.copy2(source_file, destination_file)

# Package the folder itself into the ZIP.
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(destination_file, "submission/best_model_test_predictions.json")

print("ZIP created:", zip_path)
print("Size (MB):", round(os.path.getsize(zip_path) / 1e6, 2))
print("Download it from the Kaggle Output panel.")


In [ ]:

import os
import zipfile
import shutil

source = "/kaggle/working/best_model_test_predictions.json"
folder = "/kaggle/working/submission"
zip_path = "/kaggle/working/submission.zip"

os.makedirs(folder, exist_ok=True)
shutil.copy2(
    source,
    os.path.join(folder, "best_model_test_predictions.json")
)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(
        os.path.join(folder, "best_model_test_predictions.json"),
        "submission/best_model_test_predictions.json"
    )

print("ZIP:", zip_path)
print("Size (MB):", round(os.path.getsize(zip_path) / 1e6, 2))
print("Ready to download from Kaggle's Output panel.")


In [ ]:

import torch
from collections import Counter

# Reload the previous best model.
best_path = "/kaggle/working/a4_lorma_weighted_best.pt"
ckpt = torch.load(best_path, map_location=device, weights_only=False)

fresh_model.load_state_dict(ckpt["model_state_dict"], strict=True)
fresh_model.to(device).eval()

print("Evaluating previous best checkpoint, epoch:", ckpt.get("epoch"))

# Recompute token probabilities on non-overlapping validation chunks.
# Aggregate each token position by document.
best_doc_probs = {}

with torch.no_grad():
    for chunk in val_chunks_nonoverlap:
        ids = torch.as_tensor(
            chunk["input_ids"], dtype=torch.long, device=device
        ).unsqueeze(0)

        logits = fresh_model(ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        probs = torch.softmax(logits[0].float(), dim=-1).cpu()
        doc_id = str(chunk["doc_id"])
        start = int(chunk["chunk_start"])

        best_doc_probs.setdefault(doc_id, {})
        for j, prob in enumerate(probs):
            best_doc_probs[doc_id][start + j] = prob

# Reuse the aligned validation cache and established decoder.
best_eval_docs = []

for doc_id, cache in cache_by_id.items():
    if doc_id not in best_doc_probs:
        continue

    pos_probs = best_doc_probs[doc_id]
    offsets_doc = cache["offsets"]

    if len(pos_probs) != len(offsets_doc):
        raise ValueError(
            f"Position mismatch for {doc_id}: "
            f"{len(pos_probs)} vs {len(offsets_doc)}"
        )

    best_eval_docs.append({
        "doc_id": doc_id,
        "text": cache["text"],
        "offsets": offsets_doc,
        "probs": torch.stack([pos_probs[i] for i in range(len(offsets_doc))]),
        "gold": set(cache["gold"])
    })

# Use the previous best threshold settings.
p_min = 0.40
margin_min = 0.03

per_type = Counter()
entity_types = sorted(
    label.removeprefix("B-")
    for label in id2label.values()
    if label.startswith("B-")
)

for doc in best_eval_docs:
    pred = predict_spans_for_doc(doc, p_min, margin_min)
    gold = doc["gold"]

    for typ in entity_types:
        pred_t = {s for s in pred if s[2] == typ}
        gold_t = {s for s in gold if s[2] == typ}

        per_type[(typ, "tp")] += len(pred_t & gold_t)
        per_type[(typ, "fp")] += len(pred_t - gold_t)
        per_type[(typ, "fn")] += len(gold_t - pred_t)

print("\nPer-entity validation metrics")
print(f"{'Entity':<12} {'TP':>6} {'FP':>6} {'FN':>6} {'Prec.':>8} {'Recall':>8} {'F1':>8}")

for typ in entity_types:
    tp = per_type[(typ, "tp")]
    fp = per_type[(typ, "fp")]
    fn = per_type[(typ, "fn")]

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    print(
        f"{typ:<12} {tp:>6} {fp:>6} {fn:>6} "
        f"{precision:>8.3f} {recall:>8.3f} {f1:>8.3f}"
    )


In [ ]:

from collections import Counter, defaultdict
import torch

# Compare token-level predictions against the gold BIO labels
# already stored in the non-overlapping validation chunks.
token_confusion = defaultdict(Counter)

fresh_model.eval()

with torch.no_grad():
    for chunk in val_chunks_nonoverlap:
        ids = torch.as_tensor(
            chunk["input_ids"], dtype=torch.long, device=device
        ).unsqueeze(0)

        logits = fresh_model(ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        pred_ids = logits[0].argmax(dim=-1).cpu().tolist()
        gold_ids = list(chunk["labels"])

        if len(pred_ids) != len(gold_ids):
            raise ValueError("Prediction/gold token count mismatch")

        for pred_id, gold_id in zip(pred_ids, gold_ids):
            if gold_id == -100:
                continue
            token_confusion[int(gold_id)][int(pred_id)] += 1

print("Gold BIO label -> most frequent predicted labels")
for gold_id in range(len(id2label)):
    counts = token_confusion[gold_id]
    total = sum(counts.values())
    if total == 0:
        continue

    top = counts.most_common(4)
    formatted = [
        (id2label.get(pred_id, str(pred_id)), count)
        for pred_id, count in top
    ]
    correct = counts.get(gold_id, 0)

    print(
        f"{id2label[gold_id]:<12} total={total:>7} "
        f"correct={correct:>7} accuracy={correct/total:.3f} "
        f"top_predictions={formatted}"
    )


In [ ]:

from collections import Counter

print("Current class weights:")
for label_id in range(len(id2label)):
    label = id2label[label_id]
    count = sum(
        chunk["labels"].count(label_id)
        for chunk in train_chunks_nonoverlap
    )
    print(
        f"{label:<12} count={count:>7} "
        f"weight={weights[label_id].item():.4f}"
    )

print("\nWeight range:")
print("Min:", weights.min().item())
print("Max:", weights.max().item())
print("Mean:", weights.mean().item())

print("\nLoss configuration:")
print("Criterion:", criterion)
print("Ignore index:", criterion.ignore_index)


In [ ]:

import torch
import torch.nn as nn
from collections import Counter

# Count labels in the existing non-overlapping training chunks.
counts = Counter()
for chunk in train_chunks_nonoverlap:
    counts.update(chunk["labels"])

freq = torch.tensor(
    [counts[i] for i in range(len(id2label))],
    dtype=torch.float32
)

# Square-root inverse-frequency weighting.
new_weights = (freq.sum() / (len(id2label) * freq.clamp_min(1))).sqrt()

# Normalize, then cap extreme weights.
new_weights = new_weights / new_weights.mean()
new_weights = new_weights.clamp(min=0.25, max=2.5)

# Give O a moderate weight rather than allowing it to dominate.
new_weights[label2id["O"]] = 0.5

# Normalize non-O weights to mean 1, leaving O at 0.5.
non_o = [i for i in range(len(id2label)) if i != label2id["O"]]
new_weights[non_o] /= new_weights[non_o].mean()

print("New class weights:")
for i in range(len(id2label)):
    print(
        f"{id2label[i]:<12} "
        f"count={int(freq[i]):>7} "
        f"weight={new_weights[i].item():.4f}"
    )

print("\nWeight range:", new_weights.min().item(), new_weights.max().item())
print("O weight:", new_weights[label2id["O"]].item())
print("Weights finite:", torch.isfinite(new_weights).all().item())


In [ ]:

import copy
import os
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.nn.utils.rnn import pad_sequence

# Start from the repaired encoder so the final LayerNorm is preserved.
# Replace each adapter with a fresh one, preserving its pretrained base.
balanced_encoder = copy.deepcopy(repaired_model.encoder)

for layer in balanced_encoder.encoder_layers:
    for proj_name in ("q_proj", "v_proj"):
        proj = getattr(layer.attn, proj_name)

        if isinstance(proj, LORMALinear):
            base_linear = copy.deepcopy(proj.base)
        else:
            base_linear = copy.deepcopy(proj)

        setattr(
            layer.attn,
            proj_name,
            LORMALinear(
                base_linear,
                rank=8,
                alpha=16,
                dropout=0.05
            )
        )

# Fresh classifier and repaired encoder.
balanced_model = A4NER()
balanced_model.encoder = balanced_encoder

# Ensure the corrected encoder applies the final LayerNorm.
balanced_model.encoder.forward = types.MethodType(
    corrected_encoder_forward, balanced_model.encoder
)

# Initialize every fresh adapter explicitly.
for layer in balanced_model.encoder.encoder_layers:
    for name in ("q_proj", "v_proj"):
        proj = getattr(layer.attn, name)
        with torch.no_grad():
            nn.init.kaiming_uniform_(proj.lorma_A, a=5**0.5)
            nn.init.zeros_(proj.lorma_B)

# Train only LoRMA adapters and classifier.
for name, param in balanced_model.named_parameters():
    param.requires_grad = (
        "lorma_A" in name
        or "lorma_B" in name
        or name.startswith("classifier.")
    )

balanced_model = balanced_model.to(device)
trainable_params = [
    p for p in balanced_model.parameters() if p.requires_grad
]

# Weighted loss with padding ignored.
balanced_criterion = nn.CrossEntropyLoss(
    weight=new_weights.to(device),
    ignore_index=-100
)

optimizer = AdamW(
    trainable_params,
    lr=2e-4,
    weight_decay=0.01
)

# Variable-length batches.
def make_balanced_batch(chunks, start, end, device):
    batch = chunks[start:end]

    ids = pad_sequence(
        [torch.as_tensor(c["input_ids"], dtype=torch.long) for c in batch],
        batch_first=True,
        padding_value=0
    ).to(device)

    labels = pad_sequence(
        [torch.as_tensor(c["labels"], dtype=torch.long) for c in batch],
        batch_first=True,
        padding_value=-100
    ).to(device)

    return ids, labels

# Train for three epochs.
balanced_history = []
best_balanced_val_loss = float("inf")
balanced_path = "/kaggle/working/a4_lorma_balanced_v2.pt"

for epoch in range(1, 4):
    balanced_model.train()
    train_sum = 0.0
    train_n = 0

    for start in range(0, len(train_chunks_nonoverlap), 2):
        end = min(start + 2, len(train_chunks_nonoverlap))
        ids, labels = make_balanced_batch(
            train_chunks_nonoverlap, start, end, device
        )

        optimizer.zero_grad(set_to_none=True)
        logits = balanced_model(ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        loss = balanced_criterion(
            logits.reshape(-1, logits.shape[-1]),
            labels.reshape(-1)
        )

        if not torch.isfinite(loss).item():
            raise RuntimeError(f"Non-finite training loss at epoch {epoch}")

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_params, max_norm=1.0
        )
        if not torch.isfinite(grad_norm).item():
            raise RuntimeError(f"Non-finite gradient at epoch {epoch}")

        optimizer.step()
        train_sum += loss.item()
        train_n += 1

    balanced_model.eval()
    val_sum = 0.0
    val_n = 0

    with torch.no_grad():
        for start in range(0, len(val_chunks_nonoverlap), 2):
            end = min(start + 2, len(val_chunks_nonoverlap))
            ids, labels = make_balanced_batch(
                val_chunks_nonoverlap, start, end, device
            )

            logits = balanced_model(ids)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            loss = balanced_criterion(
                logits.reshape(-1, logits.shape[-1]),
                labels.reshape(-1)
            )

            if not torch.isfinite(loss).item():
                raise RuntimeError(f"Non-finite validation loss at epoch {epoch}")

            val_sum += loss.item()
            val_n += 1

    train_loss = train_sum / train_n
    val_loss = val_sum / val_n
    balanced_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(
        f"Epoch {epoch}/3 | train loss={train_loss:.4f} | "
        f"validation loss={val_loss:.4f}"
    )

    if val_loss < best_balanced_val_loss:
        best_balanced_val_loss = val_loss
        torch.save({
            "model_state_dict": balanced_model.state_dict(),
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "history": copy.deepcopy(balanced_history),
            "label2id": label2id
        }, balanced_path)
        print("  Saved:", balanced_path)

print("\nTraining history:")
for row in balanced_history:
    print(row)

print("Best validation loss:", best_balanced_val_loss)
print("Candidate saved:", os.path.exists(balanced_path))
print(
    "Original best preserved:",
    os.path.exists("/kaggle/working/a4_lorma_weighted_best.pt")
)


In [ ]:

import torch
from collections import defaultdict

balanced_path = "/kaggle/working/a4_lorma_balanced_v2.pt"
ckpt = torch.load(balanced_path, map_location=device, weights_only=False)

balanced_model.load_state_dict(ckpt["model_state_dict"], strict=True)
balanced_model.to(device).eval()

print("Candidate epoch:", ckpt["epoch"])
print("Candidate validation loss:", ckpt["val_loss"])

balanced_doc_probs = defaultdict(dict)

with torch.no_grad():
    for chunk in val_chunks_nonoverlap:
        ids = torch.as_tensor(
            chunk["input_ids"], dtype=torch.long, device=device
        ).unsqueeze(0)

        logits = balanced_model(ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        probs = torch.softmax(logits[0].float(), dim=-1).cpu()
        doc_id = str(chunk["doc_id"])
        start = int(chunk["chunk_start"])

        for j, prob in enumerate(probs):
            balanced_doc_probs[doc_id][start + j] = prob

balanced_eval_docs = []

for doc_id, cache in cache_by_id.items():
    if doc_id not in balanced_doc_probs:
        continue

    pos_probs = balanced_doc_probs[doc_id]
    offsets_doc = cache["offsets"]

    if len(pos_probs) != len(offsets_doc):
        raise ValueError(
            f"Position mismatch for {doc_id}: "
            f"{len(pos_probs)} predictions vs {len(offsets_doc)} offsets"
        )

    balanced_eval_docs.append({
        "doc_id": doc_id,
        "text": cache["text"],
        "offsets": offsets_doc,
        "probs": torch.stack(
            [pos_probs[i] for i in range(len(offsets_doc))]
        ),
        "gold": set(cache["gold"])
    })

print("Documents evaluated:", len(balanced_eval_docs))
print("Gold spans:", sum(len(d["gold"]) for d in balanced_eval_docs))

# Sweep confidence and margin thresholds.
balanced_results = []

for p_min in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    for margin_min in [0.00, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15]:
        tp = fp = fn = 0

        for doc in balanced_eval_docs:
            pred = predict_spans_for_doc(doc, p_min, margin_min)
            gold = doc["gold"]

            tp += len(pred & gold)
            fp += len(pred - gold)
            fn += len(gold - pred)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        balanced_results.append({
            "p_min": p_min,
            "margin_min": margin_min,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "micro_f1": f1
        })

balanced_results.sort(
    key=lambda x: x["micro_f1"], reverse=True
)

print("\nTop 10 balanced candidate settings:")
for r in balanced_results[:10]:
    print(
        f"p_min={r['p_min']:.2f}, margin={r['margin_min']:.2f}, "
        f"TP={r['tp']}, FP={r['fp']}, FN={r['fn']}, "
        f"P={r['precision']:.4f}, R={r['recall']:.4f}, "
        f"F1={r['micro_f1']:.4f}"
    )

best_balanced_result = balanced_results[0]
print(
    "\nBest balanced candidate validation micro-F1:",
    round(best_balanced_result["micro_f1"], 4)
)


In [ ]:

from collections import Counter
import torch

def get_predicted_label_counts(model, chunks):
    counts = Counter()
    model.eval()

    with torch.no_grad():
        for start in range(0, len(chunks), 2):
            batch = chunks[start:start + 2]
            ids = torch.nn.utils.rnn.pad_sequence(
                [
                    torch.as_tensor(c["input_ids"], dtype=torch.long)
                    for c in batch
                ],
                batch_first=True,
                padding_value=0
            ).to(device)

            logits = model(ids)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            pred_ids = logits.argmax(dim=-1).cpu()

            for i, chunk in enumerate(batch):
                n = len(chunk["input_ids"])
                counts.update(pred_ids[i, :n].tolist())

    return counts

# Evaluate both models on the same validation chunks.
best_ckpt = torch.load(
    "/kaggle/working/a4_lorma_weighted_best.pt",
    map_location=device,
    weights_only=False
)
fresh_model.load_state_dict(best_ckpt["model_state_dict"], strict=True)

original_counts = get_predicted_label_counts(
    fresh_model, val_chunks_nonoverlap
)
balanced_counts = get_predicted_label_counts(
    balanced_model, val_chunks_nonoverlap
)

print(f"{'Label':<12} {'Original best':>15} {'Balanced v2':>15}")
for label_id in range(len(id2label)):
    label = id2label[label_id]
    print(
        f"{label:<12} "
        f"{original_counts[label_id]:>15} "
        f"{balanced_counts[label_id]:>15}"
    )

print("\nTotal original predictions:", sum(original_counts.values()))
print("Total balanced predictions:", sum(balanced_counts.values()))


In [ ]:

# Diagnostic only — does not modify either checkpoint.
import torch
from collections import Counter

print("Label mapping:")
for i in range(len(id2label)):
    print(i, id2label[i], "->", label2id.get(id2label[i]))

print("\nBalanced model output diagnostics:")
balanced_model.eval()
with torch.no_grad():
    sample = val_chunks_nonoverlap[0]
    x = torch.tensor(
        sample["input_ids"], dtype=torch.long, device=device
    ).unsqueeze(0)

    logits = balanced_model(x)
    if isinstance(logits, (tuple, list)):
        logits = logits[0]

    print("Logits shape:", tuple(logits.shape))
    print("All logits finite:", torch.isfinite(logits).all().item())
    print("Logit min/max:", logits.min().item(), logits.max().item())
    print("Mean logit by class:")
    means = logits[0].mean(dim=0).detach().cpu()
    for i, value in enumerate(means.tolist()):
        print(f"{id2label[i]:<12} {value: .4f}")

print("\nTraining label distribution:")
train_counts = Counter(
    label
    for chunk in train_chunks_nonoverlap
    for label in chunk["labels"]
)
for i in range(len(id2label)):
    print(f"{id2label[i]:<12} {train_counts[i]}")


In [ ]:

import torch
import torch.nn.functional as F

balanced_model.eval()

all_logits = []
all_gold = []

with torch.no_grad():
    for start in range(0, len(val_chunks_nonoverlap), 2):
        batch = val_chunks_nonoverlap[start:start + 2]

        ids = torch.nn.utils.rnn.pad_sequence(
            [torch.as_tensor(c["input_ids"], dtype=torch.long) for c in batch],
            batch_first=True,
            padding_value=0
        ).to(device)

        gold = torch.nn.utils.rnn.pad_sequence(
            [torch.as_tensor(c["labels"], dtype=torch.long) for c in batch],
            batch_first=True,
            padding_value=-100
        ).to(device)

        logits = balanced_model(ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        for i, chunk in enumerate(batch):
            n = len(chunk["labels"])
            all_logits.append(logits[i, :n].detach().cpu())
            all_gold.append(gold[i, :n].detach().cpu())

logits = torch.cat(all_logits)
gold = torch.cat(all_gold)

valid = gold != -100
logits = logits[valid]
gold = gold[valid]

pred = logits.argmax(dim=-1)
is_entity = gold != label2id["O"]

print("Valid tokens:", len(gold))
print("Token accuracy:", (pred == gold).float().mean().item())
print("Entity-token accuracy:",
      (pred[is_entity] == gold[is_entity]).float().mean().item())
print("O-token accuracy:",
      (pred[~is_entity] == gold[~is_entity]).float().mean().item())
print("Gold entity-token fraction:", is_entity.float().mean().item())
print("Predicted entity-token fraction:",
      (pred != label2id["O"]).float().mean().item())

# Compare the correct class logit with the O logit.
gold_logit = logits.gather(1, gold.unsqueeze(1)).squeeze(1)
o_logit = logits[:, label2id["O"]]

print("\nMean gold-label logit, gold O tokens:",
      gold_logit[~is_entity].mean().item())
print("Mean O logit, gold O tokens:",
      o_logit[~is_entity].mean().item())
print("Mean gold-label logit, gold entity tokens:",
      gold_logit[is_entity].mean().item())
print("Mean O logit, gold entity tokens:",
      o_logit[is_entity].mean().item())

print("\nActual loss weights used in the candidate:")
try:
    print("O weight:", new_weights[label2id["O"]].item())
    print("Min/max:", new_weights.min().item(), new_weights.max().item())
except Exception as e:
    print("Could not inspect new_weights:", repr(e))


In [ ]:

import torch

def validation_diagnostics(model, chunks):
    model.eval()
    total = 0
    correct = 0
    entity_total = 0
    entity_correct = 0
    predicted_entity = 0
    gold_entity = 0
    gold_entity_top2 = 0

    with torch.no_grad():
        for start in range(0, len(chunks), 2):
            batch = chunks[start:start + 2]
            ids = torch.nn.utils.rnn.pad_sequence(
                [torch.as_tensor(c["input_ids"], dtype=torch.long) for c in batch],
                batch_first=True,
                padding_value=0
            ).to(device)

            logits = model(ids)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            for i, chunk in enumerate(batch):
                n = len(chunk["labels"])
                gold = torch.as_tensor(chunk["labels"], dtype=torch.long)
                scores = logits[i, :n].detach().cpu()

                valid = gold != -100
                gold = gold[valid]
                scores = scores[valid]
                pred = scores.argmax(dim=-1)

                entity = gold != label2id["O"]
                total += len(gold)
                correct += (pred == gold).sum().item()
                entity_total += entity.sum().item()
                entity_correct += ((pred == gold) & entity).sum().item()
                predicted_entity += (pred != label2id["O"]).sum().item()
                gold_entity += entity.sum().item()

                top2 = scores.topk(2, dim=-1).indices
                gold_entity_top2 += (
                    (top2 == gold.unsqueeze(1)).any(dim=1) & entity
                ).sum().item()

    return {
        "token_accuracy": correct / max(total, 1),
        "entity_token_accuracy": entity_correct / max(entity_total, 1),
        "predicted_entity_fraction": predicted_entity / max(total, 1),
        "gold_entity_fraction": gold_entity / max(total, 1),
        "gold_entity_in_top2_fraction": gold_entity_top2 / max(entity_total, 1),
    }

# Original best checkpoint: use the original model architecture.
best_ckpt = torch.load(
    "/kaggle/working/a4_lorma_weighted_best.pt",
    map_location=device,
    weights_only=False
)
fresh_model.load_state_dict(best_ckpt["model_state_dict"], strict=True)

print("ORIGINAL BEST")
print(validation_diagnostics(fresh_model, val_chunks_nonoverlap))

print("\nBALANCED V2")
print(validation_diagnostics(balanced_model, val_chunks_nonoverlap))


In [ ]:

# Threshold-only experiment. No training and no checkpoint overwrite.
# Reuse the exact-span evaluation helpers already defined in the notebook.

best_ckpt = torch.load(
    "/kaggle/working/a4_lorma_weighted_best.pt",
    map_location=device,
    weights_only=False
)
fresh_model.load_state_dict(best_ckpt["model_state_dict"], strict=True)
fresh_model.eval()

print("Checkpoint epoch:", best_ckpt.get("epoch", "not recorded"))
print("Checkpoint validation loss:", best_ckpt.get("val_loss", "not recorded"))
print("Use the existing exact-span threshold sweep for this checkpoint.")
print("Previously best: p_min=0.40, margin_min=0.03, F1≈0.1021")


In [ ]:

# FINAL THRESHOLD SWEEP — ORIGINAL BEST CHECKPOINT
# No retraining. Does not overwrite the checkpoint or submission ZIP.

import torch
import numpy as np
from collections import defaultdict

CHECKPOINT = "/kaggle/working/a4_lorma_weighted_best.pt"

checkpoint = torch.load(
    CHECKPOINT,
    map_location=device,
    weights_only=False
)
fresh_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
fresh_model.eval()

print("Loaded:", CHECKPOINT)
print("Epoch:", checkpoint.get("epoch", "not recorded"))
print("Validation loss:", checkpoint.get("val_loss", "not recorded"))

# ---------------------------------------------------------
# 1. Gather token-level probabilities over validation chunks
# ---------------------------------------------------------
all_probs = []
all_offsets = []
all_texts = []
all_gold_labels = []

with torch.no_grad():
    for chunk in val_chunks_nonoverlap:
        ids = torch.as_tensor(
            chunk["input_ids"], dtype=torch.long, device=device
        ).unsqueeze(0)

        logits = fresh_model(ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        probs = torch.softmax(logits[0], dim=-1).cpu().numpy()
        n = len(chunk["input_ids"])

        all_probs.append(probs[:n])
        all_offsets.append(chunk.get("offsets", []))
        all_gold_labels.append(chunk["labels"])

print("Validation chunks processed:", len(all_probs))
print("IMPORTANT: probabilities are collected. Use your existing")
print("document-level overlap aggregation and exact-span decoder")
print("for the threshold sweep; chunk-level evaluation alone is not")
print("comparable to the previously reported exact-span F1.")


In [ ]:

import inspect

print("=== Available evaluation functions ===")

keywords = ("decode", "eval", "span", "threshold", "predict", "metric")

for name, obj in sorted(globals().items()):
    if (
        callable(obj)
        and any(k in name.lower() for k in keywords)
        and not name.startswith("_")
    ):
        try:
            signature = inspect.signature(obj)
        except (TypeError, ValueError):
            signature = "(signature unavailable)"

        print(f"\n{name}{signature}")

        if inspect.isfunction(obj) and name in {
            "decode_eval",
            "evaluate_spans",
            "evaluate_predictions",
            "threshold_sweep",
        }:
            try:
                print(inspect.getsource(obj))
            except (OSError, TypeError):
                print("Source unavailable")


In [ ]:

# FINAL EXACT-SPAN THRESHOLD SWEEP
# Uses the original best checkpoint; does not overwrite it.

import torch
import numpy as np
import inspect

CHECKPOINT = "/kaggle/working/a4_lorma_weighted_best.pt"

checkpoint = torch.load(
    CHECKPOINT, map_location=device, weights_only=False
)
fresh_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
fresh_model.eval()

print("Loaded original best checkpoint, epoch:", checkpoint.get("epoch", 3))
print("Existing prediction function:")
print(inspect.signature(predict_spans_for_doc))
print("Existing metrics function:")
print(inspect.signature(calc_metrics))

# Inspect the function's source to confirm its expected document format.
print("\n--- predict_spans_for_doc source ---")
print(inspect.getsource(predict_spans_for_doc))

print("\n--- calc_metrics source ---")
print(inspect.getsource(calc_metrics))


In [ ]:

# FINAL DOCUMENT-LEVEL EXACT-SPAN THRESHOLD SWEEP
# Preserves a4_lorma_weighted_best.pt and the existing submission.

import torch
import numpy as np
from collections import defaultdict

CHECKPOINT = "/kaggle/working/a4_lorma_weighted_best.pt"

# 1. Restore the known best model
ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
fresh_model.load_state_dict(ckpt["model_state_dict"], strict=True)
fresh_model.eval()

print("Loaded checkpoint:", CHECKPOINT)
print("Epoch:", ckpt.get("epoch", 3))

# 2. Build document-level averaged probabilities
# Use the validation documents and annotations already present in the notebook.
# This uses the same sliding-window strategy as inference.
val_docs = {}

# Discover the available validation data structure.
print("\nValidation-related variables:")
for name in ("val_texts", "val_golds", "val_gold", "val_data",
             "val_documents", "fold2_texts", "fold2_gold"):
    if name in globals():
        obj = globals()[name]
        try:
            print(name, type(obj).__name__, "length =", len(obj))
        except Exception:
            print(name, type(obj).__name__)

print("\nThe threshold sweep needs a mapping of document IDs to text and")
print("gold spans, plus document-level averaged probabilities.")
print("Use the existing evaluate_strict implementation as the reference.")


In [ ]:

# ============================================================
# FINAL EXACT-SPAN THRESHOLD SWEEP
# Original checkpoint | no retraining | no checkpoint overwrite
# ============================================================

import torch
import numpy as np
from collections import defaultdict

CHECKPOINT = "/kaggle/working/a4_lorma_weighted_best.pt"
MAX_TOKENS = 1024
STRIDE = 768

# Restore the original best model
ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
fresh_model.load_state_dict(ckpt["model_state_dict"], strict=True)
fresh_model.eval()

print("Checkpoint:", CHECKPOINT)
print("Epoch:", ckpt.get("epoch", 3))
print("Validation loss:", ckpt.get("val_loss", "unknown"))

# Get validation texts and gold spans from existing notebook variables.
# fold2_texts and fold2_gold are the expected source variables.
if "fold2_texts" not in globals():
    raise NameError(
        "fold2_texts is not defined. Define the Fold 2 text dictionary "
        "before running this cell."
    )
if "fold2_gold" not in globals():
    raise NameError("fold2_gold is not defined.")

def get_text(doc):
    if isinstance(doc, str):
        return doc
    if isinstance(doc, dict):
        for key in ("text", "content", "document"):
            if key in doc and isinstance(doc[key], str):
                return doc[key]
    raise ValueError(f"Cannot extract text from document: {type(doc)}")

def normalize_gold_doc(annotations):
    """Convert a document's gold annotations to (start, end, type) tuples."""
    if isinstance(annotations, dict):
        for key in ("spans", "annotations", "entities", "labels"):
            if key in annotations:
                annotations = annotations[key]
                break

    result = set()
    for item in annotations:
        if isinstance(item, (tuple, list)) and len(item) >= 3:
            result.add((int(item[0]), int(item[1]), str(item[2])))
        elif isinstance(item, dict):
            start = item.get("start", item.get("begin"))
            end = item.get("end", item.get("stop"))
            typ = item.get("type", item.get("label", item.get("entity")))
            if start is not None and end is not None and typ is not None:
                result.add((int(start), int(end), str(typ)))
            else:
                raise ValueError(f"Unknown gold annotation structure: {item}")
        else:
            raise ValueError(f"Unknown gold annotation format: {item}")
    return result

# Check data structure before evaluating.
print("Validation text container:", type(fold2_texts).__name__)
print("Gold container:", type(fold2_gold).__name__)
print("Example text entry:", repr(next(iter(fold2_texts.items()))[0:1]))
print("Example gold entry:", repr(next(iter(fold2_gold.items()))[0:1]))

# This code expects matching document IDs in the two dictionaries.
common_ids = [k for k in fold2_texts if k in fold2_gold]
if not common_ids:
    raise ValueError("No matching document IDs between fold2_texts and fold2_gold.")

print("Matching validation documents:", len(common_ids))

# Compute and average probabilities across overlapping windows.
doc_predictions = {}
gold_by_doc = {}

for doc_id in common_ids:
    text = get_text(fold2_texts[doc_id])
    encoded = tokenizer.encode(text)
    token_ids = encoded.ids
    offsets = encoded.offsets

    prob_sum = torch.zeros((len(token_ids), len(id2label)), dtype=torch.float32)
    counts = torch.zeros((len(token_ids), 1), dtype=torch.float32)

    with torch.no_grad():
        for start in range(0, len(token_ids), STRIDE):
            end = min(start + MAX_TOKENS, len(token_ids))
            x = torch.tensor(
                token_ids[start:end], dtype=torch.long, device=device
            ).unsqueeze(0)

            logits = fresh_model(x)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            probs = torch.softmax(logits[0], dim=-1).detach().cpu()
            prob_sum[start:end] += probs[:end-start]
            counts[start:end] += 1

            if end == len(token_ids):
                break

    averaged_probs = prob_sum / counts.clamp_min(1)
    doc_predictions[doc_id] = {
        "text": text,
        "offsets": offsets,
        "probs": averaged_probs,
    }
    gold_by_doc[doc_id] = normalize_gold_doc(fold2_gold[doc_id])

print("Document-level probabilities generated:", len(doc_predictions))

# Search thresholds and calculate exact-span micro F1.
p_values = np.round(np.arange(0.15, 0.651, 0.025), 3)
margin_values = np.round(np.arange(-0.10, 0.251, 0.025), 3)

results = []

for p_min in p_values:
    for margin_min in margin_values:
        predicted_all = set()
        expected_all = set()

        for doc_id, doc in doc_predictions.items():
            pred = predict_spans_for_doc(doc, float(p_min), float(margin_min))
            predicted_all.update(
                (doc_id, s, e, typ) for s, e, typ in pred
            )
            expected_all.update(
                (doc_id, s, e, typ) for s, e, typ in gold_by_doc[doc_id]
            )

        tp = len(predicted_all & expected_all)
        fp = len(predicted_all - expected_all)
        fn = len(expected_all - predicted_all)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        results.append({
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "p_min": float(p_min),
            "margin_min": float(margin_min),
        })

results.sort(key=lambda r: (r["f1"], r["precision"]), reverse=True)

print("\n=== TOP 15 EXACT-SPAN THRESHOLD SETTINGS ===")
print(" F1      Prec.   Recall  TP    FP    FN    p_min  margin")
for r in results[:15]:
    print(
        f'{r["f1"]:.4f}  {r["precision"]:.4f}  {r["recall"]:.4f}  '
        f'{r["tp"]:4d}  {r["fp"]:4d}  {r["fn"]:4d}  '
        f'{r["p_min"]:.3f}  {r["margin_min"]:.3f}'
    )

best = results[0]
print("\n=== BEST FOUND ===")
for key, value in best.items():
    print(f"{key}: {value}")

print("\nCheckpoint was not modified.")


In [ ]:

import json
from pathlib import Path

FOLD2_TEXT_PATH = "/kaggle/input/datasets/aq1de3/fold2-tex/fold2-text.json"

with open(FOLD2_TEXT_PATH, "r", encoding="utf-8") as f:
    fold2_texts = json.load(f)

print("Loaded validation texts:", type(fold2_texts).__name__)
print("Number of text records:", len(fold2_texts))
print("Number of gold records:", len(fold2_gold))

print("\nFirst text record:")
print(repr(next(iter(fold2_texts.items())) if isinstance(fold2_texts, dict)
            else fold2_texts[0])[:1500])

print("\nFirst gold record:")
print(repr(next(iter(fold2_gold.items())) if isinstance(fold2_gold, dict)
            else fold2_gold[0])[:1500])


In [ ]:

# ============================================================
# EXACT-SPAN THRESHOLD SWEEP — ORIGINAL BEST CHECKPOINT
# ============================================================

import torch
import numpy as np

CHECKPOINT = "/kaggle/working/a4_lorma_weighted_best.pt"
MAX_TOKENS = 1024
STRIDE = 768

# Restore the original best model
ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
fresh_model.load_state_dict(ckpt["model_state_dict"], strict=True)
fresh_model.eval()

print("Checkpoint epoch:", ckpt.get("epoch", 3))
print("Validation loss:", ckpt.get("val_loss", "unknown"))

# Build document-level probability arrays
doc_predictions = {}
gold_by_doc = {}

for doc_id, text in fold2_texts.items():
    encoded = tokenizer.encode(text)
    token_ids, offsets = encoded.ids, encoded.offsets

    prob_sum = torch.zeros((len(token_ids), len(id2label)))
    counts = torch.zeros((len(token_ids), 1))

    with torch.no_grad():
        for start in range(0, len(token_ids), STRIDE):
            end = min(start + MAX_TOKENS, len(token_ids))
            x = torch.tensor(
                token_ids[start:end], dtype=torch.long, device=device
            ).unsqueeze(0)

            logits = fresh_model(x)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            probs = torch.softmax(logits[0], dim=-1).detach().cpu()
            prob_sum[start:end] += probs[:end-start]
            counts[start:end] += 1

            if end == len(token_ids):
                break

    doc_predictions[doc_id] = {
        "text": text,
        "offsets": offsets,
        "probs": prob_sum / counts.clamp_min(1),
    }

    # Gold format: [start, end, type, entity_text]
    gold_by_doc[doc_id] = {
        (int(span[0]), int(span[1]), str(span[2]))
        for span in fold2_gold[doc_id]
    }

print("Documents processed:", len(doc_predictions))

# Sweep thresholds using the existing exact-span decoder
p_values = np.round(np.arange(0.15, 0.651, 0.025), 3)
margin_values = np.round(np.arange(-0.10, 0.251, 0.025), 3)
results = []

for p_min in p_values:
    for margin_min in margin_values:
        tp = fp = fn = 0

        for doc_id, doc in doc_predictions.items():
            predicted = predict_spans_for_doc(
                doc, float(p_min), float(margin_min)
            )
            expected = gold_by_doc[doc_id]

            tp += len(predicted & expected)
            fp += len(predicted - expected)
            fn += len(expected - predicted)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        results.append({
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "p_min": float(p_min),
            "margin_min": float(margin_min),
        })

results.sort(key=lambda r: (r["f1"], r["precision"]), reverse=True)

print("\nTOP 15 THRESHOLD SETTINGS")
print("F1      Precision Recall   TP    FP    FN    p_min margin")
for r in results[:15]:
    print(
        f'{r["f1"]:.4f}  {r["precision"]:.4f}    {r["recall"]:.4f}  '
        f'{r["tp"]:4d}  {r["fp"]:4d}  {r["fn"]:4d}  '
        f'{r["p_min"]:.3f}  {r["margin_min"]:.3f}'
    )

best = results[0]
print("\nBEST THRESHOLDS")
for key, value in best.items():
    print(f"{key}: {value}")

print("\nCheckpoint remains unchanged.")


In [ ]:

# Fine-grained threshold sweep around the current best.
# Uses cached document probabilities; no GPU retraining.

p_values = np.round(np.arange(0.30, 0.401, 0.01), 3)
margin_values = np.round(np.arange(0.00, 0.101, 0.01), 3)

fine_results = []

for p_min in p_values:
    for margin_min in margin_values:
        tp = fp = fn = 0

        for doc_id, doc in doc_predictions.items():
            predicted = predict_spans_for_doc(
                doc, float(p_min), float(margin_min)
            )
            expected = gold_by_doc[doc_id]

            tp += len(predicted & expected)
            fp += len(predicted - expected)
            fn += len(expected - predicted)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        fine_results.append({
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "tp": tp, "fp": fp, "fn": fn,
            "p_min": float(p_min),
            "margin_min": float(margin_min),
        })

fine_results.sort(key=lambda r: r["f1"], reverse=True)

print("TOP 10 FINE-GRAINED SETTINGS")
for r in fine_results[:10]:
    print(
        f'F1={r["f1"]:.4f} P={r["precision"]:.4f} '
        f'R={r["recall"]:.4f} TP={r["tp"]} FP={r["fp"]} FN={r["fn"]} '
        f'p_min={r["p_min"]:.2f} margin_min={r["margin_min"]:.2f}'
    )

print("\nBest fine-grained setting:", fine_results[0])


In [ ]:

import json
from pathlib import Path

best_config = {
    "checkpoint": "/kaggle/working/a4_lorma_weighted_best.pt",
    "epoch": 3,
    "p_min": 0.35,
    "margin_min": 0.05,
    "validation_fold": 2,
    "validation_precision": 0.2597701149425287,
    "validation_recall": 0.10578048209688744,
    "validation_f1": 0.1503409279893564,
    "validation_tp": 452,
    "validation_fp": 1288,
    "validation_fn": 3821,
}

config_path = Path("/kaggle/working/best_threshold_config.json")
config_path.write_text(json.dumps(best_config, indent=2), encoding="utf-8")

print("Saved:", config_path)
print(config_path.read_text())


In [ ]:

# Per-entity exact-span evaluation using the best thresholds.
# Uses cached validation probabilities; no retraining.

from collections import defaultdict

p_min = 0.35
margin_min = 0.05

entity_stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

for doc_id, doc in doc_predictions.items():
    predicted = predict_spans_for_doc(doc, p_min, margin_min)
    expected = gold_by_doc[doc_id]

    for entity_type in id2label.values():
        if entity_type == "O":
            continue

        pred_type = {s for s in predicted if s[2] == entity_type}
        gold_type = {s for s in expected if s[2] == entity_type}

        entity_stats[entity_type]["tp"] += len(pred_type & gold_type)
        entity_stats[entity_type]["fp"] += len(pred_type - gold_type)
        entity_stats[entity_type]["fn"] += len(gold_type - pred_type)

print(f"{'Entity':<12} {'TP':>6} {'FP':>6} {'FN':>6} {'Precision':>11} {'Recall':>10} {'F1':>8}")
for entity_type in sorted(entity_stats):
    s = entity_stats[entity_type]
    tp, fp, fn = s["tp"], s["fp"], s["fn"]
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    print(
        f"{entity_type:<12} {tp:>6} {fp:>6} {fn:>6} "
        f"{precision:>10.3f} {recall:>10.3f} {f1:>8.3f}"
    )


In [ ]:

# Correct per-ENTITY evaluation.
# Span types are APP, COURT, DATE, etc. — not B-APP/I-APP.

from collections import defaultdict

p_min = 0.35
margin_min = 0.05

entity_stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

for doc_id, doc in doc_predictions.items():
    predicted = predict_spans_for_doc(doc, p_min, margin_min)
    expected = gold_by_doc[doc_id]

    # Entity types present in decoded spans, not BIO label IDs.
    all_types = {span[2] for span in predicted}
    all_types |= {span[2] for span in expected}

    for entity_type in all_types:
        pred_type = {s for s in predicted if s[2] == entity_type}
        gold_type = {s for s in expected if s[2] == entity_type}

        entity_stats[entity_type]["tp"] += len(pred_type & gold_type)
        entity_stats[entity_type]["fp"] += len(pred_type - gold_type)
        entity_stats[entity_type]["fn"] += len(gold_type - pred_type)

print(
    f"{'Entity':<12} {'TP':>6} {'FP':>6} {'FN':>6} "
    f"{'Precision':>11} {'Recall':>10} {'F1':>8}"
)

for entity_type in sorted(entity_stats):
    s = entity_stats[entity_type]
    tp, fp, fn = s["tp"], s["fp"], s["fn"]
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    print(
        f"{entity_type:<12} {tp:>6} {fp:>6} {fn:>6} "
        f"{precision:>10.3f} {recall:>10.3f} {f1:>8.3f}"
    )


In [ ]:

# Diagnose whether entities disappear during threshold filtering/decoding.
# No training and no checkpoint changes.

from collections import Counter

raw_counts = Counter()
filtered_counts = Counter()
gold_counts = Counter()

for doc_id, doc in doc_predictions.items():
    probs = doc["probs"]

    # Raw argmax: no probability or margin filtering
    raw_ids = probs.argmax(dim=-1).tolist()
    raw_spans = decode_eval(
        raw_ids, doc["offsets"], doc["text"], ignore_orphan_i=True
    )

    # Winning thresholds
    filtered_spans = predict_spans_for_doc(
        doc, p_min=0.35, margin_min=0.05
    )

    for _, _, entity_type in raw_spans:
        raw_counts[entity_type] += 1

    for _, _, entity_type in filtered_spans:
        filtered_counts[entity_type] += 1

    for _, _, entity_type in gold_by_doc[doc_id]:
        gold_counts[entity_type] += 1

print(
    f"{'Entity':<12}{'Gold':>8}{'Raw spans':>12}"
    f"{'Filtered':>12}{'Raw exact TP':>14}"
)

for entity_type in sorted(gold_counts):
    raw_tp = 0
    filtered_tp = 0

    for doc_id, doc in doc_predictions.items():
        raw_ids = doc["probs"].argmax(dim=-1).tolist()
        raw_spans = decode_eval(
            raw_ids, doc["offsets"], doc["text"], ignore_orphan_i=True
        )
        filtered_spans = predict_spans_for_doc(
            doc, 0.35, 0.05
        )
        expected = gold_by_doc[doc_id]

        raw_tp += sum(
            1 for span in raw_spans
            if span[2] == entity_type and span in expected
        )
        filtered_tp += sum(
            1 for span in filtered_spans
            if span[2] == entity_type and span in expected
        )

    print(
        f"{entity_type:<12}{gold_counts[entity_type]:>8}"
        f"{raw_counts[entity_type]:>12}"
        f"{filtered_counts[entity_type]:>12}{raw_tp:>14}"
    )


In [ ]:

# Verify overall TP/FP/FN using the current winning thresholds.
# Does not train or modify the checkpoint.

p_min = 0.35
margin_min = 0.05

all_predicted = set()
all_expected = set()

for doc_id, doc in doc_predictions.items():
    predicted = predict_spans_for_doc(doc, p_min, margin_min)
    expected = gold_by_doc[doc_id]

    all_predicted.update((doc_id, s, e, typ) for s, e, typ in predicted)
    all_expected.update((doc_id, s, e, typ) for s, e, typ in expected)

tp = len(all_predicted & all_expected)
fp = len(all_predicted - all_expected)
fn = len(all_expected - all_predicted)

precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Predicted spans:", len(all_predicted))
print("Gold spans:", len(all_expected))


In [ ]:

# Per-class threshold sweep — inference only.
# Baseline: p_min=0.35, margin_min=0.05
# Tests each entity class independently against the baseline.

import numpy as np

BASE_P = 0.35
BASE_M = 0.05

TARGET_TYPES = [
    "COURT", "STAT", "AUTH", "DATE", "PREC",
    "CASENO", "APP", "A.COUNSEL", "JUDGE",
    "R.COUNSEL", "RESP", "WIT"
]

# Get filtered spans for each document with configurable thresholds per type.
def predict_class_thresholds(doc, p_by_type, m_by_type):
    probs = doc["probs"]
    pred_ids = probs.argmax(dim=-1).tolist()
    filtered_ids = []

    for i, pred_id in enumerate(pred_ids):
        if pred_id == label2id["O"]:
            filtered_ids.append(pred_id)
            continue

        tag = id2label[pred_id]
        entity_type = tag.split("-", 1)[1]
        entity_prob = float(probs[i, pred_id])
        o_prob = float(probs[i, label2id["O"]])

        p = p_by_type.get(entity_type, BASE_P)
        m = m_by_type.get(entity_type, BASE_M)

        if entity_prob < p or entity_prob - o_prob < m:
            filtered_ids.append(label2id["O"])
        else:
            filtered_ids.append(pred_id)

    return decode_eval(
        filtered_ids, doc["offsets"], doc["text"], ignore_orphan_i=True
    )

def score_predictions(pred_by_doc):
    tp = fp = fn = 0
    for doc_id, predicted in pred_by_doc.items():
        expected = gold_by_doc[doc_id]
        tp += len(predicted & expected)
        fp += len(predicted - expected)
        fn += len(expected - predicted)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return tp, fp, fn, precision, recall, f1

baseline_p = {t: BASE_P for t in TARGET_TYPES}
baseline_m = {t: BASE_M for t in TARGET_TYPES}

baseline_predictions = {
    doc_id: predict_class_thresholds(doc, baseline_p, baseline_m)
    for doc_id, doc in doc_predictions.items()
}
baseline_score = score_predictions(baseline_predictions)

print("Baseline:", baseline_score)

p_grid = np.round(np.arange(0.15, 0.601, 0.05), 2)
m_grid = np.round(np.arange(-0.05, 0.151, 0.05), 2)

class_results = []

for entity_type in TARGET_TYPES:
    best = None

    for p in p_grid:
        for m in m_grid:
            p_cfg = dict(baseline_p)
            m_cfg = dict(baseline_m)
            p_cfg[entity_type] = float(p)
            m_cfg[entity_type] = float(m)

            predictions = {
                doc_id: predict_class_thresholds(doc, p_cfg, m_cfg)
                for doc_id, doc in doc_predictions.items()
            }

            score = score_predictions(predictions)
            row = {
                "entity": entity_type,
                "p_min": float(p),
                "margin_min": float(m),
                "tp": score[0],
                "fp": score[1],
                "fn": score[2],
                "precision": score[3],
                "recall": score[4],
                "f1": score[5],
            }

            if best is None or row["f1"] > best["f1"]:
                best = row

    class_results.append(best)

print("\nBest single-class threshold change for each entity:")
print("Entity       p_min margin   TP    FP    FN  Precision Recall     F1")
for r in class_results:
    print(
        f'{r["entity"]:<12} {r["p_min"]:>5.2f} {r["margin_min"]:>6.2f} '
        f'{r["tp"]:>5} {r["fp"]:>5} {r["fn"]:>5} '
        f'{r["precision"]:>9.3f} {r["recall"]:>6.3f} {r["f1"]:>7.4f}'
    )


In [ ]:

# Compare baseline vs DATE-specific threshold.
# DATE: p_min=0.60, margin_min=-0.05
# All other entity types: p_min=0.35, margin_min=0.05

baseline_p = {t: 0.35 for t in TARGET_TYPES}
baseline_m = {t: 0.05 for t in TARGET_TYPES}

date_p = dict(baseline_p)
date_m = dict(baseline_m)
date_p["DATE"] = 0.60
date_m["DATE"] = -0.05

def evaluate_config(p_cfg, m_cfg):
    predictions = {
        doc_id: predict_class_thresholds(doc, p_cfg, m_cfg)
        for doc_id, doc in doc_predictions.items()
    }
    return score_predictions(predictions)

baseline = evaluate_config(baseline_p, baseline_m)
date_tuned = evaluate_config(date_p, date_m)

def show_score(name, score):
    tp, fp, fn, precision, recall, f1 = score
    print(
        f"{name}: TP={tp}, FP={fp}, FN={fn}, "
        f"Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}"
    )

show_score("Baseline", baseline)
show_score("DATE tuned", date_tuned)

if date_tuned[-1] > baseline[-1]:
    print("\nImprovement confirmed on this validation fold.")
    print("Keep DATE p_min=0.60 and margin_min=-0.05 for further testing.")
else:
    print("\nNo improvement from this combined configuration.")
    print("Keep the baseline thresholds for now.")


In [ ]:

import json
from pathlib import Path

best_config = {
    "checkpoint": "/kaggle/working/a4_lorma_weighted_best.pt",
    "epoch": 3,
    "validation_fold": 2,
    "default_p_min": 0.35,
    "default_margin_min": 0.05,
    "class_thresholds": {
        "DATE": {
            "p_min": 0.60,
            "margin_min": -0.05
        }
    },
    "validation_precision": 0.4105,
    "validation_recall": 0.1058,
    "validation_f1": 0.1682,
    "validation_tp": 452,
    "validation_fp": 649,
    "validation_fn": 3821
}

path = Path("/kaggle/working/best_threshold_config.json")
path.write_text(json.dumps(best_config, indent=2), encoding="utf-8")
print("Saved:", path)
print(path.read_text())


In [ ]:

# Tune STAT while keeping DATE's confirmed thresholds fixed.
# No retraining; uses cached document probabilities.

import numpy as np

stat_results = []
p_values = np.round(np.arange(0.15, 0.451, 0.05), 2)
margin_values = np.round(np.arange(-0.10, 0.101, 0.05), 2)

for stat_p in p_values:
    for stat_m in margin_values:
        p_cfg = dict(date_p)
        m_cfg = dict(date_m)

        p_cfg["STAT"] = float(stat_p)
        m_cfg["STAT"] = float(stat_m)

        score = evaluate_config(p_cfg, m_cfg)

        stat_results.append({
            "p_min": float(stat_p),
            "margin_min": float(stat_m),
            "tp": score[0],
            "fp": score[1],
            "fn": score[2],
            "precision": score[3],
            "recall": score[4],
            "f1": score[5],
        })

stat_results.sort(key=lambda x: x["f1"], reverse=True)

print("Current best with DATE tuned:", date_tuned[-1])
print("\nTOP 10 STAT SETTINGS (DATE FIXED)")
for r in stat_results[:10]:
    print(
        f'F1={r["f1"]:.4f} P={r["precision"]:.4f} '
        f'R={r["recall"]:.4f} TP={r["tp"]} FP={r["fp"]} '
        f'FN={r["fn"]} STAT_p={r["p_min"]:.2f} '
        f'STAT_margin={r["margin_min"]:.2f}'
    )


In [ ]:

import json
from pathlib import Path

best_config = {
    "checkpoint": "/kaggle/working/a4_lorma_weighted_best.pt",
    "epoch": 3,
    "validation_fold": 2,
    "default_p_min": 0.35,
    "default_margin_min": 0.05,
    "class_thresholds": {
        "DATE": {
            "p_min": 0.60,
            "margin_min": -0.05
        },
        "STAT": {
            "p_min": 0.25,
            "margin_min": -0.05
        }
    },
    "validation_precision": 0.3409,
    "validation_recall": 0.1128,
    "validation_f1": 0.1695,
    "validation_tp": 482,
    "validation_fp": 932,
    "validation_fn": 3791
}

path = Path("/kaggle/working/best_threshold_config.json")
path.write_text(json.dumps(best_config, indent=2), encoding="utf-8")

print("Saved:", path)
print(path.read_text())


In [ ]:

# Tune AUTH with DATE and STAT settings preserved.
# No retraining; uses cached validation probabilities.

auth_results = []

p_values = np.round(np.arange(0.15, 0.601, 0.05), 2)
margin_values = np.round(np.arange(-0.10, 0.151, 0.05), 2)

for auth_p in p_values:
    for auth_m in margin_values:
        p_cfg = dict(date_p)
        m_cfg = dict(date_m)

        # Preserve the current winning DATE and STAT settings.
        p_cfg["DATE"] = 0.60
        m_cfg["DATE"] = -0.05
        p_cfg["STAT"] = 0.25
        m_cfg["STAT"] = -0.05

        # Change AUTH only.
        p_cfg["AUTH"] = float(auth_p)
        m_cfg["AUTH"] = float(auth_m)

        score = evaluate_config(p_cfg, m_cfg)

        auth_results.append({
            "p_min": float(auth_p),
            "margin_min": float(auth_m),
            "tp": score[0],
            "fp": score[1],
            "fn": score[2],
            "precision": score[3],
            "recall": score[4],
            "f1": score[5],
        })

auth_results.sort(key=lambda x: x["f1"], reverse=True)

print("Current best F1: 0.1695")
print("\nTOP 10 AUTH SETTINGS")
for r in auth_results[:10]:
    print(
        f'F1={r["f1"]:.4f} P={r["precision"]:.4f} '
        f'R={r["recall"]:.4f} TP={r["tp"]} FP={r["fp"]} '
        f'FN={r["fn"]} AUTH_p={r["p_min"]:.2f} '
        f'AUTH_margin={r["margin_min"]:.2f}'
    )


In [ ]:

# Experiment 1: BIO-label diagnostic
# No training. No checkpoint changes.

import torch
from collections import defaultdict

def bio_diagnostic(model, chunks, batch_size=2):
    model.eval()
    counts = defaultdict(lambda: [0, 0])  # correct, total

    with torch.no_grad():
        for start in range(0, len(chunks), batch_size):
            batch = chunks[start:start + batch_size]

            ids = torch.nn.utils.rnn.pad_sequence(
                [
                    torch.as_tensor(c["input_ids"], dtype=torch.long)
                    for c in batch
                ],
                batch_first=True,
                padding_value=0
            ).to(device)

            gold = torch.nn.utils.rnn.pad_sequence(
                [
                    torch.as_tensor(c["labels"], dtype=torch.long)
                    for c in batch
                ],
                batch_first=True,
                padding_value=-100
            ).to(device)

            logits = model(ids)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            pred = logits.argmax(dim=-1)

            for i, chunk in enumerate(batch):
                n = len(chunk["labels"])
                y = gold[i, :n]
                p = pred[i, :n]
                valid = y != -100

                for label_id in range(len(id2label)):
                    mask = valid & (y == label_id)
                    counts[label_id][1] += mask.sum().item()
                    counts[label_id][0] += ((p == y) & mask).sum().item()

    print(f"{'Label':<12} {'Total':>8} {'Correct':>8} {'Accuracy':>10}")
    for label_id in range(len(id2label)):
        correct, total = counts[label_id]
        accuracy = correct / total if total else 0.0
        print(
            f"{id2label[label_id]:<12} {total:>8} "
            f"{correct:>8} {accuracy:>9.3%}"
        )

# Restore the original best checkpoint before evaluating.
checkpoint = torch.load(
    "/kaggle/working/a4_lorma_weighted_best.pt",
    map_location=device,
    weights_only=False
)
fresh_model.load_state_dict(checkpoint["model_state_dict"], strict=True)

print("=== TRAINING CHUNKS ===")
bio_diagnostic(fresh_model, train_chunks_nonoverlap)

print("\n=== VALIDATION CHUNKS ===")
bio_diagnostic(fresh_model, val_chunks_nonoverlap)


In [ ]:

import torch
from collections import Counter, defaultdict

fresh_model.eval()

gold_counts = Counter()
pred_counts = Counter()
confusions = defaultdict(Counter)

# Normalize the mapping so IDs always resolve to label names.
label_names = {
    int(k): str(v) for k, v in id2label.items()
}
# If id2label is name -> ID instead, reverse it.
if not any(v == "O" for v in label_names.values()):
    label_names = {int(v): str(k) for k, v in id2label.items()}

with torch.no_grad():
    for start in range(0, len(val_chunks_nonoverlap), 2):
        batch = val_chunks_nonoverlap[start:start + 2]

        ids = torch.nn.utils.rnn.pad_sequence(
            [
                torch.as_tensor(c["input_ids"], dtype=torch.long)
                for c in batch
            ],
            batch_first=True,
            padding_value=0
        ).to(device)

        gold = torch.nn.utils.rnn.pad_sequence(
            [
                torch.as_tensor(c["labels"], dtype=torch.long)
                for c in batch
            ],
            batch_first=True,
            padding_value=-100
        ).to(device)

        logits = fresh_model(ids)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        preds = logits.argmax(dim=-1)

        for i, chunk in enumerate(batch):
            n = len(chunk["labels"])
            y = gold[i, :n].cpu().tolist()
            p = preds[i, :n].cpu().tolist()

            for g, pr in zip(y, p):
                if g == -100:
                    continue
                gold_counts[g] += 1
                pred_counts[pr] += 1
                confusions[g][pr] += 1

print("=== PREDICTED COUNTS FOR ENTITY LABELS ===")
for label_id in sorted(label_names):
    label = label_names[label_id]
    if label != "O":
        print(
            f"{label:<12} "
            f"gold={gold_counts[label_id]:5d} "
            f"predicted={pred_counts[label_id]:5d}"
        )

print("\n=== TOP CONFUSIONS FOR DIFFICULT LABELS ===")
difficult_labels = [
    "B-APP", "B-RESP", "B-A.COUNSEL", "B-R.COUNSEL",
    "B-WIT", "B-PREC", "B-CASENO", "B-DATE"
]

name_to_id = {name: idx for idx, name in label_names.items()}

for label in difficult_labels:
    g = name_to_id.get(label)
    if g is None:
        print(f"\n{label}: label not found in mapping")
        continue

    total = gold_counts[g]
    print(f"\n{label} (gold tokens={total})")

    for predicted_id, count in confusions[g].most_common(4):
        predicted_name = label_names.get(
            predicted_id, f"UNKNOWN_{predicted_id}"
        )
        print(
            f"  predicted {predicted_name:<12}: "
            f"{count:5d} ({100 * count / total:.1f}%)"
            if total else
            f"  predicted {predicted_name:<12}: {count:5d}"
        )


In [ ]:

import copy
import torch
import torch.nn as nn

# Preserve the existing best checkpoint.
BEST_PATH = "/kaggle/working/a4_lorma_weighted_best.pt"
CANDIDATE_PATH = "/kaggle/working/a4_lorma_boundary_candidate.pt"

checkpoint = torch.load(
    BEST_PATH, map_location=device, weights_only=False
)
fresh_model.load_state_dict(
    checkpoint["model_state_dict"], strict=True
)

# Start from the original loss weights used in the successful baseline.
# `weights` is the class-weight tensor from the earlier training experiment.
experiment_weights = weights.detach().clone().float().cpu()

# Moderately emphasize entity beginnings only.
for label_id, label in enumerate(id2label):
    if str(label).startswith("B-"):
        experiment_weights[label_id] *= 1.5

experiment_weights = experiment_weights.to(device)
criterion_candidate = nn.CrossEntropyLoss(
    weight=experiment_weights,
    ignore_index=-100
)

optimizer_candidate = torch.optim.AdamW(
    [p for p in fresh_model.parameters() if p.requires_grad],
    lr=2e-5,
    weight_decay=0.01
)

def run_epoch(chunks, training):
    fresh_model.train(training)
    total_loss = 0.0
    total_tokens = 0

    for start in range(0, len(chunks), 2):
        batch = chunks[start:start + 2]

        input_ids = torch.nn.utils.rnn.pad_sequence(
            [
                torch.as_tensor(c["input_ids"], dtype=torch.long)
                for c in batch
            ],
            batch_first=True,
            padding_value=0
        ).to(device)

        labels = torch.nn.utils.rnn.pad_sequence(
            [
                torch.as_tensor(c["labels"], dtype=torch.long)
                for c in batch
            ],
            batch_first=True,
            padding_value=-100
        ).to(device)

        with torch.set_grad_enabled(training):
            logits = fresh_model(input_ids)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            loss = criterion_candidate(
                logits.reshape(-1, logits.shape[-1]),
                labels.reshape(-1)
            )

            if training:
                optimizer_candidate.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    fresh_model.parameters(), 1.0
                )
                optimizer_candidate.step()

        valid_tokens = (labels != -100).sum().item()
        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)

for epoch in range(1, 3):
    train_loss = run_epoch(train_chunks_nonoverlap, True)
    val_loss = run_epoch(val_chunks_nonoverlap, False)

    print(
        f"Epoch {epoch}/2 | "
        f"train loss={train_loss:.4f} | "
        f"validation loss={val_loss:.4f}"
    )

torch.save(
    {
        "model_state_dict": fresh_model.state_dict(),
        "experiment": "B-label weight multiplier 1.5",
        "base_checkpoint": BEST_PATH,
        "epochs": 2,
    },
    CANDIDATE_PATH
)

print("\nCandidate saved:", CANDIDATE_PATH)
print("Original best checkpoint remains unchanged.")


In [ ]:

import torch

BEST_PATH = "/kaggle/working/a4_lorma_weighted_best.pt"
CANDIDATE_PATH = "/kaggle/working/a4_lorma_boundary_candidate.pt"

def load_candidate(path):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    fresh_model.load_state_dict(ckpt["model_state_dict"], strict=True)
    fresh_model.eval()

def get_current_metrics():
    # Use the existing per-class threshold configuration.
    predictions = score_predictions({
        doc_id: predict_class_thresholds(
            doc_data,
            {
                "DATE": 0.60,
                "STAT": 0.25
            },
            {
                "DATE": -0.05,
                "STAT": -0.05
            }
        )
        for doc_id, doc_data in doc_predictions.items()
    })
    return predictions

# IMPORTANT:
# This comparison is valid only if predict_class_thresholds and
# score_predictions use the same document-level probability pipeline
# for each checkpoint. If predict_spans_for_doc reads cached logits,
# those logits must be regenerated for each checkpoint first.

print("Candidate checkpoint is saved safely.")
print("Before scoring, we must regenerate validation logits per checkpoint.")
print("Do not compare cached baseline logits with candidate model weights.")


In [ ]:

required_names = [
    "fresh_model",
    "device",
    "val_chunks_nonoverlap",
    "fold2_texts",
    "fold2_gold",
    "doc_predictions",
    "predict_doc",
    "decode_eval",
    "calc_metrics",
    "id2label",
    "label2id",
]

for name in required_names:
    print(f"{name}: {'available' if name in globals() else 'MISSING'}")

print("\nPrediction function:")
if "predict_doc" in globals():
    import inspect
    print(inspect.signature(predict_doc))
    print(inspect.getsource(predict_doc))

print("\nScoring functions:")
for name in ["predict_class_thresholds", "score_predictions"]:
    if name in globals():
        import inspect
        print(f"\n{name}{inspect.signature(globals()[name])}")


In [ ]:

import torch

BEST_PATH = "/kaggle/working/a4_lorma_weighted_best.pt"
CANDIDATE_PATH = "/kaggle/working/a4_lorma_boundary_candidate.pt"

def load_model_checkpoint(path):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    fresh_model.load_state_dict(ckpt["model_state_dict"], strict=True)
    fresh_model.eval()

def predict_doc_probabilities(text):
    """Return token offsets and averaged probabilities for one document."""
    encoded = tokenizer.encode(text)
    ids = encoded.ids
    offsets = encoded.offsets

    if len(ids) == 0:
        return offsets, torch.empty((0, 25))

    logit_sum = torch.zeros(len(ids), 25)
    counts = torch.zeros(len(ids), 1)

    model_device = next(fresh_model.parameters()).device

    with torch.no_grad():
        for start in range(0, len(ids), 768):
            end = min(start + 1024, len(ids))
            x = torch.tensor(
                ids[start:end], dtype=torch.long, device=model_device
            ).unsqueeze(0)

            logits = fresh_model(x)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            logits = logits[0].detach().float().cpu()
            logit_sum[start:end] += logits
            counts[start:end] += 1

            if end == len(ids):
                break

    avg_logits = logit_sum / counts.clamp_min(1)
    probs = torch.softmax(avg_logits, dim=-1)
    return offsets, probs

def regenerate_validation_predictions(checkpoint_path):
    load_model_checkpoint(checkpoint_path)
    result = {}

    for doc_id, text in fold2_texts.items():
        offsets, probs = predict_doc_probabilities(text)
        result[doc_id] = {
            "text": text,
            "offsets": offsets,
            "probs": probs
        }

    return result

print("Regenerating baseline validation predictions...")
baseline_val_probs = regenerate_validation_predictions(BEST_PATH)

print("Regenerating candidate validation predictions...")
candidate_val_probs = regenerate_validation_predictions(CANDIDATE_PATH)

print("Documents:", len(baseline_val_probs), len(candidate_val_probs))
print("Baseline token probabilities:", sum(
    len(x["probs"]) for x in baseline_val_probs.values()
))
print("Candidate token probabilities:", sum(
    len(x["probs"]) for x in candidate_val_probs.values()
))
print("Both checkpoints remain unchanged on disk.")


In [ ]:

def score_probability_cache_fixed(prob_cache):
    total_tp = total_fp = total_fn = 0

    for doc_id, item in prob_cache.items():
        text = item["text"]
        offsets = item["offsets"]
        probs = item["probs"]

        p_cfg = {"DATE": 0.60, "STAT": 0.25}
        m_cfg = {"DATE": -0.05, "STAT": -0.05}

        filtered_ids = []
        for i in range(len(offsets)):
            best_id = int(probs[i].argmax())
            label = id2label[best_id]

            if not isinstance(label, str):
                label = id2label.get(best_id, best_id)

            if label == "O":
                filtered_ids.append(best_id)
                continue

            typ = str(label).split("-", 1)[-1]
            p_min = p_cfg.get(typ, 0.35)
            margin_min = m_cfg.get(typ, 0.05)

            p_entity = float(probs[i, best_id])
            p_o = float(probs[i, label2id["O"]])

            if p_entity >= p_min and p_entity - p_o >= margin_min:
                filtered_ids.append(best_id)
            else:
                filtered_ids.append(label2id["O"])

        predicted_spans = decode_eval(
            filtered_ids, offsets, text, ignore_orphan_i=True
        )

        # Gold spans are document-specific.
        gold_spans = {
            (int(s), int(e), str(t))
            for s, e, t, *_ in fold2_gold[doc_id]
        }

        tp = len(predicted_spans & gold_spans)
        fp = len(predicted_spans - gold_spans)
        fn = len(gold_spans - predicted_spans)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / max(total_tp + total_fp, 1)
    recall = total_tp / max(total_tp + total_fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
        "precision": precision,
        "recall": recall,
        "F1": f1,
    }

print("Scoring baseline...")
baseline_metrics = score_probability_cache_fixed(baseline_val_probs)
print("Baseline:", baseline_metrics)

print("\nScoring candidate...")
candidate_metrics = score_probability_cache_fixed(candidate_val_probs)
print("Candidate:", candidate_metrics)


In [ ]:

import os
import json
import shutil

candidate_path = "/kaggle/working/a4_lorma_boundary_candidate.pt"
new_best_path = "/kaggle/working/a4_lorma_boundary_best.pt"
config_path = "/kaggle/working/best_threshold_config_v2.json"

assert os.path.exists(candidate_path), "Candidate checkpoint not found."

shutil.copy2(candidate_path, new_best_path)

config = {
    "checkpoint": new_best_path,
    "experiment": "B-label weight multiplier 1.5; 2 fine-tuning epochs",
    "validation_fold": 2,
    "default_p_min": 0.35,
    "default_margin_min": 0.05,
    "class_thresholds": {
        "DATE": {"p_min": 0.60, "margin_min": -0.05},
        "STAT": {"p_min": 0.25, "margin_min": -0.05}
    },
    "validation_precision": candidate_metrics["precision"],
    "validation_recall": candidate_metrics["recall"],
    "validation_f1": candidate_metrics["F1"],
    "validation_tp": candidate_metrics["TP"],
    "validation_fp": candidate_metrics["FP"],
    "validation_fn": candidate_metrics["FN"]
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("New best checkpoint:", new_best_path)
print("Threshold configuration:", config_path)
print("Validation F1:", round(config["validation_f1"], 6))
print("Original baseline preserved.")


In [ ]:

import torch
import os

BEST_PATH = "/kaggle/working/a4_lorma_boundary_best.pt"
NEXT_CANDIDATE = "/kaggle/working/a4_lorma_boundary_epoch4_candidate.pt"

assert os.path.exists(BEST_PATH), "Best checkpoint not found."
assert "experiment_weights" in globals(), "Loss weights unavailable."
assert "fresh_model" in globals(), "Model unavailable."

# Start from the best saved model, not the previous in-memory candidate.
ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
fresh_model.load_state_dict(ckpt["model_state_dict"], strict=True)

criterion_candidate = torch.nn.CrossEntropyLoss(
    weight=experiment_weights.to(device),
    ignore_index=-100
)

optimizer_candidate = torch.optim.AdamW(
    [p for p in fresh_model.parameters() if p.requires_grad],
    lr=2e-5,
    weight_decay=0.01
)

train_loss = run_epoch(train_chunks_nonoverlap, True)
val_loss = run_epoch(val_chunks_nonoverlap, False)

torch.save(
    {
        "model_state_dict": fresh_model.state_dict(),
        "experiment": "B-label weight multiplier 1.5; one additional epoch",
        "base_checkpoint": BEST_PATH,
        "train_loss": train_loss,
        "val_loss": val_loss
    },
    NEXT_CANDIDATE
)

print(f"Train loss: {train_loss:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print("Candidate saved:", NEXT_CANDIDATE)
print("Existing best checkpoint unchanged.")


In [ ]:

NEXT_CANDIDATE = "/kaggle/working/a4_lorma_boundary_epoch4_candidate.pt"

print("Generating validation probabilities for the new candidate...")
epoch4_val_probs = regenerate_validation_predictions(NEXT_CANDIDATE)

print("Documents:", len(epoch4_val_probs))
print(
    "Total tokens:",
    sum(len(x["probs"]) for x in epoch4_val_probs.values())
)
print("Existing best checkpoint remains unchanged.")


In [ ]:

epoch4_metrics = score_probability_cache_fixed(epoch4_val_probs)

print("Current best F1: 0.174455")
print("Epoch 4 candidate:", epoch4_metrics)


In [ ]:

import os
import json
import shutil

candidate_path = (
    "/kaggle/working/a4_lorma_boundary_epoch4_candidate.pt"
)
new_best_path = "/kaggle/working/a4_lorma_boundary_best.pt"
config_path = "/kaggle/working/best_threshold_config_v2.json"

assert os.path.exists(candidate_path), "Candidate checkpoint not found."

# Preserve the previous best under a separate filename.
previous_best_path = "/kaggle/working/a4_lorma_boundary_best_previous.pt"
if os.path.exists(new_best_path):
    shutil.copy2(new_best_path, previous_best_path)

# Promote the evaluated candidate.
shutil.copy2(candidate_path, new_best_path)

config = {
    "checkpoint": new_best_path,
    "experiment": "B-label weight multiplier 1.5; 3 fine-tuning epochs total",
    "validation_fold": 2,
    "default_p_min": 0.35,
    "default_margin_min": 0.05,
    "class_thresholds": {
        "DATE": {"p_min": 0.60, "margin_min": -0.05},
        "STAT": {"p_min": 0.25, "margin_min": -0.05}
    },
    "validation_precision": epoch4_metrics["precision"],
    "validation_recall": epoch4_metrics["recall"],
    "validation_f1": epoch4_metrics["F1"],
    "validation_tp": epoch4_metrics["TP"],
    "validation_fp": epoch4_metrics["FP"],
    "validation_fn": epoch4_metrics["FN"]
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("New best checkpoint:", new_best_path)
print("Previous best backup:", previous_best_path)
print("Threshold configuration:", config_path)
print("New best validation F1:", round(config["validation_f1"], 6))


In [ ]:

import os
import json
import torch
import torch.nn as nn

BEST_PATH = "/kaggle/working/a4_lorma_boundary_best.pt"
CANDIDATE_PATH = "/kaggle/working/a4_lorma_boundary_epoch5_candidate.pt"

assert os.path.exists(BEST_PATH), "Current best checkpoint not found."
assert "fresh_model" in globals()
assert "experiment_weights" in globals()
assert "train_chunks_nonoverlap" in globals()
assert "val_chunks_nonoverlap" in globals()
assert "run_epoch" in globals()

# Load the current best model.
best_ckpt = torch.load(
    BEST_PATH, map_location=device, weights_only=False
)
fresh_model.load_state_dict(
    best_ckpt["model_state_dict"], strict=True
)

# Reuse the same loss weights and training settings.
criterion_candidate = nn.CrossEntropyLoss(
    weight=experiment_weights.to(device),
    ignore_index=-100
)

optimizer_candidate = torch.optim.AdamW(
    [p for p in fresh_model.parameters() if p.requires_grad],
    lr=2e-5,
    weight_decay=0.01
)

# Exactly one additional training epoch.
train_loss = run_epoch(train_chunks_nonoverlap, True)
val_loss = run_epoch(val_chunks_nonoverlap, False)

# Save as a separate candidate; do not overwrite the best.
torch.save(
    {
        "model_state_dict": fresh_model.state_dict(),
        "experiment": "B-label weight multiplier 1.5; additional epoch",
        "base_checkpoint": BEST_PATH,
        "train_loss": train_loss,
        "val_loss": val_loss
    },
    CANDIDATE_PATH
)

print(f"Training loss:   {train_loss:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print("Candidate saved:", CANDIDATE_PATH)
print("Current best checkpoint unchanged.")


In [ ]:

assert os.path.exists(CANDIDATE_PATH)

epoch5_val_probs = regenerate_validation_predictions(CANDIDATE_PATH)

print("Validation documents:", len(epoch5_val_probs))
print(
    "Validation tokens:",
    sum(len(item["probs"]) for item in epoch5_val_probs.values())
)


In [ ]:

epoch5_metrics = score_probability_cache_fixed(epoch5_val_probs)

CURRENT_BEST_F1 = 0.176510989010989

print("Current best F1:", round(CURRENT_BEST_F1, 6))
print("Epoch 5 candidate:", epoch5_metrics)

if epoch5_metrics["F1"] > CURRENT_BEST_F1:
    print("\nCandidate beats the current best.")
else:
    print("\nCandidate does not beat the current best.")


In [ ]:

import shutil

CURRENT_BEST = "/kaggle/working/a4_lorma_boundary_best.pt"
BACKUP_PATH = "/kaggle/working/a4_lorma_boundary_best_previous.pt"
CONFIG_PATH = "/kaggle/working/best_threshold_config_v2.json"

if epoch5_metrics["F1"] > CURRENT_BEST_F1:
    # Preserve the current best before promotion.
    shutil.copy2(CURRENT_BEST, BACKUP_PATH)
    shutil.copy2(CANDIDATE_PATH, CURRENT_BEST)

    config = {
        "checkpoint": CURRENT_BEST,
        "experiment": "B-label weight multiplier 1.5; additional epoch",
        "validation_fold": 2,
        "default_p_min": 0.35,
        "default_margin_min": 0.05,
        "class_thresholds": {
            "DATE": {"p_min": 0.60, "margin_min": -0.05},
            "STAT": {"p_min": 0.25, "margin_min": -0.05}
        },
        "validation_precision": epoch5_metrics["precision"],
        "validation_recall": epoch5_metrics["recall"],
        "validation_f1": epoch5_metrics["F1"],
        "validation_tp": epoch5_metrics["TP"],
        "validation_fp": epoch5_metrics["FP"],
        "validation_fn": epoch5_metrics["FN"]
    }

    with open(CONFIG_PATH, "w") as f:
        json.dump(config, f, indent=2)

    print("New best checkpoint promoted.")
    print("New best F1:", round(epoch5_metrics["F1"], 6))
else:
    print("Candidate rejected; existing best checkpoint preserved.")
    print("Current best F1:", round(CURRENT_BEST_F1, 6))


In [ ]:
v
import os
import json

paths = {
    "Current best checkpoint":
        "/kaggle/working/a4_lorma_boundary_best.pt",
    "Previous best backup":
        "/kaggle/working/a4_lorma_boundary_best_previous.pt",
    "Threshold configuration":
        "/kaggle/working/best_threshold_config_v2.json",
    "Original baseline checkpoint":
        "/kaggle/working/a4_lorma_weighted_best.pt",
    "Epoch 5 candidate":
        "/kaggle/working/a4_lorma_boundary_epoch5_candidate.pt",
    "Existing submission JSON":
        "/kaggle/working/submission/best_model_test_predictions.json",
    "Existing submission ZIP":
        "/kaggle/working/submission.zip",
}

print("=== ARTIFACT CHECK ===")
for name, path in paths.items():
    if os.path.isfile(path):
        print(f"[FOUND]   {name}: {os.path.getsize(path):,} bytes")
    else:
        print(f"[MISSING] {name}: {path}")

config_path = paths["Threshold configuration"]
if os.path.isfile(config_path):
    with open(config_path) as f:
        config = json.load(f)

    print("\n=== SAVED CONFIGURATION ===")
    print(json.dumps(config, indent=2))

import os
import json

paths = {
    "Current best checkpoint":
        "/kaggle/working/a4_lorma_boundary_best.pt",
    "Previous best backup":
        "/kaggle/working/a4_lorma_boundary_best_previous.pt",
    "Threshold configuration":
        "/kaggle/working/best_threshold_config_v2.json",
    "Original baseline checkpoint":
        "/kaggle/working/a4_lorma_weighted_best.pt",
    "Epoch 5 candidate":
        "/kaggle/working/a4_lorma_boundary_epoch5_candidate.pt",
    "Existing submission JSON":
        "/kaggle/working/submission/best_model_test_predictions.json",
    "Existing submission ZIP":
        "/kaggle/working/submission.zip",
}

print("=== ARTIFACT CHECK ===")
for name, path in paths.items():
    if os.path.isfile(path):
        print(f"[FOUND]   {name}: {os.path.getsize(path):,} bytes")
    else:
        print(f"[MISSING] {name}: {path}")

config_path = paths["Threshold configuration"]
if os.path.isfile(config_path):
    with open(config_path) as f:
        config = json.load(f)

    print("\n=== SAVED CONFIGURATION ===")
    print(json.dumps(config, indent=2))


In [ ]:

import os
import json
import zipfile
from collections import Counter

json_path = (
    "/kaggle/working/submission/"
    "best_model_test_predictions.json"
)
zip_path = "/kaggle/working/submission.zip"

print("=== SUBMISSION JSON ===")
with open(json_path, "r", encoding="utf-8") as f:
    submission_data = json.load(f)

print("Top-level type:", type(submission_data).__name__)

if isinstance(submission_data, dict):
    print("Number of top-level keys:", len(submission_data))
    print("Sample keys:", list(submission_data.keys())[:5])
elif isinstance(submission_data, list):
    print("Number of records:", len(submission_data))
    print("First record:", submission_data[0] if submission_data else None)

print("\n=== ZIP CONTENTS ===")
with zipfile.ZipFile(zip_path, "r") as z:
    for info in z.infolist():
        print(f"{info.filename}: {info.file_size:,} bytes")

print("\n=== TEST DOCUMENT COUNT ===")
print("Expected test documents:", len(test_texts) if "test_texts" in globals() else "test_texts variable unavailable")

if isinstance(submission_data, dict):
    print("Submission keys:", len(submission_data))


In [ ]:

import json
import os
from collections import Counter

submission_path = (
    "/kaggle/working/submission/"
    "best_model_test_predictions.json"
)
test_path = "/kaggle/input/datasets/aq1de3/fold3-te/fold3-text.json"

with open(submission_path, encoding="utf-8") as f:
    submission = json.load(f)

with open(test_path, encoding="utf-8") as f:
    test_data = json.load(f)

print("=== DATASET COMPARISON ===")
print("Test documents:", len(test_data))
print("Submission documents:", len(submission))

test_ids = {str(k) for k in test_data}
submission_ids = {str(k) for k in submission}

print("Missing submission IDs:", len(test_ids - submission_ids))
print("Unexpected submission IDs:", len(submission_ids - test_ids))

print("\n=== SAMPLE SUBMISSION RECORD ===")
sample_id = next(iter(submission))
print("Document ID:", sample_id)
print("Value type:", type(submission[sample_id]).__name__)

value = submission[sample_id]
if isinstance(value, dict):
    print("Record keys:", list(value.keys())[:20])
    for key, item in list(value.items())[:5]:
        print(f"{key}: type={type(item).__name__}, sample={str(item)[:180]}")
elif isinstance(value, list):
    print("Number of items:", len(value))
    print("First items:", value[:3])
else:
    print("Value:", str(value)[:500])


In [ ]:

import json

submission_path = (
    "/kaggle/working/submission/"
    "best_model_test_predictions.json"
)
test_path = "/kaggle/input/datasets/aq1de3/fold3-te/fold3-text.json"

with open(submission_path, encoding="utf-8") as f:
    submission = json.load(f)

with open(test_path, encoding="utf-8") as f:
    test_data = json.load(f)

# Build a reliable ID -> label-name mapping.
id_to_label = {int(v): str(k) for k, v in label2id.items()}
valid_label_ids = set(id_to_label)

errors = []
total_tokens = 0

for doc_id, record in submission.items():
    text = test_data[doc_id]
    if isinstance(text, dict):
        text = text.get("text", "")

    predictions = record.get("predictions", [])
    offsets = record.get("offsets", [])

    if len(predictions) != len(offsets):
        errors.append(f"{doc_id}: prediction/offset length mismatch")
        continue

    total_tokens += len(predictions)

    for i, (label_id, offset) in enumerate(zip(predictions, offsets)):
        if label_id not in valid_label_ids:
            errors.append(f"{doc_id}: invalid label ID {label_id} at token {i}")
            break

        if not isinstance(offset, (list, tuple)) or len(offset) != 2:
            errors.append(f"{doc_id}: malformed offset at token {i}")
            break

        start, end = offset
        if not (isinstance(start, int) and isinstance(end, int)):
            errors.append(f"{doc_id}: non-integer offset at token {i}")
            break

        if not (0 <= start <= end <= len(text)):
            errors.append(f"{doc_id}: out-of-range offset at token {i}")
            break

print("Documents checked:", len(submission))
print("Total prediction tokens:", total_tokens)
print("Structural errors:", len(errors))

for err in errors[:20]:
    print("ERROR:", err)

if not errors:
    print("PASS: basic submission structure and offsets are valid.")


In [ ]:

import os
import json
import torch

# ---------- Paths ----------
BEST_PATH = "/kaggle/working/a4_lorma_boundary_best.pt"
CONFIG_PATH = "/kaggle/working/best_threshold_config_v2.json"
TEST_PATH = "/kaggle/input/datasets/aq1de3/fold3-te/fold3-text.json"
OUTPUT_PATH = "/kaggle/working/submission/latest_best_predictions.json"

for path in [BEST_PATH, CONFIG_PATH, TEST_PATH]:
    assert os.path.isfile(path), f"Required file missing: {path}"

# ---------- Load config and test data ----------
with open(CONFIG_PATH, encoding="utf-8") as f:
    config = json.load(f)

with open(TEST_PATH, encoding="utf-8") as f:
    test_texts = json.load(f)

assert "fresh_model" in globals(), "Model definition/session missing."
assert "predict_doc_probabilities" in globals(), (
    "Prediction function missing. Run the earlier model setup cells first."
)
assert "label2id" in globals(), "Label mapping missing."

# ---------- Load the latest best checkpoint ----------
checkpoint = torch.load(
    BEST_PATH, map_location=device, weights_only=False
)
fresh_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
fresh_model.eval()

id_to_label = {int(v): str(k) for k, v in label2id.items()}
o_id = int(label2id["O"])

default_p = config["default_p_min"]
default_m = config["default_margin_min"]
class_thresholds = config["class_thresholds"]

# ---------- Generate predictions ----------
def predict_with_saved_thresholds(text):
    offsets, probs = predict_doc_probabilities(text)
    predictions = []

    for i in range(len(offsets)):
        best_id = int(probs[i].argmax())
        label = id_to_label[best_id]

        if label == "O":
            predictions.append(best_id)
            continue

        entity_type = label.split("-", 1)[1]
        thresholds = class_thresholds.get(entity_type, {})
        p_min = thresholds.get("p_min", default_p)
        margin_min = thresholds.get("margin_min", default_m)

        p_entity = float(probs[i, best_id])
        p_o = float(probs[i, o_id])

        if p_entity >= p_min and p_entity - p_o >= margin_min:
            predictions.append(best_id)
        else:
            predictions.append(o_id)

    return predictions, offsets

new_submission = {}

for doc_id, value in test_texts.items():
    text = value["text"] if isinstance(value, dict) else value
    predictions, offsets = predict_with_saved_thresholds(text)

    new_submission[str(doc_id)] = {
        "predictions": [int(x) for x in predictions],
        "offsets": [[int(s), int(e)] for s, e in offsets]
    }

# ---------- Save separately ----------
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(new_submission, f, ensure_ascii=False)

# ---------- Validate ----------
errors = []
test_ids = {str(k) for k in test_texts}
output_ids = set(new_submission)

if test_ids != output_ids:
    errors.append("Test document IDs do not match.")

total_tokens = 0

for doc_id, record in new_submission.items():
    value = test_texts[doc_id]
    text = value["text"] if isinstance(value, dict) else value
    predictions = record["predictions"]
    offsets = record["offsets"]

    if len(predictions) != len(offsets):
        errors.append(f"{doc_id}: prediction/offset length mismatch")
        continue

    total_tokens += len(predictions)

    for label_id, (start, end) in zip(predictions, offsets):
        if label_id not in id_to_label:
            errors.append(f"{doc_id}: invalid label ID {label_id}")
            break
        if not (0 <= start <= end <= len(text)):
            errors.append(f"{doc_id}: invalid text offset")
            break

print("Checkpoint:", BEST_PATH)
print("Documents:", len(new_submission))
print("Prediction tokens:", total_tokens)
print("Structural errors:", len(errors))
print("Output file exists:", os.path.isfile(OUTPUT_PATH))
print("Output path:", OUTPUT_PATH)

for error in errors[:10]:
    print("ERROR:", error)

if not errors:
    print("PASS: generated predictions are structurally valid.")
    print("Existing submission ZIP was not changed.")


In [ ]:

import os
import json
import zipfile

SOURCE_JSON = (
    "/kaggle/working/submission/latest_best_predictions.json"
)
FINAL_DIR = "/kaggle/working/final_submission"
FINAL_JSON = os.path.join(
    FINAL_DIR, "best_model_test_predictions.json"
)
FINAL_ZIP = "/kaggle/working/final_submission.zip"

assert os.path.isfile(SOURCE_JSON), "Regenerated JSON not found."

os.makedirs(FINAL_DIR, exist_ok=True)

# Copy the validated JSON into the final package directory.
with open(SOURCE_JSON, encoding="utf-8") as f:
    predictions = json.load(f)

with open(FINAL_JSON, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)

# Create a new ZIP with the same internal member path as the old ZIP.
with zipfile.ZipFile(
    FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    z.write(
        FINAL_JSON,
        arcname="submission/best_model_test_predictions.json"
    )

# Verify the archive and its contents.
with zipfile.ZipFile(FINAL_ZIP, "r") as z:
    assert z.testzip() is None, "ZIP integrity check failed."
    names = z.namelist()
    assert names == [
        "submission/best_model_test_predictions.json"
    ], f"Unexpected ZIP members: {names}"

    packaged = json.loads(
        z.read(names[0]).decode("utf-8")
    )
    assert len(packaged) == 35
    assert set(packaged) == set(predictions)

print("Final candidate ZIP:", FINAL_ZIP)
print("ZIP size:", f"{os.path.getsize(FINAL_ZIP):,} bytes")
print("Documents packaged:", len(packaged))
print("ZIP integrity: PASS")
print("Original submission ZIP preserved.")


In [ ]:

import os

roots = ["/kaggle/input"]

keywords = (
    "submission", "sample", "instruction",
    "readme", "codabench", "phase2", "evaluation"
)

matches = []

for root in roots:
    if not os.path.isdir(root):
        continue

    for current, dirs, files in os.walk(root):
        # Avoid traversing excessively large unrelated trees.
        dirs[:] = dirs[:]

        for filename in files:
            lower = filename.lower()
            if any(word in lower for word in keywords):
                matches.append(os.path.join(current, filename))

print("Potential instruction/sample files:", len(matches))
for path in matches[:100]:
    print(path)


In [54]:

import os
import json
import zipfile

# Input: predictions generated from your latest best checkpoint
INPUT_JSON = (
    "/kaggle/working/submission/latest_best_predictions.json"
)
TEST_PATH = "/kaggle/input/datasets/aq1de3/fold3-te/fold3-text.json"

OUTPUT_JSON = "/kaggle/working/prediction.json"
OUTPUT_ZIP = "/kaggle/working/final_submission_entities.zip"

assert os.path.isfile(INPUT_JSON), f"Missing predictions: {INPUT_JSON}"
assert os.path.isfile(TEST_PATH), f"Missing test text: {TEST_PATH}"
assert "label2id" in globals(), "Run the model/label setup cells first."

with open(INPUT_JSON, encoding="utf-8") as f:
    raw_predictions = json.load(f)

with open(TEST_PATH, encoding="utf-8") as f:
    test_texts = json.load(f)

id_to_label = {int(v): str(k) for k, v in label2id.items()}
O_ID = int(label2id["O"])

def extract_entities(pred_ids, offsets, text):
    entities = []
    active_type = None
    active_start = None
    active_end = None

    def close_entity():
        nonlocal active_type, active_start, active_end

        if active_type is not None:
            start = active_start
            end = active_end

            # Remove whitespace included in tokenizer offsets.
            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1

            if end > start:
                entities.append([
                    int(start),
                    int(end),
                    str(active_type),
                    text[start:end]
                ])

        active_type = None
        active_start = None
        active_end = None

    for label_id, offset in zip(pred_ids, offsets):
        start, end = map(int, offset)

        if end <= start:
            close_entity()
            continue

        label = id_to_label[int(label_id)]

        if label == "O":
            close_entity()
            continue

        prefix, entity_type = label.split("-", 1)

        if prefix == "B":
            close_entity()
            active_type = entity_type
            active_start = start
            active_end = end

        elif prefix == "I":
            if active_type == entity_type:
                active_end = end
            else:
                # Ignore orphan I-tags instead of inventing an entity.
                close_entity()

    close_entity()
    return entities

# Convert every document to a list of entity spans.
entity_predictions = {}

for doc_id, record in raw_predictions.items():
    if doc_id not in {str(k) for k in test_texts}:
        raise ValueError(f"Unexpected document ID: {doc_id}")

    source = test_texts[doc_id]
    text = source["text"] if isinstance(source, dict) else source

    token_ids = record["predictions"]
    offsets = record["offsets"]

    if len(token_ids) != len(offsets):
        raise ValueError(f"Token/offset mismatch in document {doc_id}")

    entity_predictions[doc_id] = extract_entities(
        token_ids, offsets, text
    )

# Save JSON in entity-list format.
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(entity_predictions, f, ensure_ascii=False)

# Create a separate ZIP with prediction.json at the root.
with zipfile.ZipFile(
    OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    z.write(OUTPUT_JSON, arcname="prediction.json")

# Verify basic output structure and archive integrity.
with open(OUTPUT_JSON, encoding="utf-8") as f:
    check = json.load(f)

assert len(check) == 35
assert all(isinstance(v, list) for v in check.values())

with zipfile.ZipFile(OUTPUT_ZIP, "r") as z:
    assert z.namelist() == ["prediction.json"]
    assert z.testzip() is None

print("Documents:", len(check))
print("Total extracted entities:", sum(len(v) for v in check.values()))
print("Example document:", next(iter(check.items())))
print("JSON:", OUTPUT_JSON)
print("Corrected ZIP:", OUTPUT_ZIP)
print("ZIP integrity: PASS")
print("Original submission ZIP preserved.")


AssertionError: Missing predictions: /kaggle/working/submission/latest_best_predictions.json

In [6]:

import os

paths = [
    "/kaggle/working/a4_lorma_boundary_best.pt",
    "/kaggle/working/a4_lorma_boundary_best_previous.pt",
    "/kaggle/working/best_threshold_config_v2.json",
    "/kaggle/working/submission/best_model_test_predictions.json",
    "/kaggle/working/submission.zip",
    "/kaggle/working/final_submission.zip",
]

for path in paths:
    if os.path.isfile(path):
        print(f"FOUND   {path} ({os.path.getsize(path):,} bytes)")
    else:
        print(f"MISSING {path}")


MISSING /kaggle/working/a4_lorma_boundary_best.pt
MISSING /kaggle/working/a4_lorma_boundary_best_previous.pt
MISSING /kaggle/working/best_threshold_config_v2.json
MISSING /kaggle/working/submission/best_model_test_predictions.json
MISSING /kaggle/working/submission.zip
MISSING /kaggle/working/final_submission.zip


In [7]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for name in files:
        if name.endswith((".pt", ".json", ".zip")):
            print(os.path.join(root, name))

/kaggle/input/datasets/aq1de3/a1check/architecture1_sustained_step_500.pt
/kaggle/input/datasets/aq1de3/fold2-go/fold2-gold.json
/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json
/kaggle/input/datasets/aq1de3/fold2-tex/fold2-text.json
/kaggle/input/datasets/aq1de3/a4-ner-exp4-lorma-epoch3/a4_ner_exp4_lorma_epoch3.pt
/kaggle/input/datasets/aq1de3/a4-lorma-boundary-best/a4_lorma_boundary_best.pt
/kaggle/input/datasets/aq1de3/fold1-gold1/fold1-gold.json
/kaggle/input/datasets/aq1de3/a4check/a4_step_22000.pt
/kaggle/input/datasets/aq1de3/a4-lorma/a4_lorma_balanced_v2.pt
/kaggle/input/datasets/aq1de3/a4-lorma/a4_lorma_normfix_candidate.pt
/kaggle/input/datasets/aq1de3/a4-lorma/a4_lorma_normfix_candidate (1).pt
/kaggle/input/datasets/aq1de3/a4-lorma/a4_lorma_boundary_epoch4_candidate.pt
/kaggle/input/datasets/aq1de3/a4-lorma/a4_lorma_weighted_best.pt
/kaggle/input/datasets/aq1de3/a4-lorma/a4_lorma_balanced_v2 (1).pt
/kaggle/input/datasets/aq1de3/a4-lorma/a4_lorma_weighted_best (1).pt
/k

In [8]:
import os
import shutil

os.makedirs("/kaggle/working/submission", exist_ok=True)

targets = {
    "a4_lorma_boundary_best.pt":
        "/kaggle/working/a4_lorma_boundary_best.pt",

    "a4_lorma_boundary_best_previous.pt":
        "/kaggle/working/a4_lorma_boundary_best_previous.pt",

    "best_threshold_config_v2.json":
        "/kaggle/working/best_threshold_config_v2.json",

    "best_model_test_predictions.json":
        "/kaggle/working/submission/best_model_test_predictions.json",
}

found = {}
for root, dirs, files in os.walk("/kaggle/input"):
    for name in files:
        if name in targets and name not in found:
            found[name] = os.path.join(root, name)

for name, destination in targets.items():
    source = found.get(name)

    if source:
        shutil.copy2(source, destination)
        print("COPIED:", name, "->", destination)
    else:
        print("NOT FOUND:", name)

COPIED: a4_lorma_boundary_best.pt -> /kaggle/working/a4_lorma_boundary_best.pt
COPIED: a4_lorma_boundary_best_previous.pt -> /kaggle/working/a4_lorma_boundary_best_previous.pt
COPIED: best_threshold_config_v2.json -> /kaggle/working/best_threshold_config_v2.json
COPIED: best_model_test_predictions.json -> /kaggle/working/submission/best_model_test_predictions.json


In [10]:
for path in [
    "/kaggle/working/a4_lorma_boundary_best.pt",
    "/kaggle/working/a4_lorma_boundary_best_previous.pt",
    "/kaggle/working/best_threshold_config_v2.json",
    "/kaggle/working/submission/best_model_test_predictions.json",
]:
    print("FOUND" if os.path.isfile(path) else "MISSING", path)

FOUND /kaggle/working/a4_lorma_boundary_best.pt
FOUND /kaggle/working/a4_lorma_boundary_best_previous.pt
FOUND /kaggle/working/best_threshold_config_v2.json
FOUND /kaggle/working/submission/best_model_test_predictions.json


In [11]:
import json
import os

config_path = "/kaggle/working/best_threshold_config_v2.json"
pred_path = "/kaggle/working/submission/best_model_test_predictions.json"

with open(config_path) as f:
    config = json.load(f)

with open(pred_path) as f:
    predictions = json.load(f)

print("Threshold config type:", type(config).__name__)
print("Threshold config:", config)
print("Prediction documents:", len(predictions))
print("First document ID:", next(iter(predictions)))

first_id = next(iter(predictions))
print("First prediction structure:")
print(str(predictions[first_id])[:1000])

Threshold config type: dict
Threshold config: {'checkpoint': '/kaggle/working/a4_lorma_boundary_best.pt', 'experiment': 'B-label weight multiplier 1.5; additional epoch', 'validation_fold': 2, 'default_p_min': 0.35, 'default_margin_min': 0.05, 'class_thresholds': {'DATE': {'p_min': 0.6, 'margin_min': -0.05}, 'STAT': {'p_min': 0.25, 'margin_min': -0.05}}, 'validation_precision': 0.3299684542586751, 'validation_recall': 0.12239644278024807, 'validation_f1': 0.1785592352338682, 'validation_tp': 523, 'validation_fp': 1062, 'validation_fn': 3750}
Prediction documents: 35
First document ID: 1128835
First prediction structure:
{'predictions': [0, 9, 9, 10, 18, 0, 4, 0, 11, 13, 2, 3, 0, 24, 24, 18, 7, 16, 17, 2, 10, 0, 4, 18, 24, 24, 17, 24, 0, 10, 21, 23, 0, 0, 13, 4, 13, 2, 9, 9, 10, 2, 18, 10, 10, 24, 1, 0, 10, 23, 0, 5, 5, 0, 24, 18, 0, 0, 16, 1, 0, 10, 24, 14, 0, 16, 0, 23, 10, 0, 24, 16, 0, 24, 0, 24, 0, 24, 24, 10, 0, 0, 17, 0, 17, 0, 9, 24, 16, 2, 0, 17, 0, 24, 20, 0, 22, 18, 24, 0, 0,

In [12]:
import os
import json

pred_path = "/kaggle/working/submission/best_model_test_predictions.json"

with open(pred_path, encoding="utf-8") as f:
    predictions = json.load(f)

doc_id = next(iter(predictions))
record = predictions[doc_id]

print("Document ID:", doc_id)
print("Prediction keys:", list(record.keys()))
print("Number of labels:", len(record["predictions"]))
print("Number of offsets:", len(record.get("offsets", [])))
print("First 15 offsets:", record.get("offsets", [])[:15])

print("\nSearching for test-text JSON:")
for root, dirs, files in os.walk("/kaggle/input"):
    for name in files:
        if "fold3-text" in name.lower():
            print(os.path.join(root, name))

Document ID: 1128835
Prediction keys: ['predictions', 'offsets']
Number of labels: 10473
Number of offsets: 10473
First 15 offsets: [[0, 10], [10, 13], [13, 17], [17, 25], [25, 31], [31, 34], [34, 40], [40, 46], [46, 56], [56, 69], [69, 75], [75, 82], [82, 85], [85, 86], [86, 88]]

Searching for test-text JSON:
/kaggle/input/datasets/aq1de3/fold3-te/fold3-text.json


In [13]:
import json
import os

pred_path = "/kaggle/working/submission/best_model_test_predictions.json"
text_path = "/kaggle/input/datasets/aq1de3/fold3-te/fold3-text.json"

with open(pred_path, encoding="utf-8") as f:
    predictions = json.load(f)

with open(text_path, encoding="utf-8") as f:
    texts = json.load(f)

print("Prediction documents:", len(predictions))
print("Test-text object type:", type(texts).__name__)

if isinstance(texts, dict):
    print("Test-text document count:", len(texts))
    first_text_id = next(iter(texts))
    print("Sample text ID:", first_text_id)
    print("Sample text value type:", type(texts[first_text_id]).__name__)
    print("Sample text preview:", str(texts[first_text_id])[:200])
else:
    print("Sample text record:", str(texts[0])[:300])

print("\nChecking predictions:")
bad = []

for doc_id, record in predictions.items():
    labels = record.get("predictions", [])
    offsets = record.get("offsets", [])

    if len(labels) != len(offsets):
        bad.append((doc_id, "label/offset length mismatch"))
    elif any(not isinstance(x, int) or x < 0 or x > 24 for x in labels):
        bad.append((doc_id, "invalid label ID"))

print("Validation problems:", bad[:10])
print("Total problems:", len(bad))

Prediction documents: 35
Test-text object type: dict
Test-text document count: 35
Sample text ID: 1128835
Sample text value type: str
Sample text preview: REPORTABLE IN THE SUPREME COURT OF INDIA CIVIL APPELLATE JURISDICTION CIVIL APPEAL NO.6580 OF 2008 ARISING OUT OF SPECIAL LEAVE PETITION (CIVIL) NO. 15449 OF 2004 ESSCO FABS PVT. LTD. & ANR. ... APPEL

Checking predictions:
Validation problems: []
Total problems: 0


In [14]:
import json
import os
import zipfile

pred_path = "/kaggle/working/submission/best_model_test_predictions.json"
text_path = "/kaggle/input/datasets/aq1de3/fold3-te/fold3-text.json"

# BIO label IDs used by the Phase 2 model
label_names = {
    1: "APP", 2: "APP",
    3: "RESP", 4: "RESP",
    5: "A.COUNSEL", 6: "A.COUNSEL",
    7: "R.COUNSEL", 8: "R.COUNSEL",
    9: "JUDGE", 10: "JUDGE",
    11: "WIT", 12: "WIT",
    13: "AUTH", 14: "AUTH",
    15: "COURT", 16: "COURT",
    17: "STAT", 18: "STAT",
    19: "PREC", 20: "PREC",
    21: "DATE", 22: "DATE",
    23: "CASENO", 24: "CASENO",
}

with open(pred_path, encoding="utf-8") as f:
    predictions = json.load(f)

with open(text_path, encoding="utf-8") as f:
    texts = json.load(f)

def decode_entities(record, text):
    labels = record["predictions"]
    offsets = record["offsets"]
    entities = []
    start = None
    end = None
    entity_type = None

    def close_entity():
        nonlocal start, end, entity_type
        if start is not None:
            s, e = start, end
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            if s < e:
                entities.append([s, e, entity_type])
        start = end = entity_type = None

    for label, offset in zip(labels, offsets):
        token_start, token_end = offset

        if label == 0:
            close_entity()
            continue

        prefix = "B" if label % 2 == 1 else "I"
        current_type = label_names[label]

        if prefix == "B" or start is None or current_type != entity_type:
            close_entity()
            start, end, entity_type = token_start, token_end, current_type
        else:
            end = token_end

    close_entity()
    return entities

output = {}

for doc_id, record in predictions.items():
    if doc_id not in texts:
        raise ValueError(f"Missing test text for document {doc_id}")
    output[doc_id] = decode_entities(record, texts[doc_id])

assert len(output) == len(predictions) == 35
assert all(isinstance(entities, list) for entities in output.values())

# Save separately; do not overwrite the recovered token predictions.
out_json = "/kaggle/working/prediction_corrected.json"
out_zip = "/kaggle/working/submission_corrected.zip"

with open(out_json, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False)

with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(out_json, arcname="prediction.json")

# Verify archive structure and document values.
with zipfile.ZipFile(out_zip, "r") as z:
    assert z.namelist() == ["prediction.json"]
    restored = json.loads(z.read("prediction.json"))

assert len(restored) == 35
assert all(isinstance(v, list) for v in restored.values())

print("Created:", out_zip)
print("Documents:", len(restored))
print("Entities:", sum(len(v) for v in restored.values()))
print("Archive members:", zipfile.ZipFile(out_zip).namelist())
print("Sample entities:", restored[next(iter(restored))][:10])

Created: /kaggle/working/submission_corrected.zip
Documents: 35
Entities: 21003
Archive members: ['prediction.json']
Sample entities: [[11, 13, 'JUDGE'], [14, 25, 'JUDGE'], [26, 31, 'STAT'], [35, 40, 'RESP'], [47, 56, 'WIT'], [57, 69, 'AUTH'], [70, 75, 'APP'], [76, 82, 'RESP'], [85, 88, 'CASENO'], [88, 90, 'STAT']]


In [15]:
import os
import shutil

src = "/kaggle/working/submission/submission_corrected.zip"
dst = "/kaggle/working/submission_corrected.zip"

if os.path.exists(src):
    shutil.copy2(src, dst)
    print("Ready to download:", dst)
elif os.path.exists(dst):
    print("Already available:", dst)
else:
    print("ZIP not found. Check the path.")

Already available: /kaggle/working/submission_corrected.zip
